In [ ]:
# Cell 1: Imports, Live Game Data Retrieval in Pacific Time

import pandas as pd
import pytz
from datetime import datetime
import traceback
from caching.supabase_client import supabase

def fetch_live_games_in_pacific_time():
    """
    Fetch actively live games from 'nba_live_game_stats' using the stored 'status'
    field. A game is considered live if its status is one of:
      {"Q1", "Q2", "Q3", "Q4", "OT", "BT", "HT"}
    """
    live_statuses = {"Q1", "Q2", "Q3", "Q4", "OT", "BT", "HT"}
    
    try:
        response = supabase.table("nba_live_game_stats").select("*").execute()
        if not response.data:
            print("No data found in 'nba_live_game_stats' table.")
            return pd.DataFrame()
        
        df = pd.DataFrame(response.data)
        
        # Ensure 'game_date' exists
        if "game_date" not in df.columns:
            print("Column 'game_date' not found!")
            return pd.DataFrame()
        
        # Convert game_date (stored as fake UTC) to datetime and then to Pacific Time
        df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce", utc=True)
        pacific_tz = pytz.timezone("America/Los_Angeles")
        df["game_date_pt"] = df["game_date"].dt.tz_convert(pacific_tz)
        df["date_only_pt"] = df["game_date_pt"].dt.date
        
        # Filter for games scheduled for today in Pacific Time
        now_pt = datetime.now(pacific_tz)
        today_pt = now_pt.date()
        df_today = df[df["date_only_pt"] == today_pt].copy()
        if df_today.empty:
            print("No games match today's date in Pacific Time.")
            return pd.DataFrame()
        
        # Make sure the status field is available and in uppercase
        if "status" not in df_today.columns:
            print("Column 'status' not found!")
            return pd.DataFrame()
        df_today["status"] = df_today["status"].astype(str).str.upper()
        
        # Filter only active games (using the stored status)
        active_games = df_today[df_today["status"].isin(live_statuses)].copy()
        
        print(f"ACTIVELY LIVE GAMES: {len(active_games)}")
        if not active_games.empty:
            for _, row in active_games.iterrows():
                print(
                    f"{row.get('home_team', 'Unknown')} vs {row.get('away_team', 'Unknown')}: "
                    f"{row.get('home_score', 'N/A')}-{row.get('away_score', 'N/A')}, "
                    f"Status: {row.get('status')}"
                )
        else:
            print("No active games found!")
        
        return active_games
        
    except Exception as e:
        print(f"Error fetching live games: {e}")
        traceback.print_exc()
        return pd.DataFrame()

# Test the function
live_games = fetch_live_games_in_pacific_time()
if not live_games.empty:
    print("Successfully retrieved active games (Pacific Time).")
else:
    print("No active game data available.")


In [ ]:
# Cell 2: Imports and Helper Functions

import pandas as pd
import numpy as np
import joblib
from sqlalchemy import create_engine
import config

def convert_time_to_minutes(time_str: str) -> float:
    """
    Convert a time string "MM:SS" to a float (minutes + seconds/60).
    Returns None if conversion fails.
    """
    if pd.isna(time_str) or ":" not in str(time_str):
        return None
    try:
        minutes, seconds = str(time_str).split(":")
        return float(minutes) + float(seconds) / 60.0
    except Exception as e:
        print(f"Error converting time: {e}")
        return None
    
# Import the NBAFeatureGenerator from our unified module
from models.features import NBAFeatureGenerator

# Create an instance of the feature generator with debugging enabled
feature_generator = NBAFeatureGenerator(debug=True)

def ensure_numeric_features(df, feature_columns, flag_critical=True):
    """
    Ensures all feature columns are numeric, replacing NaN/None values with appropriate defaults.
    Also flags critical missing values for review.
    
    Args:
        df (DataFrame): Input DataFrame
        feature_columns (list): List of column names to process
        flag_critical (bool): Whether to flag critical missing values
        
    Returns:
        DataFrame: DataFrame with ensured numeric features
        dict: Dictionary of flags for critical missing values (if flag_critical=True)
    """
    # Make a copy to avoid modifying the original
    result_df = df.copy()
    
    # Default values based on column type
    default_map = {
        'home_q1': 0, 'home_q2': 0, 'home_q3': 0, 'home_q4': 0,
        'away_q1': 0, 'away_q2': 0, 'away_q3': 0, 'away_q4': 0,
        'home_score': 0, 'away_score': 0,
        'rolling_home_score': 105.0, 'rolling_away_score': 105.0,
        'score_ratio': 0.5,
        'prev_matchup_diff': 0,
        'rest_days_home': 2, 'rest_days_away': 2, 'rest_advantage': 0,
        'is_back_to_back_home': 0, 'is_back_to_back_away': 0,
        'q1_to_q2_momentum': 0, 'q2_to_q3_momentum': 0, 'q3_to_q4_momentum': 0,
        'cumulative_momentum': 0
    }
    
    # Critical columns where missing values might significantly impact predictions
    critical_columns = ['home_q1', 'home_q2', 'home_q3', 'home_q4', 
                       'away_q1', 'away_q2', 'away_q3', 'away_q4',
                       'score_ratio', 'cumulative_momentum']
    
    # For any column not in default_map, use 0 as default
    for col in feature_columns:
        if col not in default_map:
            default_map[col] = 0
    
    # Dictionary to track critical missing values
    missing_critical = {}
    
    # Process each column
    for col in feature_columns:
        if col in result_df.columns:
            # Store original NaN count for critical columns
            if flag_critical and col in critical_columns:
                nan_count = result_df[col].isna().sum()
                if nan_count > 0:
                    missing_critical[col] = nan_count
            
            # Convert to numeric, forcing errors to NaN
            result_df[col] = pd.to_numeric(result_df[col], errors='coerce')
            
            # Replace NaN values with appropriate defaults
            result_df[col] = result_df[col].fillna(default_map.get(col, 0))
        else:
            # If column doesn't exist, add it with default values
            result_df[col] = default_map.get(col, 0)
    
    # Print summary of critical missing values if requested
    if flag_critical and missing_critical:
        print("WARNING: Critical missing values detected:")
        for col, count in missing_critical.items():
            print(f"  • {col}: {count} missing values")
    
    if flag_critical:
        return result_df, missing_critical
    else:
        return result_df



In [ ]:
# Cell 2B: Helper functions for data safety and compatibility

def ensure_numeric_features(df, feature_columns):
    """
    Ensures all feature columns are numeric, replacing NaN/None values with appropriate defaults.
    
    Args:
        df (DataFrame): Input DataFrame
        feature_columns (list): List of column names to process
        
    Returns:
        DataFrame: DataFrame with ensured numeric features
    """
    # Make a copy to avoid modifying the original
    result_df = df.copy()
    
    # Default values based on column type
    default_map = {
        'home_q1': 0, 'home_q2': 0, 'home_q3': 0, 'home_q4': 0,
        'away_q1': 0, 'away_q2': 0, 'away_q3': 0, 'away_q4': 0,
        'home_score': 0, 'away_score': 0,
        'rolling_home_score': 105.0, 'rolling_away_score': 105.0,
        'score_ratio': 0.5,
        'prev_matchup_diff': 0,
        'rest_days_home': 2, 'rest_days_away': 2, 'rest_advantage': 0,
        'is_back_to_back_home': 0, 'is_back_to_back_away': 0,
        'q1_to_q2_momentum': 0, 'q2_to_q3_momentum': 0, 'q3_to_q4_momentum': 0,
        'cumulative_momentum': 0
    }
    
    # For any column not in default_map, use 0 as default
    for col in feature_columns:
        if col not in default_map:
            default_map[col] = 0
    
    # Process each column
    for col in feature_columns:
        if col in result_df.columns:
            # Convert to numeric, forcing errors to NaN
            result_df[col] = pd.to_numeric(result_df[col], errors='coerce')
            
            # Replace NaN values with appropriate defaults
            result_df[col] = result_df[col].fillna(default_map.get(col, 0))
        else:
            # If column doesn't exist, add it with default values
            result_df[col] = default_map.get(col, 0)
    
    return result_df

def calculate_derived_features(df, imputation_strategy='mean'):
    """
    Calculate derived features from game data with improved imputation.
    
    Args:
        df: DataFrame with raw features
        imputation_strategy: Strategy for imputation ('mean', 'median', 'zero', or 'default')
        
    Returns:
        DataFrame: DataFrame with derived features
    """
    result_df = df.copy()
    missing_data_flags = {}
    
    # Get quarters and check for missing data
    for idx, row in result_df.iterrows():
        # Identify missing quarter data
        missing_quarters = []
        for q in range(1, 5):
            if pd.isna(row.get(f'home_q{q}')) or pd.isna(row.get(f'away_q{q}')):
                missing_quarters.append(q)
        
        if missing_quarters and row.get('current_quarter', 0) >= max(missing_quarters):
            # Flag missing data for played quarters (critical issue)
            game_id = row.get('game_id', idx)
            missing_data_flags[game_id] = f"Missing Q{missing_quarters} data despite being in Q{row.get('current_quarter')}"
    
    # 1. Calculate current scores from quarters with improved imputation
    for idx, row in result_df.iterrows():
        # Home score
        if pd.isna(row.get('home_score')) or row.get('home_score', 0) == 0:
            # Impute missing quarter scores
            quarter_scores = []
            for i in range(1, 5):
                q_val = row.get(f'home_q{i}', None)
                if pd.isna(q_val) and row.get('current_quarter', 0) >= i:
                    # Need imputation for a played quarter
                    if imputation_strategy == 'mean':
                        # Use mean of other quarters
                        valid_quarters = [float(row.get(f'home_q{j}', 0) or 0) 
                                          for j in range(1, 5) 
                                          if not pd.isna(row.get(f'home_q{j}', None))]
                        q_val = sum(valid_quarters) / len(valid_quarters) if valid_quarters else 25.0
                    elif imputation_strategy == 'median':
                        valid_quarters = [float(row.get(f'home_q{j}', 0) or 0) 
                                          for j in range(1, 5) 
                                          if not pd.isna(row.get(f'home_q{j}', None))]
                        q_val = np.median(valid_quarters) if valid_quarters else 25.0
                    else:
                        # Default to zero
                        q_val = 0
                elif pd.isna(q_val):
                    q_val = 0
                
                quarter_scores.append(float(q_val or 0))
                result_df.at[idx, f'home_q{i}'] = float(q_val or 0)
            
            home_sum = sum(quarter_scores)
            result_df.at[idx, 'home_score'] = home_sum
        
        # Away score (similar logic)
        if pd.isna(row.get('away_score')) or row.get('away_score', 0) == 0:
            # Impute missing quarter scores
            quarter_scores = []
            for i in range(1, 5):
                q_val = row.get(f'away_q{i}', None)
                if pd.isna(q_val) and row.get('current_quarter', 0) >= i:
                    # Need imputation for a played quarter
                    if imputation_strategy == 'mean':
                        # Use mean of other quarters
                        valid_quarters = [float(row.get(f'away_q{j}', 0) or 0) 
                                          for j in range(1, 5) 
                                          if not pd.isna(row.get(f'away_q{j}', None))]
                        q_val = sum(valid_quarters) / len(valid_quarters) if valid_quarters else 25.0
                    elif imputation_strategy == 'median':
                        valid_quarters = [float(row.get(f'away_q{j}', 0) or 0) 
                                          for j in range(1, 5) 
                                          if not pd.isna(row.get(f'away_q{j}', None))]
                        q_val = np.median(valid_quarters) if valid_quarters else 25.0
                    else:
                        # Default to zero
                        q_val = 0
                elif pd.isna(q_val):
                    q_val = 0
                
                quarter_scores.append(float(q_val or 0))
                result_df.at[idx, f'away_q{i}'] = float(q_val or 0)
            
            away_sum = sum(quarter_scores)
            result_df.at[idx, 'away_score'] = away_sum
    
    # 2. Calculate score_ratio
    for idx, row in result_df.iterrows():
        home_score = float(row.get('home_score', 0))
        away_score = float(row.get('away_score', 0))
        total = home_score + away_score
        result_df.at[idx, 'score_ratio'] = (home_score / total) if total > 0 else 0.5
    
    # 3. Calculate score differential
    result_df['score_differential'] = result_df['home_score'] - result_df['away_score']
    
    # 4. Calculate half-time differentials
    result_df['first_half_diff'] = (
        result_df['home_q1'].fillna(0) + result_df['home_q2'].fillna(0) -
        result_df['away_q1'].fillna(0) - result_df['away_q2'].fillna(0)
    )
    
    result_df['pre_q4_diff'] = (
        result_df['home_q1'].fillna(0) + result_df['home_q2'].fillna(0) + result_df['home_q3'].fillna(0) -
        result_df['away_q1'].fillna(0) - result_df['away_q2'].fillna(0) - result_df['away_q3'].fillna(0)
    )
    
    # 5. Calculate momentum features
    for col in ['q1_to_q2_momentum','q2_to_q3_momentum','q3_to_q4_momentum','cumulative_momentum']:
        if col not in result_df.columns:
            result_df[col] = 0
    
    for idx, row in result_df.iterrows():
        current_quarter = int(row.get('current_quarter', 0))
        
        # Initialize with zeros to ensure these columns exist
        result_df.at[idx, 'q1_to_q2_momentum'] = 0
        result_df.at[idx, 'q2_to_q3_momentum'] = 0
        result_df.at[idx, 'q3_to_q4_momentum'] = 0
        result_df.at[idx, 'cumulative_momentum'] = 0
        
        # Calculate quarter-to-quarter momentum shifts
        if current_quarter >= 2:
            # Q1 to Q2 momentum
            home_q1 = float(row.get('home_q1') or 0)
            home_q2 = float(row.get('home_q2') or 0)
            away_q1 = float(row.get('away_q1') or 0)
            away_q2 = float(row.get('away_q2') or 0)
            
            q1_to_q2 = (home_q2 - home_q1) - (away_q2 - away_q1)
            # Cap extreme values
            q1_to_q2 = max(min(q1_to_q2, 25), -25)
            result_df.at[idx, 'q1_to_q2_momentum'] = q1_to_q2
        
        if current_quarter >= 3:
            # Q2 to Q3 momentum
            home_q2 = float(row.get('home_q2') or 0)
            home_q3 = float(row.get('home_q3') or 0)
            away_q2 = float(row.get('away_q2') or 0)
            away_q3 = float(row.get('away_q3') or 0)
            
            q2_to_q3 = (home_q3 - home_q2) - (away_q3 - away_q2)
            # Cap extreme values
            q2_to_q3 = max(min(q2_to_q3, 25), -25)
            result_df.at[idx, 'q2_to_q3_momentum'] = q2_to_q3
        
        if current_quarter >= 4:
            # Q3 to Q4 momentum
            home_q3 = float(row.get('home_q3') or 0)
            home_q4 = float(row.get('home_q4') or 0)
            away_q3 = float(row.get('away_q3') or 0)
            away_q4 = float(row.get('away_q4') or 0)
            
            q3_to_q4 = (home_q4 - home_q3) - (away_q4 - away_q3)
            # Cap extreme values
            q3_to_q4 = max(min(q3_to_q4, 25), -25)
            result_df.at[idx, 'q3_to_q4_momentum'] = q3_to_q4
        
        # Calculate cumulative momentum with weights
        weights = [0.2, 0.3, 0.5]  # Weights for Q1→Q2, Q2→Q3, Q3→Q4
        
        if current_quarter == 2:
            result_df.at[idx, 'cumulative_momentum'] = result_df.at[idx, 'q1_to_q2_momentum'] / 15.0
        elif current_quarter == 3:
            q1_to_q2 = result_df.at[idx, 'q1_to_q2_momentum']
            q2_to_q3 = result_df.at[idx, 'q2_to_q3_momentum']
            weighted_momentum = (q1_to_q2 * weights[0] + q2_to_q3 * weights[1]) / (weights[0] + weights[1])
            result_df.at[idx, 'cumulative_momentum'] = weighted_momentum / 15.0
        elif current_quarter >= 4:
            q1_to_q2 = result_df.at[idx, 'q1_to_q2_momentum']
            q2_to_q3 = result_df.at[idx, 'q2_to_q3_momentum']
            q3_to_q4 = result_df.at[idx, 'q3_to_q4_momentum']
            weighted_momentum = (q1_to_q2 * weights[0] + q2_to_q3 * weights[1] + q3_to_q4 * weights[2]) / sum(weights)
            result_df.at[idx, 'cumulative_momentum'] = weighted_momentum / 15.0
        
        # Normalize momentum to [-1, 1]
        result_df.at[idx, 'cumulative_momentum'] = max(min(result_df.at[idx, 'cumulative_momentum'], 1.0), -1.0)
    
    # 6. Add time remaining
    result_df['time_remaining_mins'] = result_df['current_quarter'].apply(
        lambda q: max(0, 48 - ((int(q) - 1) * 12 + 6))  # Approximate minutes left
    )
    result_df['time_remaining_norm'] = result_df['time_remaining_mins'] / 48.0
    
    # Print report on missing data
    if missing_data_flags:
        print("\nWARNING: Missing data detected in critical game states:")
        for game_id, message in missing_data_flags.items():
            print(f"  • Game {game_id}: {message}")
    
    return result_df

def prepare_features_for_model(df, expected_features, team_avgs=None):
    """
    Prepares features in the correct format for model input.
    
    Args:
        df (DataFrame): Input game data
        expected_features (list): Features expected by the model
        team_avgs (dict, optional): Team scoring averages for rolling features
        
    Returns:
        DataFrame: Features ready for model prediction
    """
    # Ensure all basic numeric features are present and valid
    df = ensure_numeric_features(df, expected_features)
    
    # If team_avgs provided, use them for rolling averages
    if team_avgs is not None and 'rolling_home_score' in expected_features:
        # Update rolling average features if in expected features
        for idx, row in df.iterrows():
            if 'home_team' in row and row['home_team'] in team_avgs:
                df.at[idx, 'rolling_home_score'] = team_avgs[row['home_team']]
            if 'away_team' in row and row['away_team'] in team_avgs:
                df.at[idx, 'rolling_away_score'] = team_avgs[row['away_team']]
    
    # Select only the expected features in the correct order
    X = df[expected_features].copy()
    
    return X

In [ ]:
# Cell 2C: Optimized Rest Days Calculation with Statistical Validation

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sqlalchemy import create_engine
import config
import traceback

def calculate_improved_rest_features(df: pd.DataFrame, max_lookback_days: int = 30, 
                                    data_driven_bounds: bool = True) -> pd.DataFrame:
    """
    Calculate rest-related features with better validation, memory efficiency and data-driven bounds.
    Returns the input DataFrame augmented with rest_days_home, rest_days_away,
    is_back_to_back flags, and rest_advantage.
    
    Args:
        df: Input DataFrame with game data
        max_lookback_days: Maximum days to look back for previous games
        data_driven_bounds: Use statistical analysis to determine reasonable bounds
        
    Returns:
        DataFrame with added rest features
    """
    print(f"Calculating rest features with optimized approach...")
    if df.empty:
        print("No data provided for rest calculation")
        return df
    
    # Make a copy to avoid modifying the original
    result_df = df.copy()
    
    # Early exit if no game_date
    if 'game_date' not in result_df.columns:
        print("Warning: 'game_date' column not found. Using default rest values.")
        result_df['rest_days_home'] = 2.0
        result_df['rest_days_away'] = 2.0
        result_df['is_back_to_back_home'] = 0
        result_df['is_back_to_back_away'] = 0
        result_df['rest_advantage'] = 0
        return result_df
    
    # Ensure datetime format and sort by date
    try:
        result_df['game_date'] = pd.to_datetime(result_df['game_date'], errors='coerce')
        invalid_dates = result_df['game_date'].isna().sum()
        if invalid_dates > 0:
            print(f"Warning: {invalid_dates} invalid dates found; filling with median date.")
            median_date = result_df['game_date'].median()
            result_df['game_date'] = result_df['game_date'].fillna(median_date)
    except Exception as e:
        print(f"Error processing dates: {e}")
        traceback.print_exc()
        # Default values if date processing fails
        result_df['rest_days_home'] = 2.0
        result_df['rest_days_away'] = 2.0
        result_df['is_back_to_back_home'] = 0
        result_df['is_back_to_back_away'] = 0
        result_df['rest_advantage'] = 0
        return result_df
    
    # Initialize rest features with default values
    result_df['rest_days_home'] = 2.0
    result_df['rest_days_away'] = 2.0
    result_df['is_back_to_back_home'] = 0
    result_df['is_back_to_back_away'] = 0
    
    # Create auxiliary structures for efficient lookups
    # Sort once, then use team-specific indices
    result_df = result_df.sort_values('game_date')
    
    # Extract all unique teams
    all_teams = set()
    if 'home_team' in result_df.columns:
        all_teams.update(result_df['home_team'].dropna().unique())
    if 'away_team' in result_df.columns:
        all_teams.update(result_df['away_team'].dropna().unique())
    
    # Process each team's games to calculate rest days
    for team in all_teams:
        # Get indices of all games where this team appears
        home_mask = (result_df['home_team'] == team)
        away_mask = (result_df['away_team'] == team)
        team_games = result_df[home_mask | away_mask].sort_values('game_date')
        
        if len(team_games) <= 1:
            continue  # Not enough games to calculate rest
        
        # Calculate rest days for each game (except the first one)
        for i in range(1, len(team_games)):
            current_game = team_games.iloc[i]
            prev_game = team_games.iloc[i-1]
            
            # Calculate days between games
            days_rest = (current_game['game_date'] - prev_game['game_date']).days
            
            # Apply to the correct column based on home/away
            if current_game['home_team'] == team:
                idx = current_game.name  # Get original index in result_df
                result_df.at[idx, 'rest_days_home'] = days_rest
                result_df.at[idx, 'is_back_to_back_home'] = 1 if days_rest == 1 else 0
            elif current_game['away_team'] == team:
                idx = current_game.name  # Get original index in result_df
                result_df.at[idx, 'rest_days_away'] = days_rest
                result_df.at[idx, 'is_back_to_back_away'] = 1 if days_rest == 1 else 0
    
    # Determine suitable bounds for rest advantage
    if data_driven_bounds:
        # Calculate rest advantage directly
        result_df['rest_advantage'] = result_df['rest_days_home'] - result_df['rest_days_away']
        
        # Calculate statistically reasonable bounds (3 standard deviations)
        rest_advantage_mean = result_df['rest_advantage'].mean()
        rest_advantage_std = result_df['rest_advantage'].std()
        
        min_bound = max(-5, rest_advantage_mean - 3 * rest_advantage_std)
        max_bound = min(5, rest_advantage_mean + 3 * rest_advantage_std)
        
        # Apply data-driven bounds for rest advantage
        result_df['rest_advantage'] = result_df['rest_advantage'].clip(min_bound, max_bound)
        
        print(f"Data-driven rest advantage bounds: [{min_bound:.1f}, {max_bound:.1f}]")
    else:
        # Use traditional fixed bounds
        result_df['rest_advantage'] = (result_df['rest_days_home'] - result_df['rest_days_away']).clip(-5, 5)
    
    # Validate the calculated rest features
    max_rest = max(result_df['rest_days_home'].max(), result_df['rest_days_away'].max())
    if max_rest > 10:
        # Cap extremely long rest periods
        result_df['rest_days_home'] = result_df['rest_days_home'].clip(1, 10)
        result_df['rest_days_away'] = result_df['rest_days_away'].clip(1, 10)
        print(f"Warning: Detected unusually long rest periods (max: {max_rest} days)")
        print("Rest days have been capped at 10 for consistency")
    
    # Ensure rest days are at least 1
    result_df['rest_days_home'] = np.maximum(result_df['rest_days_home'], 1)
    result_df['rest_days_away'] = np.maximum(result_df['rest_days_away'], 1)
    
    # Calculate summary statistics
    b2b_home = result_df['is_back_to_back_home'].sum()
    b2b_away = result_df['is_back_to_back_away'].sum()
    total_games = len(result_df)
    
    print(f"Rest day calculation complete for {total_games} games")
    print(f"Back-to-back games: {b2b_home} home ({b2b_home/total_games*100:.1f}%), "
          f"{b2b_away} away ({b2b_away/total_games*100:.1f}%)")
    
    return result_df

# Function to analyze rest day distribution for data-driven bounds
def analyze_rest_advantages(df):
    """Analyze the distribution of rest advantages to inform reasonable bounds"""
    if 'rest_days_home' not in df.columns or 'rest_days_away' not in df.columns:
        print("Rest day columns not found in DataFrame")
        return
        
    # Calculate rest advantage
    df['rest_advantage'] = df['rest_days_home'] - df['rest_days_away']
    
    # Calculate statistics
    rest_adv_stats = df['rest_advantage'].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    print("\nRest Advantage Statistics:")
    print(rest_adv_stats)
    
    # Suggest bounds based on percentiles (e.g., 1st to 99th)
    lower_bound = rest_adv_stats['1%']
    upper_bound = rest_adv_stats['99%']
    
    # Round to nearest integer for simplicity
    lower_bound = np.floor(lower_bound)
    upper_bound = np.ceil(upper_bound)
    
    print(f"\nSuggested rest advantage bounds: [{lower_bound}, {upper_bound}]")
    print(f"Traditional bounds: [-5, 5]")
    
    # Check if traditional bounds are reasonable
    if lower_bound < -5 or upper_bound > 5:
        print("Warning: Data suggests the traditional bounds [-5, 5] may be too restrictive")
        print(f"         Consider using [{lower_bound}, {upper_bound}] instead")
    else:
        print("The traditional bounds [-5, 5] appear reasonable for this data")
    
    return {'lower': lower_bound, 'upper': upper_bound}

In [ ]:
# Cell 2D: Robust Model Loading with Path Configuration and Historical Fallback

import os
import joblib
import glob
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
import config
import traceback
import json
from datetime import datetime

class ModelConfig:
    """Configuration class for model loading paths and versioning"""
    
    # Default search paths in priority order
    DEFAULT_PATHS = [
        os.path.join(os.getcwd(), "models"),
        os.path.join(os.getcwd(), "backend", "models"),
        os.path.join(os.environ.get("MODEL_DIR", ""), ""),  # Environment variable
    ]
    
    # Model registry file for version tracking
    REGISTRY_FILE = "model_registry.json"
    
    # Expected feature sets by model version
    FEATURE_SETS = {
        "standard": [
            'home_q1', 'home_q2', 'home_q3', 'home_q4',
            'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
        ],
        "enhanced": [
            'home_q1', 'home_q2', 'home_q3', 'home_q4',
            'score_ratio', 'prev_matchup_diff',
            'rest_days_home', 'rest_days_away', 'rest_advantage',
            'is_back_to_back_home', 'is_back_to_back_away',
            'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
        ]
    }
    
    @staticmethod
    def get_search_paths():
        """Get prioritized list of search paths"""
        paths = ModelConfig.DEFAULT_PATHS.copy()
        
        # Add config path if available
        if hasattr(config, "MODEL_PATH") and config.MODEL_PATH:
            model_dir = os.path.dirname(config.MODEL_PATH)
            if model_dir and os.path.exists(model_dir):
                paths.insert(0, model_dir)
                
        # Add config-specified model directory if available
        if hasattr(config, "MODEL_DIR") and config.MODEL_DIR:
            if os.path.exists(config.MODEL_DIR):
                paths.insert(0, config.MODEL_DIR)
        
        # Return only existing directories
        return [path for path in paths if path and os.path.exists(path)]

def find_model_file(model_name: str = "score_prediction", search_locations: list = None) -> str:
    """
    Search for a model file by name using standardized path convention.
    
    Args:
        model_name: Base name of the model
        search_locations: Optional list of directories to search
        
    Returns:
        str: Path to the found model file, or empty string if not found
    """
    # Use configured search locations if none provided
    if search_locations is None:
        search_locations = ModelConfig.get_search_paths()
        
    # If config specifies an exact path and it exists, use it
    if hasattr(config, "MODEL_PATH") and os.path.exists(config.MODEL_PATH):
        print(f"Model found at config.MODEL_PATH: {config.MODEL_PATH}")
        return config.MODEL_PATH
    
    # Search in all locations
    model_files = []
    
    for location in search_locations:
        # Try exact matches first
        exact_patterns = [
            os.path.join(location, f"{model_name}.pkl"),
            os.path.join(location, f"{model_name}_model.pkl"),
            os.path.join(location, f"final_{model_name}_model.pkl")
        ]
        
        for pattern in exact_patterns:
            if os.path.exists(pattern):
                model_files.append((pattern, os.path.getmtime(pattern)))
        
        # Then try glob patterns
        glob_patterns = [
            os.path.join(location, f"{model_name}*.pkl"),
            os.path.join(location, f"*{model_name}*.pkl")
        ]
        
        for pattern in glob_patterns:
            for file_path in glob.glob(pattern):
                model_files.append((file_path, os.path.getmtime(file_path)))
    
    if not model_files:
        print("No model file found in search locations")
        return ""
    
    # Sort by modification time (newest first)
    model_files.sort(key=lambda x: x[1], reverse=True)
    newest_model = model_files[0][0]
    
    print(f"Found model: {newest_model} (Last modified: {datetime.fromtimestamp(model_files[0][1])})")
    
    # Register this model in registry if not already present
    register_model(newest_model)
    
    return newest_model

def register_model(model_path):
    """Register model in the model registry for version tracking"""
    model_dir = os.path.dirname(model_path)
    registry_path = os.path.join(model_dir, ModelConfig.REGISTRY_FILE)
    
    model_info = {
        'path': model_path,
        'last_used': datetime.now().isoformat(),
        'file_modified': datetime.fromtimestamp(os.path.getmtime(model_path)).isoformat()
    }
    
    # Load existing registry if available
    registry = {}
    if os.path.exists(registry_path):
        try:
            with open(registry_path, 'r') as f:
                registry = json.load(f)
        except Exception as e:
            print(f"Error loading model registry: {e}")
    
    # Add or update this model
    model_id = os.path.basename(model_path)
    registry[model_id] = model_info
    
    # Save updated registry
    try:
        with open(registry_path, 'w') as f:
            json.dump(registry, f, indent=2)
    except Exception as e:
        print(f"Error updating model registry: {e}")

def create_fallback_model(feature_set="enhanced", historical_data=None):
    """
    Creates a fallback model based on historical averages when available,
    or synthetic data if no historical data is provided.
    
    Args:
        feature_set: Which feature set to use ('standard' or 'enhanced')
        historical_data: Optional DataFrame with historical game data
        
    Returns:
        Trained model for emergency use
    """
    print("Training a fallback model for emergency use...")
    
    try:
        # Define features based on selected feature set
        features = ModelConfig.FEATURE_SETS.get(feature_set, ModelConfig.FEATURE_SETS["standard"])
        
        # Use historical data if provided
        if historical_data is not None and not historical_data.empty:
            print(f"Training fallback model on {len(historical_data)} historical games")
            
            # Prepare features and target
            X = historical_data[features].fillna(0)
            y = historical_data['home_score'].fillna(historical_data['home_score'].mean())
            
            # Train a simple model
            model = GradientBoostingRegressor(n_estimators=50, max_depth=3, random_state=42)
            model.fit(X, y)
            
        else:
            # No historical data, use synthetic data
            print("No historical data available. Using synthetic data for fallback model.")
            
            # Create synthetic data that roughly matches NBA game patterns
            np.random.seed(42)
            n_samples = 1000
            
            # Quarter scores typically average 25-30 points
            q_mean, q_std = 27, 5
            
            # Generate training data
            X_data = {}
            for feature in features:
                if feature in ['home_q1', 'home_q2', 'home_q3', 'home_q4']:
                    # Quarter scores
                    X_data[feature] = np.random.normal(q_mean, q_std, n_samples).clip(10, 45)
                elif feature == 'score_ratio':
                    # Score ratio centered around 0.5 (home court advantage)
                    X_data[feature] = np.random.beta(5.2, 5, n_samples)  # Slightly skewed toward home team
                elif feature in ['rolling_home_score', 'rolling_away_score']:
                    # Team scores typically around 110 points
                    X_data[feature] = np.random.normal(110, 8, n_samples).clip(90, 130)
                elif feature == 'prev_matchup_diff':
                    # Matchup differential typically within ±10
                    X_data[feature] = np.random.normal(2, 8, n_samples).clip(-20, 20)
                elif feature in ['rest_days_home', 'rest_days_away']:
                    # Rest days typically 1-5
                    X_data[feature] = np.random.choice([1, 2, 3, 4, 5], n_samples, p=[0.2, 0.4, 0.2, 0.1, 0.1])
                elif feature == 'rest_advantage':
                    # Rest advantage typically within ±3
                    X_data[feature] = np.random.choice([-3, -2, -1, 0, 1, 2, 3], n_samples)
                elif feature.startswith('is_back_to_back'):
                    # Back-to-back indicator (about 20% of games)
                    X_data[feature] = np.random.choice([0, 1], n_samples, p=[0.8, 0.2])
                elif 'momentum' in feature:
                    # Momentum features typically between -1 and 1
                    X_data[feature] = np.random.normal(0, 0.4, n_samples).clip(-1, 1)
                else:
                    # Default for any other features
                    X_data[feature] = np.zeros(n_samples)
            
            X = pd.DataFrame(X_data)
            
            # Generate target based on features (realistic NBA scoring pattern)
            home_q_sum = X['home_q1'] + X['home_q2'] + X['home_q3'] + X['home_q4']
            
            # Add some noise to make it realistic
            y = home_q_sum + np.random.normal(0, 3, n_samples)
            
            # Train a simple model
            model = GradientBoostingRegressor(n_estimators=50, max_depth=3, random_state=42)
            model.fit(X, y)
        
        print("Fallback model trained successfully.")
        
        # Set a flag to identify this as a fallback model
        model.is_fallback = True
        model.feature_set = feature_set
        model.features = features
        
        return model
        
    except Exception as e:
        print(f"Error creating fallback model: {e}")
        traceback.print_exc()
        
        # Create an extremely simple dummy model as last resort
        class DummyModel:
            def __init__(self):
                self.is_fallback = True
                self.is_dummy = True
                self.feature_set = "standard"
                self.features = ModelConfig.FEATURE_SETS["standard"]
                
            def predict(self, X):
                # Always predict league average
                return np.full(len(X), 108.0)
                
            def __str__(self):
                return "DUMMY MODEL - Always predicts 108"
                
        print("WARNING: Using dummy model that always predicts 108")
        return DummyModel()

def verify_model_compatibility(model, expected_features=None):
    """
    Verify that a model is compatible with expected features.
    
    Args:
        model: Model to verify
        expected_features: Optional list of expected features
        
    Returns:
        bool: True if model is compatible, False otherwise
    """
    # Check if model has required attributes
    if not hasattr(model, 'predict') or not callable(model.predict):
        print("Model does not have a predict method")
        return False
    
    # Check feature compatibility if expected features are provided
    if expected_features is not None:
        if hasattr(model, 'feature_importances_'):
            # For tree-based models that have feature_importances_
            n_features = len(model.feature_importances_)
            if n_features != len(expected_features):
                print(f"Feature count mismatch: model has {n_features}, expected {len(expected_features)}")
                return False
        elif hasattr(model, 'coef_'):
            # For linear models that have coef_
            if len(model.coef_) != len(expected_features):
                print(f"Feature count mismatch: model has {len(model.coef_)}, expected {len(expected_features)}")
                return False
    
    return True

def load_model_safely(model_path: str = None, expected_features=None, use_historical_data=True):
    """
    Loads a trained model with comprehensive error handling, version compatibility
    checks, and sophisticated fallback model generation.
    
    Args:
        model_path: Path to the model file, or None to search
        expected_features: Optional list of features expected by calling code
        use_historical_data: Whether to use historical data for fallback model
        
    Returns:
        Loaded model object
    """
    # Find model if path not provided
    if model_path is None or not os.path.exists(model_path):
        model_path = find_model_file()
    
    # Load historical data for potential fallback model if requested
    historical_data = None
    if use_historical_data:
        try:
            # Try to load some historical data for fallback model
            from sqlalchemy import create_engine
            engine = create_engine(config.DATABASE_URL)
            query = """
            SELECT * FROM nba_historical_game_stats 
            ORDER BY game_date DESC 
            LIMIT 1000
            """
            historical_data = pd.read_sql(query, engine)
            print(f"Loaded {len(historical_data)} historical games for potential fallback model")
        except Exception as e:
            print(f"Could not load historical data for fallback model: {e}")
    
    # Try to load the model
    if model_path and os.path.exists(model_path):
        try:
            print(f"Loading model from: {model_path}")
            model = joblib.load(model_path)
            
            if verify_model_compatibility(model, expected_features):
                print(f"Successfully loaded compatible model from: {model_path}")
                
                # Determine feature set
                feature_set = "standard"
                if hasattr(model, 'feature_importances_') and len(model.feature_importances_) > 8:
                    feature_set = "enhanced"
                elif hasattr(model, 'feature_set'):
                    feature_set = model.feature_set
                    
                print(f"Model type: {feature_set} (features: {len(ModelConfig.FEATURE_SETS[feature_set])})")
                
                # Add model metadata if not present
                if not hasattr(model, 'is_fallback'):
                    model.is_fallback = False
                if not hasattr(model, 'feature_set'):
                    model.feature_set = feature_set
                if not hasattr(model, 'features'):
                    model.features = ModelConfig.FEATURE_SETS[feature_set]
                
                return model
            else:
                print(f"Loaded model is not compatible with required features")
        except Exception as e:
            print(f"Error loading model from {model_path}: {e}")
            traceback.print_exc()
    
    # If we get here, we need to create a fallback model
    feature_set = "enhanced" if expected_features is None or len(expected_features) > 8 else "standard"
    return create_fallback_model(feature_set, historical_data)

def get_feature_list(model=None) -> list:
    """
    Returns the list of features expected by the model.
    Uses the enhanced feature set if the model has more than 8 features.
    
    Args:
        model: Optional model object to inspect
        
    Returns:
        list: Feature list appropriate for the model
    """
    # If model has features defined, use them
    if model is not None and hasattr(model, 'features'):
        return model.features
    
    # If model has feature_set defined, use the corresponding feature list
    if model is not None and hasattr(model, 'feature_set'):
        return ModelConfig.FEATURE_SETS.get(model.feature_set, ModelConfig.FEATURE_SETS["standard"])
    
    # Otherwise, determine based on feature_importances_ or coef_
    if model is not None:
        if hasattr(model, 'feature_importances_'):
            if len(model.feature_importances_) > 8:
                return ModelConfig.FEATURE_SETS["enhanced"]
        elif hasattr(model, 'coef_'):
            if len(model.coef_) > 8:
                return ModelConfig.FEATURE_SETS["enhanced"]
    
    # Default to standard feature set
    return ModelConfig.FEATURE_SETS["standard"]

# Test model loading
print("Testing model loading with improved path handling...")
model = load_model_safely()
expected_features = get_feature_list(model)
print("\nExpected feature list for this model:")
print(expected_features)

print("\nModel type:", "Fallback" if getattr(model, 'is_fallback', False) else "Production")
if getattr(model, 'is_dummy', False):
    print("WARNING: Using extremely simplified dummy model")

In [ ]:
# Cell 2E: Enhanced Data Consistency Checks with Adaptive Validation

import pandas as pd
import numpy as np
from datetime import datetime

def validate_and_clean_features(df: pd.DataFrame, expected_features: list = None, 
                               verbose: bool = True, adaptive_ranges: bool = True) -> pd.DataFrame:
    """
    Comprehensive validation and cleaning of feature data with adaptive ranges.
    Ensures required columns exist, converts data types, fills missing values,
    and clips out-of-range values based on data distribution or configurable bounds.

    Args:
        df: Input DataFrame
        expected_features: List of column names to process
        verbose: Whether to print detailed logs
        adaptive_ranges: Use data-driven ranges for validation instead of hardcoded values
        
    Returns:
        DataFrame: DataFrame with validated and cleaned features
    """
    if df.empty:
        if verbose:
            print("No data provided for validation")
        return pd.DataFrame()
    
    clean_df = df.copy()
    
    # If no expected features are specified, choose a default set
    if expected_features is None:
        # If momentum columns exist, we assume the "enhanced" feature set
        if 'q1_to_q2_momentum' in clean_df.columns:
            expected_features = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4',
                'score_ratio', 'prev_matchup_diff',
                'rest_days_home', 'rest_days_away', 'rest_advantage',
                'is_back_to_back_home', 'is_back_to_back_away',
                'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
            ]
        else:
            # Otherwise, we assume the "original" feature set
            expected_features = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4',
                'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
            ]
    
    if verbose:
        print(f"Validating {len(clean_df)} rows against {len(expected_features)} expected features")
    
    # Ensure certain columns exist (game_id, home_team, away_team)
    required_game_cols = ['game_id', 'home_team', 'away_team']
    for col in required_game_cols:
        if col not in clean_df.columns:
            if verbose:
                print(f"Warning: Missing required column: {col}. Adding default values.")
            if col == 'game_id':
                clean_df[col] = np.arange(1000, 1000 + len(clean_df))
            else:
                clean_df[col] = "Unknown"
    
    # Default values for missing columns
    default_values = {
        'home_q1': 0, 'home_q2': 0, 'home_q3': 0, 'home_q4': 0,
        'away_q1': 0, 'away_q2': 0, 'away_q3': 0, 'away_q4': 0,
        'score_ratio': 0.5,
        'rolling_home_score': 105.0, 'rolling_away_score': 105.0,
        'prev_matchup_diff': 0,
        'rest_days_home': 2, 'rest_days_away': 2, 'rest_advantage': 0,
        'is_back_to_back_home': 0, 'is_back_to_back_away': 0,
        'q1_to_q2_momentum': 0, 'q2_to_q3_momentum': 0, 'q3_to_q4_momentum': 0, 'cumulative_momentum': 0,
        'current_quarter': 0,
        'home_score': 0, 'away_score': 0
    }
    
    # Ensure each expected feature exists and is numeric (except ID/team columns)
    for feature in expected_features:
        if feature not in clean_df.columns:
            if verbose:
                print(f"Adding missing feature: {feature} with default {default_values.get(feature, 0)}")
            clean_df[feature] = default_values.get(feature, 0)
        if feature not in ['game_id', 'home_team', 'away_team']:
            clean_df[feature] = pd.to_numeric(clean_df[feature], errors='coerce').fillna(default_values.get(feature, 0))
    
    # Determine reasonable validation ranges based on data if adaptive_ranges=True
    if adaptive_ranges:
        # Calculate data-driven validation ranges (mean ± 3*std, or percentile-based)
        validation_ranges = {}
        
        # Quarter scores (use 99th percentile as upper bound)
        for q in range(1, 5):
            home_q_col = f'home_q{q}'
            away_q_col = f'away_q{q}'
            
            # Combine home and away quarter scores for better statistics
            all_q_scores = pd.concat([
                clean_df[home_q_col][clean_df[home_q_col] > 0],
                clean_df[away_q_col][clean_df[away_q_col] > 0]
            ])
            
            if len(all_q_scores) > 10:  # Only if we have sufficient data
                q_min = 0  # Quarters can't score below 0
                q_max = all_q_scores.quantile(0.99)
                # Round up to nearest 5 for readability
                q_max = np.ceil(q_max / 5) * 5
                validation_ranges[f'q{q}_scores'] = (q_min, q_max)
            else:
                # Default if insufficient data
                validation_ranges[f'q{q}_scores'] = (0, 50)
                
        # Other numeric features
        for feature in expected_features:
            if feature not in ['game_id', 'home_team', 'away_team'] and feature not in validation_ranges:
                # Skip quarter scores (already handled)
                if any(q_name in feature for q_name in ['home_q1', 'home_q2', 'home_q3', 'home_q4', 
                                                       'away_q1', 'away_q2', 'away_q3', 'away_q4']):
                    continue
                    
                # Get non-zero/non-null values for better statistics
                feature_values = clean_df[feature]
                feature_values = feature_values[feature_values != 0]
                feature_values = feature_values.dropna()
                
                if len(feature_values) > 10:  # Only if we have sufficient data
                    if feature == 'score_ratio':
                        # Score ratio is always between 0 and 1
                        validation_ranges[feature] = (0, 1)
                    elif 'momentum' in feature:
                        # Momentum features typically range from -1 to 1
                        validation_ranges[feature] = (-1, 1)
                    elif feature in ['rolling_home_score', 'rolling_away_score']:
                        # Team scores typically between 80-140
                        score_min = max(60, feature_values.quantile(0.01))
                        score_max = min(140, feature_values.quantile(0.99))
                        validation_ranges[feature] = (score_min, score_max)
                    elif feature == 'prev_matchup_diff':
                        # Matchup differential typically within ±30
                        diff_abs_max = min(50, feature_values.abs().quantile(0.99))
                        validation_ranges[feature] = (-diff_abs_max, diff_abs_max)
                    elif feature in ['rest_days_home', 'rest_days_away']:
                        # Rest days typically 1-10
                        validation_ranges[feature] = (1, 10)
                    elif feature == 'rest_advantage':
                        # Rest advantage typically within ±5
                        adv_abs_max = min(5, feature_values.abs().quantile(0.99))
                        validation_ranges[feature] = (-adv_abs_max, adv_abs_max)
                    else:
                        # Default approach for other features: use percentiles
                        feature_min = feature_values.quantile(0.01)
                        feature_max = feature_values.quantile(0.99)
                        validation_ranges[feature] = (feature_min, feature_max)
        
        if verbose:
            print("\nData-driven validation ranges:")
            for feature, (min_val, max_val) in sorted(validation_ranges.items()):
                print(f"  {feature}: [{min_val:.2f}, {max_val:.2f}]")
    else:
        # Use fixed validation ranges
        validation_ranges = {
            'q1_scores': (0, 50),
            'q2_scores': (0, 50),
            'q3_scores': (0, 50),
            'q4_scores': (0, 50),
            'score_ratio': (0, 1),
            'rolling_home_score': (80, 130),
            'rolling_away_score': (80, 130),
            'prev_matchup_diff': (-50, 50),
            'rest_days_home': (1, 10),
            'rest_days_away': (1, 10),
            'rest_advantage': (-5, 5),
            'q1_to_q2_momentum': (-20, 20),
            'q2_to_q3_momentum': (-20, 20),
            'q3_to_q4_momentum': (-20, 20),
            'cumulative_momentum': (-1, 1)
        }
    
    # Apply validation ranges
    features_clipped = 0
    for feature, feature_range in validation_ranges.items():
        min_val, max_val = feature_range
        
        # Apply to specific quarter score columns
        if feature.startswith('q') and '_scores' in feature:
            q_num = feature[1]
            home_col = f'home_q{q_num}'
            away_col = f'away_q{q_num}'
            
            for col in [home_col, away_col]:
                if col in clean_df.columns:
                    # Count out-of-range values
                    invalid_count = ((clean_df[col] < min_val) | (clean_df[col] > max_val)).sum()
                    if invalid_count > 0:
                        features_clipped += invalid_count
                        if verbose:
                            print(f"Clipping {invalid_count} invalid values in {col} to range [{min_val:.1f}, {max_val:.1f}]")
                        clean_df[col] = clean_df[col].clip(min_val, max_val)
        
        # Apply to other columns directly
        elif feature in clean_df.columns:
            invalid_count = ((clean_df[feature] < min_val) | (clean_df[feature] > max_val)).sum()
            if invalid_count > 0:
                features_clipped += invalid_count
                if verbose:
                    print(f"Clipping {invalid_count} invalid values in {feature} to range [{min_val:.1f}, {max_val:.1f}]")
                clean_df[feature] = clean_df[feature].clip(min_val, max_val)
    
    if features_clipped > 0:
        print(f"Total features clipped to valid ranges: {features_clipped}")
    
    # Calculate home_score and away_score from quarter sums if needed
    if all(f'home_q{i}' in clean_df.columns for i in range(1, 5)):
        home_sum = clean_df[[f'home_q{i}' for i in range(1, 5)]].sum(axis=1)
        if 'home_score' in clean_df.columns:
            # Only update if current value is 0 or NA
            mask = (clean_df['home_score'].isna()) | (clean_df['home_score'] == 0)
            clean_df.loc[mask, 'home_score'] = home_sum[mask]
        else:
            clean_df['home_score'] = home_sum
            
    if all(f'away_q{i}' in clean_df.columns for i in range(1, 5)):
        away_sum = clean_df[[f'away_q{i}' for i in range(1, 5)]].sum(axis=1)
        if 'away_score' in clean_df.columns:
            # Only update if current value is 0 or NA
            mask = (clean_df['away_score'].isna()) | (clean_df['away_score'] == 0)
            clean_df.loc[mask, 'away_score'] = away_sum[mask]
        else:
            clean_df['away_score'] = away_sum
    
    # Enhanced quarter inference - more robust to edge cases
    if 'current_quarter' not in clean_df.columns:
        clean_df['current_quarter'] = 0
        
    for idx, row in clean_df.iterrows():
        # Get current quarter value
        current_q = row.get('current_quarter', 0)
        
        # If current quarter is already set and valid, skip inference
        if current_q >= 1 and current_q <= 4:
            continue
            
        # Try to infer quarter from various indicators
        q_val = 0
        
        # Method 1: Check quarter scores
        for q in range(1, 5):
            home_q_val = row.get(f'home_q{q}', 0) or 0
            away_q_val = row.get(f'away_q{q}', 0) or 0
            
            # Check for any scoring activity in this quarter
            if home_q_val > 0 or away_q_val > 0:
                q_val = max(q_val, q)
                
        # Method 2: Check status field if available
        status = str(row.get('game_status', '')).lower()
        if status in ['finished', 'complete', 'final']:
            q_val = max(q_val, 4)  # Finished games are in Q4
            
        # Method 3: Check quarter-specific momentum features if available
        if row.get('q3_to_q4_momentum', 0) != 0:
            q_val = max(q_val, 4)
        elif row.get('q2_to_q3_momentum', 0) != 0:
            q_val = max(q_val, 3)
        elif row.get('q1_to_q2_momentum', 0) != 0:
            q_val = max(q_val, 2)
            
        # Method 4: Check home/away scores
        home_score = row.get('home_score', 0) or 0
        away_score = row.get('away_score', 0) or 0
        total_score = home_score + away_score
        
        # Estimate quarter based on total score
        if total_score > 0:
            if total_score >= 160:
                q_val = max(q_val, 4)  # Likely late game
            elif total_score >= 120:
                q_val = max(q_val, 3)  # Likely mid-late game
            elif total_score >= 60:
                q_val = max(q_val, 2)  # Likely early-mid game
            else:
                q_val = max(q_val, 1)  # Likely early game
                
        clean_df.at[idx, 'current_quarter'] = q_val
    
    if verbose:
        print("\nValidation summary:")
        print(f"- Processed {len(clean_df)} rows")
        
        # Count rows by inferred quarter
        quarter_counts = clean_df['current_quarter'].value_counts().sort_index()
        print("\nRows by current quarter:")
        for quarter, count in quarter_counts.items():
            print(f"  Quarter {quarter}: {count} rows")
        
        # Calculate feature statistics
        feature_stats = pd.DataFrame({
            'min': clean_df[expected_features].min(),
            'max': clean_df[expected_features].max(),
            'mean': clean_df[expected_features].mean(),
            'missing_pct': clean_df[expected_features].isna().mean() * 100
        })
        
        print("\nFeature statistics after cleaning:")
        display(feature_stats)
    
    return clean_df

In [ ]:
# Cell 3: Get Raw Live Game Data from Supabase with Error Handling

import pandas as pd
import time
import pytz
from datetime import datetime
from caching.supabase_client import supabase
import traceback

def convert_time_to_minutes(time_str: str) -> float:
    if pd.isna(time_str) or ":" not in str(time_str):
        return None
    try:
        minutes, seconds = str(time_str).split(":")
        return float(minutes) + float(seconds) / 60.0
    except Exception as e:
        print(f"Error converting time: {e}")
        return None

def get_active_live_games():
    max_retries = 3
    retry_delay = 2  # seconds
    
    for attempt in range(max_retries):
        try:
            print(f"Attempting to fetch live game data (attempt {attempt+1}/{max_retries})...")
            response = supabase.table("nba_live_game_stats").select("*").execute()
            raw_data = response.data
            
            if not raw_data:
                print("No live game data available.")
                return pd.DataFrame()
                
            # Convert to DataFrame
            raw_df = pd.DataFrame(raw_data)
            print(f"Retrieved {len(raw_df)} games from nba_live_game_stats")
            
            # Process minutes if present
            if 'minutes' in raw_df.columns:
                raw_df['minutes_numeric'] = raw_df['minutes'].apply(convert_time_to_minutes)
                raw_df.drop(columns=['minutes'], inplace=True)
            
            # Convert game_date to datetime and filter for today's games in Pacific Time
            if "game_date" in raw_df.columns:
                raw_df["game_date"] = pd.to_datetime(raw_df["game_date"], errors="coerce", utc=True)
                pacific_tz = pytz.timezone("America/Los_Angeles")
                raw_df["game_date_pt"] = raw_df["game_date"].dt.tz_convert(pacific_tz)
                raw_df["date_only_pt"] = raw_df["game_date_pt"].dt.date
                
                now_pt = datetime.now(pacific_tz)
                today_pt = now_pt.date()
                raw_df = raw_df[raw_df["date_only_pt"] == today_pt].copy()
                print(f"Found {len(raw_df)} games scheduled for today")
            
            # Ensure the 'status' column exists and convert to uppercase
            if "status" not in raw_df.columns:
                print("Column 'status' not found!")
                return pd.DataFrame()
            raw_df["status"] = raw_df["status"].astype(str).str.upper()
            
            # Mark active games based solely on the status field
            live_statuses = {"Q1", "Q2", "Q3", "Q4", "OT", "BT", "HT"}
            raw_df['is_active_now'] = raw_df["status"].isin(live_statuses)
            
            active_df = raw_df[raw_df['is_active_now']].copy()
            print(f"Found {len(active_df)} actively live games")
            
            if active_df.empty:
                print("No actively live games found.")
                return pd.DataFrame()
            
            # Optionally, check for missing values in critical columns
            critical_cols = ['home_q1', 'home_q2', 'home_q3', 'home_q4', 
                             'away_q1', 'away_q2', 'away_q3', 'away_q4',
                             'home_score', 'away_score', 'current_quarter']
            missing_data = {}
            for col in critical_cols:
                if col in active_df.columns:
                    missing_count = active_df[col].isna().sum()
                    if missing_count > 0:
                        missing_data[col] = missing_count
            if missing_data:
                print("\nWARNING: Missing values detected in critical columns:")
                for col, count in missing_data.items():
                    print(f"  • {col}: {count} missing values ({count/len(active_df)*100:.1f}%)")
            
            return active_df
            
        except Exception as e:
            print(f"Connection error: {e}")
            traceback.print_exc()
            if attempt < max_retries - 1:
                print(f"Retrying in {retry_delay} seconds...")
                time.sleep(retry_delay)
                retry_delay *= 2
            else:
                print("Maximum retries reached. Please check your network connection and Supabase configuration.")
                return pd.DataFrame()

# Get actively live games
raw_df = get_active_live_games()

# Display the results
if not raw_df.empty:
    print("\nLatest Active Game Data:")
    display(raw_df)
else:
    print("\nNo actively live game data available for analysis.")


In [ ]:
# Cell 4: Enhanced Feature Engineering with Seasonal Random Sampling

import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from datetime import datetime, timedelta
import traceback
import os
from caching.supabase_client import supabase
import config
import random

# Use our new NBAFeatureGenerator instead of the old precompute_enhanced_features function
from models.features import NBAFeatureGenerator

def get_nba_season(date):
    """Extract NBA season from date (NBA seasons start in October and end in June)"""
    if isinstance(date, str):
        date = pd.to_datetime(date)
    year = date.year
    month = date.month
    # For October through December, the season starts in the current year
    if month >= 10:
        return f"{year}-{year+1}"
    # For January through June, the season started in the previous year
    elif month <= 6:
        return f"{year-1}-{year}"
    # For July through September, use the upcoming season
    else:
        return f"{year}-{year+1}"

def load_historical_data_with_seasonal_sampling(sample_per_season=200, min_year=2020):
    """
    Load historical game data with expanded seasonal sampling and vectorized operations.
    
    Args:
        sample_per_season: Number of games to sample per season
        min_year: Minimum year to include (for NBA season starting in that year)
        
    Returns:
        DataFrame with historical game data sampled across seasons
    """
    print("Connecting to database for expanded seasonal sampling...")
    try:
        engine = create_engine(config.DATABASE_URL)
        
        # Go back further in history (2020 instead of 2022)
        min_date = f"{min_year}-07-01"  # Start of NBA season year (July 1st)
        query = f"""
        SELECT * FROM nba_historical_game_stats 
        WHERE game_date >= '{min_date}'
        ORDER BY game_date
        """
        
        print(f"Loading games since {min_date} to identify seasons...")
        all_df = pd.read_sql(query, engine)
        
        if all_df.empty:
            print(f"No historical game data available from {min_date} onwards.")
            return pd.DataFrame()
        
        # Convert dates and extract seasons
        all_df['game_date'] = pd.to_datetime(all_df['game_date'], errors='coerce')
        
        # Vectorized season extraction
        all_df['month'] = all_df['game_date'].dt.month
        all_df['year'] = all_df['game_date'].dt.year
        
        # Vectorized condition for season assignment
        season_start_year = np.where(all_df['month'] >= 7, all_df['year'], all_df['year'] - 1)
        season_end_year = season_start_year + 1
        all_df['season'] = season_start_year.astype(str) + '-' + season_end_year.astype(str)
        
        # Get available seasons
        seasons = sorted(all_df['season'].unique())
        print(f"Found {len(seasons)} seasons: {seasons}")
        
        # Apply weighted sampling - more recent seasons get higher sampling weight
        sampled_data = []
        
        for i, season in enumerate(seasons):
            season_data = all_df[all_df['season'] == season]
            season_size = len(season_data)
            
            # Apply recency weight - newer seasons get sampled more
            recency_factor = 1 + (i / len(seasons))  # 1.0 for oldest, 2.0 for newest
            weighted_sample_size = int(sample_per_season * recency_factor)
            
            # Determine final sample size for this season
            actual_sample_size = min(weighted_sample_size, season_size)
            
            if actual_sample_size < season_size:
                # Random sampling without replacement
                sampled_season = season_data.sample(n=actual_sample_size, random_state=42)
            else:
                # Use all data if sample size exceeds available data
                sampled_season = season_data
                
            print(f"Season {season}: {len(sampled_season)} games sampled from {season_size} total " + 
                  f"(weight={recency_factor:.2f})")
            sampled_data.append(sampled_season)
        
        # Combine sampled data from all seasons
        sampled_df = pd.concat(sampled_data)
        print(f"Total sampled data: {len(sampled_df)} games across {len(seasons)} seasons")
        
        # Sort by date for consistency
        sampled_df = sampled_df.sort_values('game_date')
        
        # Drop temporary columns
        if 'month' in sampled_df.columns:
            sampled_df = sampled_df.drop(['month', 'year'], axis=1)
        
        return sampled_df
    
    except Exception as e:
        print(f"Error loading historical data with seasonal sampling: {e}")
        traceback.print_exc()
        return pd.DataFrame()

# Define function as a wrapper around our new class for backward compatibility
def precompute_enhanced_features(sample_size: int = None, skip_rest_calc: bool = False, seasonal_sampling: bool = True) -> pd.DataFrame:
    """
    Backward compatibility wrapper around NBAFeatureGenerator with seasonal sampling.
    Maintains the same interface as the original precompute_enhanced_features function.
    """
    # Load historical data with seasonal sampling
    if seasonal_sampling:
        df = load_historical_data_with_seasonal_sampling(sample_per_season=200, min_year=2022)
    else:
        df = load_historical_data(sample_size=sample_size)
        
    if df.empty:
        return pd.DataFrame()
    
    # Use the new generator
    feature_generator = NBAFeatureGenerator(debug=True)
    return feature_generator.generate_all_features(df, skip_rest_calc=skip_rest_calc)

# For backward compatibility with generate_all_quarter_features
def generate_all_quarter_features(sample_size=1000, skip_rest_calc=False, seasonal_sampling=True):
    """
    Backward compatibility wrapper for the generate_all_quarter_features function.
    """
    return precompute_enhanced_features(sample_size=sample_size, skip_rest_calc=skip_rest_calc, seasonal_sampling=seasonal_sampling)

# Define a function to load historical data from the database
def load_historical_data(sample_size=None):
    """
    Load historical game data from the database.
    
    Args:
        sample_size (int): Maximum number of games to load (None for all)
        
    Returns:
        DataFrame with historical game data
    """
    print("Connecting to database...")
    try:
        engine = create_engine(config.DATABASE_URL)
        if sample_size is not None:
            query = f"SELECT * FROM nba_historical_game_stats ORDER BY game_date LIMIT {sample_size}"
            print(f"Using limited sample of {sample_size} games")
        else:
            query = "SELECT * FROM nba_historical_game_stats ORDER BY game_date"
        
        df = pd.read_sql(query, engine)
        df['game_date'] = pd.to_datetime(df['game_date'], errors='coerce')
        print(f"Loaded {len(df)} historical games")
        return df
    except Exception as e:
        print(f"Error loading historical data: {e}")
        traceback.print_exc()
        return pd.DataFrame()

# Test the new feature generator with seasonal sampling
print("Testing feature generation with seasonal sampling...")
input_df = load_historical_data_with_seasonal_sampling(sample_per_season=200, min_year=2022)

if input_df.empty:
    print("No data loaded. Check database connection.")
else:
    # Display season distribution
    season_counts = input_df['season'].value_counts().sort_index()
    print("\nSeasonal distribution of sampled data:")
    for season, count in season_counts.items():
        print(f"  {season}: {count} games")
    
    # Create an instance of our feature generator
    feature_generator = NBAFeatureGenerator(debug=True)
    
    # Generate features for the sampled dataset
    print("\n=== Generating Features for Seasonally Sampled Dataset ===")
    features_df = feature_generator.generate_all_features(input_df, skip_rest_calc=False)
    
    # Display a summary of the generated features
    print("\nFeature Statistics Summary:")
    display(features_df.describe())
    
    # Check rest day distributions specifically
    print("\nRest Features Summary:")
    rest_cols = ['rest_days_home', 'rest_days_away', 'rest_advantage', 
                'is_back_to_back_home', 'is_back_to_back_away']
    display(features_df[rest_cols].describe())
    
    # Get percentages for back-to-back games
    home_b2b_pct = features_df['is_back_to_back_home'].mean() * 100
    away_b2b_pct = features_df['is_back_to_back_away'].mean() * 100
    print(f"\nHome teams on back-to-back: {home_b2b_pct:.1f}%")
    print(f"Away teams on back-to-back: {away_b2b_pct:.1f}%")
    
    print("\nFeature generation with seasonal sampling complete.")

In [ ]:
# Cell 4-1-Vectorized: High-Performance Feature Computation with Vectorized Operations
import pandas as pd
import numpy as np
import time

def vectorized_feature_computation(df, feature_types=None):
    """
    Compute features using vectorized operations for better performance.
    
    Args:
        df: DataFrame with game data
        feature_types: List of feature types to compute (or None for all)
        
    Returns:
        DataFrame with computed features
    """
    try:
        # Create a copy to avoid modifying the original
        result_df = df.copy()
        
        # Define feature type sets if not specified
        all_feature_types = [
            'quarter_scores', 
            'rest_days', 
            'momentum', 
            'matchup_history', 
            'game_state'
        ]
        
        if feature_types is None:
            feature_types = all_feature_types
            
        # Validate the DataFrame has the minimum required columns
        required_columns = []
        if 'quarter_scores' in feature_types:
            required_columns.extend(['home_q1', 'home_q2', 'home_q3', 'home_q4', 
                                    'away_q1', 'away_q2', 'away_q3', 'away_q4'])
        if 'rest_days' in feature_types:
            required_columns.extend(['home_rest_days', 'away_rest_days'])
        
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            print(f"Missing required columns: {missing_columns}")
            return pd.DataFrame()
        
        # Feature Group 1: Quarter Scores Analysis
        if 'quarter_scores' in feature_types:
            # Calculate quarter differentials
            for q in range(1, 5):
                result_df[f'q{q}_diff'] = result_df[f'home_q{q}'] - result_df[f'away_q{q}']
            
            # Calculate cumulative scores
            result_df['home_first_half'] = result_df['home_q1'] + result_df['home_q2']
            result_df['away_first_half'] = result_df['away_q1'] + result_df['away_q2']
            result_df['home_second_half'] = result_df['home_q3'] + result_df['home_q4']
            result_df['away_second_half'] = result_df['away_q3'] + result_df['away_q4']
            
            # Calculate half differentials
            result_df['first_half_diff'] = result_df['home_first_half'] - result_df['away_first_half']
            result_df['second_half_diff'] = result_df['home_second_half'] - result_df['away_second_half']
            
            # Calculate total scores
            result_df['home_total'] = result_df['home_first_half'] + result_df['home_second_half']
            result_df['away_total'] = result_df['away_first_half'] + result_df['away_second_half']
            result_df['total_diff'] = result_df['home_total'] - result_df['away_total']
            
        # Feature Group 2: Rest Days Analysis
        if 'rest_days' in feature_types and 'home_rest_days' in df.columns and 'away_rest_days' in df.columns:
            # Rest advantage
            result_df['rest_advantage'] = result_df['home_rest_days'] - result_df['away_rest_days']
            
            # Back-to-back indicators (if available in the data)
            if 'is_home_b2b' not in df.columns:
                result_df['is_home_b2b'] = (result_df['home_rest_days'] == 0).astype(int)
            if 'is_away_b2b' not in df.columns:
                result_df['is_away_b2b'] = (result_df['away_rest_days'] == 0).astype(int)
            
            # Rest categories
            result_df['home_rest_cat'] = pd.cut(
                result_df['home_rest_days'], 
                bins=[-1, 0, 1, 2, 10], 
                labels=['0', '1', '2', '3+']
            ).astype(str)
            
            result_df['away_rest_cat'] = pd.cut(
                result_df['away_rest_days'], 
                bins=[-1, 0, 1, 2, 10], 
                labels=['0', '1', '2', '3+']
            ).astype(str)
        
        # Feature Group 3: Momentum Analysis
        if 'momentum' in feature_types and all(f'q{i}_diff' in result_df.columns for i in range(1, 5)):
            # Quarter-to-quarter momentum shifts
            result_df['q1_to_q2_momentum'] = result_df['q2_diff'] - result_df['q1_diff']
            result_df['q2_to_q3_momentum'] = result_df['q3_diff'] - result_df['q2_diff']
            result_df['q3_to_q4_momentum'] = result_df['q4_diff'] - result_df['q3_diff']
            
            # Weighted momentum (more recent quarters have higher weight)
            result_df['cumulative_momentum'] = (
                result_df['q1_to_q2_momentum'] * 0.2 + 
                result_df['q2_to_q3_momentum'] * 0.3 + 
                result_df['q3_to_q4_momentum'] * 0.5
            )
            
            # First half to second half momentum
            result_df['half_momentum'] = result_df['second_half_diff'] - result_df['first_half_diff']
        
        # Feature Group 4: Matchup History (if available)
        if 'matchup_history' in feature_types and 'prev_matchup_diff' in df.columns:
            # No additional processing needed if prev_matchup_diff is already available
            pass
        
        # Feature Group 5: Game State Features
        if 'game_state' in feature_types:
            # Score ratio (proportion of total points scored by home team)
            home_halftime = result_df['home_first_half'] 
            away_halftime = result_df['away_first_half']
            total_halftime = home_halftime + away_halftime
            
            # Avoid division by zero
            result_df['score_ratio'] = np.where(
                total_halftime > 0,
                home_halftime / total_halftime,
                0.5  # Default to 0.5 if no points scored
            )
            
            # Score differential at halftime
            result_df['score_differential'] = home_halftime - away_halftime
            
        return result_df
        
    except Exception as e:
        print(f"Error in vectorized feature computation: {e}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame()  # Return empty DataFrame on error
    
# Example usage:
# Safeguard against the NoneType error with try-except
try:
    print("Testing vectorized feature computation for improved performance...")
    
    # Make sure features_df exists and is a valid DataFrame
    if 'features_df' in globals() and isinstance(features_df, pd.DataFrame) and not features_df.empty:
        dataset_to_use = features_df
        print(f"Using existing features_df with {len(features_df)} rows")
    else:
        # Create sample data since features_df doesn't exist or is invalid
        print("features_df not available, creating sample data instead...")
        sample_data = {
            'home_q1': np.random.randint(15, 35, 100),
            'home_q2': np.random.randint(15, 35, 100),
            'home_q3': np.random.randint(15, 35, 100),
            'home_q4': np.random.randint(15, 35, 100),
            'away_q1': np.random.randint(15, 35, 100),
            'away_q2': np.random.randint(15, 35, 100),
            'away_q3': np.random.randint(15, 35, 100),
            'away_q4': np.random.randint(15, 35, 100),
            'home_rest_days': np.random.randint(0, 5, 100),
            'away_rest_days': np.random.randint(0, 5, 100),
            'prev_matchup_diff': np.random.normal(0, 10, 100)
        }
        dataset_to_use = pd.DataFrame(sample_data)
        print(f"Created sample dataset with {len(dataset_to_use)} rows")
    
    # Execute with timing
    start_time = time.time()
    vectorized_features = vectorized_feature_computation(dataset_to_use)
    elapsed = time.time() - start_time
    print(f"Vectorized computation completed in {elapsed:.3f} seconds")
    
    # Guarantee vectorized_features is a DataFrame
    if vectorized_features is None:
        print("WARNING: Function returned None instead of DataFrame. Creating empty DataFrame.")
        vectorized_features = pd.DataFrame()
    
    # Now it's safe to access .columns
    print(f"Generated {len(vectorized_features.columns)} features")
except Exception as e:
    print(f"ERROR in vectorized feature computation test: {e}")
    import traceback
    traceback.print_exc()
    print("Creating empty DataFrame as fallback")
    vectorized_features = pd.DataFrame()
    print(f"Generated {len(vectorized_features.columns)} features (empty dataframe)")
    
# Define the variable in the global scope to ensure it exists
if 'vectorized_features' not in globals() or vectorized_features is None:
    print("WARNING: vectorized_features is not defined or is None. Creating empty DataFrame.")
    vectorized_features = pd.DataFrame()

In [ ]:
# Cell 4-2: Enhanced Dynamic Momentum Calculation

def calculate_dynamic_momentum(quarter_scores, current_quarter, score_differential, game_context=None):
    """
    Calculate momentum with improved dynamic impact based on score differential and game context.
    
    Args:
        quarter_scores: Dict with quarter scores (home_q1, away_q1, etc.)
        current_quarter: Current quarter (1-4)
        score_differential: Current absolute score differential
        game_context: Optional dict with additional game context
        
    Returns:
        Dict with momentum values and impact factor
    """
    # Extract quarter scores with safe fallbacks
    home_quarters = []
    away_quarters = []
    
    for q in range(1, 5):
        # Safe extraction with fallbacks
        home_q_val = quarter_scores.get(f'home_q{q}', None)
        away_q_val = quarter_scores.get(f'away_q{q}', None)
        
        # Convert to float and handle missing values
        home_q_val = float(home_q_val) if home_q_val is not None else 0.0
        away_q_val = float(away_q_val) if away_q_val is not None else 0.0
        
        home_quarters.append(home_q_val)
        away_quarters.append(away_q_val)
    
    # Initialize momentum values
    momentum_values = {
        'q1_to_q2': 0.0,
        'q2_to_q3': 0.0,
        'q3_to_q4': 0.0,
        'cumulative': 0.0
    }
    
    # Check if this is start of a quarter - time_remaining close to 12 minutes
    is_early_in_quarter = False
    if game_context and 'time_remaining_in_quarter' in game_context:
        time_remaining = float(game_context.get('time_remaining_in_quarter', 0))
        is_early_in_quarter = time_remaining > 11.0  # Within first minute of quarter
    
    # Calculate quarter-to-quarter momentum if data available
    if current_quarter >= 2 and home_quarters[0] > 0 and home_quarters[1] > 0:
        momentum_values['q1_to_q2'] = (home_quarters[1] - home_quarters[0]) - (away_quarters[1] - away_quarters[0])
        
    if current_quarter >= 3 and home_quarters[1] > 0 and home_quarters[2] > 0:
        # If early in Q3, reduce the impact of Q2-Q3 momentum (still forming)
        q2_q3_value = (home_quarters[2] - home_quarters[1]) - (away_quarters[2] - away_quarters[1])
        if is_early_in_quarter and current_quarter == 3:
            momentum_values['q2_to_q3'] = q2_q3_value * 0.5  # Reduced impact
        else:
            momentum_values['q2_to_q3'] = q2_q3_value
        
    if current_quarter >= 4 and home_quarters[2] > 0 and home_quarters[3] > 0:
        # If early in Q4, reduce the impact of Q3-Q4 momentum (still forming)
        q3_q4_value = (home_quarters[3] - home_quarters[2]) - (away_quarters[3] - away_quarters[2])
        if is_early_in_quarter and current_quarter == 4:
            momentum_values['q3_to_q4'] = q3_q4_value * 0.5  # Reduced impact
        else:
            momentum_values['q3_to_q4'] = q3_q4_value
    
    # Apply adaptive weighting based on game state
    # Instead of fixed weights, we'll adjust based on recency and game context
    
    # Base weights that increase with quarter number (more weight to recent quarters)
    # This provides a non-linear increase in weights by quarter
    base_weights = {
        1: 0.15,  # Q1→Q2 weight
        2: 0.25,  # Q2→Q3 weight
        3: 0.60   # Q3→Q4 weight
    }
    
    # Adjust weights based on score differential
    if score_differential < 5:
        # Close games: emphasize recent momentum even more
        adjusted_weights = {
            1: base_weights[1] * 0.8,     # Reduce early weight
            2: base_weights[2] * 1.1,     # Slightly increase mid weight
            3: base_weights[3] * 1.1      # Slightly increase late weight
        }
    elif score_differential > 15:
        # Blowouts: more balanced weighting (momentum less predictive)
        adjusted_weights = {
            1: base_weights[1] * 1.2,     # Increase early weight
            2: base_weights[2] * 1.0,     # Keep mid weight
            3: base_weights[3] * 0.9      # Decrease late weight
        }
    else:
        # Default case
        adjusted_weights = base_weights
    
    # Collect available momentum values and their weights
    available_momentum = []
    applied_weights = []
    
    if current_quarter >= 2 and momentum_values['q1_to_q2'] != 0:
        available_momentum.append(momentum_values['q1_to_q2'])
        applied_weights.append(adjusted_weights[1])
        
    if current_quarter >= 3 and momentum_values['q2_to_q3'] != 0:
        available_momentum.append(momentum_values['q2_to_q3'])
        applied_weights.append(adjusted_weights[2])
        
    if current_quarter >= 4 and momentum_values['q3_to_q4'] != 0:
        available_momentum.append(momentum_values['q3_to_q4'])
        applied_weights.append(adjusted_weights[3])
    
    # Calculate weighted momentum if we have values
    if available_momentum:
        weighted_sum = sum(m * w for m, w in zip(available_momentum, applied_weights))
        weight_sum = sum(applied_weights)
        momentum_values['cumulative'] = weighted_sum / weight_sum if weight_sum > 0 else 0
    
    # Dynamic momentum impact based on score differential - non-linear relationship
    # Use sigmoid function for smooth transition between impact levels
    if score_differential < 5:
        impact_factor = 1.0  # Full impact in very close games
    elif score_differential >= 20:
        impact_factor = 0.2  # Minimal impact in extreme blowouts
    else:
        # Sigmoid-like function for smooth transition
        # Maps 5-20 point differential to 1.0-0.2 impact factor
        normalized_diff = (score_differential - 5) / 15.0  # 0 to 1 range
        impact_factor = 1.0 - 0.8 * (normalized_diff ** 2)  # Non-linear decay
    
    # Consider time remaining - momentum matters more late in the game
    if game_context and 'time_remaining_mins' in game_context:
        time_factor = 1.0 - (game_context['time_remaining_mins'] / 48.0)  # 0 to 1
        impact_factor *= (0.7 + 0.3 * time_factor)  # 30% boost late in game
    
    # Normalize momentum to reasonable values
    for key in momentum_values:
        if key != 'cumulative':
            momentum_values[key] = max(min(momentum_values[key], 20), -20)
    
    momentum_values['cumulative'] = max(min(momentum_values['cumulative'] / 15.0, 1.0), -1.0)
    momentum_values['impact_factor'] = impact_factor
    
    return momentum_values

# Test the dynamic momentum calculation with different game scenarios
print("Testing Enhanced Momentum Calculation with different game scenarios:")

test_scenarios = [
    {
        "name": "Close game with momentum shift",
        "quarter_scores": {
            "home_q1": 25, "away_q1": 28,
            "home_q2": 35, "away_q2": 25,
            "home_q3": 30, "away_q3": 28,
            "home_q4": 0, "away_q4": 0
        },
        "current_quarter": 3,
        "score_diff": 9
    },
    {
        "name": "Blowout game with little momentum change",
        "quarter_scores": {
            "home_q1": 35, "away_q1": 22,
            "home_q2": 32, "away_q2": 20,
            "home_q3": 30, "away_q3": 18,
            "home_q4": 0, "away_q4": 0
        },
        "current_quarter": 3,
        "score_diff": 37
    },
    {
        "name": "Late game comeback brewing",
        "quarter_scores": {
            "home_q1": 20, "away_q1": 30,
            "home_q2": 22, "away_q2": 28,
            "home_q3": 30, "away_q3": 20,
            "home_q4": 15, "away_q4": 8
        },
        "current_quarter": 4,
        "score_diff": 1
    }
]

for i, scenario in enumerate(test_scenarios):
    momentum = calculate_dynamic_momentum(
        scenario["quarter_scores"],
        scenario["current_quarter"],
        scenario["score_diff"]
    )
    
    print(f"\nScenario {i+1}: {scenario['name']}")
    print(f"  Current Quarter: {scenario['current_quarter']}")
    print(f"  Score Differential: {scenario['score_diff']} points")
    print(f"  Quarter-to-Quarter Momentum:")
    print(f"    Q1→Q2: {momentum['q1_to_q2']:.1f}")
    print(f"    Q2→Q3: {momentum['q2_to_q3']:.1f}")
    if scenario['current_quarter'] >= 4:
        print(f"    Q3→Q4: {momentum['q3_to_q4']:.1f}")
    print(f"  Cumulative Momentum: {momentum['cumulative']:.2f}")
    print(f"  Momentum Impact Factor: {momentum['impact_factor']:.2f}")
    
    if momentum['impact_factor'] < 0.5:
        print("  Analysis: Momentum has reduced impact due to score differential")
    else:
        print("  Analysis: Momentum has significant impact in this close game")

In [ ]:
# Cell 4A: Feature Engineering Analysis and Validation

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time

print("=" * 80)
print("PART 1: FEATURE CORRELATION ANALYSIS")
print("=" * 80)

def analyze_feature_correlations(features_df, threshold=0.8, figsize=(20, 16)):
    """
    Analyze and visualize feature correlations with advanced metrics including
    non-linear relationships and statistical significance testing.
    
    Args:
        features_df: DataFrame with calculated features
        threshold: Correlation threshold to highlight
        figsize: Figure size for the heatmap
        
    Returns:
        Dictionary with correlation results and feature relationships
    """
    from scipy import stats
    import seaborn as sns
    
    # Select only numeric columns for correlation analysis
    numeric_features = features_df.select_dtypes(include=['int64', 'float64'])
    
    print(f"Analyzing correlations between {len(numeric_features.columns)} numeric features")
    
    # Calculate Pearson (linear) correlation matrix
    pearson_corr = numeric_features.corr(method='pearson')
    
    # Calculate Spearman (rank) correlation matrix for non-linear relationships
    spearman_corr = numeric_features.corr(method='spearman')
    
    # Identify differences between Pearson and Spearman
    # Large differences indicate non-linear relationships
    correlation_diff = np.abs(pearson_corr - spearman_corr)
    
    # Create a mask for the upper triangle of the correlation matrix
    mask = np.triu(np.ones_like(pearson_corr, dtype=bool))
    
    # Set up the matplotlib figure for Pearson correlations
    plt.figure(figsize=figsize)
    
    # Create a custom diverging colormap
    cmap = sns.diverging_palette(230, 20, as_cmap=True)
    
    # Draw the heatmap with the mask and correct aspect ratio
    sns.heatmap(pearson_corr, mask=mask, cmap=cmap, vmax=1, vmin=-1, center=0,
                square=True, linewidths=.5, cbar_kws={"shrink": .5}, annot=True, fmt=".2f")
    
    plt.title('Feature Correlation Matrix (Pearson - Linear)', fontsize=16)
    plt.tight_layout()
    plt.show()
    
    # Find highly correlated features
    high_corr_pairs = {}
    for i in range(len(pearson_corr.columns)):
        for j in range(i):
            if abs(pearson_corr.iloc[i, j]) > threshold:
                feat_i = pearson_corr.columns[i]
                feat_j = pearson_corr.columns[j]
                high_corr_pairs[(feat_i, feat_j)] = pearson_corr.iloc[i, j]
    
    # Print the results
    if high_corr_pairs:
        print(f"\nHighly correlated features (|r| > {threshold}):")
        for (feat1, feat2), corr in sorted(high_corr_pairs.items(), key=lambda x: abs(x[1]), reverse=True):
            print(f"  • {feat1} and {feat2}: r = {corr:.3f}")
        
        # Group correlations by feature to identify redundant sets
        feature_groups = {}
        for (feat1, feat2), _ in high_corr_pairs.items():
            if feat1 not in feature_groups:
                feature_groups[feat1] = set()
            if feat2 not in feature_groups:
                feature_groups[feat2] = set()
            feature_groups[feat1].add(feat2)
            feature_groups[feat2].add(feat1)
        
        # Find groups with multiple high correlations
        print("\nPotential feature groups to simplify:")
        for feature, correlated in feature_groups.items():
            if len(correlated) > 1:
                print(f"  • {feature} is highly correlated with: {', '.join(correlated)}")
    else:
        print(f"No feature pairs with correlation above {threshold} found.")
    
    # Now also display the non-linear analysis if differences are significant
    if (correlation_diff > 0.2).any().any():
        print("\nDetected potential non-linear relationships. Displaying Spearman correlation matrix...")
        
        plt.figure(figsize=figsize)
        sns.heatmap(spearman_corr, mask=mask, cmap=cmap, vmax=1, vmin=-1, center=0,
                    square=True, linewidths=.5, cbar_kws={"shrink": .5}, annot=True, fmt=".2f")
        
        plt.title('Non-linear Correlation Matrix (Spearman)', fontsize=16)
        plt.tight_layout()
        plt.show()
        
        # Find significant non-linear relationships
        non_linear_relationships = []
        for i in range(len(correlation_diff.columns)):
            for j in range(i):
                if correlation_diff.iloc[i, j] > 0.2:  # Significant difference
                    feat_i = correlation_diff.columns[i]
                    feat_j = correlation_diff.columns[j]
                    non_linear_relationships.append((
                        feat_i, feat_j, 
                        pearson_corr.iloc[i, j],
                        spearman_corr.iloc[i, j],
                        correlation_diff.iloc[i, j]
                    ))
        
        if non_linear_relationships:
            print("\nPotential non-linear relationships found:")
            for feat1, feat2, pearson, spearman, diff in sorted(non_linear_relationships, key=lambda x: x[4], reverse=True):
                print(f"  • {feat1} and {feat2}: Pearson={pearson:.3f}, Spearman={spearman:.3f}, Diff={diff:.3f}")
    
    return high_corr_pairs
# =============================================================================
print("\n" + "=" * 80)
print("PART 2: EDGE CASE VALIDATION")
print("=" * 80)

def create_edge_case_tests():
    """
    Create a set of edge case scenarios to test feature generation robustness.
    
    Returns:
        DataFrame with various edge cases
    """
    print("Creating edge case test scenarios...")
    
    # Base test case (normal game)
    normal_case = {
        'game_id': 10001,
        'home_team': 'Boston Celtics',
        'away_team': 'Los Angeles Lakers',
        'game_date': pd.Timestamp('2023-01-15'),
        'home_q1': 28, 'home_q2': 26, 'home_q3': 30, 'home_q4': 29,
        'away_q1': 25, 'away_q2': 28, 'away_q3': 24, 'away_q4': 26,
        'home_score': 113,
        'away_score': 103,
        'current_quarter': 4,
        'description': 'Normal complete game'
    }
    
    # Edge case 1: Missing quarter data
    missing_quarters = normal_case.copy()
    missing_quarters.update({
        'game_id': 10002,
        'home_q3': None, 'home_q4': None,
        'away_q3': None, 'away_q4': None,
        'home_score': 54,
        'away_score': 53,
        'current_quarter': 2,
        'description': 'Missing Q3 and Q4 (game in progress)'
    })
    
    # Edge case 2: All quarters missing
    all_quarters_missing = normal_case.copy()
    all_quarters_missing.update({
        'game_id': 10003,
        'home_q1': None, 'home_q2': None, 'home_q3': None, 'home_q4': None,
        'away_q1': None, 'away_q2': None, 'away_q3': None, 'away_q4': None,
        'home_score': 0,
        'away_score': 0,
        'current_quarter': 0,
        'description': 'Pre-game (all quarters missing)'
    })
    
    # Edge case 3: Extreme momentum shift
    extreme_momentum = normal_case.copy()
    extreme_momentum.update({
        'game_id': 10004,
        'home_q1': 10, 'home_q2': 40, 'home_q3': 10, 'home_q4': 40,
        'away_q1': 40, 'away_q2': 10, 'away_q3': 40, 'away_q4': 10,
        'home_score': 100,
        'away_score': 100,
        'description': 'Extreme momentum shifts'
    })
    
    # Edge case 4: Game with no history (new matchup)
    new_matchup = normal_case.copy()
    new_matchup.update({
        'game_id': 10005,
        'home_team': 'Expansion Team A',
        'away_team': 'Expansion Team B',
        'description': 'New teams with no history'
    })
    
    # Edge case 5: Zero scoring quarter
    zero_quarter = normal_case.copy()
    zero_quarter.update({
        'game_id': 10006,
        'home_q3': 0,
        'away_q3': 0,
        'description': 'Quarter with zero points (Q3)'
    })
    
    # Edge case 6: Missing game date
    missing_date = normal_case.copy()
    missing_date.update({
        'game_id': 10007,
        'game_date': None,
        'description': 'Missing game date'
    })
    
    # Combine all test cases
    test_cases = [
        normal_case,
        missing_quarters,
        all_quarters_missing,
        extreme_momentum,
        new_matchup,
        zero_quarter,
        missing_date
    ]
    
    # Create DataFrame
    test_df = pd.DataFrame(test_cases)
    
    # Print test case summary
    print(f"Created {len(test_df)} test cases:")
    for i, case in enumerate(test_cases):
        print(f"  {i+1}. {case['description']}")
    
    return test_df

def validate_edge_cases(feature_generator, test_df):
    """
    Process edge cases through the feature generator and validate the output.
    
    Args:
        feature_generator: Instance of NBAFeatureGenerator
        test_df: DataFrame with test cases
        
    Returns:
        DataFrame with processed features
    """
    print("\nValidating feature generation for edge cases...")
    
    # Process the test cases
    processed_df = feature_generator.generate_all_features(test_df, skip_rest_calc=True)
    
    # Check results
    validation_results = []
    
    for idx, row in processed_df.iterrows():
        game_id = row['game_id']
        desc = row['description']
        
        # Get original test case
        orig = test_df[test_df['game_id'] == game_id].iloc[0]
        
        # Define checks for each case
        if 'Missing Q3 and Q4' in desc:
            momentum_valid = pd.notna(row['q1_to_q2_momentum']) and pd.isna(row['q2_to_q3_momentum'])
            current_q_valid = row['current_quarter'] == 2
            
        elif 'Pre-game' in desc:
            momentum_valid = all(row[[
                'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum'
            ]].isna() | (row[[
                'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum'
            ]] == 0))
            current_q_valid = row['current_quarter'] == 0
            
        elif 'Extreme momentum' in desc:
            q1_to_q2 = row['q1_to_q2_momentum']
            q2_to_q3 = row['q2_to_q3_momentum']
            momentum_valid = (q1_to_q2 > 0 and q2_to_q3 < 0) or (q1_to_q2 < 0 and q2_to_q3 > 0)
            current_q_valid = row['current_quarter'] == 4
            
        elif 'new history' in desc:
            history_valid = row['prev_matchup_diff'] == 0
            momentum_valid = True
            current_q_valid = True
            
        elif 'zero points' in desc:
            momentum_valid = row['q2_to_q3_momentum'] != 0  # Should reflect the shift
            current_q_valid = row['current_quarter'] == 4
            
        elif 'Missing game date' in desc:
            # Rest features should be defaulted
            rest_valid = (row['rest_days_home'] == 2 and row['rest_days_away'] == 2)
            momentum_valid = True
            current_q_valid = True
            
        else:  # Normal case
            momentum_valid = True
            current_q_valid = row['current_quarter'] == 4
        
        # Common validations for all cases
        quarter_sums_valid = abs((row['home_q1'] + row['home_q2'] + row['home_q3'] + row['home_q4']) - 
                                 row['home_score']) < 0.01
        first_half_valid = abs(row['first_half_diff'] - 
                              ((row['home_q1'] + row['home_q2']) - (row['away_q1'] + row['away_q2']))) < 0.01
        
        # Store validation result
        validation_results.append({
            'game_id': game_id,
            'description': desc,
            'momentum_features_valid': momentum_valid,
            'current_quarter_valid': current_q_valid,
            'quarter_sums_valid': quarter_sums_valid,
            'first_half_diff_valid': first_half_valid
        })
    
    # Create validation summary DataFrame
    validation_df = pd.DataFrame(validation_results)
    
    # Print validation summary
    print("\nValidation Results:")
    for idx, result in validation_df.iterrows():
        all_valid = all([
            result['momentum_features_valid'],
            result['current_quarter_valid'],
            result['quarter_sums_valid'],
            result['first_half_diff_valid']
        ])
        status = "✅ PASSED" if all_valid else "❌ FAILED"
        print(f"Test Case {idx+1}: {result['description']} - {status}")
        
        if not all_valid:
            print(f"  Failed checks:")
            if not result['momentum_features_valid']:
                print("  - Momentum features invalid")
            if not result['current_quarter_valid']:
                print("  - Current quarter invalid")
            if not result['quarter_sums_valid']:
                print("  - Quarter sums don't match game score")
            if not result['first_half_diff_valid']:
                print("  - First half differential incorrect")
    
    # Return both raw features and validation results
    return processed_df, validation_df

# Create and run edge case tests
edge_case_df = create_edge_case_tests()
test_generator = NBAFeatureGenerator(debug=True)
processed_edge_cases, validation_results = validate_edge_cases(test_generator, edge_case_df)

# Display detailed edge case results for the most interesting cases
print("\nDetailed Feature Values for Selected Edge Cases:")
interesting_cases = ['Missing Q3 and Q4', 'Extreme momentum', 'Missing game date']
for idx, row in processed_edge_cases.iterrows():
    if any(case in row['description'] for case in interesting_cases):
        print(f"\nTest Case: {row['description']}")
        print(f"  Current Quarter: {row['current_quarter']}")
        print(f"  Score: {row['home_team']} {row['home_score']} - {row['away_score']} {row['away_team']}")
        print(f"  Quarter Scores: {row['home_q1']}-{row['home_q2']}-{row['home_q3']}-{row['home_q4']} vs "
              f"{row['away_q1']}-{row['away_q2']}-{row['away_q3']}-{row['away_q4']}")
        print(f"  Momentum Features:")
        print(f"    Q1→Q2: {row.get('q1_to_q2_momentum', 'N/A')}, "
              f"Q2→Q3: {row.get('q2_to_q3_momentum', 'N/A')}, "
              f"Q3→Q4: {row.get('q3_to_q4_momentum', 'N/A')}")
        print(f"    Cumulative: {row.get('cumulative_momentum', 'N/A')}")
        print(f"  Critical Features:")
        print(f"    First Half Diff: {row.get('first_half_diff', 'N/A')}")
        print(f"    Pre-Q4 Diff: {row.get('pre_q4_diff', 'N/A')}")

print("\nFeature engineering validation complete!")

In [ ]:
# Cell 4B: Enhanced Model Training with Time-Series Cross-Validation

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib
import time
import os
import warnings
warnings.filterwarnings('ignore')

# Mock function for precompute_enhanced_features since it's not defined
def precompute_enhanced_features(sample_size=1000, skip_rest_calc=False):
    """
    Mock implementation to generate sample features for testing
    
    Args:
        sample_size: Number of games to generate
        skip_rest_calc: Whether to skip rest day calculations
        
    Returns:
        DataFrame with features
    """
    print(f"Generating mock features for {sample_size} games...")
    
    # Create sample data with realistic values
    np.random.seed(42)
    sample_data = {
        'game_id': range(1, sample_size + 1),
        'game_date': pd.date_range(start='2021-01-01', periods=sample_size),
        'home_q1': np.random.randint(15, 35, sample_size),
        'home_q2': np.random.randint(15, 35, sample_size),
        'home_q3': np.random.randint(15, 35, sample_size),
        'home_q4': np.random.randint(15, 35, sample_size),
        'away_q1': np.random.randint(15, 35, sample_size),
        'away_q2': np.random.randint(15, 35, sample_size),
        'away_q3': np.random.randint(15, 35, sample_size),
        'away_q4': np.random.randint(15, 35, sample_size),
        'score_ratio': np.random.uniform(0.4, 0.6, sample_size),
        'rolling_home_score': np.random.uniform(90, 110, sample_size),
        'rolling_away_score': np.random.uniform(90, 110, sample_size),
        'prev_matchup_diff': np.random.normal(0, 10, sample_size),
    }
    
    # Add rest days features if not skipped
    if not skip_rest_calc:
        sample_data.update({
            'rest_days_home': np.random.randint(0, 5, sample_size),
            'rest_days_away': np.random.randint(0, 5, sample_size),
            'rest_advantage': np.random.randint(-3, 4, sample_size),
            'is_back_to_back_home': np.random.choice([0, 1], sample_size, p=[0.8, 0.2]),
            'is_back_to_back_away': np.random.choice([0, 1], sample_size, p=[0.8, 0.2]),
        })
    
    # Add momentum features
    sample_data.update({
        'q1_to_q2_momentum': np.random.normal(0, 5, sample_size),
        'q2_to_q3_momentum': np.random.normal(0, 5, sample_size),
        'q3_to_q4_momentum': np.random.normal(0, 5, sample_size),
        'cumulative_momentum': np.random.normal(0, 7, sample_size),
    })
    
    df = pd.DataFrame(sample_data)
    
    # Calculate home score for target
    df['home_score'] = df['home_q1'] + df['home_q2'] + df['home_q3'] + df['home_q4']
    
    return df

# Mock database access function
def get_target_data_from_db():
    """
    Mock function to simulate getting target data from database
    
    Returns:
        DataFrame with target data
    """
    print("Using mock target data instead of database connection")
    
    # Create sample target data
    np.random.seed(42)
    sample_size = 1200  # Slightly larger than features to test merge
    
    target_data = {
        'game_id': range(1, sample_size + 1),
        'target_home_score': np.random.randint(80, 130, sample_size),
        'game_date': pd.date_range(start='2021-01-01', periods=sample_size)
    }
    
    return pd.DataFrame(target_data)

def train_enhanced_model_with_cv(sample_size=1000, n_splits=5, save_model=True, use_mock_data=True):
    """
    Train a model using enhanced features with proper time-series cross-validation.
    
    Args:
        sample_size: Maximum number of games to use for training
        n_splits: Number of cross-validation splits
        save_model: Whether to save the final model
        use_mock_data: Whether to use mock data instead of actual data
        
    Returns:
        dict: Training results including model, metrics, and feature importance
    """
    start_time = time.time()
    
    try:
        # 1. Generate features with consistent naming
        print(f"Generating features for up to {sample_size} games...")
        features_df = precompute_enhanced_features(sample_size=sample_size, skip_rest_calc=False)
        if features_df.empty:
            print("No features generated. Aborting training.")
            return None
        
        # Check for and rename any ambiguous columns before merging
        if 'home_score' in features_df.columns:
            features_df = features_df.rename(columns={'home_score': 'home_score_orig'})
        
        # 2. Get target data (either from database or mock)
        print("Retrieving target data...")
        if use_mock_data:
            target_df = get_target_data_from_db()
        else:
            try:
                # This section would connect to the actual database
                # For now, we'll use the mock function
                target_df = get_target_data_from_db()
            except Exception as e:
                print(f"Database connection failed: {e}")
                print("Falling back to mock data...")
                target_df = get_target_data_from_db()
        
        # Ensure we have a date column for time-series CV
        if 'game_date' not in target_df.columns:
            print("Error: No game_date column in target data. Adding mock dates.")
            target_df['game_date'] = pd.date_range(start='2021-01-01', periods=len(target_df))
        
        # Convert to datetime
        target_df['game_date'] = pd.to_datetime(target_df['game_date'], errors='coerce')
        
        # Verify required columns are in target data
        required_target_cols = ['game_id', 'target_home_score', 'game_date']
        missing_cols = [col for col in required_target_cols if col not in target_df.columns]
        if missing_cols:
            print(f"Error: Missing required columns in target data: {missing_cols}")
            return None
        
        # Sort by date for time-series validation
        target_df = target_df.sort_values('game_date')
        
        # 3. Merge with explicit column naming
        print("Merging features with target data...")
        merged = pd.merge(
            features_df, 
            target_df[['game_id', 'target_home_score', 'game_date']], 
            on='game_id', 
            how='inner'
        )
        
        # Verify merge was successful
        if len(merged) == 0:
            print("Error: No rows after merging features with targets. Check game_id consistency.")
            return None
        
        # Verify game_date survived the merge
        if 'game_date' not in merged.columns:
            print("Error: game_date column lost during merge. Adding it back.")
            # We'll try to fix by joining game_date from target_df based on game_id
            game_id_to_date = dict(zip(target_df['game_id'], target_df['game_date']))
            merged['game_date'] = merged['game_id'].map(game_id_to_date)
            
            # If still missing, add mock dates
            if 'game_date' not in merged.columns or merged['game_date'].isna().all():
                print("Still missing game_date, creating artificial dates.")
                merged['game_date'] = pd.date_range(start='2021-01-01', periods=len(merged))
        
        # Sort by date for time-series splitting
        merged = merged.sort_values('game_date')
        print(f"Merged data contains {len(merged)} games from {merged['game_date'].min()} to {merged['game_date'].max()}")
        
        # 4. Prepare feature columns
        feature_cols = [
            'home_q1', 'home_q2', 'home_q3', 'home_q4',
            'score_ratio', 'rolling_home_score', 'rolling_away_score',
            'prev_matchup_diff'
        ]
        
        # Add rest day features if available
        rest_features = ['rest_days_home', 'rest_days_away', 'rest_advantage',
                         'is_back_to_back_home', 'is_back_to_back_away']
        feature_cols.extend([f for f in rest_features if f in merged.columns])
        
        # Add momentum features if available
        momentum_features = ['q1_to_q2_momentum', 'q2_to_q3_momentum', 
                            'q3_to_q4_momentum', 'cumulative_momentum']
        feature_cols.extend([f for f in momentum_features if f in merged.columns])
        
        # Check for missing features
        missing_features = [col for col in feature_cols if col not in merged.columns]
        if missing_features:
            print(f"Warning: Missing feature columns: {missing_features}")
            # Remove missing features from list
            feature_cols = [col for col in feature_cols if col in merged.columns]
            if not feature_cols:
                print("Error: No valid features remain. Aborting training.")
                return None
        
        # 5. Set up training data
        X = merged[feature_cols]
        y = merged['target_home_score']  # Using explicitly named target column
        
        # 6. Time-Series Cross-Validation
        print(f"Performing {n_splits}-fold time-series cross-validation...")
        
        # Create time-series split iterator
        tscv = TimeSeriesSplit(n_splits=n_splits)
        
        # Initialize metrics storage
        cv_scores = {
            'train_rmse': [],
            'test_rmse': [],
            'train_mae': [],
            'test_mae': [],
            'r2_scores': []
        }
        
        # Train-test splits for visualization
        splits = []
        
        # Perform cross-validation
        fold = 1
        for train_idx, test_idx in tscv.split(X):
            print(f"Fold {fold}/{n_splits}:")
            
            # Split data
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            
            # Store indices for visualization
            splits.append((train_idx, test_idx))
            
            # Train model
            model = GradientBoostingRegressor(
                n_estimators=200, 
                max_depth=5, 
                learning_rate=0.1,
                subsample=0.8,  # Added for better generalization
                random_state=42
            )
            model.fit(X_train, y_train)
            
            # Evaluate on train and test sets
            y_train_pred = model.predict(X_train)
            y_test_pred = model.predict(X_test)
            
            # Calculate metrics
            train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
            test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
            train_mae = mean_absolute_error(y_train, y_train_pred)
            test_mae = mean_absolute_error(y_test, y_test_pred)
            r2 = r2_score(y_test, y_test_pred)
            
            # Store metrics
            cv_scores['train_rmse'].append(train_rmse)
            cv_scores['test_rmse'].append(test_rmse)
            cv_scores['train_mae'].append(train_mae)
            cv_scores['test_mae'].append(test_mae)
            cv_scores['r2_scores'].append(r2)
            
            # Print fold results
            print(f"  Train RMSE: {train_rmse:.2f}, Test RMSE: {test_rmse:.2f}")
            print(f"  Train MAE: {train_mae:.2f}, Test MAE: {test_mae:.2f}")
            print(f"  R² Score: {r2:.4f}")
            print(f"  Train size: {len(X_train)}, Test size: {len(X_test)}")
            
            fold += 1
        
        # Calculate average metrics
        avg_metrics = {
            'avg_train_rmse': np.mean(cv_scores['train_rmse']),
            'avg_test_rmse': np.mean(cv_scores['test_rmse']),
            'avg_train_mae': np.mean(cv_scores['train_mae']),
            'avg_test_mae': np.mean(cv_scores['test_mae']),
            'avg_r2': np.mean(cv_scores['r2_scores']),
            'std_test_rmse': np.std(cv_scores['test_rmse']),
            'std_r2': np.std(cv_scores['r2_scores'])
        }
        
        # 7. Train final model on all data
        print("Training final model on all data...")
        final_model = GradientBoostingRegressor(
            n_estimators=200, 
            max_depth=5, 
            learning_rate=0.1,
            subsample=0.8,
            random_state=42
        )
        final_model.fit(X, y)
        
        # 8. Feature importance
        feature_importance = {}
        if hasattr(final_model, 'feature_importances_'):
            feature_importance = dict(zip(feature_cols, final_model.feature_importances_))
            sorted_importance = sorted(feature_importance.items(), key=lambda x: x[1], reverse=True)
            
            print("\nFeature Importance:")
            for feature, importance in sorted_importance:
                print(f"  {feature}: {importance:.4f}")
        
        # 9. Visualize model performance
        plt.figure(figsize=(15, 10))
        
        # Visualize time-series splits
        plt.subplot(2, 2, 1)
        for i, (train_idx, test_idx) in enumerate(splits):
            train_range = range(min(train_idx), max(train_idx) + 1)
            test_range = range(min(test_idx), max(test_idx) + 1)
            plt.scatter(train_range, [i] * len(train_range), label=f'Train Fold {i+1}', s=5, alpha=0.5)
            plt.scatter(test_range, [i] * len(test_range), label=f'Test Fold {i+1}', s=10)
        
        plt.title('Time-Series Cross-Validation Splits')
        plt.xlabel('Sample Index (ordered by date)')
        plt.ylabel('CV Iteration')
        plt.yticks(range(n_splits))
        
        # Visualize RMSE by fold
        plt.subplot(2, 2, 2)
        plt.plot(range(1, n_splits + 1), cv_scores['train_rmse'], 'o-', label='Train RMSE')
        plt.plot(range(1, n_splits + 1), cv_scores['test_rmse'], 'o-', label='Test RMSE')
        plt.title('RMSE by Fold')
        plt.xlabel('Fold')
        plt.ylabel('RMSE')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # Visualize R² by fold
        plt.subplot(2, 2, 3)
        plt.plot(range(1, n_splits + 1), cv_scores['r2_scores'], 'o-', color='green')
        plt.title('R² Score by Fold')
        plt.xlabel('Fold')
        plt.ylabel('R² Score')
        plt.grid(True, alpha=0.3)
        
        # Visualize feature importance
        plt.subplot(2, 2, 4)
        if hasattr(final_model, 'feature_importances_'):
            sorted_idx = np.argsort(final_model.feature_importances_)
            plt.barh(range(len(sorted_idx)), final_model.feature_importances_[sorted_idx])
            plt.yticks(range(len(sorted_idx)), [feature_cols[i] for i in sorted_idx])
            plt.title('Feature Importance')
            plt.xlabel('Importance')
        
        plt.tight_layout()
        plt.show()
        
        # 10. Save model if requested
        if save_model:
            model_path = 'enhanced_xgb_model_with_cv.pkl'
            joblib.dump(final_model, model_path)
            print(f"Enhanced model saved to {model_path}")
        
        # Calculate total runtime
        total_time = time.time() - start_time
        print(f"Total training time: {total_time:.2f} seconds")
        
        # Return results
        return {
            'model': final_model,
            'cv_scores': cv_scores,
            'avg_metrics': avg_metrics,
            'feature_importance': feature_importance,
            'training_time': total_time,
            'feature_columns': feature_cols
        }
    
    except Exception as e:
        print(f"An error occurred during model training: {e}")
        import traceback
        traceback.print_exc()
        return None

# Run the improved training function
print("Running enhanced model training with time-series cross-validation...")
results = train_enhanced_model_with_cv(sample_size=1000, n_splits=5, use_mock_data=True)

if results:
    print("\nTraining Results Summary:")
    print(f"Average Test RMSE: {results['avg_metrics']['avg_test_rmse']:.2f} (±{results['avg_metrics']['std_test_rmse']:.2f})")
    print(f"Average Test MAE: {results['avg_metrics']['avg_test_mae']:.2f}")
    print(f"Average R² Score: {results['avg_metrics']['avg_r2']:.4f} (±{results['avg_metrics']['std_r2']:.4f})")
    
    # Print gap between train and test performance to check for overfitting
    train_test_gap = results['avg_metrics']['avg_train_rmse'] / results['avg_metrics']['avg_test_rmse']
    print(f"Train/Test RMSE Ratio: {train_test_gap:.2f}")
    
    if train_test_gap < 0.7:
        print("WARNING: Possible overfitting detected (train error much lower than test error)")
    elif train_test_gap > 0.95:
        print("Model is generalizing well (similar performance on train and test sets)")
else:
    print("Training failed. Please check the errors above.")

In [ ]:
# Cell 4B-2: Advanced Feature Extraction and Integration

def extract_advanced_features(game_data, team_stats_df=None):
    """
    Extract advanced features from game state and historical team stats
    
    Args:
        game_data: Dict with current game information
        team_stats_df: Optional DataFrame with team stats
        
    Returns:
        Dict of advanced features
    """
    features = {}
    
    # Get basic game info
    home_team = game_data.get('home_team')
    away_team = game_data.get('away_team')
    current_quarter = game_data.get('current_quarter', 0)
    home_score = game_data.get('home_score', 0)
    away_score = game_data.get('away_score', 0)
    
    # 1. Game State Features
    # Pace calculation
    elapsed_mins = 12 * (current_quarter - 1) + (12 - game_data.get('time_remaining_in_quarter', 0))
    if elapsed_mins > 0:
        pace = (home_score + away_score) / elapsed_mins
    else:
        pace = 0
    
    features['game_pace'] = pace
    features['score_per_minute'] = pace
    features['relative_pace'] = pace / 4.5  # Compared to league average 
    
    # 2. Momentum-derived features
    if 'momentum' in game_data:
        momentum = game_data['momentum']
        features['momentum_squared'] = momentum ** 2  # Non-linear impact
        features['momentum_direction'] = 1 if momentum > 0 else (-1 if momentum < 0 else 0)
        features['momentum_increasing'] = 1 if game_data.get('momentum_change', 0) > 0 else 0
    
    # 3. Time-weighted features
    remaining_mins = max(0, 48 - elapsed_mins)
    features['game_progress'] = 1 - (remaining_mins / 48)
    features['game_time_remaining'] = remaining_mins
    
    # Weight recent quarter performance more heavily
    for q in range(1, min(current_quarter + 1, 5)):
        q_weight = 0.5 ** (current_quarter - q)  # Exponential decay
        home_q_score = game_data.get(f'home_q{q}', 0)
        away_q_score = game_data.get(f'away_q{q}', 0)
        
        features[f'weighted_home_q{q}'] = home_q_score * q_weight
        features[f'weighted_away_q{q}'] = away_q_score * q_weight
    
    # 4. Interaction features
    score_diff = home_score - away_score
    features['score_diff_by_time'] = score_diff * features['game_progress']
    
    if 'momentum' in game_data:
        features['momentum_by_diff'] = game_data['momentum'] * (abs(score_diff) / 10)
    
    # 5. Team-specific features (if team stats available)
    if team_stats_df is not None and home_team in team_stats_df.index and away_team in team_stats_df.index:
        home_stats = team_stats_df.loc[home_team]
        away_stats = team_stats_df.loc[away_team]
        
        # Offensive/defensive efficiency differential
        features['off_rtg_diff'] = home_stats['offensive_rating'] - away_stats['offensive_rating']
        features['def_rtg_diff'] = home_stats['defensive_rating'] - away_stats['defensive_rating']
        features['net_rtg_diff'] = features['off_rtg_diff'] - features['def_rtg_diff']
        
        # Pace adjustment
        features['pace_vs_avg_home'] = pace / home_stats['avg_pace']
        features['pace_vs_avg_away'] = pace / away_stats['avg_pace']
    
    return features

In [ ]:
# Cell 4C: Quarter-Specific Ensemble Comparison (Improved Version)

import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

# Mock implementation of AdaptiveEnsembleFramework
class AdaptiveEnsembleFramework:
    """
    Mock implementation of AdaptiveEnsembleFramework to enable running the notebook.
    This would normally be imported from models.features, but we're creating it here
    to fix the import error.
    """
    def predict(self, main_prediction, quarter_prediction, current_quarter, context):
        """
        Predict using adaptive ensemble logic
        
        Args:
            main_prediction: Main model prediction
            quarter_prediction: Quarter-specific prediction
            current_quarter: Current quarter of the game
            context: Game context information
            
        Returns:
            tuple: (prediction, weights, confidence)
        """
        # Define weights and confidence by quarter
        if current_quarter == 1:
            weight_main, weight_quarter, confidence = 0.3, 0.7, 0.40
            strategy = "quarter_weighted"
        elif current_quarter == 2:
            weight_main, weight_quarter, confidence = 0.6, 0.4, 0.60
            strategy = "balanced"
        elif current_quarter == 3:
            weight_main, weight_quarter, confidence = 0.8, 0.2, 0.80
            strategy = "main_weighted"
        else:
            weight_main, weight_quarter, confidence = 1.0, 0.0, 0.95
            strategy = "main_only"
        
        # Adjust weights based on context
        score_diff = context.get('score_differential', 0)
        momentum = context.get('momentum', 0)
        
        # If big lead and positive momentum, increase confidence
        if score_diff > 15 and momentum > 0:
            confidence += 0.05
            
        # If close game, reduce confidence slightly
        if score_diff < 5:
            confidence -= 0.05
            
        # Ensure confidence stays in range
        confidence = max(0.3, min(0.95, confidence))
        
        # Calculate prediction
        prediction = weight_main * main_prediction + weight_quarter * quarter_prediction
        
        # Return results
        weights = {
            'main': weight_main,
            'quarter': weight_quarter,
            'strategy': strategy
        }
        
        return prediction, weights, confidence

# Mock function for database config
class MockConfig:
    DATABASE_URL = "sqlite:///:memory:"  # Use in-memory SQLite

config = MockConfig()

# Mock function for quarter_analysis module
def analyze_quarter_differences(df, home_cols, away_cols):
    return df

def train_momentum_models(df, features, target):
    from sklearn.linear_model import LinearRegression
    return LinearRegression().fit(df[features], df[target])

def identify_momentum_shifts(df, threshold=5):
    return df

def predict_remaining_quarters(models, game_data):
    return {'q3': 25, 'q4': 27}

# Define a simplified feature generation function
def generate_all_quarter_features(sample_size=1000, skip_rest_calc=False):
    """
    Simplified function to generate mock features for testing
    """
    print(f"Generating mock features for {sample_size} games...")
    
    # Create mock data
    np.random.seed(42)
    
    # Generate dates
    start_date = pd.Timestamp('2021-01-01')
    dates = pd.date_range(start=start_date, periods=sample_size)
    
    # Generate team names
    teams = ['LAL', 'BOS', 'GSW', 'MIA', 'PHI', 'NYK', 'CHI', 'DAL', 'HOU', 'LAC',
             'MIL', 'PHX', 'DEN', 'BKN', 'UTA', 'POR', 'ATL', 'MEM', 'NOP', 'TOR',
             'SAS', 'CHA', 'IND', 'WAS', 'OKC', 'MIN', 'SAC', 'DET', 'ORL', 'CLE']
    
    # Generate game_ids
    game_ids = range(1, sample_size + 1)
    
    # Generate random home and away teams
    home_teams = np.random.choice(teams, size=sample_size)
    away_teams = np.random.choice(teams, size=sample_size)
    
    # Create DataFrame with quarter scores
    df = pd.DataFrame({
        'game_id': game_ids,
        'game_date': dates,
        'home_team': home_teams,
        'away_team': away_teams,
        'home_q1': np.random.randint(18, 35, sample_size),
        'home_q2': np.random.randint(18, 35, sample_size),
        'home_q3': np.random.randint(18, 35, sample_size),
        'home_q4': np.random.randint(18, 35, sample_size),
        'away_q1': np.random.randint(18, 35, sample_size),
        'away_q2': np.random.randint(18, 35, sample_size),
        'away_q3': np.random.randint(18, 35, sample_size),
        'away_q4': np.random.randint(18, 35, sample_size),
    })
    
    # Calculate total scores
    df['home_score'] = df['home_q1'] + df['home_q2'] + df['home_q3'] + df['home_q4']
    df['away_score'] = df['away_q1'] + df['away_q2'] + df['away_q3'] + df['away_q4']
    
    # 1. Compute standard rolling scores
    print("Computing rolling scores...")
    df.sort_values('game_date', inplace=True)
    df['rolling_home_score'] = df.groupby('home_team')['home_score'].transform(
        lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
    df.sort_values('game_date', inplace=True)
    df['rolling_away_score'] = df.groupby('away_team')['away_score'].transform(
        lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
    
    # Fill missing values with reasonable defaults
    df['rolling_home_score'] = df['rolling_home_score'].fillna(df['home_score'].mean())
    df['rolling_away_score'] = df['rolling_away_score'].fillna(df['away_score'].mean())
    
    # 2. Calculate score ratio
    df['score_ratio'] = df['home_score'] / (df['home_score'] + df['away_score'])
    
    # 3. Add rest days calculation
    if not skip_rest_calc:
        print("Calculating rest days...")
        # Get previous game dates for each team
        df['prev_game_home'] = df.sort_values('game_date').groupby('home_team')['game_date'].shift(1)
        df['prev_game_away'] = df.sort_values('game_date').groupby('away_team')['game_date'].shift(1)
        
        # Calculate days of rest as the difference between current game date and previous game date
        df['rest_days_home'] = (df['game_date'] - df['prev_game_home']).dt.days
        df['rest_days_away'] = (df['game_date'] - df['prev_game_away']).dt.days
        
        # Fill NaN values (first game of the season) with a reasonable default (3 days)
        df['rest_days_home'] = df['rest_days_home'].fillna(3)
        df['rest_days_away'] = df['rest_days_away'].fillna(3)
        
        # Calculate rest advantage (positive means home team had more rest)
        df['rest_advantage'] = df['rest_days_home'] - df['rest_days_away']
        
        # Flag back-to-back games (1 day or less between games)
        df['is_back_to_back_home'] = (df['rest_days_home'] <= 1).astype(int)
        df['is_back_to_back_away'] = (df['rest_days_away'] <= 1).astype(int)
    
    # 4. Calculate momentum-related features
    print("Calculating momentum-related features...")
    # Quarter-to-quarter shifts
    for i in range(1, 4):
        df[f'home_q{i}_to_q{i+1}_shift'] = df[f'home_q{i+1}'] - df[f'home_q{i}']
        df[f'away_q{i}_to_q{i+1}_shift'] = df[f'away_q{i+1}'] - df[f'away_q{i}']
    
    # Momentum indicators (positive means momentum for home team)
    df['q1_to_q2_momentum'] = df['home_q1_to_q2_shift'] - df['away_q1_to_q2_shift']
    df['q2_to_q3_momentum'] = df['home_q2_to_q3_shift'] - df['away_q2_to_q3_shift']
    df['q3_to_q4_momentum'] = df['home_q3_to_q4_shift'] - df['away_q3_to_q4_shift']
    
    # Cumulative momentum across all quarters
    df['cumulative_momentum'] = df['q1_to_q2_momentum'] + df['q2_to_q3_momentum'] + df['q3_to_q4_momentum']
    
    # 5. Calculate critical combined quarter metrics needed for the models
    print("Calculating combined quarter metrics...")
    # First half differential
    df['first_half_diff'] = (df['home_q1'] + df['home_q2']) - (df['away_q1'] + df['away_q2'])
    
    # Pre-Q4 differential
    df['pre_q4_diff'] = (df['home_q1'] + df['home_q2'] + df['home_q3']) - (df['away_q1'] + df['away_q2'] + df['away_q3'])
    
    # 6. Calculate previous matchup differentials
    print("Computing previous matchup differences...")
    # Create a unique identifier for each team matchup (sorted team names to handle home/away)
    df['team_pair'] = df.apply(lambda row: '_'.join(sorted([row['home_team'], row['away_team']])), axis=1)
    
    # For each game, find previous matchups between the same teams
    matchup_diffs = {}
    for team_pair in df['team_pair'].unique():
        # Get all games between these teams, sorted by date
        pair_games = df[df['team_pair'] == team_pair].sort_values('game_date')
        
        # Calculate point differential from home team perspective for each game
        pair_games['matchup_diff'] = pair_games.apply(
            lambda row: row['home_score'] - row['away_score'] if row['home_team'] == pair_games.iloc[0]['home_team'] 
            else row['away_score'] - row['home_score'], axis=1
        )
        
        # For each game, calculate average of previous matchups
        for i in range(len(pair_games)):
            game_id = pair_games.iloc[i]['game_id']
            if i == 0:  # First matchup, no history
                matchup_diffs[game_id] = 0
            else:  # Use average of previous matchups
                prev_diffs = pair_games.iloc[:i]['matchup_diff'].values
                matchup_diffs[game_id] = prev_diffs.mean()
    
    # Add the calculated previous matchup differentials to the dataframe
    df['prev_matchup_diff'] = df['game_id'].map(matchup_diffs).fillna(0)
    
    print("All required features calculated successfully.")
    return df

# --- Define quarter-specific model creation and ensemble functions ---

def create_quarter_specific_models(df):
    """
    Build separate prediction models for each quarter stage.
    Returns a dictionary of models for Q1, Q2, Q3, and Q4.
    """
    models = {}
    
    # Q1 model: using pre-game features only
    q1_features = ['rest_days_home', 'rest_days_away', 'is_back_to_back_home', 
                  'is_back_to_back_away', 'prev_matchup_diff']
    
    if all(col in df.columns for col in q1_features):
        X_q1 = df[q1_features]
        y_q1 = df['home_q1']
        from sklearn.linear_model import LinearRegression
        models['q1_model'] = LinearRegression().fit(X_q1, y_q1)
    
    # Q2 model: using Q1 data plus rest advantage
    q2_features = ['home_q1', 'away_q1', 'rest_advantage', 'prev_matchup_diff']
    
    if all(col in df.columns for col in q2_features):
        X_q2 = df[q2_features]
        y_q2 = df['home_q2']
        from sklearn.linear_model import LinearRegression
        models['q2_model'] = LinearRegression().fit(X_q2, y_q2)
    
    # Q3 model: using first half data
    q3_features = ['home_q1', 'home_q2', 'away_q1', 'away_q2', 
                  'q1_to_q2_momentum', 'first_half_diff', 'rest_advantage']
    
    if all(col in df.columns for col in q3_features):
        X_q3 = df[q3_features].copy()  # Create a copy to preserve feature names
        y_q3 = df['home_q3']
        from sklearn.linear_model import LinearRegression
        models['q3_model'] = LinearRegression().fit(X_q3, y_q3)
    
    # Q4 model: using all previous quarters
    q4_features = ['home_q1', 'home_q2', 'home_q3', 'away_q1', 'away_q2', 'away_q3', 
                  'q1_to_q2_momentum', 'q2_to_q3_momentum', 'pre_q4_diff']
    
    if all(col in df.columns for col in q4_features):
        X_q4 = df[q4_features].copy()  # Create a copy to preserve feature names
        y_q4 = df['home_q4']
        from sklearn.linear_model import LinearRegression
        models['q4_model'] = LinearRegression().fit(X_q4, y_q4)
    
    return models

def ensemble_quarter_predictions(main_prediction, quarter_sum_prediction, current_quarter):
    """
    Combine the main model's prediction with quarter-specific predictions.
    This uses a dynamic weighting that shifts as the game progresses.
    
    Args:
        main_prediction: Full-game score from the main model
        quarter_sum_prediction: Sum of played + predicted quarters
        current_quarter: Current quarter of the game (1-4)
        
    Returns:
        tuple: (ensemble_prediction, confidence, weight_main, weight_quarter)
    """
    # Define weights and confidence by quarter
    if current_quarter == 1:
        weight_main, weight_quarter, confidence = 0.3, 0.7, 0.40
    elif current_quarter == 2:
        weight_main, weight_quarter, confidence = 0.6, 0.4, 0.60
    elif current_quarter == 3:
        weight_main, weight_quarter, confidence = 0.8, 0.2, 0.80
    else:
        weight_main, weight_quarter, confidence = 1.0, 0.0, 0.95
    
    # Blend predictions based on weights
    ensemble_prediction = weight_main * main_prediction + weight_quarter * quarter_sum_prediction
    
    return ensemble_prediction, confidence, weight_main, weight_quarter

print("\n=== Starting Quarter-Specific Ensemble Comparison ===\n")

# --------------------------
# 1. Load Enhanced Features with ALL required columns
# --------------------------
try:
    print("Checking if features_df already exists...")
    if 'features_df' in globals() and isinstance(features_df, pd.DataFrame) and not features_df.empty:
        quarter_features_df = features_df.copy()
        print(f"Using existing features_df with {features_df.shape[0]} rows and {features_df.shape[1]} columns")
    else:
        print("No existing features_df found, generating new data...")
        quarter_features_df = generate_all_quarter_features(sample_size=1000, skip_rest_calc=False)
    
    print(f"Enhanced features loaded: {quarter_features_df.shape[0]} rows with {quarter_features_df.shape[1]} columns")
    
    # Verify all required columns are present
    required_columns = [
        'home_q1', 'home_q2', 'home_q3', 'home_q4', 
        'away_q1', 'away_q2', 'away_q3', 'away_q4',
        'rest_days_home', 'rest_days_away', 'rest_advantage',
        'is_back_to_back_home', 'is_back_to_back_away',
        'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum',
        'first_half_diff', 'pre_q4_diff', 'prev_matchup_diff'
    ]
    
    missing_columns = [col for col in required_columns if col not in quarter_features_df.columns]
    if missing_columns:
        print(f"Warning: Missing required columns: {missing_columns}")
    else:
        print("All required columns are present in the dataset.")
    
    # Make features_df available globally
    features_df = quarter_features_df
except Exception as e:
    print(f"Error loading features: {e}")
    import traceback
    traceback.print_exc()

# --------------------------
# 2. Create Quarter-Specific Models Using the Enhanced Data
# --------------------------
try:
    print("\nCreating quarter-specific models...")
    qs_models = create_quarter_specific_models(features_df)
    print("Quarter-specific models created:")
    for model_name in qs_models:
        print(f"  - {model_name}")
except Exception as e:
    print(f"Error creating quarter-specific models: {e}")
    import traceback
    traceback.print_exc()

# --------------------------
# 3. Define a Sample Game Data (Simulated In-Game Scenario)
# --------------------------
print("\nDefining sample game data (halftime scenario)...")
sample_game_data = {
    'home_q1': 28,
    'home_q2': 27,
    'home_q3': 0,   # not played yet
    'home_q4': 0,
    'away_q1': 26,
    'away_q2': 25,
    'away_q3': 0,
    'away_q4': 0,
    'score_ratio': 28 / (28 + 26),
    'rolling_home_score': 105,
    'rolling_away_score': 104,
    'prev_matchup_diff': 3,
    'rest_days_home': 2,
    'rest_days_away': 2,
    'rest_advantage': 0,
    'is_back_to_back_home': 0,
    'is_back_to_back_away': 0
}

# Calculate additional required features
sample_game_data['q1_to_q2_momentum'] = (sample_game_data['home_q2'] - sample_game_data['home_q1']) - (sample_game_data['away_q2'] - sample_game_data['away_q1'])
sample_game_data['first_half_diff'] = (sample_game_data['home_q1'] + sample_game_data['home_q2']) - (sample_game_data['away_q1'] + sample_game_data['away_q2'])
sample_game_data['q2_to_q3_momentum'] = 0  # Will be calculated after Q3 prediction
sample_game_data['pre_q4_diff'] = 0  # Will be calculated after Q3 prediction

current_quarter = 2  # We're at halftime

# Display current game state
print("\nCurrent Game State (After Q2):")
print(f"  Home Team: Q1={sample_game_data['home_q1']}, Q2={sample_game_data['home_q2']}, Total={sample_game_data['home_q1'] + sample_game_data['home_q2']}")
print(f"  Away Team: Q1={sample_game_data['away_q1']}, Q2={sample_game_data['away_q2']}, Total={sample_game_data['away_q1'] + sample_game_data['away_q2']}")
print(f"  Lead: {'Home' if sample_game_data['first_half_diff'] > 0 else 'Away'} by {abs(sample_game_data['first_half_diff'])} points")
print(f"  Quarter-to-Quarter Momentum: {sample_game_data['q1_to_q2_momentum']:.1f}")

# --------------------------
# 4. Main Model Prediction (From your existing model)
# --------------------------
main_model_prediction = 110.0
print(f"\nMain Game Prediction Model: {main_model_prediction:.1f} points")

# --------------------------
# 5. Generate Quarter-Specific Model Predictions
# --------------------------
print("\nQuarter-Specific Predictions:")
quarter_model_predictions = {}

# Q1 and Q2 are already played, so we don't need predictions
print(f"  - Q1: {sample_game_data['home_q1']:.1f} points (Actual)")
print(f"  - Q2: {sample_game_data['home_q2']:.1f} points (Actual)")

# Only predict quarters that haven't been played yet
if current_quarter <= 2 and 'q3_model' in qs_models:
    # Create a DataFrame for Q3 prediction to preserve feature names
    q3_features = ['home_q1', 'home_q2', 'away_q1', 'away_q2', 
                  'q1_to_q2_momentum', 'first_half_diff', 'rest_advantage']
    
    X_q3_pred = pd.DataFrame([{
        'home_q1': sample_game_data['home_q1'],
        'home_q2': sample_game_data['home_q2'],
        'away_q1': sample_game_data['away_q1'],
        'away_q2': sample_game_data['away_q2'],
        'q1_to_q2_momentum': sample_game_data['q1_to_q2_momentum'],
        'first_half_diff': sample_game_data['first_half_diff'],
        'rest_advantage': sample_game_data['rest_advantage']
    }])
    
    q3_pred = qs_models['q3_model'].predict(X_q3_pred)[0]
    quarter_model_predictions['q3'] = q3_pred
    print(f"  - Q3: {q3_pred:.1f} points (Predicted)")
    
    # Add the predicted Q3 to the sample game data for Q4 prediction
    sample_game_data['home_q3'] = q3_pred
    
    # Estimate away Q3 based on previous quarter ratios
    current_ratio = sample_game_data['home_q2'] / sample_game_data['away_q2']
    sample_game_data['away_q3'] = q3_pred / current_ratio
    
    # Calculate q2_to_q3_momentum
    sample_game_data['q2_to_q3_momentum'] = (sample_game_data['home_q3'] - sample_game_data['home_q2']) - (sample_game_data['away_q3'] - sample_game_data['away_q2'])
    
    # Calculate pre_q4_diff
    sample_game_data['pre_q4_diff'] = (sample_game_data['home_q1'] + sample_game_data['home_q2'] + sample_game_data['home_q3']) - (sample_game_data['away_q1'] + sample_game_data['away_q2'] + sample_game_data['away_q3'])

if current_quarter <= 3 and 'q4_model' in qs_models:
    # Create a DataFrame for Q4 prediction to preserve feature names
    q4_features = ['home_q1', 'home_q2', 'home_q3', 'away_q1', 'away_q2', 'away_q3', 
                  'q1_to_q2_momentum', 'q2_to_q3_momentum', 'pre_q4_diff']
    
    X_q4_pred = pd.DataFrame([{
        'home_q1': sample_game_data['home_q1'],
        'home_q2': sample_game_data['home_q2'],
        'home_q3': sample_game_data['home_q3'],
        'away_q1': sample_game_data['away_q1'],
        'away_q2': sample_game_data['away_q2'],
        'away_q3': sample_game_data['away_q3'],
        'q1_to_q2_momentum': sample_game_data['q1_to_q2_momentum'],
        'q2_to_q3_momentum': sample_game_data['q2_to_q3_momentum'],
        'pre_q4_diff': sample_game_data['pre_q4_diff']
    }])
    
    q4_pred = qs_models['q4_model'].predict(X_q4_pred)[0]
    quarter_model_predictions['q4'] = q4_pred
    print(f"  - Q4: {q4_pred:.1f} points (Predicted)")

# Calculate total predicted score from the quarters
played_quarters_sum = sample_game_data['home_q1'] + sample_game_data['home_q2']
predicted_quarters_sum = sum(quarter_model_predictions.values())
quarter_total_prediction = played_quarters_sum + predicted_quarters_sum
print(f"\nQuarter-Sum Prediction: {played_quarters_sum:.1f} (played) + {predicted_quarters_sum:.1f} (predicted) = {quarter_total_prediction:.1f} total points")

# --------------------------
# 6. Generate Ensemble Prediction USING QUARTER SUM
# --------------------------
ensemble_prediction, ensemble_confidence, weight_main, weight_quarter = ensemble_quarter_predictions(
    main_model_prediction, quarter_total_prediction, current_quarter
)
print(f"\nEnsemble Final Prediction: {ensemble_prediction:.1f} points (Confidence: {ensemble_confidence*100:.0f}%)")

# --------------------------
# 7. Compare Predictions
# --------------------------
print("\nFinal Comparison:")
print(f"  - Main Model Prediction:  {main_model_prediction:.1f} points")
print(f"  - Quarter Sum Prediction: {quarter_total_prediction:.1f} points")
print(f"  - Ensemble Prediction:    {ensemble_prediction:.1f} points (Confidence: {ensemble_confidence*100:.0f}%)")
print(f"\nEnsemble Calculation: ({weight_main:.1f} × {main_model_prediction:.1f}) + ({weight_quarter:.1f} × {quarter_total_prediction:.1f}) = {ensemble_prediction:.1f}")
print("\nNote: As the game progresses, the ensemble will give more weight to the main model's prediction.")

# --------------------------
# 8. Create Confidence Visualization
# --------------------------
def create_prediction_confidence_viz(main_prediction, quarter_prediction, current_quarter):
    """Creates a visualization showing prediction confidence across quarters"""
    
    # Define quarter weights and confidence for all quarters
    quarter_weights = {
        1: {'main': 0.3, 'quarter': 0.7, 'confidence': 0.40},
        2: {'main': 0.6, 'quarter': 0.4, 'confidence': 0.60},
        3: {'main': 0.8, 'quarter': 0.2, 'confidence': 0.80},
        4: {'main': 1.0, 'quarter': 0.0, 'confidence': 0.95},
    }
    
    # Calculate predictions and ranges for each quarter
    predictions = []
    ranges = []
    
    for q in range(1, 5):
        w_main = quarter_weights[q]['main']
        w_quarter = quarter_weights[q]['quarter']
        conf = quarter_weights[q]['confidence']
        
        # Calculate blended prediction
        pred = w_main * main_prediction + w_quarter * quarter_prediction
        
        # Calculate range based on confidence
        # As confidence increases, range decreases
        range_val = main_prediction * (1 - conf)
        
        predictions.append(pred)
        ranges.append(range_val)
    
    # Create figure
    plt.figure(figsize=(12, 6))
    
    # Create quarter labels with indication of current quarter
    quarter_labels = ['Q1', 'Q2', 'Q3', 'Q4']
    colors = ['lightgray' if q <= current_quarter else 'white' for q in range(1, 5)]
    
    # Plot prediction with confidence intervals
    for i, (pred, range_val, color) in enumerate(zip(predictions, ranges, colors)):
        q = i + 1
        
        # Draw rectangle for confidence interval
        plt.bar(i, height=range_val*2, bottom=pred-range_val, 
                width=0.6, alpha=0.3, color='lightblue', edgecolor='blue')
        
        # Draw point for prediction
        plt.plot(i, pred, 'o', markersize=8, color='blue')
        
        # Add number labels
        plt.text(i, pred, f'{pred:.1f}', ha='center', va='bottom', fontweight='bold')
        
        # Add confidence percentage
        conf_pct = quarter_weights[q]['confidence'] * 100
        plt.text(i, pred-range_val-5, f'{conf_pct:.0f}% conf.', ha='center', va='top')
    
    # Highlight current quarter
    if current_quarter <= 4:
        plt.axvspan(current_quarter-1-0.4, current_quarter-1+0.4, 
                   alpha=0.2, color='green', label=f'Current (Q{current_quarter})')
    
    # Add reference lines for main and quarter predictions
    plt.axhline(y=main_prediction, linestyle='--', color='red', alpha=0.7, 
               label=f'Main Model: {main_prediction:.1f}')
    plt.axhline(y=quarter_prediction, linestyle='--', color='blue', alpha=0.7, 
               label=f'Quarter Sum: {quarter_prediction:.1f}')
    
    # Labels and title
    plt.title('Prediction Confidence by Quarter', fontsize=14)
    plt.xlabel('Game Quarter', fontsize=12)
    plt.ylabel('Predicted Final Score', fontsize=12)
    plt.xticks(range(4), quarter_labels)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Add legend
    plt.legend(loc='upper right')
    
    # Adjust y-axis to show reasonable range
    min_y = min(quarter_prediction, main_prediction) - max(ranges) - 10
    max_y = max(quarter_prediction, main_prediction) + max(ranges) + 10
    plt.ylim(min_y, max_y)
    
    plt.tight_layout()
    plt.show()

# Generate the visualization
print("\nGenerating prediction confidence visualization...")
create_prediction_confidence_viz(main_model_prediction, quarter_total_prediction, current_quarter)

def integrate_adaptive_ensemble_into_pipeline(game_data, main_model_prediction, quarter_models_predictions):
    """
    Integration point to add the adaptive ensemble to the main prediction pipeline.
    
    Args:
        game_data: Current game state data
        main_model_prediction: Prediction from main model
        quarter_models_predictions: Dict of quarter-specific predictions
        
    Returns:
        dict: Enhanced prediction results with ensemble details
    """
    # Create ensemble
    ensemble = AdaptiveEnsembleFramework()
    
    # Extract game context information
    current_quarter = int(game_data.get('current_quarter', 0))
    
    # Get quarter-specific prediction
    quarter_prediction = quarter_models_predictions.get(f'q{current_quarter}', 
                                                      main_model_prediction * 0.95)
    
    # Prepare game context
    context = {
        'score_differential': abs(float(game_data.get('home_score', 0)) - 
                                float(game_data.get('away_score', 0))),
        'momentum': float(game_data.get('cumulative_momentum', 0)),
        'time_remaining_in_quarter': float(game_data.get('time_remaining_in_quarter', 6)),
        'game_id': game_data.get('game_id', 'unknown'),
        'home_team': game_data.get('home_team', 'Home'),
        'away_team': game_data.get('away_team', 'Away')
    }
    
    # Get ensemble prediction
    prediction, weights, confidence = ensemble.predict(
        main_model_prediction,
        quarter_prediction,
        current_quarter,
        context
    )
    
    # Create prediction package
    prediction_package = {
        'home_final': prediction,
        'away_final': game_data.get('away_score', 0) + (prediction - game_data.get('home_score', 0)),
        'main_prediction': main_model_prediction,
        'quarter_prediction': quarter_prediction,
        'strategy': weights['strategy'],
        'weights': weights,
        'confidence': confidence,
        'win_probability': 1/(1 + np.exp(-0.1 * (prediction - game_data.get('away_score', 0))))
    }
    
    return prediction_package

print("\nCell 4C execution completed successfully!")

In [ ]:
# Cell 4C-2: Enhanced Ensemble Weighting with Uncertainty Integration

def enhanced_ensemble_weighting(main_prediction, main_uncertainty, 
                              quarter_prediction, quarter_uncertainty,
                              current_quarter, game_context):
    """
    Weight models based on their relative uncertainty and game context
    
    Args:
        main_prediction: Prediction from main model
        main_uncertainty: Uncertainty estimate from main model
        quarter_prediction: Quarter-based prediction
        quarter_uncertainty: Uncertainty in quarter-based prediction
        current_quarter: Current game quarter (0-4)
        game_context: Dict with score_diff, momentum, etc.
    """
    # Base weights by quarter
    base_weights = {
        0: {'main': 0.3, 'quarter': 0.7},
        1: {'main': 0.4, 'quarter': 0.6},
        2: {'main': 0.6, 'quarter': 0.4},
        3: {'main': 0.8, 'quarter': 0.2},
        4: {'main': 0.95, 'quarter': 0.05}
    }
    
    # Adjust weights based on relative uncertainty
    uncertainty_ratio = quarter_uncertainty / main_uncertainty if main_uncertainty > 0 else 1
    
    # When quarter model is much more certain, give it more weight (clipped)
    uncertainty_adjustment = max(-0.2, min(0.2, (1 - uncertainty_ratio) * 0.3))
    
    # Apply context-specific adjustments
    score_diff = abs(game_context.get('score_differential', 0))
    momentum = game_context.get('momentum', 0)
    
    # In close games, trust the more certain model more
    context_adjustment = 0
    if score_diff < 5:  # Close game
        # If there's strong momentum, give the quarter model slightly more weight
        if abs(momentum) > 0.5:
            momentum_dir = np.sign(momentum)
            context_adjustment = -0.05  # Slightly more weight to quarter model
    
    # Calculate final weights
    main_weight = base_weights[current_quarter]['main'] - uncertainty_adjustment + context_adjustment
    main_weight = max(0.1, min(0.95, main_weight))  # Ensure weights stay reasonable
    quarter_weight = 1 - main_weight
    
    # Calculate ensemble prediction
    ensemble_prediction = (main_prediction * main_weight) + (quarter_prediction * quarter_weight)
    
    return ensemble_prediction, main_weight, quarter_weight

# Test the enhanced weighting function
test_context = {'score_differential': 4, 'momentum': 0.3}
test_main_pred = 110.0
test_quarter_pred = 105.0
test_main_uncertainty = 8.0
test_quarter_uncertainty = 5.0  # Quarter model more certain in this case

for q in range(5):
    ensemble_pred, main_w, quarter_w = enhanced_ensemble_weighting(
        test_main_pred, test_main_uncertainty,
        test_quarter_pred, test_quarter_uncertainty,
        q, test_context
    )
    print(f"Quarter {q}: Ensemble={ensemble_pred:.1f} (Main weight: {main_w:.2f}, Quarter weight: {quarter_w:.2f})")

def enhanced_ensemble_weighting_v2(main_prediction, quarter_prediction, current_quarter, 
                                 game_context=None):
    """
    Enhanced weighting function with continuous quarter transitions and better context sensitivity.
    
    Args:
        main_prediction: Prediction from main model
        quarter_prediction: Prediction from quarter-specific model
        current_quarter: Current quarter (0-4)
        game_context: Dictionary with game context information
        
    Returns:
        tuple: (ensemble_prediction, confidence, main_weight, quarter_weight)
    """
    # Initialize context
    if game_context is None:
        game_context = {}
    
    # Extract context variables
    score_diff = abs(game_context.get('score_differential', 0))
    momentum = game_context.get('momentum', 0)
    time_remaining = game_context.get('time_remaining_in_quarter', 6)
    
    # Calculate continuous quarter for smoother transitions
    quarter_progress = 1.0 - (time_remaining / 12.0)
    continuous_quarter = current_quarter + quarter_progress
    
    # Base weights that increase with quarter number using sigmoid for smooth transition
    transition_steepness = 8.0
    progress = continuous_quarter / 4.0  # 0-1 scale
    adjusted_progress = (progress - 0.5) * transition_steepness
    sigmoid_factor = 1.0 / (1.0 + np.exp(-adjusted_progress))
    
    base_main_weight = 0.3 + (0.65 * sigmoid_factor)
    
    # Context adjustments
    # Close games benefit more from quarter-specific models
    if score_diff < 5:
        score_adjustment = -0.10  # Reduce main weight (increase quarter weight)
    elif score_diff < 10:
        score_adjustment = -0.05
    elif score_diff > 20:
        score_adjustment = 0.10  # Increase main weight in blowouts
    elif score_diff > 15:
        score_adjustment = 0.05
    else:
        score_adjustment = 0.0
        
    # Strong momentum gives more weight to quarter models
    momentum_magnitude = abs(momentum)
    momentum_adjustment = -0.15 * momentum_magnitude
    
    # Apply adjustments with game progress factor
    game_progress = min(1.0, continuous_quarter / 4.0)
    progress_factor = 0.5 + (0.5 * game_progress)  # 0.5 to 1.0
    
    total_adjustment = (score_adjustment + momentum_adjustment) * progress_factor
    main_weight = max(0.1, min(0.95, base_main_weight + total_adjustment))
    quarter_weight = 1.0 - main_weight
    
    # Calculate ensemble prediction
    if isinstance(main_prediction, (list, np.ndarray)) and isinstance(quarter_prediction, (list, np.ndarray)):
        ensemble_prediction = np.array(main_prediction) * main_weight + np.array(quarter_prediction) * quarter_weight
    else:
        ensemble_prediction = main_prediction * main_weight + quarter_prediction * quarter_weight
    
    # Calculate confidence based on quarter and weights
    quarter_boost = 0.1 * current_quarter  # 0.0 for Q0, 0.4 for Q4
    weight_decisiveness = abs(main_weight - 0.5) * 2  # 0-1 based on how far from 50/50
    confidence = min(0.95, 0.5 + quarter_boost + (weight_decisiveness * 0.2))
    
    return ensemble_prediction, confidence, main_weight, quarter_weight    

In [ ]:
# Cell 4C-3: Adaptive Ensemble Strategy Selection

def adaptive_strategy_selection(game_context):
    """
    Select best weighting strategy based on game context
    
    Args:
        game_context: Dict with game state information
        
    Returns:
        str: Name of the selected strategy
        dict: Weight parameters for the selected strategy
    """
    # Extract context variables
    score_diff = abs(game_context.get('score_differential', 0))
    momentum = abs(game_context.get('momentum', 0))
    quarter = game_context.get('current_quarter', 0)
    margin_trend = game_context.get('margin_trend', 0)  # Positive if lead is growing
    
    # Define strategies
    strategies = {
        'balanced': {
            'description': 'Default balanced weights',
            'q0': {'main': 0.3, 'quarter': 0.7},
            'q1': {'main': 0.4, 'quarter': 0.6},
            'q2': {'main': 0.6, 'quarter': 0.4},
            'q3': {'main': 0.8, 'quarter': 0.2},
            'q4': {'main': 0.95, 'quarter': 0.05}
        },
        'momentum_driven': {
            'description': 'Increased weight to quarter models when momentum is strong',
            'q0': {'main': 0.3, 'quarter': 0.7},
            'q1': {'main': 0.35, 'quarter': 0.65},
            'q2': {'main': 0.5, 'quarter': 0.5},
            'q3': {'main': 0.7, 'quarter': 0.3},
            'q4': {'main': 0.9, 'quarter': 0.1}
        },
        'conservative': {
            'description': 'Heavier weighting toward main model',
            'q0': {'main': 0.4, 'quarter': 0.6},
            'q1': {'main': 0.5, 'quarter': 0.5},
            'q2': {'main': 0.7, 'quarter': 0.3},
            'q3': {'main': 0.85, 'quarter': 0.15},
            'q4': {'main': 0.98, 'quarter': 0.02}
        },
        'quarter_focused': {
            'description': 'Emphasizes quarter-specific models more',
            'q0': {'main': 0.2, 'quarter': 0.8},
            'q1': {'main': 0.3, 'quarter': 0.7},
            'q2': {'main': 0.5, 'quarter': 0.5},
            'q3': {'main': 0.7, 'quarter': 0.3},
            'q4': {'main': 0.85, 'quarter': 0.15}
        }
    }
    
    # Decision logic for strategy selection
    if score_diff < 5:  # Close game
        if momentum > 0.5:  # Strong momentum
            selected = 'momentum_driven'
        else:
            selected = 'balanced'
    elif quarter >= 3:  # Late game
        if margin_trend > 0:  # Lead is growing
            selected = 'conservative'  # Trust main model more
        else:
            selected = 'quarter_focused'  # Pay more attention to quarter dynamics
    else:
        selected = 'balanced'
    
    # Get weights for current quarter
    q_key = f'q{quarter}'
    weights = strategies[selected].get(q_key, strategies[selected]['q0'])
    
    return selected, weights

def integrated_ensemble_prediction(main_model_prediction, quarter_models_predictions, 
                                  current_quarter, game_data):
    """
    Integrated ensemble prediction using the most appropriate strategy for the game state.
    
    Args:
        main_model_prediction: Prediction from main model
        quarter_models_predictions: Dict of quarter-specific predictions
        current_quarter: Current quarter (0-4)
        game_data: Current game state data
        
    Returns:
        dict: Prediction results with ensemble details
    """
    # Extract quarter-specific prediction for current quarter
    quarter_prediction = quarter_models_predictions.get(f'q{current_quarter}', 
                                                      main_model_prediction * 0.95)
    
    # Create game context
    game_context = {
        'score_differential': abs(float(game_data.get('home_score', 0)) - 
                                float(game_data.get('away_score', 0))),
        'momentum': float(game_data.get('cumulative_momentum', 0)),
        'time_remaining_in_quarter': float(game_data.get('time_remaining_in_quarter', 6)),
        'margin_trend': float(game_data.get('margin_trend', 0))
    }
    
    # Determine best strategy for this context
    strategy_name = adaptive_strategy_selection(game_context)
    
    # Use the AdaptiveEnsembleFramework from Cell 4C-7
    ensemble = AdaptiveEnsembleFramework()
    
    # Get prediction with weights and confidence
    ensemble_pred, weights, confidence = ensemble.predict(
        main_model_prediction,
        quarter_prediction,
        current_quarter,
        game_context,
        strategy_name
    )
    
    # Compile results
    results = {
        'ensemble_prediction': ensemble_pred,
        'main_prediction': main_model_prediction,
        'quarter_prediction': quarter_prediction,
        'strategy': weights['strategy'],
        'main_weight': weights['main'],
        'quarter_weight': weights['quarter'],
        'confidence': confidence,
        'context': game_context
    }
    
    return results

def run_ab_test_on_weighting_strategies(validation_games, strategies=None):
    """
    Evaluate different weighting strategies on historical games
    
    Args:
        validation_games: DataFrame of historical games with actual final scores
        strategies: Dict of weighting strategies to test (if None, use defaults)
    
    Returns:
        DataFrame with comparison results
    """
    if strategies is None:
        # Use default strategies from adaptive_strategy_selection
        temp_context = {'score_differential': 0, 'momentum': 0, 'current_quarter': 0}
        strategies = adaptive_strategy_selection(temp_context)[0]
    
    results = []
    
    for _, game in validation_games.iterrows():
        game_id = game['game_id']
        current_quarter = game['current_quarter']
        
        # Get actual final scores
        actual_home = game['actual_home_final']
        actual_away = game['actual_away_final']
        
        # Context for strategy selection
        context = {
            'score_differential': game.get('score_differential', 0),
            'momentum': game.get('momentum', 0),
            'current_quarter': current_quarter,
            'margin_trend': game.get('margin_trend', 0)
        }
        
        # Get predictions from main and quarter models
        main_pred_home = game.get('main_prediction_home', 0)
        quarter_pred_home = game.get('quarter_prediction_home', 0)
        
        # Test each strategy
        for strategy_name, strategy_config in strategies.items():
            # Get weights for this quarter
            q_key = f'q{current_quarter}'
            weights = strategy_config.get(q_key, strategy_config['q0'])
            
            # Calculate ensemble prediction
            ensemble_home = (main_pred_home * weights['main'] + 
                            quarter_pred_home * weights['quarter'])
            
            # Calculate error
            error = abs(ensemble_home - actual_home)
            
            results.append({
                'game_id': game_id,
                'quarter': current_quarter,
                'strategy': strategy_name,
                'main_weight': weights['main'],
                'quarter_weight': weights['quarter'],
                'ensemble_prediction': ensemble_home,
                'actual_score': actual_home,
                'absolute_error': error
            })
    
    # Process results
    results_df = pd.DataFrame(results)
    
    # Calculate aggregate metrics
    summary = results_df.groupby(['quarter', 'strategy']).agg({
        'absolute_error': ['mean', 'std', 'count'],
        'main_weight': 'mean',
        'quarter_weight': 'mean'
    }).reset_index()
    
    # Flatten column names
    summary.columns = ['_'.join(col).strip('_') for col in summary.columns.values]
    
    return results_df, summary

# Test with sample data
def test_adaptive_strategy():
    # Generate sample game contexts
    contexts = [
        {'score_differential': 2, 'momentum': 0.7, 'current_quarter': 2, 'margin_trend': 0},  # Close game, high momentum
        {'score_differential': 15, 'momentum': 0.3, 'current_quarter': 3, 'margin_trend': 5},  # Big lead growing late
        {'score_differential': 8, 'momentum': 0.2, 'current_quarter': 3, 'margin_trend': -3},  # Lead shrinking late
        {'score_differential': 4, 'momentum': 0.1, 'current_quarter': 1, 'margin_trend': 0}   # Close early game
    ]
    
    # Test strategy selection
    print("Testing Adaptive Strategy Selection:")
    results = []
    for i, context in enumerate(contexts):
        strategy, weights = adaptive_strategy_selection(context)
        results.append({
            'scenario': i+1,
            'score_diff': context['score_differential'],
            'momentum': context['momentum'],
            'quarter': context['current_quarter'],
            'margin_trend': context['margin_trend'],
            'selected_strategy': strategy,
            'main_weight': weights['main'],
            'quarter_weight': weights['quarter']
        })
    
    results_df = pd.DataFrame(results)
    display(results_df)
    
    # Visualize strategy selection
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.scatter(results_df['score_diff'], results_df['momentum'], 
                c=results_df['main_weight'], cmap='viridis', 
                s=100, alpha=0.7)
    plt.colorbar(label='Main Model Weight')
    plt.xlabel('Score Differential')
    plt.ylabel('Momentum')
    plt.title('Strategy Selection by Game Context')
    
    plt.subplot(1, 2, 2)
    plt.scatter(results_df['quarter'], results_df['score_diff'],
                c=results_df['main_weight'], cmap='viridis',
                s=100, alpha=0.7)
    plt.colorbar(label='Main Model Weight')
    plt.xlabel('Quarter')
    plt.ylabel('Score Differential')
    plt.title('Strategy Selection by Game Progress')
    
    plt.tight_layout()
    plt.show()
    
    return results_df

# Run the test
test_results = test_adaptive_strategy()

In [ ]:
# Cell 4C-4: Context-Aware Weighting & Benchmark Tools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

def sigmoid_weight_transition(game_progress, smoothness=10):
    """
    Create smooth sigmoid-based transition function for model weights.
    
    Args:
        game_progress: Value between 0 and 1 indicating game progress
        smoothness: Controls how gradual the transition is (higher = sharper)
        
    Returns:
        float: Weight value between 0 and 1
    """
    # Sigmoid function centered at 0.5 (mid-game)
    return 1.0 / (1.0 + np.exp(-smoothness * (game_progress - 0.5)))

def context_aware_ensemble_weighting(main_prediction, quarter_prediction, game_context):
    """
    Enhanced ensemble weighting function that considers game context.
    
    Args:
        main_prediction: Prediction from main model
        quarter_prediction: Prediction from quarter-specific model
        game_context: Dict with game state information including:
            - current_quarter: Current quarter (0-4)
            - score_differential: Current score difference (home - away)
            - momentum: Current momentum value (-1 to 1)
            - time_remaining: Minutes remaining in game
            
    Returns:
        tuple: (weighted_prediction, main_weight, quarter_weight, reasoning)
    """
    # Extract context variables
    current_quarter = game_context.get('current_quarter', 0)
    score_diff = abs(game_context.get('score_differential', 0))
    momentum = game_context.get('momentum', 0)
    time_remaining = game_context.get('time_remaining', 48 - current_quarter * 12)
    
    # Calculate game progress as a fraction [0,1]
    total_time = 48.0  # Full game time in minutes
    game_progress = min(1.0, max(0.0, (total_time - time_remaining) / total_time))
    
    # Base weights that evolve with game progress - using sigmoid for smooth transition
    progress_factor = sigmoid_weight_transition(game_progress)
    base_main_weight = 0.3 + (0.65 * progress_factor)
    
    # Adjust for score differential - close games benefit more from quarter-specific models
    close_game_factor = max(0.0, 1.0 - (score_diff / 20.0))  # 1.0 for tie game, decreases as diff increases
    diff_adjustment = 0.1 * close_game_factor  # Max 0.1 adjustment
    
    # Adjust for momentum - strong momentum benefits quarter model
    momentum_magnitude = abs(momentum)
    momentum_adjustment = 0.1 * momentum_magnitude  # Max 0.1 adjustment
    
    # Apply context adjustments
    # Negative adjustment means more weight to quarter model
    main_weight = base_main_weight - diff_adjustment - momentum_adjustment
    
    # Ensure weights are valid
    main_weight = max(0.1, min(0.95, main_weight))
    quarter_weight = 1.0 - main_weight
    
    # Calculate final prediction
    weighted_prediction = (main_prediction * main_weight) + (quarter_prediction * quarter_weight)
    
    # Generate explanation for this weighting
    reasoning = {
        'game_progress': f"{game_progress:.2f}",
        'base_weight': f"{base_main_weight:.2f}",
        'close_game_adjustment': f"{-diff_adjustment:.2f}",
        'momentum_adjustment': f"{-momentum_adjustment:.2f}",
        'final_weights': f"Main: {main_weight:.2f}, Quarter: {quarter_weight:.2f}"
    }
    
    return weighted_prediction, main_weight, quarter_weight, reasoning

def ensemble_strategy_benchmarking(validation_games, strategies=None, ensemble_methods=None):
    """
    Comprehensive benchmarking of different ensemble strategies on historical games.
    
    Args:
        validation_games: DataFrame with validation game data
        strategies: List of strategies to test (or None for defaults)
        ensemble_methods: Dict of ensemble method functions to compare
        
    Returns:
        DataFrame with benchmark results
    """
    # Import from integrated framework if available
    try:
        # This assumes the AdaptiveEnsembleFramework is defined in the notebook
        from __main__ import dynamic_ensemble_predictions, AdaptiveEnsembleFramework
        has_integrated_framework = True
    except ImportError:
        has_integrated_framework = False
    
    if ensemble_methods is None:
        # Define methods to benchmark
        ensemble_methods = {
            'simple_context': lambda main, quarter, q, context: context_aware_ensemble_weighting(
                main, quarter, context)[0]
        }
        
        # Add integrated framework methods if available
        if has_integrated_framework:
            ensemble_methods.update({
                'adaptive': lambda main, quarter, q, context: dynamic_ensemble_predictions(
                    main, quarter, q,
                    context.get('score_differential', 0),
                    context.get('momentum', 0),
                    context.get('time_remaining_in_quarter', 6),
                    None)[0],
                'balanced_strategy': lambda main, quarter, q, context: dynamic_ensemble_predictions(
                    main, quarter, q,
                    context.get('score_differential', 0),
                    context.get('momentum', 0),
                    context.get('time_remaining_in_quarter', 6),
                    'balanced')[0],
                'momentum_strategy': lambda main, quarter, q, context: dynamic_ensemble_predictions(
                    main, quarter, q,
                    context.get('score_differential', 0),
                    context.get('momentum', 0),
                    context.get('time_remaining_in_quarter', 6),
                    'momentum_driven')[0]
            })
    
    results = []
    
    for _, game in validation_games.iterrows():
        actual_home_score = game['actual_home_final']
        actual_away_score = game['actual_away_final']
        
        for quarter in range(5):  # 0-4
            # Skip if no quarter data
            if quarter > 0 and (pd.isna(game.get(f'home_q{quarter}', None)) or 
                              game.get(f'home_q{quarter}', 0) == 0):
                continue
                
            # Get predictions
            main_prediction = game['main_prediction']
            quarter_prediction = game.get(f'q{quarter}_prediction', main_prediction * 0.95)
            
            # Create context
            context = {
                'current_quarter': quarter,
                'score_differential': abs(game.get('home_score', 0) - game.get('away_score', 0)),
                'momentum': game.get('cumulative_momentum', 0),
                'time_remaining_in_quarter': 6,  # Assume mid-quarter
                'time_remaining': (4 - quarter) * 12 + 6,  # Approx. time remaining in game
                'game_id': game['game_id'],
                'home_team': game['home_team'],
                'away_team': game['away_team']
            }
            
            # Test each ensemble method
            for method_name, method_func in ensemble_methods.items():
                try:
                    # Get ensemble prediction
                    ensemble_pred = method_func(main_prediction, quarter_prediction, quarter, context)
                    
                    # Calculate error
                    error = abs(ensemble_pred - actual_home_score)
                    
                    # Calculate winner accuracy
                    predicted_winner = 'home' if ensemble_pred > game.get('predicted_away_score', 0) else 'away'
                    actual_winner = 'home' if actual_home_score > actual_away_score else 'away'
                    correct_winner = predicted_winner == actual_winner
                    
                    # Store result
                    results.append({
                        'game_id': game['game_id'],
                        'quarter': quarter,
                        'method': method_name,
                        'actual_score': actual_home_score,
                        'predicted_score': ensemble_pred,
                        'error': error,
                        'correct_winner': correct_winner
                    })
                except Exception as e:
                    print(f"Error with method {method_name} on game {game['game_id']}: {e}")
    
    # Convert to DataFrame
    results_df = pd.DataFrame(results)
    
    # Calculate aggregate metrics by method and quarter
    if not results_df.empty:
        summary = results_df.groupby(['method', 'quarter']).agg({
            'error': ['mean', 'std', 'count'],
            'correct_winner': ['mean', 'count']
        }).reset_index()
        
        # Rename columns
        summary.columns = ['method', 'quarter', 'mae', 'std', 'count', 'winner_accuracy', 'winner_count']
        
        # Add RMSE
        squared_errors = results_df.groupby(['method', 'quarter'])['error'].apply(
            lambda x: np.sqrt((x**2).mean()))
        summary['rmse'] = squared_errors.values
        
        return summary
    
    return pd.DataFrame()

def visualize_benchmark_results(benchmark_results):
    """Visualize benchmark results with comparative charts"""
    if benchmark_results.empty:
        print("No benchmark results to visualize")
        return
        
    # Create figure
    fig, axs = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. MAE by quarter and method
    quarters = sorted(benchmark_results['quarter'].unique())
    methods = sorted(benchmark_results['method'].unique())
    
    # Group bar chart data
    quarter_labels = [f'Q{q}' if q > 0 else 'Pre' for q in quarters]
    x = np.arange(len(quarters))
    width = 0.8 / len(methods)
    
    # Plot MAE
    ax1 = axs[0, 0]
    for i, method in enumerate(methods):
        method_data = benchmark_results[benchmark_results['method'] == method]
        mae_by_quarter = {q: method_data[method_data['quarter'] == q]['mae'].values[0] 
                        if not method_data[method_data['quarter'] == q].empty else np.nan 
                        for q in quarters}
        values = [mae_by_quarter[q] for q in quarters]
        positions = x + (i - len(methods)/2 + 0.5) * width
        ax1.bar(positions, values, width=width, label=method)
    
    ax1.set_title('Mean Absolute Error by Quarter and Method')
    ax1.set_xlabel('Quarter')
    ax1.set_ylabel('MAE (points)')
    ax1.set_xticks(x)
    ax1.set_xticklabels(quarter_labels)
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    
    # Plot Winner Accuracy
    ax2 = axs[0, 1]
    for i, method in enumerate(methods):
        method_data = benchmark_results[benchmark_results['method'] == method]
        acc_by_quarter = {q: method_data[method_data['quarter'] == q]['winner_accuracy'].values[0] * 100
                        if not method_data[method_data['quarter'] == q].empty else np.nan 
                        for q in quarters}
        values = [acc_by_quarter[q] for q in quarters]
        positions = x + (i - len(methods)/2 + 0.5) * width
        ax2.bar(positions, values, width=width, label=method)
    
    ax2.set_title('Winner Prediction Accuracy by Quarter and Method')
    ax2.set_xlabel('Quarter')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_xticks(x)
    ax2.set_xticklabels(quarter_labels)
    ax2.set_ylim(0, 100)
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    
    # Plot RMSE
    ax3 = axs[1, 0]
    for i, method in enumerate(methods):
        method_data = benchmark_results[benchmark_results['method'] == method]
        rmse_by_quarter = {q: method_data[method_data['quarter'] == q]['rmse'].values[0]
                         if not method_data[method_data['quarter'] == q].empty else np.nan 
                         for q in quarters}
        values = [rmse_by_quarter[q] for q in quarters]
        positions = x + (i - len(methods)/2 + 0.5) * width
        ax3.bar(positions, values, width=width, label=method)
    
    ax3.set_title('RMSE by Quarter and Method')
    ax3.set_xlabel('Quarter')
    ax3.set_ylabel('RMSE (points)')
    ax3.set_xticks(x)
    ax3.set_xticklabels(quarter_labels)
    ax3.legend()
    ax3.grid(axis='y', alpha=0.3)
    
    # Plot sample sizes
    ax4 = axs[1, 1]
    for i, method in enumerate(methods):
        method_data = benchmark_results[benchmark_results['method'] == method]
        count_by_quarter = {q: method_data[method_data['quarter'] == q]['count'].values[0]
                          if not method_data[method_data['quarter'] == q].empty else 0
                          for q in quarters}
        values = [count_by_quarter[q] for q in quarters]
        positions = x + (i - len(methods)/2 + 0.5) * width
        ax4.bar(positions, values, width=width, label=method)
    
    ax4.set_title('Sample Size by Quarter and Method')
    ax4.set_xlabel('Quarter')
    ax4.set_ylabel('Count')
    ax4.set_xticks(x)
    ax4.set_xticklabels(quarter_labels)
    ax4.legend()
    ax4.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Test the context-aware weighting
def run_weighting_tests():
    """Run tests to demonstrate the context-aware weighting function"""
    print("Testing context-aware weighting:")
    game_contexts = [
        # Early close game with high momentum
        {'current_quarter': 1, 'score_differential': 3, 'momentum': 0.7, 'time_remaining': 36},
        # Mid-game blowout
        {'current_quarter': 2, 'score_differential': 18, 'momentum': 0.2, 'time_remaining': 24},
        # Late close game
        {'current_quarter': 4, 'score_differential': 4, 'momentum': 0.5, 'time_remaining': 6},
        # End of game
        {'current_quarter': 4, 'score_differential': 6, 'momentum': 0.3, 'time_remaining': 1}
    ]

    test_results = []
    main_prediction = 110.0
    quarter_prediction = 105.0

    for i, context in enumerate(game_contexts):
        weighted_pred, main_w, quarter_w, reasoning = context_aware_ensemble_weighting(
            main_prediction, quarter_prediction, context
        )
        
        test_results.append({
            'scenario': i+1,
            'quarter': context['current_quarter'],
            'score_diff': context['score_differential'],
            'momentum': context['momentum'],
            'time_left': context['time_remaining'],
            'main_weight': main_w,
            'quarter_weight': quarter_w,
            'prediction': weighted_pred,
            'reasoning': reasoning
        })

    # Display results
    results_df = pd.DataFrame(test_results)
    print(results_df[['scenario', 'quarter', 'score_diff', 'momentum', 'time_left', 'main_weight', 'quarter_weight', 'prediction']])
    
    return results_df

# Visualize the sigmoid weight transition
def visualize_sigmoid_transition():
    """Visualize the sigmoid weight transition function"""
    plt.figure(figsize=(10, 6))
    progress = np.linspace(0, 1, 100)
    weights = [sigmoid_weight_transition(p) for p in progress]

    plt.plot(progress, weights, 'b-', linewidth=2)
    plt.xlabel('Game Progress')
    plt.ylabel('Transition Factor')
    plt.title('Sigmoid-Based Weight Transition')
    plt.grid(True, alpha=0.3)

    # Mark key points
    quarters = [0, 0.25, 0.5, 0.75, 1.0]
    quarter_labels = ['Start', 'Q1', 'Half', 'Q3', 'End']
    for q, label in zip(quarters, quarter_labels):
        weight = sigmoid_weight_transition(q)
        plt.plot([q], [weight], 'ro')
        plt.text(q, weight + 0.05, label, ha='center')

    plt.tight_layout()
    plt.show()

# Run tests if executed directly
if __name__ == "__main__":
    run_weighting_tests()
    visualize_sigmoid_transition()

In [ ]:
# Cell 4C-5: Adaptive Strategy Selection


def select_ensemble_strategy(game_context):
    """
    Select the most appropriate ensemble strategy based on game context.
    
    Args:
        game_context: Dict with game state including:
            - current_quarter: Game quarter (0-4)
            - score_differential: Current score difference
            - momentum: Momentum indicator (-1 to 1)
            - time_remaining: Minutes remaining
            
    Returns:
        tuple: (strategy_name, strategy_params, reasoning)
    """
    # Unpack context
    quarter = game_context.get('current_quarter', 0)
    score_diff = abs(game_context.get('score_differential', 0))
    momentum = game_context.get('momentum', 0)
    time_remaining = game_context.get('time_remaining', 48 - quarter * 12)
    momentum_magnitude = abs(momentum)
    
    # Game progress as percentage
    game_progress = (48 - time_remaining) / 48.0
    
    # Define strategies
    strategies = {
        'conservative': {
            'main_weight_modifier': 0.1,  # Increase main model weight
            'description': 'Prioritizes main model predictions',
            'confidence_threshold': 0.4,
        },
        'balanced': {
            'main_weight_modifier': 0.0,  # No adjustment to weights
            'description': 'Balanced weighting between models',
            'confidence_threshold': 0.5,
        },
        'aggressive': {
            'main_weight_modifier': -0.1,  # Decrease main model weight
            'description': 'Emphasizes quarter-specific dynamics',
            'confidence_threshold': 0.6,
        },
        'momentum-focused': {
            'main_weight_modifier': -0.15,  # Strongly decrease main model weight
            'description': 'Heavily weights recent momentum shifts',
            'confidence_threshold': 0.7,
        }
    }
    
    # Decision logic
    reasoning = []
    
    # Default to balanced
    selected_strategy = 'balanced'
    
    # Close games with strong momentum get momentum-focused
    if score_diff < 10 and momentum_magnitude > 0.6:
        selected_strategy = 'momentum-focused'
        reasoning.append(f"Close game (diff: {score_diff}) with strong momentum ({momentum:.2f})")
    
    # Close late games get aggressive quarter weighting
    elif score_diff < 8 and quarter >= 3:
        selected_strategy = 'aggressive'
        reasoning.append(f"Close late game (Q{quarter}, diff: {score_diff})")
    
    # Blowouts get conservative weighting (trust the main model)
    elif score_diff > 15:
        selected_strategy = 'conservative'
        reasoning.append(f"Blowout game (diff: {score_diff})")
    
    # Early games with low confidence get balanced approach
    elif quarter <= 1:
        selected_strategy = 'balanced'
        reasoning.append(f"Early game (Q{quarter})")
    
    # Create parameters with sigmoid weight transition
    strategy_params = strategies[selected_strategy].copy()
    base_main_weight = 0.3 + (0.65 * sigmoid_weight_transition(game_progress))
    strategy_params['main_weight'] = max(0.1, min(0.95, 
                                               base_main_weight + strategy_params['main_weight_modifier']))
    strategy_params['quarter_weight'] = 1.0 - strategy_params['main_weight']
    
    # Compile reasoning
    reasoning_str = f"Strategy: {selected_strategy} - {strategies[selected_strategy]['description']}"
    if reasoning:
        reasoning_str += f" - {'; '.join(reasoning)}"
    
    return selected_strategy, strategy_params, reasoning_str

def apply_adaptive_strategy(main_prediction, quarter_prediction, game_context):
    """
    Apply the adaptive strategy to combine predictions based on game context.
    
    Args:
        main_prediction: Prediction from main model
        quarter_prediction: Prediction from quarter-specific model
        game_context: Dict with game state
        
    Returns:
        dict: Results including final prediction and explanation
    """
    # Select appropriate strategy
    strategy, params, reasoning = select_ensemble_strategy(game_context)
    
    # Apply the strategy
    final_prediction = (main_prediction * params['main_weight']) + (quarter_prediction * params['quarter_weight'])
    
    # Return results with detailed information
    return {
        'final_prediction': final_prediction,
        'strategy': strategy,
        'main_weight': params['main_weight'],
        'quarter_weight': params['quarter_weight'],
        'reasoning': reasoning,
        'main_prediction': main_prediction,
        'quarter_prediction': quarter_prediction
    }

# Test adaptive strategy selection
print("Testing adaptive strategy selection with different game scenarios:")
test_scenarios = [
    # Early game, small lead
    {'current_quarter': 1, 'score_differential': 5, 'momentum': 0.2, 'time_remaining': 36},
    # Mid-game, close with strong momentum
    {'current_quarter': 2, 'score_differential': 3, 'momentum': 0.7, 'time_remaining': 24},
    # Mid-game, blowout
    {'current_quarter': 2, 'score_differential': 18, 'momentum': 0.3, 'time_remaining': 24},
    # Late game, close
    {'current_quarter': 4, 'score_differential': 4, 'momentum': 0.4, 'time_remaining': 6},
    # Late game, blowout
    {'current_quarter': 4, 'score_differential': 20, 'momentum': 0.2, 'time_remaining': 6}
]

adaptive_results = []
main_prediction = 110.0
quarter_prediction = 105.0

for i, context in enumerate(test_scenarios):
    result = apply_adaptive_strategy(main_prediction, quarter_prediction, context)
    
    adaptive_results.append({
        'scenario': i+1,
        'quarter': context['current_quarter'],
        'score_diff': context['score_differential'],
        'momentum': context['momentum'],
        'strategy': result['strategy'],
        'main_weight': result['main_weight'],
        'quarter_weight': result['quarter_weight'],
        'prediction': result['final_prediction'],
        'reasoning': result['reasoning']
    })

# Display results
adaptive_df = pd.DataFrame(adaptive_results)
display(adaptive_df[['scenario', 'quarter', 'score_diff', 'momentum', 'strategy', 'main_weight', 'prediction', 'reasoning']])

# Visualize strategy regions
plt.figure(figsize=(10, 8))
momentum_vals = np.linspace(-1, 1, 21)
diff_vals = np.linspace(0, 25, 26)

X, Y = np.meshgrid(momentum_vals, diff_vals)
strategies = np.zeros_like(X, dtype=object)

for i in range(len(diff_vals)):
    for j in range(len(momentum_vals)):
        context = {
            'current_quarter': 3,  # Fixed for visualization
            'score_differential': Y[i, j],
            'momentum': X[i, j],
            'time_remaining': 12  # Fixed for visualization
        }
        strategy, _, _ = select_ensemble_strategy(context)
        strategies[i, j] = strategy

# Convert to numeric for plotting
strategy_map = {
    'balanced': 0,
    'conservative': 1,
    'aggressive': 2,
    'momentum-focused': 3
}
numeric_strategies = np.zeros_like(X)
for i in range(strategies.shape[0]):
    for j in range(strategies.shape[1]):
        numeric_strategies[i, j] = strategy_map[strategies[i, j]]

plt.pcolormesh(X, Y, numeric_strategies, cmap='viridis')
plt.colorbar(ticks=[0.4, 1.2, 2.0, 2.8], 
            label='Strategy').set_ticklabels(['Balanced', 'Conservative', 'Aggressive', 'Momentum-Focused'])
plt.xlabel('Momentum')
plt.ylabel('Score Differential')
plt.title('Strategy Selection Map (Quarter 3)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

class EnsembleStrategySelector:
    """
    Factory class that selects the best ensemble strategy based on historical performance
    and current game context.
    """
    
    def __init__(self, performance_data=None):
        """
        Initialize with optional performance data from previous games.
        
        Args:
            performance_data: DataFrame with historical performance by strategy
        """
        self.performance_data = performance_data
        
        # Default strategy map based on game contexts
        self.strategy_map = {
            # Format: (quarter, score_diff_range, momentum_range): strategy
            # Close games with momentum
            (1, (0, 5), (0.5, 1.0)): 'momentum_driven',
            (2, (0, 5), (0.5, 1.0)): 'momentum_driven',
            (3, (0, 5), (0.5, 1.0)): 'momentum_driven',
            (4, (0, 5), (0.5, 1.0)): 'momentum_driven',
            
            # Close games without momentum
            (1, (0, 5), (0.0, 0.5)): 'balanced',
            (2, (0, 5), (0.0, 0.5)): 'balanced',
            (3, (0, 5), (0.0, 0.5)): 'balanced',
            (4, (0, 5), (0.0, 0.5)): 'balanced',
            
            # Blowouts
            (1, (15, 100), (0.0, 1.0)): 'conservative',
            (2, (15, 100), (0.0, 1.0)): 'conservative',
            (3, (15, 100), (0.0, 1.0)): 'conservative',
            (4, (15, 100), (0.0, 1.0)): 'conservative',
            
            # Default by quarter
            (0, (0, 100), (0.0, 1.0)): 'balanced',
            (1, (0, 100), (0.0, 1.0)): 'balanced',
            (2, (0, 100), (0.0, 1.0)): 'balanced',
            (3, (0, 100), (0.0, 1.0)): 'conservative',
            (4, (0, 100), (0.0, 1.0)): 'conservative'
        }
        
        # If we have performance data, optimize the strategy map
        if self.performance_data is not None:
            self._optimize_strategy_map()
    
    def _optimize_strategy_map(self):
        """Use historical performance data to optimize strategy selection"""
        if self.performance_data.empty:
            return
            
        # Group by relevant context variables and find best strategy
        for quarter in range(5):
            quarter_data = self.performance_data[self.performance_data['quarter'] == quarter]
            if quarter_data.empty:
                continue
                
            # Group by score differential ranges
            for diff_range in [(0, 5), (5, 15), (15, 100)]:
                diff_data = quarter_data[
                    (quarter_data['score_diff'] >= diff_range[0]) & 
                    (quarter_data['score_diff'] < diff_range[1])
                ]
                if diff_data.empty:
                    continue
                    
                # Group by momentum ranges
                for momentum_range in [(0.0, 0.3), (0.3, 0.7), (0.7, 1.0)]:
                    moment_data = diff_data[
                        (abs(diff_data['momentum']) >= momentum_range[0]) & 
                        (abs(diff_data['momentum']) < momentum_range[1])
                    ]
                    if moment_data.empty:
                        continue
                        
                    # Find best strategy based on lowest error
                    best_strategy = moment_data.loc[moment_data['error'].idxmin()]['strategy']
                    
                    # Update strategy map
                    self.strategy_map[(quarter, diff_range, momentum_range)] = best_strategy
    
    def select_strategy(self, quarter, score_differential, momentum):
        """
        Select the best strategy based on game context.
        
        Args:
            quarter: Current quarter (0-4)
            score_differential: Current score differential (absolute)
            momentum: Current momentum (-1 to 1)
            
        Returns:
            str: Selected strategy name
        """
        # Normalize inputs
        quarter = int(quarter)
        score_diff = abs(float(score_differential))
        mom_magnitude = abs(float(momentum))
        
        # Find matching context in strategy map
        best_match = None
        best_score = float('inf')
        
        for (q, diff_range, mom_range), strategy in self.strategy_map.items():
            # Calculate match score (lower is better)
            q_match = abs(q - quarter)
            diff_match = 0 if diff_range[0] <= score_diff < diff_range[1] else min(
                abs(score_diff - diff_range[0]), 
                abs(score_diff - diff_range[1])
            )
            mom_match = 0 if mom_range[0] <= mom_magnitude < mom_range[1] else min(
                abs(mom_magnitude - mom_range[0]),
                abs(mom_magnitude - mom_range[1])
            )
            
            # Weight the match components - quarter match is most important
            match_score = q_match * 10 + diff_match + mom_match * 2
            
            if match_score < best_score:
                best_score = match_score
                best_match = strategy
        
        return best_match or 'balanced'  # Default to balanced if no match
    
    def update_performance(self, quarter, score_diff, momentum, strategy, error):
        """
        Update performance data with new results.
        
        Args:
            quarter: Game quarter
            score_diff: Score differential
            momentum: Momentum value
            strategy: Strategy used
            error: Prediction error
            
        Returns:
            None
        """
        if self.performance_data is None:
            self.performance_data = pd.DataFrame(columns=[
                'quarter', 'score_diff', 'momentum', 'strategy', 'error'
            ])
        
        # Add new performance data
        new_data = pd.DataFrame([{
            'quarter': quarter,
            'score_diff': score_diff,
            'momentum': momentum,
            'strategy': strategy,
            'error': error
        }])
        
        self.performance_data = pd.concat([self.performance_data, new_data], ignore_index=True)
        
        # Periodically optimize strategy map when enough new data is collected
        if len(new_data) % 20 == 0:
            self._optimize_strategy_map()

In [ ]:
# Cell 4C-6: Smooth Quarter Transitions with Dynamic Context Sensitivity

# Import the necessary functions and classes
try:
    from models.features import AdaptiveEnsembleFramework, dynamic_ensemble_predictions
except ImportError:
    # Check if AdaptiveEnsembleFramework is already defined
    try:
        AdaptiveEnsembleFramework
    except NameError:
        # Define minimal version
        class AdaptiveEnsembleFramework:
            def __init__(self):
                self.strategies = {
                    'balanced': {'description': 'Balanced weights'},
                    'momentum_driven': {'description': 'Momentum-focused'},
                    'conservative': {'description': 'Main model focused'}
                }
                
            def predict(self, main_pred, quarter_pred, quarter, context=None, strategy=None):
                weights = {'main': 0.5 + (quarter * 0.1), 'quarter': 0.5 - (quarter * 0.1), 'strategy': strategy or 'balanced'}
                prediction = main_pred * weights['main'] + quarter_pred * weights['quarter']
                confidence = 0.5 + (quarter * 0.1)
                return prediction, weights, confidence
    
    # Define dynamic_ensemble_predictions if not available
    try:
        dynamic_ensemble_predictions
    except NameError:
        def dynamic_ensemble_predictions(main_prediction, quarter_prediction, quarter,
                                      score_differential=0, momentum=0, 
                                      time_remaining_in_quarter=6, 
                                      weighting_strategy=None):
            """Simplified implementation of dynamic ensemble predictions"""
            if weighting_strategy is None:
                if score_differential < 5 and abs(momentum) > 0.5:
                    weighting_strategy = 'momentum_driven'
                elif score_differential > 15:
                    weighting_strategy = 'conservative'
                else:
                    weighting_strategy = 'balanced'
            
            # Simple weight selection
            if weighting_strategy == 'momentum_driven':
                weights = {'main': 0.4, 'quarter': 0.6}
            elif weighting_strategy == 'conservative':
                weights = {'main': 0.7, 'quarter': 0.3}
            else:  # balanced
                weights = {'main': 0.5, 'quarter': 0.5}
            
            # Apply quarter progression
            weights['main'] = min(0.95, weights['main'] + (quarter * 0.1))
            weights['quarter'] = 1.0 - weights['main']
            
            # Calculate prediction
            prediction = main_prediction * weights['main'] + quarter_prediction * weights['quarter']
            confidence = 0.5 + (quarter * 0.1)
            
            return prediction, confidence, weights['main'], weights['quarter']

def create_dynamic_context_weighting(current_quarter, time_remaining_mins, score_differential, momentum):
    """
    Calculate dynamic ensemble weights based on game context and smooth transitions
    between quarters.
    
    Args:
        current_quarter: Current quarter (1-4)
        time_remaining_mins: Minutes remaining in the game
        score_differential: Absolute point differential
        momentum: Normalized momentum value (-1 to 1)
        
    Returns:
        Dictionary of weights for different models
    """
    # Calculate game progression on a continuous 0-1 scale
    # Instead of discrete quarter steps, this creates a smooth progression
    total_mins = 48.0
    elapsed_mins = total_mins - time_remaining_mins
    game_progress = min(elapsed_mins / total_mins, 1.0)
    
    # Base weights that shift as game progresses (smooth sigmoid transition)
    # This replaces the discrete quarter-based weights with a continuous function
    main_base_weight = 0.3 + (0.7 * (1 / (1 + np.exp(-10 * (game_progress - 0.5)))))
    quarter_base_weight = 1.0 - main_base_weight
    
    # Context adjustments
    # Game closeness adjustment
    is_close_game = score_differential < 8
    closeness_factor = max(0.0, (8 - score_differential) / 8) * 0.1
    
    # Momentum adjustment - stronger momentum gives more weight to quarter models
    momentum_strength = abs(momentum)
    momentum_factor = momentum_strength * 0.1
    
    # Apply context adjustments to base weights
    if is_close_game:
        # In close games, quarter-specific models get more weight
        main_weight = max(0.2, main_base_weight - closeness_factor)
        quarter_weight = min(0.8, quarter_base_weight + closeness_factor)
    else:
        # In blowouts, main model gets more weight
        main_weight = min(0.95, main_base_weight + (1 - closeness_factor) * 0.05)
        quarter_weight = max(0.05, quarter_base_weight - (1 - closeness_factor) * 0.05)
    
    # Apply momentum adjustment
    if momentum_strength > 0.3:
        # Strong momentum gives more weight to quarter models
        main_weight = max(0.2, main_weight - momentum_factor)
        quarter_weight = min(0.8, quarter_weight + momentum_factor)
    
    # Add historical model weight as a small constant
    historical_weight = 0.0
    
    # Normalize weights to ensure they sum to 1.0
    total = main_weight + quarter_weight + historical_weight
    weights = {
        'main_model': main_weight / total,
        'quarter_model': quarter_weight / total,
        'historical_model': historical_weight / total
    }
    
    return weights

# Test the function with various game scenarios
print("Testing Dynamic Context Weighting with different game scenarios:")
test_scenarios = [
    {"name": "Early close game", "quarter": 1, "time_remaining": 42, "score_diff": 3, "momentum": 0.1},
    {"name": "Mid-game, building momentum", "quarter": 2, "time_remaining": 30, "score_diff": 5, "momentum": 0.4},
    {"name": "Late close game", "quarter": 4, "time_remaining": 6, "score_diff": 2, "momentum": 0.8},
    {"name": "Late blowout", "quarter": 4, "time_remaining": 6, "score_diff": 20, "momentum": 0.2}
]

for scenario in test_scenarios:
    weights = create_dynamic_context_weighting(
        scenario["quarter"],
        scenario["time_remaining"],
        scenario["score_diff"],
        scenario["momentum"]
    )
    print(f"\n{scenario['name']}:")
    print(f"  Quarter: {scenario['quarter']}, Time Left: {scenario['time_remaining']} mins")
    print(f"  Score Diff: {scenario['score_diff']}, Momentum: {scenario['momentum']:.2f}")
    print(f"  Weights: Main={weights['main_model']:.2f}, Quarter={weights['quarter_model']:.2f}")

def optimized_ensemble_prediction(main_prediction, quarter_predictions, game_data, 
                                 use_adaptive_ensemble=True, strategy_selector=None):
    """
    Optimized ensemble prediction function for production use in the NBA prediction pipeline.
    
    Args:
        main_prediction: Main model prediction value
        quarter_predictions: Dict of quarter-specific predictions
        game_data: Current game state data
        use_adaptive_ensemble: Whether to use the adaptive ensemble (vs. standard)
        strategy_selector: Optional EnsembleStrategySelector instance
        
    Returns:
        dict: Prediction package with ensemble details
    """
    # Extract key game data
    current_quarter = int(game_data.get('current_quarter', 0))
    time_remaining = float(game_data.get('time_remaining_in_quarter', 6))
    
    # Get current quarter prediction if available, else use main prediction
    quarter_prediction = quarter_predictions.get(f'q{current_quarter}', main_prediction * 0.95)
    
    # Calculate necessary game context
    home_score = float(game_data.get('home_score', 0))
    away_score = float(game_data.get('away_score', 0))
    score_differential = abs(home_score - away_score)
    momentum = float(game_data.get('cumulative_momentum', 0))
    
    # Determine best strategy
    if strategy_selector:
        strategy = strategy_selector.select_strategy(current_quarter, score_differential, momentum)
    else:
        strategy = None  # Will use default/auto selection
    
    # Generate prediction using appropriate ensemble method
    if use_adaptive_ensemble:
        # Use full AdaptiveEnsembleFramework
        ensemble = AdaptiveEnsembleFramework()
        
        # Prepare game context
        context = {
            'score_differential': score_differential,
            'momentum': momentum,
            'time_remaining_in_quarter': time_remaining,
            'game_id': game_data.get('game_id', 'unknown')
        }
        
        # Get prediction with weights and confidence
        prediction, weights, confidence = ensemble.predict(
            main_prediction, 
            quarter_prediction, 
            current_quarter,
            context,
            strategy
        )
        
        # Prepare result
        result = {
            'home_prediction': prediction,
            'away_prediction': game_data.get('away_score', 0) + 
                             (prediction - game_data.get('home_score', 0)),
            'weights': weights,
            'confidence': confidence,
            'strategy': weights['strategy']
        }
    else:
        # Use simplified ensemble prediction
        prediction, confidence, main_weight, quarter_weight = dynamic_ensemble_predictions(
            main_prediction,
            quarter_prediction,
            current_quarter,
            score_differential,
            momentum,
            time_remaining,
            strategy
        )
        
        # Prepare result
        result = {
            'home_prediction': prediction,
            'away_prediction': game_data.get('away_score', 0) + 
                             (prediction - game_data.get('home_score', 0)),
            'weights': {
                'main': main_weight,
                'quarter': quarter_weight,
                'strategy': strategy or 'auto'
            },
            'confidence': confidence,
            'strategy': strategy or 'auto'
        }
    
    return result

In [ ]:
# Cell 4C-7: Integrated Adaptive Ensemble Framework
"""
This cell provides a comprehensive framework for adaptively weighting ensemble predictions in 
basketball games, based on game context and progression. It contains two implementations:

1. AdaptiveEnsembleFramework: Full-featured implementation with visualizations and detailed context handling
2. OptimizedAdaptiveEnsemble: Performance-optimized version using caching and lookup tables for production use

Both implementations serve the same purpose but with different performance and feature tradeoffs.
"""

from datetime import datetime
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import expit  # Sigmoid function

class AdaptiveEnsembleFramework:
    """
    Comprehensive framework for dynamically adjusting ensemble weights based on game context,
    with smooth transitions, confidence integration, contextual awareness, and prediction history.
    """
    
    def __init__(self):
        # Base configuration for different weighting strategies
        self.strategies = {
            'balanced': {
                'description': 'Balanced weights with smooth transitions',
                'base_main_weight': {
                    0: 0.30,  # Pre-game
                    1: 0.40,  # Q1
                    2: 0.60,  # Q2
                    3: 0.80,  # Q3
                    4: 0.95   # Q4
                },
                'context_sensitivity': 0.5  # Medium sensitivity to context
            },
            'momentum_driven': {
                'description': 'Emphasizes quarter models when momentum is strong',
                'base_main_weight': {
                    0: 0.25,  # Pre-game
                    1: 0.35,  # Q1
                    2: 0.50,  # Q2
                    3: 0.70,  # Q3
                    4: 0.90   # Q4
                },
                'context_sensitivity': 0.8  # High sensitivity to context
            },
            'conservative': {
                'description': 'Emphasizes main model for stability',
                'base_main_weight': {
                    0: 0.40,  # Pre-game
                    1: 0.50,  # Q1
                    2: 0.70,  # Q2
                    3: 0.85,  # Q3
                    4: 0.98   # Q4
                },
                'context_sensitivity': 0.3  # Low sensitivity to context
            }
        }
        
        # Configuration for continuous quarter transitions
        self.transition_steepness = 8.0  # Controls smoothness of transitions
        
        # Set default strategy
        self.default_strategy = 'balanced'
        
        # Historical performance tracking for adaptive strategy selection
        self.strategy_performance = {}
        
        # Prediction history storage
        self.prediction_history = {}
    
    def select_strategy(self, game_context):
        """
        Automatically select the best strategy based on game context.
        
        Args:
            game_context: Dict with game state information
            
        Returns:
            str: Name of the selected strategy
        """
        # Extract context variables
        score_diff = abs(game_context.get('score_differential', 0))
        momentum = abs(game_context.get('momentum', 0))
        quarter = game_context.get('current_quarter', 0)
        
        # Use historical performance if available for this context
        if self.strategy_performance:
            # Look for similar contexts in history
            similar_contexts = self._find_similar_contexts(game_context)
            if similar_contexts:
                # Return best performing strategy for similar contexts
                return self._get_best_strategy(similar_contexts)
        
        # Otherwise use heuristic rules
        if score_diff < 5:  # Close game
            if momentum > 0.5:  # Strong momentum
                return 'momentum_driven'
            else:
                return 'balanced'
        elif quarter >= 3:  # Late game
            if score_diff > 15:  # Clear blowout
                return 'conservative'  # Trust main model more
            else:
                return 'balanced'
        else:
            return self.default_strategy
    
    def _find_similar_contexts(self, current_context):
        """Find past contexts similar to current game context"""
        # Placeholder for actual implementation
        # In a real implementation, this would search through historical data
        return []
    
    def _get_best_strategy(self, similar_contexts):
        """Determine best strategy based on performance in similar contexts"""
        # Placeholder for actual implementation
        return self.default_strategy
    
    def get_continuous_quarter(self, quarter, time_remaining_in_quarter=None):
        """
        Convert discrete quarter to continuous value for smooth transitions.
        
        Args:
            quarter: Current quarter (0-4)
            time_remaining_in_quarter: Minutes remaining in current quarter (0-12)
            
        Returns:
            float: Continuous quarter value (e.g., 2.75 for 75% through Q2)
        """
        if time_remaining_in_quarter is None:
            # If time not provided, just return the quarter as is
            return float(quarter)
            
        # Convert time to progress within quarter (0-1)
        quarter_duration = 12.0  # Minutes per quarter
        quarter_progress = (quarter_duration - time_remaining_in_quarter) / quarter_duration
        quarter_progress = max(0.0, min(1.0, quarter_progress))  # Clamp to [0,1]
        
        # Calculate continuous quarter
        continuous_quarter = quarter + quarter_progress
        
        return continuous_quarter
    
    def get_smooth_main_weight(self, continuous_quarter, base_weights):
        """
        Calculate smoothly transitioning main weight based on continuous quarter.
        Uses sigmoid function for smooth interpolation between quarters.
        
        Args:
            continuous_quarter: Continuous quarter value 
            base_weights: Dictionary of base weights by quarter
            
        Returns:
            float: Smoothly interpolated main weight
        """
        # Get integer quarters before and after
        q_floor = int(continuous_quarter)
        q_ceil = min(4, q_floor + 1)  # Cap at Q4
        
        # Get weights for surrounding quarters
        w_floor = base_weights.get(q_floor, 0.5)
        w_ceil = base_weights.get(q_ceil, 0.95)
        
        # Calculate fraction between quarters
        fraction = continuous_quarter - q_floor
        
        # Use sigmoid for smoother transition around midpoints
        # Adjust fraction to be centered at 0.5
        adjusted_fraction = (fraction - 0.5) * self.transition_steepness
        sigmoid_fraction = expit(adjusted_fraction)  # Apply sigmoid
        
        # Interpolate weight
        smooth_weight = w_floor + sigmoid_fraction * (w_ceil - w_floor)
        
        return smooth_weight
    
    def adjust_for_context(self, base_weight, game_context, sensitivity):
        """
        Adjust weight based on game context (score differential, momentum, etc).
        
        Args:
            base_weight: Starting weight to adjust
            game_context: Dict with game state variables
            sensitivity: How strongly context affects weights (0-1)
            
        Returns:
            float: Context-adjusted weight
        """
        if sensitivity == 0:
            return base_weight
            
        # Extract context variables
        score_diff = abs(game_context.get('score_differential', 0))
        momentum = game_context.get('momentum', 0)
        momentum_magnitude = abs(momentum)
        game_progress = game_context.get('game_progress', 0.0)  # 0-1 value
        confidence_main = game_context.get('confidence_main', 0.8)
        confidence_quarter = game_context.get('confidence_quarter', 0.7)
        
        # Calculate score-based adjustment
        # Close games benefit more from quarter-specific models
        if score_diff < 3:
            score_adjustment = -0.10  # Reduce main weight (increase quarter weight)
        elif score_diff < 7:
            score_adjustment = -0.05
        elif score_diff > 20:
            score_adjustment = 0.10  # Increase main weight in blowouts
        elif score_diff > 12:
            score_adjustment = 0.05
        else:
            score_adjustment = 0.0
            
        # Calculate momentum-based adjustment
        # Strong momentum gives more weight to quarter models
        momentum_adjustment = -0.15 * momentum_magnitude  # Negative means more weight to quarter model
        
        # Calculate confidence-based adjustment
        # Higher confidence in a model increases its weight
        confidence_gap = confidence_main - confidence_quarter
        confidence_adjustment = 0.10 * confidence_gap
        
        # Apply game progress factor - context matters more late in game
        progress_factor = 0.5 + (0.5 * game_progress)  # 0.5 to 1.0
        
        # Combine adjustments with sensitivity and progress scaling
        total_adjustment = (
            score_adjustment + 
            momentum_adjustment + 
            confidence_adjustment
        ) * sensitivity * progress_factor
        
        # Apply adjustment with limits to prevent extreme weights
        adjusted_weight = base_weight + total_adjustment
        adjusted_weight = max(0.1, min(0.95, adjusted_weight))
        
        return adjusted_weight
    
    def calculate_ensemble_weights(self, 
                                  quarter, 
                                  time_remaining_in_quarter=None,
                                  game_context=None, 
                                  strategy=None):
        """
        Calculate ensemble weights based on game state with smooth transitions.
        
        Args:
            quarter: Current quarter (0-4)
            time_remaining_in_quarter: Minutes remaining in current quarter (0-12)
            game_context: Optional dict with additional context
            strategy: Optional strategy name (auto-selected if None)
            
        Returns:
            dict: Weights for ensemble models ('main', 'quarter')
        """
        # Create default game context if not provided
        if game_context is None:
            game_context = {}
            
        # Add quarter and time to context
        game_context['current_quarter'] = quarter
        if time_remaining_in_quarter is not None:
            game_context['time_remaining_in_quarter'] = time_remaining_in_quarter
            
        # Calculate continuous quarter for smooth transitions
        continuous_quarter = self.get_continuous_quarter(quarter, time_remaining_in_quarter)
        
        # Add game progress to context
        minutes_played = (quarter - 1) * 12
        if time_remaining_in_quarter is not None:
            minutes_played += (12 - time_remaining_in_quarter)
        game_context['game_progress'] = min(1.0, minutes_played / 48.0)
            
        # Select strategy if not provided
        if strategy is None:
            strategy = self.select_strategy(game_context)
            
        # Get strategy configuration
        strategy_config = self.strategies.get(strategy, self.strategies[self.default_strategy])
        base_weights = strategy_config['base_main_weight']
        sensitivity = strategy_config['context_sensitivity']
        
        # Calculate smooth base weight
        base_main_weight = self.get_smooth_main_weight(continuous_quarter, base_weights)
        
        # Adjust for context
        final_main_weight = self.adjust_for_context(base_main_weight, game_context, sensitivity)
        
        # Calculate quarter weight as complement
        final_quarter_weight = 1.0 - final_main_weight
        
        # Create weights dictionary
        weights = {
            'main': final_main_weight,
            'quarter': final_quarter_weight,
            'strategy': strategy,
            'base_weight': base_main_weight,
            'continuous_quarter': continuous_quarter
        }
        
        return weights
    
    def predict(self, main_prediction, quarter_prediction, quarter, game_context=None, strategy=None):
        """
        Generate ensemble prediction with the appropriate weighting.
        
        Args:
            main_prediction: Prediction from main model
            quarter_prediction: Prediction from quarter-specific model
            quarter: Current quarter (0-4)
            game_context: Optional game context data
            strategy: Optional strategy override
            
        Returns:
            tuple: (prediction, weights, confidence)
        """
        # Use an empty context if none provided
        if game_context is None:
            game_context = {}
            
        # Add time_remaining_in_quarter if present in context
        time_remaining = game_context.get('time_remaining_in_quarter')
            
        # Get weights
        weights = self.calculate_ensemble_weights(
            quarter,
            time_remaining,
            game_context,
            strategy
        )
        
        # Handle array inputs 
        if isinstance(main_prediction, (list, np.ndarray)) and isinstance(quarter_prediction, (list, np.ndarray)):
            # Handle array inputs
            ensemble_prediction = (
                np.array(main_prediction) * weights['main'] + 
                np.array(quarter_prediction) * weights['quarter']
            )
        else:
            # Calculate weighted prediction for scalar inputs
            ensemble_prediction = (
                main_prediction * weights['main'] + 
                quarter_prediction * weights['quarter']
            )
        
        # Calculate confidence based on quarter and weights
        # Confidence increases with quarter number and when weights are more decisive
        quarter_boost = 0.1 * quarter  # 0.0 for Q0, 0.4 for Q4
        weight_decisiveness = abs(weights['main'] - 0.5) * 2  # 0.0-1.0 based on how far from 50/50
        base_confidence = 0.5 + quarter_boost
        confidence_boost = weight_decisiveness * 0.2  # Up to 0.2 extra confidence
        
        confidence = min(0.95, base_confidence + confidence_boost)
        
        # Store prediction in history
        prediction_data = {
            'main_prediction': main_prediction,
            'quarter_prediction': quarter_prediction,
            'ensemble_prediction': ensemble_prediction,
            'weights': weights,
            'strategy': weights['strategy'],
            'quarter': quarter,
            'confidence': confidence,
            'timestamp': datetime.now()
        }
        
        # Add to history using a unique key if context provides one
        history_key = game_context.get('game_id', str(time.time())) if game_context else str(time.time())
        if history_key not in self.prediction_history:
            self.prediction_history[history_key] = []
        self.prediction_history[history_key].append(prediction_data)
        
        return ensemble_prediction, weights, confidence

    def visualize_weight_transitions(self, strategies=None, show_context_effects=True):
        """
        Visualize how weights transition smoothly between quarters and with context.
        
        Args:
            strategies: List of strategies to visualize (None for all)
            show_context_effects: Whether to show context effects
        """
        if strategies is None:
            strategies = list(self.strategies.keys())
            
        # Create figure
        plt.figure(figsize=(15, 10))
        
        # 1. Smooth transitions by quarter
        plt.subplot(2, 2, 1)
        continuous_quarters = np.linspace(0, 4, 100)
        
        for strategy in strategies:
            strategy_config = self.strategies[strategy]
            base_weights = strategy_config['base_main_weight']
            
            weights = [self.get_smooth_main_weight(q, base_weights) for q in continuous_quarters]
            plt.plot(continuous_quarters, weights, label=strategy)
            
        plt.title('Main Model Weight by Continuous Quarter')
        plt.xlabel('Quarter')
        plt.ylabel('Main Model Weight')
        plt.xticks([0, 1, 2, 3, 4], ['Pre', 'Q1', 'Q2', 'Q3', 'Q4'])
        plt.grid(True, alpha=0.3)
        plt.legend()
        
        # 2. Effect of score differential
        if show_context_effects:
            plt.subplot(2, 2, 2)
            score_diffs = np.linspace(0, 30, 100)
            
            for strategy in strategies:
                strategy_config = self.strategies[strategy]
                base_weight = 0.6  # Mid-game weight
                sensitivity = strategy_config['context_sensitivity']
                
                weights = []
                for diff in score_diffs:
                    context = {'score_differential': diff, 'game_progress': 0.5}
                    weight = self.adjust_for_context(base_weight, context, sensitivity)
                    weights.append(weight)
                    
                plt.plot(score_diffs, weights, label=strategy)
                
            plt.title('Effect of Score Differential on Weights')
            plt.xlabel('Score Differential')
            plt.ylabel('Main Model Weight')
            plt.grid(True, alpha=0.3)
            plt.legend()
            
            # 3. Effect of momentum
            plt.subplot(2, 2, 3)
            momentum_values = np.linspace(-1, 1, 100)
            
            for strategy in strategies:
                strategy_config = self.strategies[strategy]
                base_weight = 0.6  # Mid-game weight
                sensitivity = strategy_config['context_sensitivity']
                
                weights = []
                for momentum in momentum_values:
                    context = {'momentum': momentum, 'game_progress': 0.5}
                    weight = self.adjust_for_context(base_weight, context, sensitivity)
                    weights.append(weight)
                    
                plt.plot(momentum_values, weights, label=strategy)
                
            plt.title('Effect of Momentum on Weights')
            plt.xlabel('Momentum (-1 to 1)')
            plt.ylabel('Main Model Weight')
            plt.axvline(x=0, color='black', linestyle='--', alpha=0.3)
            plt.grid(True, alpha=0.3)
            plt.legend()
            
            # 4. Combined effects in different game states
            plt.subplot(2, 2, 4)
            
            # Define some representative game states
            game_states = [
                {'name': 'Early Close', 'quarter': 1, 'time': 6, 'diff': 3, 'momentum': 0.1},
                {'name': 'Mid Blowout', 'quarter': 2, 'time': 6, 'diff': 18, 'momentum': 0.2},
                {'name': 'Mid w/Momentum', 'quarter': 2, 'time': 6, 'diff': 5, 'momentum': 0.8},
                {'name': 'Late Close', 'quarter': 4, 'time': 6, 'diff': 4, 'momentum': 0.3},
                {'name': 'Late Comeback', 'quarter': 4, 'time': 3, 'diff': 8, 'momentum': -0.7}
            ]
            
            # Calculate weights for each state and strategy
            state_labels = []
            data = []
            
            for state in game_states:
                state_labels.append(state['name'])
                
                for strategy in strategies:
                    context = {
                        'score_differential': state['diff'],
                        'momentum': state['momentum']
                    }
                    
                    weights = self.calculate_ensemble_weights(
                        state['quarter'],
                        state['time'],
                        context,
                        strategy
                    )
                    
                    data.append({
                        'Game State': state['name'],
                        'Strategy': strategy,
                        'Main Weight': weights['main'],
                        'Quarter Weight': weights['quarter']
                    })
            
            # Convert to DataFrame
            df = pd.DataFrame(data)
            
            # Plot grouped bar chart
            sns.barplot(x='Game State', y='Main Weight', hue='Strategy', data=df)
            plt.title('Ensemble Weights in Different Game States')
            plt.xlabel('Game State')
            plt.ylabel('Main Model Weight')
            plt.ylim(0, 1)
            plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        
        plt.tight_layout()
        plt.show()
        
    def visualize_prediction_behavior(self, main_pred=110, quarter_pred=100):
        """
        Visualize how ensemble predictions behave in different game states.
        
        Args:
            main_pred: Sample main model prediction
            quarter_pred: Sample quarter model prediction
        """
        # Create figure
        plt.figure(figsize=(15, 10))
        
        # 1. Prediction across quarters
        plt.subplot(2, 2, 1)
        continuous_quarters = np.linspace(0, 4, 100)
        
        for strategy in self.strategies:
            predictions = []
            
            for q in continuous_quarters:
                # Get integer quarter and remaining time
                q_int = int(q)
                q_frac = q - q_int
                time_remaining = 12 * (1 - q_frac)
                
                # Calculate weights
                weights = self.calculate_ensemble_weights(
                    q_int,
                    time_remaining,
                    None,
                    strategy
                )
                
                # Calculate prediction
                pred = main_pred * weights['main'] + quarter_pred * weights['quarter']
                predictions.append(pred)
                
            plt.plot(continuous_quarters, predictions, label=strategy)
            
        # Add reference lines for underlying models
        plt.axhline(y=main_pred, color='black', linestyle='--', label=f'Main: {main_pred}')
        plt.axhline(y=quarter_pred, color='gray', linestyle=':', label=f'Quarter: {quarter_pred}')
            
        plt.title('Ensemble Prediction Across Game Progress')
        plt.xlabel('Quarter')
        plt.ylabel('Prediction')
        plt.xticks([0, 1, 2, 3, 4], ['Pre', 'Q1', 'Q2', 'Q3', 'Q4'])
        plt.grid(True, alpha=0.3)
        plt.legend()
        
        # 2. Effect of momentum on predictions
        plt.subplot(2, 2, 2)
        momentum_values = np.linspace(-1, 1, 100)
        quarters = [1, 2, 3, 4]
        
        for q in quarters:
            predictions = []
            
            for momentum in momentum_values:
                context = {'momentum': momentum}
                weights = self.calculate_ensemble_weights(q, 6, context, 'momentum_driven')
                pred = main_pred * weights['main'] + quarter_pred * weights['quarter']
                predictions.append(pred)
                
            plt.plot(momentum_values, predictions, label=f'Q{q}')
            
        plt.title('Effect of Momentum on Predictions (Momentum-Driven Strategy)')
        plt.xlabel('Momentum (-1 to 1)')
        plt.ylabel('Prediction')
        plt.axhline(y=main_pred, color='black', linestyle='--', label=f'Main: {main_pred}')
        plt.axhline(y=quarter_pred, color='gray', linestyle=':', label=f'Quarter: {quarter_pred}')
        plt.axvline(x=0, color='black', linestyle='--', alpha=0.3)
        plt.grid(True, alpha=0.3)
        plt.legend()
        
        # 3. Score differential effect
        plt.subplot(2, 2, 3)
        score_diffs = np.linspace(0, 30, 100)
        
        for q in quarters:
            predictions = []
            
            for diff in score_diffs:
                context = {'score_differential': diff}
                weights = self.calculate_ensemble_weights(q, 6, context, 'balanced')
                pred = main_pred * weights['main'] + quarter_pred * weights['quarter']
                predictions.append(pred)
                
            plt.plot(score_diffs, predictions, label=f'Q{q}')
            
        plt.title('Effect of Score Differential on Predictions (Balanced Strategy)')
        plt.xlabel('Score Differential')
        plt.ylabel('Prediction')
        plt.axhline(y=main_pred, color='black', linestyle='--', label=f'Main: {main_pred}')
        plt.axhline(y=quarter_pred, color='gray', linestyle=':', label=f'Quarter: {quarter_pred}')
        plt.grid(True, alpha=0.3)
        plt.legend()
        
        # 4. Confidence across quarters
        plt.subplot(2, 2, 4)
        
        for strategy in self.strategies:
            confidences = []
            
            for q in continuous_quarters:
                # Get integer quarter and remaining time
                q_int = int(q)
                q_frac = q - q_int
                time_remaining = 12 * (1 - q_frac)
                
                # Calculate prediction with confidence
                _, weights, confidence = self.predict(
                    main_pred,
                    quarter_pred,
                    q_int,
                    {'time_remaining_in_quarter': time_remaining},
                    strategy
                )
                
                confidences.append(confidence)
                
            plt.plot(continuous_quarters, confidences, label=strategy)
            
        plt.title('Prediction Confidence Across Game Progress')
        plt.xlabel('Quarter')
        plt.ylabel('Confidence')
        plt.xticks([0, 1, 2, 3, 4], ['Pre', 'Q1', 'Q2', 'Q3', 'Q4'])
        plt.grid(True, alpha=0.3)
        plt.legend()
        
        plt.tight_layout()
        plt.show()

    def visualize_strategy_comparison(self, main_pred, quarter_pred, current_quarter=2, score_diff=5, momentum=0.2):
        """
        Create visualization comparing different weighting strategies.
        
        Args:
            main_pred: Main model prediction
            quarter_pred: Quarter model prediction
            current_quarter: Current quarter
            score_diff: Score differential
            momentum: Momentum value
            
        Returns:
            matplotlib Figure
        """
        # Set up the context
        game_context = {
            'current_quarter': current_quarter,
            'score_differential': score_diff,
            'momentum': momentum,
            'time_remaining_in_quarter': 6
        }
        
        # Get predictions for each strategy
        strategy_results = []
        for strategy_name in self.strategies:
            try:
                weights = self.calculate_ensemble_weights(
                    current_quarter,
                    6,  # Mid-quarter
                    game_context,
                    strategy_name
                )
                
                ensemble_pred = main_pred * weights['main'] + quarter_pred * weights['quarter']
                
                strategy_results.append({
                    'strategy': strategy_name,
                    'main_weight': weights['main'],
                    'quarter_weight': weights['quarter'],
                    'ensemble_prediction': ensemble_pred,
                    'difference_from_main': ensemble_pred - main_pred
                })
            except Exception as e:
                print(f"Error processing strategy {strategy_name}: {e}")
        
        # Create a DataFrame
        results_df = pd.DataFrame(strategy_results)
        
        # Create visualization
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        
        # Plot 1: Weights by strategy
        strategies = results_df['strategy']
        ind = np.arange(len(strategies))
        width = 0.35
        
        ax1.bar(ind, results_df['main_weight'], width, label='Main Model')
        ax1.bar(ind, results_df['quarter_weight'], width, bottom=results_df['main_weight'], label='Quarter Model')
        
        ax1.set_ylabel('Weight')
        ax1.set_title('Weights by Strategy')
        ax1.set_xticks(ind)
        ax1.set_xticklabels(strategies, rotation=45)
        ax1.legend()
        
        # Plot 2: Predictions by strategy
        ax2.bar(strategies, results_df['ensemble_prediction'], color='lightblue')
        ax2.axhline(y=main_pred, color='r', linestyle='-', label=f'Main: {main_pred}')
        ax2.axhline(y=quarter_pred, color='g', linestyle='--', label=f'Quarter: {quarter_pred}')
        
        ax2.set_ylabel('Prediction')
        ax2.set_title('Ensemble Predictions by Strategy')
        ax2.set_xticklabels(strategies, rotation=45)
        ax2.legend()
        
        plt.tight_layout()
        return fig

# Simplified interface function for generating ensemble predictions
def dynamic_ensemble_predictions(main_prediction, quarter_prediction, quarter,
                                score_differential=0, momentum=0, 
                                time_remaining_in_quarter=6, 
                                weighting_strategy=None):
    """
    Simplified interface for generating ensemble predictions with dynamic weighting.
    
    Args:
        main_prediction: Main model prediction
        quarter_prediction: Quarter-specific model prediction
        quarter: Current quarter (0-4)
        score_differential: Score differential (absolute)
        momentum: Momentum indicator (-1 to 1)
        time_remaining_in_quarter: Minutes remaining in current quarter
        weighting_strategy: Strategy to use (None for automatic)
        
    Returns:
        tuple: (ensemble_prediction, confidence, main_weight, quarter_weight)
    """
    # Create context
    game_context = {
        'score_differential': abs(score_differential),
        'momentum': momentum,
        'time_remaining_in_quarter': time_remaining_in_quarter
    }
    
    # Create ensemble
    ensemble = AdaptiveEnsembleFramework()
    
    # Generate prediction
    pred, weights, conf = ensemble.predict(
        main_prediction,
        quarter_prediction,
        quarter,
        game_context,
        weighting_strategy
    )
    
    return pred, conf, weights['main'], weights['quarter']


# ================= OPTIMIZED IMPLEMENTATION FOR PRODUCTION =================

class OptimizedAdaptiveEnsemble:
    """
    Optimized version of AdaptiveEnsembleFramework with caching and streamlined calculations
    for production use in real-time predictions. 
    
    This implementation sacrifices flexibility and visualization features for speed and efficiency,
    making it suitable for high-throughput prediction scenarios.
    """
    
    def __init__(self):
        # Pre-calculate weight tables for common game states
        self._initialize_weight_tables()
        
        # Cache for frequent predictions
        self.weight_cache = {}
        self.cache_hits = 0
        self.cache_misses = 0
    
    def _initialize_weight_tables(self):
        """Pre-calculate weight tables for common game states"""
        # Base weights by quarter and strategy
        self.base_weights = {
            'balanced': {
                0: 0.30, 1: 0.40, 2: 0.60, 3: 0.80, 4: 0.95
            },
            'momentum_driven': {
                0: 0.25, 1: 0.35, 2: 0.50, 3: 0.70, 4: 0.90
            },
            'conservative': {
                0: 0.40, 1: 0.50, 2: 0.70, 3: 0.85, 4: 0.98
            }
        }
        
        # Context adjustment lookup tables
        self.score_adjustments = {}
        for diff in range(0, 31):
            if diff < 3:
                self.score_adjustments[diff] = -0.10
            elif diff < 7:
                self.score_adjustments[diff] = -0.05
            elif diff < 12:
                self.score_adjustments[diff] = 0.0
            elif diff < 20:
                self.score_adjustments[diff] = 0.05
            else:
                self.score_adjustments[diff] = 0.10
        
        # Pre-calculated sigmoid values for continuous quarters
        self.sigmoid_values = {}
        for q in range(5):
            for t in range(13):  # 0-12 minutes remaining
                continuous_q = q + (1 - (t / 12))
                progress = continuous_q / 4.0
                adjusted_progress = (progress - 0.5) * 8.0  # transition_steepness = 8.0
                sigmoid = 1.0 / (1.0 + np.exp(-adjusted_progress))
                self.sigmoid_values[(q, t)] = sigmoid
    
    def get_ensemble_prediction(self, main_prediction, quarter_prediction, quarter, context=None, strategy=None):
        """
        Get ensemble prediction with optimal performance.
        
        Args:
            main_prediction: Prediction from main model
            quarter_prediction: Prediction from quarter-specific model
            quarter: Current quarter (0-4)
            context: Optional game context dict
            strategy: Optional strategy override
            
        Returns:
            tuple: (prediction, weights_dict, confidence)
        """
        # Default context
        if context is None:
            context = {}
        
        # Extract key context values
        score_diff = min(30, abs(int(context.get('score_differential', 0))))
        momentum = float(context.get('momentum', 0))
        time_remaining = min(12, max(0, int(context.get('time_remaining_in_quarter', 6))))
        
        # Create cache key
        cache_key = (quarter, score_diff, int(momentum * 10), time_remaining, strategy)
        
        # Check cache
        if cache_key in self.weight_cache:
            self.cache_hits += 1
            weights = self.weight_cache[cache_key]
        else:
            self.cache_misses += 1
            
            # Select strategy
            if strategy is None:
                if score_diff < 5 and abs(momentum) > 0.5:
                    strategy = 'momentum_driven'
                elif score_diff > 15:
                    strategy = 'conservative'
                else:
                    strategy = 'balanced'
            
            # Get base weight with pre-calculated sigmoid
            sigmoid = self.sigmoid_values.get((quarter, time_remaining), 0.5)
            base_main_weight = 0.3 + (0.65 * sigmoid)
            
            # Get context adjustments
            score_adjustment = self.score_adjustments.get(score_diff, 0.0)
            momentum_adjustment = -0.15 * abs(momentum)
            
            # Apply progress factor
            game_progress = min(1.0, (quarter * 12 + (12 - time_remaining)) / 48.0)
            progress_factor = 0.5 + (0.5 * game_progress)
            
            # Apply strategy sensitivity
            sensitivity = 1.0 if strategy == 'momentum_driven' else 0.5 if strategy == 'balanced' else 0.3
            
            # Calculate final weight
            total_adjustment = (score_adjustment + momentum_adjustment) * sensitivity * progress_factor
            main_weight = max(0.1, min(0.95, base_main_weight + total_adjustment))
            quarter_weight = 1.0 - main_weight
            
            # Store in weights dictionary
            weights = {
                'main': main_weight,
                'quarter': quarter_weight,
                'strategy': strategy,
                'base_weight': base_main_weight
            }
            
            # Cache result
            self.weight_cache[cache_key] = weights
        
        # Calculate prediction
        prediction = main_prediction * weights['main'] + quarter_prediction * weights['quarter']
        
        # Calculate confidence
        quarter_boost = 0.1 * quarter
        confidence = min(0.95, 0.5 + quarter_boost)
        
        return prediction, weights, confidence

    def get_cache_stats(self):
        """Return cache hit/miss statistics"""
        total = self.cache_hits + self.cache_misses
        hit_rate = (self.cache_hits / total) * 100 if total > 0 else 0
        return {
            'hits': self.cache_hits,
            'misses': self.cache_misses,
            'total': total,
            'hit_rate': f"{hit_rate:.1f}%"
        }


# ================= COMPARISON DEMONSTRATION =================

def compare_ensemble_implementations(num_predictions=1000):
    """
    Compare performance between the full framework and optimized implementation.
    
    Args:
        num_predictions: Number of predictions to generate for comparison
        
    Returns:
        dict: Performance comparison results
    """
    import time
    
    # Sample data
    main_predictions = np.random.normal(110, 5, num_predictions)
    quarter_predictions = np.random.normal(105, 7, num_predictions)
    
    # Generate random game contexts
    quarters = np.random.randint(0, 5, num_predictions)
    score_diffs = np.random.randint(0, 25, num_predictions)
    momentums = np.random.uniform(-1, 1, num_predictions)
    times = np.random.randint(1, 12, num_predictions)
    
    # Create instances
    full_framework = AdaptiveEnsembleFramework()
    optimized = OptimizedAdaptiveEnsemble()
    
    # Test full framework
    start_time = time.time()
    full_results = []
    
    for i in range(num_predictions):
        context = {
            'current_quarter': quarters[i],
            'score_differential': score_diffs[i],
            'momentum': momentums[i],
            'time_remaining_in_quarter': times[i]
        }
        
        pred, weights, conf = full_framework.predict(
            main_predictions[i],
            quarter_predictions[i],
            quarters[i],
            context
        )
        
        full_results.append(pred)
    
    full_time = time.time() - start_time
    
    # Test optimized implementation
    start_time = time.time()
    opt_results = []
    
    for i in range(num_predictions):
        context = {
            'score_differential': score_diffs[i],
            'momentum': momentums[i],
            'time_remaining_in_quarter': times[i]
        }
        
        pred, weights, conf = optimized.get_ensemble_prediction(
            main_predictions[i],
            quarter_predictions[i],
            quarters[i],
            context
        )
        
        opt_results.append(pred)
    
    opt_time = time.time() - start_time
    
    # Calculate differences
    full_results = np.array(full_results)
    opt_results = np.array(opt_results)
    differences = np.abs(full_results - opt_results)
    
    # Prepare results
    results = {
        'full_framework_time': full_time,
        'optimized_time': opt_time,
        'speedup_factor': full_time / opt_time if opt_time > 0 else 0,
        'predictions_per_second': {
            'full_framework': num_predictions / full_time,
            'optimized': num_predictions / opt_time
        },
        'prediction_differences': {
            'mean': np.mean(differences),
            'max': np.max(differences),
            'std': np.std(differences)
        },
        'cache_stats': optimized.get_cache_stats()
    }
    
    return results


# ================= DEMONSTRATION CODE =================

def run_framework_demonstration():
    """Run a demonstration of the ensemble framework features"""
    print("Demonstrating Integrated Adaptive Ensemble Framework")
    
    # Create ensemble
    ensemble = AdaptiveEnsembleFramework()
    
    # Define test scenarios
    test_scenarios = [
        {"name": "Early close game", "quarter": 1, "time": 6, "diff": 3, "momentum": 0.1},
        {"name": "Mid-game blowout", "quarter": 2, "time": 6, "diff": 18, "momentum": 0.2},
        {"name": "Mid-game with strong momentum", "quarter": 2, "time": 6, "diff": 5, "momentum": 0.8},
        {"name": "Late close game", "quarter": 4, "time": 6, "diff": 4, "momentum": 0.3},
        {"name": "Late comeback brewing", "quarter": 4, "time": 3, "diff": 8, "momentum": -0.7}
    ]
    
    # Display smooth transitions
    print("\nVisualizing weight transitions and context effects...")
    ensemble.visualize_weight_transitions()
    
    # Display prediction behavior
    print("\nVisualizing prediction behavior with different strategies...")
    ensemble.visualize_prediction_behavior(main_pred=110, quarter_pred=105)
    
    # Test in various game scenarios
    main_prediction = 110.5
    quarter_prediction = 105.5
    
    print("\nTesting ensemble predictions in different game scenarios:")
    print(f"Main model prediction: {main_prediction}, Quarter model prediction: {quarter_prediction}")
    
    results = []
    for scenario in test_scenarios:
        for strategy in list(ensemble.strategies.keys()) + [None]:  # None = auto-select
            pred, conf, main_w, quarter_w = dynamic_ensemble_predictions(
                main_prediction,
                quarter_prediction,
                scenario["quarter"],
                scenario["diff"],
                scenario["momentum"],
                scenario["time"],
                strategy
            )
            
            results.append({
                'Scenario': scenario["name"],
                'Strategy': strategy if strategy else 'auto',
                'Quarter': scenario["quarter"],
                'Score Diff': scenario["diff"],
                'Momentum': scenario["momentum"],
                'Main Weight': main_w,
                'Quarter Weight': quarter_w,
                'Prediction': pred,
                'Confidence': conf
            })
    
    # Display results
    results_df = pd.DataFrame(results)
    print("\nPrediction results:")
    print(results_df)
    
    # Create sample game context for additional visualization
    sample_context = {
        'game_id': 1001,
        'home_team': 'Boston Celtics',
        'away_team': 'Los Angeles Lakers',
        'current_quarter': 3,
        'score_differential': 6,
        'momentum': 0.4,
        'time_remaining_in_quarter': 6
    }
    
    # Get the selected strategy
    selected_strategy = ensemble.select_strategy(sample_context)
    print(f"\nAutomatically Selected Strategy for sample game: {selected_strategy}")
    
    # Generate prediction with the selected strategy
    ensemble_prediction, weights, confidence = ensemble.predict(
        main_prediction, 
        quarter_prediction,
        sample_context['current_quarter'],
        sample_context
    )
    
    print(f"Ensemble Prediction: {ensemble_prediction:.2f}")
    print(f"Weights: Main={weights['main']:.2f}, Quarter={weights['quarter']:.2f}")
    print(f"Confidence: {confidence*100:.1f}%")
    
    # Visualize strategy comparison
    print("\nGenerating strategy comparison visualization...")
    comparison_fig = ensemble.visualize_strategy_comparison(
        main_prediction,
        quarter_prediction,
        sample_context['current_quarter'],
        sample_context['score_differential'],
        sample_context['momentum']
    )
    plt.figure(comparison_fig.number)
    plt.show()

    # Compare implementations (optional - uncomment to run)
    # print("\nComparing framework and optimized implementations...")
    # comparison_results = compare_ensemble_implementations(1000)
    # print(f"Full Framework Time: {comparison_results['full_framework_time']:.3f} sec")
    # print(f"Optimized Time: {comparison_results['optimized_time']:.3f} sec")
    # print(f"Speedup Factor: {comparison_results['speedup_factor']:.1f}x")
    # print(f"Mean Prediction Difference: {comparison_results['prediction_differences']['mean']:.4f}")
    # print(f"Cache Hit Rate: {comparison_results['cache_stats']['hit_rate']}")


# Run the demonstration if executed directly
if __name__ == "__main__":
    run_framework_demonstration()

In [ ]:
# Cell 4C-8: Hyperparameter Tuning for Context Sensitivity

def evaluate_weight_function(validation_data, weight_func, metric='combined_rmse'):
    """
    Evaluate a weight function on validation data
    
    Args:
        validation_data: DataFrame with validation game data
        weight_func: Function that returns weights given game context
        metric: Metric to evaluate ('combined_rmse', 'winner_accuracy', etc.)
        
    Returns:
        float: Performance score (lower is better for error metrics)
    """
    errors = []
    winner_correct = 0
    total_games = len(validation_data)
    
    for _, game in validation_data.iterrows():
        # Extract game context
        quarter = game.get('current_quarter', 0)
        time_remaining = game.get('time_remaining_mins', 48 - (quarter * 12))
        score_diff = abs(game.get('home_score', 0) - game.get('away_score', 0))
        momentum = game.get('momentum', 0)
        
        # Get weights using the provided function
        weights = weight_func(quarter, time_remaining, score_diff, momentum)
        
        # Get main and quarter predictions
        main_home_pred = game.get('main_home_pred', 0)
        main_away_pred = game.get('main_away_pred', 0)
        quarter_home_pred = game.get('quarter_home_pred', 0)
        quarter_away_pred = game.get('quarter_away_pred', 0)
        
        # Get actual values
        actual_home = game.get('actual_home_score', 0)
        actual_away = game.get('actual_away_score', 0)
        
        # Calculate ensemble predictions
        ensemble_home = (main_home_pred * weights['main_model'] + 
                         quarter_home_pred * weights['quarter_model'])
        ensemble_away = (main_away_pred * weights['main_model'] + 
                         quarter_away_pred * weights['quarter_model'])
        
        # Calculate errors
        home_error = ensemble_home - actual_home
        away_error = ensemble_away - actual_away
        errors.append(home_error**2)
        errors.append(away_error**2)
        
        # Track winner prediction accuracy
        predicted_winner_home = ensemble_home > ensemble_away
        actual_winner_home = actual_home > actual_away
        if predicted_winner_home == actual_winner_home:
            winner_correct += 1
    
    # Calculate specified metric
    if metric == 'combined_rmse':
        return np.sqrt(np.mean(errors))
    elif metric == 'winner_accuracy':
        return 1 - (winner_correct / total_games)  # Return error rate for consistency
    elif metric == 'combined':
        # Combined metric (70% RMSE, 30% winner accuracy)
        rmse = np.sqrt(np.mean(errors))
        acc_error = 1 - (winner_correct / total_games)
        return (0.7 * rmse) + (0.3 * acc_error * 100)  # Scale accuracy error
    else:
        # Default to RMSE
        return np.sqrt(np.mean(errors))

def optimize_dynamic_weights(validation_data, metric='combined_rmse'):
    """
    Optimize parameters for dynamic weight calculation using grid search
    """
    # Parameters to tune
    closeness_thresholds = [5, 8, 10, 12]
    momentum_impacts = [0.05, 0.1, 0.15, 0.2]
    sigmoid_steepness = [5, 10, 15]
    sigmoid_midpoints = [0.4, 0.5, 0.6]
    
    best_score = float('inf')
    best_params = {}
    results = []
    
    # Simple grid search
    for threshold in closeness_thresholds:
        for impact in momentum_impacts:
            for steepness in sigmoid_steepness:
                for midpoint in sigmoid_midpoints:
                    # Create a modified weight function with these parameters
                    def test_weight_func(quarter, time_remaining, score_diff, momentum):
                        game_progress = (48 - time_remaining) / 48.0
                        main_weight = 0.3 + (0.7 * (1 / (1 + np.exp(-steepness * (game_progress - midpoint)))))
                        
                        # Closeness adjustment
                        closeness_factor = max(0.0, (threshold - score_diff) / threshold) * impact
                        
                        # Apply adjustments (simplified version)
                        if score_diff < threshold:
                            main_weight = max(0.2, main_weight - closeness_factor)
                        
                        return {'main_model': main_weight, 'quarter_model': 1 - main_weight}
                    
                    # Evaluate this configuration
                    score = evaluate_weight_function(validation_data, test_weight_func, metric)
                    
                    results.append({
                        'threshold': threshold,
                        'impact': impact,
                        'steepness': steepness,
                        'midpoint': midpoint,
                        'score': score
                    })
                    
                    if score < best_score:
                        best_score = score
                        best_params = {
                            'threshold': threshold,
                            'impact': impact,
                            'steepness': steepness,
                            'midpoint': midpoint
                        }
    
    return pd.DataFrame(results), best_params

# Test with dummy validation data
def create_dummy_validation_data(n_samples=10):
    """Create dummy validation data for testing the optimization function"""
    np.random.seed(42)
    data = []
    
    for i in range(n_samples):
        # Create a random game state
        quarter = np.random.randint(0, 5)
        time_remaining = max(0, 48 - (quarter * 12) - np.random.randint(0, 12))
        home_score = np.random.randint(quarter * 20, quarter * 30 + 10) if quarter > 0 else 0
        away_score = np.random.randint(quarter * 20, quarter * 30 + 10) if quarter > 0 else 0
        score_diff = abs(home_score - away_score)
        momentum = np.random.uniform(-1, 1)
        
        # Generate predictions
        main_home_pred = 105 + np.random.normal(0, 5)
        main_away_pred = 102 + np.random.normal(0, 5)
        quarter_home_pred = 108 + np.random.normal(0, 8)
        quarter_away_pred = 101 + np.random.normal(0, 8)
        
        # Actual scores (with some error relative to main predictions)
        actual_home = main_home_pred + np.random.normal(0, 3)
        actual_away = main_away_pred + np.random.normal(0, 3)
        
        data.append({
            'game_id': i,
            'current_quarter': quarter,
            'time_remaining_mins': time_remaining,
            'home_score': home_score,
            'away_score': away_score,
            'momentum': momentum,
            'main_home_pred': main_home_pred,
            'main_away_pred': main_away_pred,
            'quarter_home_pred': quarter_home_pred,
            'quarter_away_pred': quarter_away_pred,
            'actual_home_score': actual_home,
            'actual_away_score': actual_away
        })
    
    return pd.DataFrame(data)

# Run a quick test with limited parameter combinations
print("Testing hyperparameter optimization with small dummy dataset...")
dummy_data = create_dummy_validation_data()
results_df, best_params = optimize_dynamic_weights(dummy_data)
print(f"\nBest parameters found: {best_params}")
print("\nTop 5 parameter combinations:")
display(results_df.sort_values('score').head(5))

In [ ]:
# Cell 4D: Comprehensive Quarter-Specific Model Benchmarking

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
import time
import warnings
warnings.filterwarnings('ignore')

# Try to import optional libraries
try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not available. Install with: pip install xgboost")

try:
    from sklearn.neural_network import MLPRegressor
    NEURAL_NET_AVAILABLE = True
except ImportError:
    NEURAL_NET_AVAILABLE = False
    print("Neural network not available.")

def create_quarter_stratified_samples(features_df, train_fraction=0.8, random_state=42):
    """
    Create stratified train-test splits for each quarter to ensure fair comparison.
    
    Args:
        features_df: DataFrame with all features
        train_fraction: Fraction of data to use for training
        random_state: Random seed for reproducibility
        
    Returns:
        dict: Quarter-specific train-test splits
    """
    print("Creating quarter-stratified train-test splits...")
    
    # Use game date as a proxy for stratification if available
    if 'game_date' in features_df.columns:
        features_df['game_date'] = pd.to_datetime(features_df['game_date'], errors='coerce')
        # Create year-month strata for time-based stratification
        features_df['year_month'] = features_df['game_date'].dt.strftime('%Y-%m')
        strata_col = 'year_month'
    else:
        # Create random strata if no date is available
        features_df['random_strata'] = np.random.RandomState(random_state).randint(0, 10, size=len(features_df))
        strata_col = 'random_strata'
    
    # Initialize result dictionary
    quarter_splits = {}
    
    # Ensure unique indices for validation
    features_df = features_df.reset_index(drop=True)
    
    # Create quarter-specific splits
    for quarter in ['q1', 'q2', 'q3', 'q4']:
        # Determine target column
        target_col = f'home_{quarter}'
        
        # Skip if target doesn't exist
        if target_col not in features_df.columns:
            print(f"Warning: Target column {target_col} not found. Skipping {quarter}.")
            continue
        
        # Skip rows with missing targets
        valid_mask = ~features_df[target_col].isna()
        valid_df = features_df[valid_mask]
        
        if len(valid_df) < 10:
            print(f"Warning: Not enough valid samples for {quarter}. Need at least 10, found {len(valid_df)}.")
            continue
        
        # Create strata for stratified sampling
        strata = valid_df[strata_col].astype(str)
        
        # Use StratifiedKFold to create the train-test split
        skf = StratifiedKFold(n_splits=int(1//(1-train_fraction)), shuffle=True, random_state=random_state)
        train_idx, test_idx = next(skf.split(valid_df, strata))
        
        # Store indices
        quarter_splits[quarter] = {
            'train_idx': valid_df.index[train_idx],
            'test_idx': valid_df.index[test_idx],
            'target_col': target_col
        }
        
        print(f"  {quarter.upper()}: {len(train_idx)} train samples, {len(test_idx)} test samples")
    
    return quarter_splits

def get_quarter_feature_sets():
    """
    Define feature sets for each quarter, with common and quarter-specific features.
    
    Returns:
        dict: Quarter-specific feature lists
    """
    # Common features used across all quarter models
    common_features = [
        'prev_matchup_diff',
        'rest_advantage',
        'is_back_to_back_home',
        'is_back_to_back_away'
    ]
    
    # Quarter-specific feature sets
    quarter_features = {
        'q1': common_features + [
            'rolling_home_score',
            'rolling_away_score'
        ],
        
        'q2': common_features + [
            'home_q1',
            'away_q1',
            'rolling_home_score',
            'rolling_away_score',
            'q1_to_q2_momentum'
        ],
        
        'q3': common_features + [
            'home_q1',
            'home_q2',
            'away_q1',
            'away_q2',
            'q1_to_q2_momentum',
            'q2_to_q3_momentum',
            'first_half_diff'
        ],
        
        'q4': common_features + [
            'home_q1',
            'home_q2',
            'home_q3',
            'away_q1',
            'away_q2',
            'away_q3',
            'q1_to_q2_momentum',
            'q2_to_q3_momentum',
            'q3_to_q4_momentum',
            'pre_q4_diff',
            'cumulative_momentum'
        ]
    }
    
    return quarter_features

def get_algorithm_set():
    """
    Create a comprehensive set of algorithms to evaluate.
    
    Returns:
        dict: Mapping of algorithm names to initialized objects
    """
    # Base algorithms (always available)
    algorithms = {
        'LinearRegression': LinearRegression(),
        'Ridge': Ridge(alpha=1.0, random_state=42),
        'Lasso': Lasso(alpha=0.1, random_state=42),
        'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42),
        'RandomForest': RandomForestRegressor(
            n_estimators=100,
            max_depth=8,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=42
        ),
        'GradientBoosting': GradientBoostingRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=5,
            subsample=0.8,
            random_state=42
        )
    }
    
    # Add XGBoost if available
    if XGBOOST_AVAILABLE:
        algorithms['XGBoost'] = XGBRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        )
    
    # Add Neural Network if available
    if NEURAL_NET_AVAILABLE:
        algorithms['NeuralNetwork'] = MLPRegressor(
            hidden_layer_sizes=(50, 25),
            activation='relu',
            solver='adam',
            alpha=0.001,
            max_iter=500,
            early_stopping=True,
            random_state=42
        )
    
    return algorithms

def benchmark_quarter_models(features_df, cv_folds=5, use_stratification=True):
    """
    Comprehensive benchmarking of quarter-specific models with consistent evaluation.
    
    Args:
        features_df: DataFrame with features and targets
        cv_folds: Number of cross-validation folds
        use_stratification: Whether to use stratified sampling
        
    Returns:
        dict: Benchmark results by quarter and algorithm
    """
    print("Starting comprehensive quarter-specific model benchmarking...")
    
    # Get quarter feature sets
    quarter_feature_sets = get_quarter_feature_sets()
    
    # Get algorithms
    algorithms = get_algorithm_set()
    print(f"Using {len(algorithms)} algorithms: {', '.join(algorithms.keys())}")
    
    # If using stratification, ensure features_df index is reset
    if use_stratification:
        features_df = features_df.reset_index(drop=True)
        quarter_splits = create_quarter_stratified_samples(features_df)
    else:
        quarter_splits = None
    
    # Prepare result storage
    results = []
    detailed_results = {}
    
    # Define consistent metrics to evaluate
    scoring = {
        'rmse': 'neg_root_mean_squared_error',
        'mae': 'neg_mean_absolute_error',
        'r2': 'r2'
    }
    
    # Process each quarter
    for quarter, feature_list in quarter_feature_sets.items():
        print(f"\nEvaluating {quarter.upper()} models...")
        
        # Determine target column
        target_col = f'home_{quarter}'
        
        # Check if target exists
        if target_col not in features_df.columns:
            print(f"Target column {target_col} not found. Skipping {quarter}.")
            continue
        
        # Check for missing features
        missing_features = [f for f in feature_list if f not in features_df.columns]
        if missing_features:
            print(f"Warning: Missing features for {quarter}: {missing_features}")
            feature_list = [f for f in feature_list if f in features_df.columns]
        
        if not feature_list:
            print(f"No valid features for {quarter}. Skipping.")
            continue
        
        # Get data for this quarter
        if use_stratification and quarter in quarter_splits:
            # Use stratified samples
            train_idx = quarter_splits[quarter]['train_idx']
            test_idx = quarter_splits[quarter]['test_idx']
            
            train_df = features_df.loc[train_idx]
            test_df = features_df.loc[test_idx]
            
            X_train = train_df[feature_list]
            y_train = train_df[target_col]
            X_test = test_df[feature_list]
            y_test = test_df[target_col]
            
            cv_data = features_df.loc[train_idx]
            X_cv = cv_data[feature_list]
            y_cv = cv_data[target_col]
        else:
            # Use all data with regular cross-validation
            valid_mask = ~features_df[target_col].isna()
            valid_df = features_df[valid_mask]
            
            X_cv = valid_df[feature_list]
            y_cv = valid_df[target_col]
            
            # Create 80/20 split for final testing
            train_size = int(0.8 * len(valid_df))
            train_df = valid_df.iloc[:train_size]
            test_df = valid_df.iloc[train_size:]
            
            X_train = train_df[feature_list]
            y_train = train_df[target_col]
            X_test = test_df[feature_list]
            y_test = test_df[target_col]
        
        # Ensure no NaN values
        X_train = X_train.fillna(0)
        y_train = y_train.fillna(y_train.mean())
        X_test = X_test.fillna(0)
        y_test = y_test.fillna(y_test.mean())
        X_cv = X_cv.fillna(0)
        y_cv = y_cv.fillna(y_cv.mean())
        
        print(f"  Training on {len(X_train)} samples, testing on {len(X_test)} samples")
        print(f"  Features: {feature_list}")
        
        # Standard Scaler for preprocessing
        scaler = StandardScaler()
        
        # Benchmark each algorithm
        quarter_results = {}
        
        for alg_name, algorithm in algorithms.items():
            print(f"  Evaluating {alg_name}...")
            
            try:
                # Create pipeline with scaling
                pipeline = Pipeline([
                    ('', scaler),
                    ('model', algorithm)
                ])
                
                # Cross-validation
                start_time = time.time()
                cv_results = cross_validate(
                    pipeline, X_cv, y_cv, 
                    cv=cv_folds, 
                    scoring=scoring,
                    return_train_score=True,
                    n_jobs=-1
                )
                cv_time = time.time() - start_time
                
                # Train on full train set and evaluate on test set
                pipeline.fit(X_train, y_train)
                y_pred = pipeline.predict(X_test)
                
                # Calculate test metrics
                test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
                test_mae = mean_absolute_error(y_test, y_pred)
                test_r2 = r2_score(y_test, y_pred)
                
                # Transform CV results (negative scores back to positive)
                cv_rmse = -cv_results['test_rmse'].mean()
                cv_rmse_std = cv_results['test_rmse'].std()
                cv_mae = -cv_results['test_mae'].mean()
                cv_r2 = cv_results['test_r2'].mean()
                
                # Store results
                alg_result = {
                    'Quarter': quarter.upper(),
                    'Algorithm': alg_name,
                    'CV_RMSE': cv_rmse,
                    'CV_RMSE_STD': cv_rmse_std,
                    'CV_MAE': cv_mae,
                    'CV_R2': cv_r2,
                    'Test_RMSE': test_rmse,
                    'Test_MAE': test_mae,
                    'Test_R2': test_r2,
                    'CV_Time': cv_time,
                    'Feature_Count': len(feature_list)
                }
                
                # Store detailed results including model
                quarter_results[alg_name] = {
                    'metrics': alg_result,
                    'model': pipeline,
                    'feature_list': feature_list,
                    'feature_importance': get_feature_importance(pipeline, feature_list)
                }
                
                # Add to overall results
                results.append(alg_result)
                
                print(f"    CV RMSE: {cv_rmse:.2f} (±{cv_rmse_std:.2f})")
                print(f"    Test RMSE: {test_rmse:.2f}, Test R²: {test_r2:.3f}")
                
            except Exception as e:
                print(f"    Error evaluating {alg_name}: {str(e)}")
        
        # Store all results for this quarter
        detailed_results[quarter] = quarter_results
    
    # Convert results to DataFrame
    results_df = pd.DataFrame(results)
    
    # Create visualizations
    if not results_df.empty:
        create_benchmark_visualizations(results_df)
    
    return {
        'summary': results_df,
        'detailed': detailed_results
    }

def get_feature_importance(pipeline, feature_list):
    """Extract feature importance if the model supports it"""
    model = pipeline.named_steps.get('model')
    
    if hasattr(model, 'feature_importances_'):
        return dict(zip(feature_list, model.feature_importances_))
    elif hasattr(model, 'coef_'):
        # For linear models, use absolute coefficient values
        if len(model.coef_.shape) > 1:
            importance = np.abs(model.coef_[0])
        else:
            importance = np.abs(model.coef_)
        return dict(zip(feature_list, importance))
    else:
        return None

def create_benchmark_visualizations(results_df):
    """Create comprehensive visualizations of benchmark results"""
    # Set up the figure layout
    plt.figure(figsize=(20, 12))
    
    # 1. RMSE by quarter and algorithm
    plt.subplot(2, 2, 1)
    quarter_order = ['Q1', 'Q2', 'Q3', 'Q4']
    
    # Plot with error bars using seaborn
    sns.barplot(
        x='Quarter', 
        y='CV_RMSE', 
        hue='Algorithm',
        data=results_df,
        order=quarter_order,
        errorbar=('ci', 95)
    )
    plt.title('Cross-Validation RMSE by Quarter and Algorithm')
    plt.xlabel('Quarter')
    plt.ylabel('RMSE (Cross-Validation)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(axis='y', alpha=0.3)
    
    # 2. R² by quarter and algorithm
    plt.subplot(2, 2, 2)
    sns.barplot(
        x='Quarter', 
        y='CV_R2', 
        hue='Algorithm',
        data=results_df,
        order=quarter_order
    )
    plt.title('Cross-Validation R² by Quarter and Algorithm')
    plt.xlabel('Quarter')
    plt.ylabel('R² Score (Cross-Validation)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(axis='y', alpha=0.3)
    
    # 3. Radar chart of performance metrics for best models
    plt.subplot(2, 2, 3)
    
    # Get the best model for each quarter based on RMSE
    best_models = results_df.loc[results_df.groupby('Quarter')['CV_RMSE'].idxmin()]
    
    # Normalize metrics for radar chart
    metrics = ['CV_RMSE', 'CV_MAE', 'CV_Time']
    normalized_metrics = best_models[metrics].copy()
    
    # Invert RMSE and MAE so higher is better
    for col in ['CV_RMSE', 'CV_MAE']:
        max_val = normalized_metrics[col].max() * 1.1  # Add 10% padding
        normalized_metrics[col] = max_val - normalized_metrics[col]
    
    # Normalize each column to 0-1 scale
    for col in metrics:
        min_val = normalized_metrics[col].min()
        max_val = normalized_metrics[col].max()
        range_val = max_val - min_val
        if range_val > 0:
            normalized_metrics[col] = (normalized_metrics[col] - min_val) / range_val
        else:
            normalized_metrics[col] = 0.5  # Default if all values are the same
    
    # Prepare radar chart
    quarters = best_models['Quarter'].tolist()
    algorithms = best_models['Algorithm'].tolist()
    
    # Number of variables
    N = len(metrics)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]  # Close the loop
    
    ax = plt.subplot(2, 2, 3, polar=True)
    
    # Add lines and labels for each quarter
    for i, quarter in enumerate(quarters):
        values = normalized_metrics.iloc[i].tolist()
        values += values[:1]  # Close the loop
        
        # Plot values
        ax.plot(angles, values, linewidth=2, linestyle='solid', label=f"{quarter} ({algorithms[i]})")
        ax.fill(angles, values, alpha=0.1)
    
    # Set labels
    metric_labels = ['RMSE (inverted)', 'MAE (inverted)', 'Speed (inverted)']
    plt.xticks(angles[:-1], metric_labels)
    plt.yticks([0.2, 0.4, 0.6, 0.8], ["0.2", "0.4", "0.6", "0.8"], color="grey", size=8)
    plt.ylim(0, 1)
    
    plt.title('Best Models Comparison by Quarter')
    plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
    
    # 4. Model performance improvement by quarter
    plt.subplot(2, 2, 4)
    
    # Group by algorithm and quarter, then calculate mean of CV_RMSE
    grouped = results_df.groupby(['Algorithm', 'Quarter'])['CV_RMSE'].mean().reset_index()
    
    # Calculate average improvement from Q1 to Q4 for each algorithm
    improvements = []
    algorithms_unique = grouped['Algorithm'].unique()
    
    for alg in algorithms_unique:
        alg_data = grouped[grouped['Algorithm'] == alg]
        q1_rmse = alg_data[alg_data['Quarter'] == 'Q1']['CV_RMSE'].values
        q4_rmse = alg_data[alg_data['Quarter'] == 'Q4']['CV_RMSE'].values
        
        if len(q1_rmse) > 0 and len(q4_rmse) > 0:
            improvement_pct = (q1_rmse[0] - q4_rmse[0]) / q1_rmse[0] * 100
            improvements.append({
                'Algorithm': alg,
                'Improvement': improvement_pct
            })
    
    improvements_df = pd.DataFrame(improvements)
    
    if not improvements_df.empty:
        sns.barplot(x='Algorithm', y='Improvement', data=improvements_df)
        plt.title('RMSE Improvement from Q1 to Q4 by Algorithm (%)')
        plt.xlabel('Algorithm')
        plt.ylabel('Improvement (%)')
        plt.xticks(rotation=45)
        plt.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Summary table of best models
    print("\nBest Model for Each Quarter:")
    display(best_models[['Quarter', 'Algorithm', 'CV_RMSE', 'Test_RMSE', 'CV_R2', 'Test_R2']])

# Run the benchmark
if __name__ == "__main__":
    if 'features_df' in globals() and not features_df.empty:
        benchmark_results = benchmark_quarter_models(features_df, cv_folds=5)
        
        if 'summary' in benchmark_results and not benchmark_results['summary'].empty:
            print("\nAll Results:")
            display(benchmark_results['summary'])
            
            # Identify best model for each quarter
            print("\nRecommended Models by Quarter:")
            for quarter in ['Q1', 'Q2', 'Q3', 'Q4']:
                quarter_data = benchmark_results['summary'][benchmark_results['summary']['Quarter'] == quarter]
                if not quarter_data.empty:
                    best_idx = quarter_data['CV_RMSE'].idxmin()
                    best = quarter_data.loc[best_idx]
                    print(f"  {quarter}: {best['Algorithm']} (RMSE={best['CV_RMSE']:.2f}, R²={best['CV_R2']:.3f})")
    else:
        print("No features data available. Please generate features first.")


In [ ]:
# Cell 4D-2: Comprehensive Alternative Algorithm Evaluation for Quarter-Specific Models

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import time  

def evaluate_quarter_specific_algorithms(features_df, target_col='home_score'):
    """
    Systematically evaluate different algorithms for different quarters
    """
    # First check if XGBoost is available
    try:
        from xgboost import XGBRegressor
        xgboost_available = True
    except ImportError:
        print("XGBoost is not installed. Skipping XGBoost evaluation.")
        xgboost_available = False
    
    # Define algorithms to test
    algorithms = {
        'GradientBoosting': GradientBoostingRegressor(
            n_estimators=100, 
            learning_rate=0.1,
            max_depth=4,
            subsample=0.8,
            random_state=42
        ),
        'Ridge': Ridge(alpha=1.0, random_state=42),
        'RandomForest': RandomForestRegressor(
            n_estimators=100, 
            max_depth=5,
            min_samples_split=10,
            max_features='sqrt',
            random_state=42
        ),
        'LinearRegression': LinearRegression()
    }
    
    if xgboost_available:
        algorithms['XGBoost'] = XGBRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=4,
            subsample=0.8,
            random_state=42
        )
    
    # Define quarter-specific feature sets
    quarter_features = {
        1: ['rest_days_home', 'rest_days_away', 'is_back_to_back_home', 'is_back_to_back_away',
            'prev_matchup_diff', 'rolling_home_score', 'rolling_away_score'],
        2: ['home_q1', 'away_q1', 'rest_advantage', 'prev_matchup_diff',
            'rolling_home_score', 'rolling_away_score', 'q1_to_q2_momentum'],
        3: ['home_q1', 'home_q2', 'away_q1', 'away_q2', 'q1_to_q2_momentum', 
            'q2_to_q3_momentum', 'first_half_diff', 'rest_advantage'],
        4: ['home_q1', 'home_q2', 'home_q3', 'away_q1', 'away_q2', 'away_q3',
            'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 
            'pre_q4_diff', 'cumulative_momentum']
    }
    
    # Prepare results storage
    results = []
    
    print("Starting comprehensive algorithm evaluation for each quarter...")
    
    # Evaluate each quarter
    for quarter, features in quarter_features.items():
        print(f"\nEvaluating models for Quarter {quarter}:")
        
        # Check if all features exist
        missing_features = [f for f in features if f not in features_df.columns]
        if missing_features:
            print(f"Warning: Missing features for Quarter {quarter}: {missing_features}")
            continue
        
        # Prepare data for this quarter
        X = features_df[features].copy()
        q_target = f"home_q{quarter}"
        if q_target not in features_df.columns:
            print(f"Warning: Target column {q_target} not found. Skipping Quarter {quarter}.")
            continue
        
        y = features_df[q_target]
        
        # Handle missing values
        X = X.fillna(0)
        y = y.fillna(y.mean())
        
        # Train-test split (chronological)
        train_size = int(0.8 * len(X))
        X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
        y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]
        
        # Evaluate each algorithm
        for alg_name, algorithm in algorithms.items():
            try:
                print(f"  - Training {alg_name}...", end="")
                
                # Time the training
                start_time = time.time()
                algorithm.fit(X_train, y_train)
                train_time = time.time() - start_time
                
                # Make predictions
                y_train_pred = algorithm.predict(X_train)
                y_test_pred = algorithm.predict(X_test)
                
                # Calculate metrics
                train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
                test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
                test_mae = mean_absolute_error(y_test, y_test_pred)
                r2 = r2_score(y_test, y_test_pred)
                
                print(f" Done. Test RMSE: {test_rmse:.2f}, R²: {r2:.3f}, Time: {train_time:.2f}s")
                
                # Get feature importance if available
                if hasattr(algorithm, 'feature_importances_'):
                    importance = dict(zip(features, algorithm.feature_importances_))
                    top_features = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:3]
                    top_features_str = ', '.join([f"{f} ({v:.3f})" for f, v in top_features])
                elif hasattr(algorithm, 'coef_'):
                    importance = dict(zip(features, np.abs(algorithm.coef_)))
                    top_features = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:3]
                    top_features_str = ', '.join([f"{f} ({v:.3f})" for f, v in top_features])
                else:
                    top_features_str = "N/A"
                
                # Store result
                results.append({
                    'Quarter': f"Q{quarter}",
                    'Algorithm': alg_name,
                    'Train RMSE': train_rmse,
                    'Test RMSE': test_rmse,
                    'Test MAE': test_mae,
                    'R²': r2,
                    'Training Time (s)': train_time,
                    'Top Features': top_features_str
                })
                
            except Exception as e:
                print(f" Error: {str(e)}")
                continue
    
    # Convert to DataFrame
    results_df = pd.DataFrame(results)
    
    # Visualize results
    if not results_df.empty:
        plt.figure(figsize=(15, 10))
        
        # Plot comparison by quarter
        quarters = sorted(results_df['Quarter'].unique())
        algorithms = sorted(results_df['Algorithm'].unique())
        
        # Create subplots for RMSE and R²
        plt.subplot(2, 1, 1)
        
        # Plot grouped by algorithm within each quarter
        bar_width = 0.8 / len(algorithms)
        x = np.arange(len(quarters))
        
        for i, alg in enumerate(algorithms):
            alg_data = results_df[results_df['Algorithm'] == alg]
            alg_by_quarter = {q: alg_data[alg_data['Quarter'] == q]['Test RMSE'].values[0] 
                             if not alg_data[alg_data['Quarter'] == q].empty else np.nan 
                             for q in quarters}
            values = [alg_by_quarter[q] for q in quarters]
            positions = x + (i - len(algorithms)/2 + 0.5) * bar_width
            plt.bar(positions, values, width=bar_width, label=alg)
            
            # Add value labels
            for pos, val in zip(positions, values):
                if not np.isnan(val):
                    plt.text(pos, val + 0.1, f'{val:.2f}', ha='center', va='bottom', fontsize=8)
        
        plt.xlabel('Quarter')
        plt.ylabel('Test RMSE (lower is better)')
        plt.title('Algorithm Comparison by Quarter - RMSE')
        plt.xticks(x, quarters)
        plt.legend()
        plt.grid(axis='y', alpha=0.3)
        
        # R² plot
        plt.subplot(2, 1, 2)
        
        for i, alg in enumerate(algorithms):
            alg_data = results_df[results_df['Algorithm'] == alg]
            alg_by_quarter = {q: alg_data[alg_data['Quarter'] == q]['R²'].values[0] 
                             if not alg_data[alg_data['Quarter'] == q].empty else np.nan 
                             for q in quarters}
            values = [alg_by_quarter[q] for q in quarters]
            positions = x + (i - len(algorithms)/2 + 0.5) * bar_width
            plt.bar(positions, values, width=bar_width, label=alg)
            
            # Add value labels
            for pos, val in zip(positions, values):
                if not np.isnan(val):
                    plt.text(pos, val + 0.01, f'{val:.3f}', ha='center', va='bottom', fontsize=8)
        
        plt.xlabel('Quarter')
        plt.ylabel('R² Score (higher is better)')
        plt.title('Algorithm Comparison by Quarter - R²')
        plt.xticks(x, quarters)
        plt.legend()
        plt.grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    return results_df

# Run the evaluation using features_df from the existing dataset
print("Running comprehensive algorithm evaluation...")
algorithm_comparison = evaluate_quarter_specific_algorithms(features_df)

# Display detailed results
print("\nDetailed Algorithm Comparison Results:")
display(algorithm_comparison)

In [ ]:
# Cell 4D-3: Enhanced XGBoost Integration and Feature Selection

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_regression, SelectFromModel, RFECV
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
import traceback
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Improved XGBoost dependency handling
def ensure_xgboost():
    """
    Ensure XGBoost is available, with detailed error handling and installation instructions.
    
    Returns:
        tuple: (xgboost_available, xgb_module, error_message)
    """
    try:
        import xgboost as xgb
        return True, xgb, None
    except ImportError as e:
        error_message = f"""
XGBoost is not installed. This will limit feature selection capabilities.
To install XGBoost, run one of these commands:

pip install xgboost
conda install -c conda-forge xgboost

Note: On some systems, you may need additional dependencies. See:
https://xgboost.readthedocs.io/en/latest/install.html
        """
        print(error_message)
        return False, None, error_message

# Check XGBoost availability
xgboost_available, xgb, xgb_error = ensure_xgboost()

class AdvancedFeatureSelector:
    """
    Advanced feature selection system that uses multiple methods to identify optimal features.
    Combines mutual information, model-based selection, and recursive feature elimination.
    """
    
    def __init__(self, debug=True):
        """
        Initialize feature selector.
        
        Args:
            debug: Whether to print debug information
        """
        self.debug = debug
        self.xgboost_available = xgboost_available
        self.feature_importances = {}
        self.selected_features = {}
        self.performance_metrics = {}
    
    def print_debug(self, message):
        """Print debug message if debug mode is on"""
        if self.debug:
            print(message)
    
    def analyze_feature_importance(self, X, y, method='mutual_info'):
        """
        Calculate feature importance using specified method.
        
        Args:
            X: Feature DataFrame
            y: Target Series
            method: Method to use ('mutual_info', 'xgboost', 'ridge')
            
        Returns:
            DataFrame with feature importances
        """
        feature_names = X.columns.tolist()
        
        # Mutual information (model-agnostic)
        if method == 'mutual_info' or method == 'all':
            self.print_debug("Calculating mutual information...")
            mi_scores = mutual_info_regression(X, y)
            mi_importances = pd.DataFrame({
                'Feature': feature_names,
                'Importance': mi_scores
            }).sort_values('Importance', ascending=False)
            
            if method == 'mutual_info':
                return mi_importances
        
        # XGBoost importance
        if (method == 'xgboost' or method == 'all') and self.xgboost_available:
            self.print_debug("Calculating XGBoost feature importance...")
            try:
                xgb_model = xgb.XGBRegressor(
                    n_estimators=100,
                    learning_rate=0.1,
                    max_depth=3,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    random_state=42
                )
                xgb_model.fit(X, y)
                xgb_importances = pd.DataFrame({
                    'Feature': feature_names,
                    'Importance': xgb_model.feature_importances_
                }).sort_values('Importance', ascending=False)
                
                if method == 'xgboost':
                    return xgb_importances
            except Exception as e:
                self.print_debug(f"XGBoost importance calculation failed: {e}")
                traceback.print_exc()
        
        # Ridge regression coefficients
        if method == 'ridge' or method == 'all':
            self.print_debug("Calculating Ridge coefficient magnitudes...")
            try:
                ridge_model = Ridge(alpha=1.0, random_state=42)
                ridge_model.fit(X, y)
                ridge_importances = pd.DataFrame({
                    'Feature': feature_names,
                    'Importance': np.abs(ridge_model.coef_)
                }).sort_values('Importance', ascending=False)
                
                if method == 'ridge':
                    return ridge_importances
            except Exception as e:
                self.print_debug(f"Ridge importance calculation failed: {e}")
                traceback.print_exc()
        
        # If 'all' was selected, combine importances using rank aggregation
        if method == 'all':
            self.print_debug("Combining feature importances using rank aggregation...")
            importance_dfs = []
            
            if 'mi_importances' in locals():
                importance_dfs.append(mi_importances.copy())
                
            if 'xgb_importances' in locals():
                importance_dfs.append(xgb_importances.copy())
                
            if 'ridge_importances' in locals():
                importance_dfs.append(ridge_importances.copy())
            
            # Combine using rank aggregation
            combined_ranks = pd.DataFrame({'Feature': feature_names})
            
            for i, df in enumerate(importance_dfs):
                df = df.copy()
                df[f'Rank_{i}'] = df['Importance'].rank(ascending=False)
                combined_ranks = combined_ranks.merge(
                    df[['Feature', f'Rank_{i}']], on='Feature', how='left'
                )
            
            # Calculate mean rank
            rank_columns = [col for col in combined_ranks.columns if col.startswith('Rank_')]
            combined_ranks['Mean_Rank'] = combined_ranks[rank_columns].mean(axis=1)
            
            # Sort by mean rank
            combined_ranks = combined_ranks.sort_values('Mean_Rank')
            
            # Convert to importance score (inverse of rank)
            combined_ranks['Importance'] = 1 / combined_ranks['Mean_Rank']
            
            return combined_ranks[['Feature', 'Importance']].sort_values('Importance', ascending=False)
        
        # Fallback to mutual information if other methods failed
        return mi_importances if 'mi_importances' in locals() else None
    
    def select_features(self, X, y, quarter, n_features=None, method='hybrid'):
        """
        Select optimal features using the specified method.
        
        Args:
            X: Feature DataFrame
            y: Target Series
            quarter: Quarter number (for tracking)
            n_features: Number of features to select (None for auto)
            method: Selection method ('mutual_info', 'model_based', 'recursive', 'hybrid')
            
        Returns:
            list: Selected feature names
        """
        feature_names = X.columns.tolist()
        quarter_key = f'q{quarter}'
        
        self.print_debug(f"Selecting features for quarter {quarter} using {method} method...")
        
        if method == 'mutual_info':
            # Use mutual information to select top features
            importances = self.analyze_feature_importance(X, y, method='mutual_info')
            
            if importances is not None:
                n_to_select = n_features if n_features is not None else max(3, len(feature_names) // 2)
                selected = importances.head(n_to_select)['Feature'].tolist()
                
                self.feature_importances[quarter_key] = importances
                self.selected_features[quarter_key] = selected
                
                return selected
            
        elif method == 'model_based' and self.xgboost_available:
            # Use XGBoost model-based selection
            try:
                xgb_model = xgb.XGBRegressor(
                    n_estimators=100,
                    learning_rate=0.1,
                    max_depth=3,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    random_state=42
                )
                xgb_model.fit(X, y)
                
                # Use SelectFromModel
                selector = SelectFromModel(
                    xgb_model, 
                    threshold='median', 
                    prefit=True
                )
                
                selected_mask = selector.get_support()
                selected = [feature for feature, selected in zip(feature_names, selected_mask) if selected]
                
                # Store importances
                importances = pd.DataFrame({
                    'Feature': feature_names,
                    'Importance': xgb_model.feature_importances_
                }).sort_values('Importance', ascending=False)
                
                self.feature_importances[quarter_key] = importances
                self.selected_features[quarter_key] = selected
                
                return selected
            except Exception as e:
                self.print_debug(f"Model-based selection failed: {e}")
                traceback.print_exc()
        
        elif method == 'recursive':
            # Use recursive feature elimination with cross-validation
            try:
                # Use Ridge for RFECV as it's more stable
                ridge_model = Ridge(alpha=1.0, random_state=42)
                
                # Create RFECV selector
                selector = RFECV(
                    estimator=ridge_model,
                    step=1,
                    cv=5,
                    scoring='neg_mean_squared_error',
                    min_features_to_select=3
                )
                
                # Fit selector
                selector.fit(X, y)
                
                # Get selected features
                selected_mask = selector.support_
                selected = [feature for feature, selected in zip(feature_names, selected_mask) if selected]
                
                # Store importances using Ridge coefficients
                ridge_model.fit(X, y)
                importances = pd.DataFrame({
                    'Feature': feature_names,
                    'Importance': np.abs(ridge_model.coef_)
                }).sort_values('Importance', ascending=False)
                
                self.feature_importances[quarter_key] = importances
                self.selected_features[quarter_key] = selected
                
                return selected
            except Exception as e:
                self.print_debug(f"Recursive feature elimination failed: {e}")
                traceback.print_exc()
                
        elif method == 'hybrid':
            # Hybrid approach combining multiple methods
            self.print_debug("Using hybrid feature selection approach...")
            
            # Get importances from multiple methods
            importances = self.analyze_feature_importance(X, y, method='all')
            
            if importances is None:
                importances = self.analyze_feature_importance(X, y, method='mutual_info')
            
            if importances is not None:
                self.feature_importances[quarter_key] = importances
                
                # Select features based on importance threshold
                importance_threshold = importances['Importance'].quantile(0.5)  # Top 50%
                selected = importances[importances['Importance'] >= importance_threshold]['Feature'].tolist()
                
                # If n_features is specified, respect that limit
                if n_features is not None:
                    selected = selected[:n_features]
                
                # Ensure we have at least 3 features
                if len(selected) < 3:
                    selected = importances.head(3)['Feature'].tolist()
                
                self.selected_features[quarter_key] = selected
                return selected
        
        # Fallback to mutual information if all else fails
        return self.select_features(X, y, quarter, n_features, method='mutual_info')
    
    def optimize_feature_set(self, X, y, quarter, base_features, validation_frac=0.2):
        """
        Optimize feature set by evaluating different combinations.
        
        Args:
            X: Feature DataFrame
            y: Target Series
            quarter: Quarter number
            base_features: List of base features to consider
            validation_frac: Fraction of data to use for validation
            
        Returns:
            list: Optimized feature list
        """
        quarter_key = f'q{quarter}'
        self.print_debug(f"Optimizing feature set for quarter {quarter}...")
        
        # Try different feature selection methods
        methods = ['mutual_info', 'hybrid']
        
        if self.xgboost_available:
            methods.append('model_based')
        
        methods.append('recursive')
        
        # Split data for validation
        n_samples = len(X)
        n_val = int(n_samples * validation_frac)
        
        X_train = X.iloc[:-n_val].copy()
        y_train = y.iloc[:-n_val].copy()
        X_val = X.iloc[-n_val:].copy()
        y_val = y.iloc[-n_val:].copy()
        
        best_mse = float('inf')
        best_features = base_features
        best_model = None
        best_method = None
        
        # Try each method
        for method in methods:
            self.print_debug(f"Testing {method} feature selection...")
            
            try:
                # Select features using this method
                selected_features = self.select_features(
                    X_train, y_train, quarter, method=method
                )
                
                if not selected_features:
                    self.print_debug(f"No features selected with {method}")
                    continue
                
                # Train model with selected features
                if self.xgboost_available and method != 'recursive':
                    model = xgb.XGBRegressor(
                        n_estimators=100,
                        learning_rate=0.1,
                        max_depth=3,
                        subsample=0.8,
                        colsample_bytree=0.8,
                        random_state=42
                    )
                else:
                    model = Ridge(alpha=1.0, random_state=42)
                
                # Train on selected features
                model.fit(X_train[selected_features], y_train)
                
                # Evaluate on validation set
                y_pred = model.predict(X_val[selected_features])
                mse = mean_squared_error(y_val, y_pred)
                r2 = r2_score(y_val, y_pred)
                
                self.print_debug(f"  {method}: {len(selected_features)} features, MSE={mse:.2f}, R²={r2:.3f}")
                
                # Track best model
                if mse < best_mse:
                    best_mse = mse
                    best_features = selected_features
                    best_model = model
                    best_method = method
            
            except Exception as e:
                self.print_debug(f"Error evaluating {method}: {e}")
                traceback.print_exc()
        
        # Store performance metrics
        if best_model is not None:
            self.performance_metrics[quarter_key] = {
                'method': best_method,
                'n_features': len(best_features),
                'mse': best_mse,
                'rmse': np.sqrt(best_mse),
                'features': best_features
            }
            
            self.print_debug(f"Best model: {best_method} with {len(best_features)} features, RMSE={np.sqrt(best_mse):.2f}")
        
        return best_features
    
    def visualize_feature_importance(self, quarter):
        """
        Create visualization of feature importance for a quarter.
        
        Args:
            quarter: Quarter number
            
        Returns:
            matplotlib Figure
        """
        quarter_key = f'q{quarter}'
        
        if quarter_key not in self.feature_importances:
            print(f"No feature importance data available for quarter {quarter}")
            return None
        
        importances = self.feature_importances[quarter_key]
        selected = self.selected_features.get(quarter_key, [])
        
        # Create figure
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Plot importance values
        bars = ax.barh(
            importances['Feature'].head(15), 
            importances['Importance'].head(15),
            color=[('darkblue' if feature in selected else 'lightblue') 
                  for feature in importances['Feature'].head(15)]
        )
        
        # Add legend
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor='darkblue', label='Selected'),
            Patch(facecolor='lightblue', label='Not Selected')
        ]
        ax.legend(handles=legend_elements)
        
        # Add labels
        ax.set_title(f'Feature Importance for Quarter {quarter}')
        ax.set_xlabel('Importance')
        
        # Add performance metrics if available
        if quarter_key in self.performance_metrics:
            metrics = self.performance_metrics[quarter_key]
            metric_text = (
                f"Best Method: {metrics['method']}\n"
                f"Features: {metrics['n_features']}\n"
                f"RMSE: {metrics['rmse']:.2f}"
            )
            ax.text(
                0.95, 0.05, metric_text,
                transform=ax.transAxes,
                ha='right', va='bottom',
                bbox=dict(facecolor='white', alpha=0.8, boxstyle='round,pad=0.5')
            )
        
        plt.tight_layout()
        return fig

def optimize_quarter_specific_features(features_df, target_col='home_score'):
    """
    Identify optimal feature subsets for each quarter using XGBoost
    
    Args:
        features_df: DataFrame with all potential features
        target_col: Base target column name
        
    Returns:
        Dict with optimal feature sets for each quarter
    """
    # First check if XGBoost is available
    try:
        from xgboost import XGBRegressor
        from sklearn.feature_selection import SelectFromModel
    except ImportError:
        print("XGBoost or scikit-learn's SelectFromModel not available. Please install with:")
        print("pip install xgboost scikit-learn")
        return None
    
    # Define quarter-specific initial feature sets (candidates for optimization)
    full_quarter_features = {
        1: [
            'rest_days_home', 'rest_days_away', 'is_back_to_back_home', 'is_back_to_back_away',
            'prev_matchup_diff', 'rolling_home_score', 'rolling_away_score',
            'time_remaining_norm', 'rest_advantage'
        ],
        2: [
            'home_q1', 'away_q1', 'rest_advantage', 'prev_matchup_diff',
            'rolling_home_score', 'rolling_away_score', 'q1_to_q2_momentum',
            'time_remaining_norm', 'momentum_indicator', 'score_momentum_interaction',
            'score_ratio', 'score_differential'
        ],
        3: [
            'home_q1', 'home_q2', 'away_q1', 'away_q2', 'q1_to_q2_momentum', 
            'q2_to_q3_momentum', 'first_half_diff', 'rest_advantage',
            'time_remaining_norm', 'momentum_indicator', 'score_momentum_interaction',
            'score_ratio', 'score_differential', 'pre_q4_diff', 'cumulative_momentum'
        ],
        4: [
            'home_q1', 'home_q2', 'home_q3', 'away_q1', 'away_q2', 'away_q3',
            'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 
            'pre_q4_diff', 'cumulative_momentum', 'time_remaining_norm', 
            'momentum_indicator', 'score_momentum_interaction',
            'score_ratio', 'score_differential'
        ]
    }
    
    # Initialize dictionary for results
    optimal_feature_sets = {}
    feature_importance_data = {}
    
    print("Finding optimal feature sets for each quarter using XGBoost...")
    
    # Process each quarter
    for quarter, features in full_quarter_features.items():
        print(f"\nQuarter {quarter}:")
        
        # Check if features exist in the DataFrame
        missing_features = [f for f in features if f not in features_df.columns]
        if missing_features:
            print(f"  Warning: Missing features: {missing_features}")
            # Use only available features
            features = [f for f in features if f in features_df.columns]
        
        # Skip if no features are available
        if not features:
            print("  No valid features available. Skipping.")
            continue
        
        # Get the target column for this quarter
        q_target = f"home_q{quarter}"
        if q_target not in features_df.columns:
            print(f"  Target column {q_target} not found. Skipping.")
            continue
        
        # Define data for this quarter
        X = features_df[features].copy().fillna(0)
        y = features_df[q_target].fillna(features_df[q_target].mean())
        
        print(f"  Training on {len(X)} samples with {len(features)} initial features")
        
        # Set up train/test split (chronological)
        train_size = int(0.8 * len(X))
        X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
        y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]
        
        # Define XGBoost model for feature selection
        xgb_model = XGBRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        )
        
        # First train model on all features to get baseline performance
        xgb_model.fit(X_train, y_train)
        baseline_score = xgb_model.score(X_test, y_test)
        baseline_rmse = np.sqrt(mean_squared_error(y_test, xgb_model.predict(X_test)))
        
        print(f"  Baseline with all features: R² = {baseline_score:.3f}, RMSE = {baseline_rmse:.2f}")
        
        # Get and store feature importances
        importances = xgb_model.feature_importances_
        feature_importances = pd.DataFrame({
            'Feature': features,
            'Importance': importances
        }).sort_values('Importance', ascending=False)
        
        feature_importance_data[quarter] = feature_importances
        
        # Try different importance thresholds
        thresholds = [0.01, 0.02, 0.03, 0.05, 0.1]
        threshold_results = []
        
        for threshold in thresholds:
            # Use feature selection
            selection = SelectFromModel(xgb_model, threshold=threshold, prefit=True)
            X_train_selected = selection.transform(X_train)
            X_test_selected = selection.transform(X_test)
            
            # Skip if no features were selected
            if X_train_selected.shape[1] == 0:
                continue
            
            # Train model on selected features
            selected_model = XGBRegressor(
                n_estimators=100,
                learning_rate=0.1,
                max_depth=4,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42
            )
            selected_model.fit(X_train_selected, y_train)
            
            # Evaluate
            selected_score = selected_model.score(X_test_selected, y_test)
            selected_rmse = np.sqrt(mean_squared_error(
                y_test, selected_model.predict(X_test_selected)))
            
            # Get selected feature mask and names
            selected_mask = selection.get_support()
            selected_features = [feature for feature, selected in zip(features, selected_mask) if selected]
            
            threshold_results.append({
                'threshold': threshold,
                'n_features': len(selected_features),
                'R²': selected_score,
                'RMSE': selected_rmse,
                'feature_list': selected_features
            })
            
            print(f"  Threshold {threshold}: {len(selected_features)} features, R² = {selected_score:.3f}, RMSE = {selected_rmse:.2f}")
        
        # Find best threshold based on R² (could also use RMSE)
        if threshold_results:
            # Convert to DataFrame for easier analysis
            threshold_df = pd.DataFrame(threshold_results)
            
            # Either choose simplest model within 1% of best performance,
            # or just take the best performing model
            best_score = threshold_df['R²'].max()
            acceptable_threshold = threshold_df[threshold_df['R²'] >= best_score * 0.99]
            if not acceptable_threshold.empty:
                # Choose the one with fewest features
                best_result = acceptable_threshold.loc[acceptable_threshold['n_features'].idxmin()]
            else:
                # Just take the best
                best_result = threshold_df.loc[threshold_df['R²'].idxmax()]
            
            print(f"  Best model: threshold={best_result['threshold']}, {best_result['n_features']} features")
            print(f"  Selected features: {best_result['feature_list']}")
            
            # Store the optimal feature set
            optimal_feature_sets[quarter] = best_result['feature_list']
        else:
            print("  No valid feature selection results. Using all features.")
            optimal_feature_sets[quarter] = features
    
    # Visualize feature importance for each quarter
    for quarter, importance_df in feature_importance_data.items():
        plt.figure(figsize=(10, max(6, len(importance_df) * 0.4)))
        sns.barplot(x='Importance', y='Feature', data=importance_df.head(15))
        plt.title(f'Quarter {quarter} Feature Importance')
        plt.tight_layout()
        plt.show()
    
    return optimal_feature_sets, feature_importance_data

# Fix for dtype incompatibility in momentum calculations
def fix_momentum_dtype_warnings():
    """Fix the dtype incompatibility warnings in momentum calculations"""
    print("Updating momentum calculation to fix dtype warnings...")
    
    # Assuming this function is called from a notebook
    # where we have access to the code at the problematic line
    
    # The problematic code is in these functions:
    # - calculate_derived_features
    # - NBAFeatureGenerator.calculate_momentum_features
    
    # Here's the fix for calculate_derived_features:
    code_fix = """
    # For calculate_derived_features, change this:
    result_df.at[idx, 'cumulative_momentum'] = max(min(result_df.at[idx, 'cumulative_momentum'], 1.0), -1.0)
    
    # To this (explicit float casting):
    result_df.at[idx, 'cumulative_momentum'] = float(max(min(float(result_df.at[idx, 'cumulative_momentum']), 1.0), -1.0))
    
    # For NBAFeatureGenerator.calculate_momentum_features, similarly add explicit float() calls
    """
    
    print("To fix dtype warnings, add explicit float() casts in momentum calculations:")
    print(code_fix)
    
    # NOTE: This function just outputs the fix, since we can't directly
    # modify the class code in a notebook context without redefining the whole class
    
    return code_fix

# Run the feature optimization
optimal_features, feature_importances = optimize_quarter_specific_features(features_df)

# Display the fix for dtype warnings
fix_momentum_dtype_warnings()

def optimize_quarter_specific_features_xgboost(features_df, target_col='home_score'):
    """
    Enhanced feature optimization using advanced feature selection techniques.
    Uses multiple methods and validates results for optimal feature sets.
    
    Args:
        features_df: DataFrame with all potential features
        target_col: Base target column name
        
    Returns:
        dict: Optimal feature sets for each quarter
    """
    # Initialize feature selector
    feature_selector = AdvancedFeatureSelector(debug=True)
    
    # Define quarter-specific initial feature sets
    full_quarter_features = {
        1: [
            'rest_days_home', 'rest_days_away', 'is_back_to_back_home', 'is_back_to_back_away',
            'prev_matchup_diff', 'rolling_home_score', 'rolling_away_score',
            'time_remaining_norm', 'rest_advantage'
        ],
        2: [
            'home_q1', 'away_q1', 'rest_advantage', 'prev_matchup_diff',
            'rolling_home_score', 'rolling_away_score', 'q1_to_q2_momentum',
            'time_remaining_norm', 'momentum_indicator', 'score_momentum_interaction',
            'score_ratio', 'score_differential'
        ],
        3: [
            'home_q1', 'home_q2', 'away_q1', 'away_q2', 'q1_to_q2_momentum', 
            'q2_to_q3_momentum', 'first_half_diff', 'rest_advantage',
            'time_remaining_norm', 'momentum_indicator', 'score_momentum_interaction',
            'score_ratio', 'score_differential', 'pre_q4_diff', 'cumulative_momentum'
        ],
        4: [
            'home_q1', 'home_q2', 'home_q3', 'away_q1', 'away_q2', 'away_q3',
            'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 
            'pre_q4_diff', 'cumulative_momentum', 'time_remaining_norm', 
            'momentum_indicator', 'score_momentum_interaction',
            'score_ratio', 'score_differential'
        ]
    }
    
    # Prepare results container
    optimal_feature_sets = {}
    
    print("Optimizing quarter-specific features with advanced selection...")
    
    # Process each quarter
    for quarter, base_features in full_quarter_features.items():
        print(f"\nOptimizing features for Quarter {quarter}:")
        
        # Check which features exist in the DataFrame
        existing_features = [f for f in base_features if f in features_df.columns]
        if not existing_features:
            print(f"  No valid features found for Quarter {quarter}")
            continue
        
        # Get the target column for this quarter
        q_target = f"{target_col[:-5]}_q{quarter}" if target_col.endswith('score') else f"q{quarter}"
        if q_target not in features_df.columns:
            print(f"  Target column {q_target} not found. Skipping.")
            continue
        
        # Prepare data
        X = features_df[existing_features].copy()
        y = features_df[q_target].copy()
        
        # Handle missing values
        X = X.fillna(0)
        y = y.fillna(y.mean())
        
        print(f"  Working with {len(X)} samples and {len(existing_features)} base features")
        
        # Optimize feature set
        optimal_features = feature_selector.optimize_feature_set(
            X, y, quarter, existing_features
        )
        
        print(f"  Selected {len(optimal_features)} optimal features:")
        for feature in optimal_features:
            print(f"    • {feature}")
        
        optimal_feature_sets[quarter] = optimal_features
    
    # Visualize results for each quarter
    for quarter in full_quarter_features.keys():
        fig = feature_selector.visualize_feature_importance(quarter)
        if fig:
            plt.figure(fig.number)
            plt.show()
    
    return optimal_feature_sets

# Test the enhanced feature optimization if XGBoost is available
print("\nTesting enhanced feature optimization...")
if xgboost_available:
    optimal_features = optimize_quarter_specific_features_xgboost(features_df)
    
    # Print summary
    print("\nOptimal feature sets for each quarter:")
    for quarter, features in optimal_features.items():
        print(f"\nQuarter {quarter}: {len(features)} features")
        print(f"  {', '.join(features)}")
else:
    print("XGBoost not available. Install it for advanced feature selection.")
    # Fallback to simpler feature selection
    from sklearn.feature_selection import SelectKBest, f_regression
    
    print("\nUsing basic feature selection with SelectKBest...")
    for quarter in range(1, 5):
        q_target = f"home_q{quarter}"
        if q_target not in features_df.columns:
            continue
            
        # Get basic feature set
        base_features = [
            'home_q1', 'away_q1', 'score_ratio', 'rolling_home_score', 
            'rolling_away_score', 'prev_matchup_diff'
        ]
        base_features = [f for f in base_features if f in features_df.columns]
        
        if not base_features:
            continue
            
        # Handle quarters
        base_features = [f for f in base_features if not (
            f.startswith('home_q') or f.startswith('away_q')
        ) or int(f[-1]) < quarter]
        
        # Prepare data
        X = features_df[base_features].fillna(0)
        y = features_df[q_target].fillna(features_df[q_target].mean())
        
        # Use SelectKBest
        selector = SelectKBest(f_regression, k=min(5, len(base_features)))
        selector.fit(X, y)
        
        # Get selected features
        selected_mask = selector.get_support()
        selected_features = [f for i, f in enumerate(base_features) if selected_mask[i]]
        
        print(f"\nQuarter {quarter}: {len(selected_features)} features")
        print(f"  {', '.join(selected_features)}")

In [ ]:
# Cell 4D-4: Multi-Target Prediction Integration

class MultiTargetPredictor:
    """Predict final score, win probability, and scoring pace simultaneously"""
    
    def __init__(self, main_predictor, ensemble_builder):
        self.main_predictor = main_predictor
        self.ensemble = ensemble_builder
        self.pace_model = self._build_pace_model()
        
    def _build_pace_model(self):
        """Build model to predict remaining scoring pace"""
        from sklearn.ensemble import GradientBoostingRegressor
        
        model = GradientBoostingRegressor(
            n_estimators=100,
            max_depth=3,
            learning_rate=0.1,
            random_state=42
        )
        
        # This would be trained on historical data
        return model
    
    def predict(self, game_data):
        """Generate comprehensive prediction package"""
        # Basic score prediction using main predictor
        score_prediction = self.main_predictor.predict(game_data)
        
        # Get quarter-specific and ensemble predictions
        quarter = game_data.get('current_quarter', 0)
        quarter_prediction = self._get_quarter_prediction(game_data, quarter)
        
        ensemble_prediction, weights, confidence = self.ensemble.predict(
            score_prediction, 
            quarter_prediction,
            quarter,
            game_data
        )
        
        # Add pace prediction
        remaining_mins = game_data.get('time_remaining_mins', 48)
        if remaining_mins > 0:
            pace_features = self._extract_pace_features(game_data)
            points_per_minute = self.pace_model.predict([pace_features])[0]
            remaining_points = points_per_minute * remaining_mins
        else:
            remaining_points = 0
        
        # Calculate comprehensive win probability
        score_diff = ensemble_prediction['home'] - ensemble_prediction['away']
        win_prob_base = 1 / (1 + np.exp(-0.1 * score_diff))
        
        # Adjust win probability based on momentum and home court
        momentum_factor = game_data.get('momentum', 0) * 0.05
        home_factor = 0.05 if quarter < 3 else 0.02  # Home advantage diminishes late
        
        win_prob = np.clip(win_prob_base + momentum_factor + home_factor, 0.01, 0.99)
        
        return {
            'home_score': ensemble_prediction['home'],
            'away_score': ensemble_prediction['away'],
            'win_probability': win_prob,
            'confidence': confidence,
            'expected_pace': points_per_minute,
            'remaining_points': remaining_points,
            'weights_used': weights
        }

In [ ]:
# Cell 4E: Feature Selection and Quarter-Specific Model Updates

import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import time

print("=" * 80)
print("PART 1: FEATURE SELECTION")
print("=" * 80)

print(f"Data range at start of Cell 4E: {features_df['game_date'].min()} to {features_df['game_date'].max()}")
print(f"Seasons in data: {sorted(features_df['game_date'].apply(get_nba_season).unique())}")

def implement_feature_selection(features_df, target_col='home_score'):
    """
    Implement feature selection based on correlation analysis and feature importance.
    
    Args:
        features_df: DataFrame with all features
        target_col: Target column for prediction
        
    Returns:
        Dictionary with selected feature sets for each quarter
    """
    print("Implementing feature selection based on correlation analysis...")
    
    # Create copy to avoid modifying original
    df = features_df.copy()
    
    # 1. Remove redundant ID columns
    id_cols = ['id', 'game_id']
    essential_id = 'game_id'  # Keep one ID column
    for col in id_cols:
        if col != essential_id and col in df.columns:
            print(f"Removing redundant ID column: {col}")
            df.drop(columns=[col], inplace=True)
    
    # 2. Simplify rest features
    rest_cols = ['rest_days_home', 'rest_days_away', 'rest_advantage', 
                'is_back_to_back_home', 'is_back_to_back_away']
    
    # Only keep rest_advantage and back-to-back indicators
    simplified_rest = ['rest_advantage', 'is_back_to_back_home', 'is_back_to_back_away']
    rest_to_remove = [col for col in rest_cols if col not in simplified_rest]
    
    if all(col in df.columns for col in rest_to_remove):
        print(f"Simplifying rest features. Removing: {rest_to_remove}")
        df.drop(columns=rest_to_remove, inplace=True)
    
    # 3. Create compact momentum representation
    momentum_cols = ['q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum']
    
    # Add a simplified momentum feature if not already present
    if 'momentum_indicator' not in df.columns and all(col in df.columns for col in momentum_cols[:3]):
        print("Creating compact momentum representation...")
        
        # For each row, use the last available momentum metric
        df['momentum_indicator'] = df.apply(
            lambda row: (
                row['q3_to_q4_momentum'] if pd.notna(row['q3_to_q4_momentum']) and row['q3_to_q4_momentum'] != 0
                else row['q2_to_q3_momentum'] if pd.notna(row['q2_to_q3_momentum']) and row['q2_to_q3_momentum'] != 0
                else row['q1_to_q2_momentum'] if pd.notna(row['q1_to_q2_momentum']) and row['q1_to_q2_momentum'] != 0
                else 0
            ),
            axis=1
        )
        
        # Normalize to [-1, 1] range
        max_abs = df['momentum_indicator'].abs().max()
        if max_abs > 0:
            df['momentum_indicator'] = df['momentum_indicator'] / max_abs
            df['momentum_indicator'] = df['momentum_indicator'].clip(-1, 1)
    
    # 4. Add time remaining feature
    if 'current_quarter' in df.columns:
        print("Adding time remaining feature...")
        df['time_remaining_mins'] = df['current_quarter'].apply(
            lambda q: max(0, 48 - ((q - 1) * 12 + 6))  # Approximate minutes left
        )
        
        # Normalize time remaining to [0, 1] range (1 = full game, 0 = no time)
        df['time_remaining_norm'] = df['time_remaining_mins'] / 48.0
    
    # 5. Create interaction feature between score_ratio and momentum
    if all(f in df.columns for f in ['score_ratio', 'momentum_indicator']):
        print("Creating score-momentum interaction feature...")
        df['score_momentum_interaction'] = df['score_ratio'] * df['momentum_indicator']
    
    # 6. Select features based on mutual information with target
    # Numeric features only
    numeric_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    # Remove target and ids from features
    feature_cols = [col for col in numeric_features if col != target_col and 'id' not in col]
    
    if target_col in df.columns:
        # Calculate feature importance using mutual information
        X = df[feature_cols].fillna(0)
        y = df[target_col]
        
        # Calculate mutual information score
        mi_scores = mutual_info_regression(X, y)
        mi_features = pd.DataFrame({'Feature': feature_cols, 'MI_Score': mi_scores})
        mi_features = mi_features.sort_values('MI_Score', ascending=False)
        
        # Display top features
        print("\nTop features by mutual information with target:")
        display(mi_features.head(10))
        
        # Visualize mutual information scores
        plt.figure(figsize=(12, 8))
        sns.barplot(x='MI_Score', y='Feature', data=mi_features.head(15))
        plt.title('Feature Importance by Mutual Information')
        plt.tight_layout()
        plt.show()
    
    # 7. Define optimized feature sets for each quarter
    # Based on correlation analysis, benchmarking, and MI scores
    quarter_features = {
        'q1': [
            # Pre-game features
            'rolling_home_score', 'rolling_away_score', 
            'rest_advantage', 'is_back_to_back_home', 'is_back_to_back_away',
            'prev_matchup_diff'
        ],
        
        'q2': [
            # Q1 + pre-game features
            'home_q1', 'away_q1', 
            'rolling_home_score', 'rolling_away_score',
            'rest_advantage', 'prev_matchup_diff'
        ],
        
        'q3': [
            # Q1-Q2 features
            'home_q1', 'home_q2', 'away_q1', 'away_q2',
            'first_half_diff', 'q1_to_q2_momentum',
            'rest_advantage'
        ],
        
        'q4': [
            # All previous quarters
            'home_q1', 'home_q2', 'home_q3', 
            'away_q1', 'away_q2', 'away_q3',
            'pre_q4_diff',  # More important than individual momentum features
            'cumulative_momentum'  # Captures overall game flow
        ]
    }
    
    # Add time remaining to all quarters if available
    if 'time_remaining_norm' in df.columns:
        for quarter in quarter_features:
            quarter_features[quarter].append('time_remaining_norm')
    
    # Add momentum indicator to relevant quarters
    if 'momentum_indicator' in df.columns:
        quarter_features['q2'].append('momentum_indicator')
        quarter_features['q3'].append('momentum_indicator')
        quarter_features['q4'].append('momentum_indicator')
    
    # Add score-momentum interaction to relevant quarters
    if 'score_momentum_interaction' in df.columns:
        quarter_features['q2'].append('score_momentum_interaction')
        quarter_features['q3'].append('score_momentum_interaction')
        quarter_features['q4'].append('score_momentum_interaction')
    
    # Print final feature sets
    print("\nOptimized feature sets for quarter-specific models:")
    for quarter, features in quarter_features.items():
        print(f"\n{quarter.upper()} Features:")
        for feature in features:
            print(f"  • {feature}")
    
    return {
        'selected_features': df,
        'quarter_features': quarter_features,
        'feature_importance': mi_features if 'mi_features' in locals() else None
    }

# Run feature selection
feature_selection_results = implement_feature_selection(features_df)
enhanced_features_df = feature_selection_results['selected_features']
quarter_feature_sets = feature_selection_results['quarter_features']

print("\n" + "=" * 80)
print("PART 2: UPDATE QUARTER-SPECIFIC MODELS")
print("=" * 80)

def update_quarter_models_with_regularization(features_df, quarter_feature_sets, target_col='home_score'):
    """
    Create and tune improved quarter-specific models with regularization and anti-overfitting parameters.
    
    Args:
        features_df: DataFrame with features
        quarter_feature_sets: Dictionary with feature lists for each quarter
        target_col: Target column prefix (will be combined with quarter)
        
    Returns:
        Dictionary of tuned models for each quarter
    """
    print("Training updated quarter-specific models with regularization...")
    
    # Define quarter suffixes
    quarters = ['q1', 'q2', 'q3', 'q4']
    
    # Improved model configurations with regularization and overfitting prevention
    model_configs = {
        'q1': ('GradientBoosting', {
            'n_estimators': [100, 200],
            'learning_rate': [0.05, 0.1],
            'max_depth': [2, 3],  # Reduced from 5
            'min_samples_split': [5, 10],  # Increased from 2
            'subsample': [0.8]  # Added to reduce overfitting
        }),
        'q2': ('GradientBoosting', {
            'n_estimators': [100, 200],
            'learning_rate': [0.05, 0.1],
            'max_depth': [2, 3],  # Reduced from 5
            'min_samples_split': [5, 10],  # Increased from 2
            'subsample': [0.8]  # Added to reduce overfitting
        }),
        'q3': ('GradientBoosting', {
            'n_estimators': [100, 200],
            'learning_rate': [0.05, 0.1],
            'max_depth': [2, 3],  # Reduced from 5
            'min_samples_split': [5, 10],  # Increased from 2
            'subsample': [0.8]  # Added to reduce overfitting
        }),
        'q4': ('Ridge', {  # Changed from Linear to Ridge
            'alpha': [0.1, 1.0, 10.0],  # Regularization strength
            'fit_intercept': [True],
            'solver': ['auto']
        })
    }
    
    # Prepare results storage
    models = {}
    results = []
    cv_results = {}
    
    # Check if time-based cross-validation is possible
    use_time_cv = 'game_date' in features_df.columns
    if use_time_cv:
        print("Using time-based cross-validation with game dates")
        features_df = features_df.sort_values('game_date')
    else:
        print("Game dates not available, using standard cross-validation")
        
    # Loop through quarters
    for quarter in quarters:
        print(f"\nProcessing {quarter.upper()} model...")
        
        # Target column for this quarter
        q_target = f"{target_col[:-5]}{quarter}" if target_col.endswith('score') else quarter
        
        # Skip quarters without target data
        if q_target not in features_df.columns:
            print(f"Target column {q_target} not found for {quarter}. Skipping.")
            continue
            
        # Feature columns for this quarter
        feature_cols = quarter_feature_sets[quarter]
        missing_cols = [col for col in feature_cols if col not in features_df.columns]
        
        if missing_cols:
            print(f"Warning: Missing features for {quarter}: {missing_cols}")
            feature_cols = [col for col in feature_cols if col in features_df.columns]
            
        if not feature_cols:
            print(f"No valid features for {quarter}. Skipping.")
            continue
        
        # Prepare data
        X = features_df[feature_cols].copy()
        y = features_df[q_target]
        
        # Handle missing values
        X = X.fillna(0)
        y = y.fillna(y.mean())
        
        print(f"Training with {len(X)} samples, {len(feature_cols)} features")
        print(f"Features: {feature_cols}")
        
        # Get model type and param grid
        model_type, param_grid = model_configs[quarter]
        
        # Create base model
        if model_type == 'GradientBoosting':
            base_model = GradientBoostingRegressor(random_state=42)
            if param_grid is None:
                param_grid = {
                    'n_estimators': [100],
                    'learning_rate': [0.1],
                    'max_depth': [3],
                    'subsample': [0.8],
                    'min_samples_split': [5]
                }
        elif model_type == 'Ridge':
            base_model = Ridge(random_state=42)
            if param_grid is None:
                param_grid = {'alpha': [1.0], 'fit_intercept': [True]}
        elif model_type == 'Linear':
            base_model = LinearRegression()
            if param_grid is None:
                param_grid = {'fit_intercept': [True]}
        else:
            print(f"Unknown model type: {model_type}. Using GradientBoosting.")
            base_model = GradientBoostingRegressor(random_state=42)
            param_grid = {
                'n_estimators': [100],
                'learning_rate': [0.1],
                'max_depth': [3],
                'subsample': [0.8]
            }
        
        # Set up cross-validation
        if use_time_cv:
            cv = TimeSeriesSplit(n_splits=5)
            print("Using TimeSeriesSplit for time-based cross-validation")
        else:
            cv = KFold(n_splits=5, shuffle=True, random_state=42)
            print("Using KFold for standard cross-validation")
        
        # If param_grid has more than one parameter value, do grid search
        if any(len(values) > 1 for values in param_grid.values() if values is not None):
            print(f"Performing GridSearchCV for {quarter}...")
            
            # Remove None values from param_grid
            cleaned_param_grid = {k: v for k, v in param_grid.items() if v is not None}
            
            # Create grid search
            # Create grid search
            grid_search = GridSearchCV(
                base_model,
                cleaned_param_grid,
                cv=cv,
                scoring='neg_mean_squared_error',
                n_jobs=-1,
                verbose=0
            )
            
            # Fit grid search
            start_time = time.time()
            grid_search.fit(X, y)
            train_time = time.time() - start_time
            
            # Get best model and parameters
            best_model = grid_search.best_estimator_
            best_params = grid_search.best_params_
            
            print(f"Best parameters: {best_params}")
            print(f"Best CV score: {-grid_search.best_score_:.4f} MSE")
            
            # Store CV results
            cv_results[quarter] = {
                'best_params': best_params,
                'cv_results': grid_search.cv_results_
            }
        else:
            # Just fit the base model
            print(f"Fitting base model for {quarter}...")
            start_time = time.time()
            best_model = base_model
            best_model.fit(X, y)
            train_time = time.time() - start_time
            best_params = "Default parameters"
        
        # Calculate performance metrics
        y_pred = best_model.predict(X)
        mse = mean_squared_error(y, y_pred)
        r2 = r2_score(y, y_pred)
        
        # Calculate cross-val score
        cv_scores = cross_val_score(best_model, X, y, cv=cv, scoring='neg_mean_squared_error')
        cv_mse = -cv_scores.mean()
        cv_rmse = np.sqrt(cv_mse)
        
        # Get feature importance if available
        if hasattr(best_model, 'feature_importances_'):
            importance = dict(zip(feature_cols, best_model.feature_importances_))
            top_features = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:3]
            top_features_str = ', '.join([f"{f} ({v:.3f})" for f, v in top_features])
        elif hasattr(best_model, 'coef_'):
            importance = dict(zip(feature_cols, np.abs(best_model.coef_)))
            top_features = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:3]
            top_features_str = ', '.join([f"{f} ({v:.3f})" for f, v in top_features])
        else:
            top_features_str = "N/A"
        
        # Store results
        results.append({
            'Quarter': quarter.upper(),
            'Model Type': model_type,
            'MSE': mse,
            'RMSE': np.sqrt(mse),
            'CV MSE': cv_mse,
            'CV RMSE': cv_rmse,
            'R²': r2,
            'Training Time (s)': train_time,
            'Top Features': top_features_str,
            'Parameters': str(best_params)
        })
        
        # Store model
        models[quarter] = best_model
    
    # Convert results to DataFrame
    results_df = pd.DataFrame(results)
    
    # Display results
    print("\nUpdated model performance:")
    display(results_df)
    
    # Create a bar chart comparing CV RMSE
    plt.figure(figsize=(10, 6))
    
    # Sort by quarter
    sorted_results = results_df.sort_values('Quarter')
    
    plt.bar(sorted_results['Quarter'], sorted_results['CV RMSE'], color='darkblue')
    plt.axhline(y=sorted_results['CV RMSE'].mean(), color='red', linestyle='--', 
               label=f'Average: {sorted_results["CV RMSE"].mean():.2f}')
    
    plt.xlabel('Quarter')
    plt.ylabel('Cross-Validation RMSE')
    plt.title('Quarter-Specific Model Performance (CV RMSE)')
    plt.grid(axis='y', alpha=0.3)
    plt.legend()
    
    # Add value labels
    for i, v in enumerate(sorted_results['CV RMSE']):
        plt.text(i, v + 0.1, f'{v:.2f}', ha='center')
    
    plt.tight_layout()
    plt.show()
    
    return {
        'models': models,
        'results': results_df,
        'cv_results': cv_results
    }

def perform_season_backtesting(enhanced_df, quarter_feature_sets, n_splits=5):
    """
    Perform backtesting using chronological splits
    
    Args:
        enhanced_df: DataFrame with enhanced features
        quarter_feature_sets: Dictionary with feature lists for each quarter
        n_splits: Number of chronological splits to use for validation
        
    Returns:
        Dictionary with backtesting results
    """
    print("\n" + "=" * 80)
    print("PART 3: CHRONOLOGICAL BACKTESTING VALIDATION")
    print("=" * 80)
    
    # Check if we can determine dates
    if 'game_date' not in enhanced_df.columns:
        print("Cannot perform backtesting: no game_date column")
        return None
    
    # Ensure datetime
    enhanced_df = enhanced_df.copy()
    enhanced_df['game_date'] = pd.to_datetime(enhanced_df['game_date'])
    
    # Print date range to debug
    min_date = enhanced_df['game_date'].min()
    max_date = enhanced_df['game_date'].max()
    print(f"Date range in data: {min_date} to {max_date}")
    
    # Extract season information
    if 'season' not in enhanced_df.columns:
        enhanced_df['season'] = enhanced_df['game_date'].apply(get_nba_season)
    
    seasons = enhanced_df['season'].unique()
    print(f"Seasons in data: {sorted(seasons)}")
    
    # Sort by date
    enhanced_df = enhanced_df.sort_values('game_date')
    
    # Create time-based splits
    total_games = len(enhanced_df)
    games_per_split = total_games // n_splits
    
    print(f"Creating {n_splits} chronological splits with ~{games_per_split} games each")
    
    # Assign each game to a split
    split_ids = []
    for i in range(n_splits):
        start_idx = i * games_per_split
        end_idx = (i+1) * games_per_split if i < n_splits-1 else total_games
        split_ids.extend([f"Split-{i+1}"] * (end_idx - start_idx))
    
    enhanced_df['split'] = split_ids
    
    # Get unique splits
    splits = sorted(enhanced_df['split'].unique())
    print(f"Created splits: {splits}")
    
    # Storage for results
    backtest_results = {}
    
    for quarter in ['q1', 'q2', 'q3', 'q4']:
        q_target = f"home_{quarter}"
        feature_cols = quarter_feature_sets[quarter]
        
        # Check if target exists
        if q_target not in enhanced_df.columns:
            print(f"Target column {q_target} not found. Skipping {quarter} backtesting.")
            continue
            
        # Check if features exist
        missing_features = [f for f in feature_cols if f not in enhanced_df.columns]
        if missing_features:
            print(f"Missing features for {quarter}: {missing_features}")
            feature_cols = [f for f in feature_cols if f in enhanced_df.columns]
        
        print(f"\nBacktesting {quarter.upper()} model across chronological splits...")
        quarter_results = []
        
        # Use the first n-1 splits for training, test on the last split
        for test_idx in range(1, n_splits):
            # Training on all splits before the test split
            train_splits = [f"Split-{i+1}" for i in range(test_idx)]
            test_split = f"Split-{test_idx+1}"
            
            train_mask = enhanced_df['split'].isin(train_splits)
            test_mask = enhanced_df['split'] == test_split
            
            # Skip if no training or test data
            if train_mask.sum() == 0 or test_mask.sum() == 0:
                print(f"Insufficient data for {test_split}. Skipping.")
                continue
            
            # Split data
            X_train = enhanced_df[train_mask][feature_cols]
            y_train = enhanced_df[train_mask][q_target]
            X_test = enhanced_df[test_mask][feature_cols]
            y_test = enhanced_df[test_mask][q_target]
            
            # Fill missing values
            X_train = X_train.fillna(0)
            y_train = y_train.fillna(y_train.mean())
            X_test = X_test.fillna(0)
            y_test = y_test.fillna(y_test.mean())
            
            # Create model
            if quarter == 'q4':
                model = Ridge(alpha=1.0, random_state=42)
            else:
                model = GradientBoostingRegressor(
                    n_estimators=100,
                    learning_rate=0.1,
                    max_depth=3,
                    subsample=0.8,
                    min_samples_split=5,
                    random_state=42
                )
            
            # Train model
            model.fit(X_train, y_train)
            
            # Evaluate
            y_train_pred = model.predict(X_train)
            y_test_pred = model.predict(X_test)
            
            train_mse = mean_squared_error(y_train, y_train_pred)
            train_rmse = np.sqrt(train_mse)
            test_mse = mean_squared_error(y_test, y_test_pred)
            test_rmse = np.sqrt(test_mse)
            test_r2 = r2_score(y_test, y_test_pred)
            
            # Date range for this split
            split_dates = enhanced_df[test_mask]['game_date']
            split_start = split_dates.min()
            split_end = split_dates.max()
            
            # Store results
            quarter_results.append({
                'Split': test_split,
                'Date Range': f"{split_start.strftime('%Y-%m-%d')} to {split_end.strftime('%Y-%m-%d')}",
                'Train Splits': ', '.join(train_splits),
                'Train Size': len(X_train),
                'Test Size': len(X_test),
                'Train RMSE': train_rmse,
                'Test RMSE': test_rmse,
                'R²': test_r2,
                'Train/Test Gap': train_rmse / test_rmse if test_rmse > 0 else float('inf')
            })
            
            print(f"  {test_split}: Test RMSE = {test_rmse:.2f}, R² = {test_r2:.3f}")
        
        # Store results
        backtest_results[quarter] = pd.DataFrame(quarter_results) if quarter_results else None
    
    # Visualize backtesting results
    quarters_with_results = [q for q, df in backtest_results.items() if df is not None and not df.empty]
    
    if quarters_with_results:
        plt.figure(figsize=(12, 8))
        
        # Plot test RMSE by quarter and split
        for i, quarter in enumerate(quarters_with_results):
            results = backtest_results[quarter]
            plt.subplot(2, 2, i+1)
            sns.barplot(x='Split', y='Test RMSE', data=results)
            plt.title(f'{quarter.upper()} Model Backtesting')
            plt.ylabel('Test RMSE')
            plt.grid(axis='y', alpha=0.3)
            
            # Add values on top of bars
            for j, v in enumerate(results['Test RMSE']):
                plt.text(j, v + 0.1, f'{v:.2f}', ha='center')
        
        plt.tight_layout()
        plt.show()
    else:
        print("No visualization created - insufficient data for any quarter")
    
    return backtest_results

# Function to get NBA season from date
def get_nba_season(date):
    """Extract NBA season from date (NBA seasons start in October and end in June)"""
    if isinstance(date, str):
        date = pd.to_datetime(date)
    year = date.year
    month = date.month
    # For October through December, the season starts in the current year
    if month >= 10:
        return f"{year}-{year+1}"
    # For January through June, the season started in the previous year
    elif month <= 6:
        return f"{year-1}-{year}"
    # For July through September, use the upcoming season
    else:
        return f"{year}-{year+1}"

# Run the improved quarter-specific models with regularization
improved_models = update_quarter_models_with_regularization(enhanced_features_df, quarter_feature_sets)

# Run chronological backtesting with the same data used for model training
if 'game_date' in enhanced_features_df.columns:
    backtest_results = perform_season_backtesting(enhanced_features_df, quarter_feature_sets)
    
    # Print season distribution to verify correct data is being used
    if 'season' in enhanced_features_df.columns:
        seasons = enhanced_features_df['season'].value_counts().sort_index()
        print("\nVerifying seasons in data used for backtesting:")
        for season, count in seasons.items():
            print(f"  {season}: {count} games")
    else:
        print("\nNo 'season' column found, using dates to verify:")
        print(f"  Date range: {enhanced_features_df['game_date'].min()} to {enhanced_features_df['game_date'].max()}")
else:
    print("\nCannot perform chronological backtesting: game_date column not available")

In [ ]:
# Cell 4E-2: Comprehensive Validation Metrics Framework

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

def evaluate_prediction_accuracy(predictions_df, actual_df):
    """
    Comprehensive evaluation of prediction accuracy with multiple metrics.
    
    Args:
        predictions_df: DataFrame with predictions
        actual_df: DataFrame with actual final scores
        
    Returns:
        DataFrame with detailed accuracy metrics
    """
    # Merge predictions with actual results
    results = pd.merge(
        predictions_df,
        actual_df[['game_id', 'home_score', 'away_score']],
        on='game_id', 
        suffixes=('_pred', '')
    )
    
    # Calculate metrics by quarter
    metrics_by_quarter = []
    
    for quarter in range(5):  # 0-4 (including pre-game)
        q_results = results[results['current_quarter'] == quarter]
        if q_results.empty:
            continue
            
        # Score prediction errors
        home_errors = q_results['home_score_pred'] - q_results['home_score']
        away_errors = q_results['away_score_pred'] - q_results['away_score']
        
        # Absolute errors
        home_abs_errors = np.abs(home_errors)
        away_abs_errors = np.abs(away_errors)
        
        # Winner prediction
        actual_winners = (q_results['home_score'] > q_results['away_score']).astype(int)
        predicted_winners = (q_results['home_score_pred'] > q_results['away_score_pred']).astype(int)
        winner_accuracy = (actual_winners == predicted_winners).mean()
        
        # Margin prediction
        actual_margins = q_results['home_score'] - q_results['away_score']
        predicted_margins = q_results['home_score_pred'] - q_results['away_score_pred']
        margin_errors = np.abs(actual_margins - predicted_margins)
        
        # Confidence interval accuracy (if available)
        if 'home_lower_bound' in q_results.columns and 'home_upper_bound' in q_results.columns:
            home_in_interval = ((q_results['home_score'] >= q_results['home_lower_bound']) & 
                               (q_results['home_score'] <= q_results['home_upper_bound']))
            away_in_interval = ((q_results['away_score'] >= q_results['away_lower_bound']) & 
                               (q_results['away_score'] <= q_results['away_upper_bound']))
            interval_accuracy = (home_in_interval & away_in_interval).mean()
        else:
            interval_accuracy = None
        
        # Store metrics for this quarter
        metrics_by_quarter.append({
            'quarter': quarter,
            'sample_size': len(q_results),
            'home_rmse': np.sqrt(np.mean(home_errors**2)),
            'away_rmse': np.sqrt(np.mean(away_errors**2)),
            'home_mae': np.mean(home_abs_errors),
            'away_mae': np.mean(away_abs_errors),
            'combined_rmse': np.sqrt(np.mean(np.concatenate([home_errors**2, away_errors**2]))),
            'combined_mae': np.mean(np.concatenate([home_abs_errors, away_abs_errors])),
            'winner_accuracy': winner_accuracy,
            'margin_rmse': np.sqrt(np.mean((actual_margins - predicted_margins)**2)),
            'margin_mae': np.mean(margin_errors),
            'interval_accuracy': interval_accuracy
        })
    
    # Create DataFrame with metrics
    return pd.DataFrame(metrics_by_quarter)

# Example test of the validation metrics framework with sample data
def generate_sample_validation_data(n_samples=20):
    """Generate sample prediction and actual data for testing"""
    np.random.seed(42)
    
    # Generate game IDs
    game_ids = list(range(1001, 1001 + n_samples))
    
    # Generate predictions across different quarters
    predictions = []
    actuals = []
    
    for game_id in game_ids:
        # Random final scores
        actual_home = np.random.randint(90, 120)
        actual_away = np.random.randint(90, 120)
        
        # Add to actuals
        actuals.append({
            'game_id': game_id,
            'home_score': actual_home,
            'away_score': actual_away
        })
        
        # Generate predictions for different quarters
        for quarter in range(5):
            # More error in early quarters, less in later quarters
            error_factor = 1.0 - (quarter * 0.15)
            home_error = np.random.normal(0, 10 * error_factor)
            away_error = np.random.normal(0, 10 * error_factor)
            
            predictions.append({
                'game_id': game_id,
                'current_quarter': quarter,
                'home_score_pred': actual_home + home_error,
                'away_score_pred': actual_away + away_error,
                'home_lower_bound': actual_home + home_error - 15,
                'home_upper_bound': actual_home + home_error + 15,
                'away_lower_bound': actual_away + away_error - 15,
                'away_upper_bound': actual_away + away_error + 15
            })
    
    return pd.DataFrame(predictions), pd.DataFrame(actuals)

# Test the validation framework
print("Testing Validation Metrics Framework with sample data:")
sample_preds, sample_actuals = generate_sample_validation_data()
validation_results = evaluate_prediction_accuracy(sample_preds, sample_actuals)

print("\nValidation Results by Quarter:")
display(validation_results)

# Visualize the results
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.bar(validation_results['quarter'], validation_results['combined_rmse'], color='royalblue')
plt.title('RMSE by Quarter')
plt.xlabel('Quarter')
plt.ylabel('RMSE')
plt.xticks(validation_results['quarter'])
plt.grid(axis='y', alpha=0.3)

plt.subplot(1, 2, 2)
plt.bar(validation_results['quarter'], validation_results['winner_accuracy'], color='green')
plt.title('Winner Prediction Accuracy by Quarter')
plt.xlabel('Quarter')
plt.ylabel('Accuracy')
plt.xticks(validation_results['quarter'])
plt.ylim(0, 1)
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 4E-3: Live Betting Strategy Framework

class BettingAdvisor:
    """Provide strategic betting recommendations based on predictions"""
    
    def __init__(self, predictor, risk_tolerance='medium'):
        self.predictor = predictor
        self.risk_tolerance = risk_tolerance
        self.threshold_map = {
            'low': {'spread': 0.7, 'moneyline': 0.75, 'over_under': 0.65},
            'medium': {'spread': 0.6, 'moneyline': 0.65, 'over_under': 0.6},
            'high': {'spread': 0.55, 'moneyline': 0.6, 'over_under': 0.55}
        }
    
    def analyze_betting_opportunity(self, game_data, betting_lines):
        """Analyze current betting lines against predictions"""
        # Get prediction
        prediction = self.predictor.predict(game_data)
        
        # Calculate spreads and totals
        predicted_spread = prediction['home_score'] - prediction['away_score']
        predicted_total = prediction['home_score'] + prediction['away_score']
        
        # Get thresholds based on risk tolerance
        thresholds = self.threshold_map[self.risk_tolerance]
        
        # Analyze spread
        market_spread = betting_lines.get('spread', 0)
        spread_edge = predicted_spread - market_spread
        spread_confidence = self._calculate_confidence(
            abs(spread_edge), 
            prediction['confidence'],
            game_data.get('current_quarter', 0)
        )
        
        # Analyze over/under
        market_total = betting_lines.get('total', predicted_total)
        total_edge = predicted_total - market_total
        total_confidence = self._calculate_confidence(
            abs(total_edge),
            prediction['confidence'],
            game_data.get('current_quarter', 0)
        )
        
        # Generate recommendations
        recommendations = {
            'spread': self._get_spread_recommendation(
                spread_edge, spread_confidence, thresholds['spread']),
            'total': self._get_total_recommendation(
                total_edge, total_confidence, thresholds['over_under']),
            'moneyline': self._get_moneyline_recommendation(
                prediction['win_probability'], thresholds['moneyline'])
        }
        
        return {
            'prediction': prediction,
            'edges': {
                'spread': spread_edge,
                'total': total_edge,
                'win_prob': prediction['win_probability']
            },
            'confidence': {
                'spread': spread_confidence,
                'total': total_confidence
            },
            'recommendations': recommendations
        }

In [ ]:
# Cell 4E-4 – AdaptiveEnsemble Implementation

import json
import os
import numpy as np

class AdaptiveEnsemble:
    def __init__(self, state_file='adaptive_state.json'):
        """
        Initialize the AdaptiveEnsemble with predefined strategies and load persisted state.
        Strategies can be defined as functions or lambdas that process predictions.
        """
        # Define your strategy functions:
        self.strategies = {
            'conservative': self._conservative_strategy,
            'aggressive': self._aggressive_strategy
        }
        
        # Default strategy performance metrics (e.g., error history)
        self.performance_metrics = {'conservative': [], 'aggressive': []}
        self.state_file = state_file
        self.load_state()

    def _conservative_strategy(self, predictions):
        """
        Conservative strategy: perhaps a weighted average that trusts the main model.
        Modify the logic as needed.
        """
        # Example: return a smoothed prediction using a moving average filter
        return np.mean(predictions) if predictions else None

    def _aggressive_strategy(self, predictions):
        """
        Aggressive strategy: for close games, perhaps put more weight on recent trends.
        Modify the logic as needed.
        """
        # Example: return the last prediction or a more volatile forecast
        return predictions[-1] if predictions else None

    def select_strategy(self, game_context, predictions):
        """
        Automatically select a strategy based on game context.
        :param game_context: A string, e.g., 'close' or 'blowout'
        :param predictions: A list or array of prediction values from different models.
        :return: The selected strategy's prediction.
        """
        # Determine strategy: if game is close, use aggressive; if blowout, use conservative.
        if game_context == 'close':
            strategy = 'aggressive'
        else:
            strategy = 'conservative'
        
        # Persist the selection for performance tracking.
        result = self.strategies[strategy](predictions)
        self.update_performance(strategy, result, predictions)
        return result

    def update_performance(self, strategy, selected_prediction, predictions):
        """
        Update performance metrics for a given strategy. For demonstration, we simply record the absolute error
        between the selected prediction and the average of predictions as a proxy for performance.
        """
        target = np.mean(predictions) if predictions else 0
        error = abs(selected_prediction - target)
        self.performance_metrics[strategy].append(error)
        self.save_state()

    def save_state(self):
        """
        Save the performance metrics to disk for persistence.
        """
        state = {
            'performance_metrics': self.performance_metrics
        }
        with open(self.state_file, 'w') as f:
            json.dump(state, f)

    def load_state(self):
        """
        Load performance metrics from disk if available.
        """
        if os.path.exists(self.state_file):
            with open(self.state_file, 'r') as f:
                state = json.load(f)
                self.performance_metrics = state.get('performance_metrics', self.performance_metrics)
        else:
            # No persisted state; will create on first save.
            self.save_state()

# Example usage:
# adaptive_ensemble = AdaptiveEnsemble()
# game_context = 'close'  # or 'blowout', determined by your game context function
# predictions = [105, 108, 110]  # Replace with actual predictions from your ensemble
# final_prediction = adaptive_ensemble.select_strategy(game_context, predictions)
# print("Final adaptive prediction:", final_prediction)


In [ ]:
# Cell 4F: RandomForest Performance Investigation with Overfitting Analysis

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score, learning_curve, validation_curve
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
import time
import warnings
warnings.filterwarnings('ignore')

def investigate_random_forest(features_df, quarter_feature_sets, target_col='home_score', n_folds=5):
    """
    Investigate the high R² values of RandomForest models through comprehensive validation,
    overfitting analysis, and feature stability assessment.
    
    Args:
        features_df: DataFrame with features.
        quarter_feature_sets: Dictionary with feature lists for each quarter.
        target_col: Target column prefix (will be combined with quarter; e.g. 'home_score' -> 'home_q1', etc.)
        n_folds: Number of CV folds to use
        
    Returns:
        Dictionary with validation results.
    """
    print("Investigating RandomForest performance with comprehensive validation...")
    
    quarters = ['q1', 'q2', 'q3', 'q4']
    
    results = []
    learning_curves_data = {}
    feature_importance_data = {}
    complexity_curves_data = {}
    overfitting_metrics = {}
    
    # For comparing stability across folds
    feature_stability = {}
    
    # Define CV strategy - use KFold to ensure same splits for all analyses
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    for quarter in quarters:
        print(f"\nInvestigating {quarter.upper()} model...")
        
        # Construct target column name (e.g., "home_score" -> "home_q1")
        if target_col.endswith('score'):
            q_target = f"{target_col[:-5]}{quarter}"
        else:
            q_target = quarter
        
        if q_target not in features_df.columns:
            print(f"Target column {q_target} not found for {quarter}. Skipping.")
            continue
        
        # Get feature columns for this quarter
        feature_cols = quarter_feature_sets[quarter]
        missing_cols = [col for col in feature_cols if col not in features_df.columns]
        if missing_cols:
            print(f"Warning: Missing features for {quarter}: {missing_cols}")
            feature_cols = [col for col in feature_cols if col in features_df.columns]
        if not feature_cols:
            print(f"No valid features for {quarter}. Skipping.")
            continue
        
        # Prepare data
        X = features_df[feature_cols].copy()
        y = features_df[q_target]
        
        # Handle missing values
        X = X.fillna(0)
        y = y.fillna(y.mean())
        
        print(f"Analyzing with {len(X)} samples, {len(feature_cols)} features")
        
        # Initialize empty feature importance arrays for stability calculations
        fold_importances = []
        feature_ranks = []
        
        # 1. Cross-Validation Analysis
        print("Performing cross-validation analysis...")
        start_time = time.time()
        
        # Model with different random_state values to assess variability
        rf_model = RandomForestRegressor(
            n_estimators=100, 
            max_depth=None,  # Default depth - we'll analyze optimal depth later
            random_state=42,
            n_jobs=-1  # Use all cores for faster computation
        )
        
        # Run cross-validation
        cv_scores = cross_val_score(rf_model, X, y, cv=kf, scoring='neg_mean_squared_error')
        cv_mse = -cv_scores.mean()
        cv_rmse = np.sqrt(cv_mse)
        cv_std = cv_scores.std()
        
        # Train on full dataset for feature importance
        rf_model.fit(X, y)
        y_pred = rf_model.predict(X)
        train_mse = mean_squared_error(y, y_pred)
        train_rmse = np.sqrt(train_mse)
        r2 = r2_score(y, y_pred)
        train_time = time.time() - start_time
        
        # 2. Learning Curve Analysis - to detect overfitting
        print("Generating learning curves...")
        train_sizes = np.linspace(0.1, 1.0, 10)
        train_sizes, train_scores, test_scores = learning_curve(
            rf_model, X, y, 
            cv=kf, 
            train_sizes=train_sizes,
            scoring='neg_mean_squared_error', 
            n_jobs=-1
        )
        
        # Convert scores to positive MSE and then RMSE
        train_scores_mean = -train_scores.mean(axis=1)
        train_scores_std = train_scores.std(axis=1)
        test_scores_mean = -test_scores.mean(axis=1)
        test_scores_std = test_scores.std(axis=1)
        
        # Convert to RMSE for better interpretability
        train_rmse_mean = np.sqrt(train_scores_mean)
        train_rmse_std = train_scores_std / (2 * np.sqrt(train_scores_mean))
        test_rmse_mean = np.sqrt(test_scores_mean)
        test_rmse_std = test_scores_std / (2 * np.sqrt(test_scores_mean))
        
        learning_curves_data[quarter] = {
            'train_sizes': train_sizes,
            'train_rmse_mean': train_rmse_mean,
            'train_rmse_std': train_rmse_std,
            'test_rmse_mean': test_rmse_mean,
            'test_rmse_std': test_rmse_std
        }
        
        # 3. Complexity Curve Analysis - optimal tree depth
        print("Generating complexity curves...")
        param_range = np.arange(1, 21, 2)  # Different max_depth values
        train_scores, test_scores = validation_curve(
            RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
            X, y, param_name="max_depth", param_range=param_range,
            cv=kf, scoring='neg_mean_squared_error', n_jobs=-1
        )
        
        # Calculate mean and std for plotting
        train_scores_mean = -train_scores.mean(axis=1)
        train_scores_std = train_scores.std(axis=1)
        test_scores_mean = -test_scores.mean(axis=1)
        test_scores_std = test_scores.std(axis=1)
        
        # Convert to RMSE
        train_complexity_rmse = np.sqrt(train_scores_mean)
        test_complexity_rmse = np.sqrt(test_scores_mean)
        
        complexity_curves_data[quarter] = {
            'param_range': param_range,
            'train_rmse': train_complexity_rmse,
            'test_rmse': test_complexity_rmse
        }
        
        # 4. Feature Importance Stability Analysis
        print("Analyzing feature importance stability...")
        importance_by_fold = {}
        
        # Initialize importance storage for each feature
        for feature in feature_cols:
            importance_by_fold[feature] = []
        
        # Collect feature importance from each fold
        for train_idx, test_idx in kf.split(X):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            
            # Train model on this fold
            fold_model = RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42, n_jobs=-1)
            fold_model.fit(X_train, y_train)
            
            # Get feature importances and store
            fold_importances.append(dict(zip(feature_cols, fold_model.feature_importances_)))
            
            # Store individual fold importances
            for i, feature in enumerate(feature_cols):
                importance_by_fold[feature].append(fold_model.feature_importances_[i])
            
            # Calculate feature ranks for stability metric
            ranks = np.argsort(-fold_model.feature_importances_)
            rank_dict = {feature_cols[idx]: rank for rank, idx in enumerate(ranks)}
            feature_ranks.append(rank_dict)
        
        # Calculate importance statistics
        importance_stats = {}
        for feature in feature_cols:
            importances = importance_by_fold[feature]
            importance_stats[feature] = {
                'mean': np.mean(importances),
                'std': np.std(importances),
                'min': np.min(importances),
                'max': np.max(importances),
                'cv': np.std(importances) / np.mean(importances) if np.mean(importances) > 0 else 0
            }
        
        # Calculate rank stability using Spearman correlation
        rank_stability = []
        for i in range(len(feature_ranks)):
            for j in range(i+1, len(feature_ranks)):
                ranks_i = [feature_ranks[i][feat] for feat in feature_cols]
                ranks_j = [feature_ranks[j][feat] for feat in feature_cols]
                corr, _ = spearmanr(ranks_i, ranks_j)
                rank_stability.append(corr)
        
        mean_rank_stability = np.mean(rank_stability) if rank_stability else 0
        
        # 5. Calculate overfitting metrics
        train_test_gap = train_rmse / cv_rmse if cv_rmse > 0 else float('inf')
        overfitting_indicator = train_test_gap < 0.5  # significant gap indicates potential overfitting
        
        # Detect overfitting based on learning curve
        learning_curve_gap = (learning_curves_data[quarter]['train_rmse_mean'][-1] / 
                            learning_curves_data[quarter]['test_rmse_mean'][-1])
        
        overfitting_metrics[quarter] = {
            'train_test_gap': train_test_gap,
            'learning_curve_gap': learning_curve_gap,
            'overfitting_detected': overfitting_indicator,
            'severity': 'High' if train_test_gap < 0.3 else 
                       'Medium' if train_test_gap < 0.7 else 'Low'
        }
        
        # Store feature importance stability data
        feature_stability[quarter] = {
            'importance_stats': importance_stats,
            'mean_rank_stability': mean_rank_stability,
            'rank_stability_values': rank_stability
        }
        
        # Store fold importances for visualization
        feature_importance_data[quarter] = {
            'mean': {feature: importance_stats[feature]['mean'] for feature in feature_cols},
            'std': {feature: importance_stats[feature]['std'] for feature in feature_cols},
            'cv': {feature: importance_stats[feature]['cv'] for feature in feature_cols},
            'by_fold': fold_importances
        }
        
        # 6. Calculate permutation importance (more reliable than default feature_importances_)
        print("Calculating permutation importance...")
        result = permutation_importance(
            rf_model, X, y, n_repeats=5, random_state=42, n_jobs=-1
        )
        perm_importance = result.importances_mean
        perm_importance_std = result.importances_std
        
        # Store in results
        sorted_perm_idx = perm_importance.argsort()[::-1]
        permutation_importance_dict = {
            feature_cols[i]: {
                'importance': perm_importance[i],
                'std': perm_importance_std[i]
            } for i in sorted_perm_idx
        }
        
        # Compile results
        results.append({
            'Quarter': quarter.upper(),
            'Train MSE': train_mse,
            'Train RMSE': train_rmse,
            'CV MSE': cv_mse,
            'CV RMSE': cv_rmse,
            'CV Std': cv_std,
            'Train-Test Gap': train_test_gap,
            'R²': r2,
            'Potential Overfitting': 'Yes' if overfitting_indicator else 'No',
            'Training Time (s)': train_time,
            'Top Feature': sorted(importance_stats.items(), 
                                key=lambda x: x[1]['mean'], 
                                reverse=True)[0][0],
            'Feature Stability': mean_rank_stability,
            'Permutation Importance': permutation_importance_dict
        })
    
    results_df = pd.DataFrame(results)
    print("\nRandomForest Validation Results:")
    display(results_df)
    
    # Visualize learning curves
    plt.figure(figsize=(20, 15))
    for i, quarter in enumerate(quarters):
        if quarter in learning_curves_data:
            plt.subplot(2, 2, i+1)
            lc_data = learning_curves_data[quarter]
            
            plt.fill_between(lc_data['train_sizes'], 
                            lc_data['train_rmse_mean'] - lc_data['train_rmse_std'],
                            lc_data['train_rmse_mean'] + lc_data['train_rmse_std'], 
                            alpha=0.1, color='r')
            plt.fill_between(lc_data['train_sizes'], 
                            lc_data['test_rmse_mean'] - lc_data['test_rmse_std'],
                            lc_data['test_rmse_mean'] + lc_data['test_rmse_std'], 
                            alpha=0.1, color='g')
            
            plt.plot(lc_data['train_sizes'], lc_data['train_rmse_mean'], 'o-', color='r',
                    label=f"Training score")
            plt.plot(lc_data['train_sizes'], lc_data['test_rmse_mean'], 'o-', color='g',
                    label=f"Cross-validation score")
            
            # Add visible gap measure
            last_train = lc_data['train_rmse_mean'][-1]
            last_test = lc_data['test_rmse_mean'][-1]
            gap_ratio = last_train / last_test
            plt.text(0.5, 0.1, f'Train/Test Gap: {gap_ratio:.2f}', 
                    ha='center', va='center', transform=plt.gca().transAxes,
                    bbox=dict(facecolor='white', alpha=0.8))
            
            plt.title(f'Learning Curves for {quarter.upper()} (RandomForest)')
            plt.xlabel('Training examples')
            plt.ylabel('RMSE')
            plt.grid(True, alpha=0.3)
            plt.legend(loc='best')
    
    plt.tight_layout()
    plt.show()
    
    # Visualize complexity curves
    plt.figure(figsize=(20, 15))
    for i, quarter in enumerate(quarters):
        if quarter in complexity_curves_data:
            plt.subplot(2, 2, i+1)
            cc_data = complexity_curves_data[quarter]
            
            plt.plot(cc_data['param_range'], cc_data['train_rmse'], 'o-', color='r',
                    label=f"Training score")
            plt.plot(cc_data['param_range'], cc_data['test_rmse'], 'o-', color='g',
                    label=f"Cross-validation score")
            
            # Find optimal depth
            best_idx = np.argmin(cc_data['test_rmse'])
            best_depth = cc_data['param_range'][best_idx]
            plt.axvline(x=best_depth, color='blue', linestyle='--', 
                     label=f'Optimal depth: {best_depth}')
            
            plt.title(f'Complexity Curve for {quarter.upper()} (RandomForest)')
            plt.xlabel('max_depth parameter')
            plt.ylabel('RMSE')
            plt.xticks(cc_data['param_range'])
            plt.grid(True, alpha=0.3)
            plt.legend(loc='best')
    
    plt.tight_layout()
    plt.show()
    
    # Visualize feature importance stability
    for quarter in quarters:
        if quarter in feature_importance_data:
            # Create feature importance distribution plot
            plt.figure(figsize=(15, 8))
            
            # Extract importance data for this quarter
            fi_data = feature_importance_data[quarter]
            
            # Prepare data for boxplot
            boxplot_data = []
            feature_names = []
            
            # Sort features by mean importance
            sorted_features = sorted(fi_data['mean'].items(), key=lambda x: x[1], reverse=True)
            
            for feature, _ in sorted_features:
                # Get importance across folds
                feature_imp = [fold[feature] for fold in fi_data['by_fold']]
                boxplot_data.append(feature_imp)
                feature_names.append(feature)
            
            # Create boxplot
            plt.boxplot(boxplot_data, vert=False, labels=feature_names)
            plt.title(f'Feature Importance Stability for {quarter.upper()} (RandomForest)')
            plt.xlabel('Importance')
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
            
            # Create coefficient of variation bar chart
            plt.figure(figsize=(15, 8))
            
            # Prepare data for bar chart
            features = []
            cv_values = []
            
            # Use same sorting as boxplot
            for feature, _ in sorted_features:
                features.append(feature)
                cv_values.append(fi_data['cv'][feature])
            
            # Create bar chart
            plt.barh(features, cv_values)
            plt.title(f'Feature Importance Stability (CV) for {quarter.upper()}')
            plt.xlabel('Coefficient of Variation (lower = more stable)')
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
    
    # Summary of overfitting analysis with recommendations
    print("\nOverfitting Analysis Summary:")
    for quarter, metrics in overfitting_metrics.items():
        print(f"\n{quarter.upper()} Model:")
        print(f"  Train/Test RMSE Gap: {metrics['train_test_gap']:.2f}")
        print(f"  Learning Curve Gap: {metrics['learning_curve_gap']:.2f}")
        print(f"  Overfitting Assessment: {metrics['severity']} risk")
        
        # Provide recommendations based on overfitting severity
        print("  Recommendations:")
        if metrics['severity'] == 'High':
            print("    - Increase regularization (reduce max_depth, increase min_samples_split)")
            print("    - Consider simpler model or feature reduction")
            print("    - Add more training data if available")
        elif metrics['severity'] == 'Medium':
            print("    - Moderate regularization may help (try max_depth=10)")
            print("    - Consider feature selection to remove less important features")
        else:
            print("    - Model appears well-balanced")
            print("    - Focus on feature engineering for potential performance gains")
    
    # Feature stability summary
    print("\nFeature Stability Summary:")
    for quarter, stability in feature_stability.items():
        print(f"\n{quarter.upper()} Model:")
        print(f"  Mean Rank Stability: {stability['mean_rank_stability']:.2f} (1.0 = perfectly stable)")
        
        # List top 3 most stable and unstable features
        importance_stats = stability['importance_stats']
        sorted_by_cv = sorted(importance_stats.items(), key=lambda x: x[1]['cv'])
        
        print("  Most stable features:")
        for feature, stats in sorted_by_cv[:3]:
            print(f"    - {feature}: CV={stats['cv']:.2f}, Mean Importance={stats['mean']:.4f}")
            
        print("  Least stable features:")
        for feature, stats in sorted_by_cv[-3:]:
            print(f"    - {feature}: CV={stats['cv']:.2f}, Mean Importance={stats['mean']:.4f}")
    
    return {
        'results': results_df,
        'learning_curves': learning_curves_data,
        'feature_importance': feature_importance_data,
        'overfitting_metrics': overfitting_metrics,
        'feature_stability': feature_stability
    }

# Run RandomForest investigation using features_df and quarter_feature_sets
if __name__ == "__main__":
    # Define quarter feature sets if not already defined
    if 'quarter_feature_sets' not in globals():
        quarter_feature_sets = {
            'q1': ['rest_days_home', 'rest_days_away', 'is_back_to_back_home', 'is_back_to_back_away', 'prev_matchup_diff'],
            'q2': ['home_q1', 'away_q1', 'rest_advantage', 'prev_matchup_diff'],
            'q3': ['home_q1', 'home_q2', 'away_q1', 'away_q2', 'q1_to_q2_momentum', 'first_half_diff', 'rest_advantage'],
            'q4': ['home_q1', 'home_q2', 'home_q3', 'away_q1', 'away_q2', 'away_q3', 'q1_to_q2_momentum', 
                  'q2_to_q3_momentum', 'pre_q4_diff']
        }
    
    # Use features_df from the notebook environment if available
    if 'features_df' in globals() and not features_df.empty:
        rf_investigation = investigate_random_forest(features_df, quarter_feature_sets)
    else:
        print("No valid features_df found. Please generate or load features data first.")

In [ ]:
# Cell 4F-2: Enhanced Prediction Evolution Visualization

def create_prediction_evolution_chart(game_data, prediction_history):
    """
    Create visualization showing how predictions evolve throughout the game.
    
    Args:
        game_data: Current game information
        prediction_history: List of historical predictions
        
    Returns:
        matplotlib Figure with prediction evolution
    """
    # Create figure
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={'height_ratios': [3, 1]})
    
    # Extract data for plotting
    quarters = []
    home_scores = []
    away_scores = []
    home_predictions = []
    away_predictions = []
    win_probabilities = []
    timestamps = []
    
    for pred in prediction_history:
        quarters.append(pred.get('quarter', 0))
        home_scores.append(pred.get('current_home_score', 0))
        away_scores.append(pred.get('current_away_score', 0))
        home_predictions.append(pred.get('predicted_home_score', 0))
        away_predictions.append(pred.get('predicted_away_score', 0))
        win_probabilities.append(pred.get('win_probability', 0.5))
        timestamps.append(pred.get('timestamp'))
    
    # Only plot if we have data
    if not quarters:
        ax1.text(0.5, 0.5, "No prediction history available", 
                 ha='center', va='center')
        return fig
    
    # Convert to numpy arrays for easier manipulation
    quarters = np.array(quarters)
    x_values = np.arange(len(quarters))
    
    # Plot scores and predictions on upper axis
    ax1.plot(x_values, home_scores, 'bo-', label=f"{game_data['home_team']} Actual")
    ax1.plot(x_values, away_scores, 'ro-', label=f"{game_data['away_team']} Actual")
    ax1.plot(x_values, home_predictions, 'b--', label=f"{game_data['home_team']} Predicted")
    ax1.plot(x_values, away_predictions, 'r--', label=f"{game_data['away_team']} Predicted")
    
    # Add confidence intervals if available
    if 'home_lower_bound' in prediction_history[0] and 'home_upper_bound' in prediction_history[0]:
        home_lower = [p.get('home_lower_bound', p.get('predicted_home_score', 0)) for p in prediction_history]
        home_upper = [p.get('home_upper_bound', p.get('predicted_home_score', 0)) for p in prediction_history]
        ax1.fill_between(x_values, home_lower, home_upper, color='blue', alpha=0.2)
        
        away_lower = [p.get('away_lower_bound', p.get('predicted_away_score', 0)) for p in prediction_history]
        away_upper = [p.get('away_upper_bound', p.get('predicted_away_score', 0)) for p in prediction_history]
        ax1.fill_between(x_values, away_lower, away_upper, color='red', alpha=0.2)
    
    # Set up the primary axis
    ax1.set_title(f"Prediction Evolution: {game_data['home_team']} vs {game_data['away_team']}")
    ax1.set_ylabel("Score")
    ax1.legend(loc='upper left')
    ax1.grid(True, alpha=0.3)
    
    # Use quarter labels for x-axis
    quarter_labels = ["Pre" if q == 0 else f"Q{q}" for q in quarters]
    ax1.set_xticks(x_values)
    ax1.set_xticklabels(quarter_labels)
    
    # Plot win probability on lower axis
    ax2.plot(x_values, win_probabilities, 'g-', marker='o')
    ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)  # 50% reference line
    
    # Fill above/below 50%
    ax2.fill_between(x_values, 0.5, win_probabilities, 
                     where=(np.array(win_probabilities) >= 0.5), 
                     color='green', alpha=0.3)
    ax2.fill_between(x_values, win_probabilities, 0.5, 
                     where=(np.array(win_probabilities) <= 0.5), 
                     color='red', alpha=0.3)
    
    # Set up the secondary axis
    ax2.set_ylabel("Win Probability")
    ax2.set_xlabel("Game Progression")
    ax2.set_ylim(0, 1)
    ax2.set_xticks(x_values)
    ax2.set_xticklabels(quarter_labels)
    ax2.grid(True, alpha=0.3)
    
    # Format y-axis as percentage
    ax2.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
    
    # Add annotations for significant probability changes
    for i in range(1, len(win_probabilities)):
        change = win_probabilities[i] - win_probabilities[i-1]
        if abs(change) > 0.15:  # Only annotate significant changes
            ax2.annotate(f"{change*100:+.1f}%", 
                         xy=(x_values[i], win_probabilities[i]),
                         xytext=(0, 10 if change > 0 else -10),
                         textcoords='offset points',
                         ha='center',
                         arrowprops=dict(arrowstyle='->', color='black'))
    
    plt.tight_layout()
    return fig

# Create function to generate sample prediction history for testing
def generate_sample_prediction_history(quarters=4):
    """Generate sample game data and prediction history for testing visualization"""
    # Game data
    game_data = {
        'game_id': 1001,
        'home_team': 'Boston Celtics',
        'away_team': 'Los Angeles Lakers'
    }
    
    # Prediction history
    history = []
    
    # Starting at pre-game (quarter 0)
    for q in range(quarters + 1):
        # Current scores (0 for pre-game, increasing for each quarter)
        home_score = 0 if q == 0 else sum([25 + i for i in range(q)])
        away_score = 0 if q == 0 else sum([23 + (i % 3) for i in range(q)])
        
        # Generate progressively more accurate predictions
        error_factor = 1.0 - (q * 0.2)
        home_pred = 105 + np.random.normal(0, 10 * error_factor)
        away_pred = 98 + np.random.normal(0, 10 * error_factor)
        
        # Calculate win probability
        score_diff = home_pred - away_pred
        win_prob = 1 / (1 + np.exp(-0.1 * score_diff))
        
        # Add prediction point
        history.append({
            'quarter': q,
            'current_home_score': home_score,
            'current_away_score': away_score,
            'predicted_home_score': home_pred,
            'predicted_away_score': away_pred,
            'win_probability': win_prob,
            'timestamp': datetime.now() - timedelta(minutes=(4-q)*15),
            # Add bounds for confidence intervals
            'home_lower_bound': home_pred - (15 * error_factor),
            'home_upper_bound': home_pred + (15 * error_factor),
            'away_lower_bound': away_pred - (15 * error_factor),
            'away_upper_bound': away_pred + (15 * error_factor)
        })
    
    return game_data, history

# Test the prediction evolution chart
print("Testing Prediction Evolution Chart:")
sample_game, sample_history = generate_sample_prediction_history()
evolution_fig = create_prediction_evolution_chart(sample_game, sample_history)
plt.figure(evolution_fig.number)
plt.show()

# Create a dashboard-style visualization function 
def create_game_prediction_dashboard(predictions_df, prediction_history=None):
    """
    Create a comprehensive dashboard for multiple game predictions.
    
    Args:
        predictions_df: DataFrame with game predictions
        prediction_history: Optional dict mapping game_id to prediction history
        
    Returns:
        matplotlib Figure with dashboard
    """
    # Determine how many games to display
    n_games = min(len(predictions_df), 4)  # Max 4 games per dashboard
    
    if n_games == 0:
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.text(0.5, 0.5, "No games available for dashboard",
                ha='center', va='center', fontsize=14)
        return fig
    
    # Create figure with adaptive grid layout
    if n_games == 1:
        fig, axs = plt.subplots(1, 2, figsize=(15, 7))
        axs = np.array([axs])  # Make 2D for consistent indexing
    elif n_games == 2:
        fig, axs = plt.subplots(2, 2, figsize=(15, 12))
    else:
        # For 3-4 games
        fig, axs = plt.subplots(2, 2, figsize=(15, 12))
    
    fig.suptitle("NBA Score Prediction Dashboard", fontsize=16)
    
    # Process each game
    for i, (_, game) in enumerate(predictions_df.head(n_games).iterrows()):
        if i >= axs.shape[0] * axs.shape[1]:
            break
            
        row, col = i // 2, i % 2
        ax = axs[row, col]
        
        # Extract game data
        home_team = game['home_team']
        away_team = game['away_team']
        current_quarter = int(game.get('current_quarter', 0))
        home_score = float(game.get('current_home_score', game.get('home_score', 0)))
        away_score = float(game.get('current_away_score', game.get('away_score', 0)))
        home_pred = float(game.get('predicted_home_score', 0))
        away_pred = float(game.get('predicted_away_score', 0))
        
        # Determine status text
        if current_quarter == 0:
            status_text = "Pre-Game"
        elif game.get('is_finished', False):
            status_text = "Final"
        else:
            status_text = f"Q{current_quarter}"
        
        # Create bars for current and predicted scores
        teams = [home_team, away_team]
        current_scores = [home_score, away_score]
        predicted_scores = [home_pred, away_pred]
        
        x = np.arange(len(teams))
        width = 0.35
        
        ax.bar(x - width/2, current_scores, width, label='Current', color='lightblue')
        ax.bar(x + width/2, predicted_scores, width, label='Predicted Final', color='salmon')
        
        # Add score labels
        for i, score in enumerate(current_scores):
            ax.text(x[i] - width/2, score + 1, f"{score:.0f}", ha='center', fontweight='bold')
            
        for i, score in enumerate(predicted_scores):
            ax.text(x[i] + width/2, score + 1, f"{score:.1f}", ha='center', fontweight='bold')
        
        # Add win probability if available
        if 'win_probability' in game:
            win_prob = game['win_probability'] * 100
            ax.text(0.5, 0.05, f"Home Win: {win_prob:.1f}%", 
                   transform=ax.transAxes, ha='center', 
                   bbox=dict(facecolor='lightgreen' if win_prob > 50 else 'lightcoral', 
                             alpha=0.3, boxstyle='round'))
        
        # Configure axis
        ax.set_title(f"{home_team} vs {away_team} - {status_text}")
        ax.set_xticks(x)
        ax.set_xticklabels(teams)
        ax.legend()
        
        # Add confidence indicator if available
        if 'prediction_confidence' in game:
            conf = game['prediction_confidence'] * 100
            ax.text(0.95, 0.95, f"Confidence: {conf:.0f}%",
                   transform=ax.transAxes, ha='right', va='top',
                   bbox=dict(facecolor='white', alpha=0.7, boxstyle='round'))
            
        # Add prediction evolution if history available
        if prediction_history is not None and game.get('game_id') in prediction_history:
            history = prediction_history[game.get('game_id')]
            if len(history) > 1:
                # Add a small plot in the upper right corner showing trend
                # Create an inset for the trend
                sub_ax = ax.inset_axes([0.65, 0.65, 0.3, 0.3])
                
                # Extract trend data
                x_trend = range(len(history))
                y_home = [h.get('predicted_home_score', 0) for h in history]
                y_away = [h.get('predicted_away_score', 0) for h in history]
                
                # Plot trend lines
                sub_ax.plot(x_trend, y_home, 'b-', label='Home')
                sub_ax.plot(x_trend, y_away, 'r-', label='Away')
                
                # Configure inset
                sub_ax.set_title("Prediction Trend", fontsize=8)
                sub_ax.tick_params(labelsize=7)
                sub_ax.grid(alpha=0.3)
    
    # Adjust layout
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Leave room for suptitle
    
    return fig

# Test the dashboard visualization with sample data
print("\nTesting Prediction Dashboard:")

# Generate sample predictions for multiple games
sample_predictions = pd.DataFrame([
    {
        'game_id': 1001,
        'home_team': 'Boston Celtics',
        'away_team': 'Los Angeles Lakers',
        'current_quarter': 2,
        'home_score': 53,
        'away_score': 48,
        'predicted_home_score': 104.5,
        'predicted_away_score': 99.2,
        'win_probability': 0.68,
        'prediction_confidence': 0.65
    },
    {
        'game_id': 1002,
        'home_team': 'Golden State Warriors',
        'away_team': 'Brooklyn Nets',
        'current_quarter': 3,
        'home_score': 78,
        'away_score': 81,
        'predicted_home_score': 107.8,
        'predicted_away_score': 109.3,
        'win_probability': 0.42,
        'prediction_confidence': 0.75
    },
    {
        'game_id': 1003,
        'home_team': 'Milwaukee Bucks',
        'away_team': 'Denver Nuggets',
        'current_quarter': 0,
        'home_score': 0,
        'away_score': 0,
        'predicted_home_score': 112.3,
        'predicted_away_score': 108.7,
        'win_probability': 0.61,
        'prediction_confidence': 0.50
    }
])

# Generate sample prediction history for one game
sample_prediction_history = {
    1001: [
        {
            'predicted_home_score': 108.2,
            'predicted_away_score': 104.5,
            'quarter': 0
        },
        {
            'predicted_home_score': 106.5,
            'predicted_away_score': 101.8,
            'quarter': 1
        },
        {
            'predicted_home_score': 104.5,
            'predicted_away_score': 99.2,
            'quarter': 2
        }
    ]
}

# Create dashboard
dashboard_fig = create_game_prediction_dashboard(sample_predictions, sample_prediction_history)
plt.figure(dashboard_fig.number)
plt.show()

In [ ]:
# Cell 4F-3: Enhanced Error Analysis Pipeline

def conduct_error_analysis(validation_results, prediction_data):
    """
    Detailed error analysis to identify patterns in prediction accuracy
    
    Args:
        validation_results: DataFrame with validation metrics
        prediction_data: DataFrame with detailed prediction info
        
    Returns:
        Dict of error analysis results
    """
    analysis = {}
    
    # Group by quarter
    quarter_analysis = {}
    for quarter in range(5):  # 0-4
        quarter_preds = prediction_data[prediction_data['current_quarter'] == quarter]
        if quarter_preds.empty:
            continue
            
        # Calculate errors
        quarter_preds['home_error'] = quarter_preds['predicted_home_score'] - quarter_preds['actual_home_score']
        quarter_preds['away_error'] = quarter_preds['predicted_away_score'] - quarter_preds['actual_away_score']
        quarter_preds['abs_home_error'] = quarter_preds['home_error'].abs()
        quarter_preds['abs_away_error'] = quarter_preds['away_error'].abs()
        
        # Error distribution
        quarter_analysis[quarter] = {
            'sample_size': len(quarter_preds),
            'home_error_mean': quarter_preds['home_error'].mean(),
            'home_error_std': quarter_preds['home_error'].std(),
            'home_error_median': quarter_preds['home_error'].median(),
            'away_error_mean': quarter_preds['away_error'].mean(),
            'away_error_std': quarter_preds['away_error'].std(),
            'away_error_median': quarter_preds['away_error'].median(),
            'error_histogram': quarter_preds[['abs_home_error', 'abs_away_error']].describe()
        }
        
        # Check for bias patterns
        score_differential = quarter_preds['actual_home_score'] - quarter_preds['actual_away_score']
        quarter_preds['is_home_favorite'] = score_differential > 0
        quarter_preds['home_favored_error'] = quarter_preds['home_error'][quarter_preds['is_home_favorite']]
        quarter_preds['away_underdog_error'] = quarter_preds['away_error'][quarter_preds['is_home_favorite']]
        
        # Correlation analysis
        corr_factors = [
            'score_differential', 'total_score', 'momentum', 
            'home_win', 'quarter', 'is_home_favorite'
        ]
        
        correlations = {}
        for factor in corr_factors:
            if factor in quarter_preds.columns:
                correlations[f'home_error_vs_{factor}'] = quarter_preds['home_error'].corr(quarter_preds[factor])
                correlations[f'away_error_vs_{factor}'] = quarter_preds['away_error'].corr(quarter_preds[factor])
        
        quarter_analysis[quarter]['correlations'] = correlations
        
    analysis['quarter_analysis'] = quarter_analysis
    
    # Identify systematic error patterns
    # Example: Do we consistently underestimate high-scoring games?
    prediction_data['total_actual'] = prediction_data['actual_home_score'] + prediction_data['actual_away_score']
    prediction_data['total_predicted'] = prediction_data['predicted_home_score'] + prediction_data['predicted_away_score']
    prediction_data['total_error'] = prediction_data['total_predicted'] - prediction_data['total_actual']
    
    # Bin by total score
    bins = [160, 180, 200, 220, 240, 260]
    labels = ['Very Low', 'Low', 'Average', 'High', 'Very High']
    prediction_data['score_category'] = pd.cut(prediction_data['total_actual'], bins=bins, labels=labels)
    
    score_category_analysis = prediction_data.groupby('score_category')['total_error'].agg(['mean', 'std', 'count'])
    analysis['score_category_analysis'] = score_category_analysis
    
    return analysis

In [ ]:
# Cell 4F-4: Finalized AdaptiveEnsemble with Strategy Persistence

import pandas as pd
import numpy as np
import json
import os
from datetime import datetime

class AdaptiveEnsemble:
    """
    Finalized ensemble framework with adaptive weighting strategies and state persistence.
    """
    
    def __init__(self, strategies=None, default_strategy='balanced', persistence_file=None):
        """
        Initialize with weighting strategies and optional persistence.
        
        Args:
            strategies: Dict of named strategies (or None for defaults)
            default_strategy: Strategy to use by default
            persistence_file: File path for saving performance data
        """
        # Define default strategies if none provided
        self.strategies = strategies or {
            'balanced': {
                'description': 'Default balanced weights',
                'q0': {'main': 0.3, 'quarter': 0.7},
                'q1': {'main': 0.4, 'quarter': 0.6},
                'q2': {'main': 0.6, 'quarter': 0.4},
                'q3': {'main': 0.8, 'quarter': 0.2},
                'q4': {'main': 0.95, 'quarter': 0.05}
            },
            'momentum_driven': {
                'description': 'Increased weight to quarter models when momentum is strong',
                'q0': {'main': 0.3, 'quarter': 0.7},
                'q1': {'main': 0.35, 'quarter': 0.65},
                'q2': {'main': 0.5, 'quarter': 0.5},
                'q3': {'main': 0.7, 'quarter': 0.3},
                'q4': {'main': 0.9, 'quarter': 0.1}
            },
            'conservative': {
                'description': 'Heavier weighting toward main model',
                'q0': {'main': 0.4, 'quarter': 0.6},
                'q1': {'main': 0.5, 'quarter': 0.5},
                'q2': {'main': 0.7, 'quarter': 0.3},
                'q3': {'main': 0.85, 'quarter': 0.15},
                'q4': {'main': 0.98, 'quarter': 0.02}
            },
            'dynamic': {
                'description': 'Fully dynamic context-based weighting',
                'implementation': self._create_dynamic_context_weighting
            },
            'adaptive': {
                'description': 'Adapts based on historical performance',
                'implementation': self._create_adaptive_weighting
            }
        }
        
        self.default_strategy = default_strategy
        self.prediction_history = {}
        self.strategy_performance = {}
        
        # Set up persistence
        self.persistence_file = persistence_file or os.path.join("data", "ensemble_strategies.json")
        os.makedirs(os.path.dirname(self.persistence_file), exist_ok=True)
        
        # Load performance data if available
        self.load_performance_data()
    
    def _create_dynamic_context_weighting(self, current_quarter, time_remaining_mins, score_differential, momentum):
        """
        Calculate dynamic ensemble weights based on game context and smooth transitions.
        
        Args:
            current_quarter: Current quarter (1-4)
            time_remaining_mins: Minutes remaining in the game
            score_differential: Absolute point differential
            momentum: Normalized momentum value (-1 to 1)
            
        Returns:
            Dict of weights for different models
        """
        # Calculate game progression on a continuous 0-1 scale
        total_mins = 48.0
        elapsed_mins = total_mins - time_remaining_mins
        game_progress = min(elapsed_mins / total_mins, 1.0)
        
        # Base weights that shift as game progresses (smooth sigmoid transition)
        main_base_weight = 0.3 + (0.7 * (1 / (1 + np.exp(-10 * (game_progress - 0.5)))))
        quarter_base_weight = 1.0 - main_base_weight
        
        # Context adjustments
        # Game closeness adjustment
        is_close_game = score_differential < 8
        closeness_factor = max(0.0, (8 - score_differential) / 8) * 0.1
        
        # Momentum adjustment - stronger momentum gives more weight to quarter models
        momentum_strength = abs(momentum)
        momentum_factor = momentum_strength * 0.1
        
        # Apply context adjustments to base weights
        if is_close_game:
            # In close games, quarter-specific models get more weight
            main_weight = max(0.2, main_base_weight - closeness_factor)
            quarter_weight = min(0.8, quarter_base_weight + closeness_factor)
        else:
            # In blowouts, main model gets more weight
            main_weight = min(0.95, main_base_weight + (1 - closeness_factor) * 0.05)
            quarter_weight = max(0.05, quarter_base_weight - (1 - closeness_factor) * 0.05)
        
        # Apply momentum adjustment
        if momentum_strength > 0.3:
            # Strong momentum gives more weight to quarter models
            main_weight = max(0.2, main_weight - momentum_factor)
            quarter_weight = min(0.8, quarter_weight + momentum_factor)
        
        # Add historical model weight as a small constant
        historical_weight = 0.0
        
        # Normalize weights to ensure they sum to 1.0
        total = main_weight + quarter_weight + historical_weight
        weights = {
            'main_model': main_weight / total,
            'quarter_model': quarter_weight / total,
            'historical_model': historical_weight / total
        }
        
        return weights
    
    def _create_adaptive_weighting(self, current_quarter, time_remaining_mins, score_differential, momentum):
        """
        Create weights that adapt based on historical performance.
        
        Args:
            current_quarter: Current quarter (1-4)
            time_remaining_mins: Minutes remaining in the game
            score_differential: Absolute point differential
            momentum: Normalized momentum value (-1 to 1)
            
        Returns:
            Dict of weights for different models
        """
        # Start with dynamic context weighting as a base
        weights = self._create_dynamic_context_weighting(
            current_quarter, time_remaining_mins, score_differential, momentum
        )
        
        # If we have performance data, adjust based on historical accuracy
        if self.strategy_performance:
            # Look for similar game contexts
            context_key = f"q{current_quarter}"
            
            if context_key in self.strategy_performance:
                context_perf = self.strategy_performance[context_key]
                
                # Find the best performing strategy for this context
                best_strategy = None
                best_score = float('inf')
                
                for strategy, metrics in context_perf.items():
                    if 'rmse' in metrics and metrics['rmse'] < best_score:
                        best_score = metrics['rmse']
                        best_strategy = strategy
                
                # If we found a strategy with good performance, adjust toward it
                if best_strategy and best_strategy in self.strategies:
                    strategy_weights = self.get_weights(best_strategy, current_quarter)
                    
                    # Blend with dynamic weights (75% historical best, 25% dynamic)
                    for key in weights:
                        if key in strategy_weights:
                            weights[key] = 0.75 * strategy_weights[key] + 0.25 * weights[key]
        
        return weights
    
    def select_strategy(self, game_context):
        """
        Automatically select the best strategy based on game context.
        
        Args:
            game_context: Dict with game state information
            
        Returns:
            str: Name of the selected strategy
        """
        # Extract context variables
        score_diff = abs(game_context.get('score_differential', 0))
        momentum = abs(game_context.get('momentum', 0))
        quarter = game_context.get('current_quarter', 0)
        margin_trend = game_context.get('margin_trend', 0)  # Positive if lead is growing
        
        # Decision logic for strategy selection
        if score_diff < 5:  # Close game
            if momentum > 0.5:  # Strong momentum
                return 'momentum_driven'
            else:
                return 'balanced'
        elif quarter >= 3:  # Late game
            if score_diff > 15:  # Clear blowout
                return 'conservative'  # Trust main model more
            elif margin_trend < 0:  # Lead is shrinking
                return 'adaptive'  # Use adaptive strategy for comeback scenarios
            else:
                return 'balanced'
        else:
            return self.default_strategy
    
    def get_weights(self, strategy_name, quarter, game_context=None):
        """
        Get weights for a specific strategy and quarter.
        
        Args:
            strategy_name: Name of the strategy to use
            quarter: Current quarter (0-4)
            game_context: Optional game context for dynamic strategies
            
        Returns:
            Dict of weights
        """
        strategy = self.strategies.get(strategy_name, self.strategies[self.default_strategy])
        
        # Handle dynamic strategies
        if 'implementation' in strategy:
            if game_context is None:
                # Fall back to balanced strategy if no context
                return self.strategies['balanced'][f'q{quarter}']
            
            # Call the dynamic implementation
            try:
                return strategy['implementation'](
                    quarter,
                    game_context.get('time_remaining_mins', (4 - quarter) * 12),
                    game_context.get('score_differential', 0),
                    game_context.get('momentum', 0)
                )
            except Exception as e:
                print(f"Error in dynamic strategy implementation: {e}")
                return self.strategies['balanced'][f'q{quarter}']
        
        # For static strategies, return the quarter-specific weights
        return strategy.get(f'q{quarter}', strategy.get('q0', {'main': 0.5, 'quarter': 0.5}))
    
    def predict(self, main_prediction, quarter_prediction, quarter, game_context=None, strategy_name=None):
        """
        Generate ensemble prediction with the appropriate weighting.
        
        Args:
            main_prediction: Prediction from main model
            quarter_prediction: Prediction from quarter-specific model
            quarter: Current quarter (0-4)
            game_context: Optional game context data
            strategy_name: Optional strategy override
            
        Returns:
            Tuple of (prediction, weights_used, confidence)
        """
        # Select strategy if not specified
        if strategy_name is None:
            if game_context is not None:
                strategy_name = self.select_strategy(game_context)
            else:
                strategy_name = self.default_strategy
        
        # Get weights for this strategy and quarter
        weights = self.get_weights(strategy_name, quarter, game_context)
        
        # Calculate weighted prediction
        if isinstance(main_prediction, (list, np.ndarray)) and isinstance(quarter_prediction, (list, np.ndarray)):
            # Handle array inputs
            ensemble_prediction = (
                np.array(main_prediction) * weights['main_model'] + 
                np.array(quarter_prediction) * weights['quarter_model']
            )
        else:
            # Handle scalar inputs
            ensemble_prediction = (
                main_prediction * weights['main_model'] + 
                quarter_prediction * weights['quarter_model']
            )
        
        # Calculate confidence based on quarter
        base_confidence = 0.5 + (quarter * 0.1)  # 0.5, 0.6, 0.7, 0.8, 0.9
        
        # Store prediction details for history
        prediction_data = {
            'main_prediction': main_prediction,
            'quarter_prediction': quarter_prediction,
            'ensemble_prediction': ensemble_prediction,
            'weights': weights,
            'strategy': strategy_name,
            'quarter': quarter,
            'confidence': base_confidence,
            'timestamp': datetime.now()
        }
        
        # Add to history using a unique key if context provides one
        history_key = game_context.get('game_id', str(datetime.now().timestamp())) if game_context else str(datetime.now().timestamp())
        if history_key not in self.prediction_history:
            self.prediction_history[history_key] = []
        self.prediction_history[history_key].append(prediction_data)
        
        return ensemble_prediction, weights, base_confidence
    
    def update_performance(self, game_id, actual_final_score):
        """
        Update strategy performance metrics based on actual game results.
        
        Args:
            game_id: Identifier for the game
            actual_final_score: The actual final score of the game
        """
        if game_id not in self.prediction_history:
            return
        
        predictions = self.prediction_history[game_id]
        
        for pred in predictions:
            quarter = pred.get('quarter', 0)
            strategy = pred.get('strategy', self.default_strategy)
            predicted_score = pred.get('ensemble_prediction', 0)
            
            # Calculate error
            error = abs(predicted_score - actual_final_score)
            squared_error = error ** 2
            
            # Update performance tracking
            context_key = f"q{quarter}"
            
            if context_key not in self.strategy_performance:
                self.strategy_performance[context_key] = {}
            
            if strategy not in self.strategy_performance[context_key]:
                self.strategy_performance[context_key][strategy] = {
                    'count': 0,
                    'error_sum': 0,
                    'squared_error_sum': 0
                }
            
            # Update metrics
            perf = self.strategy_performance[context_key][strategy]
            perf['count'] += 1
            perf['error_sum'] += error
            perf['squared_error_sum'] += squared_error
            
            # Calculate running averages
            perf['mae'] = perf['error_sum'] / perf['count']
            perf['rmse'] = np.sqrt(perf['squared_error_sum'] / perf['count'])
        
        # Save updated performance data
        self.save_performance_data()
    
    def load_performance_data(self):
        """Load strategy performance data from file if available."""
        if os.path.exists(self.persistence_file):
            try:
                with open(self.persistence_file, 'r') as f:
                    self.strategy_performance = json.load(f)
                print(f"Loaded strategy performance data from {self.persistence_file}")
            except Exception as e:
                print(f"Error loading strategy performance data: {e}")
    
    def save_performance_data(self):
        """Save strategy performance data to file."""
        try:
            with open(self.persistence_file, 'w') as f:
                json.dump(self.strategy_performance, f)
            print(f"Saved strategy performance data to {self.persistence_file}")
        except Exception as e:
            print(f"Error saving strategy performance data: {e}")
    
    def get_best_strategy_for_context(self, quarter, score_differential, momentum):
        """
        Determine the best strategy for a given game context based on historical performance.
        
        Args:
            quarter: Current quarter (0-4)
            score_differential: Absolute score difference
            momentum: Momentum indicator (-1 to 1)
            
        Returns:
            str: Name of the best strategy for this context
        """
        context_key = f"q{quarter}"
        
        if context_key in self.strategy_performance:
            best_strategy = None
            best_score = float('inf')
            
            for strategy, metrics in self.strategy_performance[context_key].items():
                if 'rmse' in metrics and metrics['rmse'] < best_score:
                    best_score = metrics['rmse']
                    best_strategy = strategy
            
            if best_strategy:
                return best_strategy
        
        # Fall back to default selection logic if no performance data
        if score_differential < 5:  # Close game
            if abs(momentum) > 0.5:  # Strong momentum
                return 'momentum_driven'
            else:
                return 'balanced'
        elif quarter >= 3:  # Late game
            if score_differential > 15:  # Clear blowout
                return 'conservative'  # Trust main model more
            else:
                return 'balanced'
        else:
            return self.default_strategy

In [ ]:
# Cell 4F-5: Prediction History Management System

import os
import json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

class PredictionHistoryManager:
    """
    Efficient system for managing prediction history with persistence,
    automatic pruning, and export/import capabilities.
    """
    
    def __init__(self, data_dir="data", history_file="prediction_history.json", max_age_days=14):
        """
        Initialize the prediction history manager.
        
        Args:
            data_dir: Directory for data storage
            history_file: Filename for prediction history
            max_age_days: Maximum age in days for stored predictions
        """
        self.data_dir = data_dir
        self.history_file = os.path.join(data_dir, history_file)
        self.max_age_days = max_age_days
        self.prediction_history = {}
        
        # Create data directory if it doesn't exist
        os.makedirs(data_dir, exist_ok=True)
        
        # Load existing history if available
        self.load_history()
    
    def load_history(self):
        """Load prediction history from file if available."""
        if os.path.exists(self.history_file):
            try:
                with open(self.history_file, 'r') as f:
                    raw_history = json.load(f)
                
                # Convert string timestamps back to datetime objects
                for game_id, predictions in raw_history.items():
                    for pred in predictions:
                        if 'timestamp' in pred and isinstance(pred['timestamp'], str):
                            pred['timestamp'] = datetime.fromisoformat(pred['timestamp'])
                
                self.prediction_history = raw_history
                print(f"Loaded prediction history for {len(self.prediction_history)} games")
            except Exception as e:
                print(f"Error loading prediction history: {e}")
    
    def save_history(self):
        """Save prediction history to file."""
        if not self.prediction_history:
            return
            
        try:
            # Convert datetime objects to ISO format strings for JSON serialization
            serializable_history = {}
            
            for game_id, predictions in self.prediction_history.items():
                serializable_predictions = []
                for pred in predictions:
                    pred_copy = pred.copy()
                    if 'timestamp' in pred_copy and isinstance(pred_copy['timestamp'], datetime):
                        pred_copy['timestamp'] = pred_copy['timestamp'].isoformat()
                    serializable_predictions.append(pred_copy)
                
                serializable_history[str(game_id)] = serializable_predictions
            
            with open(self.history_file, 'w') as f:
                json.dump(serializable_history, f)
            
            print(f"Saved prediction history to {self.history_file}")
        except Exception as e:
            print(f"Error saving prediction history: {e}")
    
    def add_prediction(self, game_id, prediction_data):
        """
        Add a prediction to the history.
        
        Args:
            game_id: Unique identifier for the game
            prediction_data: Dict with prediction information
            
        Returns:
            bool: True if added successfully
        """
        try:
            # Make sure we have a timestamp
            if 'timestamp' not in prediction_data:
                prediction_data['timestamp'] = datetime.now()
            
            # Add to history
            game_id_str = str(game_id)
            if game_id_str not in self.prediction_history:
                self.prediction_history[game_id_str] = []
            
            self.prediction_history[game_id_str].append(prediction_data)
            return True
            
        except Exception as e:
            print(f"Error adding prediction: {e}")
            return False
    
    def add_multiple_predictions(self, predictions_df):
        """
        Add multiple predictions from a DataFrame.
        
        Args:
            predictions_df: DataFrame with prediction data
            
        Returns:
            int: Number of added predictions
        """
        count = 0
        
        for _, row in predictions_df.iterrows():
            game_id = row.get('game_id')
            if game_id:
                prediction_data = row.to_dict()
                if 'timestamp' not in prediction_data:
                    prediction_data['timestamp'] = datetime.now()
                
                success = self.add_prediction(game_id, prediction_data)
                if success:
                    count += 1
        
        return count
    
    def prune_old_data(self):
        """
        Remove predictions older than max_age_days.
        
        Returns:
            int: Number of pruned games
        """
        if not self.prediction_history:
            return 0
        
        cutoff_date = datetime.now() - timedelta(days=self.max_age_days)
        pruned_history = {}
        pruned_count = 0
        
        for game_id, predictions in self.prediction_history.items():
            # Check the most recent prediction timestamp
            if predictions and 'timestamp' in predictions[-1]:
                last_timestamp = predictions[-1]['timestamp']
                if isinstance(last_timestamp, str):
                    try:
                        last_timestamp = datetime.fromisoformat(last_timestamp)
                    except:
                        # If parsing fails, use the current time to avoid data loss
                        last_timestamp = datetime.now()
                
                if last_timestamp > cutoff_date:
                    pruned_history[game_id] = predictions
                else:
                    pruned_count += 1
        
        if pruned_count > 0:
            self.prediction_history = pruned_history
            print(f"Pruned {pruned_count} old games from history")
            
            # Save updated history
            self.save_history()
        
        return pruned_count
    
    def get_game_predictions(self, game_id):
        """
        Get prediction history for a specific game.
        
        Args:
            game_id: Game identifier
            
        Returns:
            list: Prediction history for the game (or empty list if not found)
        """
        game_id_str = str(game_id)
        return self.prediction_history.get(game_id_str, [])
    
    def get_latest_prediction(self, game_id):
        """
        Get the most recent prediction for a game.
        
        Args:
            game_id: Game identifier
            
        Returns:
            dict: Most recent prediction (or None if not found)
        """
        predictions = self.get_game_predictions(game_id)
        return predictions[-1] if predictions else None
    
    def export_to_csv(self, output_file=None):
        """
        Export prediction history to CSV format.
        
        Args:
            output_file: Path for the CSV file (or auto-generate if None)
            
        Returns:
            str: Path to the exported file
        """
        if not self.prediction_history:
            print("No prediction history to export")
            return None
        
        # Default filename if not provided
        if output_file is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            output_file = os.path.join(self.data_dir, f"prediction_history_{timestamp}.csv")
        
        # Convert nested dictionary to flat DataFrame
        rows = []
        
        for game_id, predictions in self.prediction_history.items():
            for pred in predictions:
                row = {'game_id': game_id}
                
                # Flatten nested dictionaries
                for key, value in pred.items():
                    if isinstance(value, dict):
                        for nested_key, nested_value in value.items():
                            row[f"{key}_{nested_key}"] = nested_value
                    else:
                        row[key] = value
                
                rows.append(row)
        
        if not rows:
            print("No valid prediction data to export")
            return None
        
        # Create DataFrame and export
        try:
            df = pd.DataFrame(rows)
            df.to_csv(output_file, index=False)
            print(f"Exported {len(df)} prediction records to {output_file}")
            return output_file
        except Exception as e:
            print(f"Error exporting to CSV: {e}")
            return None
    
    def import_from_csv(self, input_file):
        """
        Import prediction history from CSV file.
        
        Args:
            input_file: Path to the CSV file
            
        Returns:
            int: Number of imported predictions
        """
        if not os.path.exists(input_file):
            print(f"Import file not found: {input_file}")
            return 0
        
        try:
            df = pd.read_csv(input_file)
            print(f"Loaded {len(df)} records from {input_file}")
            
            # Process the imported data
            records_by_game = {}
            
            for _, row in df.iterrows():
                game_id = str(row['game_id'])
                
                if game_id not in records_by_game:
                    records_by_game[game_id] = []
                
                # Convert row to dict
                record = row.to_dict()
                
                # Handle timestamp if present
                if 'timestamp' in record:
                    try:
                        if isinstance(record['timestamp'], str):
                            record['timestamp'] = datetime.fromisoformat(record['timestamp'])
                    except:
                        record['timestamp'] = datetime.now()
                
                records_by_game[game_id].append(record)
            
            # Merge with existing history
            for game_id, records in records_by_game.items():
                if game_id in self.prediction_history:
                    # Append new records
                    self.prediction_history[game_id].extend(records)
                else:
                    # Create new entry
                    self.prediction_history[game_id] = records
            
            # Save the updated history
            self.save_history()
            
            return sum(len(records) for records in records_by_game.values())
            
        except Exception as e:
            print(f"Error importing from CSV: {e}")
            return 0
    
    def get_statistics(self):
        """
        Get statistics about the stored prediction history.
        
        Returns:
            dict: Statistics about the prediction history
        """
        if not self.prediction_history:
            return {
                'num_games': 0,
                'num_predictions': 0,
                'oldest_prediction': None,
                'newest_prediction': None
            }
        
        # Calculate statistics
        num_games = len(self.prediction_history)
        num_predictions = sum(len(preds) for preds in self.prediction_history.values())
        
        # Find oldest and newest predictions
        oldest_timestamp = None
        newest_timestamp = None
        
        for predictions in self.prediction_history.values():
            for pred in predictions:
                if 'timestamp' in pred:
                    timestamp = pred['timestamp']
                    if isinstance(timestamp, str):
                        try:
                            timestamp = datetime.fromisoformat(timestamp)
                        except:
                            continue
                    
                    if oldest_timestamp is None or timestamp < oldest_timestamp:
                        oldest_timestamp = timestamp
                    
                    if newest_timestamp is None or timestamp > newest_timestamp:
                        newest_timestamp = timestamp
        
        return {
            'num_games': num_games,
            'num_predictions': num_predictions,
            'oldest_prediction': oldest_timestamp,
            'newest_prediction': newest_timestamp,
            'disk_size_kb': os.path.getsize(self.history_file) / 1024 if os.path.exists(self.history_file) else 0
        }

In [ ]:
# Cell 4F-6 Enhanced Prediction Evolution Chart

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import matplotlib.dates as mdates

def create_enhanced_prediction_evolution_chart(game_data, prediction_history, add_confidence=True):
    """
    Create enhanced visualization showing how predictions evolve throughout the game.
    
    Args:
        game_data: Current game information (dict or Series)
        prediction_history: List of historical predictions
        add_confidence: Whether to add confidence intervals
        
    Returns:
        matplotlib Figure with prediction evolution
    """
    # Create figure
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), gridspec_kw={'height_ratios': [3, 1]})
    
    # Extract data for plotting
    quarters = []
    home_scores = []
    away_scores = []
    home_predictions = []
    away_predictions = []
    win_probabilities = []
    timestamps = []
    home_lower_bounds = []
    home_upper_bounds = []
    away_lower_bounds = []
    away_upper_bounds = []
    
    # Check if game_data is a pandas Series
    if isinstance(game_data, pd.Series):
        game_data = game_data.to_dict()
    
    # Process prediction history
    for pred in prediction_history:
        # Handle both dict and Series types
        if isinstance(pred, pd.Series):
            pred = pred.to_dict()
            
        quarters.append(pred.get('current_quarter', 0))
        home_scores.append(pred.get('current_home_score', pred.get('home_score', 0)))
        away_scores.append(pred.get('current_away_score', pred.get('away_score', 0)))
        home_predictions.append(pred.get('predicted_home_score', 0))
        away_predictions.append(pred.get('predicted_away_score', 0))
        win_probabilities.append(pred.get('win_probability', 0.5))
        timestamps.append(pred.get('timestamp', None))
        
        # Confidence intervals if available
        if add_confidence:
            # Extract bounds if present
            home_lower = pred.get('home_lower_bound')
            home_upper = pred.get('home_upper_bound')
            away_lower = pred.get('away_lower_bound')
            away_upper = pred.get('away_upper_bound')
            
            # If bounds not available, generate them
            if home_lower is None or home_upper is None:
                confidence = pred.get('prediction_confidence', 0.8)
                error_margin = 10 * (1 - confidence)  # More confidence = smaller margin
                home_pred = pred.get('predicted_home_score', 0)
                
                home_lower = home_pred - error_margin
                home_upper = home_pred + error_margin
            
            if away_lower is None or away_upper is None:
                confidence = pred.get('prediction_confidence', 0.8)
                error_margin = 10 * (1 - confidence)
                away_pred = pred.get('predicted_away_score', 0)
                
                away_lower = away_pred - error_margin
                away_upper = away_pred + error_margin
            
            home_lower_bounds.append(home_lower)
            home_upper_bounds.append(home_upper)
            away_lower_bounds.append(away_lower)
            away_upper_bounds.append(away_upper)
    
    # Only plot if we have data
    if not quarters:
        ax1.text(0.5, 0.5, "No prediction history available", 
                 ha='center', va='center', fontsize=14)
        return fig
    
    # Convert to numpy arrays for easier manipulation
    quarters = np.array(quarters)
    x_values = np.arange(len(quarters))
    
    # Use timestamps for x-axis if available
    use_timestamps = all(timestamps) and len(set(timestamps)) > 1
    
    if use_timestamps:
        # Convert timestamps to datetime if they're strings
        datetime_timestamps = []
        for ts in timestamps:
            if isinstance(ts, str):
                try:
                    dt = datetime.fromisoformat(ts)
                    datetime_timestamps.append(dt)
                except:
                    use_timestamps = False
                    break
            elif isinstance(ts, datetime):
                datetime_timestamps.append(ts)
            else:
                use_timestamps = False
                break
        
        if use_timestamps:
            timestamps = datetime_timestamps
            x_values = timestamps
    
    # Plot scores and predictions on upper axis
    home_team = game_data.get('home_team', 'Home')
    away_team = game_data.get('away_team', 'Away')
    
    ax1.plot(x_values, home_scores, 'bo-', label=f"{home_team} Actual")
    ax1.plot(x_values, away_scores, 'ro-', label=f"{away_team} Actual")
    ax1.plot(x_values, home_predictions, 'b--', label=f"{home_team} Predicted")
    ax1.plot(x_values, away_predictions, 'r--', label=f"{away_team} Predicted")
    
    # Add confidence intervals if available
    if add_confidence and home_lower_bounds and home_upper_bounds:
        ax1.fill_between(x_values, home_lower_bounds, home_upper_bounds, color='blue', alpha=0.2)
        ax1.fill_between(x_values, away_lower_bounds, away_upper_bounds, color='red', alpha=0.2)
    
    # Set up the primary axis
    ax1.set_title(f"Prediction Evolution: {home_team} vs {away_team}")
    ax1.set_ylabel("Score")
    ax1.legend(loc='upper left')
    ax1.grid(True, alpha=0.3)
    
    # Format x-axis based on data type
    if use_timestamps:
        # Format datetime x-axis
        ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')
    else:
        # Use quarter labels for x-axis
        quarter_labels = ["Pre" if q == 0 else f"Q{q}" for q in quarters]
        ax1.set_xticks(x_values)
        ax1.set_xticklabels(quarter_labels)
    
    # Plot win probability on lower axis
    ax2.plot(x_values, win_probabilities, 'g-', marker='o')
    ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)  # 50% reference line
    
    # Fill above/below 50%
    ax2.fill_between(x_values, 0.5, win_probabilities, 
                     where=(np.array(win_probabilities) >= 0.5), 
                     color='green', alpha=0.3)
    ax2.fill_between(x_values, win_probabilities, 0.5, 
                     where=(np.array(win_probabilities) <= 0.5), 
                     color='red', alpha=0.3)
    
    # Set up the secondary axis
    ax2.set_ylabel("Win Probability")
    ax2.set_xlabel("Game Progression")
    ax2.set_ylim(0, 1)
    ax2.grid(True, alpha=0.3)
    
    # Format y-axis as percentage
    ax2.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
    
    # Format x-axis based on data type (same as upper plot)
    if use_timestamps:
        # Format datetime x-axis
        ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        plt.setp(ax2.get_xticklabels(), rotation=45, ha='right')
    else:
        # Use quarter labels
        ax2.set_xticks(x_values)
        ax2.set_xticklabels(quarter_labels)
    
    # Add annotations for significant probability changes
    for i in range(1, len(win_probabilities)):
        change = win_probabilities[i] - win_probabilities[i-1]
        if abs(change) > 0.15:  # Only annotate significant changes
            ax2.annotate(f"{change*100:+.1f}%", 
                         xy=(x_values[i], win_probabilities[i]),
                         xytext=(0, 10 if change > 0 else -10),
                         textcoords='offset points',
                         ha='center',
                         arrowprops=dict(arrowstyle='->', color='black'))
    
    plt.tight_layout()
    return fig

In [ ]:
# Cell 4F-7 Advanced Anomaly Detection System

import pandas as pd
import numpy as np
from datetime import datetime
import json
import os
import matplotlib.pyplot as plt

class NBAGameAnomalyDetector:
    """
    Advanced system for detecting anomalies in game predictions and results.
    Identifies unusual patterns, statistical outliers, and potential data issues.
    """
    
    def __init__(self, data_dir="data"):
        """
        Initialize the anomaly detector.
        
        Args:
            data_dir: Directory for storing anomaly data
        """
        self.data_dir = data_dir
        self.anomalies = []
        self.anomaly_file = os.path.join(data_dir, "prediction_anomalies.json")
        
        # Create data directory if it doesn't exist
        os.makedirs(data_dir, exist_ok=True)
        
        # Load existing anomalies if available
        self.load_anomalies()
    
    def load_anomalies(self):
        """Load anomalies from file if available."""
        if os.path.exists(self.anomaly_file):
            try:
                with open(self.anomaly_file, 'r') as f:
                    self.anomalies = json.load(f)
                print(f"Loaded {len(self.anomalies)} anomalies from {self.anomaly_file}")
            except Exception as e:
                print(f"Error loading anomalies: {e}")
    
    def save_anomalies(self):
        """Save anomalies to file."""
        try:
            with open(self.anomaly_file, 'w') as f:
                json.dump(self.anomalies, f)
            print(f"Saved {len(self.anomalies)} anomalies to {self.anomaly_file}")
        except Exception as e:
            print(f"Error saving anomalies: {e}")
    
    def detect_prediction_anomalies(self, current_prediction, prediction_history, thresholds=None):
        """
        Detect anomalies in predictions based on current prediction and history.
        
        Args:
            current_prediction: Current prediction data (dict or Series)
            prediction_history: List of previous predictions for this game
            thresholds: Optional dict with custom thresholds
            
        Returns:
            list: Detected anomalies
        """
        # Convert to dict if Series
        if isinstance(current_prediction, pd.Series):
            current_prediction = current_prediction.to_dict()
        
        # Default thresholds
        default_thresholds = {
            'score_swing': 8.0,      # Points change between predictions
            'margin_swing': 10.0,     # Margin change between predictions
            'win_prob_swing': 0.2,    # Win probability change between predictions
            'score_unrealistic': 150,  # Unrealistically high score prediction
            'score_unrealistic_low': 60  # Unrealistically low score for a completed game
        }
        
        # Use custom thresholds if provided
        if thresholds is None:
            thresholds = default_thresholds
        else:
            # Update with defaults for any missing thresholds
            for key, value in default_thresholds.items():
                if key not in thresholds:
                    thresholds[key] = value
        
        detected_anomalies = []
        
        # If no history, just check for unrealistic scores
        if not prediction_history:
            home_score = float(current_prediction.get('predicted_home_score', 0))
            away_score = float(current_prediction.get('predicted_away_score', 0))
            
            # Check for unrealistically high scores
            if home_score > thresholds['score_unrealistic'] or away_score > thresholds['score_unrealistic']:
                anomaly = {
                    'type': 'unrealistic_score',
                    'game_id': current_prediction.get('game_id', 'unknown'),
                    'timestamp': datetime.now().isoformat(),
                    'details': {
                        'home_score': home_score,
                        'away_score': away_score,
                        'threshold': thresholds['score_unrealistic']
                    },
                    'severity': 'medium'
                }
                detected_anomalies.append(anomaly)
            
            return detected_anomalies
        
        # Get the most recent previous prediction
        prev_prediction = prediction_history[-1]
        
        # Convert to dict if Series
        if isinstance(prev_prediction, pd.Series):
            prev_prediction = prev_prediction.to_dict()
        
        # Extract current values
        home_score = float(current_prediction.get('predicted_home_score', 0))
        away_score = float(current_prediction.get('predicted_away_score', 0))
        win_prob = float(current_prediction.get('win_probability', 0.5))
        
        # Extract previous values
        prev_home = float(prev_prediction.get('predicted_home_score', 0))
        prev_away = float(prev_prediction.get('predicted_away_score', 0))
        prev_win_prob = float(prev_prediction.get('win_probability', 0.5))
        
        # Calculate changes
        home_change = abs(home_score - prev_home)
        away_change = abs(away_score - prev_away)
        win_prob_change = abs(win_prob - prev_win_prob)
        
        # Calculate margin changes
        current_margin = home_score - away_score
        prev_margin = prev_home - prev_away
        margin_change = abs(current_margin - prev_margin)
        
        # Check for large score swings
        if home_change > thresholds['score_swing'] or away_change > thresholds['score_swing']:
            anomaly = {
                'type': 'large_score_swing',
                'game_id': current_prediction.get('game_id', 'unknown'),
                'timestamp': datetime.now().isoformat(),
                'details': {
                    'home_change': home_change,
                    'away_change': away_change,
                    'previous': {'home': prev_home, 'away': prev_away},
                    'current': {'home': home_score, 'away': away_score},
                    'threshold': thresholds['score_swing']
                },
                'severity': 'medium'
            }
            detected_anomalies.append(anomaly)
        
        # Check for large margin swings
        if margin_change > thresholds['margin_swing']:
            anomaly = {
                'type': 'large_margin_swing',
                'game_id': current_prediction.get('game_id', 'unknown'),
                'timestamp': datetime.now().isoformat(),
                'details': {
                    'current_margin': current_margin,
                    'previous_margin': prev_margin,
                    'margin_change': margin_change,
                    'threshold': thresholds['margin_swing']
                },
                'severity': 'high' if margin_change > thresholds['margin_swing'] * 1.5 else 'medium'
            }
            detected_anomalies.append(anomaly)
        
        # Check for large win probability swings
        if win_prob_change > thresholds['win_prob_swing']:
            anomaly = {
                'type': 'large_win_prob_swing',
                'game_id': current_prediction.get('game_id', 'unknown'),
                'timestamp': datetime.now().isoformat(),
                'details': {
                    'current_win_prob': win_prob,
                    'previous_win_prob': prev_win_prob,
                    'win_prob_change': win_prob_change,
                    'threshold': thresholds['win_prob_swing']
                },
                'severity': 'high' if win_prob_change > thresholds['win_prob_swing'] * 1.5 else 'medium'
            }
            detected_anomalies.append(anomaly)
        
        # Store detected anomalies
        if detected_anomalies:
            self.anomalies.extend(detected_anomalies)
            self.save_anomalies()
        
        return detected_anomalies
    
    def detect_game_state_anomalies(self, game_data):
        """
        Detect anomalies in game state data.
        
        Args:
            game_data: Current game state data (dict or Series)
            
        Returns:
            list: Detected anomalies
        """
        # Convert to dict if Series
        if isinstance(game_data, pd.Series):
            game_data = game_data.to_dict()
        
        detected_anomalies = []
        
        # Extract game state data
        home_q1 = float(game_data.get('home_q1', 0) or 0)
        home_q2 = float(game_data.get('home_q2', 0) or 0)
        home_q3 = float(game_data.get('home_q3', 0) or 0)
        home_q4 = float(game_data.get('home_q4', 0) or 0)
        
        away_q1 = float(game_data.get('away_q1', 0) or 0)
        away_q2 = float(game_data.get('away_q2', 0) or 0)
        away_q3 = float(game_data.get('away_q3', 0) or 0)
        away_q4 = float(game_data.get('away_q4', 0) or 0)
        
        current_quarter = int(game_data.get('current_quarter', 0))
        game_status = str(game_data.get('game_status', '')).lower()
        
        # Check for inconsistent quarter progression
        if current_quarter > 0:
            # If we're in Q3 but Q1 or Q2 is zero, that's inconsistent
            if current_quarter >= 3 and (home_q1 == 0 or away_q1 == 0 or home_q2 == 0 or away_q2 == 0):
                anomaly = {
                    'type': 'inconsistent_quarter_data',
                    'game_id': game_data.get('game_id', 'unknown'),
                    'timestamp': datetime.now().isoformat(),
                    'details': {
                        'current_quarter': current_quarter,
                        'quarters': {
                            'home': [home_q1, home_q2, home_q3, home_q4],
                            'away': [away_q1, away_q2, away_q3, away_q4]
                        }
                    },
                    'severity': 'high'
                }
                detected_anomalies.append(anomaly)
        
        # Check for unusual quarter scores
        quarters_to_check = min(current_quarter, 4)
        for q in range(1, quarters_to_check + 1):
            home_score = float(game_data.get(f'home_q{q}', 0) or 0)
            away_score = float(game_data.get(f'away_q{q}', 0) or 0)
            
            # Check for unusually high quarter scores (over 50 points)
            if home_score > 50 or away_score > 50:
                anomaly = {
                    'type': 'unusual_quarter_score',
                    'game_id': game_data.get('game_id', 'unknown'),
                    'timestamp': datetime.now().isoformat(),
                    'details': {
                        'quarter': q,
                        'home_score': home_score,
                        'away_score': away_score,
                        'threshold': 50
                    },
                    'severity': 'medium'
                }
                detected_anomalies.append(anomaly)
            
            # Check for zero quarter scores (possible data error)
            if (current_quarter > q) and (home_score == 0 or away_score == 0):
                anomaly = {
                    'type': 'zero_quarter_score',
                    'game_id': game_data.get('game_id', 'unknown'),
                    'timestamp': datetime.now().isoformat(),
                    'details': {
                        'quarter': q,
                        'home_score': home_score,
                        'away_score': away_score
                    },
                    'severity': 'high'
                }
                detected_anomalies.append(anomaly)
        
        # Verify game status matches quarter
        if game_status == 'finished' and current_quarter < 4:
            anomaly = {
                'type': 'status_quarter_mismatch',
                'game_id': game_data.get('game_id', 'unknown'),
                'timestamp': datetime.now().isoformat(),
                'details': {
                    'status': game_status,
                    'current_quarter': current_quarter
                },
                'severity': 'high'
            }
            detected_anomalies.append(anomaly)
        
        # Store detected anomalies
        if detected_anomalies:
            self.anomalies.extend(detected_anomalies)
            self.save_anomalies()
        
        return detected_anomalies
    
    def analyze_anomaly_patterns(self):
        """
        Analyze anomaly patterns across games.
        
        Returns:
            dict: Analysis of anomaly patterns
        """
        if not self.anomalies:
            return {}
        
        # Convert to DataFrame for easier analysis
        anomalies_df = pd.DataFrame(self.anomalies)
        
        # Add datetime column
        anomalies_df['datetime'] = pd.to_datetime(anomalies_df['timestamp'], errors='coerce')
        
        # Count by type
        type_counts = anomalies_df['type'].value_counts().to_dict()
        
        # Count by severity
        severity_counts = anomalies_df['severity'].value_counts().to_dict()
        
        # Count by game
        game_counts = anomalies_df['game_id'].value_counts()
        games_with_anomalies = len(game_counts)
        most_anomalous_games = game_counts.nlargest(5).to_dict()
        
        # Time analysis - group by day
        if 'datetime' in anomalies_df.columns:
            anomalies_df['date'] = anomalies_df['datetime'].dt.date
            daily_counts = anomalies_df.groupby('date').size().to_dict()
            
            # Convert dates to strings for JSON serialization
            daily_counts = {str(date): count for date, count in daily_counts.items()}
        else:
            daily_counts = {}
        
        return {
            'total_anomalies': len(self.anomalies),
            'unique_games_affected': games_with_anomalies,
            'type_distribution': type_counts,
            'severity_distribution': severity_counts,
            'most_anomalous_games': most_anomalous_games,
            'daily_distribution': daily_counts
        }
    
    def visualize_anomalies(self):
        """
        Create visualizations of anomaly patterns.
        
        Returns:
            matplotlib Figure with anomaly visualizations
        """
        if not self.anomalies:
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.text(0.5, 0.5, "No anomalies detected", 
                   ha='center', va='center', fontsize=14)
            return fig
        
        # Convert to DataFrame for visualization
        anomalies_df = pd.DataFrame(self.anomalies)
        
        # Add datetime column
        anomalies_df['datetime'] = pd.to_datetime(anomalies_df['timestamp'], errors='coerce')
        
        # Create figure
        fig, axs = plt.subplots(2, 2, figsize=(16, 12))
        
        # 1. Anomalies by type
        type_counts = anomalies_df['type'].value_counts()
        
        ax1 = axs[0, 0]
        type_counts.plot.bar(ax=ax1, color='skyblue')
        ax1.set_title('Anomalies by Type')
        ax1.set_ylabel('Count')
        plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')
        
        # Add count labels
        for i, count in enumerate(type_counts):
            ax1.text(i, count + 0.1, str(count), ha='center')
        
        # 2. Anomalies by severity
        severity_counts = anomalies_df['severity'].value_counts()
        
        ax2 = axs[0, 1]
        bars = ax2.bar(severity_counts.index, severity_counts, 
                      color=['green', 'orange', 'red'])
        ax2.set_title('Anomalies by Severity')
        ax2.set_ylabel('Count')
        
        # Add count labels
        for bar in bars:
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                    str(int(height)), ha='center')
        
        # 3. Anomalies over time
        if 'datetime' in anomalies_df.columns:
            ax3 = axs[1, 0]
            
            # Group by date
            anomalies_df['date'] = anomalies_df['datetime'].dt.date
            daily_counts = anomalies_df.groupby('date').size()
            
            # Plot time series
            daily_counts.plot(ax=ax3, marker='o', linestyle='-', color='blue')
            ax3.set_title('Anomalies Over Time')
            ax3.set_ylabel('Count')
            ax3.set_xlabel('Date')
            ax3.grid(True, alpha=0.3)
        
        # 4. Top games with anomalies
        game_counts = anomalies_df['game_id'].value_counts().head(10)
        
        ax4 = axs[1, 1]
        game_counts.plot.bar(ax=ax4, color='salmon')
        ax4.set_title('Games with Most Anomalies')
        ax4.set_ylabel('Count')
        ax4.set_xlabel('Game ID')
        plt.setp(ax4.get_xticklabels(), rotation=45, ha='right')
        
        # Add count labels
        for i, count in enumerate(game_counts):
            ax4.text(i, count + 0.1, str(count), ha='center')
        
        plt.tight_layout()
        return fig

In [ ]:
# Cell 5: Updated Model with New Features from Supabase

import joblib
import pandas as pd
import numpy as np
import traceback
from sqlalchemy import create_engine

# Import configuration (ensure config.py is set up with DATABASE_URL and MODEL_PATH)
import config

MODEL_PATH = config.MODEL_PATH  # e.g., 'final_xgb_model.pkl'

# Attempt to load the model
try:
    model = joblib.load(MODEL_PATH)
    print(f"Model loaded from: {MODEL_PATH}")
except Exception as e:
    print(f"Error loading model: {e}")
    model = None

# Define the expected features for original and enhanced sets
original_features = [
    'home_q1', 'home_q2', 'home_q3', 'home_q4',
    'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
]
enhanced_features = [
    'home_q1', 'home_q2', 'home_q3', 'home_q4',
    'score_ratio', 'prev_matchup_diff',
    'rest_days_home', 'rest_days_away', 'rest_advantage',
    'is_back_to_back_home', 'is_back_to_back_away',
    'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
]

# Choose the expected feature set based on the model's capabilities
if model is not None and hasattr(model, 'feature_importances_') and len(model.feature_importances_) > 8:
    expected_features = enhanced_features
    print("Using enhanced feature set for model")
else:
    expected_features = original_features
    print("Using original feature set for model")

# Connect to the Supabase database via SQLAlchemy
engine = create_engine(config.DATABASE_URL)
try:
    new_features_df = pd.read_sql("SELECT * FROM nba_enhanced_features", engine)
    print(f"Loaded {len(new_features_df)} rows from nba_enhanced_features table.")
except Exception as e:
    print(f"Error loading features from Supabase: {e}")
    traceback.print_exc()
    new_features_df = pd.DataFrame()

# Ensure all expected features exist in the DataFrame; add missing ones with default 0
for feature in expected_features:
    if feature not in new_features_df.columns:
        print(f"Warning: feature '{feature}' not found in new_features_df; adding with default 0.")
        new_features_df[feature] = 0

# Convert all expected feature columns to numeric values
for feature in expected_features:
    new_features_df[feature] = pd.to_numeric(new_features_df[feature], errors='coerce').fillna(0)

# Prepare the feature matrix for prediction
X_features = new_features_df[expected_features].copy()

# Generate predictions if a model is available
if model is not None:
    try:
        predictions = model.predict(X_features)
        new_features_df['predicted_home_score'] = predictions
        print("Predictions generated successfully.")
        display(new_features_df[['predicted_home_score']].head())
    except Exception as e:
        print(f"Error during prediction: {e}")
        traceback.print_exc()
else:
    print("No model available for predictions. Please train or load a model first.")


In [ ]:
# Cell 5A: Ensemble Weight Visualization & A/B Testing

# Import the EnsembleWeightVisualizer class from your features module
from models.features import EnsembleWeightVisualizer

# Create weight visualizer with error history
error_history = {
    'main_model': {1: 7.0, 2: 6.2, 3: 5.5, 4: 4.7},
    'quarter_model': {1: 8.5, 2: 7.0, 3: 5.8, 4: 4.5}
}
weight_viz = EnsembleWeightVisualizer(error_history)

# Visualize all weighting strategies
print("Comparing Different Ensemble Weighting Strategies")
fig = weight_viz.visualize_all_strategies()
plt.show()

# Compare strategies for a specific game context
print("\nComparing Weights for Quarter 2 Game (Close Score, High Momentum)")
strategies_df = weight_viz.compare_strategies(
    current_quarter=2, 
    score_differential=3,  # Close game
    momentum=0.7,          # High momentum
    main_uncertainty=6.0,  # Main model uncertainty
    quarter_uncertainty=7.0 # Quarter model uncertainty
)
display(strategies_df)

# Analyze how these weights would change predictions
def analyze_prediction_impact(main_pred, quarter_pred, strategies_df):
    """Analyze how different weighting strategies affect final predictions"""
    impact_df = strategies_df.copy()
    impact_df['main_prediction'] = main_pred
    impact_df['quarter_prediction'] = quarter_pred
    impact_df['ensemble_pred'] = impact_df['main_weight'] * main_pred + impact_df['quarter_weight'] * quarter_pred
    impact_df['difference_from_main'] = impact_df['ensemble_pred'] - main_pred
    return impact_df

# Test with sample predictions
main_prediction = 105.5
quarter_prediction = 112.0
print(f"\nMain Model Prediction: {main_prediction:.1f}")
print(f"Quarter Model Prediction: {quarter_prediction:.1f}")
impact_df = analyze_prediction_impact(main_prediction, quarter_prediction, strategies_df)
display(impact_df[['strategy', 'main_weight', 'quarter_weight', 'ensemble_pred', 'difference_from_main']])

# Visualize prediction impact
plt.figure(figsize=(10, 6))
plt.bar(impact_df['strategy'], impact_df['ensemble_pred'], color='cornflowerblue')
plt.axhline(y=main_prediction, color='r', linestyle='-', label=f'Main Model: {main_prediction:.1f}')
plt.axhline(y=quarter_prediction, color='g', linestyle='--', label=f'Quarter Model: {quarter_prediction:.1f}')
plt.xlabel('Weighting Strategy')
plt.ylabel('Ensemble Prediction')
plt.title('Impact of Different Weighting Strategies on Final Prediction')
plt.xticks(rotation=45)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Implement a simple A/B test function for strategies
def ab_test_strategies(test_games_df, strategies=['standard', 'adaptive'], error_metric='abs_error'):
    """
    Run A/B test comparing different weighting strategies on historical games
    
    Args:
        test_games_df: DataFrame with test games (needs actual final scores)
        strategies: List of strategy names to compare
        error_metric: Error metric to compute ('abs_error' or 'squared_error')
    
    Returns:
        DataFrame with error metrics by strategy
    """
    from models.features import dynamic_ensemble_predictions, NBAFeatureGenerator
    
    results = []
    feature_generator = NBAFeatureGenerator(debug=False)
    
    for _, game in test_games_df.iterrows():
        actual_score = game.get('actual_home_final', game.get('home_score', 0))
        current_quarter = int(game.get('current_quarter', 0))
        
        if current_quarter <= 0:
            continue
        
        # Process game features
        game_df = pd.DataFrame([game])
        features_df = feature_generator.generate_all_features(game_df)
        
        if features_df.empty:
            continue
            
        game_data = features_df.iloc[0]
        
        # Get context data
        score_diff = float(game_data.get('score_differential', 0))
        momentum = float(game_data.get('cumulative_momentum', 0))
        main_pred = float(game.get('main_prediction', 105))
        quarter_pred = float(game.get('quarter_model_prediction', 105))
        
        # Compare strategies
        for strategy in strategies:
            pred, conf, main_w, quarter_w = dynamic_ensemble_predictions(
                main_pred, quarter_pred, current_quarter,
                score_differential=score_diff,
                momentum=momentum,
                weighting_strategy=strategy
            )
            
            # Calculate error
            error = pred - actual_score
            abs_error = abs(error)
            squared_error = error ** 2
            
            results.append({
                'game_id': game.get('game_id', ''),
                'actual_score': actual_score,
                'quarter': current_quarter,
                'strategy': strategy,
                'main_weight': main_w,
                'quarter_weight': quarter_w,
                'prediction': pred,
                'error': error,
                'abs_error': abs_error,
                'squared_error': squared_error
            })
    
    if not results:
        return pd.DataFrame()
    
    results_df = pd.DataFrame(results)
    
    # Calculate average metrics by strategy and quarter
    summary = results_df.groupby(['strategy', 'quarter']).agg({
        'abs_error': ['mean', 'std', 'count'],
        'squared_error': ['mean', 'std']
    }).reset_index()
    
    # Rename columns for better readability
    summary.columns = ['strategy', 'quarter', 'mae', 'mae_std', 'count', 'mse', 'mse_std']
    summary['rmse'] = np.sqrt(summary['mse'])
    
    return summary

# Test A/B testing with sample historical games
# These would normally come from the validation dataset
if 'validation_df' in globals() and not globals()['validation_df'].empty:
    print("\nRunning A/B Test of Weighting Strategies")
    ab_results = ab_test_strategies(
        globals()['validation_df'], 
        strategies=['standard', 'adaptive', 'conservative', 'aggressive'],
        error_metric='abs_error'
    )
    
    if not ab_results.empty:
        print("\nA/B Test Results (MAE by Quarter and Strategy):")
        display(ab_results[['strategy', 'quarter', 'mae', 'rmse', 'count']])
        
        # Visualize results
        plt.figure(figsize=(12, 6))
        for strategy in ab_results['strategy'].unique():
            strategy_data = ab_results[ab_results['strategy'] == strategy]
            plt.plot(strategy_data['quarter'], strategy_data['mae'], 'o-', 
                   label=strategy.capitalize(), linewidth=2, markersize=8)
        
        plt.xlabel('Quarter')
        plt.ylabel('Mean Absolute Error')
        plt.title('Strategy Performance by Quarter')
        plt.grid(alpha=0.3)
        plt.legend()
        plt.xticks([1, 2, 3, 4])
        plt.ylim(bottom=0)
        plt.show()

In [ ]:
# Cell 6: Optimized Data Preprocessing with Diagnostics
from models.score_prediction import load_training_data, preprocess_data
import pandas as pd
import numpy as np
import time
import os

# Configuration options
MAX_SAMPLE_SIZE = 5000  # Limit analysis to this many rows for statistics
CACHE_FILE = "preprocessed_data_cache.npz"
USE_CACHE = os.path.exists(CACHE_FILE)

# Start timing
start_time = time.time()

# Load historical training data
print("Loading data...")
df = load_training_data()
print(f"Historical data loaded. Shape: {df.shape}")
print(f"Date range: {df['game_date'].min()} to {df['game_date'].max()}")
print(f"Loading time: {time.time() - start_time:.2f} seconds")

# Check for nulls in important columns - fast operation
null_counts = df[['home_team', 'away_team', 'home_score', 'away_score']].isnull().sum()
print("\nNull counts in key columns:")
print(null_counts)

# Examine data before preprocessing (just a few rows)
print("\nSample of raw data:")
display(df.head(3))  # Show only 3 rows

try:
    # Load from cache if available
    if USE_CACHE:
        print(f"\nLoading preprocessed data from cache...")
        cache_data = np.load(CACHE_FILE, allow_pickle=True)
        X = pd.DataFrame(cache_data['X'], columns=cache_data['columns'])
        y = cache_data['y']
        print(f"Loaded from cache in {time.time() - start_time:.2f} seconds")
    else:
        # Process the data with timing
        print("\nPreprocessing data...")
        preproc_start = time.time()
        X, y = preprocess_data(df)
        print(f"Preprocessing time: {time.time() - preproc_start:.2f} seconds")
        
        # Save to cache for future runs
        try:
            print("Saving to cache for future runs...")
            np.savez(CACHE_FILE, X=X.values, columns=X.columns, y=y)
        except Exception as cache_err:
            print(f"Warning: Could not save cache: {str(cache_err)}")
    
    # Check shapes and types
    print(f"\nFeatures shape: {X.shape}")
    print(f"Target shape: {y.shape}")
    
    # Use a sample for statistics to speed up processing
    if len(X) > MAX_SAMPLE_SIZE:
        print(f"\nUsing a sample of {MAX_SAMPLE_SIZE} rows for statistics (out of {len(X)} total)")
        X_sample = X.sample(MAX_SAMPLE_SIZE, random_state=42)
    else:
        X_sample = X
    
    # Calculate basic statistics (much faster than full stats)
    stats_start = time.time()
    
    # Select only numeric columns for faster processing
    numeric_cols = X_sample.select_dtypes(include=['number']).columns
    
    # Calculate basic statistics efficiently
    feature_stats = pd.DataFrame({
        'min': X_sample[numeric_cols].min(),
        'max': X_sample[numeric_cols].max(),
        'mean': X_sample[numeric_cols].mean(),
        'null_count': X_sample[numeric_cols].isnull().sum()
    })
    
    # Limit zero count calculation to just a few important columns
    important_cols = numeric_cols[:10].tolist()  # First 10 numeric columns
    
    # Add prev_matchup_diff if it exists
    if 'prev_matchup_diff' in X_sample.columns:
        if 'prev_matchup_diff' not in important_cols:
            important_cols.append('prev_matchup_diff')
    
    # Calculate zero counts only for important columns
    zero_counts = (X_sample[important_cols] == 0).sum()
    zero_percents = zero_counts / len(X_sample) * 100
    
    for col in important_cols:
        if col in feature_stats.index:
            feature_stats.loc[col, 'zero_count'] = zero_counts[col]
            feature_stats.loc[col, 'zero_percent'] = zero_percents[col]
    
    print(f"Statistics calculation time: {time.time() - stats_start:.2f} seconds")
    
    # Display summarized statistics
    print("\nFeature statistics (top 10 features):")
    display(feature_stats.head(10))
    if len(feature_stats) > 10:
        print(f"... plus {len(feature_stats) - 10} more features")
    
    # Special focus on prev_matchup_diff if it exists
    if 'prev_matchup_diff' in X.columns:
        print(f"\nprev_matchup_diff analysis:")
        nonzero = (X_sample['prev_matchup_diff'] != 0).sum()
        print(f"Non-zero values: {nonzero} ({nonzero / len(X_sample) * 100:.2f}% of sample)")
        print(f"First 5 values: {X['prev_matchup_diff'].head(5).tolist()}")
    
    # Display just a small sample of processed features
    print("\nProcessed features sample (first 5 columns, 3 rows):")
    display(X.iloc[:3, :5])
    print(f"... plus {X.shape[1] - 5} more columns")
    
    # Total processing time
    print(f"\nTotal processing time: {time.time() - start_time:.2f} seconds")
    
except Exception as e:
    print(f"Error in preprocessing: {str(e)}")
    import traceback
    traceback.print_exc()

def ensure_features_exist(df, required_features, fallback_values=None):
    """
    Ensure all required features exist in the DataFrame, adding any missing ones.
    
    Args:
        df: DataFrame to check
        required_features: List of feature names that must exist
        fallback_values: Optional dict of default values for specific features
        
    Returns:
        DataFrame with all required features
    """
    result_df = df.copy()
    
    if fallback_values is None:
        fallback_values = {}
    
    # Default fallback is 0 for any unlisted features
    for feature in required_features:
        if feature not in result_df.columns:
            print(f"Warning: Required feature '{feature}' missing. Adding with fallback value.")
            result_df[feature] = fallback_values.get(feature, 0)
    
    return result_df

In [ ]:
# Cell 6B - NBA Score Prediction Engine - Core Implementation

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import traceback
import sys
import os
import pytz
from datetime import datetime, timedelta
from sqlalchemy import create_engine
from typing import Dict, List, Tuple, Optional, Union, Any

# ==================== CONFIGURATION =====================

CONFIG = {
    "debug": False,
    "visualize_limit": 6,
    "lookback_days": 30,
    "max_rest_days": 10,
    "log_level": "INFO",
    "momentum_cap": 25,
    "momentum_normalize_factor": 15.0,
    "league_avg_score": 220,
    "team_abbreviations": {},
}

FEATURE_SETS = {
    "original": [
        'home_q1', 'home_q2', 'home_q3', 'home_q4',
        'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
    ],
    "enhanced": [
        'home_q1', 'home_q2', 'home_q3', 'home_q4',
        'score_ratio', 'prev_matchup_diff',
        'rest_days_home', 'rest_days_away', 'rest_advantage',
        'is_back_to_back_home', 'is_back_to_back_away',
        'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
    ]
}

DEFAULT_FEATURE_VALUES = {
    'home_q1': 0, 'home_q2': 0, 'home_q3': 0, 'home_q4': 0,
    'score_ratio': 0.5, 'rolling_home_score': 105.0, 'rolling_away_score': 105.0,
    'prev_matchup_diff': 0, 'rest_days_home': 2, 'rest_days_away': 2,
    'rest_advantage': 0, 'is_back_to_back_home': 0, 'is_back_to_back_away': 0,
    'q1_to_q2_momentum': 0, 'q2_to_q3_momentum': 0, 'q3_to_q4_momentum': 0,
    'cumulative_momentum': 0
}

MODEL_PATHS = [
    'models/score_prediction_model.pkl',
    'final_xgb_model.pkl',
    'enhanced_xgb_model.pkl',
    'backend/models/score_prediction_model.pkl',
]

backend_dir = os.path.join(os.getcwd(), "backend")
if os.path.exists(backend_dir) and backend_dir not in sys.path:
    sys.path.append(backend_dir)

try:
    from caching.supabase_client import supabase
    import config
    if hasattr(config, 'MODEL_PATH') and config.MODEL_PATH:
        MODEL_PATHS.insert(0, config.MODEL_PATH)
except ImportError as e:
    class MockSupabase:
        def table(self, name):
            return self
        def select(self, cols=None):
            return self
        def eq(self, col, val):
            return self
        def gte(self, col, val):
            return self
        def lte(self, col, val):
            return self
        def limit(self, n):
            return self
        def order(self, col, desc=False):
            return self
        def execute(self):
            return type('obj', (object,), {'data': []})
    supabase = MockSupabase()
    config = type('obj', (object,), {'MODEL_PATH': 'models/score_prediction_model.pkl'})
    if CONFIG["debug"]:
        print(f"Using mock objects due to import error: {e}")

# ==================== LOGGING UTILITIES =====================

def log_message(message: str, level: str = "INFO") -> None:
    if level == "DEBUG" and not CONFIG["debug"]:
        return
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts}] {level}: {message}")

def handle_exception(e: Exception, context: str = "") -> None:
    log_message(f"Error in {context}: {str(e)}", "ERROR")
    if CONFIG["debug"]:
        traceback.print_exc()

# ==================== DATA PROCESSING FUNCTIONS =====================

def ensure_numeric_features(df: pd.DataFrame, feature_cols: List[str]) -> pd.DataFrame:
    result = df.copy()
    for col in feature_cols:
        if col not in result.columns:
            result[col] = DEFAULT_FEATURE_VALUES.get(col, 0)
        result[col] = pd.to_numeric(result[col], errors='coerce').fillna(DEFAULT_FEATURE_VALUES.get(col, 0))
    return result

def calculate_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    result_df = df.copy()
    result_df = calculate_current_scores(result_df)
    result_df = calculate_score_ratio(result_df)
    result_df['score_differential'] = result_df['home_score'] - result_df['away_score']
    result_df = calculate_momentum_features(result_df)
    result_df['time_remaining_mins'] = result_df['current_quarter'].apply(
        lambda q: max(0, 48 - ((int(q) - 1) * 12 + 6))
    )
    result_df['time_remaining_norm'] = result_df['time_remaining_mins'] / 48.0
    return result_df

def calculate_current_scores(df: pd.DataFrame) -> pd.DataFrame:
    result_df = df.copy()
    quarters = ['home_q1', 'home_q2', 'home_q3', 'home_q4']
    for q in quarters:
        if q in result_df.columns:
            result_df[q] = pd.to_numeric(result_df[q], errors='coerce').fillna(0)
    mask = (result_df['home_score'].isna()) | (result_df['home_score'] == 0)
    if mask.any():
        existing_quarters = [q for q in quarters if q in result_df.columns]
        if existing_quarters:
            result_df.loc[mask, 'home_score'] = result_df.loc[mask, existing_quarters].sum(axis=1)
    away_quarters = ['away_q1', 'away_q2', 'away_q3', 'away_q4']
    mask = (result_df['away_score'].isna()) | (result_df['away_score'] == 0)
    if mask.any():
        existing_quarters = [q for q in away_quarters if q in result_df.columns]
        if existing_quarters:
            result_df.loc[mask, 'away_score'] = result_df.loc[mask, existing_quarters].sum(axis=1)
    return result_df

def calculate_score_ratio(df: pd.DataFrame) -> pd.DataFrame:
    result_df = df.copy()
    home_scores = pd.to_numeric(result_df['home_score'], errors='coerce').fillna(0)
    away_scores = pd.to_numeric(result_df['away_score'], errors='coerce').fillna(0)
    total_scores = home_scores + away_scores
    result_df['score_ratio'] = 0.5
    mask = (total_scores > 0)
    if mask.any():
        result_df.loc[mask, 'score_ratio'] = home_scores[mask] / total_scores[mask]
    return result_df

def calculate_momentum_features(df: pd.DataFrame) -> pd.DataFrame:
    result_df = df.copy()
    momentum_cap = CONFIG["momentum_cap"]
    norm_factor = CONFIG["momentum_normalize_factor"]
    for col in ['q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum']:
        result_df[col] = 0.0
    for team in ['home', 'away']:
        for q in range(1, 5):
            col = f'{team}_q{q}'
            if col in result_df.columns:
                result_df[col] = pd.to_numeric(result_df[col], errors='coerce').fillna(0)
    current_quarters = pd.to_numeric(result_df['current_quarter'], errors='coerce').fillna(0).astype(int)
    mask = (current_quarters >= 2)
    if mask.any():
        result_df.loc[mask, 'q1_to_q2_momentum'] = (
            (result_df.loc[mask, 'home_q2'] - result_df.loc[mask, 'home_q1']) - 
            (result_df.loc[mask, 'away_q2'] - result_df.loc[mask, 'away_q1'])
        ).clip(-momentum_cap, momentum_cap)
    mask = (current_quarters >= 3)
    if mask.any():
        result_df.loc[mask, 'q2_to_q3_momentum'] = (
            (result_df.loc[mask, 'home_q3'] - result_df.loc[mask, 'home_q2']) - 
            (result_df.loc[mask, 'away_q3'] - result_df.loc[mask, 'away_q2'])
        ).clip(-momentum_cap, momentum_cap)
    mask = (current_quarters >= 4)
    if mask.any():
        result_df.loc[mask, 'q3_to_q4_momentum'] = (
            (result_df.loc[mask, 'home_q4'] - result_df.loc[mask, 'home_q3']) - 
            (result_df.loc[mask, 'away_q4'] - result_df.loc[mask, 'away_q3'])
        ).clip(-momentum_cap, momentum_cap)
    weights = [0.2, 0.3, 0.5]
    mask = (current_quarters == 2)
    if mask.any():
        result_df.loc[mask, 'cumulative_momentum'] = (
            result_df.loc[mask, 'q1_to_q2_momentum'] / norm_factor
        )
    mask = (current_quarters == 3)
    if mask.any():
        q1_to_q2 = result_df.loc[mask, 'q1_to_q2_momentum']
        q2_to_q3 = result_df.loc[mask, 'q2_to_q3_momentum']
        weighted_momentum = (q1_to_q2 * weights[0] + q2_to_q3 * weights[1]) / (weights[0] + weights[1])
        result_df.loc[mask, 'cumulative_momentum'] = weighted_momentum / norm_factor
    mask = (current_quarters >= 4)
    if mask.any():
        q1_to_q2 = result_df.loc[mask, 'q1_to_q2_momentum']
        q2_to_q3 = result_df.loc[mask, 'q2_to_q3_momentum']
        q3_to_q4 = result_df.loc[mask, 'q3_to_q4_momentum']
        weighted_momentum = (q1_to_q2 * weights[0] + q2_to_q3 * weights[1] + q3_to_q4 * weights[2]) / sum(weights)
        result_df.loc[mask, 'cumulative_momentum'] = weighted_momentum / norm_factor
    result_df['cumulative_momentum'] = result_df['cumulative_momentum'].clip(-1.0, 1.0)
    return result_df

def calculate_improved_rest_features(df: pd.DataFrame, max_lookback_days: int = 30, max_rest_days: int = 10) -> pd.DataFrame:
    if df.empty:
        return df
    result_df = df.copy()
    if 'game_date' not in result_df.columns:
        result_df['rest_days_home'] = 2.0
        result_df['rest_days_away'] = 2.0
        result_df['is_back_to_back_home'] = 0
        result_df['is_back_to_back_away'] = 0
        result_df['rest_advantage'] = 0
        return result_df
    result_df['game_date'] = pd.to_datetime(result_df['game_date'], errors='coerce')
    pacific_tz = pytz.timezone("America/Los_Angeles")
    if result_df['game_date'].dt.tz is None:
        result_df['game_date'] = result_df['game_date'].dt.tz_localize(pacific_tz)
    else:
        result_df['game_date'] = result_df['game_date'].dt.tz_convert(pacific_tz)
    result_df = result_df.sort_values('game_date')
    result_df['rest_days_home'] = 2.0
    result_df['rest_days_away'] = 2.0
    result_df['is_back_to_back_home'] = 0
    result_df['is_back_to_back_away'] = 0
    home_games = result_df[['game_id', 'game_date', 'home_team']].rename(columns={'home_team': 'team'})
    home_games['is_home'] = True
    away_games = result_df[['game_id', 'game_date', 'away_team']].rename(columns={'away_team': 'team'})
    away_games['is_home'] = False
    all_games = pd.concat([home_games, away_games])
    teams = set(result_df['home_team'].dropna().unique()).union(set(result_df['away_team'].dropna().unique()))
    for team in teams:
        team_games = all_games[all_games['team'] == team].sort_values('game_date')
        if team_games.empty:
            continue
        team_games['prev_date'] = team_games['game_date'].shift(1)
        team_games['days_since_prev'] = (team_games['game_date'] - team_games['prev_date']).dt.days.fillna(3)
        team_games['days_since_prev'] = team_games['days_since_prev'].clip(1, max_rest_days)
        team_games['is_back_to_back'] = (team_games['days_since_prev'] == 1).astype(int)
        for _, row in team_games.iterrows():
            game_id = row['game_id']
            is_home = row['is_home']
            if is_home:
                idx = result_df.index[(result_df['game_id'] == game_id) & (result_df['home_team'] == team)]
                if not idx.empty:
                    result_df.loc[idx, 'rest_days_home'] = row['days_since_prev']
                    result_df.loc[idx, 'is_back_to_back_home'] = row['is_back_to_back']
            else:
                idx = result_df.index[(result_df['game_id'] == game_id) & (result_df['away_team'] == team)]
                if not idx.empty:
                    result_df.loc[idx, 'rest_days_away'] = row['days_since_prev']
                    result_df.loc[idx, 'is_back_to_back_away'] = row['is_back_to_back']
    result_df['rest_advantage'] = (result_df['rest_days_home'] - result_df['rest_days_away']).clip(-5, 5)
    return result_df

def generate_simulated_prediction_history(game: Dict) -> List[Dict]:
    """
    Generate simulated prediction history for visualization when real history is unavailable.
    This improved version ensures predictions are realistic basketball scores.
    """
    import random
    current_quarter = int(game.get('current_quarter', 0))
    current_home_score = float(game.get('home_score', 0))
    current_away_score = float(game.get('away_score', 0))
    
    # Handle the case where predicted scores might be missing or zero
    final_home_prediction = float(game.get('predicted_home_score', 0))
    final_away_prediction = float(game.get('predicted_away_score', 0))
    
    # Use fallback values if predictions are missing or unrealistic
    if final_home_prediction < 80 or final_away_prediction < 80:
        # If we have current scores, use them to project realistic final scores
        if current_quarter > 0 and (current_home_score > 0 or current_away_score > 0):
            quarters_left = 4 - current_quarter
            if quarters_left > 0:
                # Project based on current scoring rate
                points_per_quarter_home = current_home_score / current_quarter
                points_per_quarter_away = current_away_score / current_quarter
                final_home_prediction = current_home_score + (points_per_quarter_home * quarters_left)
                final_away_prediction = current_away_score + (points_per_quarter_away * quarters_left)
            else:
                # Game is complete, use current scores
                final_home_prediction = current_home_score
                final_away_prediction = current_away_score
        else:
            # Fallback to reasonable NBA scores if we can't project from current
            final_home_prediction = 110 
            final_away_prediction = 105
    
    win_probability = float(game.get('win_probability', 0.5))
    
    # If no quarters have been played yet, just return the current prediction
    if current_quarter < 1:
        return [{
            'timestamp': datetime.now(),
            'current_quarter': current_quarter,
            'home_score': current_home_score,
            'away_score': current_away_score,
            'predicted_home_score': final_home_prediction,
            'predicted_away_score': final_away_prediction,
            'win_probability': win_probability
        }]
    
    # Create prediction history
    history = []
    now = datetime.now()
    # Generate timestamps moving backward from now
    timestamps = [now - timedelta(minutes=15*i) for i in range(current_quarter+1)]
    timestamps.reverse()  # Now timestamps go from oldest to newest
    
    # For each quarter that has been played, generate a realistic prediction
    for q in range(1, current_quarter+1):
        # Calculate what portion of the current score would have been achieved by this quarter
        q_ratio = q / 4.0  # Assume linear scoring throughout game
        
        # For quarters that have already completed, use actual partial scores if available
        # Otherwise simulate what scores might have been at this point
        simulated_home_score = current_home_score * q_ratio
        simulated_away_score = current_away_score * q_ratio
        
        # Make predictions more uncertain in early quarters, more certain in later quarters
        # but always keep them as realistic basketball scores
        pred_uncertainty = max(0.2, 1.0 - (q / 4.0))  # 0.8 for Q1, 0.2 for Q4
        
        # Calculate what percentage of the final prediction would have been made by this quarter
        # Early predictions are less accurate (further from final)
        prediction_ratio = 0.7 + (0.3 * q / 4.0)  # 0.775 for Q1, 1.0 for Q4
        
        # Scale random factor based on the size of the prediction to maintain realistic scores
        base_factor = min(12, final_home_prediction * 0.1) * pred_uncertainty
        random_factor = random.uniform(-base_factor, base_factor)
        
        # Generate predictions that converge toward the final prediction as quarters progress
        simulated_home_prediction = max(75, final_home_prediction * prediction_ratio + random_factor)
        
        # Make sure away prediction has appropriate spread from home prediction
        final_spread = final_home_prediction - final_away_prediction
        current_spread = simulated_home_prediction - final_away_prediction * prediction_ratio
        spread_adjustment = (final_spread - current_spread) * (q / 4.0)
        simulated_away_prediction = max(70, simulated_home_prediction - final_spread * prediction_ratio - random_factor * 0.5)
        
        # Calculate win probability that trends toward the final value
        # Start closer to 50% and move toward final win probability as game progresses
        wp_shift = win_probability - 0.5  # How far from even odds
        initial_wp = 0.5 + (wp_shift * (q / 4.0))  # Start closer to 50/50, move toward final
        wp_random = random.uniform(-0.15, 0.15) * (1 - q/4.0)  # More randomness in early quarters
        simulated_wp = max(min(initial_wp + wp_random, 0.99), 0.01)
        
        history.append({
            'timestamp': timestamps[q-1],
            'current_quarter': q,
            'home_score': simulated_home_score,
            'away_score': simulated_away_score,
            'predicted_home_score': simulated_home_prediction,
            'predicted_away_score': simulated_away_prediction,
            'win_probability': simulated_wp
        })
    
    # Add the current prediction as the last point
    history.append({
        'timestamp': timestamps[-1],
        'current_quarter': current_quarter,
        'home_score': current_home_score,
        'away_score': current_away_score,
        'predicted_home_score': final_home_prediction,
        'predicted_away_score': final_away_prediction,
        'win_probability': win_probability
    })
    
    return history

def calculate_adaptive_weights(current_quarter: int, score_differential: float, momentum: float,
                               main_model_error: Optional[float] = None, quarter_model_error: Optional[float] = None) -> Tuple[float, float]:
    base_quarter_weights = {0: 0.30, 1: 0.40, 2: 0.60, 3: 0.75, 4: 0.90}
    base_main_weight = base_quarter_weights.get(current_quarter, 0.5)
    adjustments = []
    if score_differential < 5:
        adjustments.append(-0.10)
    elif score_differential < 10:
        adjustments.append(-0.05)
    elif score_differential > 20:
        adjustments.append(0.10)
    elif score_differential > 15:
        adjustments.append(0.05)
    momentum_magnitude = abs(momentum)
    momentum_adjustment = -0.15 * momentum_magnitude
    adjustments.append(momentum_adjustment)
    if main_model_error is not None and quarter_model_error is not None:
        if main_model_error > 0 and quarter_model_error > 0:
            error_ratio = main_model_error / quarter_model_error
            error_adjustment = min(max(-0.15, (1.0 - error_ratio) * 0.2), 0.15)
            adjustments.append(error_adjustment)
    final_main_weight = base_main_weight + sum(adjustments)
    final_main_weight = max(0.1, min(0.95, final_main_weight))
    final_quarter_weight = 1.0 - final_main_weight
    return final_main_weight, final_quarter_weight

# ==================== NBA GAME PREDICTOR CLASS =====================

class NBAGamePredictor:
    """
    Predict NBA game outcomes and generate visualizations and recommendations.
    """
    
    def __init__(self, config: Dict = None):
        self.model = None
        self.feature_list = None
        self.prediction_history = {}
        if config:
            for key, value in config.items():
                CONFIG[key] = value
        log_message("Initializing NBA Game Predictor")
        self.load_model()
    
    def load_model(self) -> None:
        import joblib
        model_paths = MODEL_PATHS.copy()
        model_paths.extend([os.path.join(os.getcwd(), path) for path in MODEL_PATHS])
        for path in model_paths:
            try:
                if os.path.exists(path):
                    log_message(f"Attempting to load model from {path}")
                    model = joblib.load(path)
                    if hasattr(model, 'predict'):
                        log_message(f"Successfully loaded model from {path}")
                        self.model = model
                        if hasattr(model, 'feature_importances_'):
                            fc = len(model.feature_importances_)
                            is_enhanced = fc > 8
                            model_type = "enhanced" if is_enhanced else "original"
                            log_message(f"Model type: {model_type} ({fc} features)")
                            self.feature_list = FEATURE_SETS["enhanced" if is_enhanced else "original"]
                        else:
                            log_message("Model lacks feature_importances_. Using enhanced features.")
                            self.feature_list = FEATURE_SETS["enhanced"]
                        return
                    else:
                        log_message(f"Loaded object from {path} lacks a predict() method.")
                else:
                    log_message(f"Model file not found at {path}", "DEBUG")
            except Exception as e:
                log_message(f"Failed to load model from {path}: {str(e)}", "WARNING")
        log_message("No model could be loaded. Creating fallback model...", "WARNING")
        self.create_fallback_model()
    
    def create_fallback_model(self) -> None:
        from sklearn.ensemble import GradientBoostingRegressor
        log_message("Creating fallback prediction model")
        model = GradientBoostingRegressor(n_estimators=50, random_state=42)
        np.random.seed(42)
        X = pd.DataFrame({
            'home_q1': np.random.randint(20, 30, 1000),
            'home_q2': np.random.randint(20, 30, 1000),
            'home_q3': np.random.randint(20, 30, 1000),
            'home_q4': np.random.randint(20, 30, 1000),
            'score_ratio': np.random.uniform(0.4, 0.6, 1000),
            'rolling_home_score': np.random.normal(106, 5, 1000),
            'rolling_away_score': np.random.normal(103, 5, 1000),
            'prev_matchup_diff': np.random.normal(3, 8, 1000)
        })
        y = X['home_q1'] + X['home_q2'] + X['home_q3'] + X['home_q4'] + np.random.normal(0, 5, 1000)
        model.fit(X, y)
        log_message("Fallback model created and trained")
        self.model = model
        self.feature_list = FEATURE_SETS["original"]
    
    def get_feature_list(self) -> List[str]:
        if self.feature_list is not None:
            return self.feature_list
        is_enhanced = False
        if self.model and hasattr(self.model, 'feature_importances_'):
            is_enhanced = len(self.model.feature_importances_) > 8
        self.feature_list = FEATURE_SETS["enhanced" if is_enhanced else "original"]
        log_message(f"Using {'enhanced' if is_enhanced else 'original'} feature set")
        return self.feature_list
    
    def fetch_live_games_pacific(self) -> pd.DataFrame:
        try:
            log_message("Fetching all games for today...")
            pacific_tz = pytz.timezone("America/Los_Angeles")
            now_pt = datetime.now(pacific_tz)
            today_pt = now_pt.date()
            start_pt = datetime.combine(today_pt, datetime.min.time()).replace(tzinfo=pacific_tz)
            end_pt = datetime.combine(today_pt, datetime.max.time()).replace(tzinfo=pacific_tz)
            start_iso = start_pt.strftime('%Y-%m-%dT%H:%M:%S')
            end_iso = end_pt.strftime('%Y-%m-%dT%H:%M:%S')
            log_message(f"Fetching games from Supabase in range: {start_iso} to {end_iso}")
            response = supabase.table("nba_live_game_stats").select("*").gte("game_date", start_iso).lte("game_date", end_iso).execute()
            if not response.data:
                log_message(f"No games found for today ({today_pt}).", "WARNING")
                return pd.DataFrame()
            df = pd.DataFrame(response.data)
            log_message(f"Found {len(df)} rows for today in the database.")
            df['game_date'] = pd.to_datetime(df['game_date'], errors='coerce')
            if df['game_date'].dt.tz is None:
                df['game_date'] = df['game_date'].dt.tz_localize(pacific_tz)
            else:
                df['game_date'] = df['game_date'].dt.tz_convert(pacific_tz)
            df['game_date_pt'] = df['game_date']  # keep tz-aware in PT
            df['date_only_pt'] = df['game_date_pt'].dt.date
            log_message(f"Today's date in PT: {today_pt}")
            df_today = df[df['date_only_pt'] == today_pt].copy()
            log_message(f"Rows matching today's PT date: {len(df_today)}")
            df_today = self.process_game_statuses(df_today)
            return df_today
        except Exception as e:
            handle_exception(e, "fetch_live_games_pacific")
            return pd.DataFrame()
    
    def process_game_statuses(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Apply live game filtering logic based solely on the stored 'status' field.
        A game is marked as 'live' if its uppercase status is one of:
        {"Q1", "Q2", "Q3", "Q4", "OT", "BT", "HT"}; otherwise 'pregame'.
        """
        df = df.copy()
        live_statuses = {"Q1", "Q2", "Q3", "Q4", "OT", "BT", "HT"}
        if "status" in df.columns:
            df["status"] = df["status"].astype(str).str.upper()
            df["game_status"] = df["status"].apply(lambda s: "live" if s in live_statuses else "pregame")
        else:
            df["game_status"] = "pregame"
        return df
    
    def fetch_recent_historical_data(self, lookback_days: int = 30) -> pd.DataFrame:
        try:
            db_url = getattr(config, 'DATABASE_URL', None)
            if not db_url:
                log_message("DATABASE_URL not found in config", "WARNING")
                return pd.DataFrame()
            engine = create_engine(db_url)
            cutoff_date = (datetime.now() - timedelta(days=lookback_days)).strftime('%Y-%m-%d')
            query = f"""
                SELECT * 
                FROM nba_historical_game_stats
                WHERE game_date >= '{cutoff_date}'
                ORDER BY game_date
            """
            hist_df = pd.read_sql(query, engine)
            log_message(f"Fetched {len(hist_df)} historical games from database")
            if 'game_date' in hist_df.columns:
                hist_df['game_date'] = pd.to_datetime(hist_df['game_date'], errors='coerce')
                pacific_tz = pytz.timezone("America/Los_Angeles")
                if hist_df['game_date'].dt.tz is None:
                    hist_df['game_date'] = hist_df['game_date'].dt.tz_localize(pacific_tz)
                else:
                    hist_df['game_date'] = hist_df['game_date'].dt.tz_convert(pacific_tz)
            return hist_df
        except Exception as e:
            handle_exception(e, "fetch_recent_historical_data")
            return pd.DataFrame()
    
    def predict_game_scores(self, df: pd.DataFrame) -> pd.DataFrame:
        if df.empty or self.model is None:
            return pd.DataFrame()
        expected_features = self.get_feature_list()
        X = df[expected_features].copy()
        predictions = self.model.predict(X)
        results = df.copy()
        
        # Default initial predictions
        results['predicted_home_score'] = predictions
        
        # Handle case when model prediction is unrealistic or fails
        # Apply minimum realistic scores as fallback
        mask = (results['predicted_home_score'] < 80) | (results['predicted_home_score'].isna())
        if mask.any():
            log_message(f"Applying fallback predictions for {mask.sum()} games with unrealistic scores", "WARNING")
            results.loc[mask, 'predicted_home_score'] = 105.0
        
        current_time = datetime.now()
        
        for idx, row in results.iterrows():
            try:
                # Adjust prediction based on matchup differential and momentum
                diff_weight = min(0.3 + (0.1 * row.get('current_quarter', 0)), 0.6)
                momentum_adj = row.get('cumulative_momentum', 0) * 3.0
                
                # Get current quarter to adjust predictions accordingly
                current_q = int(row.get('current_quarter', 0))
                home_score = float(row.get('home_score', 0))
                away_score = float(row.get('away_score', 0))
                
                # Calculate away team score prediction based on home prediction and differential
                # Use a more realistic differential calculation based on current scores
                current_diff = home_score - away_score
                predicted_diff = (row.get('prev_matchup_diff', 0) * diff_weight) + momentum_adj
                
                # Blend current differential with historical differential based on game progress
                q_weight = min(current_q / 4.0, 1.0)
                final_diff = current_diff * q_weight + predicted_diff * (1 - q_weight)
                
                away_guess = results.at[idx, 'predicted_home_score'] - final_diff
                
                # For games in progress, project final scores based on quarters completed
                # and current scoring rates
                if current_q > 0:
                    quarters_remaining = 4 - current_q
                    if quarters_remaining > 0:
                        # Calculate points per quarter
                        home_ppq = home_score / current_q
                        away_ppq = away_score / current_q
                        
                        # Project final scores based on current pace
                        projected_home = home_score + (home_ppq * quarters_remaining)
                        projected_away = away_score + (away_ppq * quarters_remaining)
                        
                        # Blend model prediction with pace-based projection
                        blend_factor = min(0.2 + (current_q * 0.2), 0.8)  # 0.4 for Q1, 0.8 for Q3
                        results.at[idx, 'predicted_home_score'] = (projected_home * blend_factor + 
                                                                  results.at[idx, 'predicted_home_score'] * (1 - blend_factor))
                        away_guess = (projected_away * blend_factor + away_guess * (1 - blend_factor))
                
                # Ensure predictions are at least as high as current scores
                results.at[idx, 'predicted_home_score'] = max(results.at[idx, 'predicted_home_score'], home_score)
                results.at[idx, 'predicted_away_score'] = max(away_guess, away_score)
                
                # Make sure predictions are realistic basketball scores
                if current_q < 3:  # Early in the game
                    results.at[idx, 'predicted_home_score'] = max(results.at[idx, 'predicted_home_score'], 85)
                    results.at[idx, 'predicted_away_score'] = max(results.at[idx, 'predicted_away_score'], 80)
                
                # Handle cases where predictions are unrealistically high
                max_score = 150  # Maximum realistic NBA score
                results.at[idx, 'predicted_home_score'] = min(results.at[idx, 'predicted_home_score'], max_score)
                results.at[idx, 'predicted_away_score'] = min(results.at[idx, 'predicted_away_score'], max_score)
                
                # Round scores to 1 decimal place for cleaner display
                results.at[idx, 'predicted_home_score'] = round(results.at[idx, 'predicted_home_score'], 1)
                results.at[idx, 'predicted_away_score'] = round(results.at[idx, 'predicted_away_score'], 1)
                
                # Calculate win probability and other metrics
                diff = results.at[idx, 'predicted_home_score'] - results.at[idx, 'predicted_away_score']
                slope = 0.1 + 0.1 * (current_q / 4.0)
                wp = 1.0 / (1.0 + np.exp(-slope * diff))
                results.at[idx, 'win_probability'] = wp
                results.at[idx, 'projected_margin'] = diff
                results.at[idx, 'total_projected_score'] = results.at[idx, 'predicted_home_score'] + results.at[idx, 'predicted_away_score']
                
                # Store prediction history
                game_id = row.get('game_id')
                if game_id not in self.prediction_history:
                    self.prediction_history[game_id] = []
                
                # Add current prediction to history
                self.prediction_history[game_id].append({
                    'timestamp': current_time,
                    'current_quarter': current_q,
                    'home_score': home_score,
                    'away_score': away_score,
                    'predicted_home_score': results.at[idx, 'predicted_home_score'],
                    'predicted_away_score': results.at[idx, 'predicted_away_score'],
                    'win_probability': wp
                })
            except Exception as e:
                # Fallback if anything goes wrong in prediction
                log_message(f"Error in game prediction: {e}", "ERROR")
                results.at[idx, 'predicted_home_score'] = max(105, float(row.get('home_score', 0)) * 1.1)
                results.at[idx, 'predicted_away_score'] = max(100, float(row.get('away_score', 0)) * 1.1)
                results.at[idx, 'win_probability'] = 0.5
                results.at[idx, 'projected_margin'] = results.at[idx, 'predicted_home_score'] - results.at[idx, 'predicted_away_score']
                results.at[idx, 'total_projected_score'] = results.at[idx, 'predicted_home_score'] + results.at[idx, 'predicted_away_score']
        
        return results
    
    def visualize_predictions(self, predictions_df: pd.DataFrame) -> None:
        if predictions_df.empty:
            log_message("No predictions to visualize", "WARNING")
            return
        
        n_games = predictions_df.shape[0]
        fig, axs = plt.subplots(n_games, 2, figsize=(16, 5 * n_games))
        if n_games == 1:
            axs = np.array([axs])
        
        display_limit = CONFIG["visualize_limit"]
        if n_games > display_limit:
            log_message(f"Displaying first {display_limit} of {n_games} games for readability")
            predictions_to_display = predictions_df.iloc[:display_limit].copy()
        else:
            predictions_to_display = predictions_df.copy()
        
        for i, (_, game) in enumerate(predictions_to_display.iterrows()):
            # First chart: Current vs Predicted Final Scores
            ax_scores = axs[i, 0]
            teams = [game['home_team'], game['away_team']]
            current_scores = [float(game.get('home_score', 0)), float(game.get('away_score', 0))]
            predicted_scores = [float(game.get('predicted_home_score', 0)), float(game.get('predicted_away_score', 0))]
            
            x = np.arange(len(teams))
            width = 0.35
            ax_scores.bar(x - width/2, current_scores, width, label='Current', color='lightblue')
            ax_scores.bar(x + width/2, predicted_scores, width, label='Predicted Final', color='salmon')
            
            for j, v in enumerate(current_scores):
                ax_scores.text(j - width/2, v + 1, f"{v:.0f}", ha='center', fontweight='bold')
            for j, v in enumerate(predicted_scores):
                ax_scores.text(j + width/2, v + 1, f"{v:.1f}", ha='center', fontweight='bold')
            
            cq = int(game.get('current_quarter', 0))
            status_text = f"Q{cq}" if cq > 0 else "Pre-game"
            if game.get('game_status', 'unknown') == 'finished':
                status_text = "Final"
            
            title = f"{game['home_team']} vs {game['away_team']} - {status_text}"
            if 'win_probability' in game:
                win_pct = game['win_probability'] * 100
                title += f" (Home Win: {win_pct:.1f}%)"
            
            ax_scores.set_title(title)
            ax_scores.set_xticks(x)
            ax_scores.set_xticklabels(teams)
            ax_scores.legend()
            
            # Second chart: Prediction Evolution
            ax_hist = axs[i, 1]
            gid = game.get('game_id', f'game_{i}')
            
            # If we have enough history points, use them; otherwise generate simulated history
            if gid in self.prediction_history and len(self.prediction_history[gid]) > 1:
                hist = self.prediction_history[gid]
                log_message(f"Using actual prediction history with {len(hist)} points", "DEBUG")
            else:
                hist = generate_simulated_prediction_history(game)
                log_message(f"Using simulated prediction history with {len(hist)} points", "DEBUG")
                self.prediction_history[gid] = hist
            
            hist_df = pd.DataFrame(hist)
            
            if len(hist_df) >= 1:
                x_vals = range(len(hist_df))
                
                # Plot the predicted scores evolution
                ax_hist.plot(x_vals, hist_df['predicted_home_score'], 
                             label=f"{game['home_team']} Final", marker='o', color='blue')
                ax_hist.plot(x_vals, hist_df['predicted_away_score'], 
                             label=f"{game['away_team']} Final", marker='s', color='red')
                
                # Set appropriate y-axis limits for basketball scores
                min_score = min(hist_df['predicted_home_score'].min(), hist_df['predicted_away_score'].min())
                max_score = max(hist_df['predicted_home_score'].max(), hist_df['predicted_away_score'].max())
                y_padding = (max_score - min_score) * 0.1  # 10% padding
                ax_hist.set_ylim(max(0, min_score - y_padding), max_score + y_padding)
                
                # Add win probability on secondary axis if available
                if 'win_probability' in hist_df.columns:
                    ax_wp = ax_hist.twinx()
                    ax_wp.plot(x_vals, hist_df['win_probability']*100, 
                               label='Win Prob %', linestyle='--', color='green')
                    ax_wp.set_ylabel("Win Probability (%)")
                    ax_wp.set_ylim(0, 100)
                    
                    # Combine legends
                    lines, labels = ax_hist.get_legend_handles_labels()
                    lines2, labels2 = ax_wp.get_legend_handles_labels()
                    ax_hist.legend(lines + lines2, labels + labels2, loc='upper left')
                else:
                    ax_hist.legend(loc='upper left')
                
                ax_hist.set_title("Prediction Evolution")
                
                # Set x-axis labels based on quarter
                if 'current_quarter' in hist_df.columns:
                    ax_hist.set_xlabel("Quarter")
                    ax_hist.set_ylabel("Predicted Score")
                    x_labels = [f"Q{q}" for q in hist_df['current_quarter']]
                    ax_hist.set_xticks(x_vals)
                    ax_hist.set_xticklabels(x_labels)
                else:
                    ax_hist.set_xlabel("Update #")
                    ax_hist.set_ylabel("Predicted Score")
                    ax_hist.set_xticks(x_vals)
                    ax_hist.set_xticklabels([f"#{i+1}" for i in x_vals])
                    
                # Add grid for better readability
                ax_hist.grid(True, linestyle='--', alpha=0.6)
                
            else:
                ax_hist.text(0.5, 0.5, "No prediction history yet", 
                            ha='center', va='center', transform=ax_hist.transAxes, fontsize=12)
        
        plt.tight_layout()
        plt.show()
        self.print_prediction_summary(predictions_df)
    
    def print_prediction_summary(self, predictions_df: pd.DataFrame) -> None:
        print(f"\nGame Predictions - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("="*80)
        for i, (_, game) in enumerate(predictions_df.iterrows()):
            counter = f"[{i+1}/{len(predictions_df)}] " if len(predictions_df) > 10 else ""
            cq = int(game.get('current_quarter', 0))
            status_text = "Final" if game.get('game_status', 'unknown') == 'finished' else (f"Quarter {cq}" if cq > 0 else "Pre-game")
            print(f"\n{counter}{game['home_team']} vs {game['away_team']} - {status_text}")
            if cq > 0:
                print(f"Current Score: {game['home_team']} {game.get('home_score', 0):.0f} - {game['away_team']} {game.get('away_score', 0):.0f}")
            print(f"Predicted Final: {game['home_team']} {game.get('predicted_home_score', 0):.1f} - {game['away_team']} {game.get('predicted_away_score', 0):.1f}")
            if 'win_probability' in game:
                print(f"Win Probability: {game['win_probability']*100:.1f}%")
            if 'cumulative_momentum' in game and abs(game.get('cumulative_momentum', 0)) > 0.2:
                team_with_momentum = game['home_team'] if game['cumulative_momentum'] > 0 else game['away_team']
                print(f"Momentum: {team_with_momentum} ({abs(game['cumulative_momentum'])*100:.0f}%)")
            if i >= 9 and len(predictions_df) > 10:
                remaining = len(predictions_df) - 10
                print(f"\n... and {remaining} more games (not shown for brevity)")
                break
            print("-"*80)
    
    def generate_recommendations(self, predictions_df: pd.DataFrame) -> Dict[str, Dict[str, str]]:
        recs = {}
        league_avg = CONFIG["league_avg_score"]
        for _, game in predictions_df.iterrows():
            game_key = f"{game['home_team']} vs {game['away_team']}"
            
            # Get prediction values, with fallbacks to ensure reasonable values
            home_score = float(game.get('home_score', 0))
            away_score = float(game.get('away_score', 0))
            
            # Get predicted scores, with fallbacks if they're missing or unrealistic
            pred_home = float(game.get('predicted_home_score', 0))
            pred_away = float(game.get('predicted_away_score', 0))
            
            # If we have zero/missing predictions but have current scores, project based on them
            if (pred_home < 10 or pred_away < 10) and (home_score > 0 or away_score > 0):
                quarter = int(game.get('current_quarter', 0))
                if quarter > 0:
                    # Project based on quarters played so far
                    quarters_left = 4 - quarter
                    points_per_quarter_home = home_score / quarter
                    points_per_quarter_away = away_score / quarter
                    pred_home = home_score + (points_per_quarter_home * quarters_left)
                    pred_away = away_score + (points_per_quarter_away * quarters_left)
                else:
                    # Use reasonable defaults
                    pred_home = 110
                    pred_away = 105
            
            # Calculate derived metrics
            wp = float(game.get('win_probability', 0.5))
            if wp < 0.01:  # Fix extreme values
                wp = 0.5 + (0.15 * (1 if pred_home > pred_away else -1))
                
            momentum = float(game.get('cumulative_momentum', 0))
            margin = pred_home - pred_away
            total_pts = pred_home + pred_away
            quarter = int(game.get('current_quarter', 0))
            
            tips = {}
            if quarter >= 3 and wp > 0.9:
                tips["betting_tip"] = f"Strong confidence in {game['home_team']} win ({wp*100:.1f}%)"
            elif wp > 0.75:
                tips["betting_tip"] = f"Home advantage favors {game['home_team']} ({wp*100:.1f}%)"
            elif wp < 0.3:
                tips["betting_tip"] = f"Consider {game['away_team']} for upset ({(1-wp)*100:.1f}%)"
            else:
                tips["betting_tip"] = "Competitive game; consider hedging."
                
            if momentum > 0.3:
                tips["momentum_advice"] = f"Strong momentum for {game['home_team']}"
            elif momentum < -0.3:
                tips["momentum_advice"] = f"Strong momentum for {game['away_team']}"
            else:
                tips["momentum_advice"] = "Momentum appears balanced."
                
            if abs(margin) >= 8:
                fav = game['home_team'] if margin > 0 else game['away_team']
                tips["spread_tip"] = f"{fav} projected to cover by {abs(margin):.1f} points."
            else:
                tips["spread_tip"] = f"Tight margin projected ({abs(margin):.1f} points)."
                
            if total_pts > league_avg + 10:
                tips["over_under_tip"] = f"High-scoring game likely ({total_pts:.1f} points)."
            elif total_pts < league_avg - 10:
                tips["over_under_tip"] = f"Low-scoring game likely ({total_pts:.1f} points)."
            else:
                tips["over_under_tip"] = f"Projected total: {total_pts:.1f} points."
                
            if quarter == 4 and abs(margin) < 6:
                tips["clutch_tip"] = "Close game in final minutes – monitor closely."
                
            if quarter < 2:
                tips["early_game_tip"] = "Early game – watch for adjustments."
                
            # Add the predicted final scores to the recommendations
            tips["predicted_final"] = f"Projected final: {game['home_team']} {pred_home:.1f} - {game['away_team']} {pred_away:.1f}"
                
            recs[game_key] = tips
            
        return recs
    
    def run_full_prediction_pipeline(self) -> pd.DataFrame:
        try:
            log_message("Starting prediction pipeline...")
            log_message("Fetching today's games...")
            games_df = self.fetch_live_games_pacific()
            if games_df.empty:
                log_message("No games available for today", "WARNING")
                return pd.DataFrame()
            log_message(f"Processing {len(games_df)} games by status:")
            status_groups = {}
            for status in ['live', 'pregame', 'finished', 'unknown']:
                status_games = games_df[games_df['game_status'] == status]
                if not status_games.empty:
                    status_groups[status] = status_games
                    log_message(f"  - {status.upper()}: {len(status_games)} games")
            log_message("Fetching historical data for rest calculations...")
            hist_df = self.fetch_recent_historical_data(lookback_days=CONFIG["lookback_days"])
            log_message("Calculating rest features and preparing games...")
            today_games = calculate_improved_rest_features(games_df, max_rest_days=CONFIG["max_rest_days"])
            log_message("Calculating game features...")
            today_games = calculate_derived_features(today_games)
            expected = self.get_feature_list()
            today_games = ensure_numeric_features(today_games, expected)
            log_message("Generating predictions...")
            predictions_df = self.predict_game_scores(today_games)
            if predictions_df.empty:
                log_message("No predictions generated", "WARNING")
                return pd.DataFrame()
                
            # Find the visualization target first, which will generate prediction history
            # that we can use for both visualization and text summaries
            to_visualize = None
            if 'live' in status_groups:
                log_message("Preparing live game predictions...")
                to_visualize = predictions_df[predictions_df['game_status'] == 'live']
            elif 'pregame' in status_groups:
                log_message("No live games. Preparing pregame predictions...")
                to_visualize = predictions_df[predictions_df['game_status'] == 'pregame']
            elif 'finished' in status_groups:
                log_message("Preparing finished game results...")
                to_visualize = predictions_df[predictions_df['game_status'] == 'finished'].head(CONFIG["visualize_limit"])
            
            # Generate and print the summaries using the latest predictions
            self.print_game_status_summary(status_groups)
            self.print_prediction_summary(predictions_df)
            
            # Visualize if we have data
            if to_visualize is not None and not to_visualize.empty:
                print(f"\nShowing predictions for {len(to_visualize)} games:")
                self.visualize_predictions(to_visualize)
            else:
                print("\nNo games to visualize")
                
            # Update status_groups with the predicted values for recommendations
            for status in status_groups:
                game_ids = status_groups[status]['game_id'].tolist()
                status_groups[status] = predictions_df[predictions_df['game_id'].isin(game_ids)]
                
            self.print_game_recommendations(status_groups)
            return predictions_df
        except Exception as e:
            handle_exception(e, "run_full_prediction_pipeline")
            return pd.DataFrame()
    
    def print_game_status_summary(self, status_groups: Dict[str, pd.DataFrame]) -> None:
        print("\nGame Status Summary:")
        print("=" * 80)
        for status, games in status_groups.items():
            print(f"{status.upper()}: {len(games)} games")
            for idx, game in games.iterrows():
                home = game['home_team']
                away = game['away_team']
                home_score = game.get('home_score', 0)
                away_score = game.get('away_score', 0)
                if status == 'pregame':
                    print(f"  • {home} vs {away} (Upcoming)")
                else:
                    print(f"  • {home} {home_score:.0f} - {away_score:.0f} {away}")
        print("=" * 80)
    
    def print_prediction_summary(self, predictions_df: pd.DataFrame) -> None:
        print(f"\nGame Predictions - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("=" * 80)
        for i, (_, game) in enumerate(predictions_df.iterrows()):
            counter = f"[{i+1}/{len(predictions_df)}] " if len(predictions_df) > 10 else ""
            cq = int(game.get('current_quarter', 0))
            status_text = "Final" if game.get('game_status', 'unknown') == 'finished' else (f"Quarter {cq}" if cq > 0 else "Pre-game")
            print(f"\n{counter}{game['home_team']} vs {game['away_team']} - {status_text}")
            
            # Current scores for games in progress
            home_score = float(game.get('home_score', 0))
            away_score = float(game.get('away_score', 0))
            if cq > 0:
                print(f"Current Score: {game['home_team']} {home_score:.0f} - {game['away_team']} {away_score:.0f}")
            
            # Get predicted scores with fallbacks for zero/missing values
            pred_home = float(game.get('predicted_home_score', 0))
            pred_away = float(game.get('predicted_away_score', 0))
            
            # If we have zero/missing predictions but have current scores, project based on them
            if (pred_home < 10 or pred_away < 10) and (home_score > 0 or away_score > 0):
                if cq > 0:
                    # Project based on quarters played so far
                    quarters_left = 4 - cq
                    points_per_quarter_home = home_score / cq
                    points_per_quarter_away = away_score / cq
                    pred_home = home_score + (points_per_quarter_home * quarters_left)
                    pred_away = away_score + (points_per_quarter_away * quarters_left)
                else:
                    # Use reasonable defaults for pregame
                    pred_home = 110
                    pred_away = 105
            
            # Update prediction in dataframe for later use
            predictions_df.loc[predictions_df.index[i], 'predicted_home_score'] = pred_home
            predictions_df.loc[predictions_df.index[i], 'predicted_away_score'] = pred_away
            
            # Print the predicted final score
            print(f"Predicted Final: {game['home_team']} {pred_home:.1f} - {game['away_team']} {pred_away:.1f}")
            
            # Win probability
            wp = float(game.get('win_probability', 0.5))
            if wp < 0.01:  # Fix extreme values
                wp = 0.5 + (0.15 * (1 if pred_home > pred_away else -1))
                predictions_df.loc[predictions_df.index[i], 'win_probability'] = wp
            print(f"Win Probability: {wp*100:.1f}%")
            
            # Momentum info
            if 'cumulative_momentum' in game and abs(game.get('cumulative_momentum', 0)) > 0.2:
                team_with_momentum = game['home_team'] if game['cumulative_momentum'] > 0 else game['away_team']
                print(f"Momentum: {team_with_momentum} ({abs(game['cumulative_momentum'])*100:.0f}%)")
                
            # Update margin and total for later recommendations
            predictions_df.loc[predictions_df.index[i], 'projected_margin'] = pred_home - pred_away
            predictions_df.loc[predictions_df.index[i], 'total_projected_score'] = pred_home + pred_away
            
            # Limit output for very large datasets
            if i >= 9 and len(predictions_df) > 10:
                remaining = len(predictions_df) - 10
                print(f"\n... and {remaining} more games (not shown for brevity)")
                break
            print("-"*80)
        
        return predictions_df
    
    def print_game_recommendations(self, status_groups: Dict[str, pd.DataFrame]) -> None:
        if 'live' in status_groups and not status_groups['live'].empty:
            live_recs = self.generate_recommendations(status_groups['live'])
            print("\nLive Game Recommendations:")
            print("=" * 80)
            for game_name, rec in live_recs.items():
                print(f"\n{game_name}:")
                for k, v in rec.items():
                    print(f"  • {k}: {v}")
                print("-" * 40)
        if 'pregame' in status_groups and not status_groups['pregame'].empty:
            pregame_recs = self.generate_recommendations(status_groups['pregame'])
            print("\nPregame Recommendations:")
            print("=" * 80)
            for game_name, rec in pregame_recs.items():
                print(f"\n{game_name}:")
                for k, v in rec.items():
                    print(f"  • {k}: {v}")
                print("-" * 40)

# ==================== MAIN EXECUTION =====================

if __name__ == "__main__":
    predictor_config = {
        "debug": False,
        "visualize_limit": 6,
        "lookback_days": 30,
    }
    predictor = NBAGamePredictor(predictor_config)
    predictions = predictor.run_full_prediction_pipeline()

In [ ]:
# Cell 7: Early Quarter XGBoost Model Implementation

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Early Quarter XGBoost Model Enhancement\n")

# Check for XGBoost availability
try:
    import xgboost as xgb
    print("XGBoost is available for use.")
    xgboost_available = True
except ImportError:
    print("XGBoost is not installed. Some functionality will be limited.")
    print("Install XGBoost with: pip install xgboost")
    xgboost_available = False

from models.features import EarlyQuarterModelOptimizer, NBAFeatureGenerator

# Load historical data for training
from sqlalchemy import create_engine
import config

print("Loading historical data for early quarter model training...")
engine = create_engine(config.DATABASE_URL)
query = """
SELECT * FROM nba_historical_game_stats 
WHERE game_date >= CURRENT_DATE - INTERVAL '365 days'
ORDER BY game_date
"""
historical_df = pd.read_sql(query, engine)
print(f"Loaded {len(historical_df)} historical games")

if len(historical_df) < 100:
    print("Not enough historical data for effective training.")
else:
    # Generate features
    feature_generator = NBAFeatureGenerator(debug=True)
    features_df = feature_generator.generate_all_features(historical_df)
    print(f"Generated features for {len(features_df)} games")
    
    # Split data into training and testing sets
    train_df, test_df = train_test_split(features_df, test_size=0.2, random_state=42)
    print(f"Training set: {len(train_df)} games, Test set: {len(test_df)} games")
    
    # Initialize the early quarter model optimizer
    optimizer = EarlyQuarterModelOptimizer(feature_generator=feature_generator)
    
    # Initialize XGBoost models (if available)
    if xgboost_available:
        optimizer.initialize_xgboost_models()
        
        # Train models
        print("\nTraining early quarter models...")
        training_results = optimizer.train_models(train_df, target_col='home_score')
        
        # Evaluate on test data
        print("\nEvaluating models on test data...")
        eval_results = optimizer.evaluate_models(test_df, target_col='home_score')
        
        if not eval_results.empty:
            # Display evaluation results
            print("\nEarly Quarter Model Performance:")
            display(eval_results)
            
            # Compare to baseline
            baseline_rmse = {
                "1": 7.0,  # Baseline RMSE for Q1
                "2": 6.2   # Baseline RMSE for Q2
            }
            comparison = optimizer.compare_to_baseline(eval_results, baseline_rmse)
            
            if not comparison.empty:
                print("\nImprovement over baseline:")
                display(comparison[['quarter', 'model', 'feature_set', 'rmse', 'baseline_rmse', 'pct_improvement']])
                
                # Visualize comparison
                plt.figure(figsize=(12, 6))
                
                # Plot Q1 models
                q1_data = comparison[comparison['quarter'] == 'Q1']
                if not q1_data.empty:
                    plt.subplot(1, 2, 1)
                    plt.bar(q1_data['model'] + '_' + q1_data['feature_set'], q1_data['rmse'], color='cornflowerblue')
                    plt.axhline(y=baseline_rmse["1"], color='r', linestyle='--', 
                               label=f'Baseline: {baseline_rmse["1"]:.2f}')
                    plt.title('Q1 Model Performance')
                    plt.ylabel('RMSE')
                    plt.xticks(rotation=45, ha='right')
                    plt.legend()
                    plt.grid(axis='y', alpha=0.3)
                
                # Plot Q2 models
                q2_data = comparison[comparison['quarter'] == 'Q2']
                if not q2_data.empty:
                    plt.subplot(1, 2, 2)
                    plt.bar(q2_data['model'] + '_' + q2_data['feature_set'], q2_data['rmse'], color='lightcoral')
                    plt.axhline(y=baseline_rmse["2"], color='r', linestyle='--', 
                               label=f'Baseline: {baseline_rmse["2"]:.2f}')
                    plt.title('Q2 Model Performance')
                    plt.ylabel('RMSE')
                    plt.xticks(rotation=45, ha='right')
                    plt.legend()
                    plt.grid(axis='y', alpha=0.3)
                
                plt.tight_layout()
                plt.show()
                
                # Visualize feature importance for best model
                print("\nFeature Importance Analysis:")
                
                # Best Q1 model feature importance
                best_q1 = q1_data.sort_values('rmse').iloc[0]
                print(f"Best Q1 Model: {best_q1['model']}_{best_q1['feature_set']} (RMSE: {best_q1['rmse']:.2f})")
                q1_importance_fig = optimizer.visualize_feature_importance(1)
                if q1_importance_fig:
                    plt.figure(q1_importance_fig.number)
                    plt.show()
                
                # Best Q2 model feature importance
                best_q2 = q2_data.sort_values('rmse').iloc[0]
                print(f"Best Q2 Model: {best_q2['model']}_{best_q2['feature_set']} (RMSE: {best_q2['rmse']:.2f})")
                q2_importance_fig = optimizer.visualize_feature_importance(2)
                if q2_importance_fig:
                    plt.figure(q2_importance_fig.number)
                    plt.show()
                
                # Test the best models on a sample game
                print("\nTesting best models on sample game data...")
                sample_game = test_df.iloc[0].copy()
                
                # Q1 prediction
                q1_model = best_q1['model']
                q1_feature_set = best_q1['feature_set']
                q1_pred = optimizer.predict_quarter(sample_game, 1, model_name=q1_model, feature_set=q1_feature_set)
                
                # Q2 prediction
                q2_model = best_q2['model']
                q2_feature_set = best_q2['feature_set']
                q2_pred = optimizer.predict_quarter(sample_game, 2, model_name=q2_model, feature_set=q2_feature_set)
                
                print(f"Sample Game: {sample_game['home_team']} vs {sample_game['away_team']}")
                print(f"Actual Q1 score: {sample_game['home_q1']}, Predicted: {q1_pred:.2f}")
                print(f"Actual Q2 score: {sample_game['home_q2']}, Predicted: {q2_pred:.2f}")
                
                # Save the best models
                print("\nSaving best early quarter models...")
                optimizer.save_models()
    else:
        print("Cannot train XGBoost models. Please install XGBoost first.")

def create_chronological_split(df, test_fraction=0.2, date_column='game_date'):
    """
    Create chronologically ordered train-test split to prevent data leakage.
    
    Args:
        df: DataFrame to split
        test_fraction: Fraction of data to use for testing
        date_column: Column name containing dates
        
    Returns:
        tuple: (train_df, test_df)
    """
    # Ensure DataFrame is sorted by date
    if date_column in df.columns:
        sorted_df = df.sort_values(date_column).copy()
    else:
        print(f"Warning: {date_column} not found, using existing order")
        sorted_df = df.copy()
    
    # Determine split point
    split_idx = int(len(sorted_df) * (1 - test_fraction))
    
    # Split data
    train_df = sorted_df.iloc[:split_idx].copy()
    test_df = sorted_df.iloc[split_idx:].copy()
    
    return train_df, test_df

def ensure_no_data_leakage(train_df, test_df, date_column='game_date'):
    """
    Verify that no data leakage exists between train and test sets.
    
    Args:
        train_df: Training DataFrame
        test_df: Testing DataFrame
        date_column: Column name containing dates
        
    Returns:
        bool: True if no leakage, False otherwise
    """
    if date_column not in train_df.columns or date_column not in test_df.columns:
        print(f"Warning: Cannot verify data leakage without {date_column} column")
        return False
    
    train_max_date = train_df[date_column].max()
    test_min_date = test_df[date_column].min()
    
    if train_max_date >= test_min_date:
        print(f"Data leakage detected: Train max date ({train_max_date}) >= Test min date ({test_min_date})")
        return False
    
    print(f"No data leakage: Train dates end at {train_max_date}, Test dates begin at {test_min_date}")
    return True

In [ ]:
# Cell 7B: Enhanced Live Prediction Function with Momentum and Win Probability

from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import joblib
import math
import traceback

def calculate_win_probability(home_score: float, away_score: float, quarter: int, time_remaining: float = None) -> float:
    """
    Calculate win probability for the home team using a logistic function.
    """
    score_diff = home_score - away_score
    if time_remaining is not None:
        total_time = 48.0
        elapsed = total_time - time_remaining
        progress = elapsed / total_time
    else:
        progress = min(quarter / 4.0, 1.0)
    k = 0.05 + (progress * 0.15)
    return 1.0 / (1.0 + math.exp(-k * score_diff))

def calculate_momentum(home_q1, home_q2, home_q3, home_q4, away_q1, away_q2, away_q3, away_q4, current_quarter: int) -> float:
    """Calculate normalized cumulative momentum."""
    h = [float(x or 0) for x in [home_q1, home_q2, home_q3, home_q4]]
    a = [float(x or 0) for x in [away_q1, away_q2, away_q3, away_q4]]
    momentum = 0
    if current_quarter >= 2:
        m1 = (h[1] - h[0]) - (a[1] - a[0])
    else:
        m1 = 0
    if current_quarter >= 3:
        m2 = (h[2] - h[1]) - (a[2] - a[1])
    else:
        m2 = 0
    if current_quarter >= 4:
        m3 = (h[3] - h[2]) - (a[3] - a[2])
    else:
        m3 = 0
    if current_quarter == 2:
        momentum = m1
    elif current_quarter == 3:
        momentum = (m1*0.2 + m2*0.3) / (0.2+0.3)
    elif current_quarter >= 4:
        momentum = (m1*0.2 + m2*0.3 + m3*0.5) / (0.2+0.3+0.5)
    return np.clip(momentum/15.0, -1, 1)

# Example usage:
home_prob = calculate_win_probability(110, 105, 3)
momentum_val = calculate_momentum(28, 27, 22, 0, 24, 29, 26, 0, 3)
print(f"Win Probability: {home_prob:.2f}, Normalized Momentum: {momentum_val:.2f}")

def initialize_quarter_specific_models(features_df, safe_init=True):
    """
    Initialize quarter-specific models with safe handling of features.
    
    Args:
        features_df: DataFrame with features
        safe_init: Whether to use safe initialization
        
    Returns:
        dict: Quarter-specific models
    """
    models = {}
    
    # Define feature sets for each quarter with fallbacks
    quarter_features = {
        'q1': ['rest_days_home', 'rest_days_away', 'prev_matchup_diff', 
              'rolling_home_score', 'rolling_away_score'],
        'q2': ['home_q1', 'away_q1', 'prev_matchup_diff', 'rest_advantage'],
        'q3': ['home_q1', 'home_q2', 'away_q1', 'away_q2', 'first_half_diff'],
        'q4': ['home_q1', 'home_q2', 'home_q3', 'away_q1', 'away_q2', 'away_q3', 'pre_q4_diff']
    }
    
    for quarter, features in quarter_features.items():
        # Skip if DataFrame is empty
        if features_df.empty:
            continue
            
        # Safe init checks that all features exist
        if safe_init:
            # Get current existing features
            existing_features = [f for f in features if f in features_df.columns]
            
            # If missing critical features, use fallbacks
            if len(existing_features) < len(features) * 0.6:  # Missing >40% of features
                from sklearn.linear_model import Ridge
                print(f"Warning: Missing critical features for {quarter}. Using fallback model.")
                models[f'{quarter}_model'] = Ridge(alpha=1.0, random_state=42)
                continue
            
            # Use what we have, but warn
            if len(existing_features) < len(features):
                print(f"Warning: Using {len(existing_features)}/{len(features)} features for {quarter} model")
                features = existing_features
        
        # Get target and training data
        q_num = quarter[1]  # Extract quarter number from 'q1', 'q2', etc.
        target_col = f'home_q{q_num}'
        
        if target_col not in features_df.columns:
            print(f"Warning: Target column {target_col} not found for {quarter}. Using fallback model.")
            from sklearn.linear_model import Ridge
            models[f'{quarter}_model'] = Ridge(alpha=1.0, random_state=42)
            continue
        
        # Prepare data
        X = ensure_features_exist(features_df, features)
        X = X[features].copy()
        y = features_df[target_col]
        
        # Handle missing values
        X = X.fillna(0)
        y = y.fillna(y.mean())
        
        # Create and train model
        if quarter == 'q4':
            # Ridge regression for Q4 (more stable)
            from sklearn.linear_model import Ridge
            model = Ridge(alpha=1.0, random_state=42)
        else:
            # GradientBoosting for other quarters
            from sklearn.ensemble import GradientBoostingRegressor
            model = GradientBoostingRegressor(
                n_estimators=100,
                max_depth=3,
                learning_rate=0.1,
                random_state=42
            )
        
        # Train model
        model.fit(X, y)
        models[f'{quarter}_model'] = model
    
    return models


In [ ]:
#Cell 7B-2 : Enhanced Momentum Integration
import pandas as pd
import numpy as np
import time

def calculate_momentum_features(df, vectorized=True):
    """
    Calculate momentum features for game data.
    
    Args:
        df: DataFrame with game data
        vectorized: Whether to use vectorized implementation (default: True)
        
    Returns:
        DataFrame with momentum features added
    """
    if vectorized:
        return _calculate_momentum_features_vectorized(df)
    else:
        return _calculate_momentum_features_fixed(df)

def _calculate_momentum_features_fixed(df):
    """
    Calculate momentum features with proper type handling (non-vectorized).
    Use for debugging or when vectorized operations cause issues.
    
    Args:
        df: DataFrame with game data
        
    Returns:
        DataFrame with momentum features added
    """
    features_df = df.copy()
    
    # Initialize momentum columns with explicit float type
    features_df['q1_to_q2_momentum'] = 0.0
    features_df['q2_to_q3_momentum'] = 0.0
    features_df['q3_to_q4_momentum'] = 0.0
    features_df['cumulative_momentum'] = 0.0
    
    for idx, row in features_df.iterrows():
        current_quarter = int(row.get('current_quarter', 0))
        
        # Calculate quarter-to-quarter momentum with explicit float casting
        if current_quarter >= 2:
            home_q1 = float(row.get('home_q1', 0) or 0)
            home_q2 = float(row.get('home_q2', 0) or 0)
            away_q1 = float(row.get('away_q1', 0) or 0)
            away_q2 = float(row.get('away_q2', 0) or 0)
            
            q1_to_q2 = (home_q2 - home_q1) - (away_q2 - away_q1)
            features_df.at[idx, 'q1_to_q2_momentum'] = float(q1_to_q2)
        
        if current_quarter >= 3:
            home_q2 = float(row.get('home_q2', 0) or 0)
            home_q3 = float(row.get('home_q3', 0) or 0)
            away_q2 = float(row.get('away_q2', 0) or 0)
            away_q3 = float(row.get('away_q3', 0) or 0)
            
            q2_to_q3 = (home_q3 - home_q2) - (away_q3 - away_q2)
            features_df.at[idx, 'q2_to_q3_momentum'] = float(q2_to_q3)
        
        if current_quarter >= 4:
            home_q3 = float(row.get('home_q3', 0) or 0)
            home_q4 = float(row.get('home_q4', 0) or 0)
            away_q3 = float(row.get('away_q3', 0) or 0)
            away_q4 = float(row.get('away_q4', 0) or 0)
            
            q3_to_q4 = (home_q4 - home_q3) - (away_q4 - away_q3)
            features_df.at[idx, 'q3_to_q4_momentum'] = float(q3_to_q4)
        
        # Calculate weighted momentum with explicit float handling
        weights = [0.2, 0.3, 0.5]  # Weights for each quarter transition
        
        if current_quarter == 2:
            momentum = float(features_df.at[idx, 'q1_to_q2_momentum'])
            features_df.at[idx, 'cumulative_momentum'] = float(momentum / 15.0)
        elif current_quarter == 3:
            momentum = float(
                (features_df.at[idx, 'q1_to_q2_momentum'] * weights[0] + 
                 features_df.at[idx, 'q2_to_q3_momentum'] * weights[1]) / 
                (weights[0] + weights[1])
            )
            features_df.at[idx, 'cumulative_momentum'] = float(momentum / 15.0)
        elif current_quarter >= 4:
            momentum = float(
                (features_df.at[idx, 'q1_to_q2_momentum'] * weights[0] + 
                 features_df.at[idx, 'q2_to_q3_momentum'] * weights[1] + 
                 features_df.at[idx, 'q3_to_q4_momentum'] * weights[2]) / 
                sum(weights)
            )
            features_df.at[idx, 'cumulative_momentum'] = float(momentum / 15.0)
        
        # Normalize to [-1, 1] with explicit float handling
        features_df.at[idx, 'cumulative_momentum'] = float(max(min(float(features_df.at[idx, 'cumulative_momentum']), 1.0), -1.0))
    
    return features_df

def _calculate_momentum_features_vectorized(df):
    """
    Calculate momentum features using vectorized operations with proper type handling.
    Significantly improves performance while eliminating dtype warnings.
    
    Args:
        df: DataFrame with game data
        
    Returns:
        DataFrame with momentum features added
    """
    # Create copy to avoid modifying original
    features_df = df.copy()
    
    # Ensure quarters are numeric with proper handling for None/NaN values
    for team in ['home', 'away']:
        for q in range(1, 5):
            col = f'{team}_q{q}'
            if col in features_df.columns:
                # Convert to float and handle None/NaN values
                features_df[col] = pd.to_numeric(features_df[col], errors='coerce').fillna(0.0)
    
    # Ensure current_quarter is numeric
    if 'current_quarter' in features_df.columns:
        features_df['current_quarter'] = pd.to_numeric(features_df['current_quarter'], errors='coerce').fillna(0).astype(int)
    
    # Initialize momentum columns with explicit float dtype
    momentum_cols = ['q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum']
    for col in momentum_cols:
        features_df[col] = np.zeros(len(features_df), dtype=np.float64)
    
    # Calculate quarter-to-quarter momentum with vectorized operations
    # Q1 to Q2 momentum
    q2_mask = features_df['current_quarter'] >= 2
    if q2_mask.any():
        features_df.loc[q2_mask, 'q1_to_q2_momentum'] = (
            (features_df.loc[q2_mask, 'home_q2'] - features_df.loc[q2_mask, 'home_q1']) - 
            (features_df.loc[q2_mask, 'away_q2'] - features_df.loc[q2_mask, 'away_q1'])
        )
    
    # Q2 to Q3 momentum
    q3_mask = features_df['current_quarter'] >= 3
    if q3_mask.any():
        features_df.loc[q3_mask, 'q2_to_q3_momentum'] = (
            (features_df.loc[q3_mask, 'home_q3'] - features_df.loc[q3_mask, 'home_q2']) - 
            (features_df.loc[q3_mask, 'away_q3'] - features_df.loc[q3_mask, 'away_q2'])
        )
    
    # Q3 to Q4 momentum
    q4_mask = features_df['current_quarter'] >= 4
    if q4_mask.any():
        features_df.loc[q4_mask, 'q3_to_q4_momentum'] = (
            (features_df.loc[q4_mask, 'home_q4'] - features_df.loc[q4_mask, 'home_q3']) - 
            (features_df.loc[q4_mask, 'away_q4'] - features_df.loc[q4_mask, 'away_q3'])
        )
    
    # Calculate cumulative momentum using vectorized operations with weights
    weights = np.array([0.2, 0.3, 0.5])  # Weights for each quarter transition
    
    # Calculate for Q2
    q2_mask = features_df['current_quarter'] == 2
    if q2_mask.any():
        features_df.loc[q2_mask, 'cumulative_momentum'] = features_df.loc[q2_mask, 'q1_to_q2_momentum'] / 15.0
    
    # Calculate for Q3
    q3_mask = features_df['current_quarter'] == 3
    if q3_mask.any():
        # Calculate weighted average manually to ensure proper dtype handling
        weighted_sum = (
            features_df.loc[q3_mask, 'q1_to_q2_momentum'] * weights[0] +
            features_df.loc[q3_mask, 'q2_to_q3_momentum'] * weights[1]
        )
        weight_sum = weights[0] + weights[1]
        features_df.loc[q3_mask, 'cumulative_momentum'] = weighted_sum / (weight_sum * 15.0)
    
    # Calculate for Q4
    q4_mask = features_df['current_quarter'] >= 4
    if q4_mask.any():
        # Calculate weighted average manually to ensure proper dtype handling
        weighted_sum = (
            features_df.loc[q4_mask, 'q1_to_q2_momentum'] * weights[0] +
            features_df.loc[q4_mask, 'q2_to_q3_momentum'] * weights[1] +
            features_df.loc[q4_mask, 'q3_to_q4_momentum'] * weights[2]
        )
        weight_sum = weights.sum()
        features_df.loc[q4_mask, 'cumulative_momentum'] = weighted_sum / (weight_sum * 15.0)
    
    # Clip to [-1, 1] range
    for col in momentum_cols:
        # Using .clip() is faster than max/min in a loop
        features_df[col] = features_df[col].clip(-1.0, 1.0)
    
    return features_df

def calculate_dynamic_momentum_impact(momentum, score_differential, vectorized=True):
    """
    Calculate momentum impact that decreases in blowout games.
    
    Args:
        momentum: Raw momentum value(s) (-1 to 1), Series or scalar
        score_differential: Absolute score difference(s), Series or scalar
        vectorized: Whether to use vectorized implementation (default: True)
        
    Returns:
        Adjusted momentum value(s) (Series or scalar)
    """
    # Check if inputs are pandas Series or numpy arrays
    is_series = isinstance(momentum, (pd.Series, np.ndarray)) or hasattr(momentum, '__iter__')
    
    if is_series and vectorized:
        return _calculate_dynamic_momentum_impact_vectorized(momentum, score_differential)
    elif is_series:
        # Apply scalar function to each value using pandas apply
        return pd.Series([_calculate_dynamic_momentum_impact_scalar(m, s) 
                         for m, s in zip(momentum, score_differential)], 
                         index=getattr(momentum, 'index', None))
    else:
        # Single scalar values
        return _calculate_dynamic_momentum_impact_scalar(momentum, score_differential)

def _calculate_dynamic_momentum_impact_scalar(momentum, score_differential):
    """
    Calculate momentum impact for a single pair of values.
    
    Args:
        momentum: Raw momentum value (-1 to 1)
        score_differential: Absolute score difference
        
    Returns:
        float: Adjusted momentum value
    """
    # Calculate blowout factor: 1.0 for close games, approaches 0 for blowouts
    blowout_threshold = 20  # Point differential considered a complete blowout
    blowout_factor = max(0.0, 1.0 - (abs(score_differential) / blowout_threshold))
    
    # Apply to momentum - full impact in close games, reduced in blowouts
    return momentum * blowout_factor

def _calculate_dynamic_momentum_impact_vectorized(momentum_series, score_differential_series):
    """
    Vectorized version of momentum impact calculation.
    
    Args:
        momentum_series: Series of momentum values
        score_differential_series: Series of score differentials
        
    Returns:
        Series: Adjusted momentum values
    """
    # Convert inputs to numpy arrays for faster operations
    momentum = np.asarray(momentum_series, dtype=np.float64)
    score_diff = np.asarray(score_differential_series, dtype=np.float64)
    
    # Calculate blowout factor: 1.0 for close games, approaches 0 for blowouts
    blowout_threshold = 20.0  # Point differential considered a complete blowout
    blowout_factor = np.maximum(0.0, 1.0 - (np.abs(score_diff) / blowout_threshold))
    
    # Apply to momentum - full impact in close games, reduced in blowouts
    adjusted_momentum = momentum * blowout_factor
    
    return pd.Series(adjusted_momentum, index=getattr(momentum_series, 'index', None))

# Example usage section - only run this code if executing the cell directly
if __name__ == "__main__":
    # Test the momentum calculation with a simple example
    test_games = pd.DataFrame([
        {
            'game_id': 1001,
            'home_team': 'Boston Celtics',
            'away_team': 'Los Angeles Lakers',
            'current_quarter': 3,
            'home_q1': 28, 'home_q2': 30, 'home_q3': 25, 'home_q4': 0,
            'away_q1': 25, 'away_q2': 27, 'away_q3': 30, 'away_q4': 0,
            'home_score': 83,
            'away_score': 82
        }
    ])

    # Test with both implementations
    fixed_features = calculate_momentum_features(test_games, vectorized=False)
    vectorized_features = calculate_momentum_features(test_games, vectorized=True)
    
    print("Momentum features comparison:")
    momentum_cols = ['q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum']
    display(fixed_features[momentum_cols])
    
    # Test dynamic momentum impact
    score_diffs = [0, 5, 10, 15, 20, 25]
    momentum_val = 0.8
    print("\nDynamic momentum impact by score differential:")
    for diff in score_diffs:
        adjusted = calculate_dynamic_momentum_impact(momentum_val, diff, vectorized=False)
        print(f"Score diff: {diff:2d} | Momentum: {momentum_val:.1f} → Adjusted: {adjusted:.3f}")
    
    # Performance comparison
    def create_test_dataset(size=1000):
        """Create a test dataset for momentum calculation benchmarking"""
        np.random.seed(42)
        
        test_data = []
        for i in range(size):
            # Random quarter (0-4)
            quarter = np.random.randint(0, 5)
            
            # Generate random quarter scores
            home_scores = np.random.randint(15, 35, 4)
            away_scores = np.random.randint(15, 35, 4)
            
            # For quarters that haven't been played, set to 0
            if quarter < 4:
                home_scores[quarter:] = 0
                away_scores[quarter:] = 0
            
            # Create game dict
            game = {
                'game_id': 1000 + i,
                'current_quarter': quarter,
                'home_team': 'Team A',
                'away_team': 'Team B',
                'home_score': home_scores.sum(),
                'away_score': away_scores.sum()
            }
            
            # Add quarter scores
            for q in range(4):
                game[f'home_q{q+1}'] = home_scores[q]
                game[f'away_q{q+1}'] = away_scores[q]
            
            test_data.append(game)
        
        return pd.DataFrame(test_data)

    # Performance test with larger dataset
    def run_performance_test(dataset_size=5000):
        print(f"\nPerformance test with {dataset_size} records:")
        test_df = create_test_dataset(dataset_size)
        
        # Test non-vectorized implementation
        start_time = time.time()
        fixed_result = calculate_momentum_features(test_df, vectorized=False)
        fixed_time = time.time() - start_time
        print(f"Non-vectorized time: {fixed_time:.4f} seconds")
        
        # Test vectorized implementation
        start_time = time.time()
        vectorized_result = calculate_momentum_features(test_df, vectorized=True)
        vectorized_time = time.time() - start_time
        print(f"Vectorized time: {vectorized_time:.4f} seconds")
        
        # Calculate speedup
        speedup = fixed_time / vectorized_time
        print(f"Speedup factor: {speedup:.2f}x")
        
        # Verify results match
        for col in momentum_cols:
            if col in fixed_result.columns and col in vectorized_result.columns:
                max_diff = (fixed_result[col] - vectorized_result[col]).abs().max()
                print(f"Maximum difference in {col}: {max_diff:.6f}")
    
    # Uncomment to run performance tests
    # run_performance_test(5000)
    
    # Test dynamic momentum impact on vectors
    sample_momentum = pd.Series(np.random.uniform(-1, 1, 5))
    sample_score_diff = pd.Series(np.random.uniform(0, 30, 5))
    impact = calculate_dynamic_momentum_impact(sample_momentum, sample_score_diff)
    
    print("\nDynamic Momentum Impact Example:")
    for i in range(len(sample_momentum)):
        print(f"Momentum: {sample_momentum.iloc[i]:.2f}, Score Diff: {sample_score_diff.iloc[i]:.1f}, Impact: {impact.iloc[i]:.3f}")

In [ ]:
# Cell 7C: Comprehensive Monitoring System with Automatic Scheduling

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from datetime import datetime, timedelta
from IPython.display import clear_output
import traceback
import threading
import json
import os

class NBAGameMonitor:
    """
    System for monitoring NBA games with automated updates and prediction tracking.
    Builds on the NBAGamePredictor to provide continuous monitoring.
    """
    
    def __init__(self, update_interval=60, auto_save=True, debug=True):
        """
        Initialize the monitor with configurable settings.
        
        Args:
            update_interval: Seconds between updates (default: 60)
            auto_save: Whether to auto-save prediction history (default: True)
            debug: Enable detailed debug logging (default: True)
        """
        self.predictor = NBAGamePredictor()  # Assuming NBAGamePredictor is defined elsewhere
        self.update_interval = update_interval
        self.auto_save = auto_save
        self.debug = debug
        self.running = False
        self.update_thread = None
        self.prediction_history = {}
        self.update_count = 0
        self.last_update_time = None
        self.validation_results = None
        
        # File paths for saving data
        self.history_file = "prediction_history.json"
        self.validation_file = "model_validation.json"
        
        # Create directories if needed
        os.makedirs("data", exist_ok=True)
        
        # Load previous history if it exists
        self.load_history()
    
    def log(self, message, level="INFO"):
        """Log message with timestamp."""
        if not self.debug and level == "DEBUG":
            return
            
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"[{timestamp}] {level}: {message}")
    
    def load_history(self):
        """Load prediction history from file if available."""
        history_path = os.path.join("data", self.history_file)
        
        if os.path.exists(history_path):
            try:
                with open(history_path, 'r') as f:
                    raw_history = json.load(f)
                
                # Convert string timestamps back to datetime objects
                for game_id, predictions in raw_history.items():
                    for pred in predictions:
                        if 'timestamp' in pred and isinstance(pred['timestamp'], str):
                            pred['timestamp'] = datetime.fromisoformat(pred['timestamp'])
                
                self.prediction_history = raw_history
                self.log(f"Loaded prediction history for {len(self.prediction_history)} games")
            except Exception as e:
                self.log(f"Error loading prediction history: {e}", "ERROR")
    
    def save_history(self):
        """Save prediction history to file."""
        if not self.prediction_history:
            return
            
        history_path = os.path.join("data", self.history_file)
        
        try:
            # Convert datetime objects to ISO format strings for JSON serialization
            serializable_history = {}
            
            for game_id, predictions in self.prediction_history.items():
                serializable_predictions = []
                for pred in predictions:
                    pred_copy = pred.copy()
                    if 'timestamp' in pred_copy and isinstance(pred_copy['timestamp'], datetime):
                        pred_copy['timestamp'] = pred_copy['timestamp'].isoformat()
                    serializable_predictions.append(pred_copy)
                
                serializable_history[str(game_id)] = serializable_predictions
            
            with open(history_path, 'w') as f:
                json.dump(serializable_history, f)
            
            self.log(f"Saved prediction history to {history_path}")
        except Exception as e:
            self.log(f"Error saving prediction history: {e}", "ERROR")
    
    def validate_model(self, num_test_games=20):
        """Run validation on historical games to assess model performance."""
        self.log("Running model validation...")
        
        try:
            # Validate that the model is loaded
            if self.predictor.model is None:
                self.predictor.load_model()
            
            # Fetch historical games
            current_date = datetime.now().strftime('%Y-%m-%d')
            response = supabase.table("nba_historical_game_stats")\
                .select("*")\
                .lt("game_date", current_date)\
                .order('game_date', desc=True)\
                .limit(num_test_games).execute()
            
            if not response.data:
                self.log("No historical games found for validation", "WARNING")
                return
            
            # Process historical games
            historical_games = response.data
            validation_results = []
            
            self.log(f"Validating model on {len(historical_games)} historical games")
            
            for game in historical_games:
                # Get actual final scores
                actual_home_score = game['home_score']
                actual_away_score = game['away_score']
                game_id = game['game_id']
                
                # Test prediction from each quarter
                quarter_results = []
                
                for test_quarter in range(1, 5):
                    # Create simulated in-progress game
                    sim_game = {
                        'game_id': game_id,
                        'home_team': game['home_team'],
                        'away_team': game['away_team'],
                        'game_date': game['game_date'],
                        'current_quarter': test_quarter
                    }
                    
                    # Add quarter scores up to test_quarter
                    for q in range(1, 5):
                        q_key_home = f'home_q{q}'
                        q_key_away = f'away_q{q}'
                        
                        if q <= test_quarter:
                            sim_game[q_key_home] = game.get(q_key_home, 0)
                            sim_game[q_key_away] = game.get(q_key_away, 0)
                        else:
                            sim_game[q_key_home] = 0
                            sim_game[q_key_away] = 0
                    
                    # Calculate current score
                    sim_game['home_score'] = sum([sim_game.get(f'home_q{q}', 0) or 0 for q in range(1, test_quarter+1)])
                    sim_game['away_score'] = sum([sim_game.get(f'away_q{q}', 0) or 0 for q in range(1, test_quarter+1)])
                    
                    # Make prediction
                    features_df = self.predictor.prepare_features(pd.DataFrame([sim_game]))
                    if features_df.empty:
                        continue
                        
                    prediction_result = self.predictor.predict_game_scores(features_df)
                    if prediction_result.empty:
                        continue
                    
                    pred_row = prediction_result.iloc[0]
                    predicted_home = pred_row['predicted_home_final']
                    predicted_away = pred_row['predicted_away_final']
                    
                    # Calculate errors
                    home_error = predicted_home - actual_home_score
                    away_error = predicted_away - actual_away_score
                    
                    quarter_results.append({
                        'quarter': test_quarter,
                        'actual_home': actual_home_score,
                        'actual_away': actual_away_score,
                        'predicted_home': predicted_home,
                        'predicted_away': predicted_away,
                        'home_error': home_error,
                        'away_error': away_error,
                        'absolute_error': (abs(home_error) + abs(away_error)) / 2
                    })
                
                validation_results.append({
                    'game_id': game_id,
                    'home_team': game['home_team'],
                    'away_team': game['away_team'],
                    'quarter_results': quarter_results
                })
            
            # Save validation results
            self.validation_results = validation_results
            
            # Calculate and display summary metrics
            self.display_validation_results()
            
            # Save to file
            validation_path = os.path.join("data", self.validation_file)
            with open(validation_path, 'w') as f:
                json.dump(validation_results, f)
            
            self.log(f"Validation results saved to {validation_path}")
            
            return validation_results
            
        except Exception as e:
            self.log(f"Error during model validation: {e}", "ERROR")
            traceback.print_exc()
            return None
    
    def display_validation_results(self):
        """Display and visualize validation results."""
        if not self.validation_results:
            self.log("No validation results to display", "WARNING")
            return
        
        # Extract metrics by quarter
        quarter_errors = {1: [], 2: [], 3: [], 4: []}
        
        for game in self.validation_results:
            for qr in game['quarter_results']:
                quarter = qr['quarter']
                quarter_errors[quarter].append(qr['absolute_error'])
        
        # Calculate average errors by quarter
        avg_errors = {}
        for quarter, errors in quarter_errors.items():
            if errors:
                avg_errors[quarter] = sum(errors) / len(errors)
            else:
                avg_errors[quarter] = None
        
        # Display results
        print("\nModel Validation Results:")
        print("=" * 60)
        print("Average Prediction Error by Quarter:")
        for quarter, avg_error in avg_errors.items():
            if avg_error is not None:
                print(f"  Quarter {quarter}: {avg_error:.2f} points")
        
        # Visualization
        plt.figure(figsize=(10, 6))
        quarters = list(avg_errors.keys())
        errors = [avg_errors[q] for q in quarters if avg_errors[q] is not None]
        
        bars = plt.bar(quarters, errors, color='salmon')
        plt.xlabel('Quarter')
        plt.ylabel('Average Absolute Error (points)')
        plt.title('Prediction Error by Quarter')
        plt.xticks(quarters)
        plt.grid(axis='y', alpha=0.3)
        
        # Add values on top of bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                    f'{height:.2f}', ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()
    
    def start_monitoring(self, max_updates=None, run_validation=True):
        """Start the monitoring process."""
        self.running = True
        self.update_count = 0
        
        # Run model validation if requested
        if run_validation:
            self.validate_model()
        
        # Ensure the model is loaded
        if self.predictor.model is None:
            self.predictor.load_model()
        
        self.log(f"Starting monitoring with {self.update_interval} second updates")
        
        # Begin update loop
        while self.running:
            try:
                self.update_count += 1
                self.last_update_time = datetime.now()
                
                self.log(f"Update #{self.update_count} at {self.last_update_time.strftime('%H:%M:%S')}")
                
                # Clear previous output
                clear_output(wait=True)
                
                # Run full prediction pipeline
                prediction_results = self.predictor.run_full_prediction_pipeline()
                
                # Update prediction history
                if prediction_results is not None and not prediction_results.empty:
                    for _, pred in prediction_results.iterrows():
                        game_id = pred['game_id']
                        if game_id not in self.prediction_history:
                            self.prediction_history[game_id] = []
                        
                        # Add timestamp if missing
                        if 'timestamp' not in pred:
                            pred['timestamp'] = self.last_update_time
                        
                        # Store prediction
                        self.prediction_history[game_id].append(pred.to_dict())
                
                # Save history if auto-save is enabled
                if self.auto_save:
                    self.save_history()
                
                # Check if we've reached max updates
                if max_updates and self.update_count >= max_updates:
                    self.log(f"Reached maximum updates ({max_updates})")
                    break
                
                # Wait for next update
                if self.running and (max_updates is None or self.update_count < max_updates):
                    self.log(f"Waiting {self.update_interval} seconds until next update...")
                    time.sleep(self.update_interval)
                
            except KeyboardInterrupt:
                self.log("Monitoring stopped by user")
                self.running = False
                break
            except Exception as e:
                self.log(f"Error during monitoring update: {e}", "ERROR")
                traceback.print_exc()
                
                # Don't stop on errors, just wait for next update
                time.sleep(self.update_interval)
        
        self.running = False
        self.log("Monitoring stopped")
    
    def start_async_monitoring(self, max_updates=None, run_validation=True):
        """Start monitoring in a background thread."""
        if self.update_thread is not None and self.update_thread.is_alive():
            self.log("Monitoring already running")
            return
        
        self.update_thread = threading.Thread(
            target=self.start_monitoring,
            args=(max_updates, run_validation)
        )
        self.update_thread.daemon = True
        self.update_thread.start()
        self.log("Monitoring started in background thread")
    
    def stop_monitoring(self):
        """Stop the monitoring process."""
        self.running = False
        self.log("Stopping monitoring...")
        
        if self.update_thread is not None and self.update_thread.is_alive():
            self.update_thread.join(timeout=5)
            if self.update_thread.is_alive():
                self.log("Monitoring thread is still running", "WARNING")
        
        self.save_history()
        self.log("Monitoring stopped and history saved")
    
    def get_prediction_accuracy(self):
        """Calculate prediction accuracy for completed games."""
        if not self.prediction_history:
            self.log("No prediction history available")
            return None
        
        accuracy_results = []
        
        for game_id, predictions in self.prediction_history.items():
            # Check if this is a completed game with actual results
            if predictions and 'actual_home_final' in predictions[-1]:
                final_pred = predictions[-1]
                actual_home = final_pred['actual_home_final']
                actual_away = final_pred['actual_away_final']
                
                # Calculate accuracy for each prediction in the game
                game_accuracy = []
                
                for i, pred in enumerate(predictions):
                    predicted_home = pred['predicted_home_final']
                    predicted_away = pred['predicted_away_final']
                    quarter = pred['current_quarter']
                    
                    # Calculate errors
                    home_error = abs(predicted_home - actual_home)
                    away_error = abs(predicted_away - actual_away)
                    avg_error = (home_error + away_error) / 2
                    
                    # Check if prediction got winner correct
                    actual_winner = 'home' if actual_home > actual_away else 'away'
                    predicted_winner = 'home' if predicted_home > predicted_away else 'away'
                    winner_correct = (actual_winner == predicted_winner)
                    
                    game_accuracy.append({
                        'prediction_number': i + 1,
                        'quarter': quarter,
                        'home_error': home_error,
                        'away_error': away_error,
                        'avg_error': avg_error,
                        'winner_correct': winner_correct
                    })
                
                accuracy_results.append({
                    'game_id': game_id,
                    'home_team': predictions[0]['home_team'],
                    'away_team': predictions[0]['away_team'],
                    'predictions': game_accuracy
                })
        
        # Create summary
        if accuracy_results:
            self.display_accuracy_results(accuracy_results)
        
        return accuracy_results
    
    def display_accuracy_results(self, accuracy_results):
        """Display accuracy results."""
        if not accuracy_results:
            return
        
        print("\nPrediction Accuracy Analysis")
        print("=" * 60)
        
        # Overall stats
        total_predictions = 0
        correct_winner_count = 0
        error_by_quarter = {0: [], 1: [], 2: [], 3: [], 4: []}
        
        for game in accuracy_results:
            for pred in game['predictions']:
                total_predictions += 1
                if pred['winner_correct']:
                    correct_winner_count += 1
                
                quarter = pred['quarter']
                error_by_quarter[quarter].append(pred['avg_error'])
        
        # Calculate winner accuracy
        winner_accuracy = (correct_winner_count / total_predictions) if total_predictions > 0 else 0
        print(f"Overall Winner Prediction Accuracy: {winner_accuracy:.1%}")
        
        # Calculate average error by quarter
        print("\nAverage Error by Quarter:")
        for quarter, errors in error_by_quarter.items():
            if errors:
                avg_error = sum(errors) / len(errors)
                print(f"  Quarter {quarter}: {avg_error:.2f} points")
        
        # Visualize
        plt.figure(figsize=(12, 8))
        
        # First subplot - error by quarter
        plt.subplot(2, 1, 1)
        quarters = []
        avg_errors = []
        
        for quarter, errors in error_by_quarter.items():
            if errors:
                quarters.append(quarter)
                avg_errors.append(sum(errors) / len(errors))
        
        bars = plt.bar(quarters, avg_errors, color='lightblue')
        plt.xlabel('Quarter')
        plt.ylabel('Average Error (points)')
        plt.title('Prediction Error by Quarter')
        plt.xticks(quarters)
        plt.grid(axis='y', alpha=0.3)
        
        # Add values on top of bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                    f'{height:.2f}', ha='center', va='bottom')
        
        # Second subplot - winner accuracy by quarter
        plt.subplot(2, 1, 2)
        quarters = []
        accuracies = []
        
        for quarter in range(5):
            correct = 0
            total = 0
            
            for game in accuracy_results:
                for pred in game['predictions']:
                    if pred['quarter'] == quarter:
                        total += 1
                        if pred['winner_correct']:
                            correct += 1
            
            if total > 0:
                quarters.append(quarter)
                accuracies.append(correct / total)
        
        bars = plt.bar(quarters, accuracies, color='lightgreen')
        plt.xlabel('Quarter')
        plt.ylabel('Accuracy')
        plt.title('Winner Prediction Accuracy by Quarter')
        plt.xticks(quarters)
        plt.ylim(0, 1)
        plt.grid(axis='y', alpha=0.3)
        
        # Add values on top of bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                    f'{height:.2%}', ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()

def match_games_to_schedule(live_games_df, schedule_df):
    """Improved function to match games to schedule with better validation"""
    if live_games_df.empty or schedule_df.empty:
        return live_games_df
    
    # Make a copy to avoid modifying the original
    matched_games = []
    
    # Process the schedule first
    for _, scheduled_game in schedule_df.iterrows():
        schedule_game_id = scheduled_game['game_id']
        schedule_home = scheduled_game['home_team']
        schedule_away = scheduled_game['away_team']
        schedule_date = scheduled_game['game_date']
        
        # Try to find this scheduled game in live data
        matching_live = live_games_df[
            (live_games_df['home_team'] == schedule_home) & 
            (live_games_df['away_team'] == schedule_away)
        ]
        
        if not matching_live.empty:
            # Found match - use the scheduled game ID
            live_game = matching_live.iloc[0].to_dict()
            live_game['game_id'] = schedule_game_id
            live_game['verified'] = True
            matched_games.append(live_game)
        else:
            # No live data for this scheduled game - create template
            template = {
                'game_id': schedule_game_id,
                'home_team': schedule_home,
                'away_team': schedule_away,
                'game_date': schedule_date,
                'home_q1': 0, 'home_q2': 0, 'home_q3': 0, 'home_q4': 0,
                'away_q1': 0, 'away_q2': 0, 'away_q3': 0, 'away_q4': 0,
                'home_score': 0, 'away_score': 0,
                'current_quarter': 0,
                'verified': True,
                'game_status': 'SCHEDULED'
            }
            matched_games.append(template)
    
    return pd.DataFrame(matched_games)

# For testing, we'll need to define a basic NBAGamePredictor class
class NBAGamePredictor:
    def __init__(self):
        self.model = None
        print("[{}] INFO: Initializing NBA Game Predictor".format(datetime.now().strftime("%Y-%m-%d %H:%M:%S")))
        
    def load_model(self):
        # Placeholder for model loading
        self.model = "mock_model"
        
    def prepare_features(self, df):
        # Placeholder returning the input
        return df
        
    def predict_game_scores(self, features_df):
        # Mock prediction
        if features_df.empty:
            return pd.DataFrame()
            
        result = features_df.copy()
        result['predicted_home_final'] = 100
        result['predicted_away_final'] = 95
        return result
        
    def run_full_prediction_pipeline(self):
        # Mock prediction results
        return pd.DataFrame({
            'game_id': [1001, 1002],
            'home_team': ['Boston', 'LA Lakers'],
            'away_team': ['Philadelphia', 'Phoenix'],
            'current_quarter': [2, 3],
            'predicted_home_final': [105.5, 110.3],
            'predicted_away_final': [98.2, 103.7],
            'current_quarter': [2, 3]
        })

# Start with just testing the class initialization
monitor = NBAGameMonitor(update_interval=30, auto_save=False)
print("Monitor initialized successfully")

# Instead of running actual monitoring, let's just verify the logger works
monitor.log("Test log message")

In [ ]:
# Cell 7C-2: Win Prediction Validation


def analyze_win_prediction_accuracy(predictions_history, actual_results=None):
    """
    Analyze the accuracy of win probability predictions across quarters.
    
    Args:
        predictions_history: Dict of game_id -> list of prediction points
        actual_results: DataFrame with actual game results (optional)
        
    Returns:
        tuple: (accuracy_df, visualization_fig)
    """
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    
    # Prepare results container
    results = []
    
    # Process each game's prediction history
    for game_id, history in predictions_history.items():
        if not history:
            continue
            
        # Get the actual winner if available
        actual_home_win = None
        final_prediction = history[-1]  # Last prediction in history
        
        if actual_results is not None and game_id in actual_results.index:
            actual = actual_results.loc[game_id]
            actual_home_score = actual.get('home_score', 0)
            actual_away_score = actual.get('away_score', 0)
            actual_home_win = actual_home_score > actual_away_score
        
        # Process each prediction point in this game's history
        for i, pred in enumerate(history):
            quarter = pred.get('current_quarter', 0)
            win_prob = pred.get('win_probability', 0.5)
            
            # Predicted winner (home team if win_prob > 0.5)
            predicted_home_win = win_prob > 0.5
            
            # Check if prediction was correct (if we have actual results)
            correct_prediction = None
            if actual_home_win is not None:
                correct_prediction = (predicted_home_win == actual_home_win)
            
            # For later predictions in the same game, calculate change
            win_prob_change = None
            if i > 0:
                prev_win_prob = history[i-1].get('win_probability', 0.5)
                win_prob_change = win_prob - prev_win_prob
            
            # Add result
            results.append({
                'game_id': game_id,
                'quarter': quarter,
                'win_probability': win_prob, 
                'predicted_home_win': predicted_home_win,
                'correct_prediction': correct_prediction,
                'win_prob_change': win_prob_change,
                'prediction_number': i+1
            })
    
    # Convert to DataFrame
    results_df = pd.DataFrame(results)
    
    # Create visualization
    if not results_df.empty:
        fig, axs = plt.subplots(2, 1, figsize=(12, 10))
        
        # Plot 1: Win prediction accuracy by quarter
        if 'correct_prediction' in results_df.columns and not results_df['correct_prediction'].isna().all():
            # Group by quarter and calculate accuracy
            quarter_accuracy = results_df.groupby('quarter')['correct_prediction'].mean()
            
            # Plot
            axs[0].bar(quarter_accuracy.index, quarter_accuracy.values, color='lightgreen')
            axs[0].set_xlabel('Quarter')
            axs[0].set_ylabel('Prediction Accuracy')
            axs[0].set_title('Win Prediction Accuracy by Quarter')
            axs[0].set_ylim(0, 1)
            axs[0].set_xticks(quarter_accuracy.index)
            axs[0].grid(axis='y', alpha=0.3)
            
            # Add percentage labels
            for i, v in enumerate(quarter_accuracy):
                axs[0].text(i, v + 0.02, f'{v:.1%}', ha='center')
        else:
            axs[0].text(0.5, 0.5, "No accuracy data available", 
                     ha='center', va='center', transform=axs[0].transAxes)
        
        # Plot 2: Win probability confidence distribution
        axs[1].hist(results_df['win_probability'], bins=10, range=(0, 1), alpha=0.7)
        axs[1].set_xlabel('Win Probability')
        axs[1].set_ylabel('Frequency')
        axs[1].set_title('Distribution of Win Probability Predictions')
        axs[1].set_xlim(0, 1)
        axs[1].grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
    else:
        fig = plt.figure()
        plt.text(0.5, 0.5, "No prediction data available", ha='center', va='center')
    
    return results_df, fig

# Create sample prediction history for testing
sample_history = {
    1001: [
        {'current_quarter': 1, 'win_probability': 0.55, 'home_team': 'Lakers', 'away_team': 'Celtics'},
        {'current_quarter': 2, 'win_probability': 0.62, 'home_team': 'Lakers', 'away_team': 'Celtics'},
        {'current_quarter': 3, 'win_probability': 0.45, 'home_team': 'Lakers', 'away_team': 'Celtics'},
        {'current_quarter': 4, 'win_probability': 0.58, 'home_team': 'Lakers', 'away_team': 'Celtics'}
    ],
    1002: [
        {'current_quarter': 1, 'win_probability': 0.48, 'home_team': 'Warriors', 'away_team': 'Bucks'},
        {'current_quarter': 2, 'win_probability': 0.72, 'home_team': 'Warriors', 'away_team': 'Bucks'},
        {'current_quarter': 3, 'win_probability': 0.85, 'home_team': 'Warriors', 'away_team': 'Bucks'},
        {'current_quarter': 4, 'win_probability': 0.91, 'home_team': 'Warriors', 'away_team': 'Bucks'}
    ]
}

# Sample actual results for validation
sample_results = pd.DataFrame([
    {'game_id': 1001, 'home_score': 105, 'away_score': 108},  # Home team lost
    {'game_id': 1002, 'home_score': 120, 'away_score': 105}   # Home team won
]).set_index('game_id')

# Test the win prediction accuracy analysis
win_predictions_df, win_predictions_fig = analyze_win_prediction_accuracy(sample_history, sample_results)
plt.figure(win_predictions_fig.number)
plt.show()

print("Win Prediction Metrics:")
display(win_predictions_df)

In [ ]:
# Cell 7D: Enhanced NBAGameMonitor with Integrated Pipeline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import time
from IPython.display import clear_output
import json
import os
import traceback
import threading

class EnhancedNBAGameMonitor:
    """
    Enhanced monitoring system with improved data flow pipeline and state management.
    Connects live game fetching, prediction, and visualization in a continuous loop.
    """
    
    def __init__(self, update_intervals=None, auto_save=True, debug=True, visualization_mode='dashboard'):
        """
        Initialize the enhanced monitor with configurable settings.
        
        Args:
            update_intervals: Dict with update frequencies in seconds for different game states
            auto_save: Whether to auto-save prediction history (default: True)
            debug: Enable detailed debug logging (default: True)
            visualization_mode: Display mode - 'dashboard' or 'individual' (default: dashboard)
        """
        self.predictor = NBAGamePredictor()
        
        # Configure dynamic update intervals based on game state
        self.update_intervals = update_intervals or {
            'pregame': 300,      # 5 mins for pregame
            'live': 60,          # 1 min for live games
            'close_game': 30,    # 30 seconds for close games (margin < 5)
            'finished': 600,     # 10 mins for finished games
            'default': 120       # 2 mins default
        }
        
        self.auto_save = auto_save
        self.debug = debug
        self.visualization_mode = visualization_mode
        self.running = False
        self.update_thread = None
        self.prediction_history = {}
        self.update_count = 0
        self.last_update_time = None
        self.game_states = {}  # Track state of each game
        self.alerts = []       # Store system alerts
        
        # File paths for saving data
        self.data_dir = "data"
        self.history_file = os.path.join(self.data_dir, "prediction_history.json")
        self.state_file = os.path.join(self.data_dir, "game_states.json")
        
        # Create directories if needed
        os.makedirs(self.data_dir, exist_ok=True)
        
        # Load previous history if it exists
        self.load_history()
        self.load_game_states()
    
    def log(self, message, level="INFO"):
        """Log message with timestamp."""
        if not self.debug and level == "DEBUG":
            return
            
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"[{timestamp}] {level}: {message}")
    
    def load_history(self):
        """Load prediction history from file if available."""
        if os.path.exists(self.history_file):
            try:
                with open(self.history_file, 'r') as f:
                    raw_history = json.load(f)
                
                # Convert string timestamps back to datetime objects
                for game_id, predictions in raw_history.items():
                    for pred in predictions:
                        if 'timestamp' in pred and isinstance(pred['timestamp'], str):
                            pred['timestamp'] = datetime.fromisoformat(pred['timestamp'])
                
                self.prediction_history = raw_history
                self.log(f"Loaded prediction history for {len(self.prediction_history)} games")
            except Exception as e:
                self.log(f"Error loading prediction history: {e}", "ERROR")
    
    def load_game_states(self):
        """Load game states from file if available."""
        if os.path.exists(self.state_file):
            try:
                with open(self.state_file, 'r') as f:
                    self.game_states = json.load(f)
                self.log(f"Loaded states for {len(self.game_states)} games")
            except Exception as e:
                self.log(f"Error loading game states: {e}", "ERROR")
    
    def save_history(self):
        """Save prediction history to file."""
        if not self.prediction_history:
            return
            
        try:
            # Convert datetime objects to ISO format strings for JSON serialization
            serializable_history = {}
            
            for game_id, predictions in self.prediction_history.items():
                serializable_predictions = []
                for pred in predictions:
                    pred_copy = pred.copy()
                    if 'timestamp' in pred_copy and isinstance(pred_copy['timestamp'], datetime):
                        pred_copy['timestamp'] = pred_copy['timestamp'].isoformat()
                    serializable_predictions.append(pred_copy)
                
                serializable_history[str(game_id)] = serializable_predictions
            
            with open(self.history_file, 'w') as f:
                json.dump(serializable_history, f)
            
            self.log(f"Saved prediction history to {self.history_file}")
        except Exception as e:
            self.log(f"Error saving prediction history: {e}", "ERROR")
    
    def save_game_states(self):
        """Save game states to file."""
        if not self.game_states:
            return
            
        try:
            with open(self.state_file, 'w') as f:
                json.dump(self.game_states, f)
            self.log(f"Saved game states to {self.state_file}")
        except Exception as e:
            self.log(f"Error saving game states: {e}", "ERROR")
    
    def prune_old_data(self, days_to_keep=7):
        """Remove old data to prevent unlimited growth."""
        if not self.prediction_history:
            return
        
        cutoff_date = datetime.now() - timedelta(days=days_to_keep)
        pruned_history = {}
        
        for game_id, predictions in self.prediction_history.items():
            # Check the most recent prediction timestamp
            if predictions and 'timestamp' in predictions[-1]:
                last_timestamp = predictions[-1]['timestamp']
                if isinstance(last_timestamp, str):
                    last_timestamp = datetime.fromisoformat(last_timestamp)
                
                if last_timestamp > cutoff_date:
                    pruned_history[game_id] = predictions
        
        pruned_count = len(self.prediction_history) - len(pruned_history)
        if pruned_count > 0:
            self.prediction_history = pruned_history
            self.log(f"Pruned {pruned_count} old games from history")
    
    def determine_update_interval(self, game_data):
        """
        Determine appropriate update interval based on game state.
        
        Args:
            game_data: Dict or DataFrame row with game information
            
        Returns:
            int: Update interval in seconds
        """
        game_status = game_data.get('game_status', '').lower()
        
        if game_status == 'finished':
            return self.update_intervals['finished']
        
        if game_status == 'pregame' or game_data.get('current_quarter', 0) == 0:
            return self.update_intervals['pregame']
        
        # For live games, check if it's a close game
        if game_status == 'live':
            home_score = float(game_data.get('home_score', 0))
            away_score = float(game_data.get('away_score', 0))
            margin = abs(home_score - away_score)
            
            if margin < 5 and game_data.get('current_quarter', 0) >= 3:
                return self.update_intervals['close_game']
            
            return self.update_intervals['live']
        
        return self.update_intervals['default']
    
    def update_game_state(self, game_id, game_data):
        """
        Update tracked state for a specific game.
        
        Args:
            game_id: Unique identifier for the game
            game_data: Dict with current game state
        """
        str_game_id = str(game_id)
        
        if str_game_id not in self.game_states:
            self.game_states[str_game_id] = {
                'first_seen': datetime.now().isoformat(),
                'updates': 0,
                'status_history': []
            }
        
        # Update the state
        current_state = {
            'timestamp': datetime.now().isoformat(),
            'current_quarter': game_data.get('current_quarter', 0),
            'home_score': game_data.get('home_score', 0),
            'away_score': game_data.get('away_score', 0),
            'status': game_data.get('game_status', 'unknown')
        }
        
        # Check for state changes
        if self.game_states[str_game_id]['status_history']:
            last_state = self.game_states[str_game_id]['status_history'][-1]
            
            # Detect quarter changes
            if last_state['current_quarter'] != current_state['current_quarter']:
                self.log(f"Game {game_id} quarter changed: {last_state['current_quarter']} → {current_state['current_quarter']}")
            
            # Detect status changes
            if last_state['status'] != current_state['status']:
                self.log(f"Game {game_id} status changed: {last_state['status']} → {current_state['status']}")
        
        # Update the history
        self.game_states[str_game_id]['status_history'].append(current_state)
        self.game_states[str_game_id]['updates'] += 1
        self.game_states[str_game_id]['last_update'] = datetime.now().isoformat()
    
    def detect_anomalies(self, game_id, current_prediction, previous_predictions):
        """
        Detect anomalies in predictions and raise alerts if necessary.
        
        Args:
            game_id: Game identifier
            current_prediction: Current prediction data
            previous_predictions: List of previous predictions
            
        Returns:
            list: Detected anomalies (if any)
        """
        anomalies = []
        
        if not previous_predictions:
            return anomalies
        
        # Get the most recent previous prediction
        prev_prediction = previous_predictions[-1]
        
        # Check for large score prediction swings
        prev_home = float(prev_prediction.get('predicted_home_score', 0))
        prev_away = float(prev_prediction.get('predicted_away_score', 0))
        
        curr_home = float(current_prediction.get('predicted_home_score', 0))
        curr_away = float(current_prediction.get('predicted_away_score', 0))
        
        home_change = abs(curr_home - prev_home)
        away_change = abs(curr_away - prev_away)
        
        # Threshold for large prediction change (points)
        threshold = 8
        
        if home_change > threshold or away_change > threshold:
            anomaly = {
                'type': 'large_prediction_swing',
                'game_id': game_id,
                'timestamp': datetime.now().isoformat(),
                'details': {
                    'home_change': home_change,
                    'away_change': away_change,
                    'previous': {'home': prev_home, 'away': prev_away},
                    'current': {'home': curr_home, 'away': curr_away}
                }
            }
            anomalies.append(anomaly)
            
            self.log(f"Anomaly detected in game {game_id}: Large prediction swing " +
                    f"(Home: {prev_home:.1f}→{curr_home:.1f}, Away: {prev_away:.1f}→{curr_away:.1f})", 
                    "WARNING")
            
            # Add to global alerts
            self.alerts.append(anomaly)
        
        return anomalies
    
    def integrated_pipeline_step(self):
        """Execute a single step of the integrated data flow pipeline."""
        try:
            self.update_count += 1
            self.last_update_time = datetime.now()
            
            self.log(f"Pipeline update #{self.update_count} at {self.last_update_time.strftime('%H:%M:%S')}")
            
            # 1. Fetch live games
            live_games_df = self.predictor.fetch_live_games_pacific()
            
            if live_games_df.empty:
                self.log("No games available for processing")
                return self.update_intervals['default']
            
            # 2. Generate predictions for all games
            prediction_results = self.predictor.run_full_prediction_pipeline(live_games_df)
            
            # Default update interval
            next_update_interval = self.update_intervals['default']
            
            if prediction_results is not None and not prediction_results.empty:
                # Process each game prediction
                for _, game_pred in prediction_results.iterrows():
                    game_id = game_pred['game_id']
                    str_game_id = str(game_id)
                    
                    # Determine update interval
                    game_interval = self.determine_update_interval(game_pred)
                    next_update_interval = min(next_update_interval, game_interval)
                    
                    # Update game state
                    self.update_game_state(game_id, game_pred)
                    
                    # Update prediction history
                    if str_game_id not in self.prediction_history:
                        self.prediction_history[str_game_id] = []
                    
                    # Check for anomalies
                    anomalies = self.detect_anomalies(
                        game_id, 
                        game_pred.to_dict(), 
                        self.prediction_history[str_game_id]
                    )
                    
                    # Add prediction to history
                    prediction_data = game_pred.to_dict()
                    prediction_data['timestamp'] = self.last_update_time
                    if anomalies:
                        prediction_data['anomalies'] = anomalies
                    
                    self.prediction_history[str_game_id].append(prediction_data)
                
                # 3. Visualize the results
                if self.visualization_mode == 'dashboard':
                    self.display_dashboard(prediction_results)
                else:
                    self.predictor.visualize_predictions(prediction_results)
                
                # Save data if auto-save is enabled
                if self.auto_save:
                    self.save_history()
                    self.save_game_states()
            
            return next_update_interval
            
        except Exception as e:
            self.log(f"Error in pipeline step: {e}", "ERROR")
            traceback.print_exc()
            return self.update_intervals['default']
    
    def display_dashboard(self, prediction_results):
        """
        Display an integrated dashboard with all visualizations.
        
        Args:
            prediction_results: DataFrame with prediction results
        """
        # Clear previous output
        clear_output(wait=True)
        
        # Display header
        print(f"NBA Score Prediction Dashboard - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("=" * 80)
        
        # Display update info
        print(f"Update #{self.update_count} | Next update in: varies by game state")
        
        # Group games by status
        status_groups = {}
        for _, game in prediction_results.iterrows():
            status = game.get('game_status', 'unknown')
            if status not in status_groups:
                status_groups[status] = []
            status_groups[status].append(game)
        
        # Display game summary by status
        for status, games in status_groups.items():
            print(f"\n{status.upper()} GAMES ({len(games)}):")
            print("-" * 80)
            
            for game in games:
                game_id = game['game_id']
                home_team = game['home_team']
                away_team = game['away_team']
                current_quarter = game.get('current_quarter', 0)
                
                # Format based on game state
                if status.lower() == 'pregame':
                    # Display predicted outcome
                    print(f"{home_team} vs {away_team}")
                    if 'predicted_home_score' in game:
                        home_pred = game['predicted_home_score']
                        away_pred = game['predicted_away_score']
                        print(f"  Prediction: {home_team} {home_pred:.1f} - {away_pred:.1f} {away_team}")
                        
                        # Add win probability if available
                        if 'win_probability' in game:
                            win_prob = game['win_probability'] * 100
                            fav_team = home_team if win_prob > 50 else away_team
                            fav_prob = win_prob if win_prob > 50 else 100 - win_prob
                            print(f"  Favorite: {fav_team} ({fav_prob:.1f}%)")
                
                elif status.lower() in ['live', 'in progress', 'ongoing']:
                    # Display current score and prediction
                    home_score = game.get('home_score', 0)
                    away_score = game.get('away_score', 0)
                    quarter_text = f"Q{current_quarter}" if current_quarter > 0 else "Pre-game"
                    
                    print(f"{home_team} vs {away_team} - {quarter_text}")
                    print(f"  Current: {home_team} {home_score} - {away_score} {away_team}")
                    
                    if 'predicted_home_score' in game:
                        home_pred = game['predicted_home_score']
                        away_pred = game['predicted_away_score']
                        print(f"  Prediction: {home_team} {home_pred:.1f} - {away_pred:.1f} {away_team}")
                        
                        # Add win probability
                        if 'win_probability' in game:
                            win_prob = game['win_probability'] * 100
                            print(f"  Win Probability: {home_team} {win_prob:.1f}%")
                    
                    # Determine update frequency
                    update_interval = self.determine_update_interval(game)
                    print(f"  Update Frequency: Every {update_interval} seconds")
                
                else:  # Finished or other states
                    # Display final score
                    home_score = game.get('home_score', 0)
                    away_score = game.get('away_score', 0)
                    
                    print(f"{home_team} vs {away_team} - FINAL")
                    print(f"  Final Score: {home_team} {home_score} - {away_score} {away_team}")
                
                # Add separator between games
                print("-" * 40)
        
        # Display any alerts
        if self.alerts:
            print("\nALERTS:")
            print("-" * 80)
            
            # Show most recent 5 alerts
            for alert in self.alerts[-5:]:
                alert_type = alert.get('type', 'unknown')
                game_id = alert.get('game_id', 'unknown')
                timestamp = alert.get('timestamp', '')
                
                if isinstance(timestamp, str):
                    try:
                        timestamp = datetime.fromisoformat(timestamp)
                        timestamp_str = timestamp.strftime('%H:%M:%S')
                    except:
                        timestamp_str = timestamp
                else:
                    timestamp_str = str(timestamp)
                
                print(f"[{timestamp_str}] {alert_type.upper()} in game {game_id}")
                
                # Add details based on alert type
                if alert_type == 'large_prediction_swing' and 'details' in alert:
                    details = alert['details']
                    prev = details.get('previous', {})
                    curr = details.get('current', {})
                    print(f"  Prediction changed significantly:")
                    print(f"  Home: {prev.get('home', 0):.1f} → {curr.get('home', 0):.1f}")
                    print(f"  Away: {prev.get('away', 0):.1f} → {curr.get('away', 0):.1f}")
                
                print("-" * 40)
        
        # Create visualizations
        # We could add additional visualizations here
        # For now, just use the standard visualization function
        self.predictor.visualize_predictions(prediction_results)
    
    def start_monitoring(self, max_updates=None):
        """Start the monitoring process with dynamic update intervals."""
        self.running = True
        self.update_count = 0
        
        self.log(f"Starting enhanced monitoring with dynamic update intervals")
        
        # Begin update loop
        while self.running:
            try:
                # Execute pipeline step and get next update interval
                next_interval = self.integrated_pipeline_step()
                
                # Check if we've reached max updates
                if max_updates and self.update_count >= max_updates:
                    self.log(f"Reached maximum updates ({max_updates})")
                    break
                
                # Wait for next update
                if self.running and (max_updates is None or self.update_count < max_updates):
                    self.log(f"Waiting {next_interval} seconds until next update...")
                    time.sleep(next_interval)
                
            except KeyboardInterrupt:
                self.log("Monitoring stopped by user")
                self.running = False
                break
            except Exception as e:
                self.log(f"Error during monitoring: {e}", "ERROR")
                traceback.print_exc()
                time.sleep(self.update_intervals['default'])
        
        self.running = False
        self.save_history()
        self.save_game_states()
        self.log("Monitoring stopped")
    
    def start_async_monitoring(self, max_updates=None):
        """Start monitoring in a background thread."""
        if self.update_thread is not None and self.update_thread.is_alive():
            self.log("Monitoring already running")
            return
        
        self.update_thread = threading.Thread(
            target=self.start_monitoring,
            args=(max_updates,)
        )
        self.update_thread.daemon = True
        self.update_thread.start()
        self.log("Monitoring started in background thread")
    
    def stop_monitoring(self):
        """Stop the monitoring process."""
        self.running = False
        self.log("Stopping monitoring...")
        
        if self.update_thread is not None and self.update_thread.is_alive():
            self.update_thread.join(timeout=5)
            if self.update_thread.is_alive():
                self.log("Monitoring thread is still running", "WARNING")
        
        self.save_history()
        self.save_game_states()
        self.log("Monitoring stopped and data saved")

In [ ]:
# Cell 7E: Comprehensive Performance Metrics Logging and Visualization

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import json
import os
import traceback

class PredictionPerformanceTracker:
    """
    System for tracking, analyzing, and visualizing prediction performance over time.
    """
    
    def __init__(self, data_dir="data", metrics_file="prediction_metrics.json"):
        """
        Initialize the performance tracker.
        
        Args:
            data_dir: Directory for storing performance data
            metrics_file: Filename for the metrics JSON file
        """
        self.data_dir = data_dir
        self.metrics_file = os.path.join(data_dir, metrics_file)
        self.metrics_history = {}
        
        # Create data directory if it doesn't exist
        os.makedirs(data_dir, exist_ok=True)
        
        # Load existing metrics if available
        self.load_metrics()
    
    def load_metrics(self):
        """Load performance metrics from file if available."""
        if os.path.exists(self.metrics_file):
            try:
                with open(self.metrics_file, 'r') as f:
                    self.metrics_history = json.load(f)
                print(f"Loaded performance metrics from {self.metrics_file}")
            except Exception as e:
                print(f"Error loading performance metrics: {e}")
    
    def save_metrics(self):
        """Save performance metrics to file."""
        try:
            with open(self.metrics_file, 'w') as f:
                json.dump(self.metrics_history, f)
            print(f"Saved performance metrics to {self.metrics_file}")
        except Exception as e:
            print(f"Error saving performance metrics: {e}")
    
    def evaluate_prediction(self, game_id, prediction_data, actual_results):
        """
        Evaluate prediction accuracy and record metrics.
        
        Args:
            game_id: Unique identifier for the game
            prediction_data: Dict or Series with prediction information
            actual_results: Dict or Series with actual final results
            
        Returns:
            dict: Calculated performance metrics
        """
        # Convert Series to dict if needed
        if isinstance(prediction_data, pd.Series):
            prediction_data = prediction_data.to_dict()
        
        if isinstance(actual_results, pd.Series):
            actual_results = actual_results.to_dict()
        
        # Extract prediction values
        predicted_home = float(prediction_data.get('predicted_home_score', 0))
        predicted_away = float(prediction_data.get('predicted_away_score', 0))
        
        # Extract actual final scores
        actual_home = float(actual_results.get('home_score', 0))
        actual_away = float(actual_results.get('away_score', 0))
        
        # Calculate error metrics
        home_error = predicted_home - actual_home
        away_error = predicted_away - actual_away
        
        home_abs_error = abs(home_error)
        away_abs_error = abs(away_error)
        
        total_abs_error = home_abs_error + away_abs_error
        avg_abs_error = total_abs_error / 2
        
        home_squared_error = home_error ** 2
        away_squared_error = away_error ** 2
        
        total_squared_error = home_squared_error + away_squared_error
        avg_squared_error = total_squared_error / 2
        
        # Calculate margin error
        predicted_margin = predicted_home - predicted_away
        actual_margin = actual_home - actual_away
        margin_error = predicted_margin - actual_margin
        margin_abs_error = abs(margin_error)
        
        # Check winner prediction accuracy
        predicted_winner = 'home' if predicted_home > predicted_away else 'away'
        actual_winner = 'home' if actual_home > actual_away else 'away'
        winner_correct = predicted_winner == actual_winner
        
        # Check if within confidence interval if available
        within_confidence = None
        if ('home_lower_bound' in prediction_data and 
            'home_upper_bound' in prediction_data and
            'away_lower_bound' in prediction_data and
            'away_upper_bound' in prediction_data):
            
            home_lower = float(prediction_data.get('home_lower_bound', 0))
            home_upper = float(prediction_data.get('home_upper_bound', 0))
            away_lower = float(prediction_data.get('away_lower_bound', 0))
            away_upper = float(prediction_data.get('away_upper_bound', 0))
            
            home_within = home_lower <= actual_home <= home_upper
            away_within = away_lower <= actual_away <= away_upper
            
            within_confidence = home_within and away_within
        
        # Extract contextual data
        game_state = {
            'quarter': prediction_data.get('current_quarter', 0),
            'timestamp': prediction_data.get('timestamp', datetime.now().isoformat()),
            'time_remaining': prediction_data.get('time_remaining_mins', 0),
            'score_diff': prediction_data.get('score_differential', 0),
            'home_team': prediction_data.get('home_team', 'Home'),
            'away_team': prediction_data.get('away_team', 'Away')
        }
        
        # Compile metrics
        metrics = {
            'game_id': game_id,
            'game_state': game_state,
            'predictions': {
                'home': predicted_home,
                'away': predicted_away,
                'margin': predicted_margin
            },
            'actuals': {
                'home': actual_home,
                'away': actual_away,
                'margin': actual_margin
            },
            'errors': {
                'home_error': home_error,
                'away_error': away_error,
                'home_abs_error': home_abs_error,
                'away_abs_error': away_abs_error,
                'total_abs_error': total_abs_error,
                'avg_abs_error': avg_abs_error,
                'margin_error': margin_error,
                'margin_abs_error': margin_abs_error
            },
            'squared_errors': {
                'home_squared_error': home_squared_error,
                'away_squared_error': away_squared_error,
                'total_squared_error': total_squared_error,
                'avg_squared_error': avg_squared_error
            },
            'accuracy': {
                'winner_correct': winner_correct,
                'within_confidence': within_confidence
            }
        }
        
        # Store metrics in history
        game_id_str = str(game_id)
        if game_id_str not in self.metrics_history:
            self.metrics_history[game_id_str] = []
        
        self.metrics_history[game_id_str].append(metrics)
        
        # Save updated metrics
        self.save_metrics()
        
        return metrics
    
    def evaluate_game_history(self, game_id, prediction_history, actual_results):
        """
        Evaluate all predictions in a game's history.
        
        Args:
            game_id: Unique identifier for the game
            prediction_history: List of prediction data points
            actual_results: Dict or Series with actual final results
            
        Returns:
            list: Performance metrics for each prediction
        """
        metrics_list = []
        
        for pred in prediction_history:
            metrics = self.evaluate_prediction(game_id, pred, actual_results)
            metrics_list.append(metrics)
        
        return metrics_list
    
    def get_aggregated_metrics(self, filter_days=None):
        """
        Calculate aggregated performance metrics.
        
        Args:
            filter_days: Only include metrics from the last N days (optional)
            
        Returns:
            dict: Aggregated performance metrics
        """
        all_metrics = []
        
        # Flatten metrics from all games
        for game_id, metrics_list in self.metrics_history.items():
            for metrics in metrics_list:
                # Apply date filter if specified
                if filter_days is not None:
                    timestamp = metrics['game_state'].get('timestamp')
                    if timestamp:
                        if isinstance(timestamp, str):
                            try:
                                timestamp = datetime.fromisoformat(timestamp)
                            except:
                                # Skip if can't parse timestamp
                                continue
                        
                        days_ago = (datetime.now() - timestamp).days
                        if days_ago > filter_days:
                            continue
                
                all_metrics.append(metrics)
        
        if not all_metrics:
            return {}
        
        # Initialize aggregation containers
        agg_metrics = {
            'count': len(all_metrics),
            'mae': {},
            'rmse': {},
            'winner_accuracy': {},
            'confidence_accuracy': {}
        }
        
        # Separate metrics by quarter for more detailed analysis
        metrics_by_quarter = {}
        for metrics in all_metrics:
            quarter = metrics['game_state'].get('quarter', 0)
            if quarter not in metrics_by_quarter:
                metrics_by_quarter[quarter] = []
            metrics_by_quarter[quarter].append(metrics)
        
        # Overall metrics (all quarters)
        avg_abs_errors = [m['errors']['avg_abs_error'] for m in all_metrics]
        avg_squared_errors = [m['squared_errors']['avg_squared_error'] for m in all_metrics]
        winner_correct = [m['accuracy']['winner_correct'] for m in all_metrics if m['accuracy']['winner_correct'] is not None]
        within_confidence = [m['accuracy']['within_confidence'] for m in all_metrics if m['accuracy']['within_confidence'] is not None]
        
        agg_metrics['mae']['overall'] = sum(avg_abs_errors) / len(avg_abs_errors) if avg_abs_errors else None
        agg_metrics['rmse']['overall'] = np.sqrt(sum(avg_squared_errors) / len(avg_squared_errors)) if avg_squared_errors else None
        agg_metrics['winner_accuracy']['overall'] = sum(winner_correct) / len(winner_correct) if winner_correct else None
        agg_metrics['confidence_accuracy']['overall'] = sum(within_confidence) / len(within_confidence) if within_confidence else None
        
        # Metrics by quarter
        for quarter, quarter_metrics in metrics_by_quarter.items():
            q_avg_abs_errors = [m['errors']['avg_abs_error'] for m in quarter_metrics]
            q_avg_squared_errors = [m['squared_errors']['avg_squared_error'] for m in quarter_metrics]
            q_winner_correct = [m['accuracy']['winner_correct'] for m in quarter_metrics if m['accuracy']['winner_correct'] is not None]
            q_within_confidence = [m['accuracy']['within_confidence'] for m in quarter_metrics if m['accuracy']['within_confidence'] is not None]
            
            agg_metrics['mae'][f'q{quarter}'] = sum(q_avg_abs_errors) / len(q_avg_abs_errors) if q_avg_abs_errors else None
            agg_metrics['rmse'][f'q{quarter}'] = np.sqrt(sum(q_avg_squared_errors) / len(q_avg_squared_errors)) if q_avg_squared_errors else None
            agg_metrics['winner_accuracy'][f'q{quarter}'] = sum(q_winner_correct) / len(q_winner_correct) if q_winner_correct else None
            agg_metrics['confidence_accuracy'][f'q{quarter}'] = sum(q_within_confidence) / len(q_within_confidence) if q_within_confidence else None
            
            agg_metrics[f'count_q{quarter}'] = len(quarter_metrics)
        
        return agg_metrics
    
    def visualize_performance(self, filter_days=None):
        """
        Create visualizations of prediction performance.
        
        Args:
            filter_days: Only include metrics from the last N days (optional)
            
        Returns:
            matplotlib Figure with performance visualizations
        """
        aggregated = self.get_aggregated_metrics(filter_days)
        
        if not aggregated:
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.text(0.5, 0.5, "No performance data available", 
                   ha='center', va='center', fontsize=14)
            return fig
        
        # Create visualization
        fig, axs = plt.subplots(2, 2, figsize=(16, 12))
        
        # 1. Error by Quarter (MAE and RMSE)
        quarters = sorted([k for k in aggregated['mae'].keys() if k.startswith('q')])
        
        if quarters:
            mae_values = [aggregated['mae'][q] for q in quarters]
            rmse_values = [aggregated['rmse'][q] for q in quarters]
            
            ax1 = axs[0, 0]
            x = range(len(quarters))
            
            ax1.bar(x, mae_values, width=0.4, label='MAE', color='skyblue')
            ax1.bar([i + 0.4 for i in x], rmse_values, width=0.4, label='RMSE', color='salmon')
            
            ax1.set_xticks([i + 0.2 for i in x])
            ax1.set_xticklabels(quarters)
            ax1.set_ylabel('Points')
            ax1.set_title('Prediction Error by Quarter')
            ax1.legend()
            ax1.grid(axis='y', alpha=0.3)
            
            # Add value labels
            for i, v in enumerate(mae_values):
                ax1.text(i, v + 0.5, f'{v:.2f}', ha='center', va='bottom', fontsize=9)
            
            for i, v in enumerate(rmse_values):
                ax1.text(i + 0.4, v + 0.5, f'{v:.2f}', ha='center', va='bottom', fontsize=9)
        
        # 2. Winner Prediction Accuracy by Quarter
        quarters = sorted([k for k in aggregated['winner_accuracy'].keys() if k.startswith('q')])
        
        if quarters:
            accuracy_values = [aggregated['winner_accuracy'][q] * 100 for q in quarters]
            
            ax2 = axs[0, 1]
            x = range(len(quarters))
            
            ax2.bar(x, accuracy_values, width=0.6, color='lightgreen')
            
            ax2.set_xticks(x)
            ax2.set_xticklabels(quarters)
            ax2.set_ylabel('Accuracy (%)')
            ax2.set_title('Winner Prediction Accuracy by Quarter')
            ax2.set_ylim(0, 100)
            ax2.grid(axis='y', alpha=0.3)
            
            # Add value labels
            for i, v in enumerate(accuracy_values):
                ax2.text(i, v + 2, f'{v:.1f}%', ha='center', va='bottom', fontsize=9)
        
        # 3. Confidence Interval Accuracy
        quarters = sorted([k for k in aggregated['confidence_accuracy'].keys() if k.startswith('q') and aggregated['confidence_accuracy'][k] is not None])
        
        if quarters:
            conf_values = [aggregated['confidence_accuracy'][q] * 100 for q in quarters]
            
            ax3 = axs[1, 0]
            x = range(len(quarters))
            
            ax3.bar(x, conf_values, width=0.6, color='plum')
            
            ax3.set_xticks(x)
            ax3.set_xticklabels(quarters)
            ax3.set_ylabel('Within Interval (%)')
            ax3.set_title('Confidence Interval Accuracy by Quarter')
            ax3.set_ylim(0, 100)
            ax3.grid(axis='y', alpha=0.3)
            
            # Add value labels
            for i, v in enumerate(conf_values):
                ax3.text(i, v + 2, f'{v:.1f}%', ha='center', va='bottom', fontsize=9)
        
        # 4. Sample Size by Quarter
        quarters = [f'q{i}' for i in range(5)]
        counts = [aggregated.get(f'count_{q}', 0) for q in quarters]
        
        ax4 = axs[1, 1]
        x = range(len(quarters))
        
        ax4.bar(x, counts, width=0.6, color='lightblue')
        
        ax4.set_xticks(x)
        ax4.set_xticklabels(quarters)
        ax4.set_ylabel('Count')
        ax4.set_title('Sample Size by Quarter')
        ax4.grid(axis='y', alpha=0.3)
        
        # Add value labels
        for i, v in enumerate(counts):
            ax4.text(i, v + 0.5, str(v), ha='center', va='bottom', fontsize=9)
        
        # Add overall title with filter information
        if filter_days is not None:
            title = f"Prediction Performance Metrics (Last {filter_days} Days)"
        else:
            title = "Prediction Performance Metrics (All Time)"
        
        title += f" - {aggregated['count']} Total Predictions"
        fig.suptitle(title, fontsize=16)
        
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        return fig
    
    def generate_performance_report(self, filter_days=None):
        """
        Generate a text-based performance report.
        
        Args:
            filter_days: Only include metrics from the last N days (optional)
            
        Returns:
            str: Formatted performance report
        """
        aggregated = self.get_aggregated_metrics(filter_days)
        
        if not aggregated:
            return "No performance data available for reporting."
        
        # Create report
        if filter_days is not None:
            report = f"Performance Report (Last {filter_days} Days)\n"
        else:
            report = "Performance Report (All Time)\n"
        
        report += "=" * 60 + "\n\n"
        
        # Overall metrics
        report += f"Total Predictions: {aggregated['count']}\n\n"
        
        # Error metrics
        report += "Error Metrics by Quarter:\n"
        report += "-" * 40 + "\n"
        report += f"{'Quarter':<10}{'MAE':>10}{'RMSE':>10}{'Count':>10}\n"
        report += "-" * 40 + "\n"
        
        # Overall row
        overall_mae = aggregated['mae'].get('overall')
        overall_rmse = aggregated['rmse'].get('overall')
        overall_mae_str = f"{overall_mae:.2f}" if overall_mae is not None else "N/A"
        overall_rmse_str = f"{overall_rmse:.2f}" if overall_rmse is not None else "N/A"
        report += f"{'Overall':<10}{overall_mae_str:>10}{overall_rmse_str:>10}{aggregated['count']:>10}\n"
        
        # Quarter-specific rows
        for i in range(5):
            quarter_key = f'q{i}'
            mae = aggregated['mae'].get(quarter_key)
            rmse = aggregated['rmse'].get(quarter_key)
            count = aggregated.get(f'count_{quarter_key}', 0)
            
            mae_str = f"{mae:.2f}" if mae is not None else "N/A"
            rmse_str = f"{rmse:.2f}" if rmse is not None else "N/A"
            
            report += f"{quarter_key:<10}{mae_str:>10}{rmse_str:>10}{count:>10}\n"
        
        report += "\n"
        
        # Accuracy metrics
        report += "Prediction Accuracy by Quarter:\n"
        report += "-" * 50 + "\n"
        report += f"{'Quarter':<10}{'Winner %':>12}{'Confidence %':>15}{'Count':>10}\n"
        report += "-" * 50 + "\n"
        
        # Overall row
        overall_winner = aggregated['winner_accuracy'].get('overall')
        overall_conf = aggregated['confidence_accuracy'].get('overall')
        
        winner_str = f"{overall_winner*100:.1f}%" if overall_winner is not None else "N/A"
        conf_str = f"{overall_conf*100:.1f}%" if overall_conf is not None else "N/A"
        
        report += f"{'Overall':<10}{winner_str:>12}{conf_str:>15}{aggregated['count']:>10}\n"
        
        # Quarter-specific rows
        for i in range(5):
            quarter_key = f'q{i}'
            winner_acc = aggregated['winner_accuracy'].get(quarter_key)
            conf_acc = aggregated['confidence_accuracy'].get(quarter_key)
            count = aggregated.get(f'count_{quarter_key}', 0)
            
            winner_str = f"{winner_acc*100:.1f}%" if winner_acc is not None else "N/A"
            conf_str = f"{conf_acc*100:.1f}%" if conf_acc is not None else "N/A"
            
            report += f"{quarter_key:<10}{winner_str:>12}{conf_str:>15}{count:>10}\n"
        
        report += "\n"
        report += "=" * 60 + "\n"
        return report

In [ ]:
# Cell 7F: Comprehensive Validation Framework

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import traceback

class NBAModelValidator:
    """
    Comprehensive validation framework for NBA prediction models
    using historical game data to simulate live prediction scenarios.
    """
    
    def __init__(self, model_path=None, feature_list=None):
        """
        Initialize the validator.
        
        Args:
            model_path: Path to the model file (optional)
            feature_list: List of features expected by the model (optional)
        """
        self.model = None
        self.feature_list = feature_list
        
        # Load model if path provided
        if model_path:
            self.load_model(model_path)
    
    def load_model(self, model_path):
        """Load model from file."""
        try:
            self.model = joblib.load(model_path)
            print(f"Model loaded from {model_path}")
            
            # Try to determine feature list if not provided
            if self.feature_list is None and hasattr(self.model, 'feature_importances_'):
                n_features = len(self.model.feature_importances_)
                
                if n_features > 8:
                    self.feature_list = [
                        'home_q1', 'home_q2', 'home_q3', 'home_q4',
                        'score_ratio', 'prev_matchup_diff',
                        'rest_days_home', 'rest_days_away', 'rest_advantage',
                        'is_back_to_back_home', 'is_back_to_back_away',
                        'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
                    ]
                else:
                    self.feature_list = [
                        'home_q1', 'home_q2', 'home_q3', 'home_q4',
                        'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
                    ]
                
                print(f"Using detected feature list with {n_features} features")
        except Exception as e:
            print(f"Error loading model: {e}")
            traceback.print_exc()
    
    def fetch_validation_games(self, db_engine, sample_size=50, min_date=None, max_date=None, specific_teams=None):
        """
        Fetch historical games for validation.
        
        Args:
            db_engine: SQLAlchemy database engine
            sample_size: Maximum number of games to fetch
            min_date: Minimum game date (optional)
            max_date: Maximum game date (optional)
            specific_teams: List of teams to include (optional)
            
        Returns:
            DataFrame with historical games
        """
        try:
            # Build query conditionally
            query = "SELECT * FROM nba_historical_game_stats WHERE 1=1"
            
            if min_date:
                query += f" AND game_date >= '{min_date}'"
            
            if max_date:
                query += f" AND game_date <= '{max_date}'"
            
            if specific_teams:
                team_list = "', '".join(specific_teams)
                query += f" AND (home_team IN ('{team_list}') OR away_team IN ('{team_list}'))"
            
            query += f" ORDER BY game_date DESC LIMIT {sample_size}"
            
            # Fetch data
            games_df = pd.read_sql(query, db_engine)
            print(f"Fetched {len(games_df)} games for validation")
            
            return games_df
        except Exception as e:
            print(f"Error fetching validation games: {e}")
            traceback.print_exc()
            return pd.DataFrame()
    
    def prepare_game_features(self, game, current_quarter, feature_generator=None):
        """
        Prepare features for a game at a specific quarter.
        
        Args:
            game: Game data (Series or dict)
            current_quarter: Quarter to simulate (0-4)
            feature_generator: Optional NBAFeatureGenerator instance
            
        Returns:
            DataFrame with prepared features
        """
        # Convert to DataFrame if not already
        if isinstance(game, pd.Series):
            game_df = pd.DataFrame([game.to_dict()])
        elif isinstance(game, dict):
            game_df = pd.DataFrame([game])
        else:
            game_df = pd.DataFrame([game])
        
        # Simulate in-progress game by zeroing out future quarters
        for q in range(1, 5):
            if q > current_quarter:
                game_df[f'home_q{q}'] = 0
                game_df[f'away_q{q}'] = 0
        
        # Update current score based on quarters that have been played
        game_df['home_score'] = sum(float(game_df.iloc[0].get(f'home_q{q}', 0) or 0) for q in range(1, current_quarter+1))
        game_df['away_score'] = sum(float(game_df.iloc[0].get(f'away_q{q}', 0) or 0) for q in range(1, current_quarter+1))
        
        # Set current quarter
        game_df['current_quarter'] = current_quarter
        
        # If we have a feature generator, use it
        if feature_generator:
            # Generate all features
            features_df = feature_generator.generate_all_features(game_df)
            return features_df
        
        # Otherwise do basic feature preparation
        # Calculate score ratio
        home_score = float(game_df['home_score'])
        away_score = float(game_df['away_score'])
        total_score = home_score + away_score
        game_df['score_ratio'] = (home_score / total_score) if total_score > 0 else 0.5
        
        # Set dummy values for any missing features
        if self.feature_list:
            for feature in self.feature_list:
                if feature not in game_df.columns:
                    game_df[feature] = 0.0
        
        return game_df
    
    def simulate_game_predictions(self, game, ensemble=None):
        """
        Simulate predictions for a game as it progresses quarter by quarter.
        
        Args:
            game: Game data (Series or dict)
            ensemble: Optional AdaptiveEnsemble instance for weighted predictions
            
        Returns:
            dict: Simulated predictions for each quarter
        """
        if self.model is None:
            print("No model loaded for prediction")
            return {}
        
        # Initialize results
        results = {'game_id': game.get('game_id', 'unknown')}
        
        # Get actual final scores for comparison
        actual_home_score = float(game.get('home_score', 0))
        actual_away_score = float(game.get('away_score', 0))
        results['actual'] = {'home': actual_home_score, 'away': actual_away_score}
        
        # Get team names
        results['home_team'] = game.get('home_team', 'Home')
        results['away_team'] = game.get('away_team', 'Away')
        
        # Simulate predictions for each quarter
        quarter_predictions = {}
        
        for quarter in range(5):  # 0-4 (0 = pre-game)
            # Prepare features for this quarter
            features_df = self.prepare_game_features(game, quarter)
            
            if features_df.empty:
                continue
            
            try:
                # Extract features for prediction
                if self.feature_list:
                    X = features_df[self.feature_list]
                else:
                    # Fallback - use all numeric columns
                    X = features_df.select_dtypes(include=['number'])
                
                # Make prediction
                home_pred = self.model.predict(X)[0]
                
                # Estimate away score
                # Simple heuristic: assume similar home advantage as actual
                home_advantage = actual_home_score - actual_away_score
                away_pred = home_pred - (home_advantage * 0.8)  # Discount the advantage slightly
                
                # Calculate confidence (higher in later quarters)
                confidence = 0.5 + (quarter * 0.1)
                
                # Store prediction
                quarter_predictions[f'q{quarter}'] = {
                    'home': home_pred,
                    'away': away_pred,
                    'confidence': confidence
                }
                
                # Apply ensemble weighting if available
                if ensemble:
                    # Get quarter-specific prediction if applicable
                    if quarter > 0 and f'q{quarter}_model' in ensemble.models:
                        quarter_model = ensemble.models[f'q{quarter}_model']
                        quarter_pred = quarter_model.predict(X)[0]
                    else:
                        # Fallback - use similar prediction with slight adjustment
                        quarter_pred = home_pred * (1 + (random.random() - 0.5) * 0.1)
                    
                    # Get ensemble weights and prediction
                    ensemble_pred, weights, _ = ensemble.predict(
                        home_pred, quarter_pred, quarter, 
                        {
                            'score_differential': features_df['score_differential'].iloc[0],
                            'momentum': features_df['cumulative_momentum'].iloc[0] if 'cumulative_momentum' in features_df.columns else 0
                        }
                    )
                    
                    # Add ensemble prediction
                    quarter_predictions[f'q{quarter}']['ensemble'] = ensemble_pred
                    quarter_predictions[f'q{quarter}']['weights'] = weights
            
            except Exception as e:
                print(f"Error simulating quarter {quarter}: {e}")
                traceback.print_exc()
        
        results['predictions'] = quarter_predictions
        return results
    
    def batch_validate(self, games_df, feature_generator=None, ensemble=None):
        """
        Run batch validation on multiple games.
        
        Args:
            games_df: DataFrame with games to validate
            feature_generator: Optional NBAFeatureGenerator instance
            ensemble: Optional AdaptiveEnsemble instance
            
        Returns:
            list: Validation results for each game
        """
        if self.model is None:
            print("No model loaded for validation")
            return []
        
        validation_results = []
        
        for _, game in games_df.iterrows():
            result = self.simulate_game_predictions(game, ensemble)
            validation_results.append(result)
        
        return validation_results
    
    def calculate_metrics(self, validation_results):
        """
        Calculate accuracy metrics from validation results.
        
        Args:
            validation_results: List of validation result dicts
            
        Returns:
            DataFrame with accuracy metrics by quarter
        """
        metrics = []
        
        for result in validation_results:
            actual_home = result['actual']['home']
            actual_away = result['actual']['away']
            
            for quarter, preds in result['predictions'].items():
                pred_home = preds['home']
                pred_away = preds['away']
                
                # Calculate errors
                home_error = pred_home - actual_home
                away_error = pred_away - actual_away
                home_abs_error = abs(home_error)
                away_abs_error = abs(away_error)
                
                # Calculate margin error
                actual_margin = actual_home - actual_away
                pred_margin = pred_home - pred_away
                margin_error = pred_margin - actual_margin
                
                # Check winner prediction
                actual_winner = 'home' if actual_home > actual_away else 'away'
                pred_winner = 'home' if pred_home > pred_away else 'away'
                correct_winner = actual_winner == pred_winner
                
                # Store metrics
                metrics.append({
                    'game_id': result['game_id'],
                    'quarter': quarter,
                    'home_team': result['home_team'],
                    'away_team': result['away_team'],
                    'actual_home': actual_home,
                    'actual_away': actual_away,
                    'pred_home': pred_home,
                    'pred_away': pred_away,
                    'home_error': home_error,
                    'away_error': away_error,
                    'home_abs_error': home_abs_error,
                    'away_abs_error': away_abs_error,
                    'avg_abs_error': (home_abs_error + away_abs_error) / 2,
                    'margin_error': margin_error,
                    'actual_winner': actual_winner,
                    'pred_winner': pred_winner,
                    'correct_winner': correct_winner
                })
        
        return pd.DataFrame(metrics)
    
    def visualize_validation_results(self, metrics_df):
        """
        Create visualizations of validation results.
        
        Args:
            metrics_df: DataFrame with validation metrics
            
        Returns:
            matplotlib Figure with visualizations
        """
        if metrics_df.empty:
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.text(0.5, 0.5, "No validation data available", 
                   ha='center', va='center', fontsize=14)
            return fig
        
        # Create figure
        fig, axs = plt.subplots(2, 2, figsize=(16, 12))
        
        # 1. Error by Quarter
        quarter_metrics = metrics_df.groupby('quarter').agg({
            'home_abs_error': 'mean',
            'away_abs_error': 'mean',
            'avg_abs_error': 'mean',
            'correct_winner': 'mean',
            'game_id': 'count'
        }).reset_index()
        
        quarter_metrics = quarter_metrics.sort_values('quarter')
        
        ax1 = axs[0, 0]
        x = range(len(quarter_metrics))
        
        ax1.bar(x, quarter_metrics['avg_abs_error'], color='lightblue')
        ax1.set_xticks(x)
        ax1.set_xticklabels(quarter_metrics['quarter'])
        ax1.set_ylabel('Average Absolute Error (points)')
        ax1.set_title('Prediction Error by Quarter')
        ax1.grid(axis='y', alpha=0.3)
        
        # Add value labels
        for i, v in enumerate(quarter_metrics['avg_abs_error']):
            ax1.text(i, v + 0.5, f'{v:.2f}', ha='center')
        
        # 2. Winner Prediction Accuracy
        ax2 = axs[0, 1]
        accuracy = quarter_metrics['correct_winner'] * 100
        
        ax2.bar(x, accuracy, color='lightgreen')
        ax2.set_xticks(x)
        ax2.set_xticklabels(quarter_metrics['quarter'])
        ax2.set_ylabel('Accuracy (%)')
        ax2.set_title('Winner Prediction Accuracy by Quarter')
        ax2.set_ylim(0, 100)
        ax2.grid(axis='y', alpha=0.3)
        
        # Add value labels
        for i, v in enumerate(accuracy):
            ax2.text(i, v + 2, f'{v:.1f}%', ha='center')
        
        # 3. Error Distribution
        ax3 = axs[1, 0]
        ax3.hist(metrics_df['avg_abs_error'], bins=20, color='skyblue', alpha=0.7)
        ax3.set_xlabel('Average Absolute Error')
        ax3.set_ylabel('Frequency')
        ax3.set_title('Error Distribution')
        ax3.grid(axis='y', alpha=0.3)
        
        # 4. Sample Size
        ax4 = axs[1, 1]
        samples = quarter_metrics['game_id']
        
        ax4.bar(x, samples, color='salmon')
        ax4.set_xticks(x)
        ax4.set_xticklabels(quarter_metrics['quarter'])
        ax4.set_ylabel('Count')
        ax4.set_title('Sample Size by Quarter')
        ax4.grid(axis='y', alpha=0.3)
        
        # Add value labels
        for i, v in enumerate(samples):
            ax4.text(i, v + 1, str(int(v)), ha='center')
        
        # Add overall title
        fig.suptitle('Model Validation Results', fontsize=16)
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        
        return fig

In [ ]:
# Cell 7G: Continuous Learning System

import pandas as pd
import numpy as np
import joblib
import os
from datetime import datetime, timedelta
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import traceback

class NBAModelTrainer:
    """
    Continuous learning system for NBA prediction models.
    Automatically retrains models with new game data and 
    implements model performance monitoring.
    """
    
    def __init__(self, models_dir="models", data_dir="data"):
        """
        Initialize the model trainer.
        
        Args:
            models_dir: Directory for storing trained models
            data_dir: Directory for storing training data and metrics
        """
        self.models_dir = models_dir
        self.data_dir = data_dir
        self.model_metrics = {}
        self.feature_sets = {
            'standard': [
                'home_q1', 'home_q2', 'home_q3', 'home_q4',
                'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
            ],
            'enhanced': [
                'home_q1', 'home_q2', 'home_q3', 'home_q4',
                'score_ratio', 'prev_matchup_diff',
                'rest_days_home', 'rest_days_away', 'rest_advantage',
                'is_back_to_back_home', 'is_back_to_back_away',
                'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
            ]
        }
        
        # Create directories if they don't exist
        os.makedirs(models_dir, exist_ok=True)
        os.makedirs(data_dir, exist_ok=True)
        
        # Load existing metrics if available
        self.metrics_file = os.path.join(data_dir, "model_metrics.joblib")
        if os.path.exists(self.metrics_file):
            try:
                self.model_metrics = joblib.load(self.metrics_file)
                print(f"Loaded model metrics from {self.metrics_file}")
            except Exception as e:
                print(f"Error loading model metrics: {e}")
    
    def save_metrics(self):
        """Save model metrics to file."""
        try:
            joblib.dump(self.model_metrics, self.metrics_file)
            print(f"Saved model metrics to {self.metrics_file}")
        except Exception as e:
            print(f"Error saving model metrics: {e}")
    
    def fetch_training_data(self, db_engine, days_lookback=180, min_games=500):
        """
        Fetch historical game data for training.
        
        Args:
            db_engine: SQLAlchemy database engine
            days_lookback: Number of days to look back for data
            min_games: Minimum number of games to fetch
            
        Returns:
            DataFrame with training data
        """
        try:
            # Start with recent games
            lookback_date = (datetime.now() - timedelta(days=days_lookback)).strftime('%Y-%m-%d')
            
            query = f"""
            SELECT * FROM nba_historical_game_stats 
            WHERE game_date >= '{lookback_date}'
            ORDER BY game_date DESC
            """
            
            df = pd.read_sql(query, db_engine)
            print(f"Fetched {len(df)} games since {lookback_date}")
            
            # If we don't have enough games, fetch more
            if len(df) < min_games:
                additional_needed = min_games - len(df)
                
                additional_query = f"""
                SELECT * FROM nba_historical_game_stats 
                WHERE game_date < '{lookback_date}'
                ORDER BY game_date DESC
                LIMIT {additional_needed}
                """
                
                additional_df = pd.read_sql(additional_query, db_engine)
                print(f"Fetched {len(additional_df)} additional games")
                
                # Combine datasets
                df = pd.concat([df, additional_df], ignore_index=True)
            
            return df
        except Exception as e:
            print(f"Error fetching training data: {e}")
            traceback.print_exc()
            return pd.DataFrame()
    
    def prepare_features(self, df, feature_generator=None, feature_set='enhanced'):
        """
        Prepare features for training.
        
        Args:
            df: DataFrame with raw game data
            feature_generator: Optional NBAFeatureGenerator instance
            feature_set: Which feature set to use ('standard' or 'enhanced')
            
        Returns:
            DataFrame with prepared features
        """
        # If we have a feature generator, use it
        if feature_generator:
            features_df = feature_generator.generate_all_features(df)
            print(f"Generated features using feature generator: {len(features_df)} rows")
            return features_df
        
        # Otherwise do basic feature preparation
        print("No feature generator provided. Using basic feature preparation.")
        features_df = df.copy()
        
        # Ensure all required columns exist with default values
        for col in self.feature_sets[feature_set]:
            if col not in features_df.columns:
                features_df[col] = 0.0
        
        return features_df
    
    def train_model(self, features_df, target_col='home_score', feature_set='enhanced', model_params=None):
        """
        Train a new prediction model.
        
        Args:
            features_df: DataFrame with prepared features
            target_col: Target column to predict
            feature_set: Which feature set to use ('standard' or 'enhanced')
            model_params: Dictionary with model parameters (or None for defaults)
            
        Returns:
            Trained model object
        """
        # Check if we have data
        if features_df.empty:
            print("No data for training.")
            return None
        
        # Get feature columns
        feature_cols = self.feature_sets.get(feature_set, self.feature_sets['standard'])
        
        # Make sure all feature columns exist
        missing_cols = [col for col in feature_cols if col not in features_df.columns]
        if missing_cols:
            print(f"Warning: Missing feature columns: {missing_cols}")
            feature_cols = [col for col in feature_cols if col in features_df.columns]
        
        # Check if we have the target column
        if target_col not in features_df.columns:
            print(f"Error: Target column '{target_col}' not found.")
            return None
        
        # Prepare X and y
        X = features_df[feature_cols]
        y = features_df[target_col]
        
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        print(f"Training on {len(X_train)} samples, testing on {len(X_test)} samples")
        
        # Set up model params
        if model_params is None:
            model_params = {
                'n_estimators': 200,
                'max_depth': 5,
                'learning_rate': 0.1,
                'subsample': 0.8,
                'random_state': 42
            }
        
        # Create and train model
        model = GradientBoostingRegressor(**model_params)
        
        try:
            # Train the model
            start_time = datetime.now()
            model.fit(X_train, y_train)
            train_time = (datetime.now() - start_time).total_seconds()
            
            # Evaluate the model
            train_mse = mean_squared_error(y_train, model.predict(X_train))
            test_mse = mean_squared_error(y_test, model.predict(X_test))
            r2 = r2_score(y_test, model.predict(X_test))
            
            print(f"Training completed in {train_time:.2f} seconds")
            print(f"Training MSE: {train_mse:.2f}")
            print(f"Test MSE: {test_mse:.2f}")
            print(f"R² Score: {r2:.4f}")
            
            # Store metrics
            metrics = {
                'feature_set': feature_set,
                'train_mse': train_mse,
                'test_mse': test_mse,
                'train_rmse': np.sqrt(train_mse),
                'test_rmse': np.sqrt(test_mse),
                'r2': r2,
                'train_size': len(X_train),
                'test_size': len(X_test),
                'train_time': train_time,
                'timestamp': datetime.now().isoformat()
            }
            
            return model, metrics
            
        except Exception as e:
            print(f"Error training model: {e}")
            traceback.print_exc()
            return None, None
    
    def save_model(self, model, metrics, model_name=None):
        """
        Save trained model and its metrics.
        
        Args:
            model: Trained model object
            metrics: Dict with model metrics
            model_name: Name for the model file (or None for auto-generated)
            
        Returns:
            str: Path to the saved model file
        """
        if model is None:
            return None
        
        # Generate model name if not provided
        if model_name is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            feature_set = metrics.get('feature_set', 'unknown')
            test_rmse = metrics.get('test_rmse', 0)
            model_name = f"score_prediction_{feature_set}_{test_rmse:.2f}_{timestamp}.pkl"
        
        # Save the model
        model_path = os.path.join(self.models_dir, model_name)
        
        try:
            joblib.dump(model, model_path)
            print(f"Model saved to {model_path}")
            
            # Store metrics
            self.model_metrics[model_name] = metrics
            self.save_metrics()
            
            return model_path
        except Exception as e:
            print(f"Error saving model: {e}")
            return None
    
    def check_performance_drift(self, current_metrics, drift_threshold=0.1):
        """
        Check if model performance has drifted significantly.
        
        Args:
            current_metrics: Metrics from current model
            drift_threshold: Threshold for significant drift
            
        Returns:
            bool: True if significant drift detected
        """
        if not self.model_metrics:
            # No previous metrics to compare
            return False
        
        # Find the most recent model with the same feature set
        feature_set = current_metrics.get('feature_set')
        comparable_models = [
            (name, metrics) for name, metrics in self.model_metrics.items()
            if metrics.get('feature_set') == feature_set
        ]
        
        if not comparable_models:
            return False
        
        # Sort by timestamp
        comparable_models.sort(key=lambda x: x[1].get('timestamp', ''), reverse=True)
        
        # Get the most recent model
        recent_model_name, recent_metrics = comparable_models[0]
        
        # Compare RMSE
        recent_rmse = recent_metrics.get('test_rmse', float('inf'))
        current_rmse = current_metrics.get('test_rmse', float('inf'))
        
        # Calculate relative change
        if recent_rmse > 0:
            relative_change = abs(current_rmse - recent_rmse) / recent_rmse
            
            print(f"Performance drift check: Current RMSE = {current_rmse:.2f}, Previous = {recent_rmse:.2f}")
            print(f"Relative change: {relative_change:.2%}")
            
            return relative_change > drift_threshold
        
        return False
    
    def train_and_save_model(self, features_df, target_col='home_score', feature_set='enhanced'):
        """
        Train a new model and save it if it performs well.
        
        Args:
            features_df: DataFrame with prepared features
            target_col: Target column to predict
            feature_set: Which feature set to use
            
        Returns:
            tuple: (model, model_path)
        """
        # Train the model
        model, metrics = self.train_model(features_df, target_col, feature_set)
        
        if model is None or metrics is None:
            return None, None
        
        # Check if model performance has drifted
        drift_detected = self.check_performance_drift(metrics)
        
        # Save the model
        if drift_detected:
            print("Significant performance drift detected! Saving as a new version.")
            model_path = self.save_model(model, metrics)
        else:
            # Use a more generic name for incremental updates
            timestamp = datetime.now().strftime("%Y%m%d")
            model_name = f"score_prediction_{feature_set}_{timestamp}.pkl"
            model_path = self.save_model(model, metrics, model_name)
        
        return model, model_path
    
    def visualize_model_history(self):
        """
        Visualize the history of model performance.
        
        Returns:
            matplotlib Figure with performance history
        """
        if not self.model_metrics:
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.text(0.5, 0.5, "No model history available", 
                   ha='center', va='center', fontsize=14)
            return fig
        
        # Convert metrics to DataFrame
        metrics_data = []
        
        for model_name, metrics in self.model_metrics.items():
            metrics_data.append({
                'model_name': model_name,
                'feature_set': metrics.get('feature_set', 'unknown'),
                'train_rmse': metrics.get('train_rmse', None),
                'test_rmse': metrics.get('test_rmse', None),
                'r2': metrics.get('r2', None),
                'timestamp': metrics.get('timestamp', None),
                'train_size': metrics.get('train_size', 0)
            })
        
        metrics_df = pd.DataFrame(metrics_data)
        
        # Convert timestamp to datetime
        metrics_df['timestamp'] = pd.to_datetime(metrics_df['timestamp'], errors='coerce')
        
        # Sort by timestamp
        metrics_df = metrics_df.sort_values('timestamp')
        
        # Create visualization
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
        
        # Plot RMSE over time
        for feature_set in metrics_df['feature_set'].unique():
            subset = metrics_df[metrics_df['feature_set'] == feature_set]
            
            ax1.plot(subset['timestamp'], subset['test_rmse'], 'o-', 
                    label=f"{feature_set} (Test)")
            ax1.plot(subset['timestamp'], subset['train_rmse'], '--', alpha=0.5,
                    label=f"{feature_set} (Train)")
        
        ax1.set_ylabel('RMSE')
        ax1.set_title('Model Performance Over Time')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Format x-axis with dates
        ax1.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m-%d'))
        plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')
        
        # Plot R² over time
        for feature_set in metrics_df['feature_set'].unique():
            subset = metrics_df[metrics_df['feature_set'] == feature_set]
            
            ax2.plot(subset['timestamp'], subset['r2'], 'o-', 
                    label=feature_set)
        
        ax2.set_ylabel('R² Score')
        ax2.set_xlabel('Training Date')
        ax2.set_title('Model R² Score Over Time')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # Format x-axis with dates
        ax2.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m-%d'))
        plt.setp(ax2.get_xticklabels(), rotation=45, ha='right')
        
        plt.tight_layout()
        return fig

In [ ]:
# Cell 8: Improved Score Prediction Model Evaluation with Time-Based Cross-Validation

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
from datetime import datetime
import time
import traceback
import os
import json

def evaluate_model_with_time_cv(model, X, y, date_column, n_splits=5, visualize=True):
    """
    Evaluate a model using time-based cross-validation.
    
    Args:
        model: The trained model to evaluate
        X: Feature DataFrame (should NOT include datetime columns used for sorting in numeric features)
        y: Target Series
        date_column: Name of the date column for time-based splitting
        n_splits: Number of time-based CV splits
        visualize: Whether to visualize results
        
    Returns:
        Dictionary with evaluation metrics
    """
    # Verify the date column exists
    if date_column not in X.index.names and date_column not in X.columns:
        raise ValueError(f"Date column '{date_column}' not found in data")
    
    # Sort data by date, then remove the date column from X so it doesn't break numeric fitting
    if date_column in X.columns:
        sorted_indices = X[date_column].argsort()
        X_sorted = X.iloc[sorted_indices].copy()
        y_sorted = y.iloc[sorted_indices].copy()
        # Remove the date column from numeric features
        X_sorted.drop(columns=[date_column], inplace=True)
    else:
        # Already indexed by date
        X_sorted = X.copy()
        y_sorted = y.copy()
    
    # Prepare time-based cross-validation
    tscv = TimeSeriesSplit(n_splits=n_splits)
    
    # Track metrics across folds
    train_scores = []
    test_scores = []
    train_rmse = []
    test_rmse = []
    train_mae = []
    test_mae = []
    r2_scores = []
    fold_sizes = []
    
    print(f"\nPerforming {n_splits}-fold time-based cross-validation")
    for i, (train_idx, test_idx) in enumerate(tscv.split(X_sorted)):
        X_train_fold = X_sorted.iloc[train_idx]
        X_test_fold = X_sorted.iloc[test_idx]
        y_train_fold = y_sorted.iloc[train_idx]
        y_test_fold = y_sorted.iloc[test_idx]
        
        fold_sizes.append(len(test_idx))
        
        try:
            # Train model on this fold (if model is not already trained)
            if hasattr(model, 'fit'):
                model.fit(X_train_fold, y_train_fold)
            
            # Predict
            y_train_pred = model.predict(X_train_fold)
            y_test_pred = model.predict(X_test_fold)
            
            # Calculate metrics
            train_mse = mean_squared_error(y_train_fold, y_train_pred)
            test_mse = mean_squared_error(y_test_fold, y_test_pred)
            train_scores.append(train_mse)
            test_scores.append(test_mse)
            
            train_rmse.append(np.sqrt(train_mse))
            test_rmse.append(np.sqrt(test_mse))
            
            train_mae.append(mean_absolute_error(y_train_fold, y_train_pred))
            test_mae.append(mean_absolute_error(y_test_fold, y_test_pred))
            
            r2 = r2_score(y_test_fold, y_test_pred)
            r2_scores.append(r2)
            
            print(f"Fold {i+1}: Train MSE = {train_mse:.2f}, Test MSE = {test_mse:.2f}, "
                  f"Test RMSE = {np.sqrt(test_mse):.2f}, R² = {r2:.4f}")
            
        except Exception as e:
            print(f"Error in fold {i+1}: {e}")
            traceback.print_exc()
    
    if len(train_scores) == 0:
        print("\nAll folds encountered errors. No valid metrics to report.")
        return {}
    
    # Calculate aggregate metrics
    mean_train_mse = np.mean(train_scores)
    mean_test_mse = np.mean(test_scores)
    mean_train_rmse = np.mean(train_rmse)
    mean_test_rmse = np.mean(test_rmse)
    mean_train_mae = np.mean(train_mae)
    mean_test_mae = np.mean(test_mae)
    mean_r2 = np.mean(r2_scores)
    
    # Check for overfitting
    mse_ratio = mean_test_mse / mean_train_mse if mean_train_mse > 0 else float('inf')
    overfitting_detected = mse_ratio > 2.0
    
    print("\nCross-Validation Summary:")
    print(f"Mean Train MSE: {mean_train_mse:.2f}, Mean Test MSE: {mean_test_mse:.2f}")
    print(f"Mean Train RMSE: {mean_train_rmse:.2f}, Mean Test RMSE: {mean_test_rmse:.2f}")
    print(f"Mean Train MAE: {mean_train_mae:.2f}, Mean Test MAE: {mean_test_mae:.2f}")
    print(f"Mean R² Score: {mean_r2:.4f}")
    
    if overfitting_detected:
        print(f"\nWarning: Potential overfitting detected. Test/Train MSE ratio: {mse_ratio:.2f}")
        print("Consider adjusting regularization or model complexity.")
    
    # Visualize results if requested and we have valid folds
    if visualize and len(train_scores) > 0:
        plt.figure(figsize=(14, 7))
        
        plt.subplot(1, 2, 1)
        plt.plot(range(1, len(train_scores) + 1), train_scores, 'o-', label='Training MSE')
        plt.plot(range(1, len(test_scores) + 1), test_scores, 'o-', label='Validation MSE')
        plt.title('Learning Curve: MSE by Time Period')
        plt.xlabel('Fold (Earlier to Later)')
        plt.ylabel('Mean Squared Error')
        plt.grid(True, alpha=0.3)
        plt.legend()
        
        plt.subplot(1, 2, 2)
        try:
            full_preds = model.predict(X_sorted)
            errors = y_sorted - full_preds
            plt.hist(errors, bins=30, alpha=0.7, color='skyblue')
            plt.axvline(x=0, color='red', linestyle='--')
            plt.title('Prediction Error Distribution')
            plt.xlabel('Error (Actual - Predicted)')
            plt.ylabel('Frequency')
            plt.grid(True, alpha=0.3)
        except Exception as e:
            print(f"Error generating full prediction error distribution: {e}")
        
        plt.tight_layout()
        plt.show()
        
        if hasattr(model, 'feature_importances_'):
            feature_names = X_sorted.columns
            importances = model.feature_importances_
            indices = np.argsort(importances)[::-1]
            
            plt.figure(figsize=(10, 6))
            plt.title('Feature Importance')
            plt.barh(range(len(indices)), importances[indices], color='skyblue')
            plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
            plt.xlabel('Relative Importance')
            plt.tight_layout()
            plt.show()
    
    # Return metrics dictionary
    return {
        'train_mse': mean_train_mse,
        'test_mse': mean_test_mse,
        'train_rmse': mean_train_rmse,
        'test_rmse': mean_test_rmse,
        'train_mae': mean_train_mae,
        'test_mae': mean_test_mae,
        'r2': mean_r2,
        'fold_metrics': {
            'train_mse': train_scores,
            'test_mse': test_scores,
            'train_rmse': train_rmse,
            'test_rmse': test_rmse,
            'train_mae': train_mae,
            'test_mae': test_mae,
            'r2': r2_scores,
            'fold_sizes': fold_sizes
        },
        'overfitting_detected': overfitting_detected,
        'mse_ratio': mse_ratio
    }

# Check if we have a model to evaluate
if 'score_model' in globals() and globals()['score_model'] is not None:
    model_to_evaluate = globals()['score_model']
    print("Using existing 'score_model' for evaluation")
elif 'model' in globals() and globals()['model'] is not None:
    model_to_evaluate = globals()['model']
    print("Using existing 'model' for evaluation")
else:
    try:
        model_to_evaluate = joblib.load(config.MODEL_PATH)
        print(f"Loaded model from {config.MODEL_PATH} for evaluation")
    except Exception as e:
        print(f"Error loading model: {e}")
        model_to_evaluate = None

if model_to_evaluate is not None:
    print("Loading historical data for model evaluation...")
    
    try:
        from sqlalchemy import create_engine
        import config
        
        engine = create_engine(config.DATABASE_URL)
        
        query = """
        SELECT * FROM nba_historical_game_stats 
        WHERE game_date >= CURRENT_DATE - INTERVAL '365 days'
        ORDER BY game_date
        """
        
        historical_df = pd.read_sql(query, engine)
        print(f"Loaded {len(historical_df)} historical games for evaluation")
        
        if len(historical_df) >= 100:
            historical_df['game_date'] = pd.to_datetime(historical_df['game_date'])
            
            try:
                from models.features import NBAFeatureGenerator
                feature_generator = NBAFeatureGenerator(debug=False)
                features_df = feature_generator.generate_all_features(historical_df)
                print(f"Generated features for {len(features_df)} games")
            except Exception as e:
                print(f"Error generating features: {e}")
                features_df = historical_df.copy()
                
                basic_features = [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4', 
                    'away_q1', 'away_q2', 'away_q3', 'away_q4'
                ]
                
                features_df['score_ratio'] = features_df['home_score'] / (features_df['home_score'] + features_df['away_score'])
                features_df['score_ratio'] = features_df['score_ratio'].fillna(0.5)
                
                X_cols = basic_features + ['score_ratio', 'game_date']
                features_df = features_df[X_cols]
            
            if hasattr(model_to_evaluate, 'feature_importances_'):
                n_features = len(model_to_evaluate.feature_importances_)
                print(f"Model expects {n_features} features")
                
                available_features = features_df.columns.tolist()
                expected_features = None
                
                if n_features <= 8:
                    expected_features = [
                        'home_q1', 'home_q2', 'home_q3', 'home_q4',
                        'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
                    ]
                else:
                    expected_features = [
                        'home_q1', 'home_q2', 'home_q3', 'home_q4',
                        'score_ratio', 'prev_matchup_diff',
                        'rest_days_home', 'rest_days_away', 'rest_advantage',
                        'is_back_to_back_home', 'is_back_to_back_away',
                        'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
                    ]
                
                missing_features = [f for f in expected_features if f not in available_features]
                if missing_features:
                    print(f"Warning: Missing features: {missing_features}")
                    for feature in missing_features:
                        features_df[feature] = 0.0
                
                features_df['game_date'] = pd.to_datetime(historical_df['game_date'])
                
                X = features_df[expected_features + ['game_date']]
                y = historical_df['home_score']
                
                eval_metrics = evaluate_model_with_time_cv(
                    model_to_evaluate, 
                    X, 
                    y, 
                    date_column='game_date', 
                    n_splits=5, 
                    visualize=True
                )
                
                globals()['eval_metrics'] = eval_metrics
                
                try:
                    metrics_dir = "metrics"
                    os.makedirs(metrics_dir, exist_ok=True)
                    
                    # Prepare a version of metrics with only JSON-serializable types
                    save_metrics = {k: v for k, v in eval_metrics.items() if k != 'fold_metrics'}
                    save_metrics['timestamp'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                    
                    metrics_file = os.path.join(metrics_dir, "model_evaluation.json")
                    with open(metrics_file, 'w') as f:
                        json.dump(save_metrics, f, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
                    
                    print(f"Saved evaluation metrics to {metrics_file}")
                except Exception as e:
                    print(f"Error saving metrics: {e}")
            else:
                print("Model doesn't have feature_importances_ attribute. Cannot determine expected features.")
        else:
            print("Not enough historical data for effective evaluation.")
    except Exception as e:
        print(f"Error during model evaluation: {e}")
        traceback.print_exc()
else:
    print("No model available for evaluation.")


In [ ]:
# Cell 9: Enhanced Prediction Evolution Visualization with Plotly

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from datetime import datetime
import math

def calculate_confidence_intervals(prediction, quarter, score_differential=None, momentum=None):
    """
    Calculate confidence intervals based on game state.
    
    Args:
        prediction: Predicted score value
        quarter: Current quarter (0-4)
        score_differential: Points difference between teams (optional)
        momentum: Momentum value between -1 and 1 (optional)
        
    Returns:
        tuple: (lower_bound, upper_bound)
    """
    # Base confidence width based on quarter
    base_width = {
        0: 15.0,  # Pre-game: widest interval
        1: 12.0,  # Q1: still wide
        2: 8.0,   # Q2: narrower
        3: 5.0,   # Q3: getting more confident
        4: 3.0    # Q4: narrowest interval
    }.get(quarter, 10.0)
    
    # Adjust based on score differential if provided
    differential_factor = 1.0
    if score_differential is not None:
        # Close games (within 5 points) have wider intervals
        if abs(score_differential) < 5:
            differential_factor = 1.2
        # Blowouts (15+ points) have narrower intervals
        elif abs(score_differential) > 15:
            differential_factor = 0.8
    
    # Adjust based on momentum if provided
    momentum_factor = 1.0
    if momentum is not None:
        # High momentum (either direction) indicates less predictability
        if abs(momentum) > 0.6:
            momentum_factor = 1.2
    
    # Calculate interval width with all factors
    interval_width = base_width * differential_factor * momentum_factor
    
    # Return bounds
    lower_bound = prediction - interval_width
    upper_bound = prediction + interval_width
    
    return lower_bound, upper_bound

def create_prediction_evolution_chart(prediction_history, selected_game_id=None, max_games=3):
    """
    Create interactive prediction evolution visualization with Plotly.
    
    Args:
        prediction_history: Dictionary of game predictions over time
        selected_game_id: ID of game to highlight (optional)
        max_games: Maximum number of games to display
        
    Returns:
        plotly.graph_objects.Figure
    """
    # Handle empty history
    if not prediction_history:
        fig = go.Figure()
        fig.add_annotation(
            text="No prediction history available",
            showarrow=False,
            font=dict(size=16),
            xref="paper", yref="paper",
            x=0.5, y=0.5
        )
        return fig
    
    # Determine which games to plot
    if selected_game_id and selected_game_id in prediction_history:
        # Just plot the selected game
        games_to_plot = {selected_game_id: prediction_history[selected_game_id]}
    else:
        # Get the most recent/active games up to max_games
        games_to_plot = {}
        # Sort games by recency of last update
        sorted_games = sorted(
            prediction_history.items(),
            key=lambda x: x[1][-1].get('timestamp', datetime.now()) if x[1] else datetime.now(),
            reverse=True
        )
        for game_id, history in sorted_games[:max_games]:
            games_to_plot[game_id] = history
    
    # Handle zero games case
    if not games_to_plot:
        fig = go.Figure()
        fig.add_annotation(
            text="No suitable games found in prediction history",
            showarrow=False,
            font=dict(size=16),
            xref="paper", yref="paper",
            x=0.5, y=0.5
        )
        return fig
    
    # Determine subplot layout based on number of games
    num_games = len(games_to_plot)
    rows = min(num_games, 2)  # Max 2 rows
    cols = math.ceil(num_games / rows)
    
    # Create subplot figure
    fig = make_subplots(
        rows=rows, cols=cols,
        subplot_titles=[f"{history[0].get('home_team', 'Home')} vs {history[0].get('away_team', 'Away')}" 
                        if history else f"Game {game_id}" 
                        for game_id, history in games_to_plot.items()],
        specs=[[{"secondary_y": True} for _ in range(cols)] for _ in range(rows)]
    )
    
    # Plot each game
    idx = 0
    for game_id, history in games_to_plot.items():
        if not history:
            continue
        
        # Calculate row and column for this game
        row = (idx // cols) + 1
        col = (idx % cols) + 1
        
        # Extract data
        update_indices = list(range(len(history)))
        quarters = [point.get('current_quarter', 0) for point in history]
        home_scores = [point.get('predicted_home_score', 0) for point in history]
        away_scores = [point.get('predicted_away_score', 0) for point in history]
        win_probs = [point.get('win_probability', 0.5) for point in history]
        
        # Calculate confidence intervals for home and away predictions
        home_lower = []
        home_upper = []
        away_lower = []
        away_upper = []
        
        for i, point in enumerate(history):
            quarter = point.get('current_quarter', 0)
            score_diff = (point.get('current_home_score', 0) or 0) - (point.get('current_away_score', 0) or 0)
            momentum = point.get('momentum_shift', 0) or point.get('cumulative_momentum', 0) or 0
            
            h_lower, h_upper = calculate_confidence_intervals(home_scores[i], quarter, score_diff, momentum)
            a_lower, a_upper = calculate_confidence_intervals(away_scores[i], quarter, -score_diff, -momentum)
            
            home_lower.append(h_lower)
            home_upper.append(h_upper)
            away_lower.append(a_lower)
            away_upper.append(a_upper)
        
        # Get team names
        home_team = history[0].get('home_team', 'Home Team')
        away_team = history[0].get('away_team', 'Away Team')
        
        # Add home team prediction with confidence interval
        fig.add_trace(
            go.Scatter(
                x=update_indices,
                y=home_scores,
                mode='lines+markers',
                name=f"{home_team} Prediction",
                line=dict(color='blue', width=3),
                hovertemplate="Update %{x}<br>" +
                             f"{home_team}: %{{y:.1f}}<br>" +
                             "Quarter: %{text}<extra></extra>",
                text=quarters
            ),
            row=row, col=col
        )
        
        # Add home team confidence interval
        fig.add_trace(
            go.Scatter(
                x=update_indices + update_indices[::-1],
                y=home_upper + home_lower[::-1],
                fill='toself',
                fillcolor='rgba(0, 0, 255, 0.1)',
                line=dict(color='rgba(0, 0, 255, 0)'),
                hoverinfo="skip",
                showlegend=False,
                name=f"{home_team} Confidence"
            ),
            row=row, col=col
        )
        
        # Add away team prediction with confidence interval
        fig.add_trace(
            go.Scatter(
                x=update_indices,
                y=away_scores,
                mode='lines+markers',
                name=f"{away_team} Prediction",
                line=dict(color='red', width=3),
                hovertemplate="Update %{x}<br>" +
                             f"{away_team}: %{{y:.1f}}<br>" +
                             "Quarter: %{text}<extra></extra>",
                text=quarters
            ),
            row=row, col=col
        )
        
        # Add away team confidence interval
        fig.add_trace(
            go.Scatter(
                x=update_indices + update_indices[::-1],
                y=away_upper + away_lower[::-1],
                fill='toself',
                fillcolor='rgba(255, 0, 0, 0.1)',
                line=dict(color='rgba(255, 0, 0, 0)'),
                hoverinfo="skip",
                showlegend=False,
                name=f"{away_team} Confidence"
            ),
            row=row, col=col
        )
        
        # Add win probability on secondary y-axis
        fig.add_trace(
            go.Scatter(
                x=update_indices,
                y=[wp * 100 for wp in win_probs],
                mode='lines+markers',
                name=f"{home_team} Win Probability",
                line=dict(color='green', width=2, dash='dash'),
                marker=dict(symbol='diamond'),
                hovertemplate="Update %{x}<br>" +
                             f"{home_team} Win: %{{y:.1f}}%<br>" +
                             "Quarter: %{text}<extra></extra>",
                text=quarters
            ),
            secondary_y=True,
            row=row, col=col
        )
        
        # Add current quarter markers at the top
        for i, quarter in enumerate(quarters):
            if i > 0 and quarter != quarters[i-1]:
                fig.add_shape(
                    type="line",
                    x0=i, x1=i,
                    y0=min(min(home_lower), min(away_lower)),
                    y1=max(max(home_upper), max(away_upper)),
                    line=dict(color="gray", width=1, dash="dot"),
                    row=row, col=col
                )
                
                fig.add_annotation(
                    x=i,
                    y=max(max(home_upper), max(away_upper)),
                    text=f"Q{quarter}",
                    showarrow=False,
                    yshift=10,
                    row=row, col=col
                )
        
        # Add annotations for the latest predictions
        if history:
            latest = history[-1]
            latest_home = latest.get('predicted_home_score', 0)
            latest_away = latest.get('predicted_away_score', 0)
            latest_quarter = latest.get('current_quarter', 0)
            
            # Format the quarter text
            quarter_text = "Pre-game" if latest_quarter == 0 else f"Q{latest_quarter}"
            
            # Add a floating annotation with the current prediction
            fig.add_annotation(
                x=len(history) - 1,
                y=max(latest_home, latest_away) + 5,
                text=f"Latest ({quarter_text}): {home_team} {latest_home:.1f} - {away_team} {latest_away:.1f}",
                showarrow=True,
                arrowhead=1,
                arrowcolor="black",
                ax=0,
                ay=-40,
                bgcolor="rgba(255,255,255,0.8)",
                bordercolor="black",
                borderwidth=1,
                row=row, col=col
            )
        
        # Update layout for this subplot
        fig.update_xaxes(title_text="Update Index", row=row, col=col)
        fig.update_yaxes(title_text="Predicted Score", row=row, col=col)
        fig.update_yaxes(title_text="Win Probability (%)", secondary_y=True, 
                          range=[0, 100], row=row, col=col)
        
        idx += 1
    
    # Update overall layout
    fig.update_layout(
        title="Prediction Evolution Over Time",
        height=400 * rows,
        width=600 * cols,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        ),
        margin=dict(l=60, r=60, t=80, b=60)
    )
    
    return fig

def create_live_game_dashboard(predictions_df, prediction_history=None):
    """
    Create an interactive dashboard for live games with Plotly.
    
    Args:
        predictions_df: DataFrame with current game predictions
        prediction_history: Dictionary of prediction history over time
        
    Returns:
        plotly.graph_objects.Figure
    """
    if predictions_df.empty:
        fig = go.Figure()
        fig.add_annotation(
            text="No live games available",
            showarrow=False,
            font=dict(size=16),
            xref="paper", yref="paper",
            x=0.5, y=0.5
        )
        return fig
    
    # Determine number of games
    n_games = len(predictions_df)
    rows = min(n_games, 3)  # Max 3 rows
    cols = 2  # Always 2 columns (current score + prediction evolution)
    
    # Create subplot figure
    fig = make_subplots(
        rows=rows, cols=cols,
        subplot_titles=[f"{row['home_team']} vs {row['away_team']} (Q{row['current_quarter']})" 
                        for _, row in predictions_df.iterrows()],
        specs=[[{"type": "bar"}, {"secondary_y": True}] for _ in range(rows)]
    )
    
    # Process each game
    for i, (_, game) in enumerate(predictions_df.iterrows()):
        row = i + 1
        game_id = game.get('game_id', f"game_{i}")
        
        # Extract data for current scores and predictions
        teams = [game['home_team'], game['away_team']]
        current_scores = [float(game.get('current_home_score', game.get('home_score', 0)) or 0),
                          float(game.get('current_away_score', game.get('away_score', 0)) or 0)]
        predicted_scores = [float(game['predicted_home_score']), float(game['predicted_away_score'])]
        
        # Calculate confidence intervals
        home_lower, home_upper = calculate_confidence_intervals(
            predicted_scores[0], 
            game['current_quarter'],
            current_scores[0] - current_scores[1],
            game.get('momentum_shift', 0) or game.get('cumulative_momentum', 0) or 0
        )
        
        away_lower, away_upper = calculate_confidence_intervals(
            predicted_scores[1], 
            game['current_quarter'],
            current_scores[1] - current_scores[0],
            -(game.get('momentum_shift', 0) or game.get('cumulative_momentum', 0) or 0)
        )
        
        # Create confidence interval text
        home_ci_text = f"±{(home_upper - home_lower)/2:.1f}"
        away_ci_text = f"±{(away_upper - away_lower)/2:.1f}"
        
        # Add current scores
        fig.add_trace(
            go.Bar(
                x=teams,
                y=current_scores,
                name="Current Score",
                marker_color=['royalblue', 'firebrick'],
                text=current_scores,
                textposition='auto',
                hovertemplate="%{x}<br>Current: %{y}<extra></extra>"
            ),
            row=row, col=1
        )
        
        # Add predicted scores with error bars
        fig.add_trace(
            go.Bar(
                x=teams,
                y=predicted_scores,
                name="Predicted Final",
                marker_color=['lightskyblue', 'lightcoral'],
                text=[f"{predicted_scores[0]:.1f} {home_ci_text}", 
                      f"{predicted_scores[1]:.1f} {away_ci_text}"],
                textposition='auto',
                error_y=dict(
                    type='data',
                    array=[(home_upper - predicted_scores[0]), 
                           (away_upper - predicted_scores[1])],
                    arrayminus=[(predicted_scores[0] - home_lower), 
                                (predicted_scores[1] - away_lower)],
                    visible=True
                ),
                hovertemplate="%{x}<br>Predicted: %{y:.1f}<br>Range: %{text}<extra></extra>"
            ),
            row=row, col=1
        )
        
        # Add win probability annotation
        if 'win_probability' in game:
            win_prob = game['win_probability'] * 100
            winner = game['home_team'] if win_prob > 50 else game['away_team']
            win_prob_text = f"{winner} Win: {max(win_prob, 100-win_prob):.1f}%"
            
            fig.add_annotation(
                x=0.5, y=-0.15,
                xref=f"x domain", yref=f"y domain",
                text=win_prob_text,
                showarrow=False,
                font=dict(size=12, color="black"),
                bgcolor="rgba(255,255,255,0.8)",
                bordercolor="black",
                borderwidth=1,
                row=row, col=1
            )
        
        # Add prediction history if available
        if prediction_history and game_id in prediction_history:
            history = prediction_history[game_id]
            
            # Extract data for evolution chart
            update_indices = list(range(len(history)))
            quarters = [pt.get('current_quarter', 0) for pt in history]
            home_scores = [pt.get('predicted_home_score', 0) for pt in history]
            away_scores = [pt.get('predicted_away_score', 0) for pt in history]
            win_probs = [pt.get('win_probability', 0.5) for pt in history]
            
            # Add home prediction evolution
            fig.add_trace(
                go.Scatter(
                    x=update_indices,
                    y=home_scores,
                    mode='lines+markers',
                    name=f"{game['home_team']} Prediction",
                    line=dict(color='blue'),
                    hovertemplate="Update %{x}<br>Predicted: %{y:.1f}<br>Quarter: %{text}<extra></extra>",
                    text=quarters
                ),
                row=row, col=2
            )
            
            # Add away prediction evolution
            fig.add_trace(
                go.Scatter(
                    x=update_indices,
                    y=away_scores,
                    mode='lines+markers',
                    name=f"{game['away_team']} Prediction",
                    line=dict(color='red'),
                    hovertemplate="Update %{x}<br>Predicted: %{y:.1f}<br>Quarter: %{text}<extra></extra>",
                    text=quarters
                ),
                row=row, col=2
            )
            
            # Add win probability evolution
            fig.add_trace(
                go.Scatter(
                    x=update_indices,
                    y=[wp * 100 for wp in win_probs],
                    mode='lines+markers',
                    name="Win Probability",
                    line=dict(color='green', dash='dash'),
                    hovertemplate="Update %{x}<br>Win Prob: %{y:.1f}%<br>Quarter: %{text}<extra></extra>",
                    text=quarters
                ),
                secondary_y=True,
                row=row, col=2
            )
            
            # Update layout for evolution chart
            fig.update_xaxes(title_text="Update Index", row=row, col=2)
            fig.update_yaxes(title_text="Predicted Score", row=row, col=2)
            fig.update_yaxes(title_text="Win Probability (%)", secondary_y=True, 
                             range=[0, 100], row=row, col=2)
        else:
            # Add a placeholder
            fig.add_annotation(
                x=0.5, y=0.5,
                xref=f"x{row*2} domain", yref=f"y{row*2} domain",
                text="Not enough prediction history",
                showarrow=False,
                font=dict(size=12),
                row=row, col=2
            )
    
    # Update overall layout
    fig.update_layout(
        height=350 * rows,
        showlegend=False,
        margin=dict(l=50, r=50, t=30, b=50)
    )
    
    return fig

# Example usage (mock data for testing)
if __name__ == "__main__":
    # Create sample prediction history for testing
    sample_history = {
        "game1": [
            {'home_team': 'Lakers', 'away_team': 'Celtics', 'current_quarter': 1, 
             'predicted_home_score': 105.5, 'predicted_away_score': 102.3, 'win_probability': 0.58,
             'current_home_score': 25, 'current_away_score': 22, 'timestamp': datetime.now()},
            {'home_team': 'Lakers', 'away_team': 'Celtics', 'current_quarter': 2, 
             'predicted_home_score': 107.2, 'predicted_away_score': 103.1, 'win_probability': 0.62,
             'current_home_score': 52, 'current_away_score': 48, 'timestamp': datetime.now()},
            {'home_team': 'Lakers', 'away_team': 'Celtics', 'current_quarter': 3, 
             'predicted_home_score': 110.5, 'predicted_away_score': 105.8, 'win_probability': 0.65,
             'current_home_score': 78, 'current_away_score': 74, 'timestamp': datetime.now()}
        ],
        "game2": [
            {'home_team': 'Warriors', 'away_team': 'Bucks', 'current_quarter': 1, 
             'predicted_home_score': 112.5, 'predicted_away_score': 110.3, 'win_probability': 0.52,
             'current_home_score': 30, 'current_away_score': 29, 'timestamp': datetime.now()},
            {'home_team': 'Warriors', 'away_team': 'Bucks', 'current_quarter': 2, 
             'predicted_home_score': 115.2, 'predicted_away_score': 112.1, 'win_probability': 0.56,
             'current_home_score': 58, 'current_away_score': 55, 'timestamp': datetime.now()}
        ]
    }
    
    # Create sample current predictions dataframe
    sample_predictions = pd.DataFrame([
        {'game_id': 'game1', 'home_team': 'Lakers', 'away_team': 'Celtics', 'current_quarter': 3,
         'home_score': 78, 'away_score': 74, 'current_home_score': 78, 'current_away_score': 74,
         'predicted_home_score': 110.5, 'predicted_away_score': 105.8, 'win_probability': 0.65},
        {'game_id': 'game2', 'home_team': 'Warriors', 'away_team': 'Bucks', 'current_quarter': 2,
         'home_score': 58, 'away_score': 55, 'current_home_score': 58, 'current_away_score': 55,
         'predicted_home_score': 115.2, 'predicted_away_score': 112.1, 'win_probability': 0.56}
    ])
    
    # Create and display the figures
    evolution_fig = create_prediction_evolution_chart(sample_history)
    dashboard_fig = create_live_game_dashboard(sample_predictions, sample_history)
    
    # In a Jupyter notebook, you would display these with:
    # evolution_fig.show()
    # dashboard_fig.show()

In [ ]:
# Cell 10: Data Fetching Functions

def get_team_rolling_averages(days_lookback: int = 60) -> dict:
    try:
        threshold_date = (datetime.now() - timedelta(days=days_lookback)).strftime('%Y-%m-%d')
        response = supabase.table("nba_historical_game_stats").select("*").gte("game_date", threshold_date).execute()
        if not response.data:
            print(f"No historical game data available from the last {days_lookback} days.")
            return {}
        df = pd.DataFrame(response.data)
        df['game_date'] = pd.to_datetime(df['game_date'])
        df = df.sort_values('game_date')
        team_avgs = {}
        all_teams = set(df['home_team'].unique()).union(set(df['away_team'].unique()))
        for team in all_teams:
            home = df[df['home_team'] == team]['home_score']
            away = df[df['away_team'] == team]['away_score']
            all_scores = pd.concat([home, away])
            team_avgs[team] = all_scores.mean() if not all_scores.empty else 105.0
        return team_avgs
    except Exception as e:
        print(f"Error getting team rolling averages: {e}")
        traceback.print_exc()
        return {}

def get_previous_matchup_diff(home_team: str, away_team: str, max_lookback: int = 5) -> float:
    try:
        response_home = supabase.table("nba_historical_game_stats").select("*")\
            .eq("home_team", home_team).eq("away_team", away_team)\
            .order('game_date', desc=True).limit(max_lookback).execute()
        response_away = supabase.table("nba_historical_game_stats").select("*")\
            .eq("home_team", away_team).eq("away_team", home_team)\
            .order('game_date', desc=True).limit(max_lookback).execute()
        matchups = (response_home.data or []) + (response_away.data or [])
        if matchups:
            matchups.sort(key=lambda x: x['game_date'], reverse=True)
            matchups = matchups[:max_lookback]
            diffs = []
            for game in matchups:
                if game['home_team'] == home_team and game['away_team'] == away_team:
                    diffs.append(game['home_score'] - game['away_score'])
                elif game['home_team'] == away_team and game['away_team'] == home_team:
                    diffs.append(game['away_score'] - game['home_score'])
            return np.mean(diffs) if diffs else 0
        return 0
    except Exception as e:
        print(f"Error getting previous matchups: {e}")
        return 0


In [ ]:
# Cell 11: Refactored NBA Score Prediction with Clean Architecture and Ensemble Model

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import time
import traceback
import logging
from typing import Dict, List, Tuple, Optional, Any, Union
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
import joblib

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('nba_prediction')

# ===== VALIDATION FUNCTIONS =====

def validate_game_data(games_df: pd.DataFrame) -> pd.DataFrame:
    """
    Validate and clean game data to ensure it's ready for prediction.
    
    Args:
        games_df: DataFrame with raw game data
        
    Returns:
        Clean, validated DataFrame
    """
    if games_df is None or games_df.empty:
        logger.warning("No game data provided for validation")
        return pd.DataFrame()
    
    # Make a copy to avoid modifying the original
    df = games_df.copy()
    
    # Convert date fields to datetime
    if 'game_date' in df.columns:
        df['game_date'] = pd.to_datetime(df['game_date'], errors='coerce')
    
    # Ensure quarter fields are numeric
    for q in range(1, 5):
        home_q = f'home_q{q}'
        away_q = f'away_q{q}'
        
        if home_q in df.columns:
            df[home_q] = pd.to_numeric(df[home_q], errors='coerce').fillna(0)
        else:
            df[home_q] = 0
            
        if away_q in df.columns:
            df[away_q] = pd.to_numeric(df[away_q], errors='coerce').fillna(0)
        else:
            df[away_q] = 0
    
    # Calculate current quarter based on non-zero quarter scores
    df['current_quarter'] = 0
    for idx, row in df.iterrows():
        max_quarter = 0
        for q in range(1, 5):
            home_q = f'home_q{q}'
            away_q = f'away_q{q}'
            
            if ((home_q in row and pd.notnull(row[home_q]) and row[home_q] > 0) or 
                (away_q in row and pd.notnull(row[away_q]) and row[away_q] > 0)):
                max_quarter = q
        
        df.at[idx, 'current_quarter'] = max_quarter
    
    # Ensure team names are strings
    if 'home_team' in df.columns:
        df['home_team'] = df['home_team'].astype(str)
    if 'away_team' in df.columns:
        df['away_team'] = df['away_team'].astype(str)
    
    # Calculate current score as sum of quarters
    if 'home_score' not in df.columns:
        df['home_score'] = 0
    if 'away_score' not in df.columns:
        df['away_score'] = 0
        
    for idx, row in df.iterrows():
        current_q = int(row.get('current_quarter', 0) or 0)
        
        home_score = 0
        away_score = 0
        for q in range(1, current_q + 1):
            home_q = f'home_q{q}'
            away_q = f'away_q{q}'
            
            if home_q in row and away_q in row:
                home_score += float(row[home_q] or 0)
                away_score += float(row[away_q] or 0)
        
        df.at[idx, 'home_score'] = home_score
        df.at[idx, 'away_score'] = away_score
    
    # Calculate score ratio
    df['score_ratio'] = 0.5  # Default to even
    for idx, row in df.iterrows():
        home_score = float(row.get('home_score', 0) or 0)
        away_score = float(row.get('away_score', 0) or 0)
        total = home_score + away_score
        
        if total > 0:
            df.at[idx, 'score_ratio'] = home_score / total
    
    # Initialize current_home_score and current_away_score if not present
    if 'current_home_score' not in df.columns:
        df['current_home_score'] = df['home_score']
    if 'current_away_score' not in df.columns:
        df['current_away_score'] = df['away_score']
    
    logger.info(f"Validated {len(df)} games")
    return df

def match_games_to_schedule(games_df: pd.DataFrame, schedule_df: pd.DataFrame) -> pd.DataFrame:
    """
    Match games to the official schedule for consistent IDs and metadata.
    
    Args:
        games_df: DataFrame with game data to match
        schedule_df: DataFrame with official schedule
        
    Returns:
        DataFrame with games matched to schedule
    """
    if games_df.empty or schedule_df.empty:
        return games_df
    
    # Make a copy to avoid modifying original
    result_df = games_df.copy()
    result_df['matched_to_schedule'] = False
    
    # Process each game
    for idx, game in result_df.iterrows():
        home_team = game['home_team']
        away_team = game['away_team']
        
        # Look for exact match in schedule
        schedule_match = schedule_df[
            (schedule_df['home_team'] == home_team) & 
            (schedule_df['away_team'] == away_team)
        ]
        
        if not schedule_match.empty:
            # Found a match
            match_row = schedule_match.iloc[0]
            result_df.at[idx, 'matched_to_schedule'] = True
            
            # Copy over official game_id if different
            if 'game_id' in match_row and match_row['game_id'] != game['game_id']:
                result_df.at[idx, 'original_game_id'] = game['game_id']
                result_df.at[idx, 'game_id'] = match_row['game_id']
        else:
            # Try reverse order (sometimes teams are swapped)
            reverse_match = schedule_df[
                (schedule_df['home_team'] == away_team) &
                (schedule_df['away_team'] == home_team)
            ]
            
            if not reverse_match.empty:
                match_row = reverse_match.iloc[0]
                result_df.at[idx, 'matched_to_schedule'] = True
                result_df.at[idx, 'teams_reversed'] = True
                
                # Copy over official game_id if different
                if 'game_id' in match_row and match_row['game_id'] != game['game_id']:
                    result_df.at[idx, 'original_game_id'] = game['game_id']
                    result_df.at[idx, 'game_id'] = match_row['game_id']
    
    matches_count = result_df['matched_to_schedule'].sum()
    logger.info(f"Matched {matches_count} of {len(result_df)} games to official schedule")
    return result_df

# ===== FEATURE ENGINEERING MODULE =====

class FeatureEngineering:
    """Unified feature engineering module for consistent feature calculation."""
    
    @staticmethod
    def get_team_rolling_averages(days_lookback: int = 60) -> Dict[str, Dict[str, float]]:
        """
        Retrieve rolling statistics for each team.
        
        Args:
            days_lookback: Number of days to look back for calculating averages
            
        Returns:
            Dictionary mapping team names to their scoring statistics
        """
        try:
            from caching.supabase_client import supabase
            
            # Calculate threshold date
            threshold_date = (datetime.now() - timedelta(days=days_lookback)).strftime('%Y-%m-%d')
            
            # Query recent game data
            response = supabase.table("nba_historical_game_stats").select("*")\
                .gte("game_date", threshold_date).execute()
            
            if not response.data:
                logger.warning(f"No historical game data found for the past {days_lookback} days")
                return {}
            
            # Process data
            df = pd.DataFrame(response.data)
            df['game_date'] = pd.to_datetime(df['game_date'])
            df = df.sort_values('game_date')
            
            # Initialize dictionary for team statistics
            team_stats = {}
            
            # Get unique teams
            all_teams = set(df['home_team'].unique()).union(set(df['away_team'].unique()))
            
            for team in all_teams:
                # Get home games where team is home
                home_games = df[df['home_team'] == team]
                
                # Get away games where team is away
                away_games = df[df['away_team'] == team]
                
                # Collect scoring data
                home_scores = home_games['home_score'].values
                away_scores = away_games['away_score'].values
                
                # Collect quarter data
                q1_scores = np.concatenate([
                    home_games['home_q1'].values if 'home_q1' in home_games else [],
                    away_games['away_q1'].values if 'away_q1' in away_games else []
                ])
                
                q2_scores = np.concatenate([
                    home_games['home_q2'].values if 'home_q2' in home_games else [],
                    away_games['away_q2'].values if 'away_q2' in away_games else []
                ])
                
                q3_scores = np.concatenate([
                    home_games['home_q3'].values if 'home_q3' in home_games else [],
                    away_games['away_q3'].values if 'away_q3' in away_games else []
                ])
                
                q4_scores = np.concatenate([
                    home_games['home_q4'].values if 'home_q4' in home_games else [],
                    away_games['away_q4'].values if 'away_q4' in away_games else []
                ])
                
                # Combine all scores
                all_scores = np.concatenate([home_scores, away_scores])
                
                # Calculate advanced stats
                team_stats[team] = {
                    'avg_score': np.mean(all_scores) if len(all_scores) > 0 else 105.0,
                    'avg_q1': np.mean(q1_scores) if len(q1_scores) > 0 else 26.0,
                    'avg_q2': np.mean(q2_scores) if len(q2_scores) > 0 else 26.0,
                    'avg_q3': np.mean(q3_scores) if len(q3_scores) > 0 else 26.0,
                    'avg_q4': np.mean(q4_scores) if len(q4_scores) > 0 else 27.0,
                    'std_score': np.std(all_scores) if len(all_scores) > 0 else 12.0,
                    'recent_avg': np.mean(all_scores[-10:]) if len(all_scores) >= 10 else
                                np.mean(all_scores) if len(all_scores) > 0 else 105.0
                }
            
            logger.info(f"Calculated rolling statistics for {len(team_stats)} teams")
            return team_stats
        except Exception as e:
            logger.error(f"Error getting team rolling statistics: {e}")
            traceback.print_exc()
            return {}

    @staticmethod
    def get_previous_matchup_diff(home_team: str, away_team: str, max_lookback: int = 5) -> Dict[str, float]:
        """
        Get detailed statistics from previous matchups between two teams.
        
        Args:
            home_team: Home team name
            away_team: Away team name
            max_lookback: Maximum number of previous matchups to consider
            
        Returns:
            Dictionary with matchup differentials and statistics
        """
        try:
            from caching.supabase_client import supabase
            
            # Query for games where home team hosted away team
            home_response = supabase.table("nba_historical_game_stats").select("*")\
                .eq("home_team", home_team)\
                .eq("away_team", away_team)\
                .order('game_date', desc=True)\
                .limit(max_lookback).execute()
            
            # Query for games where away team hosted home team
            away_response = supabase.table("nba_historical_game_stats").select("*")\
                .eq("home_team", away_team)\
                .eq("away_team", home_team)\
                .order('game_date', desc=True)\
                .limit(max_lookback).execute()
            
            # Combine results
            home_matchups = home_response.data or []
            away_matchups = away_response.data or []
            matchups = home_matchups + away_matchups
            
            if not matchups:
                # No previous matchups found
                return {
                    'avg_diff': 0.0,
                    'first_half_diff': 0.0,
                    'second_half_diff': 0.0,
                    'q1_diff': 0.0,
                    'q2_diff': 0.0,
                    'q3_diff': 0.0,
                    'q4_diff': 0.0,
                    'games_played': 0
                }
            
            # Sort by date (most recent first)
            matchups.sort(key=lambda x: x['game_date'], reverse=True)
            matchups = matchups[:max_lookback]  # Limit to max_lookback games
            
            # Calculate differentials from home team's perspective
            diffs = []
            q1_diffs = []
            q2_diffs = []
            q3_diffs = []
            q4_diffs = []
            first_half_diffs = []
            second_half_diffs = []
            
            for game in matchups:
                if game['home_team'] == home_team and game['away_team'] == away_team:
                    # Home team was home in this matchup
                    diff = game['home_score'] - game['away_score']
                    
                    if 'home_q1' in game and 'away_q1' in game:
                        q1_diffs.append(game['home_q1'] - game['away_q1'])
                    
                    if 'home_q2' in game and 'away_q2' in game:
                        q2_diffs.append(game['home_q2'] - game['away_q2'])
                    
                    if 'home_q3' in game and 'away_q3' in game:
                        q3_diffs.append(game['home_q3'] - game['away_q3'])
                    
                    if 'home_q4' in game and 'away_q4' in game:
                        q4_diffs.append(game['home_q4'] - game['away_q4'])
                    
                elif game['home_team'] == away_team and game['away_team'] == home_team:
                    # Home team was away in this matchup
                    diff = game['away_score'] - game['home_score']
                    
                    if 'home_q1' in game and 'away_q1' in game:
                        q1_diffs.append(game['away_q1'] - game['home_q1'])
                    
                    if 'home_q2' in game and 'away_q2' in game:
                        q2_diffs.append(game['away_q2'] - game['home_q2'])
                    
                    if 'home_q3' in game and 'away_q3' in game:
                        q3_diffs.append(game['away_q3'] - game['home_q3'])
                    
                    if 'home_q4' in game and 'away_q4' in game:
                        q4_diffs.append(game['away_q4'] - game['home_q4'])
                else:
                    continue
                
                diffs.append(diff)
                
                # Calculate half differentials
                if len(q1_diffs) > 0 and len(q2_diffs) > 0:
                    first_half_diffs.append(q1_diffs[-1] + q2_diffs[-1])
                
                if len(q3_diffs) > 0 and len(q4_diffs) > 0:
                    second_half_diffs.append(q3_diffs[-1] + q4_diffs[-1])
            
            # Calculate averages and cap extreme values
            result = {
                'avg_diff': sum(diffs) / len(diffs) if diffs else 0.0,
                'first_half_diff': sum(first_half_diffs) / len(first_half_diffs) if first_half_diffs else 0.0,
                'second_half_diff': sum(second_half_diffs) / len(second_half_diffs) if second_half_diffs else 0.0,
                'q1_diff': sum(q1_diffs) / len(q1_diffs) if q1_diffs else 0.0,
                'q2_diff': sum(q2_diffs) / len(q2_diffs) if q2_diffs else 0.0,
                'q3_diff': sum(q3_diffs) / len(q3_diffs) if q3_diffs else 0.0,
                'q4_diff': sum(q4_diffs) / len(q4_diffs) if q4_diffs else 0.0,
                'games_played': len(diffs)
            }
            
            # Cap extreme values at +/- 15 points
            for key in ['avg_diff', 'first_half_diff', 'second_half_diff', 
                       'q1_diff', 'q2_diff', 'q3_diff', 'q4_diff']:
                result[key] = max(min(result[key], 15.0), -15.0)
                
            return result
        except Exception as e:
            logger.error(f"Error getting previous matchups: {e}")
            return {
                'avg_diff': 0.0,
                'first_half_diff': 0.0,
                'second_half_diff': 0.0,
                'q1_diff': 0.0,
                'q2_diff': 0.0,
                'q3_diff': 0.0,
                'q4_diff': 0.0,
                'games_played': 0
            }

    @staticmethod
    def calculate_momentum_features(games_df: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate momentum-related features for all games.
        
        Args:
            games_df: DataFrame with game data
            
        Returns:
            DataFrame with added momentum features
        """
        if games_df.empty:
            return games_df
        
        # Make a copy to avoid modifying original
        features_df = games_df.copy()
        
        # Ensure all quarter fields are present and numeric
        for q in range(1, 5):
            for team in ['home', 'away']:
                col = f'{team}_q{q}'
                if col not in features_df:
                    features_df[col] = 0.0
                else:
                    features_df[col] = pd.to_numeric(features_df[col], errors='coerce').fillna(0.0)
        
        # Initialize momentum columns with explicit float type
        momentum_cols = [
            'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 
            'cumulative_momentum', 'home_momentum', 'away_momentum',
            'momentum_differential', 'scoring_trend'
        ]
        for col in momentum_cols:
            features_df[col] = np.zeros(len(features_df), dtype=np.float64)
        
        # Calculate quarter-to-quarter momentum with vectorized operations
        # Q1 to Q2 momentum
        q2_mask = features_df['current_quarter'] >= 2
        if q2_mask.any():
            features_df.loc[q2_mask, 'q1_to_q2_momentum'] = (
                (features_df.loc[q2_mask, 'home_q2'] - features_df.loc[q2_mask, 'home_q1']) - 
                (features_df.loc[q2_mask, 'away_q2'] - features_df.loc[q2_mask, 'away_q1'])
            )
        
        # Q2 to Q3 momentum
        q3_mask = features_df['current_quarter'] >= 3
        if q3_mask.any():
            features_df.loc[q3_mask, 'q2_to_q3_momentum'] = (
                (features_df.loc[q3_mask, 'home_q3'] - features_df.loc[q3_mask, 'home_q2']) - 
                (features_df.loc[q3_mask, 'away_q3'] - features_df.loc[q3_mask, 'away_q2'])
            )
        
        # Q3 to Q4 momentum
        q4_mask = features_df['current_quarter'] >= 4
        if q4_mask.any():
            features_df.loc[q4_mask, 'q3_to_q4_momentum'] = (
                (features_df.loc[q4_mask, 'home_q4'] - features_df.loc[q4_mask, 'home_q3']) - 
                (features_df.loc[q4_mask, 'away_q4'] - features_df.loc[q4_mask, 'away_q3'])
            )
        
        # Calculate home and away team momentum separately
        for q_idx in range(1, 4):
            curr_q = q_idx + 1
            prev_q = q_idx
            
            mask = features_df['current_quarter'] >= curr_q
            if mask.any():
                # Home team momentum
                curr_q_col = f'home_q{curr_q}'
                prev_q_col = f'home_q{prev_q}'
                if curr_q_col in features_df.columns and prev_q_col in features_df.columns:
                    momentum_col = f'home_q{prev_q}_to_q{curr_q}_momentum'
                    features_df[momentum_col] = 0.0
                    features_df.loc[mask, momentum_col] = (
                        features_df.loc[mask, curr_q_col] - features_df.loc[mask, prev_q_col]
                    )
                
                # Away team momentum
                curr_q_col = f'away_q{curr_q}'
                prev_q_col = f'away_q{prev_q}'
                if curr_q_col in features_df.columns and prev_q_col in features_df.columns:
                    momentum_col = f'away_q{prev_q}_to_q{curr_q}_momentum'
                    features_df[momentum_col] = 0.0
                    features_df.loc[mask, momentum_col] = (
                        features_df.loc[mask, curr_q_col] - features_df.loc[mask, prev_q_col]
                    )
        
        # Calculate cumulative momentum using vectorized operations with weights
        weights = np.array([0.2, 0.3, 0.5])  # Weights for each quarter transition
        
        # Calculate for Q2
        q2_mask = features_df['current_quarter'] == 2
        if q2_mask.any():
            features_df.loc[q2_mask, 'cumulative_momentum'] = features_df.loc[q2_mask, 'q1_to_q2_momentum'] / 15.0
            
            # Calculate home and away momentum
            features_df.loc[q2_mask, 'home_momentum'] = (
                features_df.loc[q2_mask, 'home_q2'] - features_df.loc[q2_mask, 'home_q1']
            ) / features_df.loc[q2_mask, 'home_q1'].replace(0, 1)
            
            features_df.loc[q2_mask, 'away_momentum'] = (
                features_df.loc[q2_mask, 'away_q2'] - features_df.loc[q2_mask, 'away_q1']
            ) / features_df.loc[q2_mask, 'away_q1'].replace(0, 1)
        
        # Calculate for Q3
        q3_mask = features_df['current_quarter'] == 3
        if q3_mask.any():
            weighted_sum = (
                features_df.loc[q3_mask, 'q1_to_q2_momentum'] * weights[0] +
                features_df.loc[q3_mask, 'q2_to_q3_momentum'] * weights[1]
            )
            weight_sum = weights[0] + weights[1]
            features_df.loc[q3_mask, 'cumulative_momentum'] = weighted_sum / (weight_sum * 15.0)
            
            # Calculate home and away momentum (Q1 to Q3 change)
            features_df.loc[q3_mask, 'home_momentum'] = (
                (features_df.loc[q3_mask, 'home_q2'] + features_df.loc[q3_mask, 'home_q3']) - 
                (features_df.loc[q3_mask, 'home_q1'] * 2)
            ) / (features_df.loc[q3_mask, 'home_q1'] * 2).replace(0, 2)
            
            features_df.loc[q3_mask, 'away_momentum'] = (
                (features_df.loc[q3_mask, 'away_q2'] + features_df.loc[q3_mask, 'away_q3']) - 
                (features_df.loc[q3_mask, 'away_q1'] * 2)
            ) / (features_df.loc[q3_mask, 'away_q1'] * 2).replace(0, 2)
        
        # Calculate for Q4
        q4_mask = features_df['current_quarter'] >= 4
        if q4_mask.any():
            weighted_sum = (
                features_df.loc[q4_mask, 'q1_to_q2_momentum'] * weights[0] +
                features_df.loc[q4_mask, 'q2_to_q3_momentum'] * weights[1] +
                features_df.loc[q4_mask, 'q3_to_q4_momentum'] * weights[2]
            )
            weight_sum = weights.sum()
            features_df.loc[q4_mask, 'cumulative_momentum'] = weighted_sum / (weight_sum * 15.0)
            
            # Calculate home and away momentum (across all quarters)
            features_df.loc[q4_mask, 'home_momentum'] = (
                features_df.loc[q4_mask, 'home_q4'] - features_df.loc[q4_mask, 'home_q1']
            ) / features_df.loc[q4_mask, 'home_q1'].replace(0, 1)
            
            features_df.loc[q4_mask, 'away_momentum'] = (
                features_df.loc[q4_mask, 'away_q4'] - features_df.loc[q4_mask, 'away_q1']
            ) / features_df.loc[q4_mask, 'away_q1'].replace(0, 1)
        
        # Calculate momentum differential
        features_df['momentum_differential'] = features_df['home_momentum'] - features_df['away_momentum']
        
        # Calculate scoring trend (increasing/decreasing scoring through quarters)
        for idx, row in features_df.iterrows():
            current_q = int(row['current_quarter'])
            if current_q <= 1:
                features_df.at[idx, 'scoring_trend'] = 0.0
                continue
                
            quarters = []
            for q in range(1, current_q + 1):
                quarters.append((float(row[f'home_q{q}']) + float(row[f'away_q{q}'])))
            
            # Calculate trend using linear regression slope
            x = np.arange(len(quarters))
            if len(x) >= 2:  # Need at least 2 points for a slope
                slope = np.polyfit(x, quarters, 1)[0]
                features_df.at[idx, 'scoring_trend'] = slope / 10.0  # Normalize
            else:
                features_df.at[idx, 'scoring_trend'] = 0.0
        
        # Clip to [-1, 1] range
        for col in momentum_cols:
            features_df[col] = features_df[col].clip(-1.0, 1.0)
        
        logger.info(f"Added enhanced momentum features to {len(features_df)} games")
        return features_df

@staticmethod
def calculate_rest_features(games_df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate rest-related features for all games with improved handling of back-to-back games.
    
    Args:
        games_df: DataFrame with game data
        
    Returns:
        DataFrame with added rest features
    """
    if games_df.empty:
        return games_df
    
    # Make a copy to avoid modifying original
    features_df = games_df.copy()
    
    # Initialize rest features with defaults
    features_df['rest_days_home'] = 2.0  # Default to 2 days rest
    features_df['rest_days_away'] = 2.0
    features_df['is_back_to_back_home'] = 0
    features_df['is_back_to_back_away'] = 0
    features_df['rest_advantage'] = 0.0
    features_df['home_games_in_5_days'] = 0
    features_df['away_games_in_5_days'] = 0

    try:
        # Process each game
        for idx, row in features_df.iterrows():
            home_team = row['home_team']
            away_team = row['away_team']
            game_date = row.get('game_date')

            if pd.isna(game_date):
                continue

            # Convert game_date to string format if needed
            if isinstance(game_date, pd.Timestamp):
                game_date_str = game_date.strftime('%Y-%m-%d')
            else:
                game_date_str = str(game_date)

            five_days_ago = (pd.to_datetime(game_date) - timedelta(days=5)).strftime('%Y-%m-%d')

            # Get previous games for home team
            home_response = (
                supabase
                .table("nba_historical_game_stats")
                .select("game_date")
                .or_("home_team.eq." + home_team + ",away_team.eq." + home_team)
                .lt("game_date", game_date_str)
                .order("game_date", desc=True)
                .limit(1)
                .execute()
            )

            # Get all games in last 5 days for home team
            home_recent_response = (
                supabase
                .table("nba_historical_game_stats")
                .select("game_date")
                .or_("home_team.eq." + home_team + ",away_team.eq." + home_team)
                .lt("game_date", game_date_str)
                .gte("game_date", five_days_ago)
                .execute()
            )

            # Get previous games for away team
            away_response = (
                supabase
                .table("nba_historical_game_stats")
                .select("game_date")
                .or_("home_team.eq." + away_team + ",away_team.eq." + away_team)
                .lt("game_date", game_date_str)
                .order("game_date", desc=True)
                .limit(1)
                .execute()
            )

            # Get all games in last 5 days for away team
            away_recent_response = (
                supabase
                .table("nba_historical_game_stats")
                .select("game_date")
                .or_("home_team.eq." + away_team + ",away_team.eq." + away_team)
                .lt("game_date", game_date_str)
                .gte("game_date", five_days_ago)
                .execute()
            )

            # Calculate rest days for home team
            if home_response.data:
                prev_game = home_response.data[0]
                prev_date = pd.to_datetime(prev_game['game_date'])
                rest_days = (pd.to_datetime(game_date) - prev_date).days
                features_df.at[idx, 'rest_days_home'] = min(max(rest_days, 0), 10)  # Cap 0–10
                features_df.at[idx, 'is_back_to_back_home'] = 1 if rest_days <= 1 else 0

            # Calculate games in last 5 days for home team
            if home_recent_response.data:
                features_df.at[idx, 'home_games_in_5_days'] = len(home_recent_response.data)

            # Calculate rest days for away team
            if away_response.data:
                prev_game = away_response.data[0]
                prev_date = pd.to_datetime(prev_game['game_date'])
                rest_days = (pd.to_datetime(game_date) - prev_date).days
                features_df.at[idx, 'rest_days_away'] = min(max(rest_days, 0), 10)  # Cap 0–10
                features_df.at[idx, 'is_back_to_back_away'] = 1 if rest_days <= 1 else 0

            # Calculate games in last 5 days for away team
            if away_recent_response.data:
                features_df.at[idx, 'away_games_in_5_days'] = len(away_recent_response.data)

            # Calculate rest advantage
            home_rest = features_df.at[idx, 'rest_days_home']
            away_rest = features_df.at[idx, 'rest_days_away']
            features_df.at[idx, 'rest_advantage'] = home_rest - away_rest

            # Calculate overall fatigue factor
            home_b2b = features_df.at[idx, 'is_back_to_back_home']
            away_b2b = features_df.at[idx, 'is_back_to_back_away']
            home_games_5d = features_df.at[idx, 'home_games_in_5_days']
            away_games_5d = features_df.at[idx, 'away_games_in_5_days']

            features_df.at[idx, 'home_fatigue'] = home_b2b * 1.0 + (home_games_5d * 0.25)
            features_df.at[idx, 'away_fatigue'] = away_b2b * 1.0 + (away_games_5d * 0.25)
            features_df.at[idx, 'fatigue_differential'] = (
                features_df.at[idx, 'away_fatigue'] - features_df.at[idx, 'home_fatigue']
            )

    except Exception as e:
        logger.error(f"Error calculating rest features: {e}")
        traceback.print_exc()

    return features_df


@staticmethod
def prepare_features(
        games_df: pd.DataFrame, 
        team_stats: Optional[Dict[str, Dict[str, float]]] = None,
        include_momentum: bool = True,
        include_rest: bool = True,
        quarter_specific: bool = True
    ) -> pd.DataFrame:
        """
        Prepare all features needed for prediction with comprehensive feature set.
        
        Args:
            games_df: DataFrame with validated game data
            team_stats: Dictionary with team scoring statistics (optional)
            include_momentum: Whether to include momentum features
            include_rest: Whether to include rest-related features
            quarter_specific: Whether to include quarter-specific features
            
        Returns:
            DataFrame with all features needed for prediction
        """
        if games_df.empty:
            return pd.DataFrame()
        
        # Make a copy to avoid modifying original
        features_df = games_df.copy()
        
        # Get team rolling averages if not provided
        if team_stats is None:
            team_stats = FeatureEngineering.get_team_rolling_averages()
        
        # Add rolling scores based on team averages
        for idx, row in features_df.iterrows():
            home_team = row['home_team']
            away_team = row['away_team']
            
            # Add team average scores
            if home_team in team_stats:
                features_df.at[idx, 'rolling_home_score'] = team_stats[home_team]['avg_score']
                if quarter_specific:
                    features_df.at[idx, 'avg_home_q1'] = team_stats[home_team]['avg_q1']
                    features_df.at[idx, 'avg_home_q2'] = team_stats[home_team]['avg_q2']
                    features_df.at[idx, 'avg_home_q3'] = team_stats[home_team]['avg_q3']
                    features_df.at[idx, 'avg_home_q4'] = team_stats[home_team]['avg_q4']
            else:
                features_df.at[idx, 'rolling_home_score'] = 105.0
                if quarter_specific:
                    features_df.at[idx, 'avg_home_q1'] = 26.0
                    features_df.at[idx, 'avg_home_q2'] = 26.0
                    features_df.at[idx, 'avg_home_q3'] = 26.0
                    features_df.at[idx, 'avg_home_q4'] = 27.0
            
            if away_team in team_stats:
                features_df.at[idx, 'rolling_away_score'] = team_stats[away_team]['avg_score']
                if quarter_specific:
                    features_df.at[idx, 'avg_away_q1'] = team_stats[away_team]['avg_q1']
                    features_df.at[idx, 'avg_away_q2'] = team_stats[away_team]['avg_q2']
                    features_df.at[idx, 'avg_away_q3'] = team_stats[away_team]['avg_q3']
                    features_df.at[idx, 'avg_away_q4'] = team_stats[away_team]['avg_q4']
            else:
                features_df.at[idx, 'rolling_away_score'] = 105.0
                if quarter_specific:
                    features_df.at[idx, 'avg_away_q1'] = 26.0
                    features_df.at[idx, 'avg_away_q2'] = 26.0
                    features_df.at[idx, 'avg_away_q3'] = 26.0
                    features_df.at[idx, 'avg_away_q4'] = 27.0
            
            # Add previous matchup differential
            matchup_stats = FeatureEngineering.get_previous_matchup_diff(home_team, away_team)
            features_df.at[idx, 'prev_matchup_diff'] = matchup_stats['avg_diff']
            
            # Add detailed matchup statistics
            if quarter_specific:
                features_df.at[idx, 'prev_first_half_diff'] = matchup_stats['first_half_diff']
                features_df.at[idx, 'prev_second_half_diff'] = matchup_stats['second_half_diff']
                features_df.at[idx, 'prev_q1_diff'] = matchup_stats['q1_diff']
                features_df.at[idx, 'prev_q2_diff'] = matchup_stats['q2_diff']
                features_df.at[idx, 'prev_q3_diff'] = matchup_stats['q3_diff']
                features_df.at[idx, 'prev_q4_diff'] = matchup_stats['q4_diff']
                features_df.at[idx, 'prev_matchups_count'] = matchup_stats['games_played']
        
        # Add momentum features if requested
        if include_momentum:
            features_df = FeatureEngineering.calculate_momentum_features(features_df)
        
        # Add rest features if requested
        if include_rest:
            features_df = FeatureEngineering.calculate_rest_features(features_df)
        
        # Add quarter-specific features
        if quarter_specific:
            current_quarter = features_df['current_quarter'].fillna(0).astype(int)
            
            # Pre-game (Q0) and Q1 features
            features_df['is_q0_or_q1'] = (current_quarter <= 1).astype(int)
            
            # Q2 features
            features_df['is_q2'] = (current_quarter == 2).astype(int)
            
            # Q3 features
            features_df['is_q3'] = (current_quarter == 3).astype(int)
            
            # Q4 features
            features_df['is_q4'] = (current_quarter == 4).astype(int)
            
            # Create time remaining feature (approximation)
            features_df['time_remaining_pct'] = (4 - current_quarter) / 4
            features_df.loc[current_quarter == 0, 'time_remaining_pct'] = 1.0
            
            # Enhanced scoring trends by quarter
            for idx, row in features_df.iterrows():
                q = int(row['current_quarter'])
                if q >= 1:
                    features_df.at[idx, 'q1_score_total'] = float(row['home_q1']) + float(row['away_q1'])
                if q >= 2:
                    features_df.at[idx, 'q2_score_total'] = float(row['home_q2']) + float(row['away_q2'])
                    features_df.at[idx, 'q2_vs_q1_trend'] = features_df.at[idx, 'q2_score_total'] - features_df.at[idx, 'q1_score_total']
                if q >= 3:
                    features_df.at[idx, 'q3_score_total'] = float(row['home_q3']) + float(row['away_q3'])
                    features_df.at[idx, 'q3_vs_q2_trend'] = features_df.at[idx, 'q3_score_total'] - features_df.at[idx, 'q2_score_total']
                    features_df.at[idx, 'second_half_predicted'] = 1
                else:
                    features_df.at[idx, 'second_half_predicted'] = 0
        
        # Add score differential features
        for idx, row in features_df.iterrows():
            current_q = int(row['current_quarter'])
            if current_q > 0:
                home_score = float(row.get('home_score', 0) or 0)
                away_score = float(row.get('away_score', 0) or 0)
                features_df.at[idx, 'score_differential'] = home_score - away_score
                features_df.at[idx, 'abs_score_differential'] = abs(home_score - away_score)
                features_df.at[idx, 'close_game'] = 1 if abs(home_score - away_score) <= 10 else 0
                features_df.at[idx, 'blowout_game'] = 1 if abs(home_score - away_score) >= 20 else 0
        
        logger.info(f"Prepared comprehensive features for {len(features_df)} games")
        return features_df

# ===== QUARTER-SPECIFIC MODELS =====

class QuarterModels:
    """Container for quarter-specific prediction models."""
    
    def __init__(self):
        self.pre_game_model = None
        self.q1_model = None
        self.q2_model = None
        self.q3_model = None
        self.q4_model = None
        self.ensemble_weights = {
            0: {'main': 0.80, 'quarter': 0.20},
            1: {'main': 0.70, 'quarter': 0.30},
            2: {'main': 0.60, 'quarter': 0.40},
            3: {'main': 0.40, 'quarter': 0.60},
            4: {'main': 0.30, 'quarter': 0.70}
        }
        self.feature_lists = {}
        self.is_initialized = False
    
    def initialize_models(self):
        """Initialize all quarter-specific models."""
        try:
            # Pre-game model
            self.pre_game_model = GradientBoostingRegressor(
                n_estimators=120,
                learning_rate=0.1,
                max_depth=4,
                random_state=42,
                subsample=0.85
            )
            self.feature_lists[0] = [
                'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff',
                'rest_days_home', 'rest_days_away', 'rest_advantage',
                'is_back_to_back_home', 'is_back_to_back_away'
            ]
            
            # Q1 model
            self.q1_model = GradientBoostingRegressor(
                n_estimators=100,
                learning_rate=0.1,
                max_depth=3,
                random_state=42,
                subsample=0.85
            )
            self.feature_lists[1] = [
                'home_q1', 'away_q1', 'score_ratio',
                'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff',
                'rest_days_home', 'rest_days_away', 'rest_advantage'
            ]
            
            # Q2 model
            self.q2_model = GradientBoostingRegressor(
                n_estimators=120,
                learning_rate=0.1,
                max_depth=4,
                random_state=42,
                subsample=0.85
            )
            self.feature_lists[2] = [
                'home_q1', 'home_q2', 'away_q1', 'away_q2',
                'score_ratio', 'q1_to_q2_momentum', 'prev_matchup_diff',
                'rest_days_home', 'rest_days_away', 'prev_first_half_diff'
            ]
            
            # Q3 model
            self.q3_model = GradientBoostingRegressor(
                n_estimators=150,
                learning_rate=0.1,
                max_depth=4,
                random_state=42,
                subsample=0.8
            )
            self.feature_lists[3] = [
                'home_q1', 'home_q2', 'home_q3', 
                'away_q1', 'away_q2', 'away_q3',
                'score_ratio', 'q1_to_q2_momentum', 'q2_to_q3_momentum',
                'cumulative_momentum', 'prev_matchup_diff',
                'score_differential', 'prev_second_half_diff'
            ]
            
            # Q4 model - Use Ridge regression for better stability
            self.q4_model = Ridge(
                alpha=1.0,
                fit_intercept=True,
                normalize=False,
                random_state=42
            )
            self.feature_lists[4] = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4',
                'away_q1', 'away_q2', 'away_q3', 'away_q4',
                'score_ratio', 'score_differential',
                'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum',
                'cumulative_momentum', 'prev_q4_diff'
            ]
            
            self.is_initialized = True
            logger.info("Quarter-specific models initialized successfully")
            return True
        except Exception as e:
            logger.error(f"Error initializing quarter models: {e}")
            traceback.print_exc()
            return False
    
    def load_models(self, model_dir: str = None):
        """
        Load pre-trained quarter-specific models from disk.
        
        Args:
            model_dir: Directory containing saved models (optional)
        
        Returns:
            True if successful, False otherwise
        """
        try:
            import config
            if model_dir is None:
                model_dir = config.MODEL_DIR
                
            # Load pre-game model
            self.pre_game_model = joblib.load(f"{model_dir}/pre_game_model.joblib")
            
            # Load quarter models
            self.q1_model = joblib.load(f"{model_dir}/q1_model.joblib")
            self.q2_model = joblib.load(f"{model_dir}/q2_model.joblib")
            self.q3_model = joblib.load(f"{model_dir}/q3_model.joblib")
            self.q4_model = joblib.load(f"{model_dir}/q4_model.joblib")
            
            # Load feature lists 
            feature_lists_path = f"{model_dir}/feature_lists.joblib"
            if os.path.exists(feature_lists_path):
                self.feature_lists = joblib.load(feature_lists_path)
            
            self.is_initialized = True
            logger.info("Quarter-specific models loaded successfully")
            return True
        except Exception as e:
            logger.error(f"Error loading quarter models: {e}")
            logger.info("Initializing new models instead")
            return self.initialize_models()
    
    def get_model_for_quarter(self, quarter: int):
        """
        Get the appropriate model for a specific quarter.
        
        Args:
            quarter: Current quarter (0-4)
            
        Returns:
            Tuple of (model, feature_list)
        """
        if not self.is_initialized:
            self.initialize_models()
            
        if quarter == 0:
            return self.pre_game_model, self.feature_lists.get(0, [])
        elif quarter == 1:
            return self.q1_model, self.feature_lists.get(1, [])
        elif quarter == 2:
            return self.q2_model, self.feature_lists.get(2, [])
        elif quarter == 3:
            return self.q3_model, self.feature_lists.get(3, [])
        elif quarter >= 4:
            return self.q4_model, self.feature_lists.get(4, [])
        else:
            logger.warning(f"Invalid quarter: {quarter}, using pre-game model")
            return self.pre_game_model, self.feature_lists.get(0, [])
    
    def get_ensemble_weights(self, quarter: int) -> Dict[str, float]:
        """
        Get the appropriate ensemble weights for a specific quarter.
        
        Args:
            quarter: Current quarter (0-4)
            
        Returns:
            Dictionary of weight values
        """
        return self.ensemble_weights.get(quarter, self.ensemble_weights[0])

# ===== MODEL MANAGEMENT =====

class ModelManager:
    """Class to manage prediction models and ensembles."""
    
    def __init__(self):
        self.main_model = None
        self.quarter_models = QuarterModels()
        self.team_stats = None
        self.feature_list = None
        self.initialized = False
    
    def initialize(self):
        """Initialize the model manager and load all models."""
        if self.initialized:
            return True
            
        try:
            # Initialize quarter-specific models
            self.quarter_models.initialize_models()
            
            # Get main model from global or load from file
            self.main_model, self.feature_list = self.get_main_prediction_model()
            
            # Get team statistics
            self.team_stats = FeatureEngineering.get_team_rolling_averages()
            
            self.initialized = True
            logger.info("Model manager initialized successfully")
            return True
        except Exception as e:
            logger.error(f"Error initializing model manager: {e}")
            traceback.print_exc()
            return False
    
    def get_main_prediction_model(self) -> Tuple[Any, List[str]]:
        """
        Get the main prediction model and its expected features.
        
        Returns:
            Tuple of (model, feature_list)
        """
        model = None
        
        # Try to get model from global variables first
        if 'model' in globals() and globals()['model'] is not None:
            model = globals()['model']
            logger.info("Using model from global 'model' variable")
        elif 'score_model' in globals() and globals()['score_model'] is not None:
            model = globals()['score_model']
            logger.info("Using model from global 'score_model' variable")
        
        # If no model in globals, try to load from file
        if model is None:
            try:
                import config
                model = joblib.load(config.MODEL_PATH)
                logger.info(f"Loaded model from {config.MODEL_PATH}")
            except Exception as e:
                logger.error(f"Error loading model: {e}")
                logger.warning("Proceeding without a main prediction model")
                model = GradientBoostingRegressor(
                    n_estimators=150,
                    learning_rate=0.1,
                    max_depth=4,
                    random_state=42,
                    subsample=0.8
                )
        
        # Determine feature list based on model type
        feature_list = []
        
        if model is not None and hasattr(model, 'feature_importances_'):
            # Enhanced model with 15 features
            if len(model.feature_importances_) > 12:
                feature_list = [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4',
                    'score_ratio', 'prev_matchup_diff',
                    'rest_days_home', 'rest_days_away', 'rest_advantage',
                    'is_back_to_back_home', 'is_back_to_back_away',
                    'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
                ]
                logger.info("Using enhanced feature set with momentum and rest features")
            # Medium model with 9 features
            elif len(model.feature_importances_) > 8:
                feature_list = [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4',
                    'score_ratio', 'prev_matchup_diff',
                    'rest_days_home', 'rest_days_away', 'rest_advantage'
                ]
                logger.info("Using medium feature set with rest features")
            # Original model with 8 features
            else:
                feature_list = [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4',
                    'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
                ]
                logger.info("Using original feature set")
        else:
            # Default to enhanced feature set
            feature_list = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4',
                'score_ratio', 'prev_matchup_diff',
                'rest_days_home', 'rest_days_away', 'rest_advantage',
                'is_back_to_back_home', 'is_back_to_back_away',
                'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
            ]
            logger.warning("Could not determine feature list from model, using enhanced features")
        
        return model, feature_list
    
    def predict_with_ensemble(self, features_df: pd.DataFrame) -> pd.DataFrame:
        """
        Make predictions using the ensemble of models.
        
        Args:
            features_df: DataFrame with features
            
        Returns:
            DataFrame with predictions
        """
        if not self.initialized:
            self.initialize()
            
        result_df = features_df.copy()
        
        # Add prediction columns
        result_df['predicted_home_score'] = 0.0
        result_df['predicted_away_score'] = 0.0
        result_df['main_model_home_pred'] = 0.0
        result_df['quarter_model_home_pred'] = 0.0
        result_df['prediction_confidence'] = 0.0
        result_df['ensemble_weight_main'] = 0.0
        result_df['ensemble_weight_quarter'] = 0.0
        
        # Make predictions for each game
        for idx, row in features_df.iterrows():
            current_q = int(row.get('current_quarter', 0) or 0)
            
            # Get current scores
            current_home = float(row.get('current_home_score', row.get('home_score', 0)) or 0)
            current_away = float(row.get('current_away_score', row.get('away_score', 0)) or 0)
            
            # Get ensemble weights for this quarter
            weights = self.quarter_models.get_ensemble_weights(current_q)
            result_df.at[idx, 'ensemble_weight_main'] = weights['main']
            result_df.at[idx, 'ensemble_weight_quarter'] = weights['quarter']
            
            # Make prediction with main model if available
            main_pred_home = None
            if self.main_model is not None:
                try:
                    # Extract features for main model
                    main_features = [f for f in self.feature_list if f in features_df.columns]
                    if all(f in features_df.columns for f in main_features):
                        X_main = features_df.loc[[idx], main_features]
                        main_pred_home = float(self.main_model.predict(X_main)[0])
                        result_df.at[idx, 'main_model_home_pred'] = main_pred_home
                    else:
                        missing = [f for f in self.feature_list if f not in features_df.columns]
                        logger.warning(f"Missing features for main model: {missing}")
                except Exception as e:
                    logger.error(f"Error predicting with main model: {e}")
            
            # Make prediction with quarter-specific model
            quarter_pred_home = None
            try:
                # Get appropriate quarter model
                quarter_model, quarter_features = self.quarter_models.get_model_for_quarter(current_q)
                
                if quarter_model is not None:
                    # Extract features for quarter model
                    available_features = [f for f in quarter_features if f in features_df.columns]
                    
                    # Only predict if we have enough features
                    if len(available_features) >= len(quarter_features) * 0.75:
                        X_quarter = features_df.loc[[idx], available_features]
                        quarter_pred_home = float(quarter_model.predict(X_quarter)[0])
                        result_df.at[idx, 'quarter_model_home_pred'] = quarter_pred_home
                    else:
                        missing = [f for f in quarter_features if f not in features_df.columns]
                        logger.warning(f"Missing too many features for Q{current_q} model: {missing}")
            except Exception as e:
                logger.error(f"Error predicting with quarter-specific model: {e}")
            
            # Combine predictions using ensemble weights
            if main_pred_home is not None and quarter_pred_home is not None:
                # Check if predictions are reasonably close
                pred_diff = abs(main_pred_home - quarter_pred_home)
                if pred_diff > 15:  # Large disagreement between models
                    # Adjust weights to favor the model that's closer to current score
                    main_diff = abs(main_pred_home - current_home)
                    quarter_diff = abs(quarter_pred_home - current_home)
                    
                    if main_diff < quarter_diff:
                        # Main model seems more reasonable
                        weights = {'main': 0.8, 'quarter': 0.2}
                    else:
                        # Quarter model seems more reasonable
                        weights = {'main': 0.2, 'quarter': 0.8}
                    
                    logger.info(f"Large model disagreement ({pred_diff:.1f} pts), adjusting weights")
                
                # Calculate ensemble prediction
                ensemble_home = (main_pred_home * weights['main']) + (quarter_pred_home * weights['quarter'])
                result_df.at[idx, 'predicted_home_score'] = max(ensemble_home, current_home)
            elif main_pred_home is not None:
                # Only main model available
                result_df.at[idx, 'predicted_home_score'] = max(main_pred_home, current_home)
            elif quarter_pred_home is not None:
                # Only quarter model available
                result_df.at[idx, 'predicted_home_score'] = max(quarter_pred_home, current_home)
            else:
                # No models available, use basic prediction
                self._predict_baseline(result_df, idx, row)
                continue
            
            # Calculate away score prediction
            self._calculate_away_score(result_df, idx, row)
            
            # Calculate confidence based on quarter
            result_df.at[idx, 'prediction_confidence'] = calculate_prediction_confidence(current_q)
            
            # Calculate win probability
            home_pred = result_df.at[idx, 'predicted_home_score']
            away_pred = result_df.at[idx, 'predicted_away_score']
            result_df.at[idx, 'win_probability'] = calculate_win_probability(home_pred, away_pred, current_q)
            
            # Calculate remaining points
            result_df.at[idx, 'remaining_home_points'] = max(0, home_pred - current_home)
            result_df.at[idx, 'remaining_away_points'] = max(0, away_pred - current_away)
            
            # Calculate margin and total
            result_df.at[idx, 'projected_margin'] = home_pred - away_pred
            result_df.at[idx, 'total_projected_score'] = home_pred + away_pred
        
        return result_df
    
    def _predict_baseline(self, result_df: pd.DataFrame, idx: int, row: Dict):
        """Make baseline prediction when models are unavailable."""
        home_team = row['home_team']
        away_team = row['away_team']
        current_q = int(row.get('current_quarter', 0) or 0)
        
        # Get current scores
        current_home = float(row.get('current_home_score', row.get('home_score', 0)) or 0)
        current_away = float(row.get('current_away_score', row.get('away_score', 0)) or 0)
        
        # Get team averages
        if self.team_stats and home_team in self.team_stats:
            home_avg = self.team_stats[home_team]['avg_score']
        else:
            home_avg = 105.0
            
        if self.team_stats and away_team in self.team_stats:
            away_avg = self.team_stats[away_team]['avg_score']
        else:
            away_avg = 105.0
        
        # Calculate remaining portion of game
        if current_q == 0:
            # Pre-game prediction
            result_df.at[idx, 'predicted_home_score'] = home_avg
            result_df.at[idx, 'predicted_away_score'] = away_avg
        else:
            # In-game prediction
            remaining_pct = 1.0 - (0.25 * current_q)
            
            result_df.at[idx, 'predicted_home_score'] = current_home + (home_avg * remaining_pct)
            result_df.at[idx, 'predicted_away_score'] = current_away + (away_avg * remaining_pct)
        
        # Add confidence metrics
        result_df.at[idx, 'prediction_confidence'] = calculate_prediction_confidence(current_q)
        
        # Calculate win probability
        home_pred = result_df.at[idx, 'predicted_home_score']
        away_pred = result_df.at[idx, 'predicted_away_score']
        result_df.at[idx, 'win_probability'] = calculate_win_probability(home_pred, away_pred, current_q)
    
    def _calculate_away_score(self, result_df: pd.DataFrame, idx: int, row: Dict):
        """Calculate away score based on predicted home score and context."""
        home_team = row['home_team']
        away_team = row['away_team']
        current_q = int(row.get('current_quarter', 0) or 0)
        
        # Get current scores
        current_home = float(row.get('current_home_score', row.get('home_score', 0)) or 0)
        current_away = float(row.get('current_away_score', row.get('away_score', 0)) or 0)
        
        # Get predicted home score
        predicted_home = result_df.at[idx, 'predicted_home_score']
        
        # Calculate away score prediction
        # Using matchup differential, home advantage, and momentum
        matchup_diff = float(row.get('prev_matchup_diff', 0) or 0)
        home_adv = 3.5  # Standard home court advantage
        
        # Get momentum adjustment if available
        momentum_adj = 0.0
        if 'cumulative_momentum' in row:
            momentum = float(row.get('cumulative_momentum', 0) or 0)
            momentum_adj = momentum * 3.0  # Scale to points impact
        
        # For later quarters, preserve the current point differential more
        current_diff = current_home - current_away
        
        if current_q >= 3:
            # For later quarters, the current differential is more important
            quarter_weight = 0.7
            predicted_diff = (current_diff * quarter_weight) + (matchup_diff + home_adv + momentum_adj) * (1 - quarter_weight)
        else:
            # For earlier quarters, use more of the historical factors
            quarter_weight = 0.3 * current_q
            predicted_diff = (current_diff * quarter_weight) + (matchup_diff + home_adv + momentum_adj) * (1 - quarter_weight)
        
        # Calculate predicted away score
        predicted_away = predicted_home - predicted_diff
        
        # CRITICAL: Ensure predicted scores are never less than current scores
        predicted_away = max(predicted_away, current_away)
        
        # Store prediction
        result_df.at[idx, 'predicted_away_score'] = predicted_away

def calculate_prediction_confidence(quarter: int) -> float:
    """
    Calculate confidence percentage based on game quarter.
    
    Args:
        quarter: Current quarter (0-4)
        
    Returns:
        Confidence value between 0-1
    """
    confidence_map = {
        0: 0.3,  # Pre-game
        1: 0.45, # First quarter
        2: 0.65, # Second quarter
        3: 0.8,  # Third quarter
        4: 0.95  # Fourth quarter
    }
    return confidence_map.get(quarter, 0.3)

def calculate_win_probability(home_score: float, away_score: float, quarter: int, time_remaining: float = None) -> float:
    """
    Calculate win probability for the home team.
    
    Args:
        home_score: Current or predicted home score
        away_score: Current or predicted away score
        quarter: Current quarter (0-4)
        time_remaining: Minutes remaining in the game (optional)
        
    Returns:
        Win probability between 0-1
    """
    import math
    
    score_diff = home_score - away_score
    
    # Calculate game progress
    if time_remaining is not None:
        total_time = 48.0
        elapsed = total_time - time_remaining
        progress = elapsed / total_time
    else:
        progress = min(quarter / 4.0, 1.0)
    
    # Steepness parameter increases with game progress
    k = 0.05 + (progress * 0.15)
    
    # Logistic function for win probability
    return 1.0 / (1.0 + math.exp(-k * score_diff))

# ===== MAIN PREDICTION FUNCTION =====

def predict_final_scores(
    games_df: pd.DataFrame, 
    team_stats: Dict[str, Dict[str, float]] = None,
    prediction_history: Dict[str, List[Dict]] = None,
    official_schedule: pd.DataFrame = None
) -> pd.DataFrame:
    """
    Predict final scores for NBA games with clean architecture and ensemble model.
    
    Args:
        games_df: DataFrame with game data
        team_stats: Dictionary of team scoring averages (optional)
        prediction_history: Dictionary to store prediction history (optional)
        official_schedule: DataFrame with official game schedule (optional)
        
    Returns:
        DataFrame with predictions and confidence metrics
    """
    start_time = time.time()
    logger.info(f"Starting prediction for {len(games_df)} games")
    
    # Validate and clean input data
    validated_df = validate_game_data(games_df)
    if validated_df.empty:
        logger.warning("No valid game data available for prediction")
        return pd.DataFrame()
    
    # Match against official schedule if provided
    if official_schedule is not None and not official_schedule.empty:
        validated_df = match_games_to_schedule(validated_df, official_schedule)
    
    # Initialize model manager if needed
    if 'model_manager' not in globals() or globals()['model_manager'] is None:
        globals()['model_manager'] = ModelManager()
    
    model_manager = globals()['model_manager']
    if not model_manager.initialized:
        model_manager.initialize()
    
    # Get team statistics if not provided
    if team_stats is None:
        team_stats = model_manager.team_stats or FeatureEngineering.get_team_rolling_averages()
    
    # Prepare comprehensive features
    features_df = FeatureEngineering.prepare_features(
        validated_df, 
        team_stats=team_stats,
        include_momentum=True,
        include_rest=True,
        quarter_specific=True
    )
    
    # Make ensemble predictions
    try:
        result_df = model_manager.predict_with_ensemble(features_df)
    except Exception as e:
        logger.error(f"Error making ensemble predictions: {e}")
        traceback.print_exc()
        
        # Fallback to simple predictions
        result_df = validated_df.copy()
        
        # Add baseline predictions based on team averages
        for idx, row in result_df.iterrows():
            home_team = row['home_team']
            away_team = row['away_team']
            current_q = int(row.get('current_quarter', 0) or 0)
            
            # Get current scores
            current_home = float(row.get('current_home_score', row.get('home_score', 0)) or 0)
            current_away = float(row.get('current_away_score', row.get('away_score', 0)) or 0)
            
            # Get team averages
            home_avg = 105.0
            away_avg = 105.0
            
            if team_stats:
                if home_team in team_stats:
                    home_avg = team_stats[home_team]['avg_score']
                if away_team in team_stats:
                    away_avg = team_stats[away_team]['avg_score']
            
            # Calculate remaining portion of game
            remaining_pct = 1.0 - (0.25 * current_q)
            
            # Make fallback predictions
            result_df.at[idx, 'predicted_home_score'] = current_home + (home_avg * remaining_pct)
            result_df.at[idx, 'predicted_away_score'] = current_away + (away_avg * remaining_pct)
            
            # Add metrics
            result_df.at[idx, 'prediction_confidence'] = calculate_prediction_confidence(current_q)
            
            home_pred = result_df.at[idx, 'predicted_home_score']
            away_pred = result_df.at[idx, 'predicted_away_score']
            result_df.at[idx, 'win_probability'] = calculate_win_probability(home_pred, away_pred, current_q)
    
    # Add timestamp
    result_df['prediction_timestamp'] = datetime.now()
    
    # Update prediction history if provided
    if prediction_history is not None:
        for _, game in result_df.iterrows():
            game_id = str(game['game_id'])
            
            if game_id not in prediction_history:
                prediction_history[game_id] = []
            
            # Add current prediction to history
            prediction_record = {
                'timestamp': datetime.now(),
                'game_id': game_id,
                'home_team': game['home_team'],
                'away_team': game['away_team'],
                'current_quarter': int(game.get('current_quarter', 0) or 0),
                'current_home_score': float(game.get('current_home_score', game.get('home_score', 0)) or 0),
                'current_away_score': float(game.get('current_away_score', game.get('away_score', 0)) or 0),
                'predicted_home_score': float(game['predicted_home_score']),
                'predicted_away_score': float(game['predicted_away_score']),
                'win_probability': float(game['win_probability']),
                'prediction_confidence': float(game['prediction_confidence'])
            }
            
            # Add momentum if available
            if 'cumulative_momentum' in features_df.columns:
                prediction_record['momentum'] = float(features_df.at[game.name, 'cumulative_momentum'])
            
            # Add ensemble weights if available
            if 'ensemble_weight_main' in game:
                prediction_record['ensemble_weight_main'] = float(game['ensemble_weight_main'])
                prediction_record['ensemble_weight_quarter'] = float(game['ensemble_weight_quarter'])
            
            prediction_history[game_id].append(prediction_record)
        
        logger.info(f"Updated prediction history for {len(result_df)} games")
    
    elapsed_time = time.time() - start_time
    logger.info(f"Completed predictions in {elapsed_time:.2f} seconds")
    
    return result_df

# ===== VISUALIZATION FUNCTIONS =====

def visualize_predictions(predictions_df: pd.DataFrame, prediction_history: Dict = None) -> plt.Figure:
    """
    Create visualizations for score predictions.
    
    Args:
        predictions_df: DataFrame with predictions
        prediction_history: Optional prediction history dictionary
        
    Returns:
        matplotlib Figure object
    """
    if predictions_df.empty:
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.text(0.5, 0.5, "No predictions available", 
                ha='center', va='center', fontsize=14)
        return fig
    
    # Determine layout based on number of games
    n_games = len(predictions_df)
    rows = min(n_games, 3)  # Max 3 rows
    fig, axs = plt.subplots(rows, 2, figsize=(16, 5 * rows))
    
    # Handle case with single game
    if n_games == 1:
        axs = np.array([axs])
    
    # Process each game
    for i, (_, game) in enumerate(predictions_df.iterrows()):
        row = i % rows
        
        # Left plot: Current vs Predicted scores
        ax1 = axs[row, 0]
        
        # Extract data
        teams = [game['home_team'], game['away_team']]
        current_scores = [
            float(game.get('current_home_score', game.get('home_score', 0)) or 0),
            float(game.get('current_away_score', game.get('away_score', 0)) or 0)
        ]
        predicted_scores = [
            float(game['predicted_home_score']),
            float(game['predicted_away_score'])
        ]
        
        # Create bar chart
        x = np.arange(len(teams))
        width = 0.35
        
        current_bars = ax1.bar(x - width/2, current_scores, width, label='Current')
        predicted_bars = ax1.bar(x + width/2, predicted_scores, width, label='Predicted Final')
        
        # Add bar labels
        for bar in current_bars:
            height = bar.get_height()
            ax1.annotate(f'{int(height)}',
                        xy=(bar.get_x() + bar.get_width() / 2, height),
                        xytext=(0, 3),
                        textcoords="offset points",
                        ha='center', va='bottom')
        
        for bar in predicted_bars:
            height = bar.get_height()
            ax1.annotate(f'{height:.1f}',
                        xy=(bar.get_x() + bar.get_width() / 2, height),
                        xytext=(0, 3),
                        textcoords="offset points",
                        ha='center', va='bottom')
        
        # Customize plot
        ax1.set_xticks(x)
        ax1.set_xticklabels(teams)
        ax1.legend()
        
        # Add quarter and confidence info
        quarter = int(game.get('current_quarter', 0) or 0)
        confidence = float(game.get('prediction_confidence', 0.5)) * 100
        quarter_text = "Pre-game" if quarter == 0 else f"Quarter {quarter}"
        
        ax1.set_title(f"{teams[0]} vs {teams[1]} - {quarter_text} (Confidence: {confidence:.0f}%)")
        
        # Add win probability annotation
        if 'win_probability' in game:
            win_prob = float(game['win_probability']) * 100
            win_team = teams[0] if win_prob > 50 else teams[1]
            win_prob_display = win_prob if win_prob > 50 else (100 - win_prob)
            
            ax1.annotate(f"{win_team} Win: {win_prob_display:.1f}%",
                        xy=(0.5, -0.1),
                        xycoords='axes fraction',
                        ha='center',
                        bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="orange", alpha=0.8))
        
        # Right plot: Quarter breakdown or prediction history
        ax2 = axs[row, 1]
        
        # If prediction history available, plot the prediction trends
        if prediction_history and str(game['game_id']) in prediction_history:
            history = prediction_history[str(game['game_id'])]
            
            if len(history) > 1:  # Only plot if we have multiple predictions
                timestamps = [h['timestamp'] for h in history]
                home_predictions = [h['predicted_home_score'] for h in history]
                away_predictions = [h['predicted_away_score'] for h in history]
                
                # Convert timestamps to relative time for better display
                start_time = min(timestamps)
                rel_times = [(t - start_time).total_seconds() / 60.0 for t in timestamps]
                
                # Plot prediction trends
                ax2.plot(rel_times, home_predictions, 'b-o', label=f"{teams[0]} (Predicted)")
                ax2.plot(rel_times, away_predictions, 'r-o', label=f"{teams[1]} (Predicted)")
                
                # Add actual scores if available
                if 'current_quarter' in history[0]:
                    home_actuals = [h.get('current_home_score', 0) for h in history]
                    away_actuals = [h.get('current_away_score', 0) for h in history]
                    
                    ax2.plot(rel_times, home_actuals, 'b--', label=f"{teams[0]} (Actual)")
                    ax2.plot(rel_times, away_actuals, 'r--', label=f"{teams[1]} (Actual)")
                
                ax2.set_xlabel('Time (minutes)')
                ax2.set_ylabel('Score')
                ax2.legend(loc='upper left', fontsize=8)
                ax2.set_title('Prediction Trend Over Time')
                
                # Add quarter transitions if available
                quarters = [h.get('current_quarter', 0) for h in history]
                for q in range(1, 5):
                    # Find first instance of this quarter
                    try:
                        q_idx = quarters.index(q)
                        q_time = rel_times[q_idx]
                        ax2.axvline(x=q_time, color='gray', linestyle='--', alpha=0.5)
                        ax2.text(q_time, ax2.get_ylim()[1] * 0.95, f"Q{q}", 
                                 rotation=90, va='top', ha='right')
                    except ValueError:
                        pass  # Quarter not in history
            else:
                # Not enough history for trend
                plot_quarter_breakdown(ax2, game, teams)
        else:
            # No history, show quarter breakdown
            plot_quarter_breakdown(ax2, game, teams)
    
    # Hide unused subplots
    for i in range(n_games, rows * 2):
        row = i // 2
        col = i % 2
        if row < rows and col < 2:  # Check if axis exists in our grid
            axs[row, col].axis('off')
    
    plt.tight_layout()
    return fig

def plot_quarter_breakdown(ax, game, teams):
    """
    Plot quarter-by-quarter breakdown when prediction history isn't available.
    
    Args:
        ax: Matplotlib axis to plot on
        game: Row from predictions DataFrame
        teams: List of team names [home, away]
    """
    quarters = []
    home_scores = []
    away_scores = []
    
    # Get current quarter scores
    current_q = int(game.get('current_quarter', 0) or 0)
    for q in range(1, current_q + 1):
        home_q = f'home_q{q}'
        away_q = f'away_q{q}'
        
        if home_q in game and away_q in game:
            quarters.append(f"Q{q}")
            home_scores.append(float(game[home_q] or 0))
            away_scores.append(float(game[away_q] or 0))
    
    # Add predicted remaining quarters if not full game yet
    if current_q < 4:
        # Get current totals
        current_home = float(game.get('current_home_score', game.get('home_score', 0)) or 0)
        current_away = float(game.get('current_away_score', game.get('away_score', 0)) or 0)
        
        # Get predicted finals
        predicted_home = float(game['predicted_home_score'])
        predicted_away = float(game['predicted_away_score'])
        
        # Calculate remaining points
        remaining_home = max(0, predicted_home - current_home)
        remaining_away = max(0, predicted_away - current_away)
        
        # Calculate remaining quarters
        remaining_quarters = 4 - current_q
        
        if remaining_quarters > 0:
            quarters.append(f"Rem. {remaining_quarters}Q")
            home_scores.append(remaining_home)
            away_scores.append(remaining_away)
    
    # If no quarter data or pre-game, show predicted final distribution
    if not quarters:
        quarters = ["Predicted Final"]
        home_scores = [float(game['predicted_home_score'])]
        away_scores = [float(game['predicted_away_score'])]
    
    # Create bar chart
    x = np.arange(len(quarters))
    width = 0.35
    
    ax.bar(x - width/2, home_scores, width, label=teams[0], color='blue', alpha=0.7)
    ax.bar(x + width/2, away_scores, width, label=teams[1], color='red', alpha=0.7)
    
    # Customize chart
    ax.set_xticks(x)
    ax.set_xticklabels(quarters)
    ax.legend()
    
    current_q_text = "Pre-game" if current_q == 0 else f"Through Q{current_q}"
    ax.set_title(f"Score Breakdown ({current_q_text})")
    
    # Add labels to bars
    for i, v in enumerate(home_scores):
        ax.text(i - width/2, v + 0.5, f"{v:.1f}", ha='center')
    
    for i, v in enumerate(away_scores):
        ax.text(i + width/2, v + 0.5, f"{v:.1f}", ha='center')

# ===== USAGE EXAMPLE =====

def demo_prediction_visualization():
    """
    Demonstrate the prediction visualization with sample data.
    """
    # Sample game data (mid-game)
    sample_games = pd.DataFrame([
        {
            'game_id': '1001',
            'home_team': 'Lakers', 
            'away_team': 'Celtics',
            'current_quarter': 2,
            'home_q1': 28, 'away_q1': 26,
            'home_q2': 24, 'away_q2': 30,
            'home_score': 52, 'away_score': 56,
            'predicted_home_score': 108.5, 'predicted_away_score': 104.2,
            'win_probability': 0.62, 'prediction_confidence': 0.65
        },
        {
            'game_id': '1002',
            'home_team': 'Warriors', 
            'away_team': 'Bucks',
            'current_quarter': 3,
            'home_q1': 32, 'away_q1': 22,
            'home_q2': 28, 'away_q2': 26,
            'home_q3': 24, 'away_q3': 31,
            'home_score': 84, 'away_score': 79,
            'predicted_home_score': 120.3, 'predicted_away_score': 115.7,
            'win_probability': 0.73, 'prediction_confidence': 0.80
        },
        {
            'game_id': '1003',
            'home_team': 'Nets', 
            'away_team': 'Heat',
            'current_quarter': 0,  # Pre-game
            'home_score': 0, 'away_score': 0,
            'predicted_home_score': 112.8, 'predicted_away_score': 109.4,
            'win_probability': 0.58, 'prediction_confidence': 0.35
        }
    ])
    
    # Sample prediction history
    sample_history = {
        '1001': [
            {
                'timestamp': datetime.now() - timedelta(minutes=30),
                'current_quarter': 1,
                'current_home_score': 28, 'current_away_score': 26,
                'predicted_home_score': 110.5, 'predicted_away_score': 102.3
            },
            {
                'timestamp': datetime.now() - timedelta(minutes=15),
                'current_quarter': 2,
                'current_home_score': 52, 'current_away_score': 56,
                'predicted_home_score': 108.5, 'predicted_away_score': 104.2
            }
        ]
    }
    
    # Generate visualization
    fig = visualize_predictions(sample_games, sample_history)
    
    # Show plot
    plt.show()
    
    return fig

# Run demo if executed directly
if __name__ == "__main__":
    demo_prediction_visualization()

In [ ]:
# Cell 11A: Enhanced Layered Fallback System (Integrated Version)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import traceback
from typing import Dict, List, Union, Optional, Tuple, Any

class LayeredFallbackPredictor:
    """
    Enhanced layered fallback system for NBA score predictions that gracefully degrades
    when features or model availability changes. This system is designed to be used
    with the main prediction pipeline and supports different levels of prediction
    quality based on available data.
    
    Fallback Levels:
    1. Enhanced Model: Uses full feature set including momentum features (highest accuracy)
    2. Basic Model: Uses core features without momentum (high accuracy)
    3. Quarter Extrapolation: When model is unavailable but quarter data exists (medium accuracy)
    4. Season Averages: When only team identities are known (lower accuracy)
    5. League Averages: Ultimate fallback with minimal information (lowest accuracy)
    """
    
    def __init__(self, model=None, team_avgs=None):
        """
        Initialize the fallback prediction system.
        
        Args:
            model: The primary prediction model (optional)
            team_avgs: Dictionary of team scoring averages (optional)
        """
        self.model = model
        self.team_avgs = team_avgs or {}
        self.league_avg_score = 110.0  # NBA average as default
        self.home_advantage = 3.5      # Standard NBA home court advantage
        
        # Track which fallback level was used for analytics
        self.used_fallback_levels = {
            "enhanced_model": 0,
            "basic_model": 0,
            "quarter_extrapolation": 0,
            "season_averages": 0,
            "league_averages": 0
        }
    
    def predict(self, game_data: pd.DataFrame) -> pd.DataFrame:
        """
        Make predictions using the best available strategy based on data quality.
        
        Args:
            game_data: DataFrame with game information
            
        Returns:
            DataFrame with predictions and metadata about strategy used
        """
        if game_data.empty:
            print("No game data provided to fallback system")
            return pd.DataFrame()
        
        results = []
        for idx, game in game_data.iterrows():
            try:
                result = self._predict_single_game(game)
                results.append(result)
            except Exception as e:
                print(f"Error in fallback prediction for game {game.get('game_id', 'unknown')}: {str(e)}")
                traceback.print_exc()
                # Add emergency fallback result
                results.append(self._create_emergency_fallback(game))
        
        # Create DataFrame with results
        return pd.DataFrame(results)
    
    def _predict_single_game(self, game: pd.Series) -> Dict[str, Any]:
        """
        Predict a single game using the best available strategy.
        
        Args:
            game: Series with single game data
            
        Returns:
            Dict with prediction results and metadata
        """
        # Extract basic game information
        game_id = game.get('game_id')
        home_team = game.get('home_team')
        away_team = game.get('away_team')
        current_quarter = int(game.get('current_quarter', 0))
        
        # Initialize result dict with game info
        result = {
            'game_id': game_id,
            'home_team': home_team,
            'away_team': away_team,
            'current_quarter': current_quarter
        }
        
        # Calculate current score
        current_home_score = 0
        current_away_score = 0
        if current_quarter > 0:
            for q in range(1, current_quarter+1):
                current_home_score += float(game.get(f'home_q{q}', 0) or 0)
                current_away_score += float(game.get(f'away_q{q}', 0) or 0)
        
        result['current_home_score'] = current_home_score
        result['current_away_score'] = current_away_score
        
        # Determine data availability
        has_quarter_data = all(f'home_q{q}' in game for q in range(1, current_quarter+1))
        has_team_data = home_team in self.team_avgs and away_team in self.team_avgs
        has_enhanced_features = ('cumulative_momentum' in game) or ('q1_to_q2_momentum' in game)
        has_model = self.model is not None
        
        # LEVEL 1: Full enhanced model prediction
        if has_model and has_quarter_data and has_enhanced_features and hasattr(self.model, 'feature_importances_') and len(self.model.feature_importances_) > 8:
            print(f"Using enhanced model for {home_team} vs {away_team}")
            self.used_fallback_levels["enhanced_model"] += 1
            
            # Prepare features for enhanced model
            features = self._prepare_enhanced_features(game)
            
            try:
                # Make prediction using enhanced features
                predicted_home_score = self._predict_with_model(features)
            except ValueError as e:
                # If feature name mismatch error is encountered, log and try basic model
                print(f"Enhanced model prediction failed due to feature mismatch: {e}")
                if has_quarter_data:
                    print(f"Falling back to basic model for {home_team} vs {away_team}")
                    self.used_fallback_levels["basic_model"] += 1
                    features = self._prepare_basic_features(game)
                    predicted_home_score = self._predict_with_model(features)
                    prev_matchup_diff = float(game.get('prev_matchup_diff', 0))
                    predicted_away_score = predicted_home_score - prev_matchup_diff - self.home_advantage
                    result['fallback_model_type'] = "basic_model"
                    result['prediction_confidence'] = 0.8 * (1 + current_quarter / 4) / 2
                else:
                    raise e
            else:
                # Calculate away score based on differential factors for enhanced model
                diff_weight = min(0.3 + (0.1 * current_quarter), 0.6)
                momentum_adj = float(game.get('cumulative_momentum', 0)) * 3.0
                prev_matchup_diff = float(game.get('prev_matchup_diff', 0))
                predicted_away_score = predicted_home_score - (prev_matchup_diff * diff_weight) - momentum_adj
                result['fallback_model_type'] = "enhanced_model"
                result['prediction_confidence'] = 0.9 * (1 + current_quarter / 4) / 2
            
        # LEVEL 2: Basic model prediction
        elif has_model and has_quarter_data:
            print(f"Using basic model for {home_team} vs {away_team}")
            self.used_fallback_levels["basic_model"] += 1
            
            # Prepare features for basic model
            features = self._prepare_basic_features(game)
            predicted_home_score = self._predict_with_model(features)
            prev_matchup_diff = float(game.get('prev_matchup_diff', 0))
            predicted_away_score = predicted_home_score - prev_matchup_diff - self.home_advantage
            result['fallback_model_type'] = "basic_model"
            result['prediction_confidence'] = 0.8 * (1 + current_quarter / 4) / 2
            
        # LEVEL 3: Quarter extrapolation
        elif current_quarter > 0 and has_quarter_data:
            print(f"Using quarter extrapolation for {home_team} vs {away_team}")
            self.used_fallback_levels["quarter_extrapolation"] += 1
            
            home_ppq = current_home_score / current_quarter
            away_ppq = current_away_score / current_quarter
            
            remaining_quarters = 4 - current_quarter
            predicted_home_score = current_home_score + (home_ppq * remaining_quarters)
            predicted_away_score = current_away_score + (away_ppq * remaining_quarters)
            result['fallback_model_type'] = "quarter_extrapolation"
            result['prediction_confidence'] = 0.7 * (1 + current_quarter / 4) / 2
            
        # LEVEL 4: Season averages
        elif has_team_data:
            print(f"Using season averages for {home_team} vs {away_team}")
            self.used_fallback_levels["season_averages"] += 1
            
            home_avg = self.team_avgs[home_team]
            away_avg = self.team_avgs[away_team]
            predicted_home_score = home_avg + (self.home_advantage / 2)
            predicted_away_score = away_avg - (self.home_advantage / 2)
            result['fallback_model_type'] = "season_averages"
            result['prediction_confidence'] = 0.5
            
        # LEVEL 5: League averages (ultimate fallback)
        else:
            print(f"Using league averages for {home_team} vs {away_team}")
            self.used_fallback_levels["league_averages"] += 1
            
            predicted_home_score = self.league_avg_score + (self.home_advantage / 2)
            predicted_away_score = self.league_avg_score - (self.home_advantage / 2)
            result['fallback_model_type'] = "league_averages"
            result['prediction_confidence'] = 0.3
        
        # Ensure predictions are at least current scores
        if current_quarter > 0:
            predicted_home_score = max(predicted_home_score, current_home_score)
            predicted_away_score = max(predicted_away_score, current_away_score)
        
        # Calculate win probability
        score_diff = predicted_home_score - predicted_away_score
        result['predicted_home_score'] = predicted_home_score
        result['predicted_away_score'] = predicted_away_score
        result['win_probability'] = 1.0 / (1.0 + np.exp(-0.1 * score_diff))
        
        result['remaining_home_points'] = max(0, predicted_home_score - current_home_score)
        result['remaining_away_points'] = max(0, predicted_away_score - current_away_score)
        
        return result
    
    def _prepare_enhanced_features(self, game: pd.Series) -> pd.DataFrame:
        """
        Prepare enhanced feature set for prediction.
        
        Args:
            game: Series with game data
            
        Returns:
            DataFrame with prepared features (NaN values are filled with 0)
        """
        features = pd.DataFrame({
            'home_q1': [float(game.get('home_q1', 0) or 0)],
            'home_q2': [float(game.get('home_q2', 0) or 0)],
            'home_q3': [float(game.get('home_q3', 0) or 0)],
            'home_q4': [float(game.get('home_q4', 0) or 0)],
            'score_ratio': [float(game.get('score_ratio', 0.5) or 0.5)],
            'prev_matchup_diff': [float(game.get('prev_matchup_diff', 0) or 0)],
            'rest_days_home': [float(game.get('rest_days_home', 2) or 2)],
            'rest_days_away': [float(game.get('rest_days_away', 2) or 2)],
            'rest_advantage': [float(game.get('rest_advantage', 0) or 0)],
            # Use the naming expected by the model:
            'is_back_to_back_home': [float(game.get('is_back_to_back_home', 0) or 0)],
            'is_back_to_back_away': [float(game.get('is_back_to_back_away', 0) or 0)],
            'q1_to_q2_momentum': [float(game.get('q1_to_q2_momentum', 0) or 0)],
            'q2_to_q3_momentum': [float(game.get('q2_to_q3_momentum', 0) or 0)],
            'q3_to_q4_momentum': [float(game.get('q3_to_q4_momentum', 0) or 0)],
            'cumulative_momentum': [float(game.get('cumulative_momentum', 0) or 0)]
        })
        # Attempt to add missing feature "momentum_rest_interaction" if expected by the model
        if self.model is not None and hasattr(self.model, "feature_names_in_"):
            if "momentum_rest_interaction" in self.model.feature_names_in_ and "momentum_rest_interaction" not in features.columns:
                # Define as product of cumulative_momentum and rest_advantage
                features["momentum_rest_interaction"] = features["cumulative_momentum"] * features["rest_advantage"]
        features.fillna(0, inplace=True)
        return features
    
    def _prepare_basic_features(self, game: pd.Series) -> pd.DataFrame:
        """
        Prepare basic feature set for prediction.
        
        Args:
            game: Series with game data
            
        Returns:
            DataFrame with prepared features (NaN values are filled with 0)
        """
        home_team = game.get('home_team')
        away_team = game.get('away_team')
        
        rolling_home = self.team_avgs.get(home_team, 105.0)
        rolling_away = self.team_avgs.get(away_team, 105.0)
        
        features = pd.DataFrame({
            'home_q1': [float(game.get('home_q1', 0) or 0)],
            'home_q2': [float(game.get('home_q2', 0) or 0)],
            'home_q3': [float(game.get('home_q3', 0) or 0)],
            'home_q4': [float(game.get('home_q4', 0) or 0)],
            'score_ratio': [float(game.get('score_ratio', 0.5) or 0.5)],
            'rolling_home_score': [rolling_home],
            'rolling_away_score': [rolling_away],
            'prev_matchup_diff': [float(game.get('prev_matchup_diff', 0) or 0)]
        })
        features.fillna(0, inplace=True)
        return features
    
    def _predict_with_model(self, features: pd.DataFrame) -> float:
        """
        Make prediction using the model with error handling for feature name mismatches.
        
        Args:
            features: DataFrame with features
            
        Returns:
            float: Predicted home score
        """
        if self.model is None:
            raise ValueError("Model not available for prediction")
        
        features.fillna(0, inplace=True)
        
        try:
            prediction = float(self.model.predict(features)[0])
            return prediction
        except ValueError as e:
            if "feature names" in str(e) and hasattr(self.model, "feature_names_in_"):
                print(f"Feature name mismatch encountered: {e}")
                expected = set(self.model.feature_names_in_)
                current = set(features.columns)
                
                rename_map = {}
                # Map our naming to expected naming if possible
                if "is_home_b2b" in expected and "is_back_to_back_home" in current:
                    rename_map["is_back_to_back_home"] = "is_home_b2b"
                if "is_away_b2b" in expected and "is_back_to_back_away" in current:
                    rename_map["is_back_to_back_away"] = "is_away_b2b"
                
                if rename_map:
                    features = features.rename(columns=rename_map)
                
                # Add any missing expected columns with default 0
                for col in expected:
                    if col not in features.columns:
                        features[col] = 0.0
                # Reorder columns to match the fitted order
                features = features[list(self.model.feature_names_in_)]
                
                # Retry prediction after adjustments
                prediction = float(self.model.predict(features)[0])
                return prediction
            else:
                raise e
    
    def _create_emergency_fallback(self, game: pd.Series) -> Dict[str, Any]:
        """
        Create emergency fallback prediction when all else fails.
        
        Args:
            game: Game data
            
        Returns:
            Dict with minimal prediction
        """
        return {
            'game_id': game.get('game_id', 'unknown'),
            'home_team': game.get('home_team', 'Unknown'),
            'away_team': game.get('away_team', 'Unknown'),
            'current_quarter': int(game.get('current_quarter', 0)),
            'current_home_score': 0,
            'current_away_score': 0,
            'predicted_home_score': 105,
            'predicted_away_score': 102,
            'win_probability': 0.55,
            'fallback_model_type': "emergency_fallback",
            'prediction_confidence': 0.1
        }
    
    def get_fallback_usage_stats(self) -> Dict[str, int]:
        """
        Get statistics on which fallback levels were used.
        
        Returns:
            Dict with counts of each fallback level used
        """
        return self.used_fallback_levels
    
    def integration_test(self, test_data: Optional[pd.DataFrame] = None) -> pd.DataFrame:
        """
        Run an integration test of the fallback system with test data.
        
        Args:
            test_data: Optional DataFrame with test data, generates sample if None
            
        Returns:
            DataFrame with test results
        """
        if test_data is None:
            test_data = pd.DataFrame([
                {
                    'game_id': 1001,
                    'home_team': 'Boston Celtics',
                    'away_team': 'Miami Heat',
                    'current_quarter': 3,
                    'home_q1': 28, 'home_q2': 30, 'home_q3': 25,
                    'away_q1': 25, 'away_q2': 27, 'away_q3': 24,
                    'cumulative_momentum': 0.2,
                    'prev_matchup_diff': 4.5,
                    'rest_days_home': 2,
                    'rest_days_away': 1,
                    'rest_advantage': 1,
                    'is_back_to_back_home': 0,
                    'is_back_to_back_away': 1
                },
                {
                    'game_id': 1002,
                    'home_team': 'Los Angeles Lakers',
                    'away_team': 'Golden State Warriors',
                    'current_quarter': 2,
                    'home_q1': 31, 'home_q2': 28,
                    'away_q1': 29, 'away_q2': 32
                },
                {
                    'game_id': 1003,
                    'home_team': 'Boston Celtics',
                    'away_team': 'Los Angeles Lakers',
                    'current_quarter': 0
                },
                {
                    'game_id': 1004,
                    'home_team': 'Unknown Team',
                    'away_team': 'Mystery Team',
                    'current_quarter': 1
                }
            ])
        
        # Reset fallback usage stats
        self.used_fallback_levels = {
            "enhanced_model": 0,
            "basic_model": 0,
            "quarter_extrapolation": 0,
            "season_averages": 0,
            "league_averages": 0
        }
        
        print("Testing layered fallback system with different scenarios...")
        test_results = self.predict(test_data)
        
        print("\nFallback System Test Results:")
        display_cols = ['home_team', 'away_team', 'current_quarter', 
                        'predicted_home_score', 'predicted_away_score',
                        'fallback_model_type', 'prediction_confidence']
        
        if 'display' in globals():
            from IPython.display import display
            display(test_results[display_cols])
        else:
            print(test_results[display_cols])
        
        print("\nFallback Level Usage:")
        for level, count in self.used_fallback_levels.items():
            print(f"  {level}: {count} times")
        
        return test_results

# Instantiate the fallback system and run a test
if 'model' in globals() and globals()['model'] is not None:
    fallback_predictor = LayeredFallbackPredictor(model=globals()['model'])
else:
    fallback_predictor = LayeredFallbackPredictor()

# Add team averages if available
team_avgs = {}
if 'get_team_rolling_averages' in globals():
    try:
        team_avgs = globals()['get_team_rolling_averages']()
        fallback_predictor.team_avgs = team_avgs
    except Exception as e:
        print(f"Could not load team averages: {e}")

# Run the integration test
test_results = fallback_predictor.integration_test()

# Example of how to use with the main prediction pipeline:
"""
def run_prediction_pipeline(games_df, use_fallback=True):
    try:
        if 'model' in globals() and globals()['model'] is not None:
            predictions = make_predictions(games_df, globals()['model'])
            return predictions
        else:
            print("Main model not available, falling back to layered system")
            use_fallback = True
    except Exception as e:
        print(f"Error in main prediction: {e}")
        use_fallback = True
    
    if use_fallback:
        print("Using layered fallback system")
        return fallback_predictor.predict(games_df)
"""


In [ ]:
# Cell 11B: Refactored Strategy Selection System with Registration Pattern

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import time
import traceback
from typing import Dict, List, Tuple, Optional, Union, Any, Callable
import json

class GameContextAnalyzer:
    """
    Analyzes game state to determine context for strategy selection.
    """
    
    def __init__(self):
        # Priority order for primary context determination
        self.context_priorities = [
            "clutch_time",
            "final_minutes",
            "blowout",
            "close_game",
            "high_momentum",
            "comeback_potential",
            "run",
            "late_game",
            "early_game",
            "pregame"
        ]
    
    def analyze(self, game_data: Dict[str, Any]) -> Dict[str, Any]:
        """
        Analyze the game context to determine appropriate prediction strategy.
        
        Args:
            game_data: Dictionary with game information
            
        Returns:
            Dict with context information
        """
        # Extract key game attributes
        current_quarter = int(game_data.get('current_quarter', 0))
        home_score = float(game_data.get('home_score', 0) or 0)
        away_score = float(game_data.get('away_score', 0) or 0)
        score_diff = home_score - away_score
        abs_score_diff = abs(score_diff)
        
        # Extract momentum if available
        momentum = 0.0
        if 'cumulative_momentum' in game_data:
            momentum = float(game_data.get('cumulative_momentum', 0))
        elif 'momentum_shift' in game_data:
            momentum = float(game_data.get('momentum_shift', 0))
        
        # Extract time remaining if available
        time_remaining = float(game_data.get('time_remaining', 12))
        
        # Game phase determination
        game_phase = ""
        if current_quarter == 0:
            game_phase = "pregame"
        elif current_quarter <= 2:
            game_phase = "first_half"
        else:
            game_phase = "second_half"
        
        # Identify specific contexts that apply to this game state
        contexts = [game_phase]
        
        # Quarter-specific context
        if current_quarter == 0:
            contexts.append("pregame")
        elif current_quarter == 1:
            contexts.append("early_game")
        elif current_quarter == 4:
            contexts.append("late_game")
            
            # Final minutes
            if time_remaining <= 5:
                contexts.append("final_minutes")
                
                # Clutch time (close & final minutes)
                if abs_score_diff <= 8:
                    contexts.append("clutch_time")
        
        # Score differential contexts
        if abs_score_diff <= 5:
            contexts.append("close_game")
        elif abs_score_diff >= 20:
            contexts.append("blowout")
        elif abs_score_diff >= 12:
            contexts.append("large_lead")
        
        # Momentum-based contexts
        if abs(momentum) >= 0.6:
            contexts.append("high_momentum")
            
            # Direction of momentum
            if momentum > 0:
                contexts.append("home_momentum")
            else:
                contexts.append("away_momentum")
        
        # Potential comeback context
        if abs_score_diff >= 10 and ((score_diff > 0 and momentum < -0.3) or 
                                     (score_diff < 0 and momentum > 0.3)):
            contexts.append("comeback_potential")
        
        # Recent run context
        run_detected = False
        for q in range(1, 5):
            if q > current_quarter:
                break
                
            if q == current_quarter:
                # Check most recent quarter
                home_q = game_data.get(f'home_q{q}', 0) or 0
                away_q = game_data.get(f'away_q{q}', 0) or 0
                
                # If one team outscored the other by 8+ points
                if abs(home_q - away_q) >= 8:
                    run_detected = True
                    if home_q > away_q:
                        contexts.append("home_run")
                    else:
                        contexts.append("away_run")
        
        if run_detected:
            contexts.append("run")
        
        # Scoring pace context
        total_score = home_score + away_score
        expected_score_by_quarter = [0, 50, 105, 160, 215]  # Rough estimates
        
        if current_quarter > 0 and current_quarter < 5:
            expected_score = expected_score_by_quarter[current_quarter]
            if total_score > expected_score * 1.15:
                contexts.append("high_scoring")
            elif total_score < expected_score * 0.85:
                contexts.append("low_scoring")
        
        # Calculate a primary context (most specific/important one)
        primary_context = self._determine_primary_context(contexts)
        
        # Return the full context analysis
        return {
            'current_quarter': current_quarter,
            'score_differential': score_diff,
            'abs_score_differential': abs_score_diff,
            'momentum': momentum,
            'time_remaining': time_remaining,
            'contexts': contexts,
            'primary_context': primary_context,
            'game_phase': game_phase
        }
    
    def _determine_primary_context(self, contexts: List[str]) -> str:
        """Determine the most important context for strategy selection."""
        # Return the highest priority context that applies
        for context in self.context_priorities:
            if context in contexts:
                return context
        
        # Default to first context if no priority match
        return contexts[0] if contexts else "unknown"


class PredictionStrategy:
    """
    Base class for prediction strategies.
    """
    
    def __init__(self, name: str, description: str, preferred_contexts: List[str], model_key: Optional[str] = None):
        self.name = name
        self.description = description
        self.preferred_contexts = preferred_contexts
        self.model_key = model_key
        self.fallback_strategy = None
        self.success_rate = 0.0
        self.usage_count = 0
        self.avg_error = None
    
    def set_fallback(self, fallback_strategy: Optional['PredictionStrategy'] = None):
        """Set fallback strategy to use if this one fails."""
        self.fallback_strategy = fallback_strategy
    
    def predict(self, game_data: Dict[str, Any], model: Any = None, team_avgs: Dict[str, float] = None) -> Dict[str, Any]:
        """
        Execute prediction for this strategy.
        
        Args:
            game_data: Game data dictionary
            model: Model to use for prediction (if needed)
            team_avgs: Team scoring averages
            
        Returns:
            Dictionary with prediction results
        """
        raise NotImplementedError("Subclasses must implement predict method")


class EarlyQuartersStrategy(PredictionStrategy):
    """Strategy specialized for early quarters (Q0-Q2)."""
    
    def __init__(self):
        super().__init__(
            name="Early Quarters XGBoost",
            description="Specialized XGBoost model for Q0-Q2",
            preferred_contexts=["pregame", "early_game", "first_half"],
            model_key="early_quarters_model"
        )
    
    def predict(self, game_data: Dict[str, Any], model: Any = None, team_avgs: Dict[str, float] = None) -> Dict[str, Any]:
        # In a real implementation, this would use the model
        # For now, we'll implement a simple prediction
        current_quarter = game_data.get('current_quarter', 0)
        home_team = game_data.get('home_team', '')
        away_team = game_data.get('away_team', '')
        
        # Get team averages if available
        home_avg = team_avgs.get(home_team, 105.0) if team_avgs else 105.0
        away_avg = team_avgs.get(away_team, 105.0) if team_avgs else 105.0
        
        # For pregame, adjust with home court advantage
        if current_quarter == 0:
            home_advantage = 3.5
            predicted_home = home_avg + (home_advantage / 2)
            predicted_away = away_avg - (home_advantage / 2)
        else:
            # Get current score
            home_score = float(game_data.get('home_score', 0))
            away_score = float(game_data.get('away_score', 0))
            
            # Calculate remaining portion of game
            remaining_factor = 1.0 - (0.25 * current_quarter)
            
            # Calculate predicted final scores
            predicted_home = home_score + (home_avg * remaining_factor)
            predicted_away = away_score + (away_avg * remaining_factor)
        
        # Calculate win probability
        score_diff = predicted_home - predicted_away
        win_prob = 1.0 / (1.0 + np.exp(-0.08 * score_diff))
        
        return {
            'game_id': game_data.get('game_id'),
            'home_team': home_team,
            'away_team': away_team,
            'current_quarter': current_quarter,
            'predicted_home_score': predicted_home,
            'predicted_away_score': predicted_away,
            'win_probability': win_prob,
            'prediction_confidence': 0.7 + (0.1 * current_quarter)  # Increases with quarter
        }


class MomentumStrategy(PredictionStrategy):
    """Strategy that weights predictions based on momentum factors."""
    
    def __init__(self):
        super().__init__(
            name="Momentum-Weighted Strategy",
            description="Strategy that weights models based on momentum factors",
            preferred_contexts=["high_momentum", "comeback", "run"],
            model_key="momentum_model"
        )
    
    def predict(self, game_data: Dict[str, Any], model: Any = None, team_avgs: Dict[str, float] = None) -> Dict[str, Any]:
        # Extract momentum
        momentum = 0.0
        if 'cumulative_momentum' in game_data:
            momentum = float(game_data.get('cumulative_momentum', 0))
        elif 'momentum_shift' in game_data:
            momentum = float(game_data.get('momentum_shift', 0))
        
        current_quarter = game_data.get('current_quarter', 0)
        home_team = game_data.get('home_team', '')
        away_team = game_data.get('away_team', '')
        
        # Get current score
        home_score = float(game_data.get('home_score', 0))
        away_score = float(game_data.get('away_score', 0))
        
        # Get team averages if available
        home_avg = team_avgs.get(home_team, 105.0) if team_avgs else 105.0
        away_avg = team_avgs.get(away_team, 105.0) if team_avgs else 105.0
        
        # Calculate remaining portion of game
        remaining_factor = 1.0 - (0.25 * current_quarter)
        
        # Apply momentum adjustment
        momentum_factor = 5.0 * momentum  # Points impact
        
        # Calculate predicted final scores
        predicted_home = home_score + (home_avg * remaining_factor) + (momentum_factor if momentum > 0 else 0)
        predicted_away = away_score + (away_avg * remaining_factor) + (-momentum_factor if momentum < 0 else 0)
        
        # Calculate win probability with momentum influence
        momentum_adjusted_diff = (predicted_home - predicted_away) + (momentum * 8.0)
        win_prob = 1.0 / (1.0 + np.exp(-0.1 * momentum_adjusted_diff))
        
        return {
            'game_id': game_data.get('game_id'),
            'home_team': home_team,
            'away_team': away_team,
            'current_quarter': current_quarter,
            'predicted_home_score': predicted_home,
            'predicted_away_score': predicted_away,
            'win_probability': win_prob,
            'prediction_confidence': 0.6 + (0.1 * current_quarter),  # Slightly lower confidence due to momentum volatility
            'momentum_adjustment': momentum_factor
        }


class LateQuartersStrategy(PredictionStrategy):
    """Strategy specialized for late quarters (Q3-Q4)."""
    
    def __init__(self):
        super().__init__(
            name="Late Quarters Strategy",
            description="Specialized model for Q3-Q4",
            preferred_contexts=["late_game", "second_half", "close_game_late"],
            model_key="late_quarters_model"
        )
    
    def predict(self, game_data: Dict[str, Any], model: Any = None, team_avgs: Dict[str, float] = None) -> Dict[str, Any]:
        current_quarter = game_data.get('current_quarter', 0)
        home_team = game_data.get('home_team', '')
        away_team = game_data.get('away_team', '')
        
        # Get current score
        home_score = float(game_data.get('home_score', 0))
        away_score = float(game_data.get('away_score', 0))
        
        # Get time remaining if available
        time_remaining = float(game_data.get('time_remaining', 12))
        
        # For late quarters, current score carries more weight
        remaining_minutes = (4 - current_quarter) * 12 + time_remaining
        total_minutes = 48.0
        remaining_factor = remaining_minutes / total_minutes
        
        # Get team scoring rates
        home_scoring_rate = home_score / (1 - remaining_factor) if remaining_factor < 1 else team_avgs.get(home_team, 105.0)
        away_scoring_rate = away_score / (1 - remaining_factor) if remaining_factor < 1 else team_avgs.get(away_team, 105.0)
        
        # Calculate predicted final scores
        predicted_home = home_score + (home_scoring_rate * remaining_factor)
        predicted_away = away_score + (away_scoring_rate * remaining_factor)
        
        # Calculate win probability
        score_diff = predicted_home - predicted_away
        # Late quarter predictions should have more confidence in current leader
        win_prob = 1.0 / (1.0 + np.exp(-0.12 * score_diff))
        
        return {
            'game_id': game_data.get('game_id'),
            'home_team': home_team,
            'away_team': away_team,
            'current_quarter': current_quarter,
            'predicted_home_score': predicted_home,
            'predicted_away_score': predicted_away,
            'win_probability': win_prob,
            'prediction_confidence': 0.8 + (0.05 * current_quarter)  # Higher confidence in late quarters
        }


class BlowoutStrategy(PredictionStrategy):
    """Strategy specialized for blowout games."""
    
    def __init__(self):
        super().__init__(
            name="Blowout Specialist",
            description="Model specialized for predicting blowout games",
            preferred_contexts=["blowout", "large_lead"],
            model_key="blowout_model"
        )
    
    def predict(self, game_data: Dict[str, Any], model: Any = None, team_avgs: Dict[str, float] = None) -> Dict[str, Any]:
        current_quarter = game_data.get('current_quarter', 0)
        home_team = game_data.get('home_team', '')
        away_team = game_data.get('away_team', '')
        
        # Get current score
        home_score = float(game_data.get('home_score', 0))
        away_score = float(game_data.get('away_score', 0))
        
        # Calculate current differential
        score_diff = home_score - away_score
        abs_diff = abs(score_diff)
        
        # Calculate scoring rate for teams in blowout situations
        remaining_factor = 1.0 - (0.25 * current_quarter)
        
        # In blowouts, the leading team usually slows down and trailing team may catch up slightly
        regression_factor = 0.7  # How much the blowout continues at same pace
        
        # Points per quarter so far
        ppq_leader = home_score / current_quarter if score_diff > 0 else away_score / current_quarter
        ppq_trailer = away_score / current_quarter if score_diff > 0 else home_score / current_quarter
        
        # Adjust rates for blowout pattern
        adjusted_ppq_leader = ppq_leader * regression_factor
        adjusted_ppq_trailer = ppq_trailer * (1 + (1 - regression_factor))  # Trailing team gets slight boost
        
        # Calculate remaining points
        if score_diff > 0:  # Home team leading
            remaining_home_points = adjusted_ppq_leader * 4 * remaining_factor
            remaining_away_points = adjusted_ppq_trailer * 4 * remaining_factor
        else:  # Away team leading
            remaining_home_points = adjusted_ppq_trailer * 4 * remaining_factor
            remaining_away_points = adjusted_ppq_leader * 4 * remaining_factor
        
        # Calculate predicted final scores
        predicted_home = home_score + remaining_home_points
        predicted_away = away_score + remaining_away_points
        
        # Blowouts have very high win probability for the leader
        final_diff = predicted_home - predicted_away
        win_prob = 1.0 / (1.0 + np.exp(-0.15 * final_diff))  # Steeper curve for higher certainty
        
        return {
            'game_id': game_data.get('game_id'),
            'home_team': home_team,
            'away_team': away_team,
            'current_quarter': current_quarter,
            'predicted_home_score': predicted_home,
            'predicted_away_score': predicted_away,
            'win_probability': win_prob,
            'prediction_confidence': 0.85  # High confidence in blowout outcome
        }


class CloseGameStrategy(PredictionStrategy):
    """Strategy specialized for close games in critical moments."""
    
    def __init__(self):
        super().__init__(
            name="Close Game Specialist",
            description="Model focused on close games in critical moments",
            preferred_contexts=["close_game", "clutch_time", "final_minutes"],
            model_key="close_game_model"
        )
    
    def predict(self, game_data: Dict[str, Any], model: Any = None, team_avgs: Dict[str, float] = None) -> Dict[str, Any]:
        current_quarter = game_data.get('current_quarter', 0)
        home_team = game_data.get('home_team', '')
        away_team = game_data.get('away_team', '')
        
        # Get current score
        home_score = float(game_data.get('home_score', 0))
        away_score = float(game_data.get('away_score', 0))
        
        # Get time remaining
        time_remaining = float(game_data.get('time_remaining', 12))
        minutes_played = 48 - ((4 - current_quarter) * 12 + time_remaining)
        
        # Calculate current pace
        total_points = home_score + away_score
        points_per_minute = total_points / minutes_played if minutes_played > 0 else 4.4  # ~105 per team full game
        
        # In close games, especially late, current lead is very significant
        lead_weight = min(0.7, 0.4 + (current_quarter * 0.1))  # Increases with quarter, max 0.7
        
        # Remaining points based on pace
        remaining_minutes = (4 - current_quarter) * 12 + time_remaining
        remaining_points = points_per_minute * remaining_minutes
        
        # Team strength from averages
        home_strength = team_avgs.get(home_team, 105.0) / (team_avgs.get(home_team, 105.0) + team_avgs.get(away_team, 105.0))
        away_strength = 1 - home_strength
        
        # Distribute remaining points by team strength
        remaining_home_points = remaining_points * home_strength
        remaining_away_points = remaining_points * away_strength
        
        # Add slight bonus for home team in close games
        if abs(home_score - away_score) <= 5:
            home_clutch_advantage = 2.0  # Small home advantage in clutch
            remaining_home_points += home_clutch_advantage
            remaining_away_points -= home_clutch_advantage
        
        # Calculate predicted final scores
        predicted_home = home_score + remaining_home_points
        predicted_away = away_score + remaining_away_points
        
        # Win probability calculation - close games have higher uncertainty
        score_diff = predicted_home - predicted_away
        # Use lower coefficient for higher uncertainty
        win_prob = 1.0 / (1.0 + np.exp(-0.07 * score_diff))
        
        return {
            'game_id': game_data.get('game_id'),
            'home_team': home_team,
            'away_team': away_team,
            'current_quarter': current_quarter,
            'predicted_home_score': predicted_home,
            'predicted_away_score': predicted_away,
            'win_probability': win_prob,
            'prediction_confidence': 0.6  # Lower confidence in close games
        }


class BasicExtrapolationStrategy(PredictionStrategy):
    """Simple fallback strategy that uses basic extrapolation."""
    
    def __init__(self):
        super().__init__(
            name="Basic Score Extrapolation",
            description="Simple extrapolation based on current score and averages",
            preferred_contexts=[],  # Fallback strategy, works in any context
            model_key=None  # Doesn't require a specific model
        )
    
    def predict(self, game_data: Dict[str, Any], model: Any = None, team_avgs: Dict[str, float] = None) -> Dict[str, Any]:
        current_quarter = game_data.get('current_quarter', 0)
        home_team = game_data.get('home_team', '')
        away_team = game_data.get('away_team', '')
        
        # Get current score
        home_score = float(game_data.get('home_score', 0))
        away_score = float(game_data.get('away_score', 0))
        
        # Get team averages
        home_avg = team_avgs.get(home_team, 105.0) if team_avgs else 105.0
        away_avg = team_avgs.get(away_team, 105.0) if team_avgs else 105.0
        
        # Simple extrapolation based on current quarter
        if current_quarter == 0:
            # Pregame
            predicted_home = home_avg
            predicted_away = away_avg
        else:
            # In-game extrapolation
            remaining_factor = 1.0 - (0.25 * current_quarter)
            predicted_home = home_score + (home_avg * remaining_factor)
            predicted_away = away_score + (away_avg * remaining_factor)
        
        # Calculate win probability
        score_diff = predicted_home - predicted_away
        win_prob = 1.0 / (1.0 + np.exp(-0.08 * score_diff))
        
        return {
            'game_id': game_data.get('game_id'),
            'home_team': home_team,
            'away_team': away_team,
            'current_quarter': current_quarter,
            'predicted_home_score': predicted_home,
            'predicted_away_score': predicted_away,
            'win_probability': win_prob,
            'prediction_confidence': 0.5  # Moderate confidence for basic extrapolation
        }


class StrategyRegistry:
    """
    Registry for prediction strategies with dynamic registration.
    """
    
    def __init__(self):
        self.strategies = {}
        self.fallback_strategy = None
    
    def register(self, strategy_id: str, strategy: PredictionStrategy, fallback_id: Optional[str] = None):
        """
        Register a new prediction strategy.
        
        Args:
            strategy_id: Unique identifier for the strategy
            strategy: Strategy object to register
            fallback_id: ID of fallback strategy to use if this one fails
        """
        self.strategies[strategy_id] = strategy
        
        # Set fallback strategy if specified and exists
        if fallback_id and fallback_id in self.strategies:
            strategy.set_fallback(self.strategies[fallback_id])
        
        # If this is the first strategy, make it the default fallback
        if len(self.strategies) == 1:
            self.fallback_strategy = strategy
    
    def get_strategy(self, strategy_id: str) -> Optional[PredictionStrategy]:
        """Get a strategy by ID."""
        return self.strategies.get(strategy_id)
    
    def get_fallback_strategy(self) -> Optional[PredictionStrategy]:
        """Get the fallback strategy."""
        return self.fallback_strategy
    
    def all_strategies(self) -> Dict[str, PredictionStrategy]:
        """Get all registered strategies."""
        return self.strategies


class ErrorMetricsCollector:
    """
    Collects and analyzes prediction error metrics.
    """
    
    def __init__(self):
        self.strategy_errors = {}
        self.context_errors = {}
        self.quarter_errors = {}
        self.combined_errors = {}
    
    def add_error_record(self, 
                        prediction: Dict[str, Any],
                        actual: Dict[str, Any]):
        """
        Add an error record for a prediction.
        
        Args:
            prediction: Prediction dictionary
            actual: Actual results dictionary
        """
        # Ensure we have the required fields
        if 'strategy_id' not in prediction or 'context' not in prediction:
            return
        
        strategy_id = prediction['strategy_id']
        context = prediction['context']
        
        # Calculate error metrics
        try:
            pred_home_score = float(prediction.get('predicted_home_score', 0))
            pred_away_score = float(prediction.get('predicted_away_score', 0))
            
            actual_home_score = float(actual.get('home_score', 0))
            actual_away_score = float(actual.get('away_score', 0))
            
            # Mean Absolute Error
            home_error = abs(pred_home_score - actual_home_score)
            away_error = abs(pred_away_score - actual_away_score)
            mae = (home_error + away_error) / 2
            
            # Bias (over/under prediction)
            home_bias = pred_home_score - actual_home_score
            away_bias = pred_away_score - actual_away_score
            bias = (home_bias + away_bias) / 2
            
            # Winner prediction accuracy
            actual_winner = 'home' if actual_home_score > actual_away_score else 'away'
            predicted_winner = 'home' if pred_home_score > pred_away_score else 'away'
            winner_correct = int(actual_winner == predicted_winner)
            
            # Create error record
            error_record = {
                'timestamp': datetime.now().isoformat(),
                'game_id': prediction.get('game_id'),
                'strategy_id': strategy_id,
                'primary_context': context.get('primary_context', 'unknown'),
                'quarter': context.get('current_quarter', 0),
                'mae': mae,
                'bias': bias,
                'winner_correct': winner_correct,
                'pred_confidence': prediction.get('prediction_confidence', 0.8)
            }
            
            # Add to strategy errors
            if strategy_id not in self.strategy_errors:
                self.strategy_errors[strategy_id] = []
            self.strategy_errors[strategy_id].append(error_record)
            
            # Add to context errors
            primary_context = context.get('primary_context', 'unknown')
            if primary_context not in self.context_errors:
                self.context_errors[primary_context] = []
            self.context_errors[primary_context].append(error_record)
            
            # Add to quarter errors
            quarter = context.get('current_quarter', 0)
            if quarter not in self.quarter_errors:
                self.quarter_errors[quarter] = []
            self.quarter_errors[quarter].append(error_record)
            
            # Add to combined errors
            combined_key = f"{strategy_id}_{primary_context}"
            if combined_key not in self.combined_errors:
                self.combined_errors[combined_key] = []
            self.combined_errors[combined_key].append(error_record)
            
        except Exception as e:
            print(f"Error adding metrics: {str(e)}")
    
    def update_strategy_metrics(self, registry: StrategyRegistry):
        """
        Update strategy success rates and error metrics.
        
        Args:
            registry: Strategy registry to update
        """
        for strategy_id, records in self.strategy_errors.items():
            if not records:
                continue
                
            strategy = registry.get_strategy(strategy_id)
            if not strategy:
                continue
                
            # Update success rate
            strategy.usage_count = len(records)
            correct_predictions = sum(r['winner_correct'] for r in records)
            strategy.success_rate = correct_predictions / len(records) if records else 0
            
            # Update error rate
            total_error = sum(r['mae'] for r in records)
            strategy.avg_error = total_error / len(records) if records else None
    
    def get_error_summary(self) -> Dict[str, Any]:
        """Get summary of error metrics."""
        summary = {
            'by_strategy': {},
            'by_context': {},
            'by_quarter': {},
            'overall': {
                'records': 0,
                'avg_mae': None,
                'avg_winner_accuracy': None
            }
        }
        
        # Collect all records
        all_records = []
        
        # Strategy metrics
        for strategy_id, records in self.strategy_errors.items():
            if not records:
                continue
                
            total_mae = sum(r['mae'] for r in records)
            correct_predictions = sum(r['winner_correct'] for r in records)
            
            summary['by_strategy'][strategy_id] = {
                'records': len(records),
                'avg_mae': total_mae / len(records),
                'winner_accuracy': correct_predictions / len(records)
            }
            
            all_records.extend(records)
        
        # Context metrics
        for context, records in self.context_errors.items():
            if not records:
                continue
                
            total_mae = sum(r['mae'] for r in records)
            correct_predictions = sum(r['winner_correct'] for r in records)
            
            summary['by_context'][context] = {
                'records': len(records),
                'avg_mae': total_mae / len(records),
                'winner_accuracy': correct_predictions / len(records)
            }
        
        # Quarter metrics
        for quarter, records in self.quarter_errors.items():
            if not records:
                continue
                
            total_mae = sum(r['mae'] for r in records)
            correct_predictions = sum(r['winner_correct'] for r in records)
            
            summary['by_quarter'][quarter] = {
                'records': len(records),
                'avg_mae': total_mae / len(records),
                'winner_accuracy': correct_predictions / len(records)
            }
        
        # Overall metrics
        if all_records:
            total_mae = sum(r['mae'] for r in all_records)
            correct_predictions = sum(r['winner_correct'] for r in all_records)
            
            summary['overall'] = {
                'records': len(all_records),
                'avg_mae': total_mae / len(all_records),
                'avg_winner_accuracy': correct_predictions / len(all_records)
            }
        
        return summary
    
    def visualize_error_metrics(self):
        """
        Visualize error metrics by strategy and quarter.
        """
        error_summary = self.get_error_summary()
        
        # Skip if no data
        if not error_summary['by_strategy']:
            print("No error metrics available for visualization.")
            return
        
        # Create figure with two subplots
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Strategy MAE Comparison
        strategies = list(error_summary['by_strategy'].keys())
        mae_values = [error_summary['by_strategy'][s]['avg_mae'] for s in strategies]
        accuracy_values = [error_summary['by_strategy'][s]['winner_accuracy'] for s in strategies]
        records_count = [error_summary['by_strategy'][s]['records'] for s in strategies]
        
        # Sort by MAE (lower is better)
        sorted_indices = np.argsort(mae_values)
        sorted_strategies = [strategies[i] for i in sorted_indices]
        sorted_mae = [mae_values[i] for i in sorted_indices]
        sorted_accuracy = [accuracy_values[i] for i in sorted_indices]
        sorted_records = [records_count[i] for i in sorted_indices]
        
        # MAE Plot
        bars1 = ax1.barh(sorted_strategies, sorted_mae, color='skyblue')
        ax1.set_xlabel('Mean Absolute Error')
        ax1.set_title('Strategy Prediction Error (lower is better)')
        ax1.grid(axis='x', alpha=0.3)
        
        # Add sample size labels
        for i, bar in enumerate(bars1):
            ax1.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2, 
                    f"n={sorted_records[i]}", va='center')
        
        # Accuracy Plot
        bars2 = ax2.barh(sorted_strategies, sorted_accuracy, color='lightgreen')
        ax2.set_xlabel('Winner Prediction Accuracy')
        ax2.set_title('Strategy Winner Accuracy (higher is better)')
        ax2.set_xlim(0, 1)
        ax2.grid(axis='x', alpha=0.3)
        
        # Add percentage labels
        for i, bar in enumerate(bars2):
            ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
                    f"{sorted_accuracy[i]:.1%}", va='center')
        
        plt.tight_layout()
        plt.show()
        
        # Plot error by quarter if available
        if error_summary['by_quarter']:
            plt.figure(figsize=(10, 6))
            
            quarters = sorted(error_summary['by_quarter'].keys(), key=lambda x: int(x))
            quarter_mae = [error_summary['by_quarter'][q]['avg_mae'] for q in quarters]
            quarter_acc = [error_summary['by_quarter'][q]['winner_accuracy'] for q in quarters]
            
            plt.plot(quarters, quarter_mae, 'o-', label='MAE', color='red')
            plt.xlabel('Quarter')
            plt.ylabel('Mean Absolute Error')
            plt.grid(alpha=0.3)
            
            # Add second y-axis for accuracy
            ax3 = plt.gca().twinx()
            ax3.plot(quarters, quarter_acc, 's-', label='Accuracy', color='green')
            ax3.set_ylabel('Winner Accuracy')
            ax3.set_ylim(0, 1)
            
            # Add labels
            for i, q in enumerate(quarters):
                plt.text(q, quarter_mae[i] + 0.5, f"{quarter_mae[i]:.1f}", ha='center')
                ax3.text(q, quarter_acc[i] + 0.02, f"{quarter_acc[i]:.1%}", ha='center')
            
            # Combine legends
            lines1, labels1 = plt.gca().get_legend_handles_labels()
            lines2, labels2 = ax3.get_legend_handles_labels()
            ax3.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
            
            plt.title('Prediction Performance by Quarter')
            plt.tight_layout()
            plt.show()


class StrategySelector:
    """
    Selects the most appropriate prediction strategy based on game context.
    """
    
    def __init__(self, registry: StrategyRegistry, model_registry: Dict[str, Any] = None):
        """
        Initialize the strategy selector.
        
        Args:
            registry: Strategy registry containing available strategies
            model_registry: Dictionary mapping model keys to model objects
        """
        self.registry = registry
        self.model_registry = model_registry or {}
        self.context_analyzer = GameContextAnalyzer()
        self.metrics_collector = ErrorMetricsCollector()
        self.selection_history = []
    
    def select_strategy(self, game_data: Dict[str, Any]) -> Dict[str, Any]:
        """
        Select the most appropriate prediction strategy based on game context.
        
        Args:
            game_data: Dictionary with game information
            
        Returns:
            Dict with selected strategy and context information
        """
        # Analyze the game context
        context = self.context_analyzer.analyze(game_data)
        
        # Score strategies based on context match and past performance
        strategy_scores = {}
        
        for strategy_id, strategy in self.registry.all_strategies().items():
            # Base score
            score = 1.0
            
            # Context match bonus
            preferred_contexts = strategy.preferred_contexts
            current_contexts = context['contexts']
            context_matches = [c for c in current_contexts if c in preferred_contexts]
            
            context_score = len(context_matches) * 2.0
            
            # Bonus for matching the primary context
            if context['primary_context'] in preferred_contexts:
                context_score += 3.0
            
            # Performance history bonus
            success_bonus = 0.0
            if strategy.usage_count > 0:
                success_rate = strategy.success_rate
                # Scale by usage - more confident with more samples
                usage_scale = min(1.0, strategy.usage_count / 10.0)
                success_bonus = success_rate * 5.0 * usage_scale
            
            # Error metrics penalty
            error_penalty = 0.0
            if strategy.avg_error is not None:
                # Scale error to a 0-5 range (assuming error is in points)
                max_acceptable_error = 15.0
                error_scale = min(strategy.avg_error / max_acceptable_error, 1.0)
                error_penalty = error_scale * 5.0
            
            # Calculate total score
            total_score = score + context_score + success_bonus - error_penalty
            
            strategy_scores[strategy_id] = {
                'total_score': total_score,
                'context_score': context_score,
                'success_bonus': success_bonus,
                'error_penalty': error_penalty
            }
        
        # Select the strategy with the highest score
        if not strategy_scores:
            # No strategies available, use fallback
            fallback = self.registry.get_fallback_strategy()
            selected_strategy_id = "basic_extrapolation"
            selected_strategy = fallback
        else:
            selected_strategy_id = max(strategy_scores, key=lambda k: strategy_scores[k]['total_score'])
            selected_strategy = self.registry.get_strategy(selected_strategy_id)
        
        # Check if model is available for selected strategy
        model_key = selected_strategy.model_key
        fallback_used = False
        fallback_reason = None
        
        if model_key and model_key not in self.model_registry:
            # Model not available, use fallback
            fallback = selected_strategy.fallback_strategy or self.registry.get_fallback_strategy()
            if fallback:
                fallback_used = True
                fallback_reason = f"Model {model_key} not available"
                selected_strategy = fallback
                # Update selected strategy ID
                for sid, strategy in self.registry.all_strategies().items():
                    if strategy == fallback:
                        selected_strategy_id = sid
                        break
        
        # Record the selection
        selection_record = {
            'timestamp': datetime.now().isoformat(),
            'game_id': game_data.get('game_id'),
            'selected_strategy_id': selected_strategy_id,
            'context': context,
            'strategy_scores': strategy_scores,
            'fallback_used': fallback_used,
            'fallback_reason': fallback_reason
        }
        self.selection_history.append(selection_record)
        
        # Increment usage count
        selected_strategy.usage_count += 1
        
        # Return selection result
        return {
            'strategy_id': selected_strategy_id,
            'strategy': selected_strategy,
            'context': context,
            'selection_record': selection_record
        }
    
    def execute_prediction(self, 
                        game_data: Dict[str, Any],
                        team_avgs: Dict[str, float] = None) -> Dict[str, Any]:
        """
        Execute prediction for a game using the selected strategy.
        
        Args:
            game_data: Dictionary with game information
            team_avgs: Dictionary of team scoring averages
            
        Returns:
            Dict with prediction results and metadata
        """
        # Select appropriate strategy
        strategy_selection = self.select_strategy(game_data)
        strategy_id = strategy_selection['strategy_id']
        strategy = strategy_selection['strategy']
        context = strategy_selection['context']
        
        # Track execution time
        start_time = time.time()
        
        try:
            # Get relevant model if needed
            model = None
            if strategy.model_key is not None and self.model_registry is not None:
                model = self.model_registry.get(strategy.model_key)
            
            # Execute prediction
            prediction_result = strategy.predict(game_data, model, team_avgs)
            
            # Track execution time
            execution_time = time.time() - start_time
            
            # Add metadata to result
            prediction_result.update({
                'strategy_id': strategy_id,
                'strategy_name': strategy.name,
                'strategy_description': strategy.description,
                'context': context,
                'execution_time': execution_time,
                'prediction_timestamp': datetime.now().isoformat()
            })
            
            # Return the prediction with all metadata
            return prediction_result
            
        except Exception as e:
            # Error during execution
            error_msg = str(e)
            print(f"Error executing strategy {strategy_id}: {error_msg}")
            
            # Try to use fallback strategy
            fallback = strategy.fallback_strategy or self.registry.get_fallback_strategy()
            
            if fallback:
                try:
                    print(f"Using fallback strategy: {fallback.name}")
                    
                    # Get model for fallback if needed
                    fallback_model = None
                    if fallback.model_key is not None and self.model_registry is not None:
                        fallback_model = self.model_registry.get(fallback.model_key)
                    
                    # Execute fallback prediction
                    fallback_result = fallback.predict(game_data, fallback_model, team_avgs)
                    
                    # Get fallback strategy ID
                    fallback_id = "unknown_fallback"
                    for sid, s in self.registry.all_strategies().items():
                        if s == fallback:
                            fallback_id = sid
                            break
                    
                    # Track execution time
                    execution_time = time.time() - start_time
                    
                    # Add metadata about fallback
                    fallback_result.update({
                        'strategy_id': fallback_id,
                        'strategy_name': fallback.name,
                        'original_strategy_id': strategy_id,
                        'fallback_reason': error_msg,
                        'context': context,
                        'execution_time': execution_time,
                        'prediction_timestamp': datetime.now().isoformat()
                    })
                    
                    # Increment usage count for fallback
                    fallback.usage_count += 1
                    
                    return fallback_result
                    
                except Exception as fallback_error:
                    # Both original and fallback failed
                    print(f"Fallback strategy also failed: {str(fallback_error)}")
                    return {
                        'game_id': game_data.get('game_id'),
                        'error': error_msg,
                        'fallback_error': str(fallback_error),
                        'execution_time': time.time() - start_time
                    }
            
            # No fallback available
            return {
                'game_id': game_data.get('game_id'),
                'error': error_msg,
                'execution_time': time.time() - start_time
            }
    
    def update_strategy_metrics(self, actual_results: Dict[str, Dict[str, Any]]):
        """
        Update strategy metrics based on actual game results.
        
        Args:
            actual_results: Dictionary mapping game_id to actual results
        """
        # Update metrics in the collector
        self.metrics_collector.update_strategy_metrics(self.registry)
    
    def get_error_summary(self) -> Dict[str, Any]:
        """Get summary of prediction errors."""
        return self.metrics_collector.get_error_summary()
    
    def visualize_performance(self):
        """Visualize strategy performance metrics."""
        self.metrics_collector.visualize_error_metrics()


# Setup and demonstration 
def setup_prediction_system():
    """Set up the prediction system with registered strategies."""
    # Create registry and register strategies
    registry = StrategyRegistry()
    
    # Create and register strategies
    registry.register("early_quarters", EarlyQuartersStrategy(), "basic_extrapolation")
    registry.register("momentum", MomentumStrategy(), "late_quarters")
    registry.register("late_quarters", LateQuartersStrategy(), "basic_extrapolation")
    registry.register("blowout", BlowoutStrategy(), "basic_extrapolation")
    registry.register("close_game", CloseGameStrategy(), "late_quarters")
    registry.register("basic_extrapolation", BasicExtrapolationStrategy())
    
    # Create strategy selector
    selector = StrategySelector(registry)
    
    return selector

def demo_prediction_system():
    """Demonstrate the prediction system with sample games."""
    # Set up the system
    selector = setup_prediction_system()
    
    # Create sample game scenarios
    sample_games = [
        # Pregame
        {
            'game_id': 1001,
            'home_team': 'Lakers',
            'away_team': 'Celtics',
            'current_quarter': 0,
            'home_score': 0,
            'away_score': 0
        },
        # Early close game
        {
            'game_id': 1002,
            'home_team': 'Warriors',
            'away_team': 'Bucks',
            'current_quarter': 1,
            'home_q1': 28,
            'away_q1': 26,
            'home_score': 28,
            'away_score': 26,
            'cumulative_momentum': 0.2
        },
        # Blowout in Q3
        {
            'game_id': 1003,
            'home_team': 'Heat',
            'away_team': 'Nuggets',
            'current_quarter': 3,
            'home_q1': 32, 'home_q2': 35, 'home_q3': 30,
            'away_q1': 22, 'away_q2': 20, 'away_q3': 18,
            'home_score': 97,
            'away_score': 60,
            'cumulative_momentum': 0.7
        },
        # Close late game
        {
            'game_id': 1004,
            'home_team': 'Nets',
            'away_team': 'Suns',
            'current_quarter': 4,
            'home_q1': 28, 'home_q2': 25, 'home_q3': 24, 'home_q4': 18,
            'away_q1': 30, 'away_q2': 27, 'away_q3': 22, 'away_q4': 20,
            'home_score': 95,
            'away_score': 99,
            'cumulative_momentum': -0.3,
            'time_remaining': 3.5
        },
        # Comeback potential
        {
            'game_id': 1005,
            'home_team': 'Bulls',
            'away_team': 'Mavs',
            'current_quarter': 3,
            'home_q1': 20, 'home_q2': 22, 'home_q3': 32,
            'away_q1': 30, 'away_q2': 33, 'away_q3': 18,
            'home_score': 74,
            'away_score': 81,
            'cumulative_momentum': 0.6,  # Strong home momentum (comeback potential)
            'time_remaining': 0.5
        }
    ]
    
    # Team averages for demonstration
    team_avgs = {
        'Lakers': 112.5,
        'Celtics': 110.8,
        'Warriors': 116.2,
        'Bucks': 114.5,
        'Heat': 108.3,
        'Nuggets': 115.7,
        'Nets': 112.0,
        'Suns': 113.5,
        'Bulls': 108.0,
        'Mavs': 112.8
    }
    
    # Process each game
    print("Strategy Selection and Prediction Examples\n")
    print("=" * 80)
    
    for game in sample_games:
        # Analyze context
        context = selector.context_analyzer.analyze(game)
        
        print(f"\nGame {game['game_id']}: {game['home_team']} vs {game['away_team']} (Q{game['current_quarter']})")
        print(f"Primary Context: {context['primary_context']}")
        print(f"All Contexts: {', '.join(context['contexts'])}")
        
        # Select and execute strategy
        prediction = selector.execute_prediction(game, team_avgs)
        
        print(f"Selected Strategy: {prediction['strategy_name']} ({prediction['strategy_id']})")
        print(f"Strategy Description: {prediction['strategy_description']}")
        
        print("Prediction Results:")
        print(f"  Home Score: {prediction['predicted_home_score']:.1f}")
        print(f"  Away Score: {prediction['predicted_away_score']:.1f}")
        print(f"  Win Probability: {prediction['win_probability']:.1%}")
        print(f"  Confidence: {prediction['prediction_confidence']:.2f}")
        
        if 'execution_time' in prediction:
            print(f"  Execution Time: {prediction['execution_time']*1000:.2f} ms")
        
        print("-" * 80)
    
    # Simulate some validations for demo purposes
    print("\nSimulating validation with actual results...")
    
    # Mock actual results
    actual_results = {
        1001: {'game_id': 1001, 'home_score': 110, 'away_score': 105},
        1002: {'game_id': 1002, 'home_score': 115, 'away_score': 110},
        1003: {'game_id': 1003, 'home_score': 120, 'away_score': 85},
        1004: {'game_id': 1004, 'home_score': 105, 'away_score': 110},
        1005: {'game_id': 1005, 'home_score': 110, 'away_score': 105}  # Comeback completed
    }
    
    # Process each game again and update metrics
    for game in sample_games:
        # Execute prediction
        prediction = selector.execute_prediction(game, team_avgs)
        
        # Update error metrics
        game_id = game['game_id']
        if game_id in actual_results:
            selector.metrics_collector.add_error_record(prediction, actual_results[game_id])
    
    # Update strategy metrics
    selector.update_strategy_metrics(actual_results)
    
    # Visualize performance
    selector.visualize_performance()
    
    return {
        'selector': selector,
        'sample_games': sample_games,
        'error_summary': selector.get_error_summary()
    }

# Run the demonstration
if __name__ == "__main__":
    demo_results = demo_prediction_system()

In [ ]:
# Cell 12: Refactored Dashboard with Confidence Display and Error Handling

from IPython.display import clear_output, display, HTML
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
import pandas as pd
import warnings
import traceback

class PredictionDashboard:
    """
    A dashboard class for visualizing game predictions with confidence metrics.
    Separates data preparation from visualization for better maintainability.
    """
    
    def __init__(self, use_plotly=False):
        """
        Initialize the dashboard.
        
        Args:
            use_plotly: If True, tries to use Plotly for interactive visualizations
        """
        self.use_plotly = use_plotly
        self._check_plotly_available()
        self.default_colors = {
            'home': '#1f77b4',  # blue
            'away': '#d62728',  # red
            'confidence': '#2ca02c',  # green
            'background': '#f8f9fa',
            'grid': '#dddddd'
        }
    
    def _check_plotly_available(self):
        """Check if Plotly is available and set flag accordingly."""
        if self.use_plotly:
            try:
                import plotly.graph_objects as go
                import plotly.express as px
                self.plotly_available = True
            except ImportError:
                warnings.warn("Plotly not available. Using Matplotlib instead.")
                self.plotly_available = False
        else:
            self.plotly_available = False
    
    def prepare_game_data(self, team_predictions, prediction_history=None):
        """
        Prepare game data for visualization.
        
        Args:
            team_predictions: DataFrame with team predictions
            prediction_history: Dictionary mapping game_id to list of predictions
            
        Returns:
            List of prepared game data dictionaries
        """
        if team_predictions is None or team_predictions.empty:
            return []
        
        prepared_games = []
        
        for _, game in team_predictions.iterrows():
            try:
                # Extract basic game information
                game_id = game.get('game_id', None)
                if game_id is None:
                    continue  # Skip games without ID
                
                home_team = game.get('home_team', 'Home')
                away_team = game.get('away_team', 'Away')
                current_quarter = int(game.get('current_quarter', 0))
                
                # Get current scores
                current_home_score = float(game.get('current_home_score', game.get('home_score', 0)) or 0)
                current_away_score = float(game.get('current_away_score', game.get('away_score', 0)) or 0)
                
                # Get predicted scores
                predicted_home_score = float(game.get('predicted_home_score', 0))
                predicted_away_score = float(game.get('predicted_away_score', 0))
                
                # Get confidence metrics if available
                confidence = float(game.get('prediction_confidence', 0.5))
                win_probability = float(game.get('win_probability', 0.5))
                
                # Calculate margins and totals
                current_margin = current_home_score - current_away_score
                predicted_margin = predicted_home_score - predicted_away_score
                current_total = current_home_score + current_away_score
                predicted_total = predicted_home_score + predicted_away_score
                
                # Get prediction history for this game if available
                game_history = None
                if prediction_history is not None and game_id in prediction_history:
                    game_history = prediction_history[game_id]
                
                # Prepare game data
                game_data = {
                    'game_id': game_id,
                    'home_team': home_team,
                    'away_team': away_team,
                    'current_quarter': current_quarter,
                    'current_scores': {
                        'home': current_home_score,
                        'away': current_away_score,
                        'margin': current_margin,
                        'total': current_total
                    },
                    'predicted_scores': {
                        'home': predicted_home_score,
                        'away': predicted_away_score,
                        'margin': predicted_margin,
                        'total': predicted_total
                    },
                    'metrics': {
                        'confidence': confidence,
                        'win_probability': win_probability
                    },
                    'history': game_history
                }
                
                prepared_games.append(game_data)
                
            except Exception as e:
                warnings.warn(f"Error preparing game data: {str(e)}")
                continue
        
        return prepared_games
    
    def create_live_dashboard(self, team_predictions, prediction_history=None, include_text=True):
        """
        Create a dashboard visualization of live game predictions with confidence metrics.
        
        Args:
            team_predictions: DataFrame with team predictions
            prediction_history: Dictionary mapping game_id to list of predictions
            include_text: Whether to include text summary in addition to visualization
            
        Returns:
            None (displays visualization in notebook)
        """
        try:
            # Clear previous output
            clear_output(wait=True)
            
            # Prepare data
            prepared_games = self.prepare_game_data(team_predictions, prediction_history)
            
            if not prepared_games:
                print("No live games to display.")
                return
            
            # Create visualizations
            if self.plotly_available:
                self._create_plotly_dashboard(prepared_games)
            else:
                self._create_matplotlib_dashboard(prepared_games)
            
            # Display text summary if requested
            if include_text:
                self._display_text_summary(prepared_games)
            
        except Exception as e:
            print(f"Error creating dashboard: {str(e)}")
            traceback.print_exc()
    
    def _create_matplotlib_dashboard(self, prepared_games):
        """Create dashboard using Matplotlib."""
        n_games = len(prepared_games)
        fig, axs = plt.subplots(n_games, 2, figsize=(15, 5*n_games))
        fig.suptitle(f"Live Game Predictions - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", fontsize=16)
        
        # Handle single game case
        if n_games == 1:
            axs = np.array([axs])
        
        for i, game in enumerate(prepared_games):
            # Game scores visualization
            ax_scores = axs[i, 0]
            self._plot_game_scores(ax_scores, game)
            
            # Prediction history visualization
            ax_history = axs[i, 1]
            self._plot_prediction_history(ax_history, game)
        
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()
    
    def _plot_game_scores(self, ax, game):
        """Plot game scores in Matplotlib ax."""
        teams = [game['home_team'], game['away_team']]
        current_scores = [game['current_scores']['home'], game['current_scores']['away']]
        predicted_scores = [game['predicted_scores']['home'], game['predicted_scores']['away']]
        
        x = np.arange(len(teams))
        width = 0.35
        
        # Plot bars
        ax.bar(x - width/2, current_scores, width, label='Current', color=self.default_colors['home'])
        ax.bar(x + width/2, predicted_scores, width, label='Predicted Final', color=self.default_colors['away'])
        
        # Add labels and styling
        ax.set_xticks(x)
        ax.set_xticklabels(teams)
        ax.legend()
        
        confidence = int(game['metrics']['confidence'] * 100)
        quarter_text = 'PRE-GAME' if game['current_quarter'] == 0 else f"Q{game['current_quarter']}"
        ax.set_title(f"{game['home_team']} vs {game['away_team']} - {quarter_text} (Confidence: {confidence}%)")
        
        # Add value labels
        for j, v in enumerate(current_scores):
            ax.text(j - width/2, v + 1, str(int(v)), ha='center')
        for j, v in enumerate(predicted_scores):
            ax.text(j + width/2, v + 1, f"{v:.1f}", ha='center')
    
    def _plot_prediction_history(self, ax, game):
        """Plot prediction history in Matplotlib ax."""
        history = game.get('history')
        
        if history and len(history) > 1:
            # Extract data
            timestamps = range(len(history))
            home_scores = [h.get('predicted_home_score', 0) for h in history]
            away_scores = [h.get('predicted_away_score', 0) for h in history]
            
            # Draw lines
            ax.plot(timestamps, home_scores, 'o-', label=f"{game['home_team']} Final", 
                   marker='o', color=self.default_colors['home'])
            ax.plot(timestamps, away_scores, 's-', label=f"{game['away_team']} Final", 
                   marker='s', color=self.default_colors['away'])
            
            # Add win probability if available
            if all('win_probability' in h for h in history):
                win_probs = [h.get('win_probability', 0.5) for h in history]
                ax_twin = ax.twinx()
                ax_twin.plot(timestamps, win_probs, 'd-', label='Win Prob', 
                           color=self.default_colors['confidence'])
                ax_twin.set_ylim(0, 1)
                ax_twin.set_ylabel('Win Probability')
                
                # Add legend items from both axes
                lines1, labels1 = ax.get_legend_handles_labels()
                lines2, labels2 = ax_twin.get_legend_handles_labels()
                ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
            else:
                ax.legend()
            
            # Styling
            ax.set_title("Prediction Evolution")
            ax.set_xlabel("Update Index")
            ax.set_ylabel("Predicted Score")
            ax.grid(alpha=0.3)
            
        else:
            ax.text(0.5, 0.5, "Not enough prediction history yet", 
                   ha='center', va='center', transform=ax.transAxes)
    
    def _create_plotly_dashboard(self, prepared_games):
        """Create dashboard using Plotly for interactive visualization."""
        try:
            import plotly.graph_objects as go
            import plotly.subplots as sp
            
            # Create a 2-column layout for each game
            rows = len(prepared_games)
            fig = sp.make_subplots(
                rows=rows, 
                cols=2,
                subplot_titles=[f"{g['home_team']} vs {g['away_team']}" for g in prepared_games for _ in range(2)],
                specs=[[{"type": "bar"}, {"type": "scatter"}] for _ in range(rows)]
            )
            
            for i, game in enumerate(prepared_games):
                # Add 1 to i since plotly is 1-indexed
                row = i + 1
                
                # Game scores chart
                teams = [game['home_team'], game['away_team']]
                current_scores = [game['current_scores']['home'], game['current_scores']['away']]
                predicted_scores = [game['predicted_scores']['home'], game['predicted_scores']['away']]
                
                fig.add_trace(
                    go.Bar(
                        x=teams,
                        y=current_scores,
                        name='Current',
                        marker_color='royalblue',
                        hovertemplate='%{x}: %{y}<br><extra>Current Score</extra>',
                        text=current_scores,
                        textposition='auto'
                    ),
                    row=row, col=1
                )
                
                fig.add_trace(
                    go.Bar(
                        x=teams,
                        y=predicted_scores,
                        name='Predicted Final',
                        marker_color='firebrick',
                        hovertemplate='%{x}: %{y:.1f}<br><extra>Predicted Final</extra>',
                        text=[f"{s:.1f}" for s in predicted_scores],
                        textposition='auto'
                    ),
                    row=row, col=1
                )
                
                # Add confidence indicator
                confidence = game['metrics']['confidence'] * 100
                quarter_text = 'PRE-GAME' if game['current_quarter'] == 0 else f"Q{game['current_quarter']}"
                fig.update_xaxes(title_text=f"{quarter_text} (Confidence: {confidence:.0f}%)", row=row, col=1)
                
                # Prediction history chart
                history = game.get('history')
                if history and len(history) > 1:
                    # Extract data
                    timestamps = list(range(len(history)))
                    home_scores = [h.get('predicted_home_score', 0) for h in history]
                    away_scores = [h.get('predicted_away_score', 0) for h in history]
                    
                    fig.add_trace(
                        go.Scatter(
                            x=timestamps,
                            y=home_scores,
                            mode='lines+markers',
                            name=f"{game['home_team']} Prediction",
                            marker_color='royalblue',
                            hovertemplate='Update %{x}: %{y:.1f}<br><extra>Home Prediction</extra>'
                        ),
                        row=row, col=2
                    )
                    
                    fig.add_trace(
                        go.Scatter(
                            x=timestamps,
                            y=away_scores,
                            mode='lines+markers',
                            name=f"{game['away_team']} Prediction",
                            marker_color='firebrick',
                            hovertemplate='Update %{x}: %{y:.1f}<br><extra>Away Prediction</extra>'
                        ),
                        row=row, col=2
                    )
                    
                    # Add win probability if available
                    if all('win_probability' in h for h in history):
                        win_probs = [h.get('win_probability', 0.5) * 100 for h in history]
                        
                        fig.add_trace(
                            go.Scatter(
                                x=timestamps,
                                y=win_probs,
                                mode='lines+markers',
                                name='Win Probability',
                                marker_color='green',
                                yaxis='y2',
                                hovertemplate='Update %{x}: %{y:.1f}%<br><extra>Win Probability</extra>'
                            ),
                            row=row, col=2
                        )
                        
                        # Add secondary y-axis for win probability
                        fig.update_layout(
                            yaxis2=dict(
                                title='Win Probability (%)',
                                titlefont=dict(color='green'),
                                tickfont=dict(color='green'),
                                anchor='x',
                                overlaying='y',
                                side='right',
                                range=[0, 100]
                            )
                        )
                    
                    fig.update_xaxes(title_text="Update", row=row, col=2)
                    fig.update_yaxes(title_text="Predicted Score", row=row, col=2)
                    
                else:
                    # Add placeholder text for insufficient history
                    fig.add_trace(
                        go.Scatter(
                            x=[0],
                            y=[0],
                            mode='text',
                            text=['Not enough prediction history yet'],
                            textposition='middle center',
                            hoverinfo='none'
                        ),
                        row=row, col=2
                    )
                    
            
            # Update layout
            fig.update_layout(
                title=f"Live Game Predictions - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
                showlegend=False,
                height=300 * rows,
                template='plotly_white'
            )
            
            fig.show()
            
        except Exception as e:
            warnings.warn(f"Error creating Plotly dashboard: {str(e)}. Falling back to Matplotlib.")
            traceback.print_exc()
            # Fallback to matplotlib
            self._create_matplotlib_dashboard(prepared_games)
    
    def _display_text_summary(self, prepared_games):
        """Display text summary of game predictions."""
        print(f"Live Game Predictions - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("="*80)
        
        for game in prepared_games:
            quarter_text = "Pre-game" if game['current_quarter'] == 0 else f"Quarter {game['current_quarter']}"
            print(f"\n{game['home_team']} vs {game['away_team']} - {quarter_text}")
            print(f"Current Score: {game['home_team']} {game['current_scores']['home']} - {game['away_team']} {game['current_scores']['away']}")
            print(f"Predicted Final: {game['home_team']} {game['predicted_scores']['home']:.1f} - {game['away_team']} {game['predicted_scores']['away']:.1f}")
            
            win_prob = game['metrics']['win_probability']
            favorite = game['home_team'] if win_prob > 0.5 else game['away_team']
            win_pct = max(win_prob, 1-win_prob) * 100
            print(f"Win Probability: {favorite} {win_pct:.1f}%")
            
            confidence = game['metrics']['confidence'] * 100
            print(f"Prediction Confidence: {confidence:.0f}%")
            
            print("-"*80)
    
    def get_html_dashboard(self, team_predictions, prediction_history=None):
        """
        Generate HTML for the dashboard (for web display).
        
        Args:
            team_predictions: DataFrame with team predictions
            prediction_history: Dictionary mapping game_id to list of predictions
            
        Returns:
            HTML string for the dashboard
        """
        try:
            # Prepare data
            prepared_games = self.prepare_game_data(team_predictions, prediction_history)
            
            if not prepared_games:
                return "<p>No live games to display.</p>"
            
            # Build HTML
            html = f"""
            <div class="nba-prediction-dashboard">
                <style>
                    .nba-prediction-dashboard {{
                        font-family: Arial, sans-serif;
                        max-width: 1200px;
                        margin: 0 auto;
                        padding: 10px;
                    }}
                    .dashboard-header {{
                        text-align: center;
                        margin-bottom: 20px;
                    }}
                    .games-container {{
                        display: flex;
                        flex-wrap: wrap;
                        gap: 20px;
                        justify-content: center;
                    }}
                    .game-card {{
                        border: 1px solid #ddd;
                        border-radius: 8px;
                        padding: 15px;
                        width: 350px;
                        box-shadow: 0 2px 4px rgba(0,0,0,0.1);
                        background-color: #fff;
                    }}
                    .game-header {{
                        border-bottom: 1px solid #eee;
                        padding-bottom: 10px;
                        margin-bottom: 10px;
                        font-weight: bold;
                        font-size: 1.2em;
                    }}
                    .score-display {{
                        display: flex;
                        justify-content: space-between;
                        margin: 15px 0;
                    }}
                    .team-score {{
                        text-align: center;
                        width: 45%;
                    }}
                    .team-name {{
                        font-weight: bold;
                        margin-bottom: 5px;
                    }}
                    .current-score {{
                        font-size: 1.8em;
                        font-weight: bold;
                    }}
                    .predicted-score {{
                        font-size: 1.4em;
                        color: #666;
                    }}
                    .vs-divider {{
                        display: flex;
                        align-items: center;
                        justify-content: center;
                        width: 10%;
                    }}
                    .confidence-bar {{
                        margin: 15px 0;
                        background-color: #f0f0f0;
                        border-radius: 4px;
                        height: 8px;
                    }}
                    .confidence-level {{
                        background-color: #4CAF50;
                        height: 100%;
                        border-radius: 4px;
                    }}
                    .metrics {{
                        display: flex;
                        justify-content: space-between;
                        margin-top: 15px;
                        font-size: 0.9em;
                        color: #666;
                    }}
                    .refresh-time {{
                        text-align: right;
                        font-size: 0.8em;
                        color: #999;
                        margin-top: 10px;
                    }}
                </style>
                
                <div class="dashboard-header">
                    <h2>NBA Game Predictions</h2>
                    <p>Updated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
                </div>
                
                <div class="games-container">
            """
            
            # Add game cards
            for game in prepared_games:
                # Determine game status text
                if game['current_quarter'] == 0:
                    status_text = "PRE-GAME"
                else:
                    status_text = f"QUARTER {game['current_quarter']}"
                
                # Determine win probability text
                win_prob = game['metrics']['win_probability']
                favorite = game['home_team'] if win_prob > 0.5 else game['away_team']
                win_pct = max(win_prob, 1-win_prob) * 100
                
                # Add game card HTML
                html += f"""
                    <div class="game-card">
                        <div class="game-header">
                            {game['home_team']} vs {game['away_team']}
                            <span style="float: right; background-color: #007bff; color: white; padding: 2px 6px; border-radius: 4px; font-size: 0.8em;">
                                {status_text}
                            </span>
                        </div>
                        
                        <div class="score-display">
                            <div class="team-score">
                                <div class="team-name">{game['home_team']}</div>
                                <div class="current-score">{int(game['current_scores']['home'])}</div>
                                <div class="predicted-score">Predicted: {game['predicted_scores']['home']:.1f}</div>
                            </div>
                            
                            <div class="vs-divider">VS</div>
                            
                            <div class="team-score">
                                <div class="team-name">{game['away_team']}</div>
                                <div class="current-score">{int(game['current_scores']['away'])}</div>
                                <div class="predicted-score">Predicted: {game['predicted_scores']['away']:.1f}</div>
                            </div>
                        </div>
                        
                        <div>
                            <div style="font-size: 0.9em; margin-bottom: 5px;">Win Probability</div>
                            <div class="confidence-bar">
                                <div class="confidence-level" style="width: {win_pct}%;"></div>
                            </div>
                            <div style="text-align: center; margin-top: 5px; font-size: 0.9em;">
                                {favorite}: {win_pct:.1f}%
                            </div>
                        </div>
                        
                        <div class="metrics">
                            <div>Prediction Confidence: {game['metrics']['confidence']*100:.0f}%</div>
                            <div>Margin: {abs(game['predicted_scores']['margin']):.1f}</div>
                        </div>
                        
                        <div class="refresh-time">
                            Last update: {datetime.now().strftime('%H:%M:%S')}
                        </div>
                    </div>
                """
            
            # Close containers
            html += """
                </div>
            </div>
            """
            
            return html
            
        except Exception as e:
            error_html = f"""
            <div style="border: 1px solid #f44336; padding: 10px; border-radius: 4px; background-color: #ffebee;">
                <h3 style="color: #f44336;">Error Generating Dashboard</h3>
                <p>{str(e)}</p>
            </div>
            """
            return error_html


# Example usage:
def demo_prediction_dashboard():
    """Demonstrate the refactored prediction dashboard with sample data."""
    # Create sample prediction data
    sample_games = pd.DataFrame([
        {
            'game_id': 'game1',
            'home_team': 'Lakers',
            'away_team': 'Celtics',
            'current_quarter': 3,
            'home_score': 85,
            'away_score': 82,
            'predicted_home_score': 112.5,
            'predicted_away_score': 108.2,
            'prediction_confidence': 0.85,
            'win_probability': 0.67
        },
        {
            'game_id': 'game2',
            'home_team': 'Warriors',
            'away_team': 'Bucks',
            'current_quarter': 2,
            'home_score': 58,
            'away_score': 61,
            'predicted_home_score': 118.3,
            'predicted_away_score': 121.7,
            'prediction_confidence': 0.72,
            'win_probability': 0.42
        }
    ])
    
    # Create sample prediction history
    sample_history = {
        'game1': [
            {
                'timestamp': '2023-03-13T19:30:00',
                'current_quarter': 1,
                'current_home_score': 28,
                'current_away_score': 25,
                'predicted_home_score': 107.2,
                'predicted_away_score': 102.6,
                'win_probability': 0.62
            },
            {
                'timestamp': '2023-03-13T20:00:00',
                'current_quarter': 2,
                'current_home_score': 56,
                'current_away_score': 52,
                'predicted_home_score': 110.3,
                'predicted_away_score': 106.1,
                'win_probability': 0.64
            },
            {
                'timestamp': '2023-03-13T20:30:00',
                'current_quarter': 3,
                'current_home_score': 85,
                'current_away_score': 82,
                'predicted_home_score': 112.5,
                'predicted_away_score': 108.2,
                'win_probability': 0.67
            }
        ],
        'game2': [
            {
                'timestamp': '2023-03-13T19:45:00',
                'current_quarter': 1,
                'current_home_score': 29,
                'current_away_score': 32,
                'predicted_home_score': 115.5,
                'predicted_away_score': 119.8,
                'win_probability': 0.38
            },
            {
                'timestamp': '2023-03-13T20:15:00',
                'current_quarter': 2,
                'current_home_score': 58,
                'current_away_score': 61,
                'predicted_home_score': 118.3,
                'predicted_away_score': 121.7,
                'win_probability': 0.42
            }
        ]
    }
    
    # Create dashboard
    dashboard = PredictionDashboard(use_plotly=True)
    
    # Display dashboard
    print("Creating dashboard visualization...")
    dashboard.create_live_dashboard(sample_games, sample_history)
    
    # Generate HTML version
    html_dashboard = dashboard.get_html_dashboard(sample_games, sample_history)
    print(f"\nGenerated HTML dashboard ({len(html_dashboard)} characters)")
    
    # Show HTML in notebook
    from IPython.display import HTML
    display(HTML("<h3>HTML Dashboard Preview:</h3>"))
    display(HTML(html_dashboard))
    
    return {
        'dashboard': dashboard,
        'sample_games': sample_games,
        'sample_history': sample_history
    }

# Run the demonstration if this cell is executed directly
if __name__ == '__main__':
    demo_results = demo_prediction_dashboard()

In [ ]:
# Cell 13: Enhanced Model Validation with Structured Results and Cross-Validation

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Union, Any, Callable
from sklearn.model_selection import KFold
import traceback

class ModelValidator:
    """
    Enhanced model validation framework for NBA score prediction models with
    cross-validation support and structured results reporting.
    """
    
    def __init__(self, supabase_client=None, model=None):
        """
        Initialize the model validator.
        
        Args:
            supabase_client: Supabase client for data access
            model: Pre-trained prediction model to validate
        """
        self.supabase = supabase_client
        self.model = model
        self.results_directory = './validation_results'
        
        # Create results directory if it doesn't exist
        os.makedirs(self.results_directory, exist_ok=True)
    
    def validate_on_historical_games(self, 
                                    num_games: int = 10, 
                                    model_override=None, 
                                    team_avgs: Optional[Dict[str, float]] = None,
                                    visualize: bool = True,
                                    cross_validate: bool = False,
                                    n_splits: int = 5) -> Dict[str, Any]:
        """
        Validate the prediction model on historical games with structured results.
        
        Args:
            num_games: Number of recent games to use for validation
            model_override: Optional model to use instead of the initialized one
            team_avgs: Pre-computed team rolling averages
            visualize: Whether to generate visualizations
            cross_validate: Whether to perform cross-validation
            n_splits: Number of splits for cross-validation
        
        Returns:
            Dict with structured validation results
        """
        print("Starting validation on historical games...")
        
        # Initialize results structure
        validation_results = {
            'timestamp': datetime.now().isoformat(),
            'num_games': num_games,
            'cross_validation': cross_validate,
            'n_splits': n_splits if cross_validate else None,
            'overall_metrics': {},
            'quarter_metrics': {},
            'game_results': [],
            'error_distribution': {},
            'cross_val_results': [] if cross_validate else None
        }
        
        try:
            # Use override model if provided, otherwise use the initialized model
            model_to_use = model_override if model_override is not None else self.model
            
            # Verify that we have a valid model
            if model_to_use is None:
                print("No prediction model loaded. Skipping validation.")
                validation_results['error'] = "No model available"
                return validation_results
            
            # Fetch historical games for validation
            historical_games = self._fetch_historical_games(num_games)
            if not historical_games or len(historical_games) == 0:
                print("No historical games found for validation.")
                validation_results['error'] = "No historical games found"
                return validation_results
            
            # Convert to DataFrame if we got a list
            if isinstance(historical_games, list):
                historical_games = pd.DataFrame(historical_games)
            
            # Get team averages if not provided
            if team_avgs is None:
                team_avgs = self._get_team_rolling_averages()
            
            if cross_validate:
                # Perform cross-validation
                cv_results = self._perform_cross_validation(
                    historical_games, model_to_use, team_avgs, n_splits
                )
                validation_results['cross_val_results'] = cv_results
                
                # Update overall metrics with cross-validation results
                validation_results['overall_metrics'] = {
                    'mean_absolute_error': np.mean([r['mean_absolute_error'] for r in cv_results]),
                    'root_mean_squared_error': np.mean([r['root_mean_squared_error'] for r in cv_results]),
                    'winner_accuracy': np.mean([r['winner_accuracy'] for r in cv_results]),
                    'r2_score': np.mean([r['r2_score'] for r in cv_results if 'r2_score' in r])
                }
            else:
                # Standard validation on all games
                all_results = []
                
                print(f"Validating model on {len(historical_games)} games")
                
                for idx, game in historical_games.iterrows():
                    actual_home = game['home_score']
                    actual_away = game['away_score']
                    game_id = game['game_id']
                    
                    # Validate across quarters
                    quarter_results = self._validate_game_by_quarters(
                        game, model_to_use, team_avgs
                    )
                    
                    # Add game metadata
                    game_result = {
                        'game_id': game_id,
                        'home_team': game['home_team'],
                        'away_team': game['away_team'],
                        'actual_home_score': actual_home,
                        'actual_away_score': actual_away,
                        'actual_margin': actual_home - actual_away,
                        'actual_winner': 'home' if actual_home > actual_away else 'away',
                        'quarter_results': quarter_results
                    }
                    
                    all_results.append(game_result)
                
                # Update results structure with game results
                validation_results['game_results'] = all_results
                
                # Calculate per-quarter metrics
                quarter_metrics = self._calculate_quarter_metrics(all_results)
                validation_results['quarter_metrics'] = quarter_metrics
                
                # Calculate overall metrics
                validation_results['overall_metrics'] = self._calculate_overall_metrics(all_results)
                
                # Calculate error distribution
                validation_results['error_distribution'] = self._calculate_error_distribution(all_results)
            
            # Generate visualizations if requested
            if visualize:
                self.visualize_validation_results(validation_results)
            
            return validation_results
            
        except Exception as e:
            print(f"Error during validation: {e}")
            traceback.print_exc()
            validation_results['error'] = str(e)
            validation_results['traceback'] = traceback.format_exc()
            return validation_results
    
    def _fetch_historical_games(self, num_games: int) -> pd.DataFrame:
        """Fetch historical games from the database."""
        try:
            if self.supabase is None:
                print("No Supabase client provided. Cannot fetch games.")
                return pd.DataFrame()
            
            response = self.supabase.table("nba_historical_game_stats").select("*").order('game_date', desc=True).limit(num_games).execute()
            
            if not response.data:
                return pd.DataFrame()
            
            return pd.DataFrame(response.data)
            
        except Exception as e:
            print(f"Error fetching historical games: {e}")
            return pd.DataFrame()
    
    def _get_team_rolling_averages(self) -> Dict[str, float]:
        """Calculate rolling averages for teams."""
        # This implementation would depend on your actual data schema
        # Here's a placeholder that returns an empty dictionary
        try:
            if self.supabase is None:
                return {}
            
            # In a real implementation, you would fetch team stats and calculate averages
            # For now, return an empty dict
            return {}
            
        except Exception as e:
            print(f"Error calculating team averages: {e}")
            return {}
    
    def _validate_game_by_quarters(self, 
                                  game: pd.Series, 
                                  model: Any, 
                                  team_avgs: Dict[str, float]) -> List[Dict[str, Any]]:
        """
        Validate a game by quarter, simulating predictions at each stage.
        
        Args:
            game: Game data as a pandas Series
            model: Prediction model
            team_avgs: Team rolling averages
            
        Returns:
            List of quarter-by-quarter prediction results
        """
        quarter_results = []
        actual_home = game['home_score']
        actual_away = game['away_score']
        
        for quarter in range(1, 5):
            # Create simulated game state up through this quarter
            sim_game = {
                'game_id': game['game_id'],
                'home_team': game['home_team'],
                'away_team': game['away_team'],
                'game_date': game.get('game_date'),
                'current_quarter': quarter
            }
            
            # Calculate scores through the current quarter
            home_score = 0
            away_score = 0
            
            for q in range(1, quarter+1):
                home_q = f'home_q{q}'
                away_q = f'away_q{q}'
                
                if home_q in game and away_q in game:
                    home_score += float(game[home_q]) if pd.notna(game[home_q]) else 0
                    away_score += float(game[away_q]) if pd.notna(game[away_q]) else 0
            
            # Update simulated game with current scores
            sim_game['home_score'] = home_score
            sim_game['away_score'] = away_score
            
            # Add quarter-specific scores
            for q in range(1, 5):
                home_q = f'home_q{q}'
                away_q = f'away_q{q}'
                
                if q <= quarter and home_q in game and away_q in game:
                    sim_game[home_q] = float(game[home_q]) if pd.notna(game[home_q]) else 0
                    sim_game[away_q] = float(game[away_q]) if pd.notna(game[away_q]) else 0
                else:
                    sim_game[home_q] = 0
                    sim_game[away_q] = 0
            
            # Create dataframe for prediction
            sim_df = pd.DataFrame([sim_game])
            
            # Predict the final score
            try:
                pred_result = self._run_prediction(sim_df, model, team_avgs)
                
                if pred_result.empty:
                    # Skip this quarter if prediction failed
                    continue
                
                pred_row = pred_result.iloc[0]
                predicted_home = float(pred_row['predicted_home_score'])
                predicted_away = float(pred_row['predicted_away_score'])
                
                # Calculate errors
                home_error = predicted_home - actual_home
                away_error = predicted_away - actual_away
                abs_error = (abs(home_error) + abs(away_error)) / 2
                
                # Calculate winner prediction accuracy
                actual_winner = 'home' if actual_home > actual_away else 'away'
                predicted_winner = 'home' if predicted_home > predicted_away else 'away'
                winner_correct = actual_winner == predicted_winner
                
                # Record results for this quarter
                quarter_result = {
                    'quarter': quarter,
                    'current_home_score': home_score,
                    'current_away_score': away_score,
                    'predicted_home_score': predicted_home,
                    'predicted_away_score': predicted_away,
                    'predicted_margin': predicted_home - predicted_away,
                    'predicted_winner': predicted_winner,
                    'home_error': home_error,
                    'away_error': away_error,
                    'absolute_error': abs_error,
                    'pct_error_home': (abs(home_error) / actual_home) * 100 if actual_home > 0 else float('inf'),
                    'pct_error_away': (abs(away_error) / actual_away) * 100 if actual_away > 0 else float('inf'),
                    'winner_correct': winner_correct
                }
                
                quarter_results.append(quarter_result)
                
            except Exception as e:
                print(f"Error validating quarter {quarter} for game {game['game_id']}: {e}")
                # Continue with next quarter
        
        return quarter_results
    
    def _run_prediction(self, 
                       game_df: pd.DataFrame, 
                       model: Any, 
                       team_avgs: Dict[str, float]) -> pd.DataFrame:
        """
        Run prediction on a game.
        
        This is a placeholder method that should be replaced with your actual prediction logic.
        It simulates calling a predict_final_scores function that would exist in your codebase.
        
        Args:
            game_df: Game data as a DataFrame
            model: Prediction model
            team_avgs: Team rolling averages
            
        Returns:
            DataFrame with prediction results
        """
        try:
            # This is where you would call your actual prediction function
            # For example: return predict_final_scores(game_df, team_avgs)
            
            # Since we don't have the actual prediction function, we'll simulate it
            # In a real implementation, replace this with your actual prediction logic
            
            # Placeholder implementation - this would be replaced with actual prediction
            result_df = game_df.copy()
            
            # Simple simulation of prediction - this is not a real model prediction!
            # In a real implementation, you would use the provided model
            for _, row in result_df.iterrows():
                current_quarter = int(row['current_quarter'])
                home_score = float(row['home_score']) if 'home_score' in row else 0
                away_score = float(row['away_score']) if 'away_score' in row else 0
                
                # Simple extrapolation
                remaining_factor = 1.0 - (current_quarter / 4.0)
                
                # Get team scoring rates from averages
                home_team = row['home_team']
                away_team = row['away_team']
                
                home_avg = team_avgs.get(home_team, 105)
                away_avg = team_avgs.get(away_team, 105)
                
                # Predict remaining points
                remaining_home = home_avg * remaining_factor
                remaining_away = away_avg * remaining_factor
                
                # Calculate predicted final scores
                predicted_home = home_score + remaining_home
                predicted_away = away_score + remaining_away
                
                result_df.at[_, 'predicted_home_score'] = predicted_home
                result_df.at[_, 'predicted_away_score'] = predicted_away
            
            return result_df
            
        except Exception as e:
            print(f"Error running prediction: {e}")
            return pd.DataFrame()
    
    def _calculate_quarter_metrics(self, game_results: List[Dict[str, Any]]) -> Dict[int, Dict[str, float]]:
        """
        Calculate metrics by quarter.
        
        Args:
            game_results: List of game results
            
        Returns:
            Dict mapping quarter to metrics
        """
        quarter_metrics = {
            1: {'mae': [], 'rmse': [], 'winner_accuracy': []},
            2: {'mae': [], 'rmse': [], 'winner_accuracy': []},
            3: {'mae': [], 'rmse': [], 'winner_accuracy': []},
            4: {'mae': [], 'rmse': [], 'winner_accuracy': []}
        }
        
        # Collect metrics by quarter
        for game in game_results:
            for qr in game['quarter_results']:
                quarter = qr['quarter']
                if quarter not in quarter_metrics:
                    continue
                    
                quarter_metrics[quarter]['mae'].append(qr['absolute_error'])
                quarter_metrics[quarter]['rmse'].append(qr['absolute_error'] ** 2)
                quarter_metrics[quarter]['winner_accuracy'].append(int(qr['winner_correct']))
        
        # Calculate means
        results = {}
        for quarter, metrics in quarter_metrics.items():
            if not metrics['mae']:
                # Skip quarters with no data
                continue
                
            results[quarter] = {
                'mean_absolute_error': np.mean(metrics['mae']),
                'root_mean_squared_error': np.sqrt(np.mean(metrics['rmse'])),
                'winner_accuracy': np.mean(metrics['winner_accuracy']),
                'sample_size': len(metrics['mae'])
            }
        
        return results
    
    def _calculate_overall_metrics(self, game_results: List[Dict[str, Any]]) -> Dict[str, float]:
        """
        Calculate overall metrics across all games and quarters.
        
        Args:
            game_results: List of game results
            
        Returns:
            Dict of overall metrics
        """
        all_maes = []
        all_mses = []
        all_winner_correct = []
        
        for game in game_results:
            for qr in game['quarter_results']:
                all_maes.append(qr['absolute_error'])
                all_mses.append(qr['absolute_error'] ** 2)
                all_winner_correct.append(int(qr['winner_correct']))
        
        if not all_maes:
            return {}
            
        return {
            'mean_absolute_error': np.mean(all_maes),
            'root_mean_squared_error': np.sqrt(np.mean(all_mses)),
            'winner_accuracy': np.mean(all_winner_correct),
            'sample_size': len(all_maes)
        }
    
    def _calculate_error_distribution(self, game_results: List[Dict[str, Any]]) -> Dict[str, Any]:
        """
        Calculate error distribution statistics.
        
        Args:
            game_results: List of game results
            
        Returns:
            Dict with error distribution statistics
        """
        all_errors = []
        
        for game in game_results:
            for qr in game['quarter_results']:
                all_errors.append(qr['absolute_error'])
        
        if not all_errors:
            return {}
            
        return {
            'min': np.min(all_errors),
            'max': np.max(all_errors),
            'median': np.median(all_errors),
            'p25': np.percentile(all_errors, 25),
            'p75': np.percentile(all_errors, 75),
            'p90': np.percentile(all_errors, 90),
            'std_dev': np.std(all_errors)
        }
    
    def _perform_cross_validation(self,
                                 games_df: pd.DataFrame,
                                 model: Any,
                                 team_avgs: Dict[str, float],
                                 n_splits: int = 5) -> List[Dict[str, Any]]:
        """
        Perform k-fold cross-validation on the games dataset.
        
        Args:
            games_df: DataFrame of historical games
            model: Prediction model
            team_avgs: Team rolling averages
            n_splits: Number of CV splits
            
        Returns:
            List of metrics for each fold
        """
        print(f"Performing {n_splits}-fold cross-validation...")
        
        # Initialize k-fold cross-validation
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
        
        # Prepare results container
        cv_results = []
        
        # Perform cross-validation
        for fold, (train_idx, test_idx) in enumerate(kf.split(games_df)):
            print(f"Processing fold {fold+1}/{n_splits}")
            
            # Split data
            train_games = games_df.iloc[train_idx]
            test_games = games_df.iloc[test_idx]
            
            # Train model on training set if model has a fit method
            # This assumes the model can be retrained - adapt if yours can't
            if hasattr(model, 'fit'):
                # Prepare training data
                X_train, y_train = self._prepare_training_data(train_games)
                
                # Train model
                if X_train is not None and y_train is not None:
                    model.fit(X_train, y_train)
            
            # Validate on test set
            fold_results = []
            for idx, game in test_games.iterrows():
                # Validate quarter by quarter
                quarter_results = self._validate_game_by_quarters(
                    game, model, team_avgs
                )
                
                # Add to fold results
                if quarter_results:
                    fold_results.extend(quarter_results)
            
            # Calculate fold metrics
            if fold_results:
                all_abs_errors = [r['absolute_error'] for r in fold_results]
                all_squared_errors = [r['absolute_error'] ** 2 for r in fold_results]
                all_winner_correct = [int(r['winner_correct']) for r in fold_results]
                
                fold_metrics = {
                    'fold': fold + 1,
                    'train_size': len(train_games),
                    'test_size': len(test_games),
                    'mean_absolute_error': np.mean(all_abs_errors),
                    'root_mean_squared_error': np.sqrt(np.mean(all_squared_errors)),
                    'winner_accuracy': np.mean(all_winner_correct),
                    'error_std_dev': np.std(all_abs_errors),
                    'sample_size': len(all_abs_errors)
                }
                
                cv_results.append(fold_metrics)
        
        return cv_results
    
    def _prepare_training_data(self, train_games: pd.DataFrame) -> Tuple[Optional[pd.DataFrame], Optional[pd.Series]]:
        """
        Prepare training data for model fitting.
        
        This is a placeholder method that should be replaced with your actual data preparation logic.
        
        Args:
            train_games: Training games DataFrame
            
        Returns:
            Tuple of (X_train, y_train) or (None, None) if preparation fails
        """
        try:
            # This would be replaced with your actual feature engineering logic
            # For example, you might extract features like:
            # - Team performance metrics
            # - Game context features
            # - Historical matchup data
            
            # Placeholder implementation - returns None, None
            return None, None
            
        except Exception as e:
            print(f"Error preparing training data: {e}")
            return None, None
    
    def visualize_validation_results(self, validation_results: Dict[str, Any]):
        """
        Generate visualizations for validation results.
        
        Args:
            validation_results: Validation results dictionary
        """
        # Skip visualization if no quarter metrics available
        if not validation_results['quarter_metrics']:
            print("No metrics available for visualization.")
            return
        
        # Create figure for quarter metrics
        plt.figure(figsize=(12, 6))
        
        # Prepare data for plotting
        quarters = sorted(validation_results['quarter_metrics'].keys())
        maes = [validation_results['quarter_metrics'][q]['mean_absolute_error'] for q in quarters]
        rmses = [validation_results['quarter_metrics'][q]['root_mean_squared_error'] for q in quarters]
        accuracies = [validation_results['quarter_metrics'][q]['winner_accuracy'] * 100 for q in quarters]
        sample_sizes = [validation_results['quarter_metrics'][q]['sample_size'] for q in quarters]
        
        # Plot 1: Error metrics by quarter
        plt.subplot(1, 2, 1)
        width = 0.35
        x = np.arange(len(quarters))
        
        # Plot MAE bars
        plt.bar(x - width/2, maes, width, label='MAE', color='skyblue')
        
        # Plot RMSE bars
        plt.bar(x + width/2, rmses, width, label='RMSE', color='salmon')
        
        # Add sample size annotations
        for i, (m, r, s) in enumerate(zip(maes, rmses, sample_sizes)):
            plt.text(i - width/2, m + 0.5, f"n={s}", ha='center', va='bottom', fontsize=9)
        
        plt.xlabel('Information Available Through Quarter')
        plt.ylabel('Error (points)')
        plt.title('Prediction Error by Quarter')
        plt.xticks(x, quarters)
        plt.legend()
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        
        # Plot 2: Winner prediction accuracy by quarter
        plt.subplot(1, 2, 2)
        plt.plot(quarters, accuracies, 'o-', color='green', linewidth=2)
        
        # Add accuracy annotations
        for i, (q, acc) in enumerate(zip(quarters, accuracies)):
            plt.text(q, acc + 2, f"{acc:.1f}%", ha='center')
        
        plt.xlabel('Information Available Through Quarter')
        plt.ylabel('Winner Prediction Accuracy (%)')
        plt.title('Winner Prediction Accuracy by Quarter')
        plt.ylim(0, 100)
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Cross-validation results if available
        if validation_results.get('cross_val_results'):
            self._visualize_cross_validation_results(validation_results['cross_val_results'])
        
        # Error distribution visualization
        if validation_results.get('error_distribution'):
            self._visualize_error_distribution(validation_results['error_distribution'])
    
    def _visualize_cross_validation_results(self, cv_results: List[Dict[str, Any]]):
        """
        Visualize cross-validation results.
        
        Args:
            cv_results: List of cross-validation fold results
        """
        if not cv_results:
            return
            
        plt.figure(figsize=(10, 6))
        
        # Extract data
        folds = [r['fold'] for r in cv_results]
        maes = [r['mean_absolute_error'] for r in cv_results]
        rmses = [r['root_mean_squared_error'] for r in cv_results]
        accuracies = [r['winner_accuracy'] * 100 for r in cv_results]
        
        # Calculate means for reference lines
        mean_mae = np.mean(maes)
        mean_rmse = np.mean(rmses)
        mean_acc = np.mean(accuracies)
        
        # Create plot
        plt.subplot(1, 2, 1)
        plt.bar(folds, maes, alpha=0.7, label='MAE')
        plt.bar(folds, rmses, alpha=0.5, label='RMSE')
        
        # Add reference lines
        plt.axhline(y=mean_mae, color='blue', linestyle='--', label=f'Mean MAE: {mean_mae:.2f}')
        plt.axhline(y=mean_rmse, color='red', linestyle='--', label=f'Mean RMSE: {mean_rmse:.2f}')
        
        plt.xlabel('Fold')
        plt.ylabel('Error (points)')
        plt.title('Cross-Validation Error Metrics by Fold')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # Accuracy plot
        plt.subplot(1, 2, 2)
        plt.bar(folds, accuracies, color='green', alpha=0.7)
        
        # Add reference line
        plt.axhline(y=mean_acc, color='green', linestyle='--', label=f'Mean: {mean_acc:.1f}%')
        
        # Add accuracy annotations
        for i, acc in enumerate(accuracies):
            plt.text(i+1, acc + 2, f"{acc:.1f}%", ha='center')
        
        plt.xlabel('Fold')
        plt.ylabel('Winner Accuracy (%)')
        plt.title('Cross-Validation Winner Prediction Accuracy')
        plt.ylim(0, 100)
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    def _visualize_error_distribution(self, error_distribution: Dict[str, float]):
        """
        Visualize error distribution statistics.
        
        Args:
            error_distribution: Dictionary with error distribution statistics
        """
        if not error_distribution:
            return
            
        plt.figure(figsize=(10, 4))
        
        # Create box plot-like visualization
        plt.plot([0, 1], [error_distribution['median'], error_distribution['median']], 'r-', linewidth=2)
        plt.plot([0, 1], [error_distribution['p25'], error_distribution['p25']], 'b--')
        plt.plot([0, 1], [error_distribution['p75'], error_distribution['p75']], 'b--')
        plt.fill_between([0, 1], error_distribution['p25'], error_distribution['p75'], color='lightblue', alpha=0.5)
        
        # Add min/max lines
        plt.plot([0, 1], [error_distribution['min'], error_distribution['min']], 'g:')
        plt.plot([0, 1], [error_distribution['max'], error_distribution['max']], 'r:')
        
        # Add annotations
        plt.text(1.01, error_distribution['median'], f"Median: {error_distribution['median']:.1f}", va='center')
        plt.text(1.01, error_distribution['p25'], f"Q1: {error_distribution['p25']:.1f}", va='center')
        plt.text(1.01, error_distribution['p75'], f"Q3: {error_distribution['p75']:.1f}", va='center')
        plt.text(1.01, error_distribution['min'], f"Min: {error_distribution['min']:.1f}", va='center')
        plt.text(1.01, error_distribution['max'], f"Max: {error_distribution['max']:.1f}", va='center')
        plt.text(1.01, error_distribution['p90'], f"90%: {error_distribution['p90']:.1f}", va='center')
        
        plt.axhline(y=error_distribution['p90'], color='purple', linestyle=':', alpha=0.7)
        
        plt.xlim(0, 1.3)
        plt.ylim(0, error_distribution['max'] * 1.1)
        plt.title('Prediction Error Distribution')
        plt.ylabel('Absolute Error (points)')
        plt.grid(axis='y', alpha=0.3)
        plt.gca().set_xticks([])
        
        plt.tight_layout()
        plt.show()
    
    def save_validation_results(self, validation_results: Dict[str, Any], filename: Optional[str] = None):
        """
        Save validation results to a file.
        
        Args:
            validation_results: Validation results dictionary
            filename: Optional filename (if None, auto-generates based on timestamp)
        """
        if filename is None:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            filename = f"validation_results_{timestamp}.json"
        
        filepath = os.path.join(self.results_directory, filename)
        
        try:
            with open(filepath, 'w') as f:
                json.dump(validation_results, f, indent=2)
            
            print(f"Validation results saved to {filepath}")
            return filepath
            
        except Exception as e:
            print(f"Error saving validation results: {e}")
            return None
    
    def load_validation_results(self, filename: str) -> Dict[str, Any]:
        """
        Load validation results from a file.
        
        Args:
            filename: Filename to load
            
        Returns:
            Dictionary with validation results
        """
        filepath = os.path.join(self.results_directory, filename)
        
        try:
            with open(filepath, 'r') as f:
                validation_results = json.load(f)
            
            print(f"Validation results loaded from {filepath}")
            return validation_results
            
        except Exception as e:
            print(f"Error loading validation results: {e}")
            return {}


# Usage example
def demo_model_validation(model=None, supabase=None):
    """Demonstrate the enhanced model validation framework."""
    # Initialize validator
    validator = ModelValidator(supabase_client=supabase, model=model)
    
    # Run validation
    print("Running standard validation...")
    validation_results = validator.validate_on_historical_games(
        num_games=10,
        model_override=model,
        visualize=True,
        cross_validate=False
    )
    
    # Save validation results
    validator.save_validation_results(validation_results)
    
    # Run cross-validation with fewer games for speed
    print("\nRunning cross-validation...")
    cv_results = validator.validate_on_historical_games(
        num_games=5,
        model_override=model,
        visualize=True,
        cross_validate=True,
        n_splits=3
    )
    
    # Save cross-validation results
    validator.save_validation_results(cv_results, "cross_validation_results.json")
    
    return {
        'validator': validator,
        'standard_results': validation_results,
        'cv_results': cv_results
    }

# Direct execution example
if __name__ == "__main__":
    # If this is run as a script, try to load model and run demo
    try:
        from config import DATABASE_URL
        import joblib
        import os
        
        # Try to load model from common locations
        model_path = os.environ.get('MODEL_PATH', './models/score_prediction_model.pkl')
        if os.path.exists(model_path):
            model = joblib.load(model_path)
            print(f"Loaded model from {model_path}")
        else:
            print("Model not found. Running with simulation only.")
            model = None
        
        # Try to set up Supabase connection
        try:
            from caching.supabase_client import supabase
            print("Using existing Supabase connection")
        except ImportError:
            print("Supabase client not available. Running with simulation only.")
            supabase = None
        
        # Run validation
        demo_results = demo_model_validation(model=model, supabase=supabase)
        
    except Exception as e:
        print(f"Error during demo: {e}")
        traceback.print_exc()

In [ ]:
# Cell 14: Fetch Scheduled Game Data with Centralized Timezone Handling

import pytz
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import traceback
from functools import lru_cache
from typing import Dict, List, Optional, Tuple, Union

# ---- Timezone Utility Functions ----

def get_pacific_time():
    """Get current time in Pacific timezone."""
    pacific_tz = pytz.timezone('America/Los_Angeles')
    return datetime.now(pacific_tz)

def get_pacific_date_str():
    """Get current date in Pacific timezone as YYYY-MM-DD string."""
    return get_pacific_time().strftime('%Y-%m-%d')

def convert_to_pacific(dt, from_timezone=None):
    """Convert datetime to Pacific timezone.
    
    Args:
        dt: Datetime to convert (string or datetime object)
        from_timezone: Source timezone if dt is naive
        
    Returns:
        Timezone-aware datetime in Pacific time
    """
    pacific_tz = pytz.timezone('America/Los_Angeles')
    
    # Handle string dates
    if isinstance(dt, str):
        try:
            # If time component is missing, assume start of day
            if len(dt) <= 10:  # YYYY-MM-DD format
                dt_obj = datetime.strptime(dt, '%Y-%m-%d')
                # Add typical NBA game start time (7:30 PM Pacific)
                dt_obj = dt_obj.replace(hour=19, minute=30)
                # Localize to Pacific time
                return pacific_tz.localize(dt_obj)
            else:
                # Parse with time component
                dt_obj = pd.to_datetime(dt)
                # If timezone naive, localize to provided timezone or UTC
                if dt_obj.tzinfo is None:
                    source_tz = pytz.timezone(from_timezone) if from_timezone else pytz.UTC
                    dt_obj = source_tz.localize(dt_obj)
                # Convert to Pacific
                return dt_obj.astimezone(pacific_tz)
        except Exception as e:
            print(f"Error parsing date '{dt}': {e}")
            return get_pacific_time()
    
    # Handle datetime objects
    if isinstance(dt, datetime):
        # If timezone naive, localize to provided timezone or UTC
        if dt.tzinfo is None:
            source_tz = pytz.timezone(from_timezone) if from_timezone else pytz.UTC
            dt = source_tz.localize(dt)
        # Convert to Pacific
        return dt.astimezone(pacific_tz)
    
    # Default fallback
    print(f"Unsupported date format: {dt}")
    return get_pacific_time()

# ---- Caching Mechanism ----

# Cache for schedule data (expires after 30 minutes)
SCHEDULE_CACHE = {}
CACHE_EXPIRY = 30 * 60  # 30 minutes in seconds

def get_cached_data(key, expiry=CACHE_EXPIRY):
    """Get data from cache if it exists and hasn't expired."""
    if key in SCHEDULE_CACHE:
        timestamp, data = SCHEDULE_CACHE[key]
        if time.time() - timestamp <= expiry:
            return data
    return None

def set_cached_data(key, data):
    """Store data in cache with current timestamp."""
    SCHEDULE_CACHE[key] = (time.time(), data)
    return data

# ---- API Error Handling ----

def safe_api_call(func, *args, max_retries=3, retry_delay=2, **kwargs):
    """Execute an API call with retry logic and error handling.
    
    Args:
        func: Function to call
        max_retries: Maximum number of retries (default: 3)
        retry_delay: Delay between retries in seconds (default: 2)
        *args, **kwargs: Arguments to pass to func
        
    Returns:
        API response or None on failure
    """
    retries = 0
    last_error = None
    
    while retries < max_retries:
        try:
            return func(*args, **kwargs)
        except Exception as e:
            retries += 1
            last_error = e
            
            # Check for rate limiting and adjust retry delay
            if "rate limit" in str(e).lower() or "429" in str(e):
                retry_delay = min(retry_delay * 2, 30)  # Exponential backoff up to 30 seconds
                print(f"Rate limit hit. Retrying in {retry_delay} seconds...")
            else:
                print(f"API error: {e}. Retry {retries}/{max_retries} in {retry_delay} seconds...")
                
            time.sleep(retry_delay)
    
    # Log the error after all retries fail
    print(f"API call failed after {max_retries} retries. Last error: {last_error}")
    traceback.print_exc()
    return None

# ---- Unified Game Status Detection ----

def determine_game_status(game_data: Dict) -> Tuple[str, int, float]:
    """
    Determine game status, current quarter, and time remaining.
    
    Args:
        game_data: Dictionary with game information
        
    Returns:
        Tuple of (status, current_quarter, time_remaining_mins)
    """
    # Extract basic game info
    game_date_str = game_data.get('game_date', get_pacific_date_str())
    game_date = convert_to_pacific(game_date_str)
    now = get_pacific_time()
    
    # Check if game is marked as finished
    if game_data.get('is_finished', False):
        return 'FINAL', 4, 0
    
    # Calculate quarter based on data
    current_q = 0
    for q in range(1, 5):
        home_q_val = float(game_data.get(f'home_q{q}', 0) or 0)
        away_q_val = float(game_data.get(f'away_q{q}', 0) or 0)
        if home_q_val > 0 or away_q_val > 0:
            current_q = q
    
    # Use provided current_quarter if available
    if 'current_quarter' in game_data and game_data['current_quarter'] > current_q:
        current_q = int(game_data['current_quarter'])
    
    # All four quarters have scores - game should be final
    if current_q == 4:
        home_q4 = float(game_data.get('home_q4', 0) or 0)
        away_q4 = float(game_data.get('away_q4', 0) or 0)
        if home_q4 > 0 and away_q4 > 0:
            # Special case: if there's OT data or game is specifically marked unfinished, don't mark as FINAL
            has_ot = ('home_ot' in game_data and game_data['home_ot'] > 0) or \
                     ('away_ot' in game_data and game_data['away_ot'] > 0)
            explicitly_unfinished = game_data.get('is_finished') is False
            
            if has_ot or explicitly_unfinished:
                return 'LIVE', 4, 2  # Assume late 4th quarter or OT
            return 'FINAL', 4, 0
    
    # Game in progress
    if current_q > 0:
        # Calculate approximate time remaining
        quarter_length = 12  # minutes
        time_remaining = (4 - current_q) * quarter_length + \
                         (quarter_length / 2)  # Assume middle of current quarter
                         
        # If specific time remaining is provided, use it
        if 'time_remaining_in_quarter' in game_data:
            specified_time = float(game_data['time_remaining_in_quarter'])
            time_remaining = (4 - current_q) * quarter_length + \
                            min(specified_time, quarter_length)
        
        return 'LIVE', current_q, time_remaining
    
    # Game hasn't started yet
    if current_q == 0:
        # Calculate time until game starts
        game_time = game_date
        current_time = now
        
        # If game date is today but in the past, it might be delayed
        if game_time.date() == current_time.date() and game_time < current_time:
            time_diff = (current_time - game_time).total_seconds() / 60
            if time_diff > 60:  # More than an hour past scheduled start
                return 'DELAYED', 0, 48 + 15  # Full game + typical delay
            return 'DELAYED', 0, 48  # Full game time
        
        # Game is in the future
        if game_time > current_time:
            mins_to_start = (game_time - current_time).total_seconds() / 60
            return 'SCHEDULED', 0, 48 + max(0, mins_to_start)
    
    # Default fallback
    return 'UNKNOWN', current_q, 48  # Full game time remaining

# ---- Main Functions ----

def fetch_scheduled_games(date_str: str = None) -> pd.DataFrame:
    """
    Fetch scheduled games from the database for a specific date (Pacific Time).
    
    Args:
        date_str: Date string in YYYY-MM-DD format (defaults to today in PT)
        
    Returns:
        DataFrame with scheduled games
    """
    # Use today's date in Pacific Time if none provided
    if date_str is None:
        date_str = get_pacific_date_str()
    
    print(f"Fetching scheduled games for {date_str} (Pacific Time)")
    
    # Check cache first
    cache_key = f"schedule_{date_str}"
    cached_data = get_cached_data(cache_key)
    if cached_data is not None:
        print(f"Using cached schedule data for {date_str}")
        return cached_data
    
    # Fetch from API with error handling
    try:
        response = safe_api_call(
            supabase.table("nba_game_schedule").select("*").eq("game_date", date_str).execute
        )
        
        if not response or not response.data:
            print(f"No scheduled games found for {date_str}")
            return pd.DataFrame()
        
        # Create DataFrame and process it
        games_df = pd.DataFrame(response.data)
        
        # Ensure each game has a status
        enhanced_games = []
        for _, game in games_df.iterrows():
            game_dict = game.to_dict()
            status, quarter, time_remaining = determine_game_status(game_dict)
            
            # Add status information
            game_dict['game_status'] = status
            game_dict['current_quarter'] = quarter
            game_dict['time_remaining_mins'] = time_remaining
            
            enhanced_games.append(game_dict)
        
        result_df = pd.DataFrame(enhanced_games)
        
        # Cache the result
        set_cached_data(cache_key, result_df)
        
        # Report status breakdown
        if not result_df.empty:
            status_counts = result_df['game_status'].value_counts()
            print("Game status summary:")
            for status, count in status_counts.items():
                print(f"  - {status}: {count} games")
        
        return result_df
        
    except Exception as e:
        print(f"Error fetching scheduled games: {e}")
        traceback.print_exc()
        return pd.DataFrame()

# Test the function
today_schedule = fetch_scheduled_games()
display(today_schedule)

In [ ]:
# Cell 14B - Add explicit game status tracking with Pacific Time Zone support

import pytz
from datetime import datetime, timedelta

def determine_game_status(games_df):
    """
    Adds explicit status field to game data with proper Pacific Time Zone handling:
    - LIVE: Currently in progress
    - SCHEDULED: Upcoming game
    - FINAL: Completed game
    
    Also adds estimated time remaining.
    """
    if games_df is None or games_df.empty:
        return games_df
        
    result_df = games_df.copy()
    result_df['game_status'] = 'UNKNOWN'
    result_df['time_remaining_mins'] = 0
    
    # Current time in Pacific Time Zone (Los Angeles)
    pacific_tz = pytz.timezone('America/Los_Angeles')
    now_pacific = datetime.now(pacific_tz)
    today_pacific = now_pacific.strftime('%Y-%m-%d')
    
    print(f"Current Pacific Time: {now_pacific.strftime('%Y-%m-%d %H:%M:%S %Z')}")
    
    for idx, game in result_df.iterrows():
        # Extract game info
        game_date = game.get('game_date', today_pacific)
        
        # Calculate quarter based on data
        current_q = 0
        for q in range(1, 5):
            home_q_val = float(game.get(f'home_q{q}', 0) or 0)
            away_q_val = float(game.get(f'away_q{q}', 0) or 0)
            if home_q_val > 0 or away_q_val > 0:
                current_q = q
        
        # Set current_quarter field consistent with data
        result_df.at[idx, 'current_quarter'] = current_q
        
        # Convert game_date to datetime object in Pacific time
        try:
            if isinstance(game_date, str):
                # If time component is missing, assume start of day
                if len(game_date) <= 10:  # YYYY-MM-DD format
                    game_date_obj = datetime.strptime(game_date, '%Y-%m-%d')
                    # Add typical NBA game start time (7:30 PM Pacific)
                    game_date_obj = game_date_obj.replace(hour=19, minute=30)
                    # Localize to Pacific time
                    game_date_obj = pacific_tz.localize(game_date_obj)
                else:
                    # Parse with time component
                    game_date_obj = pd.to_datetime(game_date).tz_localize(pacific_tz)
            else:
                # Handle if already datetime object but no timezone
                game_date_obj = pd.to_datetime(game_date)
                if game_date_obj.tzinfo is None:
                    game_date_obj = game_date_obj.tz_localize(pacific_tz)
        except Exception as e:
            print(f"Error parsing game date '{game_date}': {e}")
            game_date_obj = now_pacific
        
        # Determine status based on available data and Pacific time
        if current_q > 0 and current_q < 4:
            # Game in progress
            result_df.at[idx, 'game_status'] = 'LIVE'
            result_df.at[idx, 'time_remaining_mins'] = (4 - current_q) * 12
        elif current_q == 4:
            # Potentially finished or in 4th quarter
            home_q4 = float(game.get('home_q4', 0) or 0)
            away_q4 = float(game.get('away_q4', 0) or 0)
            if home_q4 > 0 and away_q4 > 0:
                # Check if complete or still in Q4
                if game.get('is_finished', False):
                    result_df.at[idx, 'game_status'] = 'FINAL'
                    result_df.at[idx, 'time_remaining_mins'] = 0
                else:
                    result_df.at[idx, 'game_status'] = 'LIVE'
                    result_df.at[idx, 'time_remaining_mins'] = 6  # Estimate mid-Q4
            else:
                result_df.at[idx, 'game_status'] = 'LIVE'
                result_df.at[idx, 'time_remaining_mins'] = 12  # Full Q4 remaining
        elif current_q == 0:
            # Game hasn't started yet - check date against Pacific time
            if game_date_obj.date() < now_pacific.date():
                # Past date but no scores - likely a data issue or postponed game
                result_df.at[idx, 'game_status'] = 'UNKNOWN'
            elif game_date_obj.date() == now_pacific.date():
                # Today's game
                if game_date_obj < now_pacific:
                    # Start time has passed, but no scores - may be delayed or data issue
                    result_df.at[idx, 'game_status'] = 'DELAYED'
                else:
                    # Start time is later today
                    result_df.at[idx, 'game_status'] = 'SCHEDULED'
                    # Calculate minutes until game starts
                    mins_to_start = (game_date_obj - now_pacific).total_seconds() / 60
                    result_df.at[idx, 'time_remaining_mins'] = 48 + max(0, mins_to_start)
            else:
                # Future date
                result_df.at[idx, 'game_status'] = 'SCHEDULED'
                result_df.at[idx, 'time_remaining_mins'] = 48  # Full game
        
        # Special case: Check for actual final flag (for historical test games)
        if 'actual_home_final' in game and pd.notna(game['actual_home_final']):
            result_df.at[idx, 'game_status'] = 'FINAL'
            result_df.at[idx, 'time_remaining_mins'] = 0
    
    # Summary of statuses
    status_counts = result_df['game_status'].value_counts()
    print("Game status summary:")
    for status, count in status_counts.items():
        print(f"  - {status}: {count} games")
    
    return result_df

In [ ]:
# Cell 15: Enhanced Monitoring Function with Correct Team Matching

from models.dynamic_recommendation import generate_recommendations
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import traceback
import pytz
import os
import joblib
from supabase import create_client

# Make the official schedule globally available
OFFICIAL_SCHEDULE = None

def normalize_team_name(name):
    """Normalize team names for consistent matching"""
    if not name:
        return ""

    # Convert to string and lowercase
    name = str(name).lower().strip()

    # Standard team name mappings
    mappings = {
        'lakers': 'los angeles lakers',
        'la lakers': 'los angeles lakers',
        'clippers': 'los angeles clippers',
        'la clippers': 'los angeles clippers',
        'blazers': 'portland trail blazers',
        'sixers': 'philadelphia 76ers',
        'philly': 'philadelphia 76ers',
        'knicks': 'new york knicks',
        'ny knicks': 'new york knicks',
        'nets': 'brooklyn nets',
        'mavs': 'dallas mavericks',
        'cavs': 'cleveland cavaliers',
        'wolves': 'minnesota timberwolves',
        't-wolves': 'minnesota timberwolves',
        'celts': 'boston celtics',
        'pels': 'new orleans pelicans',
        'warriors': 'golden state warriors',
        'gsw': 'golden state warriors',
        'heat': 'miami heat',
        'bulls': 'chicago bulls',
        'hawks': 'atlanta hawks',
        'suns': 'phoenix suns',
        'bucks': 'milwaukee bucks',
        'jazz': 'utah jazz',
        'nuggets': 'denver nuggets',
        'rockets': 'houston rockets',
        'pacers': 'indiana pacers',
        'spurs': 'san antonio spurs',
        'kings': 'sacramento kings',
        'magic': 'orlando magic',
        'wizards': 'washington wizards',
        'raptors': 'toronto raptors',
        'thunder': 'oklahoma city thunder',
        'okc': 'oklahoma city thunder',
        'pistons': 'detroit pistons',
        'hornets': 'charlotte hornets',
        'grizzlies': 'memphis grizzlies',
        'grizz': 'memphis grizzlies'
    }

    # Check if the name is in mappings
    for key, value in mappings.items():
        if key in name:
            return value

    return name

def load_and_cache_official_schedule():
    """Loads and caches the official NBA schedule for today"""
    # Get today's date in Pacific Time (NBA standard timezone)
    pacific_tz = pytz.timezone('America/Los_Angeles')
    today = datetime.now(pacific_tz).strftime('%Y-%m-%d')

    try:
        # Try to fetch from database
        response = supabase.table("nba_game_schedule").select("*").eq("game_date", today).execute()

        if response.data:
            schedule_df = pd.DataFrame(response.data)
            print(f"Loaded {len(schedule_df)} games from official schedule for {today}")

            # Add normalized team names for better matching
            schedule_df['home_team_norm'] = schedule_df['home_team'].apply(normalize_team_name)
            schedule_df['away_team_norm'] = schedule_df['away_team'].apply(normalize_team_name)

            global OFFICIAL_SCHEDULE
            OFFICIAL_SCHEDULE = schedule_df
            
            # Print the matchups
            for _, game in schedule_df.iterrows():
                print(f"  - {game['home_team']} vs {game['away_team']}")
            
            return schedule_df
        else:
            print(f"No scheduled games found for today ({today})")
            return pd.DataFrame()
    except Exception as e:
        print(f"Error loading official schedule: {e}")
        traceback.print_exc()
        return pd.DataFrame()

def fetch_live_games_with_schedule_matching():
    """Fetch live games and match against the official schedule"""
    # First load the official schedule
    global OFFICIAL_SCHEDULE
    if OFFICIAL_SCHEDULE is None or OFFICIAL_SCHEDULE.empty:
        OFFICIAL_SCHEDULE = load_and_cache_official_schedule()

    try:
        # Get current date in Pacific Time
        pacific_tz = pytz.timezone('America/Los_Angeles')
        today_pacific = datetime.now(pacific_tz).strftime('%Y-%m-%d')

        print(f"Fetching live games for {today_pacific} (Pacific Time)")

        response = supabase.table("nba_live_game_stats").select("*").execute()
        
        if not response.data:
            print(f"No live games found. Checking for scheduled games...")
            
            # If there are scheduled games but no live games, convert schedule to live format
            if not OFFICIAL_SCHEDULE.empty:
                print("No live games found, but scheduled games exist. Creating pre-game entries.")
                live_games_df = convert_scheduled_to_live_format(OFFICIAL_SCHEDULE)
                return live_games_df
            else:
                print("No live or scheduled games found.")
                return pd.DataFrame()

        live_df = pd.DataFrame(response.data)
        
        # Filter for today's games if we have a date column
        if 'game_date' in live_df.columns:
            # Convert to datetime to handle different date formats
            live_df['game_date'] = pd.to_datetime(live_df['game_date'])
            today_dt = pd.to_datetime(today_pacific)
            live_df = live_df[live_df['game_date'].dt.date == today_dt.date()]
        
        print(f"Found {len(live_df)} live game records")

        # Validate and clean game data
        live_df = validate_game_data(live_df)

        # Match against the schedule if available
        if not OFFICIAL_SCHEDULE.empty:
            matched_df = match_teams_to_schedule(live_df, OFFICIAL_SCHEDULE)
            print(f"Matched {matched_df['matched_to_schedule'].sum()} of {len(matched_df)} games to schedule")
            return matched_df
        else:
            # If no schedule, just return the validated live data
            return live_df

    except Exception as e:
        print(f"Error fetching live games: {e}")
        traceback.print_exc()

        # If live data fails but we have the schedule, use that
        if not OFFICIAL_SCHEDULE.empty:
            print("Using schedule data as fallback")
            return convert_scheduled_to_live_format(OFFICIAL_SCHEDULE)

        return pd.DataFrame()

def find_recent_games_for_testing():
    """Find recent completed games to use for testing the prediction model"""
    print("No live games found. Fetching recent completed games for testing...")

    # Get dates to try, in order of preference
    try:
        dates_to_try = []
        today = datetime.now()

        # Try yesterday first, then previous days
        for i in range(1, 5):  # Try up to 4 previous days
            date = (today - timedelta(days=i)).strftime('%Y-%m-%d')
            dates_to_try.append(date)

        # Try each date in sequence
        for test_date in dates_to_try:
            response = supabase.table("nba_historical_game_stats")\
                .select("*")\
                .eq("game_date", test_date)\
                .limit(5).execute()

            historical_games = response.data
            if historical_games:
                print(f"Found {len(historical_games)} games from {test_date} for testing.")

                # Simulate these as 'live' games by setting them to random quarters
                import random

                live_games = []
                for game in historical_games:
                    # Randomly select a quarter for simulation (1-4)
                    sim_quarter = random.randint(1, 4)

                    # Create a simulated live game where we only know scores up to the simulated quarter
                    sim_game = {
                        'game_id': game['game_id'],
                        'home_team': game['home_team'],
                        'away_team': game['away_team'],
                        'game_date': game['game_date'],
                        'current_quarter': sim_quarter,
                        'simulated_quarter': sim_quarter,  # Mark which quarter we're simulating
                        'matched_to_schedule': True  # Pretend it's matched
                    }

                    # Add quarter scores up to the simulated quarter
                    for q in range(1, 5):
                        q_field_home = f'home_q{q}'
                        q_field_away = f'away_q{q}'

                        if q <= sim_quarter and q_field_home in game and q_field_away in game:
                            sim_game[q_field_home] = game[q_field_home]
                            sim_game[q_field_away] = game[q_field_away]
                        else:
                            sim_game[q_field_home] = 0
                            sim_game[q_field_away] = 0

                    # Calculate the current score based on quarters we "know"
                    sim_game['home_score'] = sum([
                        float(sim_game.get(f'home_q{q}', 0) or 0) for q in range(1, sim_quarter+1)
                    ])
                    sim_game['away_score'] = sum([
                        float(sim_game.get(f'away_q{q}', 0) or 0) for q in range(1, sim_quarter+1)
                    ])

                    # Save the actual final scores for validation
                    sim_game['actual_final_home'] = game['home_score']
                    sim_game['actual_final_away'] = game['away_score']

                    live_games.append(sim_game)

                return pd.DataFrame(live_games)

        print("No recent games found for testing.")
        return pd.DataFrame()
    except Exception as e:
        print(f"Error finding recent games: {e}")
        traceback.print_exc()
        return pd.DataFrame()

def get_team_rolling_averages(days_lookback=60):
    """
    Retrieves the rolling scoring average for each team from historical data.
    
    Args:
        days_lookback: Number of days to look back for calculating the average (default: 60)
        
    Returns:
        Dictionary mapping team names to their rolling scoring average
    """
    try:
        # Calculate the date threshold
        threshold_date = (datetime.now() - timedelta(days=days_lookback)).strftime('%Y-%m-%d')
        
        # Fetch recent historical game data
        response = supabase.table("nba_historical_game_stats").select("*").gte("game_date", threshold_date).execute()
        historical_data = response.data
        
        if not historical_data:
            print(f"No historical game data available from the last {days_lookback} days.")
            return {}
        
        df = pd.DataFrame(historical_data)
        df['game_date'] = pd.to_datetime(df['game_date'])
        df = df.sort_values('game_date')
        
        # Initialize dictionary for team averages
        team_avgs = {}
        
        # Get unique teams
        all_teams = set(df['home_team'].unique()) | set(df['away_team'].unique())
        
        for team in all_teams:
            # Get home games where team is home
            home_games = df[df['home_team'] == team][['game_date', 'home_score']].rename(
                columns={'home_score': 'score'})
            
            # Get away games where team is away
            away_games = df[df['away_team'] == team][['game_date', 'away_score']].rename(
                columns={'away_score': 'score'})
            
            # Combine all games
            team_games = pd.concat([home_games, away_games]).sort_values('game_date')
            
            if not team_games.empty:
                # Calculate recent average (last 5 games if available)
                recent_games = team_games.tail(5)
                team_avgs[team] = recent_games['score'].mean()
            else:
                # Fallback to a reasonable default
                team_avgs[team] = 105.0  # NBA average is approximately 100-110 points per game
        
        return team_avgs
    except Exception as e:
        print(f"Error getting team rolling averages: {e}")
        return {}  # Return empty dict on error

def get_rest_data(team_name, game_date):
    """
    Calculates rest days for a team before a specific game date.
    
    Args:
        team_name: Name of the team
        game_date: Date of the current game
        
    Returns:
        Dictionary with rest days and back-to-back status
    """
    try:
        # Ensure game_date is datetime
        if isinstance(game_date, str):
            game_date = pd.to_datetime(game_date)
        
        # Try to find previous game with a simple query approach
        try:
            # First attempt - check home games
            home_response = supabase.table("nba_historical_game_stats")\
                .select("game_date")\
                .eq("home_team", team_name)\
                .lt("game_date", game_date.strftime('%Y-%m-%d'))\
                .order("game_date", desc=True)\
                .limit(1)\
                .execute()
                
            # Second attempt - check away games
            away_response = supabase.table("nba_historical_game_stats")\
                .select("game_date")\
                .eq("away_team", team_name)\
                .lt("game_date", game_date.strftime('%Y-%m-%d'))\
                .order("game_date", desc=True)\
                .limit(1)\
                .execute()
            
            # Combine results and find most recent
            all_games = home_response.data + away_response.data
            if not all_games:
                # If no previous game found, return default
                return {
                    'rest_days': 3,  # Default if no history
                    'is_back_to_back': False
                }
                
            # Sort by date (most recent first)
            all_games.sort(key=lambda x: x['game_date'], reverse=True)
            prev_game_date = pd.to_datetime(all_games[0]['game_date'])
            
        except Exception as e:
            print(f"Error in first query approach: {e}")
            # Try with normalized team name as fallback
            norm_team = normalize_team_name(team_name)
            if norm_team != team_name:
                try:
                    # Look for the team's previous game with normalized name
                    home_response = supabase.table("nba_historical_game_stats")\
                        .select("game_date")\
                        .eq("home_team", norm_team)\
                        .lt("game_date", game_date.strftime('%Y-%m-%d'))\
                        .order("game_date", desc=True)\
                        .limit(1)\
                        .execute()
                        
                    away_response = supabase.table("nba_historical_game_stats")\
                        .select("game_date")\
                        .eq("away_team", norm_team)\
                        .lt("game_date", game_date.strftime('%Y-%m-%d'))\
                        .order("game_date", desc=True)\
                        .limit(1)\
                        .execute()
                    
                    all_games = home_response.data + away_response.data
                    if not all_games:
                        return {
                            'rest_days': 3,
                            'is_back_to_back': False
                        }
                        
                    all_games.sort(key=lambda x: x['game_date'], reverse=True)
                    prev_game_date = pd.to_datetime(all_games[0]['game_date'])
                except Exception as e2:
                    print(f"Error in fallback query: {e2}")
                    return {
                        'rest_days': 2,  # Default if all queries fail
                        'is_back_to_back': False
                    }
            else:
                return {
                    'rest_days': 2,
                    'is_back_to_back': False
                }
        
        # Calculate days between games
        rest_days = (game_date - prev_game_date).days
        
        # Cap to reasonable values
        rest_days = max(min(rest_days, 10), 0)
        
        return {
            'rest_days': rest_days,
            'is_back_to_back': rest_days <= 1
        }
    except Exception as e:
        print(f"Error getting rest data for {team_name}: {e}")
        # Return default values on error
        return {
            'rest_days': 2,  # Reasonable default
            'is_back_to_back': False
        }

def get_previous_matchup_diff(home_team, away_team, max_lookback=5):
    """Gets the point differential from previous matchups between two teams."""
    try:
        # Use separate queries for home and away configurations to avoid syntax issues
        response_home = supabase.table("nba_historical_game_stats").select("*")\
            .eq("home_team", home_team)\
            .eq("away_team", away_team)\
            .order('game_date', desc=True)\
            .limit(max_lookback).execute()
            
        response_away = supabase.table("nba_historical_game_stats").select("*")\
            .eq("home_team", away_team)\
            .eq("away_team", home_team)\
            .order('game_date', desc=True)\
            .limit(max_lookback).execute()
        
        # Combine results
        home_matchups = response_home.data
        away_matchups = response_away.data
        matchups = home_matchups + away_matchups
        
        # Sort by date (most recent first)
        if matchups:
            matchups.sort(key=lambda x: x['game_date'], reverse=True)
            matchups = matchups[:max_lookback]
        
        if not matchups:
            return 0
        
        # Calculate point differential from home team perspective
        differentials = []
        for game in matchups:
            if game['home_team'] == home_team and game['away_team'] == away_team:
                diff = game['home_score'] - game['away_score']
            elif game['home_team'] == away_team and game['away_team'] == home_team:
                diff = game['away_score'] - game['home_score']
            else:
                continue
            differentials.append(diff)
        
        # Cap extreme values
        avg_diff = sum(differentials) / len(differentials) if differentials else 0
        return max(min(avg_diff, 15), -15)  # Cap at +/- 15 points
    except Exception as e:
        print(f"Error getting previous matchups: {e}")
        return 0

def convert_scheduled_to_live_format(schedule_df):
    # If you actually need to convert scheduled games to a live format, re-implement
    # For now, return an empty or minimal DataFrame
    return pd.DataFrame()
    
    for _, game in schedule_df.iterrows():
        # Create a basic live game structure
        live_game = {
            'game_id': game['game_id'],
            'home_team': game['home_team'],
            'away_team': game['away_team'],
            'game_date': game['game_date'],
            'matched_to_schedule': True,
            'current_quarter': 0,  # Pre-game
            'home_q1': 0, 'home_q2': 0, 'home_q3': 0, 'home_q4': 0,
            'away_q1': 0, 'away_q2': 0, 'away_q3': 0, 'away_q4': 0,
            'home_score': 0,
            'away_score': 0
        }
        live_format.append(live_game)
    
    return pd.DataFrame(live_format)

def validate_game_data(df):
    """Validate and clean game data"""
    # Make a copy to avoid modifying the original
    df = df.copy()
    
    # Convert date fields to datetime
    if 'game_date' in df.columns:
        df['game_date'] = pd.to_datetime(df['game_date'], errors='coerce')
    
    # Ensure quarter fields are numeric
    for q in range(1, 5):
        home_q = f'home_q{q}'
        away_q = f'away_q{q}'
        
        if home_q in df.columns:
            df[home_q] = pd.to_numeric(df[home_q], errors='coerce').fillna(0)
        if away_q in df.columns:
            df[away_q] = pd.to_numeric(df[away_q], errors='coerce').fillna(0)
    
    # Calculate current quarter based on non-zero quarter scores
    df['current_quarter'] = 0
    for idx, row in df.iterrows():
        max_quarter = 0
        for q in range(1, 5):
            home_q = f'home_q{q}'
            away_q = f'away_q{q}'
            
            if ((home_q in row and pd.notnull(row[home_q]) and row[home_q] > 0) or 
                (away_q in row and pd.notnull(row[away_q]) and row[away_q] > 0)):
                max_quarter = q
        
        df.at[idx, 'current_quarter'] = max_quarter
    
    # Ensure team names are strings
    if 'home_team' in df.columns:
        df['home_team'] = df['home_team'].astype(str)
    if 'away_team' in df.columns:
        df['away_team'] = df['away_team'].astype(str)
    
    # Calculate current score as sum of quarters
    if 'home_score' not in df.columns:
        df['home_score'] = 0
    if 'away_score' not in df.columns:
        df['away_score'] = 0
        
    for idx, row in df.iterrows():
        current_q = int(row.get('current_quarter', 0) or 0)
        
        home_score = 0
        away_score = 0
        for q in range(1, current_q + 1):
            home_q = f'home_q{q}'
            away_q = f'away_q{q}'
            
            if home_q in row:
                home_score += float(row[home_q] or 0)
            if away_q in row:
                away_score += float(row[away_q] or 0)
        
        df.at[idx, 'home_score'] = home_score
        df.at[idx, 'away_score'] = away_score
    
    # Calculate score ratio
    df['score_ratio'] = 0.5  # Default to even
    for idx, row in df.iterrows():
        home_score = float(row.get('home_score', 0) or 0)
        away_score = float(row.get('away_score', 0) or 0)
        total = home_score + away_score
        
        if total > 0:
            df.at[idx, 'score_ratio'] = home_score / total
    
    # Initialize matched_to_schedule column if not present
    if 'matched_to_schedule' not in df.columns:
        df['matched_to_schedule'] = False
    
    return df

def match_teams_to_schedule(live_df, schedule_df):
    # If you actually need to match team names, re-implement the real logic
    # For now, just return live_df as-is or with a 'verified' column
    live_df['verified'] = False
    return live_df
    # Copy to avoid modifying the original
    live_df = live_df.copy()
    
    # Add normalized team names for better matching
    live_df['home_team_norm'] = live_df['home_team'].apply(normalize_team_name)
    live_df['away_team_norm'] = live_df['away_team'].apply(normalize_team_name)
    
    # Make sure schedule has normalized teams too
    if 'home_team_norm' not in schedule_df.columns:
        schedule_df = schedule_df.copy()
        schedule_df['home_team_norm'] = schedule_df['home_team'].apply(normalize_team_name)
        schedule_df['away_team_norm'] = schedule_df['away_team'].apply(normalize_team_name)
    
    # Try to match by exact normalized team names
    for live_idx, live_game in live_df.iterrows():
        # Skip already matched games
        if live_game['matched_to_schedule']:
            continue
            
        home_norm = live_game['home_team_norm']
        away_norm = live_game['away_team_norm']
        
        # Try both home/away combinations
        home_away_match = schedule_df[
            (schedule_df['home_team_norm'] == home_norm) &
            (schedule_df['away_team_norm'] == away_norm)
        ]
        
        away_home_match = schedule_df[
            (schedule_df['home_team_norm'] == away_norm) &
            (schedule_df['away_team_norm'] == home_norm)
        ]
        
        if not home_away_match.empty:
            # Found exact match
            match = home_away_match.iloc[0]
            live_df.at[live_idx, 'matched_to_schedule'] = True
            # Copy over the official game_id if needed
            if 'game_id' in match and match['game_id'] != live_game['game_id']:
                live_df.at[live_idx, 'original_game_id'] = live_game['game_id']
                live_df.at[live_idx, 'game_id'] = match['game_id']
                
        elif not away_home_match.empty:
            # Found match with teams reversed - this indicates a data issue
            print(f"Warning: Found teams in reverse order for game {live_game['game_id']}")
            match = away_home_match.iloc[0]
            live_df.at[live_idx, 'matched_to_schedule'] = True
            live_df.at[live_idx, 'teams_reversed'] = True
            # Copy over the official game_id if needed
            if 'game_id' in match and match['game_id'] != live_game['game_id']:
                live_df.at[live_idx, 'original_game_id'] = live_game['game_id']
                live_df.at[live_idx, 'game_id'] = match['game_id']
    
    # For unmatched games, try looser matching
    if not live_df[~live_df['matched_to_schedule']].empty:
        for live_idx, live_game in live_df[~live_df['matched_to_schedule']].iterrows():
            best_match = None
            best_score = 0
            is_reversed = False
            
            for _, sched_game in schedule_df.iterrows():
                # Try both directions
                direct_match = (live_game['home_team_norm'] in sched_game['home_team_norm'] or 
                               sched_game['home_team_norm'] in live_game['home_team_norm']) and \
                              (live_game['away_team_norm'] in sched_game['away_team_norm'] or 
                               sched_game['away_team_norm'] in live_game['away_team_norm'])
                               
                reverse_match = (live_game['home_team_norm'] in sched_game['away_team_norm'] or 
                                sched_game['away_team_norm'] in live_game['home_team_norm']) and \
                               (live_game['away_team_norm'] in sched_game['home_team_norm'] or 
                                sched_game['home_team_norm'] in live_game['away_team_norm'])
                
                # Simple scoring - each match is worth 1 point
                score = direct_match + reverse_match * 0.9  # Slightly prefer direct matches
                
                if score > best_score:
                    best_score = score
                    best_match = sched_game
                    is_reversed = reverse_match > direct_match
            
            if best_match is not None and best_score > 0:
                live_df.at[live_idx, 'matched_to_schedule'] = True
                live_df.at[live_idx, 'match_confidence'] = best_score
                live_df.at[live_idx, 'teams_reversed'] = is_reversed
                
                # Copy over the official game_id if needed
                if 'game_id' in best_match and best_match['game_id'] != live_game['game_id']:
                    live_df.at[live_idx, 'original_game_id'] = live_game['game_id']
                    live_df.at[live_idx, 'game_id'] = best_match['game_id']
    
    return live_df

def run_enhanced_prediction_pipeline():
    """
    Run the full prediction pipeline with all improvements
    
    1. Ensure official schedule is loaded
    2. Fetch live games with schedule matching
    3. If no live games, use historical games for testing
    4. Calculate enhanced features
    5. Generate predictions using the appropriate model
    6. Calculate confidence metrics and win probabilities
    7. Return predictions with validation metrics when possible
    """
    # 1. Load official schedule
    global OFFICIAL_SCHEDULE
    if OFFICIAL_SCHEDULE is None:
        OFFICIAL_SCHEDULE = load_and_cache_official_schedule()

    # 2. Fetch live games
    live_games_df = fetch_live_games_with_schedule_matching()

    # 3. Fall back to historical games if needed
    if live_games_df.empty:
        live_games_df = find_recent_games_for_testing()

        if live_games_df.empty:
            print("No games available for prediction")
            return pd.DataFrame()

    # 4. Calculate enhanced features
    try:
        # Get team rolling averages
        team_avgs = get_team_rolling_averages()

        # Calculate rest features
        for idx, game in live_games_df.iterrows():
            try:
                home_team = game['home_team']
                away_team = game['away_team']
                game_date = pd.to_datetime(game['game_date'])

                # Get rest data for both teams
                home_rest = get_rest_data(home_team, game_date)
                away_rest = get_rest_data(away_team, game_date)

                # Update DataFrame
                live_games_df.at[idx, 'rest_days_home'] = home_rest['rest_days']
                live_games_df.at[idx, 'rest_days_away'] = away_rest['rest_days']
                live_games_df.at[idx, 'is_back_to_back_home'] = int(home_rest['is_back_to_back'])
                live_games_df.at[idx, 'is_back_to_back_away'] = int(away_rest['is_back_to_back'])
                live_games_df.at[idx, 'rest_advantage'] = home_rest['rest_days'] - away_rest['rest_days']

                # Get previous matchup difference
                live_games_df.at[idx, 'prev_matchup_diff'] = get_previous_matchup_diff(home_team, away_team)
            except Exception as e:
                print(f"Error calculating rest features for game {game.get('game_id')}: {e}")
                # Set default values
                live_games_df.at[idx, 'rest_days_home'] = 2
                live_games_df.at[idx, 'rest_days_away'] = 2
                live_games_df.at[idx, 'is_back_to_back_home'] = 0
                live_games_df.at[idx, 'is_back_to_back_away'] = 0
                live_games_df.at[idx, 'rest_advantage'] = 0
                live_games_df.at[idx, 'prev_matchup_diff'] = 0

        # Calculate momentum features
        for idx, game in live_games_df.iterrows():
            try:
                current_quarter = int(game.get('current_quarter', 0) or 0)

                # Initialize momentum features
                live_games_df.at[idx, 'q1_to_q2_momentum'] = 0
                live_games_df.at[idx, 'q2_to_q3_momentum'] = 0
                live_games_df.at[idx, 'q3_to_q4_momentum'] = 0
                live_games_df.at[idx, 'cumulative_momentum'] = 0

                # Calculate quarter-to-quarter momentum
                if current_quarter >= 2:
                    # Q1 to Q2
                    home_q1 = float(game.get('home_q1', 0) or 0)
                    home_q2 = float(game.get('home_q2', 0) or 0)
                    away_q1 = float(game.get('away_q1', 0) or 0)
                    away_q2 = float(game.get('away_q2', 0) or 0)

                    q1_to_q2 = (home_q2 - home_q1) - (away_q2 - away_q1)
                    # Cap to prevent extreme values
                    q1_to_q2 = max(min(q1_to_q2, 15), -15)
                    live_games_df.at[idx, 'q1_to_q2_momentum'] = q1_to_q2

                if current_quarter >= 3:
                    # Q2 to Q3
                    home_q2 = float(game.get('home_q2', 0) or 0)
                    home_q3 = float(game.get('home_q3', 0) or 0)
                    away_q2 = float(game.get('away_q2', 0) or 0)
                    away_q3 = float(game.get('away_q3', 0) or 0)

                    q2_to_q3 = (home_q3 - home_q2) - (away_q3 - away_q2)
                    q2_to_q3 = max(min(q2_to_q3, 15), -15)
                    live_games_df.at[idx, 'q2_to_q3_momentum'] = q2_to_q3

                if current_quarter >= 4:
                    # Q3 to Q4
                    home_q3 = float(game.get('home_q3', 0) or 0)
                    home_q4 = float(game.get('home_q4', 0) or 0)
                    away_q3 = float(game.get('away_q3', 0) or 0)
                    away_q4 = float(game.get('away_q4', 0) or 0)

                    q3_to_q4 = (home_q4 - home_q3) - (away_q4 - away_q3)
                    q3_to_q4 = max(min(q3_to_q4, 15), -15)
                    live_games_df.at[idx, 'q3_to_q4_momentum'] = q3_to_q4

                # Calculate weighted momentum
                weights = [0.2, 0.3, 0.5]  # Weights for each momentum period

                if current_quarter == 2:
                    momentum = live_games_df.at[idx, 'q1_to_q2_momentum']
                elif current_quarter == 3:
                    momentum = (
                        live_games_df.at[idx, 'q1_to_q2_momentum'] * weights[0] +
                        live_games_df.at[idx, 'q2_to_q3_momentum'] * weights[1]
                    ) / (weights[0] + weights[1])
                elif current_quarter >= 4:
                    momentum = (
                        live_games_df.at[idx, 'q1_to_q2_momentum'] * weights[0] +
                        live_games_df.at[idx, 'q2_to_q3_momentum'] * weights[1] +
                        live_games_df.at[idx, 'q3_to_q4_momentum'] * weights[2]
                    ) / sum(weights)
                else:
                    momentum = 0

                # Normalize to [-1, 1]
                live_games_df.at[idx, 'cumulative_momentum'] = max(min(momentum / 15.0, 1.0), -1.0)

            except Exception as e:
                print(f"Error calculating momentum for game {game.get('game_id')}: {e}")
                # Set defaults
                live_games_df.at[idx, 'q1_to_q2_momentum'] = 0
                live_games_df.at[idx, 'q2_to_q3_momentum'] = 0
                live_games_df.at[idx, 'q3_to_q4_momentum'] = 0
                live_games_df.at[idx, 'cumulative_momentum'] = 0

    except Exception as e:
        print(f"Error calculating enhanced features: {e}")
        traceback.print_exc()
        # Continue with basic features

    # 5. Make predictions
    try:
        # Load model from global scope or load it if not present
        model_to_use = None
        if 'model' in globals() and globals()['model'] is not None:
            model_to_use = globals()['model']
            print("Using 'model' from global scope")
        else:
            # Try to load model from various locations
            try:
                model_path = config.MODEL_PATH
                if os.path.exists(model_path):
                    model_to_use = joblib.load(model_path)
                    print(f"Loaded model from {model_path}")
            except Exception as e:
                print(f"Failed to load model: {e}")

        if model_to_use is None:
            print("No model available for prediction")
            return live_games_df  # Return with features but no predictions

        # Determine feature list based on model
        if hasattr(model_to_use, 'feature_importances_'):
            feature_count = len(model_to_use.feature_importances_)
            if feature_count > 8:
                # Enhanced model
                features_to_use = [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4',
                    'score_ratio', 'prev_matchup_diff',
                    'rest_days_home', 'rest_days_away', 'rest_advantage',
                    'is_back_to_back_home', 'is_back_to_back_away',
                    'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
                ]
                print("Using enhanced feature set for prediction")
            else:
                # Original model
                features_to_use = [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4',
                    'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
                ]

                # Add rolling scores if needed
                if 'rolling_home_score' not in live_games_df.columns:
                    live_games_df['rolling_home_score'] = live_games_df['home_team'].apply(
                        lambda t: team_avgs.get(t, 105.0))
                    live_games_df['rolling_away_score'] = live_games_df['away_team'].apply(
                        lambda t: team_avgs.get(t, 105.0))

                print("Using original feature set for prediction")
        else:
            # Default to enhanced features
            features_to_use = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4',
                'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff',
                'rest_days_home', 'rest_days_away', 'rest_advantage',
                'is_back_to_back_home', 'is_back_to_back_away',
                'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
            ]
            print("Using default enhanced feature set")

        # Make sure all required features exist and have reasonable values
        for feature in features_to_use:
            if feature not in live_games_df.columns:
                print(f"Adding missing feature: {feature}")
                live_games_df[feature] = 0
            
            # Convert to numeric and handle NaN values
            live_games_df[feature] = pd.to_numeric(live_games_df[feature], errors='coerce').fillna(0)

        # Prepare feature matrix
        X_pred = live_games_df[features_to_use].copy()

        # Generate predictions
        predictions = model_to_use.predict(X_pred)
        live_games_df['predicted_home_score'] = predictions

        # Calculate additional metrics
        for idx, row in live_games_df.iterrows():
            # Estimate away score (if model only predicts home score)
            # Use simple home court advantage (avg ~3.5 points) and previous matchup differential
            home_advantage = 3.5
            matchup_diff = row.get('prev_matchup_diff', 0)
            live_games_df.at[idx, 'predicted_away_score'] = row['predicted_home_score'] - home_advantage - matchup_diff

            # Calculate remaining quarters
            current_quarter = int(row.get('current_quarter', 0) or 0)
            if current_quarter > 0 and current_quarter < 4:
                # Get current score
                current_home_score = sum([float(row.get(f'home_q{q}', 0) or 0) for q in range(1, current_quarter + 1)])
                current_away_score = sum([float(row.get(f'away_q{q}', 0) or 0) for q in range(1, current_quarter + 1)])
                
                # Calculate remaining score
                remaining_home = row['predicted_home_score'] - current_home_score
                remaining_away = row['predicted_away_score'] - current_away_score
                
                # Store remaining scores
                live_games_df.at[idx, 'remaining_home_score'] = max(0, remaining_home)
                live_games_df.at[idx, 'remaining_away_score'] = max(0, remaining_away)
            
            # Calculate win probability
            score_diff = row['predicted_home_score'] - row['predicted_away_score']
            # Simple logistic function for win probability
            win_prob = 1 / (1 + np.exp(-0.1 * score_diff))
            live_games_df.at[idx, 'win_probability'] = win_prob
            
            # Calculate confidence based on quarter
            if current_quarter == 0:
                confidence = 0.5  # Pre-game
            elif current_quarter == 1:
                confidence = 0.6  # 1st quarter
            elif current_quarter == 2:
                confidence = 0.7  # 2nd quarter
            elif current_quarter == 3:
                confidence = 0.85  # 3rd quarter
            else:
                confidence = 0.95  # 4th quarter
            
            live_games_df.at[idx, 'prediction_confidence'] = confidence
            
            # If we have actual final scores (for testing), calculate errors
            if 'actual_final_home' in row and 'actual_final_away' in row:
                live_games_df.at[idx, 'home_score_error'] = row['predicted_home_score'] - row['actual_final_home']
                live_games_df.at[idx, 'away_score_error'] = row['predicted_away_score'] - row['actual_final_away']
                
                # Calculate percentage error
                if row['actual_final_home'] > 0:
                    live_games_df.at[idx, 'home_error_pct'] = abs(row['home_score_error'] / row['actual_final_home']) * 100
                if row['actual_final_away'] > 0:
                    live_games_df.at[idx, 'away_error_pct'] = abs(row['away_score_error'] / row['actual_final_away']) * 100

        # 6. Generate recommendations
        for idx, row in live_games_df.iterrows():
            try:
                model_outputs = {
                    "win_probability": row['win_probability'],
                    "momentum_shift": row.get('cumulative_momentum', 0),
                    "projected_margin": row['predicted_home_score'] - row['predicted_away_score'],
                    "total_projected_score": row['predicted_home_score'] + row['predicted_away_score'],
                    "quarter": row.get('current_quarter', 0),
                    "time_remaining": None  # Would need to be provided by data source
                }
                
                recommendations = generate_recommendations(model_outputs)
                
                # Store top recommendations
                for rec_key, rec_value in recommendations.items():
                    live_games_df.at[idx, f'rec_{rec_key}'] = rec_value
            except Exception as e:
                print(f"Error generating recommendations for game {row.get('game_id')}: {e}")

    except Exception as e:
        print(f"Error making predictions: {e}")
        traceback.print_exc()

    # Return the dataframe with predictions
    return live_games_df

In [ ]:
# Cell 16 - Fetch official schedule from database

import numpy as np
import pandas as pd  # Added import for pandas
from datetime import datetime, timedelta
import time
import traceback
import pytz
import os
import joblib
import json
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from pathlib import Path
import logging
from typing import Dict, List, Optional, Tuple, Union

# Setup structured logging
logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger('nba_monitor')

# Import Supabase client if available, otherwise create a placeholder for demo purposes
try:
    from supabase import create_client, Client
    
    # Try to get environment variables
    import os
    SUPABASE_URL = os.getenv("SUPABASE_URL")
    SUPABASE_KEY = os.getenv("SUPABASE_KEY")
    
    # Create a supabase client if credentials are available
    if SUPABASE_URL and SUPABASE_KEY:
        supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
        logger.info("Supabase client initialized with credentials.")
    else:
        logger.warning("Supabase credentials not found. Using mock client for demo purposes.")
        supabase = None
except ImportError:
    logger.warning("Supabase package not installed. Using mock client for demo purposes.")
    supabase = None

class MockSupabaseClient:
    """A mock Supabase client for testing when real client isn't available"""
    
    def table(self, table_name):
        return MockSupabaseQuery(table_name)
    
class MockSupabaseQuery:
    """A mock query builder"""
    
    def __init__(self, table_name):
        self.table_name = table_name
        self.filters = []
        
    def select(self, columns):
        return self
        
    def eq(self, column, value):
        self.filters.append((column, "=", value))
        return self
        
    def lt(self, column, value):
        self.filters.append((column, "<", value))
        return self
        
    def gt(self, column, value):
        self.filters.append((column, ">", value))
        return self
        
    def gte(self, column, value):
        self.filters.append((column, ">=", value))
        return self
        
    def order(self, column, desc=False):
        return self
        
    def limit(self, num):
        return self
        
    def execute(self):
        # Return empty mock data
        return MockSupabaseResponse([])
        
class MockSupabaseResponse:
    """A mock response object"""
    
    def __init__(self, data):
        self.data = data


class DataFetcher:
    """
    Handles all data fetching operations from the database.
    """
    
    def __init__(self, supabase_client):
        """Initialize with Supabase client."""
        self.supabase = supabase_client if supabase_client else MockSupabaseClient()

    def get_official_schedule(self, date=None):
        """Load the official NBA schedule for a specific date (defaults to today)."""
        # Get today's date in Pacific Time if not specified
        if date is None:
            pacific_tz = pytz.timezone('America/Los_Angeles')
            date = datetime.now(pacific_tz).strftime('%Y-%m-%d')
            
        try:
            # Fetch from database
            response = self.supabase.table("nba_game_schedule").select("*").eq("game_date", date).execute()

            if response.data:
                schedule_df = pd.DataFrame(response.data)
                logger.info(f"Loaded {len(schedule_df)} games from official schedule for {date}")
                return schedule_df
            else:
                logger.warning(f"No scheduled games found for {date}")
                return pd.DataFrame()
        except Exception as e:
            logger.error(f"Error loading official schedule: {e}")
            traceback.print_exc()
            return pd.DataFrame()
    
    def get_live_games(self):
        """Fetch current live games."""
        try:
            # Get current date in Pacific Time
            pacific_tz = pytz.timezone('America/Los_Angeles')
            today_pacific = datetime.now(pacific_tz).strftime('%Y-%m-%d')

            logger.info(f"Fetching live games for {today_pacific} (Pacific Time)")
            response = self.supabase.table("nba_live_game_stats").select("*").execute()
            
            if not response.data:
                logger.info("No live games found.")
                return pd.DataFrame()

            return pd.DataFrame(response.data)
            
        except Exception as e:
            logger.error(f"Error fetching live games: {e}")
            traceback.print_exc()
            return pd.DataFrame()
    
    def get_team_rolling_averages(self, days_lookback=60):
        """Get team rolling scoring averages from historical data."""
        try:
            # Calculate the date threshold
            threshold_date = (datetime.now() - timedelta(days=days_lookback)).strftime('%Y-%m-%d')
            
            # Fetch recent historical game data
            response = self.supabase.table("nba_historical_game_stats").select("*").gte("game_date", threshold_date).execute()
            historical_data = response.data
            
            if not historical_data:
                logger.warning(f"No historical game data available from the last {days_lookback} days.")
                return {}
            
            df = pd.DataFrame(historical_data)
            df['game_date'] = pd.to_datetime(df['game_date'])
            df = df.sort_values('game_date')
            
            # Initialize dictionary for team averages
            team_avgs = {}
            
            # Get unique teams
            all_teams = set(df['home_team'].unique()) | set(df['away_team'].unique())
            
            for team in all_teams:
                # Get home games where team is home
                home_games = df[df['home_team'] == team][['game_date', 'home_score']].rename(
                    columns={'home_score': 'score'})
                
                # Get away games where team is away
                away_games = df[df['away_team'] == team][['game_date', 'away_score']].rename(
                    columns={'away_score': 'score'})
                
                # Combine all games
                team_games = pd.concat([home_games, away_games]).sort_values('game_date')
                
                if not team_games.empty:
                    # Calculate recent average (last 5 games if available)
                    recent_games = team_games.tail(5)
                    team_avgs[team] = recent_games['score'].mean()
                else:
                    # Fallback to a reasonable default
                    team_avgs[team] = 105.0  # NBA average is approximately 100-110 points per game
            
            return team_avgs
        except Exception as e:
            logger.error(f"Error getting team rolling averages: {e}")
            traceback.print_exc()
            return {}
    
    def get_rest_data(self, team_name, game_date):
        """Calculate rest days for a team before a specific game."""
        try:
            # Ensure game_date is datetime
            if isinstance(game_date, str):
                game_date = pd.to_datetime(game_date)
            
            # Try to find previous game
            home_response = self.supabase.table("nba_historical_game_stats")\
                .select("game_date")\
                .eq("home_team", team_name)\
                .lt("game_date", game_date.strftime('%Y-%m-%d'))\
                .order("game_date", desc=True)\
                .limit(1)\
                .execute()
                
            away_response = self.supabase.table("nba_historical_game_stats")\
                .select("game_date")\
                .eq("away_team", team_name)\
                .lt("game_date", game_date.strftime('%Y-%m-%d'))\
                .order("game_date", desc=True)\
                .limit(1)\
                .execute()
            
            # Combine results and find most recent
            all_games = home_response.data + away_response.data
            if not all_games:
                # If no previous game found, return default
                return {
                    'rest_days': 3,  # Default if no history
                    'is_back_to_back': False
                }
                
            # Sort by date (most recent first)
            all_games.sort(key=lambda x: x['game_date'], reverse=True)
            prev_game_date = pd.to_datetime(all_games[0]['game_date'])
            
            # Calculate days between games
            rest_days = (game_date - prev_game_date).days
            
            # Cap to reasonable values
            rest_days = max(min(rest_days, 10), 0)
            
            return {
                'rest_days': rest_days,
                'is_back_to_back': rest_days <= 1
            }
        except Exception as e:
            logger.error(f"Error getting rest data for {team_name}: {e}")
            traceback.print_exc()
            # Return default values on error
            return {
                'rest_days': 2,  # Reasonable default
                'is_back_to_back': False
            }
    
    def get_previous_matchup_diff(self, home_team, away_team, max_lookback=5):
        """Get point differential from previous matchups between two teams."""
        try:
            # Use separate queries for home and away configurations
            response_home = self.supabase.table("nba_historical_game_stats").select("*")\
                .eq("home_team", home_team)\
                .eq("away_team", away_team)\
                .order('game_date', desc=True)\
                .limit(max_lookback).execute()
                
            response_away = self.supabase.table("nba_historical_game_stats").select("*")\
                .eq("home_team", away_team)\
                .eq("away_team", home_team)\
                .order('game_date', desc=True)\
                .limit(max_lookback).execute()
            
            # Combine results
            home_matchups = response_home.data
            away_matchups = response_away.data
            matchups = home_matchups + away_matchups
            
            # Sort by date (most recent first)
            if matchups:
                matchups.sort(key=lambda x: x['game_date'], reverse=True)
                matchups = matchups[:max_lookback]
            
            if not matchups:
                return 0
            
            # Calculate point differential from home team perspective
            differentials = []
            for game in matchups:
                if game['home_team'] == home_team and game['away_team'] == away_team:
                    diff = game['home_score'] - game['away_score']
                elif game['home_team'] == away_team and game['away_team'] == home_team:
                    diff = game['away_score'] - game['home_score']
                else:
                    continue
                differentials.append(diff)
            
            # Cap extreme values
            avg_diff = sum(differentials) / len(differentials) if differentials else 0
            return max(min(avg_diff, 15), -15)  # Cap at +/- 15 points
        except Exception as e:
            logger.warning(f"Error getting previous matchups: {e}")
            return 0
    
    def find_recent_games_for_testing(self, days_to_look_back=5, limit=5):
        """Find recent completed games to use for testing the prediction model."""
        logger.info("Fetching recent completed games for testing...")

        # Get dates to try, in order of preference
        try:
            dates_to_try = []
            today = datetime.now()

            # Try previous days
            for i in range(1, days_to_look_back+1):
                date = (today - timedelta(days=i)).strftime('%Y-%m-%d')
                dates_to_try.append(date)

            # Try each date in sequence
            for test_date in dates_to_try:
                response = self.supabase.table("nba_historical_game_stats")\
                    .select("*")\
                    .eq("game_date", test_date)\
                    .limit(limit).execute()

                historical_games = response.data
                if historical_games:
                    logger.info(f"Found {len(historical_games)} games from {test_date} for testing.")

                    # Simulate these as 'live' games by setting them to random quarters
                    import random

                    live_games = []
                    for game in historical_games:
                        # Randomly select a quarter for simulation (1-4)
                        sim_quarter = random.randint(1, 4)

                        # Create a simulated live game where we only know scores up to the simulated quarter
                        sim_game = {
                            'game_id': game['game_id'],
                            'home_team': game['home_team'],
                            'away_team': game['away_team'],
                            'game_date': game['game_date'],
                            'current_quarter': sim_quarter,
                            'simulated_quarter': sim_quarter,  # Mark which quarter we're simulating
                            'matched_to_schedule': True,  # Pretend it's matched
                            'game_status': 'LIVE'
                        }

                        # Add quarter scores up to the simulated quarter
                        for q in range(1, 5):
                            q_field_home = f'home_q{q}'
                            q_field_away = f'away_q{q}'

                            if q <= sim_quarter and q_field_home in game and q_field_away in game:
                                sim_game[q_field_home] = game[q_field_home]
                                sim_game[q_field_away] = game[q_field_away]
                            else:
                                sim_game[q_field_home] = 0
                                sim_game[q_field_away] = 0

                        # Calculate the current score based on quarters we "know"
                        sim_game['home_score'] = sum([
                            float(sim_game.get(f'home_q{q}', 0) or 0) for q in range(1, sim_quarter+1)
                        ])
                        sim_game['away_score'] = sum([
                            float(sim_game.get(f'away_q{q}', 0) or 0) for q in range(1, sim_quarter+1)
                        ])

                        # Save the actual final scores for validation
                        sim_game['actual_home_final'] = game['home_score']
                        sim_game['actual_away_final'] = game['away_score']

                        live_games.append(sim_game)

                    return pd.DataFrame(live_games)

            logger.warning("No recent games found for testing.")
            return pd.DataFrame()
        except Exception as e:
            logger.error(f"Error finding recent games: {e}")
            traceback.print_exc()
            return pd.DataFrame()


class DataProcessor:
    """
    Handles data processing, cleaning, and feature engineering.
    """
    
    @staticmethod
    def normalize_team_name(name):
        """Normalize team names for consistent matching."""
        if not name:
            return ""

        # Convert to string and lowercase
        name = str(name).lower().strip()

        # Standard team name mappings
        mappings = {
            'lakers': 'los angeles lakers',
            'la lakers': 'los angeles lakers',
            'clippers': 'los angeles clippers',
            'la clippers': 'los angeles clippers',
            'blazers': 'portland trail blazers',
            'sixers': 'philadelphia 76ers',
            'philly': 'philadelphia 76ers',
            'knicks': 'new york knicks',
            'ny knicks': 'new york knicks',
            'nets': 'brooklyn nets',
            'mavs': 'dallas mavericks',
            'cavs': 'cleveland cavaliers',
            'wolves': 'minnesota timberwolves',
            't-wolves': 'minnesota timberwolves',
            'celts': 'boston celtics',
            'pels': 'new orleans pelicans',
            'warriors': 'golden state warriors',
            'gsw': 'golden state warriors',
            'heat': 'miami heat',
            'bulls': 'chicago bulls',
            'hawks': 'atlanta hawks',
            'suns': 'phoenix suns',
            'bucks': 'milwaukee bucks',
            'jazz': 'utah jazz',
            'nuggets': 'denver nuggets',
            'rockets': 'houston rockets',
            'pacers': 'indiana pacers',
            'spurs': 'san antonio spurs',
            'kings': 'sacramento kings',
            'magic': 'orlando magic',
            'wizards': 'washington wizards',
            'raptors': 'toronto raptors',
            'thunder': 'oklahoma city thunder',
            'okc': 'oklahoma city thunder',
            'pistons': 'detroit pistons',
            'hornets': 'charlotte hornets',
            'grizzlies': 'memphis grizzlies',
            'grizz': 'memphis grizzlies'
        }

        # Check if the name is in mappings
        for key, value in mappings.items():
            if key in name:
                return value

        return name
    
    def prepare_schedule_data(self, schedule_df):
        """Add normalized team names to schedule data."""
        if schedule_df.empty:
            return schedule_df
            
        # Add normalized team names for better matching
        result_df = schedule_df.copy()
        result_df['home_team_norm'] = result_df['home_team'].apply(self.normalize_team_name)
        result_df['away_team_norm'] = result_df['away_team'].apply(self.normalize_team_name)
        
        return result_df
    
    def validate_game_data(self, df):
        """Validate and clean game data."""
        if df.empty:
            return df
            
        # Make a copy to avoid modifying the original
        df = df.copy()
        
        # Convert date fields to datetime
        if 'game_date' in df.columns:
            df['game_date'] = pd.to_datetime(df['game_date'], errors='coerce')
        
        # Ensure quarter fields are numeric
        for q in range(1, 5):
            home_q = f'home_q{q}'
            away_q = f'away_q{q}'
            
            if home_q in df.columns:
                df[home_q] = pd.to_numeric(df[home_q], errors='coerce').fillna(0)
            else:
                df[home_q] = 0
                
            if away_q in df.columns:
                df[away_q] = pd.to_numeric(df[away_q], errors='coerce').fillna(0)
            else:
                df[away_q] = 0
        
        # Calculate current quarter based on non-zero quarter scores
        df['current_quarter'] = 0
        for idx, row in df.iterrows():
            max_quarter = 0
            for q in range(1, 5):
                home_q = f'home_q{q}'
                away_q = f'away_q{q}'
                
                if ((home_q in row and pd.notnull(row[home_q]) and row[home_q] > 0) or 
                    (away_q in row and pd.notnull(row[away_q]) and row[away_q] > 0)):
                    max_quarter = q
            
            # Only update if detected quarter is higher than existing value
            existing_q = row.get('current_quarter', 0)
            if pd.isna(existing_q) or existing_q < max_quarter:
                df.at[idx, 'current_quarter'] = max_quarter
        
        # Ensure team names are strings
        if 'home_team' in df.columns:
            df['home_team'] = df['home_team'].astype(str)
        if 'away_team' in df.columns:
            df['away_team'] = df['away_team'].astype(str)
        
        # Calculate current score as sum of quarters
        if 'home_score' not in df.columns:
            df['home_score'] = 0
        if 'away_score' not in df.columns:
            df['away_score'] = 0
            
        for idx, row in df.iterrows():
            current_q = int(row.get('current_quarter', 0) or 0)
            
            home_score = 0
            away_score = 0
            for q in range(1, current_q + 1):
                home_q = f'home_q{q}'
                away_q = f'away_q{q}'
                
                if home_q in row and pd.notnull(row[home_q]):
                    home_score += float(row[home_q] or 0)
                if away_q in row and pd.notnull(row[away_q]):
                    away_score += float(row[away_q] or 0)
            
            # Only update if calculated scores are different from existing values
            if abs(df.at[idx, 'home_score'] - home_score) > 0.1 or pd.isna(df.at[idx, 'home_score']):
                df.at[idx, 'home_score'] = home_score
            if abs(df.at[idx, 'away_score'] - away_score) > 0.1 or pd.isna(df.at[idx, 'away_score']):
                df.at[idx, 'away_score'] = away_score
        
        # Calculate score ratio
        df['score_ratio'] = 0.5  # Default to even
        for idx, row in df.iterrows():
            home_score = float(row.get('home_score', 0) or 0)
            away_score = float(row.get('away_score', 0) or 0)
            total = home_score + away_score
            
            if total > 0:
                df.at[idx, 'score_ratio'] = home_score / total
        
        # Initialize matched_to_schedule column if not present
        if 'matched_to_schedule' not in df.columns:
            df['matched_to_schedule'] = False
        
        # Add game status column
        if 'game_status' not in df.columns:
            df['game_status'] = 'UNKNOWN'
            for idx, row in df.iterrows():
                status, quarter, time_remaining = self._determine_game_status(row)
                df.at[idx, 'game_status'] = status
                df.at[idx, 'time_remaining_mins'] = time_remaining
        
        # Add normalized team names
        df['home_team_norm'] = df['home_team'].apply(self.normalize_team_name)
        df['away_team_norm'] = df['away_team'].apply(self.normalize_team_name)
        
        return df
    
    def _determine_game_status(self, game_row):
        """Determine game status based on quarter and score data."""
        # Default values
        status = 'UNKNOWN'
        quarter = int(game_row.get('current_quarter', 0) or 0)
        time_remaining = 48  # Full game
        
        # Check if game is marked as finished
        if game_row.get('is_finished', False):
            return 'FINAL', quarter, 0
        
        # Game has started (at least one quarter with points)
        if quarter > 0:
            # All quarters have data - likely finished
            if quarter == 4:
                home_q4 = float(game_row.get('home_q4', 0) or 0)
                away_q4 = float(game_row.get('away_q4', 0) or 0)
                if home_q4 > 0 and away_q4 > 0:
                    return 'FINAL', 4, 0
            
            # In progress
            status = 'LIVE'
            time_remaining = (4 - quarter) * 12
            if quarter == 4:
                # Assume we're midway through the 4th
                time_remaining = 6
        else:
            # No quarters with data - scheduled or delayed
            # If we have a game date, check if it's in the past
            if 'game_date' in game_row:
                game_date = pd.to_datetime(game_row['game_date'])
                now = datetime.now(tz=game_date.tzinfo) if game_date.tzinfo else datetime.now()
                
                if game_date < now:
                    # Game should have started but no data - likely delayed
                    status = 'DELAYED'
                else:
                    # Game is in the future
                    status = 'SCHEDULED'
                    # Add time until game starts to the 48 minutes
                    mins_to_start = (game_date - now).total_seconds() / 60
                    time_remaining = 48 + max(0, mins_to_start)
            else:
                # No date info - assume scheduled
                status = 'SCHEDULED'
        
        return status, quarter, time_remaining
    
    def match_teams_to_schedule(self, live_df, schedule_df):
        """Match live games to the official schedule using team names."""
        if live_df.empty or schedule_df.empty:
            return live_df
            
        # Copy to avoid modifying the original
        live_df = live_df.copy()
        
        # Make sure both dataframes have normalized team names
        live_df = self.validate_game_data(live_df)
        schedule_df = self.prepare_schedule_data(schedule_df)
        
        # Try to match by exact normalized team names
        for live_idx, live_game in live_df.iterrows():
            # Skip already matched games
            if live_game.get('matched_to_schedule', False):
                continue
                
            home_norm = live_game['home_team_norm']
            away_norm = live_game['away_team_norm']
            
            # Try both home/away combinations
            home_away_match = schedule_df[
                (schedule_df['home_team_norm'] == home_norm) &
                (schedule_df['away_team_norm'] == away_norm)
            ]
            
            away_home_match = schedule_df[
                (schedule_df['home_team_norm'] == away_norm) &
                (schedule_df['away_team_norm'] == home_norm)
            ]
            
            if not home_away_match.empty:
                # Found exact match
                match = home_away_match.iloc[0]
                live_df.at[live_idx, 'matched_to_schedule'] = True
                # Copy over the official game_id if needed
                if 'game_id' in match and match['game_id'] != live_game['game_id']:
                    live_df.at[live_idx, 'original_game_id'] = live_game['game_id']
                    live_df.at[live_idx, 'game_id'] = match['game_id']
                    
            elif not away_home_match.empty:
                # Found match with teams reversed - this indicates a data issue
                logger.warning(f"Found teams in reverse order for game {live_game['game_id']}")
                match = away_home_match.iloc[0]
                live_df.at[live_idx, 'matched_to_schedule'] = True
                live_df.at[live_idx, 'teams_reversed'] = True
                # Copy over the official game_id if needed
                if 'game_id' in match and match['game_id'] != live_game['game_id']:
                    live_df.at[live_idx, 'original_game_id'] = live_game['game_id']
                    live_df.at[live_idx, 'game_id'] = match['game_id']
        
        # For unmatched games, try looser matching
        if not live_df[~live_df['matched_to_schedule']].empty:
            for live_idx, live_game in live_df[~live_df['matched_to_schedule']].iterrows():
                best_match = None
                best_score = 0
                is_reversed = False
                
                for _, sched_game in schedule_df.iterrows():
                    # Try both directions
                    direct_match = (live_game['home_team_norm'] in sched_game['home_team_norm'] or 
                                   sched_game['home_team_norm'] in live_game['home_team_norm']) and \
                                  (live_game['away_team_norm'] in sched_game['away_team_norm'] or 
                                   sched_game['away_team_norm'] in live_game['away_team_norm'])
                                   
                    reverse_match = (live_game['home_team_norm'] in sched_game['away_team_norm'] or 
                                    sched_game['away_team_norm'] in live_game['home_team_norm']) and \
                                   (live_game['away_team_norm'] in sched_game['home_team_norm'] or 
                                    sched_game['home_team_norm'] in live_game['away_team_norm'])
                    
                    # Simple scoring - each match is worth 1 point
                    score = direct_match + reverse_match * 0.9  # Slightly prefer direct matches
                    
                    if score > best_score:
                        best_score = score
                        best_match = sched_game
                        is_reversed = reverse_match > direct_match
                
                if best_match is not None and best_score > 0:
                    live_df.at[live_idx, 'matched_to_schedule'] = True
                    live_df.at[live_idx, 'match_confidence'] = best_score
                    live_df.at[live_idx, 'teams_reversed'] = is_reversed
                    
                    # Copy over the official game_id if needed
                    if 'game_id' in best_match and best_match['game_id'] != live_game['game_id']:
                        live_df.at[live_idx, 'original_game_id'] = live_game['game_id']
                        live_df.at[live_idx, 'game_id'] = best_match['game_id']
        
        return live_df
    
    def convert_scheduled_to_live_format(self, schedule_df):
        """Convert scheduled games to a live data format."""
        if schedule_df.empty:
            return pd.DataFrame()
            
        live_format = []
        
        for _, game in schedule_df.iterrows():
            # Create a basic live game structure
            live_game = {
                'game_id': game['game_id'],
                'home_team': game['home_team'],
                'away_team': game['away_team'],
                'game_date': game['game_date'],
                'matched_to_schedule': True,
                'current_quarter': 0,  # Pre-game
                'home_q1': 0, 'home_q2': 0, 'home_q3': 0, 'home_q4': 0,
                'away_q1': 0, 'away_q2': 0, 'away_q3': 0, 'away_q4': 0,
                'home_score': 0,
                'away_score': 0,
                'game_status': 'SCHEDULED'}

            # Calculate time until game
            try:
                game_dt = pd.to_datetime(game['game_date'])
                now = datetime.now(tz=game_dt.tzinfo) if game_dt.tzinfo else datetime.now()
                time_until_game = max(0, (game_dt - now).total_seconds() / 60)
                live_game['time_remaining_mins'] = 48 + time_until_game
            except:
                live_game['time_remaining_mins'] = 48
                
            live_format.append(live_game)
        
        return pd.DataFrame(live_format)
    
    def calculate_enhanced_features(self, games_df, team_avgs, data_fetcher):
        """Calculate enhanced features for prediction."""
        if games_df.empty:
            return pd.DataFrame()
            
        # Make a copy to avoid modifying the original
        enhanced_df = games_df.copy()
        
        # Add rolling averages
        enhanced_df['rolling_home_score'] = enhanced_df['home_team'].apply(
            lambda t: team_avgs.get(t, 105.0))
        enhanced_df['rolling_away_score'] = enhanced_df['away_team'].apply(
            lambda t: team_avgs.get(t, 105.0))
            
        # Calculate rest features
        for idx, game in enhanced_df.iterrows():
            try:
                home_team = game['home_team']
                away_team = game['away_team']
                game_date = pd.to_datetime(game['game_date'])

                # Get rest data for both teams
                home_rest = data_fetcher.get_rest_data(home_team, game_date)
                away_rest = data_fetcher.get_rest_data(away_team, game_date)

                # Update DataFrame
                enhanced_df.at[idx, 'rest_days_home'] = home_rest['rest_days']
                enhanced_df.at[idx, 'rest_days_away'] = away_rest['rest_days']
                enhanced_df.at[idx, 'is_back_to_back_home'] = int(home_rest['is_back_to_back'])
                enhanced_df.at[idx, 'is_back_to_back_away'] = int(away_rest['is_back_to_back'])
                enhanced_df.at[idx, 'rest_advantage'] = home_rest['rest_days'] - away_rest['rest_days']

                # Get previous matchup difference
                enhanced_df.at[idx, 'prev_matchup_diff'] = data_fetcher.get_previous_matchup_diff(home_team, away_team)
            except Exception as e:
                logger.warning(f"Error calculating rest features for game {game.get('game_id')}: {e}")
                # Set default values
                enhanced_df.at[idx, 'rest_days_home'] = 2
                enhanced_df.at[idx, 'rest_days_away'] = 2
                enhanced_df.at[idx, 'is_back_to_back_home'] = 0
                enhanced_df.at[idx, 'is_back_to_back_away'] = 0
                enhanced_df.at[idx, 'rest_advantage'] = 0
                enhanced_df.at[idx, 'prev_matchup_diff'] = 0
        
        # Calculate momentum features
        for idx, game in enhanced_df.iterrows():
            try:
                current_quarter = int(game.get('current_quarter', 0) or 0)

                # Initialize momentum features
                enhanced_df.at[idx, 'q1_to_q2_momentum'] = 0
                enhanced_df.at[idx, 'q2_to_q3_momentum'] = 0
                enhanced_df.at[idx, 'q3_to_q4_momentum'] = 0
                enhanced_df.at[idx, 'cumulative_momentum'] = 0

                # Calculate quarter-to-quarter momentum
                if current_quarter >= 2:
                    # Q1 to Q2
                    home_q1 = float(game.get('home_q1', 0) or 0)
                    home_q2 = float(game.get('home_q2', 0) or 0)
                    away_q1 = float(game.get('away_q1', 0) or 0)
                    away_q2 = float(game.get('away_q2', 0) or 0)

                    q1_to_q2 = (home_q2 - home_q1) - (away_q2 - away_q1)
                    # Cap to prevent extreme values
                    q1_to_q2 = max(min(q1_to_q2, 15), -15)
                    enhanced_df.at[idx, 'q1_to_q2_momentum'] = q1_to_q2

                if current_quarter >= 3:
                    # Q2 to Q3
                    home_q2 = float(game.get('home_q2', 0) or 0)
                    home_q3 = float(game.get('home_q3', 0) or 0)
                    away_q2 = float(game.get('away_q2', 0) or 0)
                    away_q3 = float(game.get('away_q3', 0) or 0)

                    q2_to_q3 = (home_q3 - home_q2) - (away_q3 - away_q2)
                    q2_to_q3 = max(min(q2_to_q3, 15), -15)
                    enhanced_df.at[idx, 'q2_to_q3_momentum'] = q2_to_q3

                if current_quarter >= 4:
                    # Q3 to Q4
                    home_q3 = float(game.get('home_q3', 0) or 0)
                    home_q4 = float(game.get('home_q4', 0) or 0)
                    away_q3 = float(game.get('away_q3', 0) or 0)
                    away_q4 = float(game.get('away_q4', 0) or 0)

                    q3_to_q4 = (home_q4 - home_q3) - (away_q4 - away_q3)
                    q3_to_q4 = max(min(q3_to_q4, 15), -15)
                    enhanced_df.at[idx, 'q3_to_q4_momentum'] = q3_to_q4

                # Calculate weighted momentum
                weights = [0.2, 0.3, 0.5]  # Weights for each momentum period

                if current_quarter == 2:
                    momentum = enhanced_df.at[idx, 'q1_to_q2_momentum']
                elif current_quarter == 3:
                    momentum = (
                        enhanced_df.at[idx, 'q1_to_q2_momentum'] * weights[0] +
                        enhanced_df.at[idx, 'q2_to_q3_momentum'] * weights[1]
                    ) / (weights[0] + weights[1])
                elif current_quarter >= 4:
                    momentum = (
                        enhanced_df.at[idx, 'q1_to_q2_momentum'] * weights[0] +
                        enhanced_df.at[idx, 'q2_to_q3_momentum'] * weights[1] +
                        enhanced_df.at[idx, 'q3_to_q4_momentum'] * weights[2]
                    ) / sum(weights)
                else:
                    momentum = 0

                # Normalize to [-1, 1]
                enhanced_df.at[idx, 'cumulative_momentum'] = max(min(momentum / 15.0, 1.0), -1.0)

            except Exception as e:
                logger.warning(f"Error calculating momentum for game {game.get('game_id')}: {e}")
                # Set defaults
                enhanced_df.at[idx, 'q1_to_q2_momentum'] = 0
                enhanced_df.at[idx, 'q2_to_q3_momentum'] = 0
                enhanced_df.at[idx, 'q3_to_q4_momentum'] = 0
                enhanced_df.at[idx, 'cumulative_momentum'] = 0
                
        return enhanced_df


class ModelManager:
    """
    Handles model loading and feature extraction.
    """
    
    def __init__(self):
        """Initialize the model manager."""
        self.model = None
        
    def load_model(self, model_path=None):
        """Load a prediction model from disk."""
        try:
            if model_path is None:
                # Try to use config.MODEL_PATH if available
                try:
                    import config
                    model_path = config.MODEL_PATH
                except (ImportError, AttributeError):
                    model_path = os.path.join(os.getcwd(), 'models', 'score_prediction_model.pkl')
            
            if os.path.exists(model_path):
                self.model = joblib.load(model_path)
                logger.info(f"Model loaded from {model_path}")
                return True
            else:
                logger.error(f"Model file not found at {model_path}")
                return False
        except Exception as e:
            logger.error(f"Error loading model: {e}")
            traceback.print_exc()
            return False
    
    def get_feature_list(self):
        """Determine which features to use based on model type."""
        if self.model is None:
            logger.error("No model loaded")
            return []
            
        # If model has feature_importances_, it's a tree-based model
        if hasattr(self.model, 'feature_importances_'):
            feature_count = len(self.model.feature_importances_)
            if feature_count > 8:
                # Enhanced model
                return [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4',
                    'score_ratio', 'prev_matchup_diff',
                    'rest_days_home', 'rest_days_away', 'rest_advantage',
                    'is_back_to_back_home', 'is_back_to_back_away',
                    'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
                ]
            else:
                # Original model
                return [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4',
                    'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
                ]
        # If model has coefficients, it's a linear model
        elif hasattr(self.model, 'coef_'):
            feature_count = len(self.model.coef_)
            if feature_count > 8:
                # Enhanced model
                return [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4',
                    'score_ratio', 'prev_matchup_diff',
                    'rest_days_home', 'rest_days_away', 'rest_advantage',
                    'is_back_to_back_home', 'is_back_to_back_away',
                    'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
                ]
            else:
                # Original model
                return [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4',
                    'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
                ]
        # Default to comprehensive feature list
        return [
            'home_q1', 'home_q2', 'home_q3', 'home_q4',
            'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff',
            'rest_days_home', 'rest_days_away', 'rest_advantage',
            'is_back_to_back_home', 'is_back_to_back_away',
            'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
        ]


class PredictionEngine:
    """
    Handles the actual prediction process.
    """
    
    def __init__(self, model_manager):
        """Initialize with a model manager."""
        self.model_manager = model_manager
        self.prediction_history = {}
        self.season_adjustment = 1.0
    
    def run_predictions(self, games_df):
        """
        Run predictions for all games in the DataFrame.
        
        Args:
            games_df: DataFrame with game data and enhanced features
            
        Returns:
            DataFrame with predictions and additional metrics
        """
        if games_df.empty:
            logger.info("No games available for prediction")
            return pd.DataFrame()
        
        # Ensure model is loaded
        if self.model_manager.model is None:
            logger.error("No model available for prediction")
            return games_df
        
        # Identify features to use
        feature_list = self.model_manager.get_feature_list()
        
        # Ensure all features exist
        enhanced_df = games_df.copy()
        for feature in feature_list:
            if feature not in enhanced_df.columns:
                logger.info(f"Adding missing feature: {feature}")
                enhanced_df[feature] = 0
                
            # Convert to numeric
            enhanced_df[feature] = pd.to_numeric(enhanced_df[feature], errors='coerce').fillna(0)
        
        # Extract feature matrix
        X_pred = enhanced_df[feature_list]
        
        # Generate predictions
        try:
            predictions = self.model_manager.model.predict(X_pred)
            enhanced_df['predicted_home_score'] = predictions
            
            # Calculate away score predictions and other metrics
            for idx, row in enhanced_df.iterrows():
                # Adjust for home court advantage and matchup history
                home_advantage = 3.5
                matchup_diff = row.get('prev_matchup_diff', 0)
                enhanced_df.at[idx, 'predicted_away_score'] = row['predicted_home_score'] - home_advantage - matchup_diff
                
                # Ensure predictions are at least current scores
                current_home = float(row.get('home_score', 0) or 0)
                current_away = float(row.get('away_score', 0) or 0)
                
                enhanced_df.at[idx, 'predicted_home_score'] = max(enhanced_df.at[idx, 'predicted_home_score'], current_home)
                enhanced_df.at[idx, 'predicted_away_score'] = max(enhanced_df.at[idx, 'predicted_away_score'], current_away)
                
                # Calculate score differential
                score_diff = enhanced_df.at[idx, 'predicted_home_score'] - enhanced_df.at[idx, 'predicted_away_score']
                
                # Calculate win probability using logistic function
                # Higher differential = higher probability, with steeper curve in later quarters
                quarter = int(row.get('current_quarter', 0) or 0)
                k_factor = 0.05 + (quarter * 0.025)  # Steeper curve in later quarters
                win_prob = 1.0 / (1.0 + np.exp(-k_factor * score_diff))
                enhanced_df.at[idx, 'win_probability'] = win_prob
                
                # Calculate remaining score
                if quarter > 0:
                    enhanced_df.at[idx, 'remaining_home_score'] = enhanced_df.at[idx, 'predicted_home_score'] - current_home
                    enhanced_df.at[idx, 'remaining_away_score'] = enhanced_df.at[idx, 'predicted_away_score'] - current_away
                
                # Calculate prediction confidence (increases by quarter)
                confidence_by_quarter = {0: 0.5, 1: 0.6, 2: 0.7, 3: 0.85, 4: 0.95}
                enhanced_df.at[idx, 'prediction_confidence'] = confidence_by_quarter.get(quarter, 0.5)
                
                # Calculate total projected score
                enhanced_df.at[idx, 'total_projected_score'] = enhanced_df.at[idx, 'predicted_home_score'] + enhanced_df.at[idx, 'predicted_away_score']
                
                # If we have actual finals, calculate errors
                if 'actual_home_final' in row and 'actual_away_final' in row:
                    enhanced_df.at[idx, 'home_score_error'] = enhanced_df.at[idx, 'predicted_home_score'] - row['actual_home_final']
                    enhanced_df.at[idx, 'away_score_error'] = enhanced_df.at[idx, 'predicted_away_score'] - row['actual_away_final']
                    enhanced_df.at[idx, 'home_error_pct'] = abs(enhanced_df.at[idx, 'home_score_error'] / row['actual_home_final']) * 100
                    enhanced_df.at[idx, 'away_error_pct'] = abs(enhanced_df.at[idx, 'away_score_error'] / row['actual_away_final']) * 100
            
        except Exception as e:
            logger.error(f"Error making predictions: {e}")
            traceback.print_exc()
        
        # Generate betting recommendations
        try:
            from models.dynamic_recommendation import generate_recommendations
            
            for idx, row in enhanced_df.iterrows():
                model_outputs = {
                    "win_probability": row.get('win_probability', 0.5),
                    "momentum_shift": row.get('cumulative_momentum', 0),
                    "projected_margin": row.get('predicted_home_score', 0) - row.get('predicted_away_score', 0),
                    "total_projected_score": row.get('predicted_home_score', 0) + row.get('predicted_away_score', 0),
                    "quarter": row.get('current_quarter', 0),
                    "time_remaining": row.get('time_remaining_mins', None)
                }
                
                try:
                    recommendations = generate_recommendations(model_outputs)
                    
                    # Store top recommendations
                    for rec_key, rec_value in recommendations.items():
                        enhanced_df.at[idx, f'rec_{rec_key}'] = rec_value
                except Exception as rec_error:
                    logger.warning(f"Error generating recommendations for game {row.get('game_id')}: {rec_error}")
        except ImportError:
            logger.warning("Recommendation module not available, skipping recommendations.")
        except Exception as e:
            logger.error(f"Error in recommendation generation: {e}")
        
        return enhanced_df
    
    def update_prediction_history(self, predictions_df):
        """Update the prediction history with new predictions."""
        if predictions_df.empty:
            return
            
        current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        for _, game in predictions_df.iterrows():
            game_key = f"{game['home_team']} vs {game['away_team']}"
            
            if game_key not in self.prediction_history:
                self.prediction_history[game_key] = []
            
            # Add current prediction
            self.prediction_history[game_key].append({
                'timestamp': current_time,
                'current_quarter': game['current_quarter'],
                'current_home_score': game.get('home_score', 0),
                'current_away_score': game.get('away_score', 0),
                'predicted_home_score': game['predicted_home_score'],
                'predicted_away_score': game['predicted_away_score'],
                'win_probability': game.get('win_probability', 0.5)
            })


class Visualizer:
    """
    Handles display and visualization of predictions.
    """
    
    def display_predictions(self, predictions_df, show_recommendations=True):
        """Display current predictions in a clean format."""
        if predictions_df.empty:
            print("\nNo predictions to display.")
            return
            
        print("\n===== PREDICTION RESULTS =====")
        
        for _, game in predictions_df.iterrows():
            current_q = game.get('current_quarter', 0)
            q_str = f"Quarter {current_q}" if current_q > 0 else "Pre-game"
            
            print(f"\n{game['home_team']} vs {game['away_team']} - {q_str}")
            
            if current_q > 0:
                print(f"Current: {game['home_team']} {game.get('home_score', 0):.0f} - " 
                      f"{game['away_team']} {game.get('away_score', 0):.0f}")
            
            print(f"Predicted: {game['home_team']} {game['predicted_home_score']:.1f} - " 
                  f"{game['away_team']} {game['predicted_away_score']:.1f}")
            
            if 'win_probability' in game:
                win_pct = game['win_probability'] * 100
                print(f"Win Probability: {win_pct:.1f}%")
            
            # If we have error metrics available, show them
            if 'home_score_error' in game:
                print(f"Validation - Prediction Error: {game['home_score_error']:.1f} points")
        
        # Display recommendations if requested
        if show_recommendations:
            print("\n===== GAME RECOMMENDATIONS =====")
            
            for _, game in predictions_df.iterrows():
                game_name = f"{game['home_team']} vs {game['away_team']}"
                print(f"\n{game_name}:")
                
                # Get all recommendation fields
                recs = {}
                for col in game.index:
                    if col.startswith('rec_'):
                        rec_key = col[4:]  # Remove 'rec_' prefix
                        if pd.notna(game[col]) and game[col]:
                            recs[rec_key] = game[col]
                
                if recs:
                    for rec_type, recommendation in recs.items():
                        print(f"  • {rec_type}: {recommendation}")
                else:
                    # Simple default recommendations if none available
                    score_diff = game.get('predicted_home_score', 0) - game.get('predicted_away_score', 0)
                    win_prob = game.get('win_probability', 0.5)
                    
                    if win_prob > 0.7:
                        print(f"  • betting_tip: Home team strong favorite.")
                    elif win_prob < 0.3:
                        print(f"  • betting_tip: Home team significant underdog.")
                    else:
                        print(f"  • betting_tip: Game appears competitive.")
                        
                    print(f"  • margin: Projected margin: {score_diff:.1f} points")
    
    def plot_prediction_history(self, prediction_history):
        """Plot the prediction history for all games."""
        if not prediction_history:
            print("No prediction history available to plot.")
            return
            
        # Check if we have at least one game with multiple predictions
        has_history = False
        for game_key, history in prediction_history.items():
            if len(history) > 1:
                has_history = True
                break
                
        if not has_history:
            print("Not enough history to plot trends.")
            return
            
        plt.figure(figsize=(12, 8))
        
        # For each game, plot prediction evolution
        for game_key, history in prediction_history.items():
            if len(history) > 1:
                times = range(len(history))
                home_preds = [p['predicted_home_score'] for p in history]
                away_preds = [p['predicted_away_score'] for p in history]
                
                # Extract team names
                teams = game_key.split(' vs ')
                home_team = teams[0]
                away_team = teams[1] if len(teams) > 1 else "Away"
                
                plt.plot(times, home_preds, 'o-', label=f"{home_team} (Home)")
                plt.plot(times, away_preds, 's-', label=f"{away_team} (Away)")
        
        plt.xlabel('Update Index')
        plt.ylabel('Predicted Final Score')
        plt.title('Prediction Evolution Over Time')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # Add quarters as background
        quarters = None
        for game_key, history in prediction_history.items():
            if len(history) > 1:
                quarters = [h.get('current_quarter', 0) for h in history]
                break
                
        if quarters:
            last_q = -1
            colors = ['#f8f9fa', '#e9ecef', '#dee2e6', '#ced4da']
            
            for i, q in enumerate(quarters):
                if q != last_q:
                    plt.axvline(x=i, color='gray', linestyle='--', alpha=0.5)
                    last_q = q
        
        plt.tight_layout()
        plt.show()
        
        # Also plot win probability evolution if available
        has_win_prob = False
        for game_key, history in prediction_history.items():
            if len(history) > 1 and 'win_probability' in history[0]:
                has_win_prob = True
                break
                
        if has_win_prob:
            plt.figure(figsize=(12, 6))
            
            for game_key, history in prediction_history.items():
                if len(history) > 1 and 'win_probability' in history[0]:
                    times = range(len(history))
                    win_probs = [p.get('win_probability', 0.5) * 100 for p in history]
                    
                    # Extract team names
                    teams = game_key.split(' vs ')
                    home_team = teams[0]
                    away_team = teams[1] if len(teams) > 1 else "Away"
                    
                    plt.plot(times, win_probs, 'o-', label=f"{home_team} Win Probability")
            
            plt.xlabel('Update Index')
            plt.ylabel('Win Probability (%)')
            plt.title('Win Probability Evolution Over Time')
            plt.axhline(y=50, color='gray', linestyle='--', label='Even (50%)')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.ylim(0, 100)
            plt.tight_layout()
            plt.show()
    
    def validate_predictions(self, predictions_df):
        """Validate predictions against actual results if available."""
        has_actuals = (
            not predictions_df.empty and 
            'actual_home_final' in predictions_df.columns and
            'actual_away_final' in predictions_df.columns
        )
        
        if not has_actuals:
            return
            
        print("\n===== VALIDATION RESULTS =====")
        
        # Calculate overall metrics
        home_errors = []
        away_errors = []
        
        for _, game in predictions_df.iterrows():
            if pd.isna(game['actual_home_final']) or pd.isna(game['actual_away_final']):
                continue
                
            home_error = abs(game['predicted_home_score'] - game['actual_home_final'])
            away_error = abs(game['predicted_away_score'] - game['actual_away_final'])
            
            home_errors.append(home_error)
            away_errors.append(away_error)
            
            # Print individual game results
            print(f"\n{game['home_team']} vs {game['away_team']} (Q{game['current_quarter']}):")
            print(f"  Predicted: {game['home_team']} {game['predicted_home_score']:.1f} - {game['away_team']} {game['predicted_away_score']:.1f}")
            print(f"  Actual:    {game['home_team']} {game['actual_home_final']:.1f} - {game['away_team']} {game['actual_away_final']:.1f}")
            print(f"  Error:     {home_error:.1f} pts ({home_error/game['actual_home_final']*100:.1f}%) / {away_error:.1f} pts ({away_error/game['actual_away_final']*100:.1f}%)")
        
        if home_errors and away_errors:
            # Calculate summary metrics
            avg_home_error = sum(home_errors) / len(home_errors)
            avg_away_error = sum(away_errors) / len(away_errors)
            avg_error = (avg_home_error + avg_away_error) / 2
            
            print("\nSummary:")
            print(f"  Average Home Error: {avg_home_error:.2f} points")
            print(f"  Average Away Error: {avg_away_error:.2f} points")
            print(f"  Overall Error:      {avg_error:.2f} points")


class DataStorage:
    """
    Handles saving and loading of results.
    """
    
    def __init__(self, output_dir=None):
        """Initialize with an output directory."""
        if output_dir is None:
            self.output_dir = Path("monitoring_output")
        else:
            self.output_dir = Path(output_dir)
            
        self.output_dir.mkdir(exist_ok=True)
    
    def save_results(self, predictions_df, prediction_history):
        """Save prediction results to disk."""
        if predictions_df.empty:
            return
            
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        
        # Save predictions to CSV
        predictions_file = self.output_dir / f"predictions_{timestamp}.csv"
        predictions_df.to_csv(predictions_file, index=False)
        logger.info(f"Saved predictions to {predictions_file}")
        
        # Save prediction history to JSON
        history_file = self.output_dir / f"prediction_history_{timestamp}.json"
        with open(history_file, 'w') as f:
            json.dump(prediction_history, f, indent=2)
        logger.info(f"Saved prediction history to {history_file}")


class GameMonitor:
    """
    Orchestrates the entire game monitoring pipeline.
    """
    
    def __init__(self, supabase_client=None, model_path=None):
        """Initialize the monitoring pipeline."""
        # Initialize components
        self.data_fetcher = DataFetcher(supabase_client)
        self.data_processor = DataProcessor()

In [ ]:
# Cell 17: Improved Test Prediction on Active Live Games

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import pytz
import traceback
import os
import json
import time
from typing import Dict, List, Optional, Tuple, Union

# Add the missing import for supabase
from caching.supabase_client import supabase

# ---- Timezone Utility Functions ----

def get_timezone(tz_name='America/Los_Angeles'):
    """Get a timezone object from its name."""
    return pytz.timezone(tz_name)

def get_current_time(tz_name='America/Los_Angeles'):
    """Get current time in specified timezone."""
    tz = get_timezone(tz_name)
    return datetime.now(tz)

def get_date_str(dt=None, tz_name='America/Los_Angeles', format='%Y-%m-%d'):
    """Get formatted date string in specified timezone."""
    if dt is None:
        dt = get_current_time(tz_name)
    return dt.strftime(format)

def convert_to_timezone(dt, target_tz_name='America/Los_Angeles', source_tz_name=None):
    """Convert datetime to target timezone.
    
    Args:
        dt: Datetime to convert (string or datetime object)
        target_tz_name: Target timezone name
        source_tz_name: Source timezone name if dt is naive
        
    Returns:
        Timezone-aware datetime in target timezone
    """
    target_tz = get_timezone(target_tz_name)
    
    # Handle string dates
    if isinstance(dt, str):
        try:
            # If time component is missing, assume start of day
            if len(dt) <= 10:  # YYYY-MM-DD format
                dt_obj = datetime.strptime(dt, '%Y-%m-%d')
                # Add typical NBA game start time (7:30 PM)
                dt_obj = dt_obj.replace(hour=19, minute=30)
                # Localize to target timezone
                return target_tz.localize(dt_obj)
            else:
                # Parse with time component
                dt_obj = pd.to_datetime(dt)
                # If timezone naive, localize to provided timezone or UTC
                if dt_obj.tzinfo is None:
                    source_tz = get_timezone(source_tz_name) if source_tz_name else pytz.UTC
                    dt_obj = source_tz.localize(dt_obj)
                # Convert to target timezone
                return dt_obj.astimezone(target_tz)
        except Exception as e:
            print(f"Error parsing date '{dt}': {e}")
            return get_current_time(target_tz_name)
    
    # Handle datetime objects
    if isinstance(dt, datetime):
        # If timezone naive, localize to provided timezone or UTC
        if dt.tzinfo is None:
            source_tz = get_timezone(source_tz_name) if source_tz_name else pytz.UTC
            dt = source_tz.localize(dt)
        # Convert to target timezone
        return dt.astimezone(target_tz)
    
    # Default fallback
    print(f"Unsupported date format: {dt}")
    return get_current_time(target_tz_name)

# ---- API Utility Functions ----

def safe_api_call(func, *args, max_retries=3, retry_delay=2, **kwargs):
    """Execute an API call with retry logic and error handling.
    
    Args:
        func: Function to call
        max_retries: Maximum number of retries (default: 3)
        retry_delay: Delay between retries in seconds (default: 2)
        *args, **kwargs: Arguments to pass to func
        
    Returns:
        API response or None on failure
    """
    retries = 0
    last_error = None
    
    while retries < max_retries:
        try:
            return func(*args, **kwargs)
        except Exception as e:
            retries += 1
            last_error = e
            
            # Check for rate limiting and adjust retry delay
            if "rate limit" in str(e).lower() or "429" in str(e):
                retry_delay = min(retry_delay * 2, 30)  # Exponential backoff up to 30 seconds
                print(f"Rate limit hit. Retrying in {retry_delay} seconds...")
            else:
                print(f"API error: {e}. Retry {retries}/{max_retries} in {retry_delay} seconds...")
                
            time.sleep(retry_delay)
    
    # Log the error after all retries fail
    print(f"API call failed after {max_retries} retries. Last error: {last_error}")
    traceback.print_exc()
    return None

# ---- Game Status Detection Function ----

def determine_game_status(game_data):
    """
    Determine game status based on quarter and score data.
    
    Args:
        game_data: Dictionary with game information
        
    Returns:
        Tuple of (status_str, current_quarter, time_remaining_mins)
    """
    # Default values
    status = 'UNKNOWN'
    current_quarter = int(game_data.get('current_quarter', 0) or 0)
    time_remaining = 48  # Full game time
    
    # Check if game is marked as finished
    if game_data.get('is_finished', False):
        return 'FINAL', current_quarter or 4, 0
    
    # Calculate max quarter with data
    max_quarter = 0
    for q in range(1, 5):
        home_q_val = float(game_data.get(f'home_q{q}', 0) or 0)
        away_q_val = float(game_data.get(f'away_q{q}', 0) or 0)
        if home_q_val > 0 or away_q_val > 0:
            max_quarter = q
    
    # Use the higher of calculated or provided quarter
    current_quarter = max(current_quarter, max_quarter)
    
    # Game has started (at least one quarter with points)
    if current_quarter > 0:
        # All quarters have significant data - likely finished
        if current_quarter == 4:
            home_q4 = float(game_data.get('home_q4', 0) or 0)
            away_q4 = float(game_data.get('away_q4', 0) or 0)
            if home_q4 > 0 and away_q4 > 0:
                # Double-check for overtime
                home_ot = float(game_data.get('home_ot', 0) or 0)
                away_ot = float(game_data.get('away_ot', 0) or 0)
                if home_ot > 0 or away_ot > 0:
                    return 'LIVE', 4, 5  # Overtime
                return 'FINAL', 4, 0
        
        # In progress
        status = 'LIVE'
        
        # Calculate approximate time remaining
        if current_quarter == 4:
            time_remaining = 6  # Assume mid-4th quarter
        else:
            time_remaining = (4 - current_quarter) * 12 + 6  # Add half of current quarter
            
        # If specific time remaining is provided, use it
        if 'time_remaining_in_quarter' in game_data:
            try:
                quarter_time = float(game_data['time_remaining_in_quarter'])
                time_remaining = (4 - current_quarter) * 12 + quarter_time
            except (ValueError, TypeError):
                pass
    else:
        # No quarters with data - scheduled or delayed
        if 'game_date' in game_data:
            try:
                # Convert game date to datetime in Pacific time
                game_date = convert_to_timezone(game_data['game_date'])
                now = get_current_time()
                
                # If game date is today
                if game_date.date() == now.date():
                    if game_date < now:
                        # Game should have started but no data - likely delayed
                        status = 'DELAYED'
                    else:
                        # Game is later today
                        status = 'SCHEDULED'
                        # Add time until game starts
                        mins_to_start = (game_date - now).total_seconds() / 60
                        time_remaining = 48 + max(0, mins_to_start)
                else:
                    # Different day
                    if game_date.date() < now.date():
                        # Past date - might be postponed or data issue
                        status = 'UNKNOWN'
                    else:
                        # Future date
                        status = 'SCHEDULED'
            except Exception as e:
                print(f"Error processing game date: {e}")
        else:
            # No date info - assume scheduled
            status = 'SCHEDULED'
    
    return status, current_quarter, time_remaining

# ---- NBA Live Game Functions ----

def fetch_live_games_by_date(date_str=None):
    """
    Fetch live games for a specific date from the database.
    
    Args:
        date_str: Date string in YYYY-MM-DD format (defaults to today in PT)
        
    Returns:
        DataFrame with live games
    """
    # Use today's date in Pacific Time if none provided
    if date_str is None:
        date_str = get_date_str()
    
    print(f"Fetching live games for {date_str} (Pacific Time)")
    
    try:
        # Get timestamps for the day in Pacific Time
        pacific_tz = get_timezone()
        start_dt = datetime.strptime(date_str, '%Y-%m-%d')
        start_dt = pacific_tz.localize(start_dt)
        end_dt = start_dt + timedelta(days=1) - timedelta(seconds=1)
        
        # Convert to UTC for database query
        start_utc = start_dt.astimezone(pytz.UTC).isoformat()
        end_utc = end_dt.astimezone(pytz.UTC).isoformat()
        
        # Query using date range instead of exact match
        response = safe_api_call(
            supabase.table("nba_live_game_stats").select("*")
                   .gte("game_date", start_utc)
                   .lte("game_date", end_utc)
                   .execute
        )
        
        if not response or not response.data:
            print(f"No live games found for {date_str}")
            return pd.DataFrame()
        
        # Create DataFrame
        live_df = pd.DataFrame(response.data)
        
        # Process and validate game data
        live_df = validate_game_data(live_df)
        
        # Match with schedule if possible
        live_df = match_with_schedule(live_df)
        
        # Report game statuses
        if not live_df.empty:
            status_counts = live_df['game_status'].value_counts()
            print("Game status summary:")
            for status, count in status_counts.items():
                print(f"  - {status}: {count} games")
        
        return live_df
        
    except Exception as e:
        print(f"Error fetching live games by date: {e}")
        traceback.print_exc()
        return pd.DataFrame()

def validate_game_data(df):
    """
    Validate and clean game data.
    
    Args:
        df: DataFrame with raw game data
        
    Returns:
        DataFrame with validated and cleaned game data
    """
    if df.empty:
        return df
        
    # Make a copy to avoid modifying the original
    df = df.copy()
    
    # Convert date fields to datetime
    if 'game_date' in df.columns:
        df['game_date'] = pd.to_datetime(df['game_date'], errors='coerce')
    
    # Ensure quarter fields are numeric
    for q in range(1, 5):
        home_q = f'home_q{q}'
        away_q = f'away_q{q}'
        
        if home_q in df.columns:
            df[home_q] = pd.to_numeric(df[home_q], errors='coerce').fillna(0)
        else:
            df[home_q] = 0
            
        if away_q in df.columns:
            df[away_q] = pd.to_numeric(df[away_q], errors='coerce').fillna(0)
        else:
            df[away_q] = 0
    
    # Ensure team names are strings
    if 'home_team' in df.columns:
        df['home_team'] = df['home_team'].astype(str)
    if 'away_team' in df.columns:
        df['away_team'] = df['away_team'].astype(str)
    
    # Calculate current score as sum of quarters
    if 'home_score' not in df.columns:
        df['home_score'] = 0
    if 'away_score' not in df.columns:
        df['away_score'] = 0
    
    # Set up quarter-related fields and current scores    
    for idx, row in df.iterrows():
        # Calculate current quarter based on non-zero quarter scores
        max_quarter = 0
        for q in range(1, 5):
            home_q_val = float(row.get(f'home_q{q}', 0) or 0)
            away_q_val = float(row.get(f'away_q{q}', 0) or 0)
            if home_q_val > 0 or away_q_val > 0:
                max_quarter = q
        
        # Set current_quarter field if needed
        current_q = int(row.get('current_quarter', 0) or 0)
        if current_q < max_quarter:
            df.at[idx, 'current_quarter'] = max_quarter
        
        # Calculate home and away scores
        current_q = max(current_q, max_quarter)
        home_score = sum([float(row.get(f'home_q{q}', 0) or 0) for q in range(1, current_q + 1)])
        away_score = sum([float(row.get(f'away_q{q}', 0) or 0) for q in range(1, current_q + 1)])
        
        df.at[idx, 'home_score'] = home_score
        df.at[idx, 'away_score'] = away_score
        
        # Calculate score ratio
        total_score = home_score + away_score
        df.at[idx, 'score_ratio'] = home_score / total_score if total_score > 0 else 0.5
        
        # Determine game status
        status, quarter, time_remaining = determine_game_status(row)
        df.at[idx, 'game_status'] = status
        df.at[idx, 'time_remaining_mins'] = time_remaining
    
    # Add matched_to_schedule column if not present
    if 'matched_to_schedule' not in df.columns:
        df['matched_to_schedule'] = False
    
    return df

def match_with_schedule(df):
    """
    Match live games with the official schedule.
    
    Args:
        df: DataFrame with live game data
        
    Returns:
        DataFrame with schedule matching information
    """
    if df.empty:
        return df
    
    # Make a copy to avoid modifying the original
    df = df.copy()
    
    try:
        # Try to fetch the schedule for the same date
        if 'game_date' in df.columns and not df.empty:
            # Get the date for the first game
            first_game_date = pd.to_datetime(df.iloc[0]['game_date'])
            date_str = first_game_date.strftime('%Y-%m-%d')
            
            # Fetch schedule
            response = safe_api_call(
                supabase.table("nba_game_schedule").select("*").eq("game_date", date_str).execute
            )
            
            if response and response.data:
                schedule_df = pd.DataFrame(response.data)
                
                # Check each live game against schedule
                for idx, game in df.iterrows():
                    home_team = str(game['home_team']).lower()
                    away_team = str(game['away_team']).lower()
                    
                    # Look for matching teams in schedule
                    for _, sched in schedule_df.iterrows():
                        sched_home = str(sched['home_team']).lower()
                        sched_away = str(sched['away_team']).lower()
                        
                        # Check for match in either direction
                        if (home_team in sched_home and away_team in sched_away) or \
                           (home_team in sched_away and away_team in sched_home):
                            df.at[idx, 'matched_to_schedule'] = True
                            # Use the official game_id if available
                            if 'game_id' in sched:
                                df.at[idx, 'schedule_game_id'] = sched['game_id']
                            break
    except Exception as e:
        print(f"Error matching with schedule: {e}")
        traceback.print_exc()
    
    return df

def get_active_games():
    """
    Get only active games from live games data.
    
    Returns:
        DataFrame with active games
    """
    # Fetch today's live games
    all_games = fetch_live_games_by_date()
    
    if all_games.empty:
        print("No games found today.")
        return pd.DataFrame()
    
    # Filter ONLY for games with LIVE status
    live_status_games = all_games[all_games['game_status'] == 'LIVE']
    
    if not live_status_games.empty:
        print(f"Found {len(live_status_games)} active games")
        return live_status_games
    
    # If no active games, return empty DataFrame
    print("No active games found.")
    return pd.DataFrame()

def create_generic_game_fallback():
    """
    Create a fallback generic NBA game record.
    
    Returns:
        DataFrame with a simulated game record
    """
    print(f"Creating fallback generic NBA game record")
    
    # Current time in Pacific
    pacific_tz = get_timezone()
    now_pt = get_current_time()
    
    # Create dummy record
    dummy_game = {
        'game_id': int(datetime.now().timestamp()),  # Unique ID based on timestamp
        'game_date': now_pt.strftime('%Y-%m-%d %H:%M:%S%z'),
        'home_team': 'Home Team',
        'away_team': 'Away Team',
        'home_score': 0,
        'away_score': 0,
        'home_q1': 0, 'home_q2': 0, 'home_q3': 0, 'home_q4': 0,
        'away_q1': 0, 'away_q2': 0, 'away_q3': 0, 'away_q4': 0,
        'current_quarter': 1,
        'is_finished': False,
        'game_status': 'SCHEDULED',
        'time_remaining_mins': 48,
        'matched_to_schedule': True
    }
    
    return pd.DataFrame([dummy_game])

def create_upcoming_games_from_schedule():
    """
    Create upcoming games from the schedule if no active games found.
    
    Returns:
        DataFrame with upcoming scheduled games
    """
    print("Checking for upcoming scheduled games")
    
    try:
        # Get today's date
        today = get_date_str()
        
        # Fetch schedule for today and upcoming days
        response = safe_api_call(
            supabase.table("nba_game_schedule").select("*")
                   .gte("game_date", today)
                   .order("game_date")
                   .limit(5)
                   .execute
        )
        
        if not response or not response.data:
            print("No upcoming games found in schedule")
            return pd.DataFrame()
        
        # Create DataFrame
        schedule_df = pd.DataFrame(response.data)
        
        # Convert to game format
        games_list = []
        for _, game in schedule_df.iterrows():
            game_dict = {
                'game_id': game.get('game_id', int(datetime.now().timestamp())),
                'game_date': game.get('game_date', today),
                'home_team': game.get('home_team', 'Home Team'),
                'away_team': game.get('away_team', 'Away Team'),
                'home_score': 0,
                'away_score': 0,
                'home_q1': 0, 'home_q2': 0, 'home_q3': 0, 'home_q4': 0,
                'away_q1': 0, 'away_q2': 0, 'away_q3': 0, 'away_q4': 0,
                'current_quarter': 0,
                'is_finished': False,
                'game_status': 'SCHEDULED',
                'time_remaining_mins': 48,
                'matched_to_schedule': True
            }
            games_list.append(game_dict)
        
        if games_list:
            print(f"Found {len(games_list)} upcoming games in schedule")
            return pd.DataFrame(games_list)
        else:
            return pd.DataFrame()
            
    except Exception as e:
        print(f"Error fetching upcoming games: {e}")
        traceback.print_exc()
        return pd.DataFrame()

# Custom implementation of predict_games to handle the import error
def predict_games_local(games_df, model=None):
    """
    Local implementation of predict_games for when the import fails.
    
    Args:
        games_df: DataFrame with game data
        model: Pre-loaded model (unused in this implementation)
        
    Returns:
        DataFrame with prediction results added
    """
    print("Using local game prediction implementation")
    
    # Make a copy to avoid modifying the original
    result_df = games_df.copy()
    
    # Process each game
    for idx, game in result_df.iterrows():
        current_q = int(game.get('current_quarter', 0) or 0)
        home_score = float(game.get('home_score', 0) or 0)
        away_score = float(game.get('away_score', 0) or 0)
        game_status = game.get('game_status', 'UNKNOWN')
        
        if game_status == 'FINAL':
            # For finished games, use the actual final score
            result_df.at[idx, 'predicted_home_score'] = home_score
            result_df.at[idx, 'predicted_away_score'] = away_score
            # Determine the winner after the fact
            result_df.at[idx, 'win_probability'] = 1.0 if home_score > away_score else 0.0
            
        elif game_status == 'LIVE' and current_q > 0:
            # For live games, project based on quarters played
            quarters_played = current_q
            quarters_remaining = 4 - quarters_played
            
            # Calculate average points per quarter so far
            home_ppq = home_score / quarters_played
            away_ppq = away_score / quarters_played
            
            # Project remaining quarters (simple linear projection)
            projected_home = home_score + (home_ppq * quarters_remaining)
            projected_away = away_score + (away_ppq * quarters_remaining)
            
            # Adjust for typical 4th quarter scoring changes
            if current_q == 3:  # Going into 4th quarter
                # Teams typically score 3-5% differently in 4th quarters
                home_q4_factor = 1.03 if home_score > away_score else 0.97
                away_q4_factor = 1.03 if away_score > home_score else 0.97
                
                # Apply the adjustment
                projected_home = home_score + (home_ppq * quarters_remaining * home_q4_factor)
                projected_away = away_score + (away_ppq * quarters_remaining * away_q4_factor)
            
            # Save projections
            result_df.at[idx, 'predicted_home_score'] = round(projected_home, 1)
            result_df.at[idx, 'predicted_away_score'] = round(projected_away, 1)
            
            # Calculate win probability based on lead and time remaining
            score_diff = home_score - away_score
            time_played_pct = current_q / 4.0
            
            # Simple win probability model
            if score_diff == 0:
                win_prob = 0.5  # Tied game
            else:
                # Larger leads later in the game are more significant
                normalized_diff = score_diff / (10 * (1 - time_played_pct + 0.1))
                win_prob = 1 / (1 + np.exp(-normalized_diff))  # Sigmoid function
            
            result_df.at[idx, 'win_probability'] = win_prob
            
        else:
            # Pre-game - use league averages with slight home court advantage
            result_df.at[idx, 'predicted_home_score'] = 110.5  # Average home score
            result_df.at[idx, 'predicted_away_score'] = 105.8  # Average away score
            result_df.at[idx, 'win_probability'] = 0.6  # Home court advantage
    
    # Add confidence and recommendations
    for idx, game in result_df.iterrows():
        game_status = game.get('game_status', 'UNKNOWN')
        
        # Add prediction confidence based on stage of game
        if game_status == 'FINAL':
            result_df.at[idx, 'prediction_confidence'] = 1.0  # 100% confidence for finished games
        elif game_status == 'LIVE':
            # Confidence increases as game progresses
            current_q = int(game.get('current_quarter', 0) or 0)
            result_df.at[idx, 'prediction_confidence'] = min(0.5 + (current_q * 0.125), 0.95)
        else:
            result_df.at[idx, 'prediction_confidence'] = 0.65  # Base confidence for pre-game
    
    return result_df

def fetch_and_predict_game(games_df=None):
    """
    Run predictions on active games.
    
    Args:
        games_df: Pre-loaded games DataFrame (optional)
        
    Returns:
        DataFrame with prediction results
    """
    # Get active games if not provided
    if games_df is None or games_df.empty:
        games_df = get_active_games()
    
    if games_df.empty:
        print("No active games available for prediction.")
        
        # Try to get upcoming scheduled games
        upcoming_games = create_upcoming_games_from_schedule()
        
        if not upcoming_games.empty:
            print("Using upcoming scheduled games for prediction")
            games_df = upcoming_games
        else:
            # Create a generic fallback game
            print("Using generic game fallback for prediction")
            games_df = create_generic_game_fallback()
    
    # Use local prediction implementation since the import has been failing
    print("Using local prediction implementation")
    return predict_games_local(games_df)

def display_game_prediction(prediction_df):
    """
    Display game prediction results in a user-friendly format.
    
    Args:
        prediction_df: DataFrame with prediction results
    """
    if prediction_df.empty:
        print("No prediction results to display.")
        return
    
    # Sort by game status (LIVE first, then SCHEDULED, then FINAL)
    status_order = {'LIVE': 0, 'SCHEDULED': 1, 'FINAL': 2, 'UNKNOWN': 3}
    prediction_df['status_order'] = prediction_df['game_status'].map(status_order).fillna(3)
    sorted_df = prediction_df.sort_values('status_order')
    
    # Process each game
    for idx, game in sorted_df.iterrows():
        home_team = game['home_team']
        away_team = game['away_team']
        current_q = int(game.get('current_quarter', 0) or 0)
        game_status = game.get('game_status', 'UNKNOWN')
        
        print("\n========================================")
        print(f"GAME: {home_team} vs {away_team}")
        print(f"STATUS: {game_status}, Quarter: {current_q if current_q > 0 else 'Pre-game'}")
        
        # Current score if game has started
        if current_q > 0:
            home_score = game.get('home_score', 0)
            away_score = game.get('away_score', 0)
            print(f"CURRENT SCORE: {home_team} {home_score} - {away_team} {away_score}")
            
            # Quarter scores
            print("QUARTER BREAKDOWN:")
            for q in range(1, current_q + 1):
                home_q = game.get(f'home_q{q}', 0)
                away_q = game.get(f'away_q{q}', 0)
                print(f"  Q{q}: {home_team} {home_q} - {away_team} {away_q}")
        
        # Prediction results
        if 'predicted_home_score' in game and 'predicted_away_score' in game:
            home_pred = game['predicted_home_score']
            away_pred = game['predicted_away_score']
            
            if game_status == 'FINAL':
                print("\nFINAL SCORE:")
            else:
                print("\nPREDICTION RESULTS:")
                
            print(f"  {'Final' if game_status == 'FINAL' else 'Projected Final'}: {home_team} {home_pred:.1f} - {away_team} {away_pred:.1f}")
            
            # Win probability
            if 'win_probability' in game and game_status != 'FINAL':
                win_prob = game['win_probability'] * 100
                print(f"  Win Probability: {home_team} {win_prob:.1f}%")
            
            # Show remaining points to score if game in progress
            if game_status == 'LIVE' and current_q > 0:
                remaining_home = max(0, home_pred - game.get('home_score', 0))
                remaining_away = max(0, away_pred - game.get('away_score', 0))
                print(f"  Remaining Points: {home_team} +{remaining_home:.1f}, {away_team} +{remaining_away:.1f}")
            
            # Show prediction confidence
            if 'prediction_confidence' in game and game_status != 'FINAL':
                confidence = game['prediction_confidence'] * 100
                print(f"  Prediction Confidence: {confidence:.1f}%")
        
        # Recommendations if available
        recs = {}
        for col in game.index:
            if col.startswith('rec_'):
                key = col[4:]
                if pd.notna(game[col]) and game[col]:
                    recs[key] = game[col]
        
        if recs:
            print("\nRECOMMENDATIONS:")
            for key, value in recs.items():
                print(f"  {key.replace('_', ' ').title()}: {value}")
        
        print("========================================")

# ---- Main Test Function ----

def test_prediction_on_active_games():
    """
    Test prediction on all active live games.
    """
    print("Testing NBA score prediction model on active live games")
    
    # Current time in preferred timezone
    current_time = get_current_time()
    print(f"Current Time: {current_time.strftime('%Y-%m-%d %H:%M:%S %Z')}")
    
    # Fetch active games and make prediction
    active_games = get_active_games()
    predictions = fetch_and_predict_game(active_games)
    
    # Display results
    display_game_prediction(predictions)
    
    return predictions

# Run the test on active games
active_game_predictions = test_prediction_on_active_games()

In [ ]:
# Cell 17A – Integrated Live Dashboard (Dash-Based with Merged Features)

import dash
from dash import dcc, html, Input, Output
import dash_bootstrap_components as dbc
import plotly.graph_objs as go
import pandas as pd
import numpy as np
# Fix the datetime import by importing the classes directly
from datetime import datetime, timedelta
import pytz
import traceback

# Add the missing import for supabase
from caching.supabase_client import supabase

# ----- Timezone and Data Utilities -----
def get_current_time_pt():
    """Get current time in Pacific timezone."""
    pacific_tz = pytz.timezone("America/Los_Angeles")
    return datetime.now(pacific_tz)  # Now works with the fixed import

def fetch_live_games():
    """
    Fetch active live games from database.
    Returns DataFrame with live game data.
    """
    try:
        # Get current date in Pacific Time
        pacific_tz = pytz.timezone("America/Los_Angeles")
        now_pt = datetime.now(pacific_tz)  
        today_pt = now_pt.date()
        
        # Define start and end of today in PT
        start_pt = datetime.combine(today_pt, datetime.min.time()).replace(tzinfo=pacific_tz)
        end_pt = datetime.combine(today_pt, datetime.max.time()).replace(tzinfo=pacific_tz)
        
        # Convert to UTC for database query
        start_utc = start_pt.astimezone(pytz.UTC).isoformat()
        end_utc = end_pt.astimezone(pytz.UTC).isoformat()
        
        print(f"Fetching games for range: {start_utc} to {end_utc}")
        
        # Fetch all games for today
        response = supabase.table("nba_live_game_stats").select("*").gte("game_date", start_utc).lte("game_date", end_utc).execute()
        
        if not response.data:
            print("No games found for today")
            return pd.DataFrame()
            
        # Convert to DataFrame
        all_games = pd.DataFrame(response.data)
        
        # Process game data
        all_games = process_game_data(all_games)
        
        # Filter for active games
        active_games = all_games[all_games['game_status'] == 'live'].copy()
        
        if active_games.empty:
            print("No actively live games found")
            # If no active games, return all games
            return all_games
        
        print(f"Found {len(active_games)} active games")
        return active_games
        
    except Exception as e:
        print(f"Error fetching live games: {e}")
        traceback.print_exc()
        return pd.DataFrame()

def process_game_data(df):
    """Process and determine game status for all games."""
    if df.empty:
        return df
    
    # Create a copy to avoid modifying original
    df = df.copy()
    
    # Convert date columns
    if 'game_date' in df.columns:
        df['game_date'] = pd.to_datetime(df['game_date'], errors='coerce', utc=True)
        
    # Ensure numeric columns are numeric
    for col in ['home_score', 'away_score', 'current_quarter']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    
    # Make quarter columns numeric
    for q in range(1, 5):
        for team in ['home', 'away']:
            col = f'{team}_q{q}'
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    
    # Determine game status for each game
    df['game_status'] = 'unknown'  # Default status
    
    for idx, row in df.iterrows():
        # Check if game is marked as finished
        if row.get('is_finished', False):
            df.at[idx, 'game_status'] = 'finished'
            continue
            
        # Calculate highest quarter with data
        max_quarter = 0
        for q in range(1, 5):
            home_q = float(row.get(f'home_q{q}', 0) or 0)
            away_q = float(row.get(f'away_q{q}', 0) or 0)
            if home_q > 0 or away_q > 0:
                max_quarter = q
        
        # Update current_quarter if needed
        current_q = int(row.get('current_quarter', 0) or 0)
        if current_q < max_quarter:
            df.at[idx, 'current_quarter'] = max_quarter
            current_q = max_quarter
        
        # Determine status based on quarter data
        if current_q > 0:
            # If all quarters have data and it's Q4, likely finished
            if current_q == 4 and max_quarter == 4:
                df.at[idx, 'game_status'] = 'finished'
            else:
                df.at[idx, 'game_status'] = 'live'
        else:
            # No quarter data - scheduled game
            df.at[idx, 'game_status'] = 'upcoming'
        
        # Calculate scores from quarters if needed
        if current_q > 0:
            home_score = sum([float(row.get(f'home_q{q}', 0) or 0) for q in range(1, current_q + 1)])
            away_score = sum([float(row.get(f'away_q{q}', 0) or 0) for q in range(1, current_q + 1)])
            
            # Update scores if needed
            if row.get('home_score', 0) != home_score:
                df.at[idx, 'home_score'] = home_score
            if row.get('away_score', 0) != away_score:
                df.at[idx, 'away_score'] = away_score
    
    return df

# Custom prediction function
def predict_games_local(df):
    """Local prediction function for games."""
    print("Using local prediction implementation")
    
    if df.empty:
        return df
        
    result_df = df.copy()
    
    # Process each game
    for idx, row in result_df.iterrows():
        current_q = int(row.get('current_quarter', 0) or 0)
        home_score = float(row.get('home_score', 0) or 0)
        away_score = float(row.get('away_score', 0) or 0)
        game_status = row.get('game_status', '').lower()
        
        if game_status == 'finished':
            # For finished games, use the actual final score
            result_df.at[idx, 'predicted_home_score'] = home_score
            result_df.at[idx, 'predicted_away_score'] = away_score
            # Determine the winner after the fact
            result_df.at[idx, 'win_probability'] = 1.0 if home_score > away_score else 0.0
            
        elif game_status == 'live' and current_q > 0:
            # For live games, project based on quarters played
            quarters_played = current_q
            quarters_remaining = 4 - quarters_played
            
            # Calculate average points per quarter so far
            home_ppq = home_score / quarters_played if quarters_played > 0 else 0
            away_ppq = away_score / quarters_played if quarters_played > 0 else 0
            
            # Project remaining quarters (simple linear projection)
            projected_home = home_score + (home_ppq * quarters_remaining)
            projected_away = away_score + (away_ppq * quarters_remaining)
            
            # Adjust for typical 4th quarter scoring changes
            if current_q == 3:  # Going into 4th quarter
                # Teams typically score 3-5% differently in 4th quarters
                home_q4_factor = 1.03 if home_score > away_score else 0.97
                away_q4_factor = 1.03 if away_score > home_score else 0.97
                
                # Apply the adjustment
                projected_home = home_score + (home_ppq * quarters_remaining * home_q4_factor)
                projected_away = away_score + (away_ppq * quarters_remaining * away_q4_factor)
            
            # Save projections
            result_df.at[idx, 'predicted_home_score'] = round(projected_home, 1)
            result_df.at[idx, 'predicted_away_score'] = round(projected_away, 1)
            
            # Calculate win probability based on lead and time remaining
            score_diff = home_score - away_score
            time_played_pct = current_q / 4.0
            
            # Simple win probability model
            if score_diff == 0:
                win_prob = 0.5  # Tied game
            else:
                # Larger leads later in the game are more significant
                normalized_diff = score_diff / (10 * (1 - time_played_pct + 0.1))
                win_prob = 1 / (1 + np.exp(-normalized_diff))  # Sigmoid function
            
            result_df.at[idx, 'win_probability'] = win_prob
            
        else:
            # Pre-game - use league averages with slight home court advantage
            result_df.at[idx, 'predicted_home_score'] = 110.5  # Average home score
            result_df.at[idx, 'predicted_away_score'] = 105.8  # Average away score
            result_df.at[idx, 'win_probability'] = 0.6  # Home court advantage
    
    # Add confidence and recommendations
    for idx, row in result_df.iterrows():
        game_status = row.get('game_status', '').lower()
        
        # Add prediction confidence based on stage of game
        if game_status == 'finished':
            result_df.at[idx, 'prediction_confidence'] = 1.0  # 100% confidence for finished games
        elif game_status == 'live':
            # Confidence increases as game progresses
            current_q = int(row.get('current_quarter', 0) or 0)
            result_df.at[idx, 'prediction_confidence'] = min(0.5 + (current_q * 0.125), 0.95)
        else:
            result_df.at[idx, 'prediction_confidence'] = 0.65  # Base confidence for pre-game
    
    return result_df

def get_dashboard_data():
    """
    Get real dashboard data from the database.
    Returns a processed DataFrame with game information.
    """
    try:
        # Fetch live games
        df = fetch_live_games()
        
        if df.empty:
            print("No games available for dashboard")
            return create_dummy_data()
        
        # Add timestamp column
        df['timestamp'] = get_current_time_pt().strftime("%Y-%m-%d %H:%M:%S")
        
        # Ensure required columns
        required_cols = [
            'game_id', 'home_team', 'away_team', 'game_status', 
            'current_quarter', 'home_score', 'away_score'
        ]
        
        for col in required_cols:
            if col not in df.columns:
                if col in ['home_score', 'away_score', 'current_quarter']:
                    df[col] = 0
                else:
                    df[col] = "Unknown"
        
        # Always use the local prediction implementation
        return predict_games_local(df)
    except Exception as e:
        print(f"Error in get_dashboard_data: {e}")
        traceback.print_exc()
        # Make sure we always return something
        return create_dummy_data()

def create_dummy_data():
    """Create dummy data when no real data is available."""
    try:
        now = get_current_time_pt().strftime("%Y-%m-%d %H:%M:%S")
        data = {
            "game_id": ["game_1", "game_2", "game_3"],
            "home_team": ["Lakers", "Bulls", "Celtics"],
            "away_team": ["Warriors", "Heat", "Knicks"],
            "game_status": ["upcoming", "upcoming", "upcoming"],
            "current_quarter": [0, 0, 0],
            "home_score": [0, 0, 0],
            "away_score": [0, 0, 0],
            "predicted_home_score": [108, 105, 110],
            "predicted_away_score": [102, 103, 112],
            "win_probability": [0.55, 0.52, 0.45],
            "timestamp": [now, now, now]
        }
        return pd.DataFrame(data)
    except Exception as e:
        print(f"Error creating dummy data: {e}")
        # Absolute last resort - hardcoded dummy data with no time function calls
        data = {
            "game_id": ["emergency_fallback"],
            "home_team": ["Home Team"],
            "away_team": ["Away Team"],
            "game_status": ["upcoming"],
            "current_quarter": [0],
            "home_score": [0],
            "away_score": [0],
            "predicted_home_score": [100],
            "predicted_away_score": [100],
            "win_probability": [0.5],
            "timestamp": ["Emergency fallback data"]
        }
        return pd.DataFrame(data)

def get_prediction_history(game_id):
    """
    Get prediction history for a specific game.
    In production, this would fetch from a database.
    """
    try:
        # Try to fetch from history table or create realistic history
        # For now, using a simple simulation
        game_data = get_dashboard_data()
        game = game_data[game_data['game_id'] == game_id]
        
        if game.empty:
            return []
            
        game = game.iloc[0]
        current_quarter = int(game.get('current_quarter', 0))
        
        if current_quarter <= 0:
            return []  # No history for upcoming games
            
        # Create simulated history
        history = []
        
        # For each quarter up to current
        for q in range(1, current_quarter + 1):
            # Simulate predictions getting more accurate as game progresses
            accuracy_factor = q / 4  # Gets closer to 1 as game progresses
            noise_factor = 1 - accuracy_factor
            
            # Final predictions as reference
            final_home = float(game.get('predicted_home_score', 100))
            final_away = float(game.get('predicted_away_score', 100))
            final_wp = float(game.get('win_probability', 0.5))
            
            # Add some noise to early predictions
            home_noise = np.random.normal(0, 10 * noise_factor)
            away_noise = np.random.normal(0, 10 * noise_factor)
            wp_noise = np.random.normal(0, 0.15 * noise_factor)
            
            history.append({
                "current_quarter": q,
                "predicted_home_score": final_home + home_noise,
                "predicted_away_score": final_away + away_noise,
                "win_probability": max(0.01, min(0.99, final_wp + wp_noise)),
                "timestamp": (get_current_time_pt() - timedelta(minutes=15*(current_quarter-q))).strftime("%H:%M:%S")
            })
            
        return history
            
    except Exception as e:
        print(f"Error getting prediction history: {e}")
        traceback.print_exc()
        return []

# ----- Dash App Setup -----
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
server = app.server  # For deployment purposes

# Get initial dashboard data
try:
    df_dashboard = get_dashboard_data()
    available_games = df_dashboard["game_id"].unique()
    initial_game = available_games[0] if len(available_games) > 0 else None
except Exception as e:
    print(f"Error getting initial data: {e}")
    df_dashboard = create_dummy_data()
    available_games = df_dashboard["game_id"].unique()
    initial_game = available_games[0] if len(available_games) > 0 else None

app.layout = dbc.Container([
    dbc.Row([
        dbc.Col(html.H2("NBA Score Prediction Dashboard"), width=12)
    ]),
    dbc.Row([
        dbc.Col(
            dcc.Dropdown(
                id='game-dropdown',
                options=[{'label': f"{row['home_team']} vs {row['away_team']} ({row['game_status'].upper()})", 
                          'value': row['game_id']} for idx, row in df_dashboard.iterrows()],
                value=initial_game,
                clearable=False,
                style={"width": "100%"}
            ),
            width=6
        ),
        dbc.Col(
            html.Div(id='last-update', style={'textAlign': 'right', 'fontSize': '16px'}),
            width=6
        )
    ], className="my-2"),
    dbc.Row([
        dbc.Col(
            html.Div(id='game-status-indicator', className="alert alert-info"),
            width=12
        )
    ], className="my-2"),
    dbc.Row([
        dbc.Col(dcc.Graph(id='main-dashboard-graph'), width=12)
    ]),
    dbc.Row([
        dbc.Col(dcc.Graph(id='history-graph'), width=12)
    ]),
    dcc.Interval(
        id='interval-component',
        interval=30*1000,  # Refresh every 30 seconds
        n_intervals=0
    )
], fluid=True)

# ----- Callback to Update Dashboard -----
@app.callback(
    [Output('main-dashboard-graph', 'figure'),
     Output('history-graph', 'figure'),
     Output('last-update', 'children'),
     Output('game-dropdown', 'options'),
     Output('game-status-indicator', 'children'),
     Output('game-status-indicator', 'className')],
    [Input('interval-component', 'n_intervals'),
     Input('game-dropdown', 'value')]
)
def update_dashboard(n_intervals, selected_game):
    try:
        # Fetch fresh dashboard data
        df = get_dashboard_data()
        
        # Update dropdown options
        dropdown_options = [
            {'label': f"{row['home_team']} vs {row['away_team']} ({row['game_status'].upper()})", 
            'value': row['game_id']} 
            for idx, row in df.iterrows()
        ]
        
        # Handle case when no games are available
        if df.empty or selected_game is None:
            empty_fig = go.Figure()
            empty_fig.update_layout(title="No game data available")
            return (empty_fig, empty_fig, "No data available", dropdown_options,
                    "No active games found", "alert alert-warning")
        
        # Filter for selected game
        game_data = df[df['game_id'] == selected_game]
        
        # If selected game is not in current data, take the first available game
        if game_data.empty and len(df) > 0:
            selected_game = df.iloc[0]['game_id']
            game_data = df[df['game_id'] == selected_game]
        
        if game_data.empty:
            empty_fig = go.Figure()
            empty_fig.update_layout(title="No game data available")
            return (empty_fig, empty_fig, "No data available", dropdown_options,
                    "Selected game not found", "alert alert-warning")
        
        game = game_data.iloc[0]
        timestamp = game.get('timestamp', get_current_time_pt().strftime("%Y-%m-%d %H:%M:%S"))
        
        # Create Main Graph: Bar Chart for Current vs Predicted Scores
        teams = [game['home_team'], game['away_team']]
        current_scores = [game['home_score'], game['away_score']]
        predicted_scores = [game['predicted_home_score'], game['predicted_away_score']]
        
        # Status indicator text and class
        game_status = game['game_status'].upper()
        current_q = int(game.get('current_quarter', 0))
        
        if game_status == 'LIVE':
            status_class = "alert alert-success"
            status_text = f"LIVE - Quarter {current_q}"
        elif game_status == 'FINISHED':
            status_class = "alert alert-secondary"
            status_text = "FINISHED"
        else:
            status_class = "alert alert-info"
            status_text = "UPCOMING"
        
        main_fig = go.Figure(data=[
            go.Bar(name='Current Score', x=teams, y=current_scores, marker_color='skyblue'),
            go.Bar(name='Predicted Final', x=teams, y=predicted_scores, marker_color='salmon')
        ])
        
        title = f"{game['home_team']} vs {game['away_team']} - {game_status}"
        if current_q > 0:
            title += f" (Q{current_q})"
            
        main_fig.update_layout(
            barmode='group', 
            title=title,
            xaxis_title="Team", 
            yaxis_title="Score"
        )
        
        # Add score labels to bars
        for i, (team, score, pred) in enumerate(zip(teams, current_scores, predicted_scores)):
            main_fig.add_annotation(
                x=team, y=score,
                text=f"{score:.0f}",
                showarrow=False,
                yshift=10
            )
            main_fig.add_annotation(
                x=team, y=pred,
                text=f"{pred:.1f}",
                showarrow=False,
                yshift=10
            )
        
        # Add Win Probability annotation if available
        if game_status.lower() == 'live' and 'win_probability' in game:
            win_prob = game['win_probability'] * 100
            prob_text = (f"{game['home_team']} Win: {win_prob:.1f}%" if win_prob > 50 
                        else f"{game['away_team']} Win: {100-win_prob:.1f}%")
            main_fig.add_annotation(
                x=0.5, y=0.05, 
                xref="paper", yref="paper",
                text=prob_text, 
                showarrow=False,
                font=dict(color="black", size=14),
                bgcolor="lightyellow", 
                opacity=0.8
            )
        
        # Create History Graph: Line Chart of Prediction Evolution
        history = get_prediction_history(selected_game)
        
        if history and len(history) >= 2:
            quarters = [pt["current_quarter"] for pt in history]
            home_preds = [pt["predicted_home_score"] for pt in history]
            away_preds = [pt["predicted_away_score"] for pt in history]
            win_probs = [pt["win_probability"] for pt in history]
            
            history_fig = go.Figure()
            history_fig.add_trace(go.Scatter(
                x=quarters, y=home_preds,
                mode='lines+markers',
                name=f"{game['home_team']} Prediction",
                line=dict(color='blue')
            ))
            history_fig.add_trace(go.Scatter(
                x=quarters, y=away_preds,
                mode='lines+markers',
                name=f"{game['away_team']} Prediction",
                line=dict(color='red')
            ))
            
            # Secondary axis for win probability
            history_fig.add_trace(go.Scatter(
                x=quarters, y=[wp * 100 for wp in win_probs],
                mode='lines+markers',
                name="Win Probability (%)",
                line=dict(color='green', dash='dash'),
                yaxis='y2'
            ))
            
            history_fig.update_layout(
                title="Prediction Evolution by Quarter",
                xaxis_title="Quarter",
                yaxis_title="Predicted Score",
                yaxis2=dict(
                    overlaying='y',
                    side='right',
                    title="Win Probability (%)",
                    range=[0, 100]
                )
            )
        else:
            history_fig = go.Figure()
            
            if game_status.lower() == 'upcoming':
                msg = "Game has not started - No prediction history available"
            else:
                msg = "No prediction history available"
                
            history_fig.update_layout(title=msg)
        
        last_update_text = f"Last Update: {timestamp}"
        
        return main_fig, history_fig, last_update_text, dropdown_options, status_text, status_class
    
    except Exception as e:
        print(f"Error in dashboard update: {e}")
        traceback.print_exc()
        # Create a fallback response
        empty_fig = go.Figure()
        empty_fig.update_layout(title="Error loading dashboard data")
        return (
            empty_fig, empty_fig, 
            f"Error: {str(e)}", 
            [{'label': 'Error', 'value': 'error'}],
            "Dashboard Error", 
            "alert alert-danger"
        )

# ----- To Run the Dashboard (for standalone execution) -----
if __name__ == '__main__':
    app.run_server(debug=True)

In [ ]:
# Cell 17B – Enhanced Visualization Components

import plotly.graph_objs as go
import numpy as np

def create_prediction_evolution_chart(data, actual_data=None, win_prob_data=None, confidence_intervals=None):
    """
    Create an enhanced prediction evolution chart.
    
    Parameters:
      - data: List/array of prediction values over time.
      - actual_data: (Optional) List/array of actual scores.
      - win_prob_data: (Optional) List/array of win probability values.
      - confidence_intervals: (Optional) Tuple (lower_bounds, upper_bounds) arrays for confidence intervals.
    
    Returns:
      - fig: A Plotly figure object.
    """
    time_points = list(range(len(data)))
    
    fig = go.Figure()
    
    # Plot prediction evolution.
    fig.add_trace(go.Scatter(
        x=time_points,
        y=data,
        mode='lines+markers',
        name='Predicted Score',
        line=dict(width=2)
    ))
    
    # Plot actual scores if provided.
    if actual_data is not None:
        fig.add_trace(go.Scatter(
            x=time_points,
            y=actual_data,
            mode='lines+markers',
            name='Actual Score',
            line=dict(width=2, dash='dash')
        ))
    
    # Plot win probability trend if provided.
    if win_prob_data is not None:
        fig.add_trace(go.Scatter(
            x=time_points,
            y=win_prob_data,
            mode='lines+markers',
            name='Win Probability',
            yaxis='y2',
            line=dict(width=2, color='green')
        ))
        # Set up a secondary y-axis.
        fig.update_layout(
            yaxis2=dict(
                title='Win Probability',
                overlaying='y',
                side='right',
                range=[0, 1]
            )
        )
    
    # Plot confidence intervals if provided.
    if confidence_intervals is not None:
        lower_bounds, upper_bounds = confidence_intervals
        fig.add_trace(go.Scatter(
            x=time_points + time_points[::-1],
            y=upper_bounds + lower_bounds[::-1],
            fill='toself',
            fillcolor='rgba(0,100,80,0.2)',
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo="skip",
            showlegend=True,
            name='Confidence Interval'
        ))
    
    # Add a dynamic confidence indicator annotation.
    if confidence_intervals is not None:
        last_confidence = upper_bounds[-1] - lower_bounds[-1]
        fig.add_annotation(
            x=time_points[-1],
            y=data[-1],
            text=f"Confidence: {last_confidence:.2f}",
            showarrow=True,
            arrowhead=1
        )
    
    fig.update_layout(
        title="Prediction Evolution Over Game Time",
        xaxis_title="Time Point (Interval/Quarter)",
        yaxis_title="Score",
        legend_title="Metrics",
        margin=dict(l=40, r=40, t=50, b=40)
    )
    
    return fig

# Example usage:
# predictions = [100, 102, 105, 107, 110]
# actual_scores = [98, 103, 106, 108, 111]
# win_probabilities = [0.55, 0.57, 0.60, 0.63, 0.65]
# lower_conf = [98, 100, 102, 104, 107]
# upper_conf = [102, 104, 108, 110, 113]
# fig = create_prediction_evolution_chart(predictions, actual_scores, win_probabilities, (lower_conf, upper_conf))
# fig.show()


In [ ]:
# Cell 18: Enhanced Model Training with Improved Methodology (Optimized)

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import joblib
import os
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import logging
from sqlalchemy import create_engine, text
from typing import Dict, List, Tuple, Any, Optional
import config
import time

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('model_training')

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

class FeatureEngineering:
    """Optimized feature engineering module for NBA score prediction."""
    
    @staticmethod
    def calculate_rest_features(df: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate rest-related features using vectorized operations.
        
        Args:
            df: DataFrame with game data including dates and teams
            
        Returns:
            DataFrame with rest features added
        """
        start_time = time.time()
        logger.info("Calculating rest-related features")
        
        # Copy input to avoid modifying original
        result_df = df.copy()
        
        # Ensure game_date is datetime and sort
        result_df['game_date'] = pd.to_datetime(result_df['game_date'])
        result_df = result_df.sort_values(['game_date'])
        
        # Initialize rest features with defaults
        result_df['rest_days_home'] = 2  # Default
        result_df['rest_days_away'] = 2  # Default
        result_df['is_home_b2b'] = 0     # Default: not back-to-back
        result_df['is_away_b2b'] = 0     # Default: not back-to-back
        
        # Get unique teams - faster approach
        teams = set(df['home_team'].unique()) | set(df['away_team'].unique())
        
        # Process each team - but with optimized approach
        for team in teams:
            # Create team schedule with both home and away games
            team_schedule = pd.concat([
                result_df[result_df['home_team'] == team][['game_id', 'game_date']].assign(team_role='home'),
                result_df[result_df['away_team'] == team][['game_id', 'game_date']].assign(team_role='away')
            ]).sort_values('game_date')
            
            if team_schedule.empty:
                continue
                
            # Calculate rest days using vectorized operations
            team_schedule['prev_date'] = team_schedule['game_date'].shift(1)
            team_schedule['days_rest'] = (team_schedule['game_date'] - team_schedule['prev_date']).dt.days
            team_schedule['days_rest'] = team_schedule['days_rest'].fillna(5).clip(1, 10)
            team_schedule['is_b2b'] = (team_schedule['days_rest'] == 1).astype(int)
            
            # Update main dataframe efficiently - home games
            home_schedule = team_schedule[team_schedule['team_role'] == 'home']
            if not home_schedule.empty:
                home_idx = result_df[result_df['home_team'] == team].index
                result_df.loc[home_idx, 'rest_days_home'] = result_df.loc[home_idx, 'game_id'].map(
                    home_schedule.set_index('game_id')['days_rest']
                ).fillna(2)
                result_df.loc[home_idx, 'is_home_b2b'] = result_df.loc[home_idx, 'game_id'].map(
                    home_schedule.set_index('game_id')['is_b2b']
                ).fillna(0)
            
            # Update main dataframe efficiently - away games
            away_schedule = team_schedule[team_schedule['team_role'] == 'away']
            if not away_schedule.empty:
                away_idx = result_df[result_df['away_team'] == team].index
                result_df.loc[away_idx, 'rest_days_away'] = result_df.loc[away_idx, 'game_id'].map(
                    away_schedule.set_index('game_id')['days_rest']
                ).fillna(2)
                result_df.loc[away_idx, 'is_away_b2b'] = result_df.loc[away_idx, 'game_id'].map(
                    away_schedule.set_index('game_id')['is_b2b']
                ).fillna(0)
        
        # Calculate rest advantage
        result_df['rest_advantage'] = result_df['rest_days_home'] - result_df['rest_days_away']
        result_df['rest_advantage'] = result_df['rest_advantage'].clip(-5, 5)
        
        elapsed = time.time() - start_time
        logger.info(f"Rest features calculation complete in {elapsed:.2f} seconds")
        return result_df
    
    @staticmethod
    def calculate_momentum_features(df: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate momentum-related features using vectorized operations.
        
        Args:
            df: DataFrame with quarter scores
            
        Returns:
            DataFrame with momentum features added
        """
        start_time = time.time()
        logger.info("Calculating momentum-related features")
        
        # Copy input to avoid modifying original
        result_df = df.copy()
        
        # Initialize momentum features with zeros if they don't exist
        for col in ['q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum']:
            if col not in result_df.columns:
                result_df[col] = 0
                
        # Make sure quarter scores are numeric
        for q in range(1, 5):
            home_q = f'home_q{q}'
            away_q = f'away_q{q}'
            if home_q in result_df.columns:
                result_df[home_q] = pd.to_numeric(result_df[home_q], errors='coerce').fillna(0)
            else:
                result_df[home_q] = 0
                
            if away_q in result_df.columns:
                result_df[away_q] = pd.to_numeric(result_df[away_q], errors='coerce').fillna(0)
            else:
                result_df[away_q] = 0
        
        # Calculate quarter-to-quarter momentum using vectorized operations
        result_df['q1_to_q2_momentum'] = (result_df['home_q2'] - result_df['home_q1']) - (result_df['away_q2'] - result_df['away_q1'])
        result_df['q2_to_q3_momentum'] = (result_df['home_q3'] - result_df['home_q2']) - (result_df['away_q3'] - result_df['away_q2'])
        result_df['q3_to_q4_momentum'] = (result_df['home_q4'] - result_df['home_q3']) - (result_df['away_q4'] - result_df['away_q3'])
        
        # Cap extreme values
        result_df['q1_to_q2_momentum'] = result_df['q1_to_q2_momentum'].clip(-20, 20)
        result_df['q2_to_q3_momentum'] = result_df['q2_to_q3_momentum'].clip(-20, 20)
        result_df['q3_to_q4_momentum'] = result_df['q3_to_q4_momentum'].clip(-20, 20)
        
        # Calculate weighted cumulative momentum
        result_df['cumulative_momentum'] = (
            result_df['q1_to_q2_momentum'] * 0.2 + 
            result_df['q2_to_q3_momentum'] * 0.3 + 
            result_df['q3_to_q4_momentum'] * 0.5
        )
        
        # Normalize to [-1, 1]
        result_df['cumulative_momentum'] = result_df['cumulative_momentum'] / 15.0
        result_df['cumulative_momentum'] = result_df['cumulative_momentum'].clip(-1, 1)
        
        elapsed = time.time() - start_time
        logger.info(f"Momentum features calculation complete in {elapsed:.2f} seconds")
        return result_df
    
    @staticmethod
    def calculate_matchup_features(df: pd.DataFrame, max_matchups=3, max_days=365) -> pd.DataFrame:
        """
        Calculate matchup-related features with optimizations.
        
        Args:
            df: DataFrame with game data
            max_matchups: Maximum number of previous matchups to consider
            max_days: Maximum number of days to look back for previous matchups
            
        Returns:
            DataFrame with matchup features added
        """
        start_time = time.time()
        logger.info("Calculating matchup-related features")
        
        # Copy input to avoid modifying original
        result_df = df.copy()
        
        # Initialize matchup features
        result_df['prev_matchup_diff'] = 0
        
        # Ensure datetime
        result_df['game_date'] = pd.to_datetime(result_df['game_date'])
        
        # Create team pair identifier for faster lookup
        result_df['team_pair'] = result_df.apply(
            lambda x: tuple(sorted([str(x['home_team']), str(x['away_team'])])), 
            axis=1
        )
        
        # Group by team pair
        team_pairs = result_df.groupby('team_pair')
        
        # Process each pair more efficiently
        for team_pair, games in team_pairs:
            # Sort by date
            games = games.sort_values('game_date')
            
            # Skip pairs with only one game
            if len(games) <= 1:
                continue
                
            # Process each game in this matchup
            for i, (idx, game) in enumerate(games.iterrows()):
                # Skip first game (no previous matchups)
                if i == 0:
                    continue
                    
                # Find previous matchups within time window
                prev_cutoff_date = game['game_date'] - timedelta(days=max_days)
                prev_games = games[
                    (games['game_date'] < game['game_date']) & 
                    (games['game_date'] >= prev_cutoff_date)
                ].tail(max_matchups)  # Limit to most recent matchups
                
                if prev_games.empty:
                    continue
                
                # Calculate point differentials from home team perspective
                diffs = []
                for _, prev in prev_games.iterrows():
                    if prev['home_team'] == game['home_team']:
                        # Same home team
                        diff = prev['home_score'] - prev['away_score']
                    else:
                        # Teams reversed
                        diff = prev['away_score'] - prev['home_score']
                    diffs.append(diff)
                
                # Calculate mean and cap extreme values
                avg_diff = np.mean(diffs)
                avg_diff = np.clip(avg_diff, -15, 15)
                
                # Update the dataframe
                result_df.loc[idx, 'prev_matchup_diff'] = avg_diff
        
        # Drop temporary column
        result_df.drop('team_pair', axis=1, inplace=True)
        
        elapsed = time.time() - start_time
        logger.info(f"Matchup features calculation complete in {elapsed:.2f} seconds")
        return result_df
    
    @staticmethod
    def calculate_all_features(df: pd.DataFrame, max_matchups=3, max_days=365) -> pd.DataFrame:
        """
        Calculate all features for model training with optimizations.
        
        Args:
            df: DataFrame with base game data
            max_matchups: Maximum number of previous matchups to consider
            max_days: Maximum number of days to look back for previous matchups
            
        Returns:
            DataFrame with all features added
        """
        # Make sure game_date is datetime
        df['game_date'] = pd.to_datetime(df['game_date'])
        
        # Calculate score ratio
        if 'home_score' in df.columns and 'away_score' in df.columns:
            total_score = df['home_score'] + df['away_score']
            df['score_ratio'] = df['home_score'] / total_score.where(total_score > 0, 1)
        
        # Apply feature calculations
        df = FeatureEngineering.calculate_rest_features(df)
        df = FeatureEngineering.calculate_momentum_features(df)
        df = FeatureEngineering.calculate_matchup_features(df, max_matchups, max_days)
        
        # Add interaction features
        df['momentum_rest_interaction'] = df['cumulative_momentum'] * df['rest_advantage']
        
        return df

class TimeBasedModelEvaluation:
    """Time-based model evaluation for sports prediction models."""
    
    def __init__(self, n_splits=3, test_size=0.2, gap=20):
        """
        Initialize time-based model evaluation with reduced splits.
        
        Args:
            n_splits: Number of time-based splits (reduced for speed)
            test_size: Proportion of data for testing
            gap: Number of days gap between train and test sets (reduced)
        """
        self.n_splits = n_splits
        self.test_size = test_size
        self.gap = gap
        
    def create_time_based_splits(self, df: pd.DataFrame) -> List[Tuple[np.ndarray, np.ndarray]]:
        """
        Create time-based splits for time series cross-validation.
        
        Args:
            df: DataFrame with 'game_date' column
            
        Returns:
            List of (train_indices, test_indices) tuples
        """
        # Ensure date is in datetime format
        df['game_date'] = pd.to_datetime(df['game_date'])
        
        # Sort by date
        df = df.sort_values('game_date')
        
        # Total number of samples
        n_samples = len(df)
        
        # Size of each test fold
        test_fold_size = int(n_samples * self.test_size)
        
        # Create splits
        splits = []
        
        for i in range(self.n_splits):
            # Calculate end index for this fold's test set
            test_end = n_samples - i * test_fold_size
            test_start = test_end - test_fold_size
            
            # Ensure valid indices
            if test_start < 0:
                continue
                
            # Get test dates
            test_start_date = df.iloc[test_start]['game_date']
            
            # Calculate train end date (with gap)
            train_end_date = test_start_date - timedelta(days=self.gap)
            
            # Get train indices (all samples before train_end_date)
            train_indices = df[df['game_date'] < train_end_date].index.values
            
            # Get test indices
            test_indices = df.iloc[test_start:test_end].index.values
            
            # Only add valid splits (with enough train and test samples)
            if len(train_indices) > 10 and len(test_indices) > 10:
                splits.append((train_indices, test_indices))
        
        return splits
    
    def evaluate_model(self, model, X: pd.DataFrame, y: pd.Series, date_column: str) -> Dict[str, Any]:
        """
        Evaluate model using time-based cross-validation.
        
        Args:
            model: Model to evaluate
            X: Feature DataFrame
            y: Target Series
            date_column: Name of date column
            
        Returns:
            Dictionary with evaluation metrics
        """
        df = X.copy()
        df[date_column] = pd.to_datetime(df[date_column])
        df['target'] = y
        
        # Create time-based splits
        splits = self.create_time_based_splits(df)
        
        # Track metrics
        train_scores = []
        test_scores = []
        train_mse = []
        test_mse = []
        test_mae = []
        feature_importances = []
        
        # Drop date and target columns from features
        feature_cols = [col for col in df.columns if col != date_column and col != 'target']
        
        # Evaluate on each split
        for i, (train_idx, test_idx) in enumerate(splits):
            # Get train and test sets
            X_train = df.loc[train_idx, feature_cols]
            y_train = df.loc[train_idx, 'target']
            X_test = df.loc[test_idx, feature_cols]
            y_test = df.loc[test_idx, 'target']
            
            # Train model
            model.fit(X_train, y_train)
            
            # Score model
            train_pred = model.predict(X_train)
            test_pred = model.predict(X_test)
            
            # Calculate metrics
            train_r2 = r2_score(y_train, train_pred)
            test_r2 = r2_score(y_test, test_pred)
            train_mse_val = mean_squared_error(y_train, train_pred)
            test_mse_val = mean_squared_error(y_test, test_pred)
            test_mae_val = mean_absolute_error(y_test, test_pred)
            
            # Save metrics
            train_scores.append(train_r2)
            test_scores.append(test_r2)
            train_mse.append(train_mse_val)
            test_mse.append(test_mse_val)
            test_mae.append(test_mae_val)
            
            # Save feature importances if available
            if hasattr(model, 'feature_importances_'):
                feature_importances.append(model.feature_importances_)
            
            # Log progress
            logger.info(f"Split {i+1}/{len(splits)}: Train R² = {train_r2:.4f}, Test R² = {test_r2:.4f}, Test RMSE = {np.sqrt(test_mse_val):.2f}")
        
        # Calculate average metrics
        avg_train_r2 = np.mean(train_scores)
        avg_test_r2 = np.mean(test_scores)
        avg_train_mse = np.mean(train_mse)
        avg_test_mse = np.mean(test_mse)
        avg_test_mae = np.mean(test_mae)
        
        # Calculate average feature importances if available
        avg_feature_importances = None
        if feature_importances:
            avg_feature_importances = np.mean(feature_importances, axis=0)
            
        return {
            'train_r2': avg_train_r2,
            'test_r2': avg_test_r2,
            'train_mse': avg_train_mse,
            'test_mse': avg_test_mse,
            'test_rmse': np.sqrt(avg_test_mse),
            'test_mae': avg_test_mae,
            'feature_importances': avg_feature_importances,
            'feature_names': feature_cols
        }
    
def load_games_data(start_date=None, end_date=None, min_games=1000, sample_frac=None) -> pd.DataFrame:
    """
    Load historical NBA game data with option to sample for faster development.
    
    Args:
        start_date: Optional start date filter (YYYY-MM-DD)
        end_date: Optional end date filter (YYYY-MM-DD)
        min_games: Minimum number of games to fetch
        sample_frac: Optional fraction of data to sample (e.g., 0.5 for 50%)
        
    Returns:
        DataFrame with game data
    """
    logger.info("Loading historical game data from database")
    start_time = time.time()
    
    try:
        # Create database connection
        engine = create_engine(config.DATABASE_URL)
        
        # Prepare query parameters
        params = {}
        where_clauses = []
        
        if start_date:
            where_clauses.append("game_date >= :start_date")
            params['start_date'] = start_date
            
        if end_date:
            where_clauses.append("game_date <= :end_date")
            params['end_date'] = end_date
        
        # Build query
        query = "SELECT * FROM nba_historical_game_stats"
        if where_clauses:
            query += " WHERE " + " AND ".join(where_clauses)
        query += " ORDER BY game_date"
        
        # Execute query
        with engine.connect() as conn:
            result = conn.execute(text(query), params)
            data = result.fetchall()
            
            # Convert to DataFrame
            df = pd.DataFrame(data, columns=result.keys())
            
        logger.info(f"Successfully loaded {len(df)} historical games")
        
        # Sample data if requested (for faster development)
        if sample_frac and 0 < sample_frac < 1:
            # Stratified sampling by year to maintain time distribution
            df['year'] = pd.to_datetime(df['game_date']).dt.year
            stratified_sample = df.groupby('year').apply(lambda x: x.sample(frac=sample_frac))
            df = stratified_sample.reset_index(drop=True)
            df.drop('year', axis=1, inplace=True)
            logger.info(f"Sampled down to {len(df)} games for faster processing")
        
        # If we don't have enough data, try without date filters
        if len(df) < min_games and (start_date or end_date):
            logger.info(f"Not enough data ({len(df)} < {min_games}). Trying without date filters.")
            return load_games_data(start_date=None, end_date=None, min_games=min_games, sample_frac=sample_frac)
        
        elapsed = time.time() - start_time
        logger.info(f"Data loading complete in {elapsed:.2f} seconds")
        return df
        
    except Exception as e:
        logger.error(f"Error loading game data: {e}")
        
        # Fallback to simulated data if necessary for development
        if os.getenv('DEVELOPMENT_MODE', '').lower() == 'true':
            logger.warning("Using simulated data for development")
            return generate_simulated_data(min_games)
        
        raise e

def generate_simulated_data(n_games=500):
    """Generate simulated game data for development purposes."""
    np.random.seed(42)
    
    # Create date range (reduced sample size for speed)
    start_date = datetime(2020, 1, 1)
    dates = [start_date + timedelta(days=i) for i in range(n_games)]
    
    # Create team names
    teams = [
        "Atlanta Hawks", "Boston Celtics", "Brooklyn Nets", "Charlotte Hornets",
        "Chicago Bulls", "Cleveland Cavaliers", "Dallas Mavericks", "Denver Nuggets",
        "Detroit Pistons", "Golden State Warriors", "Houston Rockets", "Indiana Pacers",
        "Los Angeles Clippers", "Los Angeles Lakers", "Memphis Grizzlies", "Miami Heat",
        "Milwaukee Bucks", "Minnesota Timberwolves", "New Orleans Pelicans", "New York Knicks",
        "Oklahoma City Thunder", "Orlando Magic", "Philadelphia 76ers", "Phoenix Suns",
        "Portland Trail Blazers", "Sacramento Kings", "San Antonio Spurs", "Toronto Raptors",
        "Utah Jazz", "Washington Wizards"
    ]
    
    # Generate random games
    games = []
    for i in range(n_games):
        # Select two random teams
        team_indices = np.random.choice(len(teams), 2, replace=False)
        home_team = teams[team_indices[0]]
        away_team = teams[team_indices[1]]
        
        # Generate random quarter scores
        home_q1 = np.random.randint(15, 35)
        home_q2 = np.random.randint(15, 35)
        home_q3 = np.random.randint(15, 35)
        home_q4 = np.random.randint(15, 35)
        
        away_q1 = np.random.randint(15, 35)
        away_q2 = np.random.randint(15, 35)
        away_q3 = np.random.randint(15, 35)
        away_q4 = np.random.randint(15, 35)
        
        # Calculate total scores
        home_score = home_q1 + home_q2 + home_q3 + home_q4
        away_score = away_q1 + away_q2 + away_q3 + away_q4
        
        # Create game record
        game = {
            'game_id': i + 1000,
            'game_date': dates[i],
            'home_team': home_team,
            'away_team': away_team,
            'home_q1': home_q1,
            'home_q2': home_q2,
            'home_q3': home_q3,
            'home_q4': home_q4,
            'away_q1': away_q1,
            'away_q2': away_q2,
            'away_q3': away_q3,
            'away_q4': away_q4,
            'home_score': home_score,
            'away_score': away_score
        }
        
        games.append(game)
    
    return pd.DataFrame(games)

def optimize_hyperparameters(X_train, y_train, cv=None, n_iter=20):
    """
    Optimize model hyperparameters using randomized search (faster than grid search).
    
    Args:
        X_train: Training features
        y_train: Training target
        cv: Cross-validation strategy
        n_iter: Number of parameter settings to try
        
    Returns:
        Tuple of (best_model, search_results)
    """
    logger.info("Optimizing model hyperparameters")
    start_time = time.time()
    
    # Define parameter distribution for random search
    param_distributions = {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.05, 0.1, 0.15],
        'max_depth': [3, 4, 5],
        'subsample': [0.7, 0.8, 0.9],
        'min_samples_split': [2, 4, 6],
    }
    
    # Create base model
    base_model = GradientBoostingRegressor(random_state=42)
    
    # Set up randomized search (faster than grid search)
    if cv is None:
        # Use 3-fold time series CV by default
        cv = TimeSeriesSplit(n_splits=3)
        
    random_search = RandomizedSearchCV(
        base_model,
        param_distributions=param_distributions,
        n_iter=n_iter,  # Try fewer combinations for speed
        cv=cv,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1,
        random_state=42
    )
    
    # Run random search
    random_search.fit(X_train, y_train)
    
    # Get best model
    best_model = random_search.best_estimator_
    
    elapsed = time.time() - start_time
    logger.info(f"Hyperparameter optimization complete in {elapsed:.2f} seconds")
    logger.info(f"Best parameters: {random_search.best_params_}")
    logger.info(f"Best score: {-random_search.best_score_:.4f} MSE")
    
    return best_model, random_search

def visualize_feature_importance(model, feature_names, top_n=10):
    """
    Visualize feature importance (limited to top features for clarity).
    
    Args:
        model: Trained model with feature_importances_ attribute
        feature_names: List of feature names
        top_n: Number of top features to display (reduced)
    """
    if not hasattr(model, 'feature_importances_'):
        logger.warning("Model does not have feature_importances_ attribute")
        return
    
    # Extract feature importances
    importances = model.feature_importances_
    
    # Create DataFrame of features and importances
    feature_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    # Display top features
    top_features = feature_df.head(top_n)
    
    # Plot
    plt.figure(figsize=(10, 6))
    sns.barplot(x='Importance', y='Feature', data=top_features)
    plt.title(f'Top {top_n} Feature Importances')
    plt.tight_layout()
    plt.show()
    
def train_enhanced_model(fast_mode=True):
    """
    Train an enhanced NBA score prediction model with improved methodology.
    
    Args:
        fast_mode: If True, use optimizations for faster training
    
    Returns:
        Trained model
    """
    overall_start_time = time.time()
    logger.info("Starting enhanced model training")
    
    # Step 1: Load historical game data
    try:
        # Sample data for faster processing in fast_mode
        sample_frac = 0.5 if fast_mode else None
        # Use more recent data for faster processing
        start_date = '2020-10-01' if fast_mode else '2018-10-01'
        
        df = load_games_data(start_date=start_date, sample_frac=sample_frac)
    except Exception as e:
        logger.error(f"Failed to load data: {e}")
        return None
    
    # Step 2: Calculate features
    try:
        # Use optimized parameters for matchup features
        max_matchups = 3 if fast_mode else 5  # Consider fewer previous matchups
        max_days = 180 if fast_mode else 365  # Look back over shorter time period
        
        df = FeatureEngineering.calculate_all_features(df, max_matchups, max_days)
    except Exception as e:
        logger.error(f"Feature engineering failed: {e}")
        return None
    
    # Step 3: Define features and target
    features = [
        'home_q1', 'home_q2', 'home_q3', 'home_q4',
        'score_ratio', 'prev_matchup_diff',
        'rest_days_home', 'rest_days_away', 'rest_advantage',
        'is_home_b2b', 'is_away_b2b',
        'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum',
        'momentum_rest_interaction'
    ]
    
    target = 'home_score'
    
    # Make sure all feature columns exist and are numeric
    for col in features:
        if col not in df.columns:
            logger.warning(f"Missing feature column: {col}")
            df[col] = 0
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    
    # Step 4: Prepare data
    X = df[features]
    y = df[target]
    
    # Step 5: Split data using time-based approach
    # Get last 20% for final testing
    train_size = int(0.8 * len(df))
    X_train_full, X_test = X.iloc[:train_size], X.iloc[train_size:]
    y_train_full, y_test = y.iloc[:train_size], y.iloc[train_size:]
    
    # In fast mode, use a smaller validation split
    if fast_mode:
        # Use 90% of training data for actual training
        train_val_split = int(0.9 * len(X_train_full))
    else:
        # Use 80% of training data for actual training
        train_val_split = int(0.8 * len(X_train_full))
        
    X_train, X_val = X_train_full.iloc[:train_val_split], X_train_full.iloc[train_val_split:]
    y_train, y_val = y_train_full.iloc[:train_val_split], y_train_full.iloc[train_val_split:]
    
    logger.info(f"Train set: {X_train.shape[0]} samples")
    logger.info(f"Validation set: {X_val.shape[0]} samples")
    logger.info(f"Test set: {X_test.shape[0]} samples")
    
    # Step 6: Optimize hyperparameters - use fewer iterations in fast mode
    try:
        n_iter = 10 if fast_mode else 20
        optimized_model, search_results = optimize_hyperparameters(X_train, y_train, n_iter=n_iter)
        logger.info(f"Hyperparameter optimization complete")
    except Exception as e:
        logger.error(f"Hyperparameter optimization failed: {e}")
        # Fallback to default model
        optimized_model = GradientBoostingRegressor(
            n_estimators=100 if fast_mode else 200,
            learning_rate=0.1,
            max_depth=4,
            subsample=0.8,
            random_state=42
        )
    
    # Step 7: Evaluate model with time-based cross-validation
    # In fast mode, use fewer splits and smaller gap
    n_splits = 3 if fast_mode else 5
    gap_days = 20 if fast_mode else 30
    
    evaluator = TimeBasedModelEvaluation(n_splits=n_splits, test_size=0.2, gap=gap_days)
    
    # Add date to features for evaluation
    X_eval = X.copy()
    X_eval['game_date'] = df['game_date']
    
    try:
        evaluation_results = evaluator.evaluate_model(
            optimized_model, X_eval, y, 'game_date'
        )
        
        logger.info("Cross-validation results:")
        logger.info(f"Average Train R²: {evaluation_results['train_r2']:.4f}")
        logger.info(f"Average Test R²: {evaluation_results['test_r2']:.4f}")
        logger.info(f"Average Test RMSE: {evaluation_results['test_rmse']:.2f}")
        logger.info(f"Average Test MAE: {evaluation_results['test_mae']:.2f}")
    except Exception as e:
        logger.error(f"Time-based evaluation failed: {e}")
        # Continue without time-based evaluation
    
    # Step 8: Fit final model on combined training and validation data
    final_model = optimized_model
    final_model.fit(X_train_full, y_train_full)
    
    # Step 9: Evaluate on test set
    test_pred = final_model.predict(X_test)
    test_mse = mean_squared_error(y_test, test_pred)
    test_r2 = r2_score(y_test, test_pred)
    test_mae = mean_absolute_error(y_test, test_pred)
    
    logger.info("Final model test set performance:")
    logger.info(f"Test MSE: {test_mse:.2f}")
    logger.info(f"Test RMSE: {np.sqrt(test_mse):.2f}")
    logger.info(f"Test R²: {test_r2:.4f}")
    logger.info(f"Test MAE: {test_mae:.2f}")
    
    # Step 10: Visualize feature importance - limited in fast mode
    try:
        top_n = 10 if fast_mode else 20
        visualize_feature_importance(final_model, features, top_n=top_n)
    except Exception as e:
        logger.error(f"Feature importance visualization failed: {e}")
    
    # Skip quarter analysis in fast mode
    if not fast_mode:
        try:
            analyze_performance_by_quarter(df, final_model, features)
        except Exception as e:
            logger.error(f"Quarter analysis failed: {e}")
    
    # Step 12: Save the enhanced model
    model_path = os.path.join(os.getcwd(), 'enhanced_score_prediction_model.pkl')
    joblib.dump(final_model, model_path)
    logger.info(f"Enhanced model saved to {model_path}")
    
    # Make the model available globally
    globals()['model'] = final_model
    
    overall_elapsed = time.time() - overall_start_time
    logger.info(f"Total model training time: {overall_elapsed:.2f} seconds")
    
    return final_model

def analyze_performance_by_quarter(df, model, features):
    """
    Analyze model performance by quarter.
    
    Args:
        df: Full DataFrame with all data
        model: Trained model
        features: List of feature names
    """
    logger.info("Analyzing performance by quarter")
    
    # Create a copy for analysis
    analysis_df = df.copy()
    
    # Define quarters
    quarters = range(1, 5)
    
    # Track results
    quarter_results = []
    
    # Split by time
    train_size = int(0.8 * len(df))
    test_df = df.iloc[train_size:]
    
    # Get true target
    y_true = test_df['home_score']
    
    # Test predictions for different quarters
    for quarter in quarters:
        # Create quarter-specific test data
        X_test_q = test_df[features].copy()
        
        # Zero out information from future quarters
        for q in range(quarter + 1, 5):
            q_col = f'home_q{q}'
            if q_col in X_test_q.columns:
                X_test_q[q_col] = 0
        
        # Zero out momentum features that wouldn't be available
        if quarter < 2:
            X_test_q['q1_to_q2_momentum'] = 0
            X_test_q['q2_to_q3_momentum'] = 0
            X_test_q['q3_to_q4_momentum'] = 0
            X_test_q['cumulative_momentum'] = 0
        elif quarter < 3:
            X_test_q['q2_to_q3_momentum'] = 0
            X_test_q['q3_to_q4_momentum'] = 0
            # Keep q1_to_q2_momentum and recalculate cumulative_momentum
            X_test_q['cumulative_momentum'] = X_test_q['q1_to_q2_momentum'] / 15.0
        elif quarter < 4:
            X_test_q['q3_to_q4_momentum'] = 0
            # Recalculate cumulative_momentum with just q1->q2 and q2->q3
            X_test_q['cumulative_momentum'] = (
                X_test_q['q1_to_q2_momentum'] * 0.2 + 
                X_test_q['q2_to_q3_momentum'] * 0.3
            ) / (0.2 + 0.3) / 15.0
        
        # Generate predictions
        y_pred_q = model.predict(X_test_q)
        
        # Calculate error metrics
        mse = mean_squared_error(y_true, y_pred_q)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_true, y_pred_q)
        r2 = r2_score(y_true, y_pred_q)
        
        # Save results
        quarter_results.append({
            'Quarter': quarter,
            'MSE': mse,
            'RMSE': rmse,
            'MAE': mae,
            'R²': r2
        })
    
    # Create DataFrame and display results
    quarter_df = pd.DataFrame(quarter_results)
    logger.info("\nPerformance by quarter:")
    logger.info(quarter_df.to_string(index=False))
    
    # Plot results
    plt.figure(figsize=(10, 6))
    plt.bar(quarter_df['Quarter'], quarter_df['RMSE'], color='salmon')
    plt.xlabel('Information Available Through Quarter')
    plt.ylabel('Root Mean Square Error')
    plt.title('Prediction Error by Available Quarter Information')
    plt.xticks([1, 2, 3, 4])
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
    
    plt.figure(figsize=(10, 6))
    plt.plot(quarter_df['Quarter'], quarter_df['R²'], 'o-', color='blue', linewidth=2)
    plt.xlabel('Information Available Through Quarter')
    plt.ylabel('R² Score')
    plt.title('Model Accuracy by Available Quarter Information')
    plt.xticks([1, 2, 3, 4])
    plt.ylim(0, 1)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Run the enhanced model training with fast mode enabled
if __name__ == "__main__":
    logger.info("Starting model training process")
    enhanced_model = train_enhanced_model(fast_mode=True)
    if enhanced_model is not None:
        logger.info("Model training completed successfully")
    else:
        logger.error("Model training failed")

In [ ]:
# Cell 18A – PredictionCache Implementation

import redis
import json
import time

class PredictionCache:
    def __init__(self, host='localhost', port=6379, db=0, ttl=86400):
        """
        Initialize the cache for prediction history using Redis.
        :param host: Redis server hostname.
        :param port: Redis server port.
        :param db: Redis database index.
        :param ttl: Time-to-live for cache entries in seconds (default is 1 day).
        """
        self.r = redis.Redis(host=host, port=port, db=db, decode_responses=True)
        self.ttl = ttl

    def cache_prediction(self, game_id, prediction_data):
        """
        Cache prediction data for a game.
        :param game_id: Unique identifier for the game.
        :param prediction_data: A dictionary or serializable data containing prediction details.
        """
        key = f"prediction:{game_id}:{int(time.time())}"
        self.r.set(key, json.dumps(prediction_data), ex=self.ttl)

    def get_cached_predictions(self, game_id):
        """
        Retrieve all cached predictions for a given game.
        :param game_id: Unique identifier for the game.
        :return: A list of prediction records.
        """
        pattern = f"prediction:{game_id}:*"
        keys = self.r.keys(pattern)
        predictions = [json.loads(self.r.get(key)) for key in keys]
        return predictions

    def prune_old_data(self):
        """
        Optionally implement additional pruning logic if necessary.
        Note: Redis will automatically remove keys past their TTL.
        """
        # This method can be extended if you need manual cleanup or logging.
        pass

    def export_cache(self, filename):
        """
        Export the current prediction cache to a JSON file.
        :param filename: The file path where to save the cache data.
        """
        keys = self.r.keys("prediction:*")
        data = {key: json.loads(self.r.get(key)) for key in keys}
        with open(filename, 'w') as f:
            json.dump(data, f)
    
    def import_cache(self, filename):
        """
        Import prediction cache data from a JSON file.
        :param filename: The file path to load the cache data from.
        """
        with open(filename, 'r') as f:
            data = json.load(f)
        for key, value in data.items():
            self.r.set(key, json.dumps(value), ex=self.ttl)

# Example usage:
# cache = PredictionCache()
# cache.cache_prediction("game_123", {"prediction": 110, "confidence": 0.87})
# print(cache.get_cached_predictions("game_123"))
# cache.export_cache("prediction_cache.json")
# cache.import_cache("prediction_cache.json")


In [ ]:
# Cell 19 - Visualize Feature Importance and Test Quarter-Specific Performance

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import os

print("Loading NBA game data...")

# Safely attempt to get an existing DataFrame from globals
working_df = None
for df_name in ['games_df', 'preprocessed_df', 'training_df', 'df']:
    if df_name in globals():
        working_df = globals()[df_name]
        print(f"Using existing {df_name} from previous cells")
        break

if working_df is None:
    # No existing DataFrame found, try to load from file
    data_path = "./data/nba_games.csv"  # Adjust path as needed
    if os.path.exists(data_path):
        working_df = pd.read_csv(data_path)
        print(f"Loaded data from {data_path}: {working_df.shape[0]} rows, {working_df.shape[1]} columns")
    else:
        # Create synthetic NBA game data for testing
        print("No data found in memory or file. Creating sample NBA game data for testing.")
        np.random.seed(42)
        n_samples = 1000
        working_df = pd.DataFrame({
            'game_id': range(1, n_samples + 1),
            'home_q1': np.random.randint(15, 35, n_samples),
            'home_q2': np.random.randint(15, 35, n_samples),
            'home_q3': np.random.randint(15, 35, n_samples),
            'home_q4': np.random.randint(15, 35, n_samples),
            'away_q1': np.random.randint(15, 35, n_samples),
            'away_q2': np.random.randint(15, 35, n_samples),
            'away_q3': np.random.randint(15, 35, n_samples),
            'away_q4': np.random.randint(15, 35, n_samples),
            'home_rest_days': np.random.randint(0, 5, n_samples),
            'away_rest_days': np.random.randint(0, 5, n_samples),
            'is_home_b2b': np.random.choice([0, 1], n_samples, p=[0.8, 0.2]),
            'is_away_b2b': np.random.choice([0, 1], n_samples, p=[0.8, 0.2]),
            'prev_matchup_diff': np.random.normal(0, 10, n_samples),
            # Add missing features that the model might expect
            'rolling_home_score': np.random.randint(80, 120, n_samples),
            'rolling_away_score': np.random.randint(80, 120, n_samples),
        })
        # Calculate additional features
        working_df['rest_advantage'] = working_df['home_rest_days'] - working_df['away_rest_days']
        working_df['home_total'] = working_df[['home_q1', 'home_q2', 'home_q3', 'home_q4']].sum(axis=1)
        working_df['away_total'] = working_df[['away_q1', 'away_q2', 'away_q3', 'away_q4']].sum(axis=1)
        working_df['final_score_diff'] = working_df['home_total'] - working_df['away_total']
        
        # Add momentum features
        working_df['q1_diff'] = working_df['home_q1'] - working_df['away_q1']
        working_df['q2_diff'] = working_df['home_q2'] - working_df['away_q2']
        working_df['q3_diff'] = working_df['home_q3'] - working_df['away_q3']
        working_df['q4_diff'] = working_df['home_q4'] - working_df['away_q4']
        working_df['q1_to_q2_momentum'] = working_df['q2_diff'] - working_df['q1_diff']
        working_df['q2_to_q3_momentum'] = working_df['q3_diff'] - working_df['q2_diff']
        working_df['q3_to_q4_momentum'] = working_df['q4_diff'] - working_df['q3_diff']
        working_df['cumulative_momentum'] = (working_df['q1_to_q2_momentum'] * 0.2 + 
                                             working_df['q2_to_q3_momentum'] * 0.3 + 
                                             working_df['q3_to_q4_momentum'] * 0.5)
        # Add game state features
        half_time_home = working_df['home_q1'] + working_df['home_q2']
        half_time_away = working_df['away_q1'] + working_df['away_q2']
        working_df['score_differential'] = half_time_home - half_time_away
        working_df['score_ratio'] = half_time_home / (half_time_home + half_time_away)
        print(f"Created sample data with {working_df.shape[0]} rows, {working_df.shape[1]} columns")

# Ensure we have the required columns for target calculation
# First, check if we can compute home_total and away_total if they don't exist
if 'home_total' not in working_df.columns and all(f'home_q{i}' in working_df.columns for i in range(1, 5)):
    working_df['home_total'] = working_df[['home_q1', 'home_q2', 'home_q3', 'home_q4']].sum(axis=1)
    print("Calculated home_total from quarter data")
    
if 'away_total' not in working_df.columns and all(f'away_q{i}' in working_df.columns for i in range(1, 5)):
    working_df['away_total'] = working_df[['away_q1', 'away_q2', 'away_q3', 'away_q4']].sum(axis=1)
    print("Calculated away_total from quarter data")

# Define target variable based on the NBA score prediction context
if 'final_score_diff' in working_df.columns:
    target_column = 'final_score_diff'
    print(f"Using existing column '{target_column}' as target")
elif 'score_differential' in working_df.columns:
    target_column = 'score_differential'
    print(f"Using existing column '{target_column}' as target")
elif 'home_total' in working_df.columns and 'away_total' in working_df.columns:
    working_df['final_score_diff'] = working_df['home_total'] - working_df['away_total']
    target_column = 'final_score_diff'
    print(f"Created and using '{target_column}' as target")
else:
    # As a last resort, if we have quarter data, create what we need
    if all(f'home_q{i}' in working_df.columns for i in range(1, 5)) and all(f'away_q{i}' in working_df.columns for i in range(1, 5)):
        working_df['home_total'] = working_df[['home_q1', 'home_q2', 'home_q3', 'home_q4']].sum(axis=1)
        working_df['away_total'] = working_df[['away_q1', 'away_q2', 'away_q3', 'away_q4']].sum(axis=1)
        working_df['final_score_diff'] = working_df['home_total'] - working_df['away_total']
        target_column = 'final_score_diff'
        print(f"Created and using '{target_column}' as target")
    else:
        # If we still can't create a target, show what columns we do have
        print("Available columns:", working_df.columns.tolist())
        raise KeyError("No valid target column found in the data and unable to create one from available columns.")

# Define features if not already defined
features_defined = 'features' in globals() and len(features) > 0

# Check if model exists and has feature_names_in_ attribute
model_has_features = 'model' in globals() and hasattr(model, 'feature_names_in_')

if model_has_features:
    print("Using features from the existing model")
    required_features = model.feature_names_in_.tolist()
    
    # Check if all required features exist in the DataFrame
    missing_features = [f for f in required_features if f not in working_df.columns]
    if missing_features:
        print(f"Warning: Missing features in data: {missing_features}")
        print("Creating missing features with default values")
        for feature in missing_features:
            # Use appropriate default values based on feature name
            if 'rolling' in feature:
                working_df[feature] = np.random.randint(80, 120, len(working_df))
            else:
                working_df[feature] = 0
    
    features = required_features
    print(f"Using {len(features)} features from the model")
    
elif not features_defined:
    # Use features relevant to NBA score prediction based on your project
    score_features = [col for col in working_df.columns if col.startswith('home_q') or col.startswith('away_q')]
    game_state_features = ['score_ratio', 'score_differential'] if 'score_ratio' in working_df.columns else []
    momentum_features = ['q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'] if 'q1_to_q2_momentum' in working_df.columns else []
    rest_features = ['home_rest_days', 'away_rest_days', 'rest_advantage', 'is_home_b2b', 'is_away_b2b'] if 'home_rest_days' in working_df.columns else []
    matchup_features = ['prev_matchup_diff'] if 'prev_matchup_diff' in working_df.columns else []
    rolling_features = ['rolling_home_score', 'rolling_away_score'] if 'rolling_home_score' in working_df.columns else []
    
    # Combine all feature groups
    features = score_features + game_state_features + momentum_features + rest_features + matchup_features + rolling_features
    # Ensure features exist in the DataFrame
    features = [f for f in features if f in working_df.columns]
    
    if not features:
        raise ValueError("No valid features found in the DataFrame. Check your column names.")
    
    print(f"Using {len(features)} automatically selected features for analysis.")

# Create train/test split if not already done
if 'X_test' not in globals() or 'y_test' not in globals():
    print("Creating train/test split...")
    X = working_df[features]
    y = working_df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # If no model exists yet, train one
    if 'model' not in globals():
        from sklearn.ensemble import GradientBoostingRegressor
        print("Training GradientBoostingRegressor model...")
        model = GradientBoostingRegressor(random_state=42)
        model.fit(X_train, y_train)
        
        # Display initial model performance
        train_preds = model.predict(X_train)
        test_preds = model.predict(X_test)
        train_mse = mean_squared_error(y_train, train_preds)
        test_mse = mean_squared_error(y_test, test_preds)
        print(f"Model trained. Train MSE: {train_mse:.4f}, Test MSE: {test_mse:.4f}, Test RMSE: {np.sqrt(test_mse):.4f}")

# Visualize feature importance if the model provides it
if 'model' in globals() and hasattr(model, 'feature_importances_'):
    # Diagnose feature length mismatch
    feature_len = len(features)
    importance_len = len(model.feature_importances_)
    print(f"Features length: {feature_len}, Feature importances length: {importance_len}")
    
    # Handle length mismatch
    if feature_len != importance_len:
        if hasattr(model, 'feature_names_in_'):
            print("Using model's feature_names_in_ attribute")
            features = model.feature_names_in_.tolist()
        elif feature_len > importance_len:
            print("Truncating features list to match feature_importances_")
            features = features[:importance_len]
        else:
            print("Adding placeholder features to match feature_importances_")
            features = features + [f"Unknown_{i}" for i in range(feature_len, importance_len)]
    
    # Create DataFrame column by column to avoid length issues
    feature_importances = pd.DataFrame()
    feature_importances['Feature'] = features
    feature_importances['Importance'] = model.feature_importances_
    feature_importances = feature_importances.sort_values('Importance', ascending=False)
    
    plt.figure(figsize=(12, 8))
    sns.barplot(x='Importance', y='Feature', data=feature_importances.head(15))
    plt.title('Top 15 Features by Importance in Enhanced In-Game Model')
    plt.tight_layout()
    plt.show()
    
    # Group features by type for a pie chart visualization
    feature_groups = {
        'Quarter Scores': [col for col in features if col.startswith('home_q') or col.startswith('away_q')],
        'Game State': [col for col in features if col in ['score_ratio', 'score_differential']],
        'Matchup History': [col for col in features if col in ['prev_matchup_diff']],
        'Rest': [col for col in features if col in ['home_rest_days', 'away_rest_days', 'rest_advantage', 'is_home_b2b', 'is_away_b2b']],
        'Momentum': [col for col in features if col in ['q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum']],
        'Rolling Stats': [col for col in features if 'rolling' in col]
    }
    
    group_importance = {}
    for group, feats in feature_groups.items():
        valid_feats = [f for f in feats if f in features]
        if valid_feats:
            group_importance[group] = sum(
                model.feature_importances_[features.index(f)] for f in valid_feats
            )
    
    if group_importance:
        plt.figure(figsize=(10, 8))
        plt.pie(
            group_importance.values(), 
            labels=group_importance.keys(), 
            autopct='%1.1f%%', 
            shadow=True, 
            startangle=140
        )
        plt.axis('equal')
        plt.title('Feature Group Contribution to In-Game Predictions')
        plt.tight_layout()
        plt.show()

# Analyze model performance by quarter
print("\nAnalyzing performance by quarter...")

quarter_analysis = []
for current_quarter in range(1, 5):
    # Create a copy of the test data for this quarter analysis
    quarter_X_test = X_test.copy()
    
    # Get the expected feature names from the model
    if hasattr(model, 'feature_names_in_'):
        required_features = model.feature_names_in_
        
        # Create a new DataFrame with all the required features
        new_quarter_X_test = pd.DataFrame()
        for feature in required_features:
            if feature in quarter_X_test.columns:
                new_quarter_X_test[feature] = quarter_X_test[feature]
            else:
                # Feature is missing, add it with zeros
                print(f"Adding missing feature '{feature}' to quarter_X_test")
                new_quarter_X_test[feature] = 0
                
        quarter_X_test = new_quarter_X_test
    
    # Zero out data for future quarters
    for q in range(current_quarter + 1, 5):
        for col_prefix in ['home_q', 'away_q']:
            col_name = f"{col_prefix}{q}"
            if col_name in quarter_X_test.columns:
                quarter_X_test[col_name] = 0
    
    # Zero out momentum features not available yet
    if 'q1_to_q2_momentum' in quarter_X_test.columns and current_quarter < 2:
        quarter_X_test['q1_to_q2_momentum'] = 0
    if 'q2_to_q3_momentum' in quarter_X_test.columns and current_quarter < 3:
        quarter_X_test['q2_to_q3_momentum'] = 0
    if 'q3_to_q4_momentum' in quarter_X_test.columns and current_quarter < 4:
        quarter_X_test['q3_to_q4_momentum'] = 0
    
    # Adjust cumulative momentum based on available quarters
    if 'cumulative_momentum' in quarter_X_test.columns:
        if current_quarter < 2:
            quarter_X_test['cumulative_momentum'] = 0
        elif current_quarter < 3:
            if 'q1_to_q2_momentum' in quarter_X_test.columns:
                quarter_X_test['cumulative_momentum'] = quarter_X_test['q1_to_q2_momentum']
        elif current_quarter < 4:
            if 'q1_to_q2_momentum' in quarter_X_test.columns and 'q2_to_q3_momentum' in quarter_X_test.columns:
                quarter_X_test['cumulative_momentum'] = (
                    quarter_X_test['q1_to_q2_momentum'] * 0.2 + quarter_X_test['q2_to_q3_momentum'] * 0.3
                ) / 0.5
    
    # Ensure all feature columns are in the same order as expected by the model
    if hasattr(model, 'feature_names_in_'):
        quarter_X_test = quarter_X_test[model.feature_names_in_]
    
    # Generate predictions for the current quarter scenario
    try:
        quarter_preds = model.predict(quarter_X_test)
        quarter_mse = mean_squared_error(y_test, quarter_preds)
        quarter_mae = np.mean(np.abs(y_test - quarter_preds))
        
        quarter_analysis.append({
            'Quarter': current_quarter,
            'MSE': quarter_mse,
            'MAE': quarter_mae,
            'RMSE': np.sqrt(quarter_mse)
        })
    except Exception as e:
        print(f"Error predicting for quarter {current_quarter}: {e}")
        # Print detailed debugging information
        if hasattr(model, 'feature_names_in_'):
            print("Model expects these features:", model.feature_names_in_)
        print("Features in quarter_X_test:", quarter_X_test.columns.tolist())
        print("Shape of quarter_X_test:", quarter_X_test.shape)
        missing = [f for f in model.feature_names_in_ if f not in quarter_X_test.columns]
        if missing:
            print("Missing features:", missing)

if quarter_analysis:
    quarter_df = pd.DataFrame(quarter_analysis)
    print(quarter_df)
    
    plt.figure(figsize=(10, 6))
    plt.bar(quarter_df['Quarter'], quarter_df['RMSE'], color='salmon')
    plt.xlabel('Information Available Through Quarter')
    plt.ylabel('Root Mean Square Error')
    plt.title('Prediction Error by Available Quarter Information')
    plt.xticks([1, 2, 3, 4])
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("No quarter analysis data was generated due to errors.")

In [ ]:
# Cell 19A – Anomaly Detection & Alerts Implementation

import smtplib
from email.mime.text import MIMEText
import logging

class AnomalyDetector:
    def __init__(self, swing_threshold=10, confidence_threshold=0.5):
        """
        Initialize the AnomalyDetector.
        :param swing_threshold: The minimum absolute difference between consecutive predictions that is considered anomalous.
        :param confidence_threshold: The minimum acceptable prediction confidence value.
        """
        self.swing_threshold = swing_threshold
        self.confidence_threshold = confidence_threshold
        self.previous_prediction = None

    def detect(self, current_prediction, current_confidence):
        """
        Detect anomalies based on unusual prediction swings or low confidence.
        :param current_prediction: Current prediction value (numeric).
        :param current_confidence: Current prediction confidence (numeric between 0 and 1).
        :return: List of alert messages (empty if no anomaly detected).
        """
        alerts = []
        if self.previous_prediction is not None:
            swing = abs(current_prediction - self.previous_prediction)
            if swing > self.swing_threshold:
                alerts.append(f"Unusual prediction swing detected: {swing} (Threshold: {self.swing_threshold})")
        if current_confidence < self.confidence_threshold:
            alerts.append(f"Low prediction confidence detected: {current_confidence} (Threshold: {self.confidence_threshold})")
        
        # Update the previous prediction for next detection
        self.previous_prediction = current_prediction
        return alerts

class Notifier:
    def __init__(self, smtp_server, smtp_port, username, password, from_addr, to_addrs):
        """
        Initialize the Notifier with SMTP settings.
        :param smtp_server: SMTP server address.
        :param smtp_port: SMTP server port.
        :param username: SMTP username.
        :param password: SMTP password.
        :param from_addr: Email address to send from.
        :param to_addrs: List of email addresses to notify.
        """
        self.smtp_server = smtp_server
        self.smtp_port = smtp_port
        self.username = username
        self.password = password
        self.from_addr = from_addr
        self.to_addrs = to_addrs

    def send_alert(self, subject, message):
        """
        Send an alert email.
        :param subject: Subject line for the email.
        :param message: Body message for the alert.
        """
        msg = MIMEText(message)
        msg['Subject'] = subject
        msg['From'] = self.from_addr
        msg['To'] = ", ".join(self.to_addrs)
        try:
            with smtplib.SMTP(self.smtp_server, self.smtp_port) as server:
                server.starttls()
                server.login(self.username, self.password)
                server.sendmail(self.from_addr, self.to_addrs, msg.as_string())
            logging.info("Alert email sent successfully.")
        except Exception as e:
            logging.error(f"Failed to send alert email: {e}")

# Example integration within your live monitoring loop:
if __name__ == "__main__":
    # Setup logging for anomaly detection
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
    
    # Instantiate anomaly detector and notifier
    anomaly_detector = AnomalyDetector(swing_threshold=10, confidence_threshold=0.5)
    notifier = Notifier(
        smtp_server="smtp.example.com",  # Update with your SMTP server
        smtp_port=587,                   # Update with your SMTP port
        username="user@example.com",     # Update with your SMTP username
        password="password",             # Update with your SMTP password
        from_addr="user@example.com",    # Update with your from email address
        to_addrs=["alert@example.com"]   # Update with one or more recipient emails
    )
    
    # Simulated predictions and confidence values (replace with real-time data in production)
    simulated_predictions = [100, 105, 120, 115, 130]
    simulated_confidences = [0.9, 0.8, 0.4, 0.85, 0.95]
    
    for pred, conf in zip(simulated_predictions, simulated_confidences):
        alerts = anomaly_detector.detect(pred, conf)
        if alerts:
            alert_message = "\n".join(alerts)
            logging.warning(f"Anomaly Detected: {alert_message}")
            notifier.send_alert("Critical Anomaly Detected", alert_message)
        else:
            logging.info("No anomalies detected.")

In [ ]:
# Cell 20 -- Fetch scheduled games

import pandas as pd
import pytz
from datetime import datetime, timedelta
import os
import json
import random

# Initialize Supabase client
try:
    # Try to import from the project structure
    try:
        from backend.caching.supabase_client import supabase  # type: ignore
        print("Imported supabase client from project structure")
    except ImportError:
        # If that fails, try to import from the current directory
        try:
            from supabase_client import supabase  # type: ignore
            print("Imported supabase client from current directory")
        except ImportError:
            # If both imports fail, check if credentials exist as environment variables
            from supabase import create_client  # Make sure the 'supabase' package is installed
            supabase_url = os.environ.get("SUPABASE_URL")
            supabase_key = os.environ.get("SUPABASE_KEY")
            
            if supabase_url and supabase_key:
                supabase = create_client(supabase_url, supabase_key)
                print("Created supabase client from environment variables")
            else:
                # Look for a config file
                config_paths = [
                    "./config.json",
                    "./backend/config.json",
                    "../backend/config.json"
                ]
                
                config_found = False
                for path in config_paths:
                    if os.path.exists(path):
                        with open(path, 'r') as f:
                            config = json.load(f)
                            if 'SUPABASE_URL' in config and 'SUPABASE_KEY' in config:
                                supabase = create_client(config['SUPABASE_URL'], config['SUPABASE_KEY'])
                                print(f"Created supabase client from config file: {path}")
                                config_found = True
                                break
                
                if not config_found:
                    # Create a mock supabase client for testing
                    print("WARNING: No Supabase credentials found. Creating mock client for testing.")
                    
                    class MockSupabase:
                        def table(self, table_name):
                            self.current_table = table_name
                            return self
                        
                        def select(self, *args):
                            self.select_args = args
                            return self
                        
                        def eq(self, column, value):
                            self.eq_column = column
                            self.eq_value = value
                            return self
                        
                        def execute(self):
                            if self.current_table == "nba_game_schedule" and self.eq_column == "game_date":
                                teams = [
                                    "LAL", "LAC", "GSW", "PHX", "SAC", 
                                    "DEN", "UTA", "POR", "MEM", "DAL", 
                                    "HOU", "SAS", "OKC", "MIN", "NOP",
                                    "MIL", "CHI", "CLE", "DET", "IND", 
                                    "BOS", "NYK", "BKN", "PHI", "TOR",
                                    "MIA", "ORL", "ATL", "CHA", "WAS"
                                ]
                                seed = int(self.eq_value.replace('-', ''))
                                random.seed(seed)
                                num_games = random.randint(3, 5)
                                games = []
                                random.shuffle(teams)
                                for i in range(num_games):
                                    home_team = teams[i*2]
                                    away_team = teams[i*2+1]
                                    hour = random.randint(16, 20)
                                    minute = random.choice([0, 30])
                                    start_time = f"{hour:02d}:{minute:02d}:00"
                                    games.append({
                                        "id": f"mock-{self.eq_value}-{i+1}",
                                        "game_date": self.eq_value,
                                        "start_time": start_time,
                                        "home_team": home_team,
                                        "away_team": away_team,
                                        "status": "scheduled",
                                        "arena": f"{home_team} Arena",
                                        "city": "City",
                                        "state": "State",
                                        "country": "USA"
                                    })
                                return type('obj', (object,), {'data': games})
                            return type('obj', (object,), {'data': []})
                    
                    supabase = MockSupabase()
except Exception as e:
    print(f"Error initializing Supabase client: {e}")
    print("Creating mock Supabase client as fallback")
    
    class SimpleSupabase:
        def table(self, _): return self
        def select(self, *_): return self
        def eq(self, *_): return self
        def execute(self): return type('obj', (object,), {'data': []})
    
    supabase = SimpleSupabase()

def fetch_scheduled_games(date_str=None, timezone='America/Los_Angeles'):
    """
    Fetch scheduled games from the database for a specific date.
    
    Args:
        date_str: Date string in YYYY-MM-DD format (defaults to today in specified timezone)
        timezone: Timezone name (default: 'America/Los_Angeles' for Pacific Time)
        
    Returns:
        DataFrame with scheduled games or empty DataFrame if none found
    """
    try:
        if date_str is None:
            tz = pytz.timezone(timezone)
            date_str = datetime.now(tz).strftime('%Y-%m-%d')
        
        print(f"Fetching scheduled games for {date_str} ({timezone})")
        response = supabase.table("nba_game_schedule").select("*").eq("game_date", date_str).execute()
        
        if response and hasattr(response, 'data') and response.data:
            df = pd.DataFrame(response.data)
            print(f"Found {len(df)} scheduled games")
            return df
        else:
            print(f"No scheduled games found for {date_str}")
            return pd.DataFrame()
            
    except Exception as e:
        print(f"Error fetching scheduled games: {e}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame()

# Test fetching today's schedule
today_schedule = fetch_scheduled_games()
from IPython.display import display
display(today_schedule)

# Test fetching tomorrow's schedule
tz = pytz.timezone('America/Los_Angeles')
tomorrow = (datetime.now(tz) + timedelta(days=1)).strftime('%Y-%m-%d')
tomorrow_schedule = fetch_scheduled_games(tomorrow)
display(tomorrow_schedule)


In [ ]:
# Cell 20A - Compute momentum features for prediction

def compute_momentum_features(df):
    """
    Compute momentum-based features for prediction.
    
    Args:
        df: DataFrame with game data including quarter scores
        
    Returns:
        DataFrame with momentum features added
    """
    import numpy as np
    
    try:
        # Make a copy to avoid modifying the original
        features_df = df.copy()
        
        # Initialize momentum columns
        features_df['q1_to_q2_momentum'] = 0
        features_df['q2_to_q3_momentum'] = 0
        features_df['q3_to_q4_momentum'] = 0
        features_df['cumulative_momentum'] = 0
        
        # Weights for quarter transitions (more weight to recent quarters)
        weights = [0.2, 0.3, 0.5]
        
        for idx, row in features_df.iterrows():
            try:
                # Get current quarter
                current_quarter = int(row.get('current_quarter', 0))
                
                # Extract quarter scores with error handling
                home_q1 = float(row.get('home_q1', 0) or 0)
                home_q2 = float(row.get('home_q2', 0) or 0)
                home_q3 = float(row.get('home_q3', 0) or 0)
                home_q4 = float(row.get('home_q4', 0) or 0)
                
                away_q1 = float(row.get('away_q1', 0) or 0)
                away_q2 = float(row.get('away_q2', 0) or 0)
                away_q3 = float(row.get('away_q3', 0) or 0)
                away_q4 = float(row.get('away_q4', 0) or 0)
                
                # Calculate quarter-to-quarter momentum
                if current_quarter >= 2:
                    features_df.at[idx, 'q1_to_q2_momentum'] = (home_q2 - home_q1) - (away_q2 - away_q1)
                
                if current_quarter >= 3:
                    features_df.at[idx, 'q2_to_q3_momentum'] = (home_q3 - home_q2) - (away_q3 - away_q2)
                    
                if current_quarter >= 4:
                    features_df.at[idx, 'q3_to_q4_momentum'] = (home_q4 - home_q3) - (away_q4 - away_q3)
                
                # Calculate cumulative momentum based on available quarters
                if current_quarter == 2:
                    cum_momentum = features_df.at[idx, 'q1_to_q2_momentum']
                elif current_quarter == 3:
                    cum_momentum = (
                        features_df.at[idx, 'q1_to_q2_momentum'] * weights[0] + 
                        features_df.at[idx, 'q2_to_q3_momentum'] * weights[1]
                    ) / (weights[0] + weights[1])
                elif current_quarter >= 4:
                    cum_momentum = (
                        features_df.at[idx, 'q1_to_q2_momentum'] * weights[0] + 
                        features_df.at[idx, 'q2_to_q3_momentum'] * weights[1] + 
                        features_df.at[idx, 'q3_to_q4_momentum'] * weights[2]
                    ) / sum(weights)
                else:
                    cum_momentum = 0
                
                # Normalize momentum to range [-1, 1]
                features_df.at[idx, 'cumulative_momentum'] = np.clip(cum_momentum / 15.0, -1.0, 1.0)
                
            except Exception as e:
                print(f"Error processing row {idx}: {e}")
                # Keep default values for this row
        
        return features_df
        
    except Exception as e:
        print(f"Error computing momentum features: {e}")
        import traceback
        traceback.print_exc()
        return df  # Return original dataframe if processing fails

In [ ]:
# Cell 21: Enhanced feature engineering
def compute_momentum_features(df):
    """Compute momentum-based features for prediction."""
    features_df = df.copy()
    
    # Add momentum columns
    features_df['q1_to_q2_momentum'] = 0
    features_df['q2_to_q3_momentum'] = 0
    features_df['q3_to_q4_momentum'] = 0
    features_df['cumulative_momentum'] = 0
    
    for idx, row in features_df.iterrows():
        current_quarter = int(row.get('current_quarter', 0))
        
        # Quarter scores
        home_q1 = float(row.get('home_q1', 0))
        home_q2 = float(row.get('home_q2', 0))
        home_q3 = float(row.get('home_q3', 0))
        home_q4 = float(row.get('home_q4', 0))
        
        away_q1 = float(row.get('away_q1', 0))
        away_q2 = float(row.get('away_q2', 0))
        away_q3 = float(row.get('away_q3', 0))
        away_q4 = float(row.get('away_q4', 0))
        
        # Calculate quarter-to-quarter momentum
        if current_quarter >= 2:
            features_df.at[idx, 'q1_to_q2_momentum'] = (home_q2 - home_q1) - (away_q2 - away_q1)
            
        if current_quarter >= 3:
            features_df.at[idx, 'q2_to_q3_momentum'] = (home_q3 - home_q2) - (away_q3 - away_q2)
            
        if current_quarter >= 4:
            features_df.at[idx, 'q3_to_q4_momentum'] = (home_q4 - home_q3) - (away_q4 - away_q3)
        
        # Weighted momentum calculation
        weights = [0.2, 0.3, 0.5]  # Weight recent quarters more heavily
        
        if current_quarter == 2:
            features_df.at[idx, 'cumulative_momentum'] = features_df.at[idx, 'q1_to_q2_momentum']
        elif current_quarter == 3:
            features_df.at[idx, 'cumulative_momentum'] = (
                features_df.at[idx, 'q1_to_q2_momentum'] * weights[0] + 
                features_df.at[idx, 'q2_to_q3_momentum'] * weights[1]
            ) / (weights[0] + weights[1])
        elif current_quarter >= 4:
            features_df.at[idx, 'cumulative_momentum'] = (
                features_df.at[idx, 'q1_to_q2_momentum'] * weights[0] + 
                features_df.at[idx, 'q2_to_q3_momentum'] * weights[1] + 
                features_df.at[idx, 'q3_to_q4_momentum'] * weights[2]
            ) / sum(weights)
        
        # Normalize momentum to range [-1, 1]
        if features_df.at[idx, 'cumulative_momentum'] != 0:
            max_momentum = 15.0  # Typical max quarter differential
            features_df.at[idx, 'cumulative_momentum'] = max(min(
                features_df.at[idx, 'cumulative_momentum'] / max_momentum, 1.0
            ), -1.0)
    
    return features_df

In [ ]:
# Cell 22 - Fetch recent games for testing

def fetch_recent_games_for_testing(limit=5, days_lookback=5, simulate_live=True):
    """
    Find recent completed games to use for testing the prediction model.
    
    Args:
        limit: Maximum number of games to return
        days_lookback: Number of days to look back for recent games
        simulate_live: Whether to simulate games as in-progress 
        
    Returns:
        DataFrame with recent games (optionally simulated as in-progress)
    """
    try:
        # Look back up to specified days for recent games
        dates_to_try = []
        today = datetime.now()
        
        for i in range(1, days_lookback + 1):
            date = (today - timedelta(days=i)).strftime('%Y-%m-%d')
            dates_to_try.append(date)
        
        print(f"Searching for recent games in the last {days_lookback} days...")
        
        # Try each date until we find games
        for test_date in dates_to_try:
            response = supabase.table("nba_historical_game_stats")\
                .select("*")\
                .eq("game_date", test_date)\
                .limit(limit).execute()
            
            if response.data:
                print(f"Found {len(response.data)} historical games from {test_date}")
                
                # Convert to DataFrame
                historical_df = pd.DataFrame(response.data)
                
                if not simulate_live:
                    # Return historical games as-is
                    return historical_df
                
                # Simulate as in-progress games
                import random
                live_games = []
                
                for _, game in historical_df.iterrows():
                    # Randomly select a quarter (1-4) for simulation
                    sim_quarter = random.randint(1, 4)
                    
                    # Create simulated live game with data up to selected quarter
                    sim_game = {
                        'game_id': game['game_id'],
                        'home_team': game['home_team'],
                        'away_team': game['away_team'],
                        'game_date': game['game_date'],
                        'current_quarter': sim_quarter,
                        'simulated': True,  # Flag as simulated
                        'time_remaining': random.randint(1, 12),  # Random minutes remaining
                        'verified': True  # Mark as verified since from historical data
                    }
                    
                    # Add quarter scores up to simulated quarter
                    for q in range(1, 5):
                        sim_game[f'home_q{q}'] = game.get(f'home_q{q}', 0) if q <= sim_quarter else 0
                        sim_game[f'away_q{q}'] = game.get(f'away_q{q}', 0) if q <= sim_quarter else 0
                    
                    # Calculate current score based on visible quarters
                    sim_game['home_score'] = sum(
                        [sim_game.get(f'home_q{q}', 0) for q in range(1, sim_quarter+1)]
                    )
                    sim_game['away_score'] = sum(
                        [sim_game.get(f'away_q{q}', 0) for q in range(1, sim_quarter+1)]
                    )
                    
                    # Store actual final scores for validation
                    sim_game['actual_home_final'] = game['home_score']
                    sim_game['actual_away_final'] = game['away_score']
                    
                    live_games.append(sim_game)
                
                simulated_df = pd.DataFrame(live_games)
                print(f"Created {len(simulated_df)} simulated live games at various quarters")
                return simulated_df
                
        print("No recent games found for testing")
        return pd.DataFrame()
        
    except Exception as e:
        print(f"Error finding historical games: {e}")
        traceback.print_exc()
        return pd.DataFrame()

In [ ]:
# Cell 23 - Load trained score prediction model with versioning + validation.

def load_model(model_path=None, model_version=None, validate=True):
    """
    Load the trained score prediction model with versioning and validation.
    
    Args:
        model_path: Custom path to the model file (overrides defaults)
        model_version: Expected model version (if None, accepts any version)
        validate: Whether to validate the model after loading
        
    Returns:
        Loaded model or None if loading fails
    """
    try:
        import joblib
        import os
        import hashlib
        
        # Model loading search strategy:
        # 1. Use provided model_path if specified
        # 2. Try config.MODEL_PATH from configuration
        # 3. Look in standard locations (./models/, current directory)
        # 4. Return None if no model found
        
        search_paths = []
        
        # 1. Add provided path if specified
        if model_path:
            search_paths.append(model_path)
        
        # 2. Add path from config if available
        try:
            import config
            if hasattr(config, 'MODEL_PATH'):
                search_paths.append(config.MODEL_PATH)
        except ImportError:
            print("Config module not found, skipping config.MODEL_PATH")
        
        # 3. Add standard locations
        search_paths.extend([
            os.path.join(os.getcwd(), 'models', 'score_prediction_model.pkl'),
            os.path.join(os.getcwd(), 'models', 'enhanced_score_prediction_model.pkl'),
            os.path.join(os.getcwd(), 'score_prediction_model.pkl'),
            'score_prediction_model.pkl',
            'enhanced_xgb_model.pkl'
        ])
        
        # Search for model in paths
        for path in search_paths:
            if os.path.exists(path):
                print(f"Found model at {path}")
                model = joblib.load(path)
                
                # Check model version if specified
                if model_version is not None:
                    if hasattr(model, 'version') and model.version != model_version:
                        print(f"Warning: Model version mismatch. Expected {model_version}, found {model.version}")
                        continue
                
                # Validate model
                if validate and not validate_model(model, path):
                    print(f"Model validation failed for {path}")
                    continue
                    
                print(f"Successfully loaded model from {path}")
                
                # Add path attribute to model for reference
                model.model_path = path
                
                # Generate model hash for tracking
                with open(path, 'rb') as f:
                    model_bytes = f.read()
                    model_hash = hashlib.md5(model_bytes).hexdigest()
                model.model_hash = model_hash
                
                return model
        
        print("No valid model found in any of the search paths")
        return None
        
    except Exception as e:
        print(f"Error loading model: {e}")
        traceback.print_exc()
        return None

def validate_model(model, model_path):
    """
    Validate the model to ensure it has expected properties and methods.
    
    Args:
        model: The model to validate
        model_path: Path the model was loaded from
        
    Returns:
        bool: True if model passes validation, False otherwise
    """
    try:
        # Check if model has predict method
        if not hasattr(model, 'predict') or not callable(model.predict):
            print("Model missing predict method")
            return False
            
        # Check if model has feature names (if applicable)
        if hasattr(model, 'feature_names_in_'):
            # Model has explicit feature names, which is good
            feature_count = len(model.feature_names_in_)
            print(f"Model expects {feature_count} features: {model.feature_names_in_}")
        elif hasattr(model, 'feature_importances_'):
            # Model has feature importances but not names
            feature_count = len(model.feature_importances_)
            print(f"Model uses {feature_count} features (names unknown)")
        else:
            # Can't determine feature count
            print("Warning: Cannot determine feature count for model")
        
        # Check model type
        model_type = type(model).__name__
        print(f"Model type: {model_type}")
        
        # Model-specific validation
        if 'GradientBoosting' in model_type:
            if not hasattr(model, 'n_estimators'):
                print("GradientBoostingRegressor missing n_estimators")
                return False
            print(f"GradientBoostingRegressor with {model.n_estimators} estimators")
        elif 'Ridge' in model_type:
            if not hasattr(model, 'alpha'):
                print("RidgeRegressor missing alpha parameter")
                return False
            print(f"RidgeRegressor with alpha={model.alpha}")
        
        # Add model metadata
        model.validation_date = datetime.now().isoformat()
        model.model_type = model_type
        
        return True
        
    except Exception as e:
        print(f"Error validating model: {e}")
        return False

# Test loading the model
model = load_model()
if model:
    print(f"Loaded {type(model).__name__} model with hash: {model.model_hash}")
else:
    print("Failed to load any model")

In [ ]:
# Cell 24 -- Get rolling scoring averages for all teams

def get_team_rolling_averages(days_lookback=60, cache_key=None, refresh_cache=False):
    """
    Get rolling scoring averages for all teams.
    
    Args:
        days_lookback: Number of days to look back for statistics
        cache_key: Optional cache key for retrieving/storing results
        refresh_cache: Whether to force refresh the cache
        
    Returns:
        Dict with team averages
    """
    # Use caching to improve performance
    if cache_key and not refresh_cache:
        if cache_key in globals():
            print(f"Using cached team averages ({len(globals()[cache_key])} teams)")
            return globals()[cache_key]
    
    try:
        # Calculate threshold date
        threshold_date = (datetime.now() - timedelta(days=days_lookback)).strftime('%Y-%m-%d')
        print(f"Fetching team averages from {threshold_date} to present")
        
        # Fetch recent games for calculating averages
        response = supabase.table("nba_historical_game_stats").select("*").gte("game_date", threshold_date).execute()
        
        if not response.data:
            print("No historical data found for team averages")
            return {}
        
        # Calculate team averages
        team_avgs = {}
        df = pd.DataFrame(response.data)
        
        # Get unique teams
        home_teams = set(df['home_team'].unique())
        away_teams = set(df['away_team'].unique())
        all_teams = home_teams.union(away_teams)
        
        print(f"Calculating averages for {len(all_teams)} teams")
        
        # Constants for score calculations
        DEFAULT_AVG_SCORE = 105.0  # NBA league average if no data available
        MIN_GAMES_THRESHOLD = 3    # Minimum games needed for reliable average
        
        for team in all_teams:
            # Get home and away games
            home_games = df[df['home_team'] == team]['home_score']
            away_games = df[df['away_team'] == team]['away_score']
            
            # Combine and calculate average
            all_scores = pd.concat([home_games, away_games])
            if len(all_scores) >= MIN_GAMES_THRESHOLD:
                team_avgs[team] = all_scores.mean()
                
                # Calculate offensive and defensive ratings if needed later
                home_allowed = df[df['home_team'] == team]['away_score'].mean() if len(df[df['home_team'] == team]) > 0 else DEFAULT_AVG_SCORE
                away_allowed = df[df['away_team'] == team]['home_score'].mean() if len(df[df['away_team'] == team]) > 0 else DEFAULT_AVG_SCORE
                
                # Store these as additional metrics
                team_avgs[f"{team}_defensive"] = (home_allowed + away_allowed) / 2
            else:
                print(f"Warning: Insufficient data for {team}, using default average")
                team_avgs[team] = DEFAULT_AVG_SCORE
        
        # Cache results if requested
        if cache_key:
            globals()[cache_key] = team_avgs
            print(f"Cached team averages under key '{cache_key}'")
        
        return team_avgs
    
    except Exception as e:
        print(f"Error getting team averages: {e}")
        traceback.print_exc()
        return {}

def get_previous_matchup_diff(home_team, away_team, max_lookback=5):
    """
    Gets the point differential from previous matchups between two teams.
    
    Args:
        home_team: Home team name
        away_team: Away team name
        max_lookback: Maximum number of previous matchups to consider
        
    Returns:
        float: Average point differential from home team perspective
    """
    try:
        print(f"Looking for previous matchups between {home_team} and {away_team}")
        
        # Use separate queries for home and away configurations for reliable results
        response_home = supabase.table("nba_historical_game_stats").select("*")\
            .eq("home_team", home_team)\
            .eq("away_team", away_team)\
            .order('game_date', desc=True)\
            .limit(max_lookback).execute()
            
        response_away = supabase.table("nba_historical_game_stats").select("*")\
            .eq("home_team", away_team)\
            .eq("away_team", home_team)\
            .order('game_date', desc=True)\
            .limit(max_lookback).execute()
        
        # Combine results
        home_matchups = response_home.data if response_home.data else []
        away_matchups = response_away.data if response_away.data else []
        matchups = home_matchups + away_matchups
        
        # Sort by date (most recent first)
        if matchups:
            matchups.sort(key=lambda x: x['game_date'], reverse=True)
            matchups = matchups[:max_lookback]
            
            print(f"Found {len(matchups)} previous matchups")
        else:
            print("No previous matchups found")
            return 0
        
        # Calculate point differential from home team perspective
        differentials = []
        for game in matchups:
            if game['home_team'] == home_team and game['away_team'] == away_team:
                # Home team was home in this matchup
                diff = game['home_score'] - game['away_score']
            elif game['home_team'] == away_team and game['away_team'] == home_team:
                # Home team was away in this matchup (reverse diff)
                diff = game['away_score'] - game['home_score']
            else:
                continue
                
            differentials.append(diff)
            print(f"  {game['game_date']}: Differential {diff:+.1f} pts")
        
        if not differentials:
            return 0
            
        # Return average differential, capped to prevent extreme values
        avg_diff = sum(differentials) / len(differentials)
        # Cap extreme values for prediction stability
        MAX_DIFF = 15.0  # Maximum historical differential to consider
        capped_diff = max(min(avg_diff, MAX_DIFF), -MAX_DIFF)
        
        print(f"Average point differential: {avg_diff:.2f} (capped to {capped_diff:.2f})")
        return capped_diff
        
    except Exception as e:
        print(f"Error getting previous matchups: {e}")
        traceback.print_exc()
        return 0

def prepare_features(games_df, model=None):
    """
    Prepare features for prediction based on model type.
    
    Args:
        games_df: DataFrame with game data
        model: Prediction model (optional, used to determine feature requirements)
        
    Returns:
        DataFrame with features prepared for prediction
    """
    if games_df.empty:
        print("No games available for feature preparation")
        return pd.DataFrame()
    
    try:
        print(f"Preparing features for {len(games_df)} games")
        
        # Determine required features based on model
        is_enhanced_model = False
        expected_features = []
        
        if model is not None and hasattr(model, 'feature_importances_'):
            feature_count = len(model.feature_importances_)
            is_enhanced_model = (feature_count > 8)
            
            # Check if model has explicit feature names
            if hasattr(model, 'feature_names_in_'):
                expected_features = list(model.feature_names_in_)
                print(f"Using explicit model features: {expected_features}")
            else:
                # Infer features based on model type
                if is_enhanced_model:
                    expected_features = [
                        'home_q1', 'home_q2', 'home_q3', 'home_q4', 
                        'score_ratio', 'prev_matchup_diff',
                        'rest_days_home', 'rest_days_away', 'rest_advantage',
                        'is_back_to_back_home', 'is_back_to_back_away',
                        'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
                    ]
                    print("Using enhanced feature set (15 features)")
                else:
                    expected_features = [
                        'home_q1', 'home_q2', 'home_q3', 'home_q4', 
                        'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
                    ]
                    print("Using standard feature set (8 features)")
        else:
            # Default to original features
            expected_features = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4', 
                'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
            ]
            print("Using default feature set (no model provided)")
        
        # Get team averages
        team_avgs = get_team_rolling_averages(cache_key='team_averages')
        
        # Prepare feature DataFrame
        features = []
        
        for idx, game in games_df.iterrows():
            # Get basic game data
            game_id = game['game_id']
            home_team = game['home_team']
            away_team = game['away_team']
            current_quarter = int(game['current_quarter'])
            
            # Quarter scores with error handling
            home_q1 = float(game.get('home_q1', 0) or 0)
            home_q2 = float(game.get('home_q2', 0) or 0)
            home_q3 = float(game.get('home_q3', 0) or 0)
            home_q4 = float(game.get('home_q4', 0) or 0)
            
            away_q1 = float(game.get('away_q1', 0) or 0)
            away_q2 = float(game.get('away_q2', 0) or 0)
            away_q3 = float(game.get('away_q3', 0) or 0)
            away_q4 = float(game.get('away_q4', 0) or 0)
            
            # Calculate score ratio
            home_score = float(game.get('home_score', 0) or 0)
            away_score = float(game.get('away_score', 0) or 0)
            total_score = home_score + away_score
            score_ratio = home_score / total_score if total_score > 0 else 0.5
            
            # Get matchup history
            prev_matchup_diff = get_previous_matchup_diff(home_team, away_team)
            
            # Create base feature dictionary
            feature_dict = {
                'game_id': game_id,
                'home_team': home_team,
                'away_team': away_team,
                'home_q1': home_q1,
                'home_q2': home_q2,
                'home_q3': home_q3,
                'home_q4': home_q4,
                'score_ratio': score_ratio,
                'prev_matchup_diff': prev_matchup_diff,
                'current_quarter': current_quarter
            }
            
            # Add rolling averages
            if 'rolling_home_score' in expected_features:
                feature_dict['rolling_home_score'] = team_avgs.get(home_team, 105.0)
                feature_dict['rolling_away_score'] = team_avgs.get(away_team, 105.0)
            
            # Add enhanced features if needed
            if is_enhanced_model:
                # Default rest features (simplified for demonstration)
                feature_dict['rest_days_home'] = 2  # Default 2 days rest
                feature_dict['rest_days_away'] = 2
                feature_dict['rest_advantage'] = 0
                feature_dict['is_back_to_back_home'] = 0
                feature_dict['is_back_to_back_away'] = 0
                
                # Add momentum features if needed
                if ('q1_to_q2_momentum' in expected_features or 
                    'q2_to_q3_momentum' in expected_features or 
                    'q3_to_q4_momentum' in expected_features or 
                    'cumulative_momentum' in expected_features):
                    
                    # Calculate momentum features
                    feature_dict['q1_to_q2_momentum'] = 0
                    feature_dict['q2_to_q3_momentum'] = 0
                    feature_dict['q3_to_q4_momentum'] = 0
                    feature_dict['cumulative_momentum'] = 0
                    
                    if current_quarter >= 2:
                        feature_dict['q1_to_q2_momentum'] = (home_q2 - home_q1) - (away_q2 - away_q1)
                    
                    if current_quarter >= 3:
                        feature_dict['q2_to_q3_momentum'] = (home_q3 - home_q2) - (away_q3 - away_q2)
                    
                    if current_quarter >= 4:
                        feature_dict['q3_to_q4_momentum'] = (home_q4 - home_q3) - (away_q4 - away_q3)
                    
                    # Calculate cumulative momentum
                    weights = [0.2, 0.3, 0.5]  # Quarter momentum weights
                    
                    if current_quarter == 2:
                        feature_dict['cumulative_momentum'] = feature_dict['q1_to_q2_momentum']
                    elif current_quarter == 3:
                        feature_dict['cumulative_momentum'] = (
                            feature_dict['q1_to_q2_momentum'] * weights[0] + 
                            feature_dict['q2_to_q3_momentum'] * weights[1]
                        ) / (weights[0] + weights[1])
                    elif current_quarter >= 4:
                        feature_dict['cumulative_momentum'] = (
                            feature_dict['q1_to_q2_momentum'] * weights[0] + 
                            feature_dict['q2_to_q3_momentum'] * weights[1] + 
                            feature_dict['q3_to_q4_momentum'] * weights[2]
                        ) / sum(weights)
                    
                    # Normalize momentum to [-1, 1]
                    feature_dict['cumulative_momentum'] = max(
                        min(feature_dict['cumulative_momentum'] / 15.0, 1.0), -1.0
                    )
            
            features.append(feature_dict)
        
        # Create DataFrame
        features_df = pd.DataFrame(features)
        
        # Ensure all expected features exist with proper types
        for feature in expected_features:
            if feature not in features_df.columns:
                print(f"Adding missing feature: {feature}")
                features_df[feature] = 0
            
            # Ensure numeric
            features_df[feature] = pd.to_numeric(features_df[feature], errors='coerce').fillna(0)
        
        return features_df
        
    except Exception as e:
        print(f"Error preparing features: {e}")
        traceback.print_exc()
        return pd.DataFrame()

def run_predictions(model, games_df):
    """
    Run predictions using the model and games data.
    
    Args:
        model: Prediction model
        games_df: DataFrame with games data
        
    Returns:
        DataFrame with prediction results
    """
    if games_df.empty:
        print("No games available for prediction")
        return pd.DataFrame()
    
    if model is None:
        print("No model provided for prediction")
        return pd.DataFrame()
    
    try:
        print(f"Running predictions for {len(games_df)} games")
        
        # Step 1: Prepare features
        features_df = prepare_features(games_df, model)
        if features_df.empty:
            print("Failed to prepare features")
            return pd.DataFrame()
        
        # Step 2: Get the right feature columns for prediction
        model_features = []
        
        # Check if model has explicit feature names
        if hasattr(model, 'feature_names_in_'):
            model_features = list(model.feature_names_in_)
            print(f"Using {len(model_features)} features from model definition")
        elif hasattr(model, 'feature_importances_'):
            # Infer features from importances length
            feature_count = len(model.feature_importances_)
            if feature_count > 8:
                model_features = [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4', 
                    'score_ratio', 'prev_matchup_diff',
                    'rest_days_home', 'rest_days_away', 'rest_advantage',
                    'is_back_to_back_home', 'is_back_to_back_away',
                    'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
                ]
                print(f"Using {len(model_features)} enhanced features based on model importances")
            else:
                model_features = [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4', 
                    'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
                ]
                print(f"Using {len(model_features)} standard features based on model importances")
        else:
            # Default to original features
            model_features = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4', 
                'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
            ]
            print(f"Using {len(model_features)} default features (model type unknown)")
        
        # Make sure all required features exist
        for feature in model_features:
            if feature not in features_df.columns:
                print(f"Adding missing feature '{feature}' (required by model)")
                features_df[feature] = 0
        
        # Step 3: Make predictions
        X_pred = features_df[model_features]
        print(f"Making predictions using {X_pred.shape[1]} features")
        predictions = model.predict(X_pred)
        
        # Step 4: Create results DataFrame with predictions
        results = []
        
        for i, (idx, game) in enumerate(games_df.iterrows()):
            # Get game info
            game_id = game['game_id']
            home_team = game['home_team']
            away_team = game['away_team']
            current_quarter = int(game['current_quarter'])
            home_score = float(game['home_score'])
            away_score = float(game['away_score'])
            
            # Get prediction
            predicted_home_score = predictions[i]
            
            # Calculate away score prediction
            # Constants for prediction adjustments
            DIFF_WEIGHT_BASE = 0.3     # Base weight for matchup differential
            DIFF_WEIGHT_INCREMENT = 0.1  # Weight increment per quarter
            MOMENTUM_SCALE = 3.0       # Points impact of momentum
            
            # Get matchup differential
            prev_matchup_diff = features_df.loc[i, 'prev_matchup_diff'] if 'prev_matchup_diff' in features_df.columns else 0
            
            # Scale effect based on game progress
            diff_weight = min(DIFF_WEIGHT_BASE + (DIFF_WEIGHT_INCREMENT * current_quarter), 0.6)
            
            # Factor in momentum if available
            momentum_adj = 0
            if 'cumulative_momentum' in features_df.columns:
                momentum = features_df.loc[i, 'cumulative_momentum']
                momentum_adj = momentum * MOMENTUM_SCALE
            
            # Calculate predicted away score
            predicted_away_score = predicted_home_score - (prev_matchup_diff * diff_weight) - momentum_adj
            
            # Ensure predictions are at least current scores
            predicted_home_score = max(predicted_home_score, home_score)
            predicted_away_score = max(predicted_away_score, away_score)
            
            # Calculate win probability using logistic function
            score_diff = predicted_home_score - predicted_away_score
            game_progress = min(current_quarter / 4.0, 1.0)
            
            # Constants for win probability calculation
            K_BASE = 0.05      # Base confidence factor
            K_INCREMENT = 0.15  # Increment for game progress
            
            k_factor = K_BASE + (game_progress * K_INCREMENT)
            win_probability = 1.0 / (1.0 + np.exp(-k_factor * score_diff))
            
            # Calculate confidence based on quarter
            confidence_by_quarter = {
                0: 0.6,  # Pre-game
                1: 0.7,  # Q1
                2: 0.8,  # Q2
                3: 0.9,  # Q3
                4: 0.95  # Q4
            }
            prediction_confidence = confidence_by_quarter.get(current_quarter, 0.7)
            
            # Create result dictionary
            result = {
                'game_id': game_id,
                'home_team': home_team,
                'away_team': away_team,
                'current_quarter': current_quarter,
                'current_home_score': home_score,
                'current_away_score': away_score,
                'predicted_home_final': predicted_home_score,
                'predicted_away_final': predicted_away_score,
                'remaining_home_points': predicted_home_score - home_score,
                'remaining_away_points': predicted_away_score - away_score,
                'win_probability': win_probability,
                'prediction_confidence': prediction_confidence,
                'projected_margin': predicted_home_score - predicted_away_score,
                'total_projected_score': predicted_home_score + predicted_away_score,
                'prediction_timestamp': datetime.now().isoformat()
            }
            
            # Add actual finals if available (for historical testing)
            if 'actual_home_final' in game:
                result['actual_home_final'] = game['actual_home_final']
                result['actual_away_final'] = game['actual_away_final']
                
                # Calculate prediction error if actuals available
                if 'actual_home_final' in result:
                    home_error = abs(predicted_home_score - result['actual_home_final'])
                    away_error = abs(predicted_away_score - result['actual_away_final'])
                    result['avg_error'] = (home_error + away_error) / 2
                    result['winner_correct'] = ((predicted_home_score > predicted_away_score) == 
                                              (result['actual_home_final'] > result['actual_away_final']))
            
            results.append(result)
        
        return pd.DataFrame(results)
        
    except Exception as e:
        print(f"Error in prediction pipeline: {e}")
        traceback.print_exc()
        return pd.DataFrame()

In [ ]:
# Cell 25: Add Missing Match Functions

def fetch_live_games() -> pd.DataFrame:
    """Fetch live game data from Supabase."""
    try:
        print("Fetching live games from nba_live_game_stats...")
        response = supabase.table("nba_live_game_stats").select("*").execute()
        if response.data:
            print(f"Found {len(response.data)} live games")
            return pd.DataFrame(response.data)
        else:
            print("No live games found.")
            return pd.DataFrame()
    except Exception as e:
        print(f"Error fetching live games: {e}")
        traceback.print_exc()
        return pd.DataFrame()

def match_live_to_scheduled(live_games_df: pd.DataFrame, scheduled_games_df: pd.DataFrame) -> pd.DataFrame:
    """Match live games to scheduled games using team names."""
    if live_games_df.empty or scheduled_games_df.empty:
        return live_games_df
    matched_games = live_games_df.copy()
    matched_games['verified'] = False
    for idx, game in matched_games.iterrows():
        home_team = str(game.get('home_team', '')).lower()
        away_team = str(game.get('away_team', '')).lower()
        for _, sched in scheduled_games_df.iterrows():
            sched_home = str(sched.get('home_team', '')).lower()
            sched_away = str(sched.get('away_team', '')).lower()
            if (home_team in sched_home and away_team in sched_away) or (home_team in sched_away and away_team in sched_home):
                matched_games.at[idx, 'game_id'] = sched['game_id']
                matched_games.at[idx, 'verified'] = True
                break
    return matched_games

def compute_momentum_features(df: pd.DataFrame) -> pd.DataFrame:
    """Compute momentum-based features for prediction."""
    features_df = df.copy()
    features_df['q1_to_q2_momentum'] = 0
    features_df['q2_to_q3_momentum'] = 0
    features_df['q3_to_q4_momentum'] = 0
    features_df['cumulative_momentum'] = 0
    for idx, row in features_df.iterrows():
        current_quarter = int(row.get('current_quarter', 0))
        h = [float(row.get(f'home_q{i}', 0)) for i in range(1, 5)]
        a = [float(row.get(f'away_q{i}', 0)) for i in range(1, 5)]
        if current_quarter >= 2:
            features_df.at[idx, 'q1_to_q2_momentum'] = (h[1] - h[0]) - (a[1] - a[0])
        if current_quarter >= 3:
            features_df.at[idx, 'q2_to_q3_momentum'] = (h[2] - h[1]) - (a[2] - a[1])
        if current_quarter >= 4:
            features_df.at[idx, 'q3_to_q4_momentum'] = (h[3] - h[2]) - (a[3] - a[2])
        weights = [0.2, 0.3, 0.5]
        if current_quarter == 2:
            cum = features_df.at[idx, 'q1_to_q2_momentum']
        elif current_quarter == 3:
            cum = (features_df.at[idx, 'q1_to_q2_momentum']*weights[0] + features_df.at[idx, 'q2_to_q3_momentum']*weights[1]) / (weights[0]+weights[1])
        elif current_quarter >= 4:
            cum = (features_df.at[idx, 'q1_to_q2_momentum']*weights[0] + features_df.at[idx, 'q2_to_q3_momentum']*weights[1] + features_df.at[idx, 'q3_to_q4_momentum']*weights[2]) / sum(weights)
        else:
            cum = 0
        features_df.at[idx, 'cumulative_momentum'] = np.clip(cum/15.0, -1, 1)
    return features_df


In [ ]:
# Cell 26: Enhanced NBA Game Status Detection with Pacific Time Handling

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import pytz, traceback, os
from sqlalchemy import create_engine
import config
from caching.supabase_client import supabase

def log_with_timestamp(message: str, level: str = "INFO"):
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts}] {level}: {message}")

def fetch_live_games_in_pacific_time():
    """
    Fetch all game data for today in Pacific Time with improved status detection.
    Returns:
        DataFrame with all games (pregame, live, and finished) with proper status flags.
    """
    try:
        log_with_timestamp("Fetching all games for today (Pacific Time)...")
        
        # Get current date in Pacific Time
        pacific_tz = pytz.timezone("America/Los_Angeles")
        now_pt = datetime.now(pacific_tz)
        today_pt = now_pt.date()
        
        # Define start and end of today in PT
        start_pt = datetime.combine(today_pt, datetime.min.time()).replace(tzinfo=pacific_tz)
        end_pt = datetime.combine(today_pt, datetime.max.time()).replace(tzinfo=pacific_tz)
        
        # Convert start and end to UTC ISO format strings
        start_utc = start_pt.astimezone(pytz.utc).isoformat()
        end_utc = end_pt.astimezone(pytz.utc).isoformat()
        
        log_with_timestamp(f"Fetching games from Supabase in range:\n  {start_utc} to {end_utc}")
        
        # Query Supabase for games whose 'game_date' is between start_utc and end_utc
        response = supabase.table("nba_live_game_stats").select("*")\
                          .gte("game_date", start_utc)\
                          .lte("game_date", end_utc)\
                          .execute()
        
        if not response.data:
            log_with_timestamp(f"No games found for today ({today_pt}).", "WARNING")
            return pd.DataFrame()
        
        df = pd.DataFrame(response.data)
        log_with_timestamp(f"Found {len(df)} rows for today in the database.")
        
        # Convert 'game_date' (stored in UTC) to a tz-aware datetime, then to Pacific Time
        df['game_date'] = pd.to_datetime(df['game_date'], errors='coerce', utc=True)
        df['game_date_pt'] = df['game_date'].dt.tz_convert(pacific_tz)
        df['date_only_pt'] = df['game_date_pt'].dt.date
        
        log_with_timestamp(f"Today's date in Pacific Time: {today_pt}")
        df_today = df[df['date_only_pt'] == today_pt].copy()
        log_with_timestamp(f"Rows matching today's PT date: {len(df_today)}")
        
        if df_today.empty:
            log_with_timestamp("No rows match today's date in Pacific Time.", "WARNING")
            return pd.DataFrame()
        
        # Ensure 'is_finished' is a proper boolean column
        if 'is_finished' not in df_today.columns:
            df_today['is_finished'] = False
        else:
            try:
                df_today['is_finished'] = df_today['is_finished'].astype(bool)
            except Exception:
                df_today['is_finished'] = df_today['is_finished'].apply(lambda x: bool(x) if pd.notna(x) else False)
        
        # Ensure 'current_quarter' exists and is numeric
        if 'current_quarter' not in df_today.columns:
            df_today['current_quarter'] = 0
        else:
            df_today['current_quarter'] = pd.to_numeric(df_today['current_quarter'], errors='coerce').fillna(0)
        
        # Set initial game status to 'unknown'
        df_today['game_status'] = 'unknown'
        
        # Process each game to determine its status
        for idx, row in df_today.iterrows():
            # First check if game is explicitly marked as finished
            if row.get('is_finished', False):
                df_today.at[idx, 'game_status'] = 'finished'
                continue
                
            # Check if all 4 quarters have data (likely finished but not marked)
            all_quarters_filled = True
            for q in range(1, 5):
                home_val = float(row.get(f'home_q{q}', 0) or 0)
                away_val = float(row.get(f'away_q{q}', 0) or 0)
                if home_val == 0 and away_val == 0:
                    all_quarters_filled = False
                    break
                    
            if all_quarters_filled:
                df_today.at[idx, 'game_status'] = 'finished'
                df_today.at[idx, 'current_quarter'] = 4
                continue
                
            # If any quarter has points, it's at least started
            quarters_played = 0
            for q in range(1, 5):
                home_val = float(row.get(f'home_q{q}', 0) or 0)
                away_val = float(row.get(f'away_q{q}', 0) or 0)
                if home_val > 0 or away_val > 0:
                    quarters_played = max(quarters_played, q)
            
            if quarters_played > 0:
                # Game has started
                df_today.at[idx, 'game_status'] = 'live'
                
                # Update current_quarter if needed
                if int(row.get('current_quarter', 0)) < quarters_played:
                    df_today.at[idx, 'current_quarter'] = quarters_played
            else:
                # No quarters played yet - it's pre-game
                df_today.at[idx, 'game_status'] = 'pregame'
                
            # Additional check: if home_score and away_score exist and are > 0,
            # but no quarter data, it's probably live but missing quarter breakdown
            if quarters_played == 0 and df_today.at[idx, 'game_status'] == 'pregame':
                home_score = float(row.get('home_score', 0) or 0)
                away_score = float(row.get('away_score', 0) or 0)
                if home_score > 0 or away_score > 0:
                    df_today.at[idx, 'game_status'] = 'live'
                    # Guess quarter based on score
                    total_score = home_score + away_score
                    if total_score > 180:  # Late game
                        df_today.at[idx, 'current_quarter'] = 4
                    elif total_score > 120:  # Mid game
                        df_today.at[idx, 'current_quarter'] = 3
                    elif total_score > 60:   # Early-mid game
                        df_today.at[idx, 'current_quarter'] = 2
                    else:                   # Early game
                        df_today.at[idx, 'current_quarter'] = 1
        
        # Get active live games
        active_live_games = df_today[df_today['game_status'] == 'live'].copy()
        log_with_timestamp(f"Found {len(active_live_games)} active live games")
        
        # Log overall game status breakdown
        log_with_timestamp("Game status breakdown:")
        status_counts = df_today['game_status'].value_counts()
        for status, count in status_counts.items():
            print(f"  {status.upper()}: {count} games")
        
        log_with_timestamp(f"Returning {len(df_today)} total games.")
        return df_today
        
    except Exception as e:
        log_with_timestamp(f"Error fetching games in Pacific Time: {str(e)}", "ERROR")
        traceback.print_exc()
        return pd.DataFrame()

def get_active_live_games():
    """
    Get only actively live games from all today's games.
    Returns:
        DataFrame with only live games.
    """
    all_games = fetch_live_games_in_pacific_time()
    if all_games.empty:
        log_with_timestamp("No games available for today", "WARNING")
        return pd.DataFrame()
    
    # Filter for only live games
    live_games = all_games[all_games['game_status'] == 'live'].copy()
    
    if live_games.empty:
        log_with_timestamp("No actively live games found", "WARNING")
        return pd.DataFrame()
    
    log_with_timestamp(f"Found {len(live_games)} actively live games")
    return live_games

# ---------------- Example Usage for Cell 26 ----------------
if __name__ == "__main__":
    # Get all games
    all_games_pt = fetch_live_games_in_pacific_time()
    if all_games_pt.empty:
        print("No games available in Pacific Time.")
    else:
        print("Games by status (Pacific Time):")
        status_counts = all_games_pt['game_status'].value_counts()
        for status, count in status_counts.items():
            print(f"  {status.upper()}: {count} games")
    
    # Get only active live games
    active_games = get_active_live_games()
    if active_games.empty:
        print("\nNo actively live games found.")
    else:
        print(f"\nFound {len(active_games)} actively live games:")
        for idx, game in active_games.iterrows():
            print(f"  {game['home_team']} vs {game['away_team']} - Q{int(game['current_quarter'])}")

In [ ]:
# Cell 27: Core Uncertainty Estimation Module

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional, Union, Any
import json
from datetime import datetime

class UncertaintyEstimator:
    """
    Core class for estimating uncertainty in NBA score predictions.
    
    This class implements a context-aware uncertainty estimation approach that
    adjusts confidence intervals based on:
    1. Game quarter (uncertainty decreases as game progresses)
    2. Score differential (close games have higher uncertainty)
    3. Team momentum (high momentum can increase uncertainty)
    4. Historical calibration data
    
    Methodology:
    - Base uncertainty is derived from the quarter of play, with earlier quarters
      having wider intervals than later quarters
    - Context adjustments modify this base uncertainty based on game state
    - Intervals are centered around the predicted value
    """
    
    def __init__(self, confidence_level: float = 0.9):
        """
        Initialize the uncertainty estimator with default parameters.
        
        Args:
            confidence_level: Target confidence level (default: 0.9 for 90% coverage)
        """
        # Base width factors for different quarters
        # These control how wide the intervals are relative to the base uncertainty
        self.width_factors = {
            0: 3.0,  # Pre-game: widest intervals
            1: 2.5,  # 1st quarter
            2: 2.0,  # 2nd quarter 
            3: 1.5,  # 3rd quarter
            4: 1.0   # 4th quarter: narrowest intervals
        }
        
        # Confidence interval target (90% coverage by default)
        self.confidence_level = confidence_level
        
        # Calibration metrics dictionary to track interval performance
        self.calibration_data = {
            'total_predictions': 0,
            'in_interval_count': 0,
            'by_quarter': {q: {'count': 0, 'in_interval': 0} for q in range(5)}
        }
    
    def calculate_prediction_interval(
            self, 
            predicted_value: float, 
            quarter: int, 
            score_differential: float = 0, 
            momentum: float = 0,
            time_remaining: Optional[float] = None
        ) -> Tuple[float, float, float]:
        """
        Calculate confidence interval for a prediction with context awareness.
        
        Args:
            predicted_value: The predicted final score
            quarter: Current quarter (0-4, where 0 is pre-game)
            score_differential: Absolute score difference between teams
            momentum: Team momentum (-1 to 1, higher absolute value means more momentum)
            time_remaining: Minutes remaining in current quarter (optional)
            
        Returns:
            Tuple of (lower_bound, upper_bound, interval_width)
        """
        # Get base width factor for this quarter (or default to 2.0)
        base_factor = self.width_factors.get(quarter, 2.0)
        
        # Calculate base uncertainty (decreases as game progresses)
        # Formula: Maximum uncertainty at pre-game (quarter 0), decreases linearly 
        # to minimum at quarter 4. Scale of 10 points represents typical NBA scoring variation.
        base_uncertainty = 10.0 * (4.0 - min(quarter, 4)) / 4.0
        
        # Apply context adjustments
        context_adjustment = self._calculate_context_adjustment(
            quarter, score_differential, momentum, time_remaining)
        
        # Calculate adjusted interval width
        interval_width = base_uncertainty * base_factor * context_adjustment
        
        # Ensure interval width is reasonable (between min and max bounds)
        min_width = 2.0  # Minimum width to account for inherent randomness
        max_width = 30.0  # Maximum width to prevent extreme intervals
        interval_width = min(max(interval_width, min_width), max_width)
        
        # Calculate bounds centered on prediction
        lower_bound = predicted_value - interval_width / 2
        upper_bound = predicted_value + interval_width / 2
        
        return lower_bound, upper_bound, interval_width
    
    def _calculate_context_adjustment(
            self,
            quarter: int,
            score_differential: float,
            momentum: float = 0,
            time_remaining: Optional[float] = None
        ) -> float:
        """
        Calculate adjustment factor based on game context.
        
        Args:
            quarter: Current quarter (0-4)
            score_differential: Absolute score difference between teams
            momentum: Team momentum (-1 to 1)
            time_remaining: Minutes remaining in current quarter
            
        Returns:
            float: Context adjustment factor
        """
        context_adjustment = 1.0
        
        # 1. Adjust for close games vs blowouts
        if score_differential <= 5.0:
            # Close games have higher uncertainty
            context_adjustment *= 1.1
        elif score_differential >= 15.0:
            # Blowouts have lower uncertainty
            context_adjustment *= 0.9
        
        # 2. Adjust for momentum
        if abs(momentum) >= 0.5:
            # High momentum (either direction) increases uncertainty
            context_adjustment *= 1.15
        
        # 3. Final minutes adjustment
        if quarter == 4 and time_remaining is not None and time_remaining <= 5.0:
            # Late in close games, prediction becomes more certain
            context_adjustment *= 0.85
            
        # 4. Special case for pre-game predictions
        if quarter == 0:
            # Pre-game predictions rely on team averages which introduces more uncertainty
            context_adjustment *= 1.2
            
        return context_adjustment
    
    def update_calibration(self, 
                          predicted_value: float, 
                          actual_value: float,
                          quarter: int,
                          lower_bound: float,
                          upper_bound: float):
        """
        Update calibration metrics based on whether actual fell within interval.
        
        Args:
            predicted_value: The predicted score
            actual_value: The actual final score
            quarter: Quarter when prediction was made (0-4)
            lower_bound: Lower bound of confidence interval
            upper_bound: Upper bound of confidence interval
        """
        # Increment total predictions counter
        self.calibration_data['total_predictions'] += 1
        
        # Check if actual value is within interval
        in_interval = lower_bound <= actual_value <= upper_bound
        
        # Update overall metrics
        if in_interval:
            self.calibration_data['in_interval_count'] += 1
        
        # Update quarter-specific metrics
        if 0 <= quarter <= 4:
            self.calibration_data['by_quarter'][quarter]['count'] += 1
            if in_interval:
                self.calibration_data['by_quarter'][quarter]['in_interval'] += 1
    
    def get_calibration_report(self) -> Dict[str, Any]:
        """
        Get a report on uncertainty calibration.
        
        Returns:
            Dict with calibration metrics including overall coverage and
            quarter-specific coverage
        """
        report = {
            'total_predictions': self.calibration_data['total_predictions'],
            'overall_coverage': 0,
            'target_coverage': self.confidence_level,
            'by_quarter': {}
        }
        
        # Calculate overall coverage
        if report['total_predictions'] > 0:
            report['overall_coverage'] = (
                self.calibration_data['in_interval_count'] / 
                report['total_predictions']
            )
        
        # Calculate per-quarter coverage
        for quarter, data in self.calibration_data['by_quarter'].items():
            if data['count'] > 0:
                coverage = data['in_interval'] / data['count']
                report['by_quarter'][quarter] = {
                    'count': data['count'],
                    'coverage': coverage,
                    'width_factor': self.width_factors.get(quarter, 0)
                }
        
        # Add calibration quality assessment
        # Well-calibrated means the actual coverage is within 5% of the target
        report['is_well_calibrated'] = (
            abs(report['overall_coverage'] - self.confidence_level) < 0.05
            if report['total_predictions'] >= 30 else None
        )
        
        return report
    
    def apply_intervals_to_predictions(self, predictions_df: pd.DataFrame) -> pd.DataFrame:
        """
        Apply prediction intervals to all predictions in a DataFrame.
        
        Args:
            predictions_df: DataFrame with prediction data
            
        Returns:
            DataFrame with confidence intervals added
        """
        # Create copy to avoid modifying original
        result_df = predictions_df.copy()
        
        # Add interval columns if they don't exist
        for col in ['home_lower_bound', 'home_upper_bound', 
                   'away_lower_bound', 'away_upper_bound', 
                   'home_interval_width', 'away_interval_width']:
            if col not in result_df.columns:
                result_df[col] = np.nan
        
        # Process each prediction
        for idx, row in result_df.iterrows():
            quarter = int(row.get('current_quarter', 0))
            
            # Get score differential
            home_score = float(row.get('current_home_score', row.get('home_score', 0)))
            away_score = float(row.get('current_away_score', row.get('away_score', 0)))
            score_diff = abs(home_score - away_score)
            
            # Get momentum if available
            momentum = float(row.get('momentum_shift', row.get('cumulative_momentum', 0)))
            
            # Get time remaining if available
            time_remaining = row.get('time_remaining')
            
            # Calculate intervals for home score
            home_pred = float(row.get('predicted_home_final', row.get('predicted_home_score', 0)))
            home_lower, home_upper, home_width = self.calculate_prediction_interval(
                home_pred, quarter, score_diff, momentum, time_remaining
            )
            
            # Calculate intervals for away score
            away_pred = float(row.get('predicted_away_final', row.get('predicted_away_score', 0)))
            away_lower, away_upper, away_width = self.calculate_prediction_interval(
                away_pred, quarter, score_diff, momentum, time_remaining
            )
            
            # Store in DataFrame
            result_df.loc[idx, 'home_lower_bound'] = home_lower
            result_df.loc[idx, 'home_upper_bound'] = home_upper
            result_df.loc[idx, 'home_interval_width'] = home_width
            
            result_df.loc[idx, 'away_lower_bound'] = away_lower
            result_df.loc[idx, 'away_upper_bound'] = away_upper
            result_df.loc[idx, 'away_interval_width'] = away_width
        
        return result_df


class ConfidenceVisualizer:
    """
    Specialized class for visualizing prediction confidence.
    
    This class provides methods to create various visualizations
    of prediction confidence and uncertainty.
    """
    
    def __init__(self, color_scheme: str = 'default'):
        """
        Initialize the visualizer with a color scheme.
        
        Args:
            color_scheme: Color scheme name ('default', 'dark', 'light')
        """
        # Color schemes
        self.color_schemes = {
            'default': {
                'background': '#f8f9fa',
                'text': '#212529',
                'home_team': '#0d6efd',
                'away_team': '#dc3545',
                'confidence_high': '#198754',
                'confidence_med': '#fd7e14',
                'confidence_low': '#dc3545',
                'grid': '#e9ecef',
                'axis': '#6c757d'
            },
            'dark': {
                'background': '#212529',
                'text': '#f8f9fa',
                'home_team': '#0dcaf0',
                'away_team': '#f85c70',
                'confidence_high': '#20c997',
                'confidence_med': '#fd7e14',
                'confidence_low': '#dc3545',
                'grid': '#495057',
                'axis': '#adb5bd'
            },
            'light': {
                'background': '#ffffff',
                'text': '#212529',
                'home_team': '#0d6efd',
                'away_team': '#dc3545',
                'confidence_high': '#198754',
                'confidence_med': '#fd7e14',
                'confidence_low': '#dc3545',
                'grid': '#f1f3f5',
                'axis': '#6c757d'
            }
        }
        
        # Set active color scheme
        self.colors = self.color_schemes.get(color_scheme, self.color_schemes['default'])
        
        # Quarter-specific colors for visual differentiation
        self.quarter_colors = {
            0: "#6c757d",  # Gray for pregame
            1: "#20c997",  # Teal for Q1
            2: "#0dcaf0",  # Cyan for Q2
            3: "#0d6efd",  # Blue for Q3
            4: "#6610f2"   # Purple for Q4
        }
    
    def visualize_confidence_intervals(self, predictions_df: pd.DataFrame) -> plt.Figure:
        """
        Create a figure visualizing prediction intervals by quarter.
        
        Args:
            predictions_df: DataFrame with prediction data including intervals
            
        Returns:
            matplotlib Figure object
        """
        # Create figure
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Extract by quarter
        grouped = predictions_df.groupby('current_quarter')
        x_positions = []
        x_labels = []
        
        for i, (quarter, group) in enumerate(grouped):
            # Get mean interval widths
            if 'home_interval_width' in group.columns and 'away_interval_width' in group.columns:
                mean_width = (group['home_interval_width'].mean() + group['away_interval_width'].mean()) / 2
            else:
                # Estimate from upper/lower bounds
                home_width = (group['home_upper_bound'] - group['home_lower_bound']).mean()
                away_width = (group['away_upper_bound'] - group['away_lower_bound']).mean()
                mean_width = (home_width + away_width) / 2
            
            # Calculate error
            if 'actual_home_final' in group.columns and 'actual_away_final' in group.columns:
                home_error = abs(group['predicted_home_final'] - group['actual_home_final']).mean()
                away_error = abs(group['predicted_away_final'] - group['actual_away_final']).mean()
                mean_error = (home_error + away_error) / 2
            else:
                mean_error = None
            
            # Position on x-axis
            x_pos = i
            x_positions.append(x_pos)
            x_labels.append(f"Q{quarter}")
            
            # Plot interval width
            ax.bar(x_pos, mean_width, 
                  color=self.quarter_colors.get(quarter, self.colors['confidence_med']),
                  alpha=0.7)
            
            # Plot error if available
            if mean_error is not None:
                ax.plot(x_pos, mean_error, 'o', color='black', markersize=8)
            
            # Add interval size text
            ax.text(x_pos, mean_width + 1, f"{mean_width:.1f}", 
                   ha='center', va='bottom', fontsize=10)
        
        # Customize plot
        ax.set_ylabel("Points", fontsize=12)
        ax.set_title("Prediction Uncertainty by Quarter", fontsize=14)
        ax.set_xticks(x_positions)
        ax.set_xticklabels(x_labels)
        ax.grid(axis='y', linestyle='--', alpha=0.7)
        
        # Add legend
        if any('actual_home_final' in group.columns for _, group in grouped):
            from matplotlib.lines import Line2D
            legend_elements = [
                Line2D([0], [0], marker='o', color='w', markerfacecolor='black', 
                      markersize=8, label='Mean Error'),
                plt.Rectangle((0,0), 1, 1, fc=self.colors['confidence_med'], 
                             alpha=0.7, label='Interval Width')
            ]
            ax.legend(handles=legend_elements, loc='upper right')
            
        plt.tight_layout()
        return fig
    
    def create_confidence_indicator(self, prediction_data: Dict[str, Any]) -> str:
        """
        Create an SVG confidence indicator for a prediction.
        
        Args:
            prediction_data: Dict with prediction data including intervals
            
        Returns:
            String with SVG code for the indicator
        """
        # Extract key information
        home_team = prediction_data.get('home_team', 'Home')
        away_team = prediction_data.get('away_team', 'Away')
        quarter = int(prediction_data.get('current_quarter', 0))
        home_score = float(prediction_data.get('current_home_score', prediction_data.get('home_score', 0)))
        away_score = float(prediction_data.get('current_away_score', prediction_data.get('away_score', 0)))
        
        # Predicted scores
        home_pred = float(prediction_data.get('predicted_home_final', prediction_data.get('predicted_home_score', 0)))
        away_pred = float(prediction_data.get('predicted_away_final', prediction_data.get('predicted_away_score', 0)))
        
        # Intervals
        home_lower = float(prediction_data.get('home_lower_bound', home_pred - 10))
        home_upper = float(prediction_data.get('home_upper_bound', home_pred + 10))
        away_lower = float(prediction_data.get('away_lower_bound', away_pred - 10))
        away_upper = float(prediction_data.get('away_upper_bound', away_pred + 10))
        
        # Win probability
        win_prob = float(prediction_data.get('win_probability', 0.5))
        
        # Generate SVG using the template method
        svg = self._generate_confidence_svg(
            home_team, away_team, quarter, 
            home_score, away_score,
            home_pred, away_pred,
            home_lower, home_upper,
            away_lower, away_upper,
            win_prob
        )
        
        return svg
    
    def _generate_confidence_svg(self,
                               home_team: str,
                               away_team: str,
                               quarter: int,
                               home_score: float,
                               away_score: float,
                               home_pred: float,
                               away_pred: float,
                               home_lower: float,
                               home_upper: float,
                               away_lower: float,
                               away_upper: float,
                               win_prob: float) -> str:
        """
        Generate SVG code for confidence indicator.
        
        Args:
            home_team: Home team name
            away_team: Away team name
            quarter: Current quarter (0-4)
            home_score: Current home score
            away_score: Current away score
            home_pred: Predicted home score
            away_pred: Predicted away score
            home_lower: Home score lower bound
            home_upper: Home score upper bound
            away_lower: Away score lower bound
            away_upper: Away score upper bound
            win_prob: Win probability (0-1)
            
        Returns:
            String with SVG code
        """
        # SVG dimensions
        width = 360
        height = 220
        
        # Get quarter color
        quarter_color = self.quarter_colors.get(quarter, self.quarter_colors[0])
        
        # Get win probability color
        if win_prob > 0.65:
            win_color = self.colors['confidence_high']
        elif win_prob > 0.45:
            win_color = self.colors['confidence_med']
        else:
            win_color = self.colors['confidence_low']
            
        # Calculate vertical scale and positioning
        max_score = max(home_upper, away_upper, 130)
        y_scale = (height - 100) / max_score
        
        # Create SVG
        svg = f"""
        <svg width="100%" height="100%" viewBox="0 0 {width} {height}" 
             xmlns="http://www.w3.org/2000/svg" style="max-width:500px;">
            <!-- Background -->
            <rect x="0" y="0" width="{width}" height="{height}" 
                  fill="{self.colors['background']}" rx="10" ry="10" />
            
            <!-- Header with Game Status -->
            <rect x="0" y="0" width="{width}" height="40" 
                  fill="{quarter_color}" rx="10" ry="10" />
            <text x="{width/2}" y="25" text-anchor="middle" 
                  font-family="system-ui" font-size="16" fill="white" font-weight="bold">
                {f"QUARTER {quarter}" if quarter > 0 else "PRE-GAME"} PREDICTION
            </text>
            
            <!-- Team Names and Current Score -->
            <text x="80" y="60" text-anchor="middle" 
                  font-family="system-ui" font-size="14" font-weight="bold" 
                  fill="{self.colors['home_team']}">
                {home_team}
            </text>
            <text x="{width-80}" y="60" text-anchor="middle" 
                  font-family="system-ui" font-size="14" font-weight="bold" 
                  fill="{self.colors['away_team']}">
                {away_team}
            </text>
            
            <text x="80" y="80" text-anchor="middle" 
                  font-family="system-ui" font-size="18" font-weight="bold" 
                  fill="{self.colors['text']}">
                {home_score:.0f}
            </text>
            <text x="{width-80}" y="80" text-anchor="middle" 
                  font-family="system-ui" font-size="18" font-weight="bold" 
                  fill="{self.colors['text']}">
                {away_score:.0f}
            </text>
            
            <!-- Confidence Intervals -->
            <rect x="60" y="{height - 70 - (home_upper * y_scale)}" 
                  width="40" height="{(home_upper - home_lower) * y_scale}" 
                  fill="{self.colors['home_team']}" fill-opacity="0.25" rx="5" ry="5" />
                  
            <rect x="{width-100}" y="{height - 70 - (away_upper * y_scale)}" 
                  width="40" height="{(away_upper - away_lower) * y_scale}" 
                  fill="{self.colors['away_team']}" fill-opacity="0.25" rx="5" ry="5" />
            
            <!-- Predicted Final Scores -->
            <line x1="40" x2="120" y1="{height - 70 - (home_pred * y_scale)}" 
                  y2="{height - 70 - (home_pred * y_scale)}" 
                  stroke="{self.colors['home_team']}" stroke-width="2.5" stroke-dasharray="2" />
                  
            <line x1="{width-120}" x2="{width-40}" y1="{height - 70 - (away_pred * y_scale)}" 
                  y2="{height - 70 - (away_pred * y_scale)}" 
                  stroke="{self.colors['away_team']}" stroke-width="2.5" stroke-dasharray="2" />
            
            <!-- Prediction Labels -->
            <text x="80" y="{height - 70 - (home_pred * y_scale) - 8}" text-anchor="middle" 
                  font-family="system-ui" font-size="14" font-weight="bold" 
                  fill="{self.colors['home_team']}">
                {home_pred:.1f}
            </text>
            
            <text x="{width-80}" y="{height - 70 - (away_pred * y_scale) - 8}" text-anchor="middle" 
                  font-family="system-ui" font-size="14" font-weight="bold" 
                  fill="{self.colors['away_team']}">
                {away_pred:.1f}
            </text>
            
            <!-- Win Probability Bar -->
            <text x="20" y="{height-45}" font-family="system-ui" font-size="12" 
                  fill="{self.colors['text']}">
                Win Probability:
            </text>
            
            <rect x="20" y="{height-35}" width="{width-40}" height="10" 
                  fill="#e9ecef" rx="5" ry="5" />
            <rect x="20" y="{height-35}" width="{(width-40) * win_prob}" height="10" 
                  fill="{win_color}" rx="5" ry="5" />
            
            <!-- Win Probability Value -->
            <text x="{width-20}" y="{height-45}" text-anchor="end" 
                  font-family="system-ui" font-size="14" font-weight="bold" 
                  fill="{win_color}">
                {win_prob*100:.1f}%
            </text>
            
            <!-- Timestamp -->
            <text x="{width-10}" y="{height-10}" text-anchor="end" 
                  font-family="system-ui" font-size="10" fill="{self.colors['axis']}">
                Updated: {datetime.now().strftime('%H:%M:%S')}
            </text>
        </svg>
        """
        
        return svg


class CalibrationEvaluator:
    """
    Evaluates and improves confidence interval calibration.
    
    This class implements methods to evaluate how well-calibrated
    uncertainty estimates are and adjust calibration factors.
    """
    
    def __init__(self, uncertainty_estimator=None):
        """
        Initialize the calibration evaluator.
        
        Args:
            uncertainty_estimator: UncertaintyEstimator instance to calibrate
        """
        self.uncertainty_estimator = uncertainty_estimator or UncertaintyEstimator()
        
        # Track calibration results
        self.calibration_history = []
    
    def evaluate_calibration(self, predictions_df: pd.DataFrame, 
                           actual_results_df: pd.DataFrame) -> Dict[str, Any]:
        """
        Evaluate how well-calibrated the prediction intervals are.
        
        Args:
            predictions_df: DataFrame with predictions and intervals
            actual_results_df: DataFrame with actual results
            
        Returns:
            Dict with calibration metrics
        """
        # Merge predictions with actuals
        merged = pd.merge(
            predictions_df,
            actual_results_df[['game_id', 'home_score', 'away_score']],
            on='game_id',
            how='inner',
            suffixes=('_pred', '_actual')
        )
        
        if merged.empty:
            return {'error': 'No matching games found'}
        
        # Initialize metrics
        results = {
            'total_evaluated': len(merged),
            'overall_coverage': 0,
            'by_quarter': {},
            'calibration_error': 0,
            'target_coverage': self.uncertainty_estimator.confidence_level
        }
        
        # Check if actual values fall within prediction intervals
        home_correct = ((merged['home_score'] >= merged['home_lower_bound']) & 
                      (merged['home_score'] <= merged['home_upper_bound']))
        away_correct = ((merged['away_score'] >= merged['away_lower_bound']) & 
                      (merged['away_score'] <= merged['away_upper_bound']))
        both_correct = home_correct & away_correct
        
        # Calculate overall coverage
        results['overall_coverage'] = both_correct.mean()
        results['home_coverage'] = home_correct.mean()
        results['away_coverage'] = away_correct.mean()
        
        # Calibration error (how far from target)
        results['calibration_error'] = abs(
            results['overall_coverage'] - results['target_coverage']
        )
        
        # Calculate per-quarter coverage
        for quarter in range(5):
            quarter_data = merged[merged['current_quarter'] == quarter]
            if not quarter_data.empty:
                q_home_correct = ((quarter_data['home_score'] >= quarter_data['home_lower_bound']) & 
                                (quarter_data['home_score'] <= quarter_data['home_upper_bound']))
                q_away_correct = ((quarter_data['away_score'] >= quarter_data['away_lower_bound']) & 
                                (quarter_data['away_score'] <= quarter_data['away_upper_bound']))
                q_both_correct = q_home_correct & q_away_correct
                
                results['by_quarter'][quarter] = {
                    'count': len(quarter_data),
                    'coverage': q_both_correct.mean(),
                    'width_factor': self.uncertainty_estimator.width_factors.get(quarter, 0)
                }
        
        # Calculate average interval width
        results['avg_home_width'] = (
            merged['home_upper_bound'] - merged['home_lower_bound']
        ).mean()
        results['avg_away_width'] = (
            merged['away_upper_bound'] - merged['away_lower_bound']
        ).mean()
        
        # Store calibration result
        self.calibration_history.append({
            'timestamp': datetime.now().isoformat(),
            'metrics': results,
            'width_factors': self.uncertainty_estimator.width_factors.copy()
        })
        
        return results
    
    def calibrate_width_factors(self, 
                              predictions_df: pd.DataFrame, 
                              actual_results_df: pd.DataFrame, 
                              adjustment_rate: float = 0.5) -> Dict[int, float]:
        """
        Calibrate width factors based on observed coverage rates.
        
        Args:
            predictions_df: DataFrame with predictions and intervals
            actual_results_df: DataFrame with actual results
            adjustment_rate: How much to adjust factors (0-1)
            
        Returns:
            Dict with updated width factors
        """
        # Evaluate current calibration
        calibration = self.evaluate_calibration(predictions_df, actual_results_df)
        
        # Initialize new factors with current values
        new_factors = self.uncertainty_estimator.width_factors.copy()
        
        # Adjust factors by quarter based on observed coverage
        for quarter, metrics in calibration.get('by_quarter', {}).items():
            if metrics['count'] >= 5:  # Only adjust with sufficient data
                current_coverage = metrics['coverage']
                target_coverage = self.uncertainty_estimator.confidence_level
                
                # Calculate adjustment ratio
                if current_coverage > 0:
                    adjustment_ratio = target_coverage / current_coverage
                    
                    # Apply adjustment with dampening
                    new_factors[quarter] = new_factors.get(quarter, 1.0) * (
                        1.0 + (adjustment_ratio - 1.0) * adjustment_rate
                    )
                    
                    # Ensure factor stays in reasonable range
                    new_factors[quarter] = min(max(new_factors[quarter], 0.5), 5.0)
        
        # Update estimator with new factors
        self.uncertainty_estimator.width_factors = new_factors
        
        # Record calibration update
        self.calibration_history.append({
            'timestamp': datetime.now().isoformat(),
            'action': 'calibrate',
            'before': calibration.get('width_factors', {}),
            'after': new_factors
        })
        
        return new_factors
    
    def visualize_calibration_history(self) -> plt.Figure:
        """
        Visualize the history of calibration metrics.
        
        Returns:
            matplotlib Figure object
        """
        if not self.calibration_history:
            return plt.figure()
        
        # Extract data from history
        timestamps = []
        overall_coverage = []
        quarter_coverage = {q: [] for q in range(5)}
        width_factors = {q: [] for q in range(5)}
        
        for entry in self.calibration_history:
            if 'metrics' in entry:
                metrics = entry['metrics']
                timestamps.append(entry.get('timestamp', ''))
                overall_coverage.append(metrics.get('overall_coverage', 0))
                
                for q, q_metrics in metrics.get('by_quarter', {}).items():
                    quarter_coverage[q].append(q_metrics.get('coverage', 0))
                    width_factors[q].append(entry.get('width_factors', {}).get(q, 0))
        
        # Create figure with subplots
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
        
        # Plot coverage rates
        ax1.axhline(y=self.uncertainty_estimator.confidence_level, color='black', 
                   linestyle='--', label=f'Target ({self.uncertainty_estimator.confidence_level:.0%})')
        
        ax1.plot(range(len(timestamps)), overall_coverage, 'o-', 
               label='Overall', linewidth=2, color='blue')
        
        # Plot quarter-specific coverage
        colors = ['gray', 'green', 'orange', 'red', 'purple']  # For quarters 0-4
        for q in range(5):
            if quarter_coverage[q]:
                ax1.plot(range(len(timestamps)), quarter_coverage[q], 's-', 
                       label=f'Q{q}', color=colors[q], alpha=0.7)
        
        ax1.set_ylabel('Coverage Rate')
        ax1.set_title('Confidence Interval Coverage Over Time')
        ax1.grid(True, alpha=0.3)
        ax1.legend()
        
        # Plot width factors
        for q in range(5):
            if width_factors[q]:
                ax2.plot(range(len(timestamps)), width_factors[q], 's-', 
                       label=f'Q{q} Factor', color=colors[q])
        
        ax2.set_xlabel('Calibration Iteration')
        ax2.set_ylabel('Width Factor')
        ax2.set_title('Confidence Interval Width Factors Over Time')
        ax2.grid(True, alpha=0.3)
        ax2.legend()
        
        plt.tight_layout()
        return fig


# Integration functions

def add_uncertainty_to_predictions(predictions_df: pd.DataFrame, 
                                 uncertainty_estimator: Optional[UncertaintyEstimator] = None) -> pd.DataFrame:
    """
    Add uncertainty estimates to prediction results.
    
    Args:
        predictions_df: DataFrame with prediction results
        uncertainty_estimator: Optional custom uncertainty estimator
        
    Returns:
        DataFrame with uncertainty bounds added
    """
    # Create estimator if not provided
    if uncertainty_estimator is None:
        uncertainty_estimator = UncertaintyEstimator()
    
    # Apply intervals to predictions
    result_df = uncertainty_estimator.apply_intervals_to_predictions(predictions_df)
    
    # Add additional uncertainty-related fields
    for idx, row in result_df.iterrows():
        # Get prediction confidence based on interval width
        home_width = row['home_upper_bound'] - row['home_lower_bound']
        away_width = row['away_upper_bound'] - row['away_lower_bound']
        avg_width = (home_width + away_width) / 2
        
        # Add normalized prediction confidence (1.0 is highest confidence)
        # Formula maps interval width to confidence - wider intervals = lower confidence
        max_expected_width = 30.0  # Maximum reasonable interval width
        confidence = max(0.0, min(1.0, 1.0 - (avg_width / max_expected_width)))
        
        # Store confidence
        result_df.loc[idx, 'prediction_confidence'] = confidence
        
        # Add uncertainty ratio (ratio of uncertainty to predicted value)
        home_pred = row['predicted_home_final']
        away_pred = row['predicted_away_final']
        
        if home_pred > 0:
            result_df.loc[idx, 'home_uncertainty_ratio'] = home_width / home_pred
        if away_pred > 0:
            result_df.loc[idx, 'away_uncertainty_ratio'] = away_width / away_pred
    
    return result_df


# Function to create confidence visualizations for a game
def create_game_confidence_visualization(game_data: Dict[str, Any], 
                                       color_scheme: str = 'default') -> Dict[str, Any]:
    """
    Create confidence visualizations for a single game prediction.
    
    Args:
        game_data: Dict with game prediction data
        color_scheme: Color scheme for visualization
        
    Returns:
        Dict with visualization components (SVG indicator, interval data)
    """
    # Create visualizer
    visualizer = ConfidenceVisualizer(color_scheme=color_scheme)
    
    # Generate SVG indicator
    svg_indicator = visualizer.create_confidence_indicator(game_data)
    
    # Extract confidence interval data for API response
    confidence_data = {
        'home_team': game_data.get('home_team', 'Home'),
        'away_team': game_data.get('away_team', 'Away'),
        'home_score': {
            'current': game_data.get('home_score', 0),
            'predicted': game_data.get('predicted_home_final', 0),
            'lower_bound': game_data.get('home_lower_bound', 0),
            'upper_bound': game_data.get('home_upper_bound', 0),
            'interval_width': game_data.get('home_upper_bound', 0) - game_data.get('home_lower_bound', 0)
        },
        'away_score': {
            'current': game_data.get('away_score', 0),
            'predicted': game_data.get('predicted_away_final', 0),
            'lower_bound': game_data.get('away_lower_bound', 0),
            'upper_bound': game_data.get('away_upper_bound', 0),
            'interval_width': game_data.get('away_upper_bound', 0) - game_data.get('away_lower_bound', 0)
        },
        'win_probability': game_data.get('win_probability', 0.5),
        'quarter': game_data.get('current_quarter', 0),
        'visualization_svg': svg_indicator
    }
    
    return confidence_data


# Create instances for testing and demonstration
def demonstrate_uncertainty_estimation():
    """Demonstrate uncertainty estimation functionality."""
    # Create sample prediction data
    sample_data = [
        {
            'game_id': 1001,
            'home_team': 'Lakers',
            'away_team': 'Celtics',
            'current_quarter': 2,
            'home_score': 52,
            'away_score': 48,
            'predicted_home_final': 105.5,
            'predicted_away_final': 98.7,
            'win_probability': 0.65
        },
        {
            'game_id': 1002,
            'home_team': 'Warriors',
            'away_team': 'Bucks',
            'current_quarter': 4,
            'home_score': 98,
            'away_score': 92,
            'predicted_home_final': 106.2,
            'predicted_away_final': 100.8,
            'win_probability': 0.85
        }
    ]
    
    df = pd.DataFrame(sample_data)
    
    # Create instances
    uncertainty_estimator = UncertaintyEstimator()
    confidence_visualizer = ConfidenceVisualizer()
    calibration_evaluator = CalibrationEvaluator(uncertainty_estimator)
    
    # Add uncertainty to predictions
    predictions_with_uncertainty = add_uncertainty_to_predictions(df, uncertainty_estimator)
    
    print("Predictions with Uncertainty Estimates:")
    if 'get_ipython' in globals():
        from IPython.display import display
        display(predictions_with_uncertainty)
    else:
        print(predictions_with_uncertainty)
    
    # Create visualization for first game
    first_game = predictions_with_uncertainty.iloc[0].to_dict()
    svg = confidence_visualizer.create_confidence_indicator(first_game)
    
    # Display SVG
    if 'get_ipython' in globals():
        from IPython.display import HTML
        display(HTML(svg))
    else:
        print("SVG visualization created (length: {len(svg)} characters)")
    
    return predictions_with_uncertainty

# Run the demonstration if in notebook
if 'get_ipython' in globals():
    uncertainty_estimator = UncertaintyEstimator()
    confidence_visualizer = ConfidenceVisualizer()
    calibration_evaluator = CalibrationEvaluator(uncertainty_estimator)
    demonstrate_uncertainty_estimation()

In [ ]:
# Cell 27A: Auto-calibrating Confidence Intervals

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional, Union, Any

def calibrate_confidence_intervals(validation_results: pd.DataFrame, 
                                 current_width_factors: Dict[int, float],
                                 target_coverage: float = 0.9,
                                 dampen_factor: float = 0.5) -> Dict[int, float]:
    """
    Adjust confidence interval widths based on historical accuracy.
    
    This function examines the historical coverage rates of prediction intervals
    and adjusts the width factors to try to achieve the target coverage rate.
    
    Args:
        validation_results: DataFrame with validation results containing 'quarter',
                           'actual_score', 'lower_bound', and 'upper_bound' columns
        current_width_factors: Dict mapping quarters (0-4) to current width factors
        target_coverage: Target coverage rate (default: 0.9 for 90% confidence)
        dampen_factor: Factor to dampen adjustments (0-1, lower = more conservative)
    
    Returns:
        Dict mapping quarters to adjusted width factors
    """
    coverage_by_quarter = {}
    
    # Calculate actual coverage rates
    for quarter in range(5):  # 0-4 for pre-game and quarters 1-4
        quarter_results = validation_results[validation_results['quarter'] == quarter]
        if not quarter_results.empty:
            in_interval = ((quarter_results['actual_score'] >= quarter_results['lower_bound']) & 
                          (quarter_results['actual_score'] <= quarter_results['upper_bound']))
            coverage_by_quarter[quarter] = in_interval.mean()
    
    # Adjust width factors
    adjusted_factors = {}
    for quarter, coverage in coverage_by_quarter.items():
        # Handle edge cases for coverage
        if coverage <= 0.01:  # Avoid division by zero or very small values
            # If coverage is near zero, significantly widen intervals
            adjusted_factors[quarter] = current_width_factors.get(quarter, 1.0) * 1.5
            continue
            
        # If coverage is too low, widen the intervals (ratio > 1)
        # If coverage is too high, narrow the intervals (ratio < 1)
        adjustment_ratio = target_coverage / coverage
        
        # Apply dampening to prevent overcorrection
        dampened_adjustment = 1.0 + (adjustment_ratio - 1.0) * dampen_factor
        
        # Calculate new factor with dampening
        new_factor = current_width_factors.get(quarter, 1.0) * dampened_adjustment
        
        # Ensure factors stay within reasonable bounds
        adjusted_factors[quarter] = max(min(new_factor, 5.0), 0.5)
    
    # Ensure all quarters have factors, using defaults for missing quarters
    for quarter in range(5):
        if quarter not in adjusted_factors:
            # Default mapping of quarter to width factor if not calculated
            default_factors = {0: 3.0, 1: 2.5, 2: 2.0, 3: 1.5, 4: 1.0}
            adjusted_factors[quarter] = current_width_factors.get(quarter, default_factors[quarter])
    
    return adjusted_factors


def validate_confidence_intervals(predictions_df: pd.DataFrame, 
                               actual_results_df: pd.DataFrame, 
                               confidence_level: float = 0.9) -> pd.DataFrame:
    """
    Validate that confidence intervals contain actual values at the expected rate.
    
    Args:
        predictions_df: DataFrame with predictions and confidence intervals containing
                      at minimum 'game_id', 'current_quarter', 'home_lower_bound',
                      'home_upper_bound', 'away_lower_bound', 'away_upper_bound'
        actual_results_df: DataFrame with actual final scores containing
                         at minimum 'game_id', 'home_score', 'away_score'
        confidence_level: Expected coverage rate (default: 0.9 for 90% confidence)
    
    Returns:
        DataFrame with validation metrics by quarter including coverage rates
        and interval widths
    """
    # Merge predictions with actual results
    merged = pd.merge(
        predictions_df,
        actual_results_df[['game_id', 'home_score', 'away_score']],
        on='game_id', 
        suffixes=('_pred', '_actual')
    )
    
    if merged.empty:
        print("Warning: No matching games found between predictions and actuals")
        return pd.DataFrame()
    
    # Calculate validation metrics
    results = []
    for quarter in range(5):
        quarter_data = merged[merged['current_quarter'] == quarter]
        if quarter_data.empty:
            continue
        
        # Check if actual scores fall within predicted intervals
        home_in_interval = ((quarter_data['home_score'] >= quarter_data['home_lower_bound']) & 
                           (quarter_data['home_score'] <= quarter_data['home_upper_bound']))
        
        away_in_interval = ((quarter_data['away_score'] >= quarter_data['away_lower_bound']) & 
                           (quarter_data['away_score'] <= quarter_data['away_upper_bound']))
        
        # Calculate coverage rates
        home_coverage = home_in_interval.mean()
        away_coverage = away_in_interval.mean()
        avg_coverage = (home_coverage + away_coverage) / 2
        
        # Calculate average interval widths
        home_width = (quarter_data['home_upper_bound'] - quarter_data['home_lower_bound']).mean()
        away_width = (quarter_data['away_upper_bound'] - quarter_data['away_lower_bound']).mean()
        
        results.append({
            'quarter': quarter,
            'home_coverage': home_coverage,
            'away_coverage': away_coverage,
            'avg_coverage': avg_coverage,
            'target_coverage': confidence_level,
            'coverage_error': avg_coverage - confidence_level,
            'home_interval_width': home_width,
            'away_interval_width': away_width,
            'avg_interval_width': (home_width + away_width) / 2,
            'sample_size': len(quarter_data)
        })
    
    return pd.DataFrame(results)


def generate_test_calibration_data(uncertainty_estimator, 
                                 num_samples: int = 50, 
                                 target_coverage: float = 0.9) -> pd.DataFrame:
    """
    Generate synthetic calibration data for testing calibration methods.
    
    Args:
        uncertainty_estimator: An instance of UncertaintyEstimator class
        num_samples: Number of synthetic samples to generate
        target_coverage: Target coverage rate to simulate
        
    Returns:
        DataFrame with synthetic validation data
    """
    np.random.seed(42)  # For reproducibility
    validation_data = []
    
    # Generate samples across quarters
    for i in range(num_samples):
        # Randomly select quarter
        quarter = np.random.randint(0, 5)
        
        # Generate predicted score (around 110 points with some variance)
        pred_score = np.random.normal(110, 5)
        
        # Sample score differential and momentum
        score_diff = np.random.uniform(0, 20)
        momentum = np.random.uniform(-0.8, 0.8)
        
        # Get interval from uncertainty estimator
        lower, upper, width = uncertainty_estimator.calculate_prediction_interval(
            pred_score, quarter, score_diff, momentum)
        
        # Generate actual score
        # Simulate a actual_within_interval with target_coverage probability
        if np.random.random() < target_coverage:
            # Generate score within interval
            actual_score = np.random.uniform(lower, upper)
        else:
            # Generate score outside interval
            if np.random.random() < 0.5:
                actual_score = np.random.uniform(upper, upper + width)
            else:
                actual_score = np.random.uniform(lower - width, lower)
        
        validation_data.append({
            'quarter': quarter,
            'predicted_score': pred_score,
            'lower_bound': lower,
            'upper_bound': upper,
            'interval_width': width,
            'actual_score': actual_score,
            'in_interval': (lower <= actual_score <= upper),
            'error': abs(pred_score - actual_score),
            'score_diff': score_diff,
            'momentum': momentum
        })
    
    return pd.DataFrame(validation_data)


def run_calibration_test(uncertainty_estimator) -> Dict[str, Any]:
    """
    Run a complete test of the calibration system.
    
    This function generates synthetic data, validates intervals, 
    calibrates the width factors, and visualizes the results.
    
    Args:
        uncertainty_estimator: An instance of UncertaintyEstimator
        
    Returns:
        Dict with test results including calibrated factors and metrics
    """
    # Step 1: Generate test validation data
    print("Generating synthetic validation data...")
    validation_data = generate_test_calibration_data(
        uncertainty_estimator, num_samples=200, target_coverage=0.9)
    
    # Calculate initial coverage
    initial_coverage = validation_data['in_interval'].mean()
    print(f"Initial overall coverage: {initial_coverage:.2%}")
    
    # Coverage by quarter
    quarter_coverage = validation_data.groupby('quarter')['in_interval'].mean()
    print("\nCoverage by quarter:")
    for quarter, coverage in quarter_coverage.items():
        print(f"Quarter {quarter}: {coverage:.2%}")
    
    # Step 2: Run calibration
    print("\nCalibrating width factors...")
    initial_factors = uncertainty_estimator.width_factors.copy()
    calibrated_factors = calibrate_confidence_intervals(
        validation_data, initial_factors)
    
    print("\nCalibration Results:")
    print("Quarter | Initial Factor | Calibrated Factor | Change")
    print("--------|----------------|-------------------|-------")
    for quarter in range(5):
        initial = initial_factors.get(quarter, 1.0)
        calibrated = calibrated_factors.get(quarter, 1.0)
        change = calibrated - initial
        print(f"   {quarter}    |      {initial:.2f}      |       {calibrated:.2f}        | {change:+.2f}")
    
    # Step 3: Visualize calibration
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot coverage by quarter
    ax.bar(
        [f"Q{q}" for q in range(5)],
        [quarter_coverage.get(q, 0) for q in range(5)],
        alpha=0.6,
        label="Initial Coverage"
    )
    
    # Add target line
    ax.axhline(y=0.9, color='red', linestyle='--', label="Target (90%)")
    
    # Customize plot
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Coverage Rate")
    ax.set_xlabel("Quarter")
    ax.set_title("Confidence Interval Coverage by Quarter")
    ax.grid(axis='y', alpha=0.3)
    ax.legend()
    
    plt.tight_layout()
    
    return {
        'validation_data': validation_data,
        'initial_factors': initial_factors,
        'calibrated_factors': calibrated_factors,
        'initial_coverage': initial_coverage,
        'quarter_coverage': quarter_coverage,
        'visualization': fig
    }


def integrate_with_main_pipeline(game_predictions: pd.DataFrame,
                              actual_results: pd.DataFrame,
                              uncertainty_estimator,
                              apply_calibration: bool = True) -> Dict[str, Any]:
    """
    Integrate the calibration system with the main prediction pipeline.
    
    This function combines validation, calibration, and application of 
    uncertainty estimates within the main prediction workflow.
    
    Args:
        game_predictions: DataFrame with game predictions
        actual_results: DataFrame with actual game results
        uncertainty_estimator: Instance of UncertaintyEstimator
        apply_calibration: Whether to apply calibration updates (default: True)
        
    Returns:
        Dict with processing results and updated predictions
    """
    # Step 1: Apply current uncertainty estimates
    predictions_with_intervals = uncertainty_estimator.apply_intervals_to_predictions(
        game_predictions)
    
    # Step 2: Validate intervals against actuals
    validation_metrics = validate_confidence_intervals(
        predictions_with_intervals, actual_results)
    
    results = {
        'predictions': predictions_with_intervals,
        'validation_metrics': validation_metrics,
        'width_factors': uncertainty_estimator.width_factors.copy()
    }
    
    # Skip calibration if requested or if no validation data
    if not apply_calibration or validation_metrics.empty:
        return results
    
    # Step 3: Update calibration if we have actuals
    # Prepare data in format expected by calibrate_confidence_intervals
    validation_data = []
    
    # Merge actuals with predictions
    merged = pd.merge(
        predictions_with_intervals,
        actual_results[['game_id', 'home_score', 'away_score']],
        on='game_id'
    )
    
    if not merged.empty:
        # Process home scores
        home_data = merged[['current_quarter', 'home_lower_bound', 
                          'home_upper_bound', 'home_score']].copy()
        home_data.columns = ['quarter', 'lower_bound', 'upper_bound', 'actual_score']
        validation_data.append(home_data)
        
        # Process away scores
        away_data = merged[['current_quarter', 'away_lower_bound', 
                          'away_upper_bound', 'away_score']].copy()
        away_data.columns = ['quarter', 'lower_bound', 'upper_bound', 'actual_score']
        validation_data.append(away_data)
        
        # Combine and calibrate
        validation_df = pd.concat(validation_data, ignore_index=True)
        
        calibrated_factors = calibrate_confidence_intervals(
            validation_df, uncertainty_estimator.width_factors)
        
        # Update estimator with new factors
        if apply_calibration:
            uncertainty_estimator.width_factors = calibrated_factors
        
        # Record calibration changes
        results['calibrated_factors'] = calibrated_factors
        results['calibration_change'] = {
            q: calibrated_factors.get(q, 0) - results['width_factors'].get(q, 0)
            for q in range(5)
        }
        
        # Re-apply with calibrated factors if needed
        if apply_calibration:
            results['calibrated_predictions'] = uncertainty_estimator.apply_intervals_to_predictions(
                game_predictions)
    
    return results


# Example usage with the UncertaintyEstimator from Cell 27
if 'UncertaintyEstimator' in globals():
    # Create an estimator instance
    test_estimator = UncertaintyEstimator()
    
    # Run calibration test with synthetic data
    print("Running calibration test with synthetic data...")
    test_results = run_calibration_test(test_estimator)
    
    # Display results
    if 'get_ipython' in globals():
        # In Jupyter notebook, show the plot
        from IPython.display import display
        display(test_results['visualization'])
    
    print("\nSynthetic test complete. The calibration system is ready for integration.")


In [ ]:
# Cell 27B: Interactive Confidence Visualization for PWA Integration

import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Optional, Union, Any
from datetime import datetime

class InteractiveConfidenceDisplay:
    """
    Class for creating interactive visualizations of prediction confidence
    optimized for Progressive Web App (PWA) integration.
    
    This class provides methods to generate SVG-based visualizations
    in different styles for displaying NBA score predictions with
    confidence intervals.
    """
    
    def __init__(self):
        """Initialize the display generator with default settings."""
        # Quarter-specific colors for visual differentiation
        self.quarter_colors = {
            0: "#6c757d",  # Gray for pregame
            1: "#20c997",  # Teal for Q1
            2: "#0dcaf0",  # Cyan for Q2
            3: "#0d6efd",  # Blue for Q3
            4: "#6610f2"   # Purple for Q4
        }
        
        # Color schemes for teams and UI elements
        self.color_scheme = {
            'home_team': "#0066cc",
            'away_team': "#cc0000",
            'background': "#f8f9fa",
            'win_positive': "#198754",  # Green for positive win probability
            'win_negative': "#dc3545",  # Red for negative win probability
            'win_neutral': "#fd7e14",   # Orange for neutral/even win probability
            'grid': "#dddddd",
            'text': "#212529",
            'highlight': "#4CAF50"
        }
    
    def create_display(self, 
                      prediction_data: Dict[str, Any], 
                      design_style: str = 'pwa') -> str:
        """
        Create an interactive SVG visualization showing prediction confidence.
        
        Args:
            prediction_data: Dict with prediction details including team names,
                           scores, predictions, and confidence intervals
            design_style: Design style to use:
                        - 'minimal': Simple, clean visualization
                        - 'detailed': More detailed visualization with additional information
                        - 'pwa': Optimized for mobile Progressive Web App display
        
        Returns:
            str: SVG/HTML markup for the confidence display
        """
        # Validate inputs
        if not isinstance(prediction_data, dict):
            raise ValueError("Prediction data must be a dictionary")
        
        # Extract prediction data with fallbacks for missing values
        home_team = prediction_data.get('home_team', 'Home')
        away_team = prediction_data.get('away_team', 'Away')
        current_quarter = int(prediction_data.get('current_quarter', 0))
        home_score = float(prediction_data.get('home_score', 0))
        away_score = float(prediction_data.get('away_score', 0))
        
        # Get predicted scores
        predicted_home = float(prediction_data.get('predicted_home_score', 
                                                 prediction_data.get('predicted_home_final', 0)))
        predicted_away = float(prediction_data.get('predicted_away_score', 
                                                 prediction_data.get('predicted_away_final', 0)))
        
        # Get confidence interval data with reasonable defaults
        home_lower = float(prediction_data.get('home_lower_bound', predicted_home * 0.9))
        home_upper = float(prediction_data.get('home_upper_bound', predicted_home * 1.1))
        away_lower = float(prediction_data.get('away_lower_bound', predicted_away * 0.9))
        away_upper = float(prediction_data.get('away_upper_bound', predicted_away * 1.1))
        
        # Win probability
        win_prob = float(prediction_data.get('win_probability', 0.5))
        
        # Choose the appropriate visualization style
        if design_style == 'minimal':
            return self._create_minimal_display(
                home_team, away_team, current_quarter,
                home_score, away_score,
                predicted_home, predicted_away,
                home_lower, home_upper,
                away_lower, away_upper,
                win_prob
            )
        elif design_style == 'detailed':
            return self._create_detailed_display(
                home_team, away_team, current_quarter,
                home_score, away_score,
                predicted_home, predicted_away,
                home_lower, home_upper,
                away_lower, away_upper,
                win_prob
            )
        elif design_style == 'pwa':
            return self._create_pwa_display(
                home_team, away_team, current_quarter,
                home_score, away_score,
                predicted_home, predicted_away,
                home_lower, home_upper,
                away_lower, away_upper,
                win_prob
            )
        else:
            # Default to PWA style if unrecognized style is specified
            return self._create_pwa_display(
                home_team, away_team, current_quarter,
                home_score, away_score,
                predicted_home, predicted_away,
                home_lower, home_upper,
                away_lower, away_upper,
                win_prob
            )
    
    def _create_minimal_display(self,
                              home_team: str,
                              away_team: str,
                              current_quarter: int,
                              home_score: float,
                              away_score: float,
                              predicted_home: float,
                              predicted_away: float,
                              home_lower: float,
                              home_upper: float,
                              away_lower: float,
                              away_upper: float,
                              win_prob: float) -> str:
        """
        Create a clean, minimal SVG visualization with just the essentials.
        
        This version is optimized for simple embedding and smaller screen sizes.
        
        Args:
            Various parameters describing the game state and predictions
            
        Returns:
            str: SVG markup for the minimal visualization
        """
        # SVG dimensions
        svg_width = 400
        svg_height = 200
        
        # Calculate position and scaling - ensure we have room for all scores
        max_score = max(home_upper, away_upper, 130)  # Minimum bound of 130 points
        y_scale = (svg_height - 60) / max_score
        
        # Generate SVG
        svg = f"""
        <svg width="100%" height="100%" viewBox="0 0 {svg_width} {svg_height}" 
             xmlns="http://www.w3.org/2000/svg" style="max-width:400px;" class="confidence-display minimal">
            <!-- Title -->
            <text x="{svg_width/2}" y="20" text-anchor="middle" font-family="Arial" font-size="16" font-weight="bold">
                {f"Quarter {current_quarter}" if current_quarter > 0 else "Pre-Game"} Prediction
            </text>
            
            <!-- Team names -->
            <text x="100" y="50" text-anchor="middle" font-family="Arial" font-size="14" fill="{self.color_scheme['home_team']}">
                {home_team}
            </text>
            <text x="{svg_width-100}" y="50" text-anchor="middle" font-family="Arial" font-size="14" fill="{self.color_scheme['away_team']}">
                {away_team}
            </text>
            
            <!-- Current scores -->
            <text x="100" y="75" text-anchor="middle" font-family="Arial" font-size="18" font-weight="bold">
                {home_score:.0f}
            </text>
            <text x="{svg_width-100}" y="75" text-anchor="middle" font-family="Arial" font-size="18" font-weight="bold">
                {away_score:.0f}
            </text>
            
            <!-- Confidence intervals -->
            <rect x="80" y="{svg_height - (home_upper * y_scale)}" width="40" height="{(home_upper - home_lower) * y_scale}" 
                  fill="{self.color_scheme['home_team']}" fill-opacity="0.3" />
            <rect x="{svg_width-120}" y="{svg_height - (away_upper * y_scale)}" width="40" height="{(away_upper - away_lower) * y_scale}" 
                  fill="{self.color_scheme['away_team']}" fill-opacity="0.3" />
            
            <!-- Predicted scores -->
            <line x1="80" x2="120" y1="{svg_height - (predicted_home * y_scale)}" y2="{svg_height - (predicted_home * y_scale)}" 
                  stroke="{self.color_scheme['home_team']}" stroke-width="2" />
            <line x1="{svg_width-120}" x2="{svg_width-80}" y1="{svg_height - (predicted_away * y_scale)}" y2="{svg_height - (predicted_away * y_scale)}" 
                  stroke="{self.color_scheme['away_team']}" stroke-width="2" />
            
            <!-- Predicted score labels -->
            <text x="100" y="{svg_height - (predicted_home * y_scale) - 5}" text-anchor="middle" font-family="Arial" font-size="12">
                {predicted_home:.1f}
            </text>
            <text x="{svg_width-100}" y="{svg_height - (predicted_away * y_scale) - 5}" text-anchor="middle" font-family="Arial" font-size="12">
                {predicted_away:.1f}
            </text>
            
            <!-- Win probability indicator -->
            <rect x="{svg_width/2 - 75}" y="{svg_height - 25}" width="150" height="15" fill="#eeeeee" rx="7" ry="7" />
            <rect x="{svg_width/2 - 75}" y="{svg_height - 25}" width="{150 * win_prob}" height="15" 
                  fill="{self._get_win_probability_color(win_prob)}" rx="7" ry="7" />
            <text x="{svg_width/2}" y="{svg_height - 12}" text-anchor="middle" font-family="Arial" font-size="10">
                {home_team} Win: {win_prob*100:.1f}%
            </text>
        </svg>
        """
        
        return svg
    
    def _create_detailed_display(self,
                               home_team: str,
                               away_team: str,
                               current_quarter: int,
                               home_score: float,
                               away_score: float,
                               predicted_home: float,
                               predicted_away: float,
                               home_lower: float,
                               home_upper: float,
                               away_lower: float,
                               away_upper: float,
                               win_prob: float) -> str:
        """
        Create a more detailed SVG with additional information and interactive elements.
        
        This version includes axis labels, range indicators, and detailed explanatory text.
        
        Args:
            Various parameters describing the game state and predictions
            
        Returns:
            str: SVG markup for the detailed visualization
        """
        # SVG dimensions
        svg_width = 500
        svg_height = 300
        
        # Calculate position and scaling
        max_score = max(home_upper, away_upper, 130)
        y_scale = (svg_height - 100) / max_score
        
        # Create tooltips for key elements
        home_tooltip = f"Predicted: {predicted_home:.1f} (range: {home_lower:.1f}-{home_upper:.1f})"
        away_tooltip = f"Predicted: {predicted_away:.1f} (range: {away_lower:.1f}-{away_upper:.1f})"
        
        # Calculate interval widths for display
        home_width = home_upper - home_lower
        away_width = away_upper - away_lower
        
        # Generate SVG with more details and tooltip elements
        svg = f"""
        <svg width="100%" height="100%" viewBox="0 0 {svg_width} {svg_height}" 
             xmlns="http://www.w3.org/2000/svg" style="max-width:600px;" class="confidence-display detailed">
            <!-- Title and Game Info -->
            <text x="{svg_width/2}" y="25" text-anchor="middle" font-family="Arial" font-size="18" font-weight="bold">
                {f"Quarter {current_quarter}" if current_quarter > 0 else "Pre-Game"} Prediction
            </text>
            <text x="{svg_width/2}" y="45" text-anchor="middle" font-family="Arial" font-size="14">
                {home_team} vs {away_team}
            </text>
            
            <!-- Background grid lines -->
            <g stroke="{self.color_scheme['grid']}" stroke-width="1">
                <line x1="50" y1="80" x2="{svg_width-50}" y2="80" />
                <line x1="50" y1="130" x2="{svg_width-50}" y2="130" />
                <line x1="50" y1="180" x2="{svg_width-50}" y2="180" />
                <line x1="50" y1="230" x2="{svg_width-50}" y2="230" />
            </g>
            
            <!-- Score axis labels -->
            <text x="40" y="80" text-anchor="end" font-family="Arial" font-size="10">{int(svg_height/y_scale)}</text>
            <text x="40" y="130" text-anchor="end" font-family="Arial" font-size="10">{int((svg_height-50)/y_scale)}</text>
            <text x="40" y="180" text-anchor="end" font-family="Arial" font-size="10">{int((svg_height-100)/y_scale)}</text>
            <text x="40" y="230" text-anchor="end" font-family="Arial" font-size="10">{int((svg_height-150)/y_scale)}</text>
            
            <!-- Team columns with confidence intervals -->
            <!-- Home team -->
            <g class="home-team">
                <text x="150" y="70" text-anchor="middle" font-family="Arial" font-size="16" font-weight="bold" fill="{self.color_scheme['home_team']}">
                    {home_team}
                </text>
                <text x="150" y="90" text-anchor="middle" font-family="Arial" font-size="20" font-weight="bold">
                    {home_score:.0f}
                </text>
                
                <!-- Confidence interval -->
                <rect x="130" y="{svg_height - (home_upper * y_scale)}" width="40" height="{(home_upper - home_lower) * y_scale}" 
                      fill="{self.color_scheme['home_team']}" fill-opacity="0.3">
                    <title>{home_tooltip}</title>
                </rect>
                
                <!-- Predicted score marker -->
                <line x1="130" x2="170" y1="{svg_height - (predicted_home * y_scale)}" y2="{svg_height - (predicted_home * y_scale)}" 
                      stroke="{self.color_scheme['home_team']}" stroke-width="3" />
                
                <!-- Predicted score label -->
                <text x="150" y="{svg_height - (predicted_home * y_scale) - 10}" text-anchor="middle" 
                      font-family="Arial" font-size="14" font-weight="bold">
                    {predicted_home:.1f}
                </text>
                
                <!-- Range indicators -->
                <text x="180" y="{svg_height - (home_upper * y_scale)}" text-anchor="start" font-family="Arial" font-size="12">
                    ↑ {home_upper:.1f}
                </text>
                <text x="180" y="{svg_height - (home_lower * y_scale)}" text-anchor="start" font-family="Arial" font-size="12">
                    ↓ {home_lower:.1f}
                </text>
                
                <!-- Confidence width -->
                <text x="180" y="{svg_height - ((home_upper + home_lower) * y_scale/2)}" text-anchor="start" 
                      font-family="Arial" font-size="10" fill="#666666">
                    ±{home_width/2:.1f}
                </text>
            </g>
            
            <!-- Away team -->
            <g class="away-team">
                <text x="350" y="70" text-anchor="middle" font-family="Arial" font-size="16" font-weight="bold" fill="{self.color_scheme['away_team']}">
                    {away_team}
                </text>
                <text x="350" y="90" text-anchor="middle" font-family="Arial" font-size="20" font-weight="bold">
                    {away_score:.0f}
                </text>
                
                <!-- Confidence interval -->
                <rect x="330" y="{svg_height - (away_upper * y_scale)}" width="40" height="{(away_upper - away_lower) * y_scale}" 
                      fill="{self.color_scheme['away_team']}" fill-opacity="0.3">
                    <title>{away_tooltip}</title>
                </rect>
                
                <!-- Predicted score marker -->
                <line x1="330" x2="370" y1="{svg_height - (predicted_away * y_scale)}" y2="{svg_height - (predicted_away * y_scale)}" 
                      stroke="{self.color_scheme['away_team']}" stroke-width="3" />
                
                <!-- Predicted score label -->
                <text x="350" y="{svg_height - (predicted_away * y_scale) - 10}" text-anchor="middle" 
                      font-family="Arial" font-size="14" font-weight="bold">
                    {predicted_away:.1f}
                </text>
                
                <!-- Range indicators -->
                <text x="320" y="{svg_height - (away_upper * y_scale)}" text-anchor="end" font-family="Arial" font-size="12">
                    ↑ {away_upper:.1f}
                </text>
                <text x="320" y="{svg_height - (away_lower * y_scale)}" text-anchor="end" font-family="Arial" font-size="12">
                    ↓ {away_lower:.1f}
                </text>
                
                <!-- Confidence width -->
                <text x="320" y="{svg_height - ((away_upper + away_lower) * y_scale/2)}" text-anchor="end" 
                      font-family="Arial" font-size="10" fill="#666666">
                    ±{away_width/2:.1f}
                </text>
            </g>
            
            <!-- Win probability gauge -->
            <g class="win-probability">
                <text x="{svg_width/2}" y="{svg_height-50}" text-anchor="middle" font-family="Arial" font-size="14" font-weight="bold">
                    Win Probability
                </text>
                
                <!-- Probability bar -->
                <rect x="{svg_width/2 - 100}" y="{svg_height-40}" width="200" height="20" fill="#eeeeee" rx="10" ry="10" />
                <rect x="{svg_width/2 - 100}" y="{svg_height-40}" width="{200 * win_prob}" height="20" 
                      fill="{self._get_win_probability_color(win_prob)}" rx="10" ry="10">
                    <title>Win probability: {win_prob*100:.1f}%</title>
                </rect>
                
                <!-- Home team marker -->
                <text x="{svg_width/2 - 110}" y="{svg_height-30}" text-anchor="end" font-family="Arial" font-size="12">
                    {home_team}
                </text>
                
                <!-- Away team marker -->
                <text x="{svg_width/2 + 110}" y="{svg_height-30}" text-anchor="start" font-family="Arial" font-size="12">
                    {away_team}
                </text>
                
                <!-- Probability percentage -->
                <text x="{svg_width/2}" y="{svg_height-30}" text-anchor="middle" font-family="Arial" font-size="14" font-weight="bold">
                    {win_prob*100:.1f}%
                </text>
            </g>
            
            <!-- Confidence level explanation -->
            <text x="{svg_width/2}" y="{svg_height-10}" text-anchor="middle" font-family="Arial" font-size="10" fill="#666666">
                Shaded areas show 90% confidence intervals for final score predictions
            </text>
        </svg>
        """
        
        return svg
    
    def _create_pwa_display(self,
                          home_team: str,
                          away_team: str,
                          current_quarter: int,
                          home_score: float,
                          away_score: float,
                          predicted_home: float,
                          predicted_away: float,
                          home_lower: float,
                          home_upper: float,
                          away_lower: float,
                          away_upper: float,
                          win_prob: float) -> str:
        """
        Create a mobile-optimized visualization for Progressive Web App.
        
        This version is fully responsive, has touch-friendly elements, and 
        uses a modern design system optimized for mobile devices.
        
        Args:
            Various parameters describing the game state and predictions
            
        Returns:
            str: SVG markup for the PWA-optimized visualization
        """
        # SVG dimensions - mobile optimized
        svg_width = 360  # Standard mobile width
        svg_height = 240
        
        # Get quarter-specific color
        quarter_color = self.quarter_colors.get(current_quarter, self.quarter_colors[0])
        
        # Calculate position and scaling
        max_score = max(home_upper, away_upper, 130)
        y_scale = (svg_height - 80) / max_score
        
        # Calculate win probability styling
        win_color = self._get_win_probability_color(win_prob)
        
        # Determine status text based on win probability
        home_win_text = self._get_win_probability_text(win_prob)
        
        # Create detailed tooltips
        home_tooltip = f"Home prediction: {predicted_home:.1f} points\nConfidence range: {home_lower:.1f} to {home_upper:.1f}\nCurrent score: {home_score}"
        away_tooltip = f"Away prediction: {predicted_away:.1f} points\nConfidence range: {away_lower:.1f} to {away_upper:.1f}\nCurrent score: {away_score}"
        win_tooltip = f"Win probability: {win_prob*100:.1f}%\nBased on score projection and historical patterns"
        
        # Generate responsive SVG optimized for mobile PWA
        svg = f"""
        <svg width="100%" height="100%" viewBox="0 0 {svg_width} {svg_height}" 
             xmlns="http://www.w3.org/2000/svg" style="max-width:600px;" 
             class="prediction-chart" data-quarter="{current_quarter}">
            <!-- Main Container -->
            <rect x="0" y="0" width="{svg_width}" height="{svg_height}" fill="{self.color_scheme['background']}" rx="10" ry="10" />
            
            <!-- Header with Game Status -->
            <rect x="0" y="0" width="{svg_width}" height="40" fill="{quarter_color}" rx="10" ry="10" />
            <text x="{svg_width/2}" y="25" text-anchor="middle" font-family="system-ui" font-size="16" fill="white" font-weight="bold">
                {f"QUARTER {current_quarter}" if current_quarter > 0 else "PRE-GAME"} PREDICTION
            </text>
            
            <!-- Team Header -->
            <text x="80" y="60" text-anchor="middle" font-family="system-ui" font-size="14" font-weight="bold">
                {home_team}
            </text>
            <text x="{svg_width-80}" y="60" text-anchor="middle" font-family="system-ui" font-size="14" font-weight="bold">
                {away_team}
            </text>
            
            <!-- Current Score -->
            <text x="80" y="80" text-anchor="middle" font-family="system-ui" font-size="18" font-weight="bold">
                {home_score:.0f}
            </text>
            <text x="{svg_width-80}" y="80" text-anchor="middle" font-family="system-ui" font-size="18" font-weight="bold">
                {away_score:.0f}
            </text>
            
            <!-- VS divider -->
            <text x="{svg_width/2}" y="80" text-anchor="middle" font-family="system-ui" font-size="16">
                VS
            </text>
            
            <!-- Confidence Intervals with Modern Styling and Tooltips-->
            <g class="home-interval-group">
                <rect x="60" y="{svg_height - (home_upper * y_scale)}" width="40" height="{(home_upper - home_lower) * y_scale}" 
                      fill="{quarter_color}" fill-opacity="0.25" rx="5" ry="5" class="interval home-interval">
                    <title>{home_tooltip}</title>
                </rect>
                
                <!-- Interactive elements -->
                <rect x="50" y="{svg_height - (home_upper * y_scale) - 10}" width="60" height="{(home_upper - home_lower) * y_scale + 20}" 
                      fill="transparent" class="interval-hover home-hover" style="cursor:pointer" />
            </g>
            
            <g class="away-interval-group">
                <rect x="{svg_width-100}" y="{svg_height - (away_upper * y_scale)}" width="40" height="{(away_upper - away_lower) * y_scale}" 
                      fill="{quarter_color}" fill-opacity="0.25" rx="5" ry="5" class="interval away-interval">
                    <title>{away_tooltip}</title>
                </rect>
                
                <!-- Interactive elements -->
                <rect x="{svg_width-110}" y="{svg_height - (away_upper * y_scale) - 10}" width="60" height="{(away_upper - away_lower) * y_scale + 20}" 
                      fill="transparent" class="interval-hover away-hover" style="cursor:pointer" />
            </g>
            
            <!-- Predicted Final Scores -->
            <line x1="40" x2="120" y1="{svg_height - (predicted_home * y_scale)}" y2="{svg_height - (predicted_home * y_scale)}" 
                  stroke="{quarter_color}" stroke-width="2.5" stroke-dasharray="2" />
                  
            <line x1="{svg_width-120}" x2="{svg_width-40}" y1="{svg_height - (predicted_away * y_scale)}" y2="{svg_height - (predicted_away * y_scale)}" 
                  stroke="{quarter_color}" stroke-width="2.5" stroke-dasharray="2" />
            
            <!-- Prediction Labels -->
            <text x="80" y="{svg_height - (predicted_home * y_scale) - 8}" text-anchor="middle" font-family="system-ui" font-size="15" font-weight="bold">
                {predicted_home:.1f}
            </text>
            
            <text x="{svg_width-80}" y="{svg_height - (predicted_away * y_scale) - 8}" text-anchor="middle" font-family="system-ui" font-size="15" font-weight="bold">
                {predicted_away:.1f}
            </text>
            
            <!-- Divider line -->
            <line x1="20" y1="{svg_height-45}" x2="{svg_width-20}" y2="{svg_height-45}" stroke="#dee2e6" stroke-width="1" />
            
            <!-- Win Probability Section -->
            <text x="20" y="{svg_height-25}" text-anchor="start" font-family="system-ui" font-size="12">
                {home_team} WIN PROBABILITY: 
            </text>
            
            <!-- Modern Progress Bar with Tooltip -->
            <g class="win-probability-group">
                <rect x="20" y="{svg_height-15}" width="{svg_width-40}" height="8" fill="#e9ecef" rx="4" ry="4" />
                <rect x="20" y="{svg_height-15}" width="{(svg_width-40) * win_prob}" height="8" fill="{win_color}" rx="4" ry="4">
                    <title>{win_tooltip}</title>
                </rect>
                
                <!-- Interactive overlay -->
                <rect x="20" y="{svg_height-25}" width="{svg_width-40}" height="20" fill="transparent" style="cursor:pointer" />
            </g>
            
            <!-- Win Percentage -->
            <text x="{svg_width-20}" y="{svg_height-25}" text-anchor="end" font-family="system-ui" font-size="14" font-weight="bold" fill="{win_color}">
                {win_prob*100:.1f}% {home_win_text}
            </text>
            
            <!-- Add interactive JS capabilities -->
            <script type="text/javascript"><![CDATA[
            (function() {{
                // Add hover effects for intervals
                const intervalHovers = document.querySelectorAll('.interval-hover');
                intervalHovers.forEach(hover => {{
                    hover.addEventListener('mouseover', function() {{
                        const interval = this.previousElementSibling;
                        interval.setAttribute('fill-opacity', '0.5');
                    }});
                    hover.addEventListener('mouseout', function() {{
                        const interval = this.previousElementSibling;
                        interval.setAttribute('fill-opacity', '0.25');
                    }});
                    
                    // Mobile touch events
                    hover.addEventListener('touchstart', function() {{
                        const interval = this.previousElementSibling;
                        interval.setAttribute('fill-opacity', '0.5');
                    }});
                    hover.addEventListener('touchend', function() {{
                        const interval = this.previousElementSibling;
                        interval.setAttribute('fill-opacity', '0.25');
                    }});
                }});
            }})();
            ]]></script>
        </svg>
        """
        
        return svg
    
    def _get_win_probability_color(self, win_prob: float) -> str:
        """Get the appropriate color for a win probability value."""
        if win_prob > 0.65:
            return self.color_scheme['win_positive']  # Strong favorite
        elif win_prob > 0.45:
            return self.color_scheme['win_neutral']   # Even match
        else:
            return self.color_scheme['win_negative']  # Underdog
    
    def _get_win_probability_text(self, win_prob: float) -> str:
        """Get descriptive text for win probability status."""
        if win_prob > 0.65:
            return "FAVORED"
        elif win_prob > 0.45:
            return "EVEN"
        else:
            return "UNDERDOG"


class PWAConfidenceDisplay:
    """
    Class for creating Progressive Web App (PWA) visualizations and components
    for displaying prediction confidence in responsive, mobile-friendly formats.
    
    This class builds on the InteractiveConfidenceDisplay to create complete
    PWA-ready components with responsive design and touch interactions.
    """
    
    def __init__(self):
        """Initialize the PWA display component generator."""
        self.confidence_display = InteractiveConfidenceDisplay()
    
    def create_game_card(self, game_data: Dict[str, Any]) -> str:
        """
        Create a complete game card component with predictions and confidence.
        
        Args:
            game_data: Dict with game information and predictions
            
        Returns:
            str: HTML component for the game card
        """
        # Extract basic game info
        game_id = game_data.get('game_id', '0')
        home_team = game_data.get('home_team', 'Home')
        away_team = game_data.get('away_team', 'Away')
        current_quarter = game_data.get('current_quarter', 0)
        game_status = game_data.get('game_status', 'SCHEDULED')
        
        # Generate the SVG visualization
        svg_visualization = self.confidence_display.create_display(game_data, 'pwa')
        
        # Create the complete card with responsive design
        card_html = f"""
        <div class="game-prediction-card" data-game-id="{game_id}" data-quarter="{current_quarter}">
            <style>
                .game-prediction-card {{
                    font-family: system-ui, -apple-system, "Segoe UI", Roboto, "Helvetica Neue";
                    border-radius: 12px;
                    overflow: hidden;
                    box-shadow: 0 2px 8px rgba(0,0,0,0.1);
                    margin-bottom: 16px;
                    background-color: #fff;
                    width: 100%;
                    max-width: 600px;
                }}
                
                .game-header {{
                    display: flex;
                    justify-content: space-between;
                    padding: 12px 16px;
                    border-bottom: 1px solid #dee2e6;
                }}
                
                .game-teams {{
                    font-weight: bold;
                    font-size: 1.1rem;
                }}
                
                .game-status {{
                    font-size: 0.9rem;
                    padding: 4px 8px;
                    border-radius: 4px;
                    font-weight: 500;
                    background-color: #6c757d;
                    color: white;
                }}
                
                .status-live {{
                    background-color: #dc3545;
                }}
                
                .status-final {{
                    background-color: #198754;
                }}
                
                .game-visualization {{
                    padding: 8px;
                }}
                
                /* Responsive adjustments */
                @media (max-width: 480px) {{
                    .game-teams {{
                        font-size: 1rem;
                    }}
                    
                    .game-status {{
                        font-size: 0.8rem;
                    }}
                }}
            </style>
            
            <div class="game-header">
                <div class="game-teams">{home_team} vs {away_team}</div>
                <div class="game-status {self._get_status_class(game_status)}">
                    {self._format_game_status(game_status, current_quarter)}
                </div>
            </div>
            
            <div class="game-visualization">
                {svg_visualization}
            </div>
        </div>
        """
        
        return card_html
    
    def create_game_dashboard(self, games_data: List[Dict[str, Any]]) -> str:
        """
        Create a complete dashboard of game predictions for the PWA.
        
        Args:
            games_data: List of game data dictionaries
            
        Returns:
            str: Complete HTML for the dashboard
        """
        # Start the dashboard HTML
        dashboard_html = """
        <div class="prediction-dashboard">
            <style>
                .prediction-dashboard {
                    font-family: system-ui, -apple-system, "Segoe UI", Roboto, "Helvetica Neue";
                    max-width: 1200px;
                    margin: 0 auto;
                    padding: 16px;
                }
                
                .dashboard-header {
                    text-align: center;
                    margin-bottom: 24px;
                }
                
                .dashboard-title {
                    font-size: 1.5rem;
                    font-weight: bold;
                    margin-bottom: 8px;
                }
                
                .dashboard-subtitle {
                    font-size: 1rem;
                    color: #6c757d;
                }
                
                .games-grid {
                    display: grid;
                    grid-template-columns: repeat(auto-fill, minmax(300px, 1fr));
                    gap: 16px;
                }
                
                /* Responsive adjustments */
                @media (max-width: 768px) {
                    .games-grid {
                        grid-template-columns: 1fr;
                    }
                }
            </style>
            
            <div class="dashboard-header">
                <div class="dashboard-title">NBA Score Predictions</div>
                <div class="dashboard-subtitle">Updated: """ + datetime.now().strftime("%Y-%m-%d %H:%M:%S") + """</div>
            </div>
            
            <div class="games-grid">
        """
        
        # Add game cards to dashboard
        for game_data in games_data:
            game_card = self.create_game_card(game_data)
            dashboard_html += game_card
        
        # Close the dashboard HTML
        dashboard_html += """
            </div>
        </div>
        """
        
        return dashboard_html
    
    def _get_status_class(self, status: str) -> str:
        """Get the CSS class for a game status."""
        status = status.upper()
        if status == 'LIVE':
            return 'status-live'
        elif status == 'FINAL':
            return 'status-final'
        else:
            return ''
    
    def _format_game_status(self, status: str, quarter: int) -> str:
        """Format the game status text."""
        status = status.upper()
        if status == 'LIVE':
            return f"LIVE Q{quarter}"
        elif status == 'FINAL':
            return "FINAL"
        else:
            return status


def create_interactive_confidence_display(prediction_data: Dict[str, Any], 
                                        design_style: str = 'pwa') -> str:
    """
    Create an interactive SVG visualization showing prediction confidence.
    
    This is a wrapper function that uses the InteractiveConfidenceDisplay class.
    
    Args:
        prediction_data: Dict with prediction details
        design_style: 'minimal', 'detailed', or 'pwa' for different visualization styles
        
    Returns:
        str: SVG/HTML markup for the confidence display
    """
    display_generator = InteractiveConfidenceDisplay()
    return display_generator.create_display(prediction_data, design_style)


def create_pwa_game_card(game_data: Dict[str, Any]) -> str:
    """
    Create a complete game card component for a PWA.
    
    This is a wrapper function that uses the PWAConfidenceDisplay class.
    
    Args:
        game_data: Dict with game information and predictions
        
    Returns:
        str: HTML component for the game card
    """
    pwa_display = PWAConfidenceDisplay()
    return pwa_display.create_game_card(game_data)


def generate_sample_prediction_data() -> Dict[str, Any]:
    """Generate sample prediction data for testing."""
    return {
        'game_id': 1001,
        'home_team': 'Lakers',
        'away_team': 'Celtics',
        'current_quarter': 2,
        'game_status': 'LIVE',
        'home_score': 54,
        'away_score': 51,
        'predicted_home_score': 112.5,
        'predicted_away_score': 104.8,
        'home_lower_bound': 105.2,
        'home_upper_bound': 119.8,
        'away_lower_bound': 97.5,
        'away_upper_bound': 112.1,
        'win_probability': 0.68
    }


# Example usage and testing
if __name__ == "__main__":
    # Generate sample prediction data
    sample_data = generate_sample_prediction_data()
    
    # Create display generator - renamed from 'display' to 'display_generator' to avoid naming conflicts
    display_generator = InteractiveConfidenceDisplay()
    
    # Test all styles
    for style in ['minimal', 'detailed', 'pwa']:
        print(f"\nTesting {style} style visualization:")
        svg = display_generator.create_display(sample_data, style)
        print(f"Generated {len(svg)} characters of SVG/HTML")
        
        # In a notebook, display the visualization
        if 'get_ipython' in globals():
            from IPython.display import HTML, display as ipython_display
            ipython_display(HTML(svg))
    
    # Test PWA components
    pwa_display = PWAConfidenceDisplay()
    game_card = pwa_display.create_game_card(sample_data)
    print(f"\nGenerated game card: {len(game_card)} characters")
    
    # Create a dashboard with multiple games
    games = [
        sample_data,
        {**sample_data, 'game_id': 1002, 'home_team': 'Warriors', 'away_team': 'Bucks', 
         'current_quarter': 4, 'win_probability': 0.51},
        {**sample_data, 'game_id': 1003, 'home_team': 'Heat', 'away_team': 'Nets', 
         'current_quarter': 0, 'game_status': 'SCHEDULED', 'win_probability': 0.45}
    ]
    
    dashboard = pwa_display.create_game_dashboard(games)
    print(f"\nGenerated dashboard: {len(dashboard)} characters")
    
    # In a notebook, display the dashboard
    if 'get_ipython' in globals():
        from IPython.display import HTML, display as ipython_display
        ipython_display(HTML(dashboard))

In [ ]:
# Cell 27C: Confidence Calibration

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional, Union, Any
from datetime import datetime
import json
import os

class CalibrationSystem:
    """
    A complete system for calibrating confidence intervals for NBA score predictions.
    
    This class handles the validation, calibration, and optimization of confidence
    interval width factors to ensure that interval coverage matches the desired
    target rate (e.g., 90% of actual values falling within predicted intervals).
    
    The system supports iterative calibration, visualization of results, and
    integration with the UncertaintyEstimator from Cell 27.
    """
    
    def __init__(self, 
                 initial_factors: Optional[Dict[int, float]] = None,
                 target_coverage: float = 0.9,
                 dampen_factor: float = 0.5,
                 min_factor: float = 0.5,
                 max_factor: float = 5.0):
        """
        Initialize the calibration system.
        
        Args:
            initial_factors: Starting width factors for different quarters. 
                          If None, uses default values.
            target_coverage: Target confidence interval coverage (default: 0.9 for 90%)
            dampen_factor: Factor to dampen adjustments (0-1, lower = more conservative)
            min_factor: Minimum allowed width factor
            max_factor: Maximum allowed width factor
        """
        # Default width factors if none provided
        if initial_factors is None:
            initial_factors = {
                0: 3.0,  # Pre-game: widest intervals
                1: 2.5,  # 1st quarter 
                2: 2.0,  # 2nd quarter
                3: 1.5,  # 3rd quarter
                4: 1.0   # 4th quarter: narrowest intervals
            }
        
        self.current_factors = initial_factors.copy()
        self.initial_factors = initial_factors.copy()
        self.target_coverage = target_coverage
        self.dampen_factor = dampen_factor
        self.min_factor = min_factor
        self.max_factor = max_factor
        
        # Track calibration history
        self.calibration_history = []
        
        # Track metrics
        self.metrics_history = []
    
    def calibrate_width_factors(self, 
                              validation_df: pd.DataFrame,
                              reset_to_initial: bool = False) -> Dict[int, float]:
        """
        Calibrate confidence interval width factors based on validation results.
        
        Args:
            validation_df: DataFrame with validation results containing:
                          - quarter: Game quarter
                          - in_interval: Boolean indicating if actual fell in prediction interval
            reset_to_initial: Whether to start from initial factors (default: False)
            
        Returns:
            Dict with updated width factors
        """
        # Reset to initial factors if requested
        if reset_to_initial:
            self.current_factors = self.initial_factors.copy()
        
        # Calculate actual coverage by quarter
        coverage_by_quarter = {}
        
        for quarter in range(5):  # 0-4
            quarter_data = validation_df[validation_df['quarter'] == quarter]
            if quarter_data.empty:
                continue
                
            # Calculate actual coverage rate
            if 'in_interval' in quarter_data.columns:
                coverage = quarter_data['in_interval'].mean()
            elif 'home_in_interval' in quarter_data.columns and 'away_in_interval' in quarter_data.columns:
                # If we have separate columns for home and away, calculate combined coverage
                both_in_interval = quarter_data['home_in_interval'] & quarter_data['away_in_interval']
                coverage = both_in_interval.mean()
            else:
                # Default to 0.5 if we can't determine coverage
                coverage = 0.5
                
            coverage_by_quarter[quarter] = coverage
        
        # Adjust factors
        updated_factors = self.current_factors.copy()
        adjustments = {}
        
        for quarter, coverage in coverage_by_quarter.items():
            if coverage < 0.01 or self.target_coverage == 0:
                # Avoid division by zero or very small values
                continue
                
            # Calculate adjustment ratio
            # If coverage is too low, increase factor; if too high, decrease factor
            adjustment_ratio = self.target_coverage / coverage
            
            # Apply adjustment with dampening to prevent overcorrection
            adjustment = (adjustment_ratio - 1.0) * self.dampen_factor
            updated_factors[quarter] = self.current_factors.get(quarter, 1.0) * (1.0 + adjustment)
            
            # Enforce bounds on width factors
            updated_factors[quarter] = max(min(updated_factors[quarter], self.max_factor), self.min_factor)
            
            # Record adjustment
            adjustments[quarter] = {
                'old_factor': self.current_factors.get(quarter, 1.0),
                'new_factor': updated_factors[quarter],
                'coverage': coverage,
                'adjustment_ratio': adjustment_ratio,
                'adjustment': adjustment
            }
        
        # Record calibration event
        self.calibration_history.append({
            'timestamp': datetime.now().isoformat(),
            'old_factors': self.current_factors.copy(),
            'new_factors': updated_factors.copy(),
            'coverage': coverage_by_quarter,
            'adjustments': adjustments,
            'target_coverage': self.target_coverage
        })
        
        # Update current factors
        self.current_factors = updated_factors.copy()
        
        return self.current_factors
    
    def validate_intervals(self, 
                         predictions_df: pd.DataFrame, 
                         actual_results_df: pd.DataFrame) -> pd.DataFrame:
        """
        Validate that confidence intervals contain actual values at the expected rate.
        
        Args:
            predictions_df: DataFrame with predictions and confidence intervals
            actual_results_df: DataFrame with actual final scores
        
        Returns:
            DataFrame with validation metrics
        """
        # Merge predictions with actual results
        merged = pd.merge(
            predictions_df,
            actual_results_df[['game_id', 'home_score', 'away_score']],
            on='game_id',
            how='inner'
        )
        
        if merged.empty:
            print("Warning: No matching games found between predictions and actuals")
            return pd.DataFrame()
        
        # Calculate validation metrics
        results = []
        for quarter in range(5):
            # Select data for this quarter
            quarter_data = merged[merged['current_quarter'] == quarter]
            if quarter_data.empty:
                continue
            
            # Check if actual scores fall within predicted intervals
            home_in_interval = ((quarter_data['home_score'] >= quarter_data['home_lower_bound']) & 
                              (quarter_data['home_score'] <= quarter_data['home_upper_bound']))
            
            away_in_interval = ((quarter_data['away_score'] >= quarter_data['away_lower_bound']) & 
                              (quarter_data['away_score'] <= quarter_data['away_upper_bound']))
            
            # Combined coverage (both home and away within intervals)
            both_in_interval = home_in_interval & away_in_interval
            
            # Calculate coverage rates
            home_coverage = home_in_interval.mean()
            away_coverage = away_in_interval.mean()
            combined_coverage = both_in_interval.mean()
            
            # Calculate interval widths
            home_width = (quarter_data['home_upper_bound'] - quarter_data['home_lower_bound']).mean()
            away_width = (quarter_data['away_upper_bound'] - quarter_data['away_lower_bound']).mean()
            avg_width = (home_width + away_width) / 2
            
            # Calculate errors
            if 'predicted_home_score' in quarter_data.columns and 'predicted_away_score' in quarter_data.columns:
                home_error = (quarter_data['predicted_home_score'] - quarter_data['home_score']).abs().mean()
                away_error = (quarter_data['predicted_away_score'] - quarter_data['away_score']).abs().mean()
                avg_error = (home_error + away_error) / 2
            else:
                home_error = away_error = avg_error = np.nan
            
            # Create result row
            results.append({
                'quarter': quarter,
                'home_coverage': home_coverage,
                'away_coverage': away_coverage,
                'combined_coverage': combined_coverage,
                'target_coverage': self.target_coverage,
                'coverage_error': combined_coverage - self.target_coverage,
                'home_interval_width': home_width,
                'away_interval_width': away_width,
                'avg_interval_width': avg_width,
                'home_error': home_error,
                'away_error': away_error,
                'avg_error': avg_error,
                'width_factor': self.current_factors.get(quarter, np.nan),
                'sample_size': len(quarter_data)
            })
        
        # Create DataFrame from results
        validation_df = pd.DataFrame(results)
        
        # Record metrics
        self.metrics_history.append({
            'timestamp': datetime.now().isoformat(),
            'validation_metrics': validation_df.to_dict(orient='records'),
            'current_factors': self.current_factors.copy()
        })
        
        return validation_df
    
    def prepare_validation_data(self, 
                              predictions_df: pd.DataFrame, 
                              actual_results_df: pd.DataFrame) -> pd.DataFrame:
        """
        Prepare validation data in the format needed for calibration.
        
        Args:
            predictions_df: DataFrame with predictions and intervals
            actual_results_df: DataFrame with actual results
            
        Returns:
            DataFrame with validation data ready for calibration
        """
        # Merge predictions with actuals
        merged = pd.merge(
            predictions_df,
            actual_results_df[['game_id', 'home_score', 'away_score']],
            on='game_id',
            how='inner'
        )
        
        if merged.empty:
            print("Warning: No matching games found for validation")
            return pd.DataFrame()
        
        # Process each row
        validation_rows = []
        
        for _, row in merged.iterrows():
            quarter = int(row.get('current_quarter', 0))
            
            # Check if home score is within interval
            home_in_interval = (row['home_score'] >= row['home_lower_bound'] and 
                              row['home_score'] <= row['home_upper_bound'])
            
            # Check if away score is within interval
            away_in_interval = (row['away_score'] >= row['away_lower_bound'] and 
                              row['away_score'] <= row['away_upper_bound'])
            
            # Add home row
            validation_rows.append({
                'game_id': row['game_id'],
                'quarter': quarter,
                'team_type': 'home',
                'predicted_score': row.get('predicted_home_score', np.nan),
                'actual_score': row['home_score'],
                'lower_bound': row['home_lower_bound'],
                'upper_bound': row['home_upper_bound'],
                'interval_width': row['home_upper_bound'] - row['home_lower_bound'],
                'in_interval': home_in_interval,
                'error': abs(row.get('predicted_home_score', row['home_score']) - row['home_score']),
                'home_in_interval': home_in_interval,
                'away_in_interval': away_in_interval,
                'both_in_interval': home_in_interval and away_in_interval
            })
            
            # Add away row
            validation_rows.append({
                'game_id': row['game_id'],
                'quarter': quarter,
                'team_type': 'away',
                'predicted_score': row.get('predicted_away_score', np.nan),
                'actual_score': row['away_score'],
                'lower_bound': row['away_lower_bound'],
                'upper_bound': row['away_upper_bound'],
                'interval_width': row['away_upper_bound'] - row['away_lower_bound'],
                'in_interval': away_in_interval,
                'error': abs(row.get('predicted_away_score', row['away_score']) - row['away_score']),
                'home_in_interval': home_in_interval,
                'away_in_interval': away_in_interval,
                'both_in_interval': home_in_interval and away_in_interval
            })
        
        return pd.DataFrame(validation_rows)
    
    def run_calibration_loop(self, 
                           predictions_history: List[pd.DataFrame], 
                           actual_results: pd.DataFrame,
                           iterations: int = 3,
                           reset_to_initial: bool = True,
                           apply_to_uncertainty_estimator: Optional[Any] = None) -> Dict[str, Any]:
        """
        Run an iterative calibration process to optimize confidence intervals.
        
        Args:
            predictions_history: List of DataFrames with historical predictions
            actual_results: DataFrame with actual game results
            iterations: Number of calibration iterations
            reset_to_initial: Whether to reset to initial factors before calibration
            apply_to_uncertainty_estimator: Optional UncertaintyEstimator to update
                                          with calibrated factors
            
        Returns:
            Dict with calibration results and metrics
        """
        # Reset to initial factors if requested
        if reset_to_initial:
            self.current_factors = self.initial_factors.copy()
            
        # Track iteration history
        iteration_history = []
        
        # Process iterations
        for iteration in range(iterations):
            print(f"Calibration iteration {iteration+1}/{iterations}")
            
            # Apply current factors to generate intervals
            all_validation_data = []
            
            for i, pred_df in enumerate(predictions_history):
                print(f"  Processing prediction set {i+1}/{len(predictions_history)}")
                
                # Apply current width factors to create intervals
                pred_with_intervals = self._apply_width_factors(pred_df)
                
                # Validate against actual results
                validation_data = self.prepare_validation_data(
                    pred_with_intervals, actual_results)
                
                # Add to collection
                if not validation_data.empty:
                    all_validation_data.append(validation_data)
            
            # Combine validation data
            if not all_validation_data:
                print("  Warning: No validation data available for calibration")
                continue
                
            combined_validation = pd.concat(all_validation_data, ignore_index=True)
            
            # Calculate metrics
            validation_metrics = self.validate_intervals(
                pd.concat([self._apply_width_factors(df) for df in predictions_history]),
                actual_results
            )
            
            # Print current metrics
            print("\nCurrent coverage by quarter:")
            for _, row in validation_metrics.iterrows():
                print(f"  Quarter {int(row['quarter'])}: {row['combined_coverage']:.1%} (target: {self.target_coverage:.1%})")
            
            # Calibrate factors
            new_factors = self.calibrate_width_factors(combined_validation)
            
            # Print updated factors
            print("\nUpdated width factors:")
            for quarter, factor in new_factors.items():
                print(f"  Quarter {quarter}: {factor:.2f}")
            
            # Record iteration
            iteration_history.append({
                'iteration': iteration + 1,
                'metrics': validation_metrics.to_dict(orient='records'),
                'factors': new_factors.copy()
            })
        
        # Apply to uncertainty estimator if provided
        if apply_to_uncertainty_estimator is not None:
            print("\nApplying calibrated factors to UncertaintyEstimator")
            apply_to_uncertainty_estimator.width_factors = self.current_factors.copy()
        
        # Create final results
        results = {
            'initial_factors': self.initial_factors,
            'final_factors': self.current_factors,
            'iteration_history': iteration_history,
            'validation_metrics': validation_metrics.to_dict(orient='records') if 'validation_metrics' in locals() else [],
            'calibration_history': self.calibration_history
        }
        
        # Visualize results
        self.visualize_calibration_results(iteration_history)
        
        return results
    
    def _apply_width_factors(self, predictions_df: pd.DataFrame) -> pd.DataFrame:
        """
        Apply current width factors to generate confidence intervals.
        
        Args:
            predictions_df: DataFrame with game predictions
            
        Returns:
            DataFrame with confidence intervals added
        """
        # Make a copy to avoid modifying the original
        result_df = predictions_df.copy()
        
        # Add interval columns if they don't exist
        for col in ['home_lower_bound', 'home_upper_bound', 
                   'away_lower_bound', 'away_upper_bound']:
            if col not in result_df.columns:
                result_df[col] = np.nan
        
        # Apply width factors to each row
        for idx, row in result_df.iterrows():
            quarter = int(row.get('current_quarter', 0))
            width_factor = self.current_factors.get(quarter, 1.0)
            
            # Get predictions
            home_pred = float(row.get('predicted_home_score', row.get('predicted_home_final', 0)))
            away_pred = float(row.get('predicted_away_score', row.get('predicted_away_final', 0)))
            
            # Estimate uncertainty based on quarter
            base_uncertainty = 10.0 * (4.0 - min(quarter, 4)) / 4.0
            half_width = base_uncertainty * width_factor / 2
            
            # Create intervals
            result_df.loc[idx, 'home_lower_bound'] = home_pred - half_width
            result_df.loc[idx, 'home_upper_bound'] = home_pred + half_width
            result_df.loc[idx, 'away_lower_bound'] = away_pred - half_width
            result_df.loc[idx, 'away_upper_bound'] = away_pred + half_width
        
        return result_df
    
    def visualize_calibration_results(self, 
                                    iteration_history: List[Dict[str, Any]] = None,
                                    show_plot: bool = True) -> plt.Figure:
        """
        Visualize the calibration process and results.
        
        Args:
            iteration_history: List of iteration results (if None, uses self.calibration_history)
            show_plot: Whether to show the plot (default: True)
            
        Returns:
            matplotlib Figure object
        """
        # Use calibration history if no iteration history provided
        if iteration_history is None and self.calibration_history:
            # Convert calibration history to iteration format
            iteration_history = []
            for i, entry in enumerate(self.calibration_history):
                iteration_history.append({
                    'iteration': i + 1,
                    'factors': entry.get('new_factors', {}),
                    'coverage': entry.get('coverage', {})
                })
        
        if not iteration_history:
            print("No calibration history to visualize")
            return plt.figure()
        
        # Create figure
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
        
        # Colors and markers for quarters
        markers = ['o', 's', '^', 'd', '*']
        colors = ['blue', 'green', 'red', 'purple', 'orange']
        
        # Data to plot
        iterations = [entry.get('iteration', i+1) for i, entry in enumerate(iteration_history)]
        
        # Plot coverages on the first axis
        ax1.set_title('Coverage Rate by Quarter During Calibration')
        ax1.axhline(y=self.target_coverage, color='black', linestyle='--', 
                   label=f'Target ({self.target_coverage:.0%})')
        
        # Track which quarters we've seen
        seen_quarters = set()
        
        # Plot coverage for each quarter
        for i, entry in enumerate(iteration_history):
            # Get coverage data - handle different formats
            if 'coverage' in entry:
                coverage_data = entry['coverage']
            elif 'metrics' in entry:
                # Convert metrics list to quarter coverage dict
                coverage_data = {}
                for metric in entry['metrics']:
                    if 'quarter' in metric and 'combined_coverage' in metric:
                        coverage_data[metric['quarter']] = metric['combined_coverage']
            else:
                continue
            
            # Plot each quarter's coverage
            for quarter, coverage in coverage_data.items():
                quarter = int(quarter)  # Ensure it's an integer
                
                # Add quarter to seen set
                seen_quarters.add(quarter)
                
                # Plot point for this iteration
                color_idx = quarter % len(colors)
                marker_idx = quarter % len(markers)
                
                # For first iteration, add to legend
                if i == 0:
                    ax1.scatter(iterations[i], coverage, 
                              marker=markers[marker_idx], color=colors[color_idx],
                              s=80, label=f'Quarter {quarter}')
                else:
                    ax1.scatter(iterations[i], coverage, 
                              marker=markers[marker_idx], color=colors[color_idx],
                              s=80)
        
        # Connect points for the same quarter
        for quarter in seen_quarters:
            quarter_coverage = []
            iteration_nums = []
            
            for i, entry in enumerate(iteration_history):
                # Extract coverage based on format
                if 'coverage' in entry and quarter in entry['coverage']:
                    quarter_coverage.append(entry['coverage'][quarter])
                    iteration_nums.append(iterations[i])
                elif 'metrics' in entry:
                    for metric in entry['metrics']:
                        if metric.get('quarter') == quarter and 'combined_coverage' in metric:
                            quarter_coverage.append(metric['combined_coverage'])
                            iteration_nums.append(iterations[i])
            
            # Plot connecting line if we have multiple points
            if len(quarter_coverage) > 1:
                color_idx = quarter % len(colors)
                ax1.plot(iteration_nums, quarter_coverage, 
                       color=colors[color_idx], alpha=0.7)
        
        # Configure first axis
        ax1.set_ylabel('Coverage Rate')
        ax1.set_ylim(0, 1.1)
        ax1.grid(True, alpha=0.3)
        ax1.legend(loc='best')
        
        # Plot width factors on the second axis
        ax2.set_title('Width Factors by Quarter During Calibration')
        
        # Plot factors for each quarter
        for quarter in sorted(seen_quarters):
            quarter_factors = []
            iteration_nums = []
            
            for i, entry in enumerate(iteration_history):
                if 'factors' in entry and quarter in entry['factors']:
                    quarter_factors.append(entry['factors'][quarter])
                    iteration_nums.append(iterations[i])
            
            # Plot if we have data
            if quarter_factors:
                color_idx = quarter % len(colors)
                marker_idx = quarter % len(markers)
                
                ax2.plot(iteration_nums, quarter_factors, 
                       marker=markers[marker_idx], color=colors[color_idx],
                       label=f'Quarter {quarter}', linewidth=2)
        
        # Configure second axis
        ax2.set_xlabel('Calibration Iteration')
        ax2.set_ylabel('Width Factor')
        ax2.grid(True, alpha=0.3)
        ax2.legend(loc='best')
        
        # Final layout
        plt.tight_layout()
        
        if show_plot:
            plt.show()
        
        return fig
    
    def save_calibration_state(self, filepath: str) -> bool:
        """
        Save the current calibration state to a JSON file.
        
        Args:
            filepath: Path to save the calibration state
            
        Returns:
            bool: True if successful, False otherwise
        """
        try:
            state = {
                'current_factors': self.current_factors,
                'initial_factors': self.initial_factors,
                'target_coverage': self.target_coverage,
                'dampen_factor': self.dampen_factor,
                'min_factor': self.min_factor,
                'max_factor': self.max_factor,
                'calibration_history': self.calibration_history,
                'saved_at': datetime.now().isoformat()
            }
            
            with open(filepath, 'w') as f:
                json.dump(state, f, indent=2)
                
            print(f"Calibration state saved to {filepath}")
            return True
            
        except Exception as e:
            print(f"Error saving calibration state: {e}")
            return False
    
    def load_calibration_state(self, filepath: str) -> bool:
        """
        Load calibration state from a JSON file.
        
        Args:
            filepath: Path to the calibration state file
            
        Returns:
            bool: True if successful, False otherwise
        """
        try:
            if not os.path.exists(filepath):
                print(f"Calibration state file not found: {filepath}")
                return False
                
            with open(filepath, 'r') as f:
                state = json.load(f)
            
            # Load core parameters
            self.current_factors = state.get('current_factors', self.current_factors)
            self.initial_factors = state.get('initial_factors', self.initial_factors)
            self.target_coverage = state.get('target_coverage', self.target_coverage)
            self.dampen_factor = state.get('dampen_factor', self.dampen_factor)
            self.min_factor = state.get('min_factor', self.min_factor)
            self.max_factor = state.get('max_factor', self.max_factor)
            self.calibration_history = state.get('calibration_history', [])
            
            print(f"Calibration state loaded from {filepath}")
            print(f"Current factors: {self.current_factors}")
            return True
            
        except Exception as e:
            print(f"Error loading calibration state: {e}")
            return False


def calibrate_confidence_intervals(validation_df: pd.DataFrame, 
                                 current_factors: Dict[int, float],
                                 target_coverage: float = 0.9,
                                 dampen_factor: float = 0.5) -> Dict[int, float]:
    """
    Calibrate confidence interval width factors based on validation results.
    
    Standalone function version for backward compatibility.
    
    Args:
        validation_df: DataFrame with validation results
        current_factors: Dict mapping quarters to current width factors
        target_coverage: Target coverage rate (default: 0.9 for 90% intervals)
        dampen_factor: Factor to dampen adjustments (0-1, lower = more conservative)
        
    Returns:
        Dict with updated width factors
    """
    # Create calibration system and apply calibration
    calibration_system = CalibrationSystem(
        initial_factors=current_factors,
        target_coverage=target_coverage,
        dampen_factor=dampen_factor
    )
    
    return calibration_system.calibrate_width_factors(validation_df)


def validate_confidence_intervals(predictions_df: pd.DataFrame, 
                               actual_results_df: pd.DataFrame, 
                               confidence_level: float = 0.9) -> pd.DataFrame:
    """
    Validate that confidence intervals contain actual values at the expected rate.
    
    Standalone function version for backward compatibility.
    
    Args:
        predictions_df: DataFrame with predictions and confidence intervals
        actual_results_df: DataFrame with actual final scores
        confidence_level: Expected coverage rate (default: 0.9 for 90% confidence)
    
    Returns:
        DataFrame with validation metrics
    """
    # Create calibration system and run validation
    calibration_system = CalibrationSystem(target_coverage=confidence_level)
    
    return calibration_system.validate_intervals(predictions_df, actual_results_df)


def run_confidence_calibration_loop(predictions_history: List[pd.DataFrame], 
                                  actual_results: pd.DataFrame,
                                  initial_factors: Optional[Dict[int, float]] = None,
                                  iterations: int = 3) -> Tuple[Dict[int, float], List[Dict[str, Any]]]:
    """
    Run an iterative calibration process to optimize confidence intervals.
    
    Standalone function version for backward compatibility.
    
    Args:
        predictions_history: List of DataFrames with historical predictions
        actual_results: DataFrame with actual game results
        initial_factors: Starting width factors (default: {0:3.0, 1:2.5, 2:2.0, 3:1.5, 4:1.0})
        iterations: Number of calibration iterations
        
    Returns:
        tuple: (final_factors, calibration_history)
    """
    # Create calibration system
    calibration_system = CalibrationSystem(initial_factors=initial_factors)
    
    # Run calibration loop
    results = calibration_system.run_calibration_loop(
        predictions_history, actual_results, iterations)
    
    # Extract expected return values for backward compatibility
    return calibration_system.current_factors, results['iteration_history']


# Example usage and demo
def run_calibration_demo():
    """Run a demonstration of the calibration system with sample data."""
    print("Running calibration system demonstration...")
    
    # Create sample predictions
    sample_predictions = []
    for _ in range(3):  # Three sample prediction batches
        preds = pd.DataFrame([
            {'game_id': 1001, 'current_quarter': 1, 'predicted_home_score': 105, 'predicted_away_score': 100},
            {'game_id': 1002, 'current_quarter': 2, 'predicted_home_score': 110, 'predicted_away_score': 102},
            {'game_id': 1003, 'current_quarter': 3, 'predicted_home_score': 95, 'predicted_away_score': 90},
            {'game_id': 1004, 'current_quarter': 4, 'predicted_home_score': 118, 'predicted_away_score': 112}
        ])
        sample_predictions.append(preds)

    # Sample actual results
    sample_actuals = pd.DataFrame([
        {'game_id': 1001, 'home_score': 108, 'away_score': 97},
        {'game_id': 1002, 'home_score': 112, 'away_score': 105},
        {'game_id': 1003, 'home_score': 92, 'away_score': 96},
        {'game_id': 1004, 'home_score': 120, 'away_score': 110}
    ])
    
    # Create calibration system
    calibration_system = CalibrationSystem()
    
    # Run calibration loop
    results = calibration_system.run_calibration_loop(
        sample_predictions, sample_actuals, iterations=2)
    
    # Print final results
    print("\nFinal calibrated factors:")
    for quarter, factor in calibration_system.current_factors.items():
        print(f"Quarter {quarter}: {factor:.2f}")
    
    return results


# Run demonstration if executed directly
if __name__ == "__main__":
    demo_results = run_calibration_demo()

In [ ]:
# Cell 27D: Complete Context-Aware Confidence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import pickle
import os
import json
from sklearn.metrics import mean_absolute_error
from typing import Dict, Tuple, List, Optional, Union, Any


class ContextClassifier:
    """
    Classifies NBA game contexts to help determine appropriate confidence intervals.
    
    This class analyzes game state data to identify special situations like close games,
    blowouts, high-momentum scenarios, and high-leverage moments where prediction
    uncertainty may differ from standard patterns.
    """
    
    def __init__(self, thresholds: Optional[Dict[str, Any]] = None):
        """
        Initialize the context classifier with configurable thresholds.
        
        Args:
            thresholds: Dictionary of threshold values for different classifications.
                      If None, uses default thresholds.
        """
        # Default thresholds if none provided
        self.thresholds = thresholds or {
            'close_game': 5.0,       # Score differential at or below this is a close game
            'blowout': 15.0,         # Score differential at or above this is a blowout
            'high_momentum': 0.5,    # Absolute momentum at or above this is high momentum
            'final_minutes': 5.0,    # Minutes remaining at or below this is "final minutes"
            'win_prob_threshold': 0.45,  # Win prob between this and (1-this) is a toss-up
        }
    
    def is_close_game(self, score_diff: float) -> bool:
        """
        Determine if a game is considered "close" based on score differential.
        
        Args:
            score_diff: Absolute score difference between teams
            
        Returns:
            bool: True if the game is close
        """
        return score_diff <= self.thresholds['close_game']
    
    def is_blowout(self, score_diff: float) -> bool:
        """
        Determine if a game is considered a "blowout" based on score differential.
        
        Args:
            score_diff: Absolute score difference between teams
            
        Returns:
            bool: True if the game is a blowout
        """
        return score_diff >= self.thresholds['blowout']
    
    def is_high_momentum(self, momentum: float) -> bool:
        """
        Determine if a team has high momentum.
        
        Args:
            momentum: Team momentum value (-1 to 1)
            
        Returns:
            bool: True if momentum is high (in either direction)
        """
        return abs(momentum) >= self.thresholds['high_momentum']
    
    def is_final_minutes(self, quarter: int, time_remaining: Optional[float]) -> bool:
        """
        Determine if the game is in its final minutes.
        
        Args:
            quarter: Current quarter (0-4, where 0 is pre-game)
            time_remaining: Minutes remaining in the quarter
            
        Returns:
            bool: True if in final minutes of 4th quarter
        """
        return (quarter == 4 and 
                time_remaining is not None and 
                time_remaining <= self.thresholds['final_minutes'])
    
    def is_high_leverage_situation(self, 
                                 quarter: int, 
                                 score_diff: float, 
                                 time_remaining: Optional[float] = None,
                                 win_probability: Optional[float] = None) -> bool:
        """
        Determine if this is a high-leverage situation requiring specialized handling.
        
        High-leverage situations are critical moments that can significantly impact 
        the outcome of the game, such as close games in the final minutes.
        
        Args:
            quarter: Current quarter (0-4)
            score_diff: Absolute score difference between teams
            time_remaining: Minutes remaining in the quarter
            win_probability: Current win probability (0-1)
            
        Returns:
            bool: True if this is a high-leverage situation
        """
        # Final minutes of 4th quarter in close game
        if self.is_final_minutes(quarter, time_remaining) and self.is_close_game(score_diff):
            return True
        
        # Close win probability regardless of quarter
        if win_probability is not None:
            threshold = self.thresholds['win_prob_threshold']
            is_toss_up = (win_probability >= threshold and 
                          win_probability <= (1 - threshold))
                          
            if is_toss_up and self.is_close_game(score_diff):
                return True
        
        return False
    
    def get_game_context(self, 
                        quarter: int,
                        score_diff: float,
                        momentum: float = 0.0,
                        time_remaining: Optional[float] = None,
                        win_probability: Optional[float] = None) -> Dict[str, bool]:
        """
        Get a complete analysis of the game context.
        
        Args:
            quarter: Current quarter (0-4)
            score_diff: Absolute score difference
            momentum: Team momentum (-1 to 1)
            time_remaining: Minutes remaining in quarter
            win_probability: Win probability (0-1)
            
        Returns:
            Dict with context classifications
        """
        return {
            'close_game': self.is_close_game(score_diff),
            'blowout': self.is_blowout(score_diff),
            'high_momentum': self.is_high_momentum(momentum),
            'final_minutes': self.is_final_minutes(quarter, time_remaining),
            'high_leverage': self.is_high_leverage_situation(
                quarter, score_diff, time_remaining, win_probability)
        }


class ContextAwareConfidenceAdjuster:
    """
    Adjusts confidence intervals based on game context.
    
    This class implements context-specific adjustments to base confidence
    intervals to account for different game situations that may affect
    prediction certainty.
    """
    
    def __init__(self, adjustment_factors: Optional[Dict[str, float]] = None):
        """
        Initialize the context adjuster with configurable adjustment factors.
        
        Args:
            adjustment_factors: Dictionary mapping context types to adjustment
                              multipliers. If None, uses default values.
        """
        # Default adjustment factors if none provided
        self.adjustment_factors = adjustment_factors or {
            'close_game': 1.1,       # Increase width for close games (more uncertainty)
            'blowout': 0.9,          # Decrease width for blowouts (more certainty)
            'high_momentum': 1.15,   # Increase width for high momentum (more volatility)
            'final_minutes': 0.85,   # Decrease width in final minutes (more certainty)
            'overtime': 1.2,         # Wider intervals in overtime (more uncertainty)
            'high_leverage': 0.8,    # Special handling for high-leverage situations
        }
        
        # Context classifier for identifying game situations
        self.context_classifier = ContextClassifier()
    
    def calculate_adjustment(self, 
                          quarter: int, 
                          score_diff: float, 
                          momentum: float = 0.0,
                          time_remaining: Optional[float] = None,
                          win_probability: Optional[float] = None) -> float:
        """
        Calculate the context-specific adjustment factor for confidence intervals.
        
        Args:
            quarter: Current quarter (0-4)
            score_diff: Absolute score difference
            momentum: Momentum value (-1 to 1)
            time_remaining: Minutes remaining in quarter
            win_probability: Current win probability
            
        Returns:
            float: Combined adjustment factor
        """
        # Start with base factor of 1.0
        adjustment = 1.0
        
        # Get game context
        context = self.context_classifier.get_game_context(
            quarter, score_diff, momentum, time_remaining, win_probability)
        
        # Apply adjustments based on context
        if context['close_game']:
            adjustment *= self.adjustment_factors['close_game']
        elif context['blowout']:
            adjustment *= self.adjustment_factors['blowout']
        else:
            # Linear interpolation between close and blowout
            close_threshold = self.context_classifier.thresholds['close_game']
            blowout_threshold = self.context_classifier.thresholds['blowout']
            
            # Avoid division by zero
            if blowout_threshold > close_threshold:
                blend = (score_diff - close_threshold) / (blowout_threshold - close_threshold)
                blend = max(0.0, min(1.0, blend))  # Clamp to [0, 1]
                
                adjustment *= ((1 - blend) * self.adjustment_factors['close_game'] + 
                              blend * self.adjustment_factors['blowout'])
        
        # Apply momentum adjustment
        if context['high_momentum']:
            adjustment *= self.adjustment_factors['high_momentum']
        
        # Apply final minutes adjustment
        if context['final_minutes']:
            adjustment *= self.adjustment_factors['final_minutes']
        
        # Apply high leverage adjustment
        if context['high_leverage']:
            # High leverage situations get specialized treatment, but never below 0.7
            adjustment = max(0.7, adjustment * self.adjustment_factors['high_leverage'])
        
        # Ensure adjustment is within reasonable bounds
        adjustment = max(0.5, min(2.0, adjustment))
        
        return adjustment
    
    def set_adjustment_factor(self, context_type: str, value: float) -> bool:
        """
        Update a specific adjustment factor.
        
        Args:
            context_type: Type of context to adjust (e.g., 'close_game')
            value: New adjustment factor value
            
        Returns:
            bool: True if update was successful
        """
        if context_type in self.adjustment_factors:
            self.adjustment_factors[context_type] = value
            return True
        return False
    
    def set_threshold(self, threshold_type: str, value: float) -> bool:
        """
        Update a threshold in the context classifier.
        
        Args:
            threshold_type: Type of threshold to update (e.g., 'close_game')
            value: New threshold value
            
        Returns:
            bool: True if update was successful
        """
        if threshold_type in self.context_classifier.thresholds:
            self.context_classifier.thresholds[threshold_type] = value
            return True
        return False


class ContextAwareConfidenceCalibration:
    """
    Advanced confidence calibration system that adapts to game context
    and provides specialized handling for high-leverage situations.
    
    This class combines base width factors with context-aware adjustments
    to create confidence intervals that account for different game situations.
    It also provides calibration functionality to improve interval accuracy
    over time.
    """
    
    def __init__(self, 
                 base_width_factors: Optional[Dict[int, float]] = None,
                 context_adjuster: Optional[ContextAwareConfidenceAdjuster] = None,
                 cache_path: str = "confidence_calibration_cache.pkl"):
        """
        Initialize the calibration system.
        
        Args:
            base_width_factors: Initial width factors by quarter (0-4).
                              If None, uses default values.
            context_adjuster: Custom context adjuster. If None, creates default.
            cache_path: Path to save/load calibration cache
        """
        # Default width factors by quarter if none provided
        self.width_factors = base_width_factors or {
            0: 3.0,  # Pre-game: wide intervals
            1: 2.5,  # 1st quarter
            2: 2.0,  # 2nd quarter
            3: 1.5,  # 3rd quarter
            4: 1.0,  # 4th quarter
        }
        
        # Context-aware adjustment system
        self.context_adjuster = context_adjuster or ContextAwareConfidenceAdjuster()
        
        # Cache for calibration history
        self.cache_path = cache_path
        self.calibration_history = []
        
        # Target coverage rate (default 90%)
        self.target_coverage = 0.9
        
        # Load cached calibration if available
        self._load_calibration_cache()
    
    def _load_calibration_cache(self) -> bool:
        """
        Load calibration cache from disk if available.
        
        Returns:
            bool: True if cache was loaded successfully
        """
        if os.path.exists(self.cache_path):
            try:
                with open(self.cache_path, 'rb') as f:
                    cached_data = pickle.load(f)
                    self.width_factors = cached_data.get('width_factors', self.width_factors)
                    
                    # Load context adjustments if available
                    if 'context_adjustments' in cached_data:
                        for key, value in cached_data['context_adjustments'].items():
                            self.context_adjuster.set_adjustment_factor(key, value)
                    
                    self.calibration_history = cached_data.get('calibration_history', [])
                    print(f"Loaded calibration cache from {self.cache_path}")
                return True
            except Exception as e:
                print(f"Error loading calibration cache: {e}")
        return False
    
    def _save_calibration_cache(self) -> bool:
        """
        Save current calibration to disk.
        
        Returns:
            bool: True if cache was saved successfully
        """
        try:
            with open(self.cache_path, 'wb') as f:
                pickle.dump({
                    'width_factors': self.width_factors,
                    'context_adjustments': self.context_adjuster.adjustment_factors,
                    'calibration_history': self.calibration_history,
                    'updated_at': datetime.now().isoformat()
                }, f)
            print(f"Saved calibration cache to {self.cache_path}")
            return True
        except Exception as e:
            print(f"Error saving calibration cache: {e}")
            return False
    
    def export_calibration_json(self, filepath: str) -> bool:
        """
        Export calibration data to a human-readable JSON file.
        
        Args:
            filepath: Path to save the JSON export
            
        Returns:
            bool: True if export was successful
        """
        try:
            export_data = {
                'width_factors': self.width_factors,
                'context_adjustments': self.context_adjuster.adjustment_factors,
                'context_thresholds': self.context_adjuster.context_classifier.thresholds,
                'target_coverage': self.target_coverage,
                'calibration_summary': {
                    'history_entries': len(self.calibration_history),
                    'last_updated': datetime.now().isoformat()
                }
            }
            
            with open(filepath, 'w') as f:
                json.dump(export_data, f, indent=2)
                
            print(f"Exported calibration data to {filepath}")
            return True
        except Exception as e:
            print(f"Error exporting calibration data: {e}")
            return False
    
    def calculate_prediction_interval(self, 
                                    predicted_value: float, 
                                    quarter: int, 
                                    score_diff: float, 
                                    momentum: float = 0.0,
                                    time_remaining: Optional[float] = None, 
                                    win_probability: Optional[float] = None) -> Tuple[float, float, float]:
        """
        Calculate confidence interval for a prediction with context awareness.
        
        Args:
            predicted_value: The predicted score
            quarter: Current quarter (0-4)
            score_diff: Absolute score difference
            momentum: Momentum value (-1 to 1)
            time_remaining: Minutes remaining in quarter
            win_probability: Win probability estimate
            
        Returns:
            Tuple[float, float, float]: (lower_bound, upper_bound, interval_width)
        """
        # Get base width factor for this quarter
        base_factor = self.width_factors.get(quarter, 2.0)
        
        # Calculate base uncertainty (decreases as game progresses)
        base_uncertainty = 10.0 * (4.0 - min(quarter, 4)) / 4.0
        
        # Get context adjustment
        context_adjustment = self.context_adjuster.calculate_adjustment(
            quarter, score_diff, momentum, time_remaining, win_probability)
        
        # Calculate interval width
        interval_width = base_uncertainty * base_factor * context_adjustment
        
        # Ensure interval width is reasonable
        min_width = 2.0  # Minimum width to account for inherent randomness
        max_width = 30.0  # Maximum width to prevent extreme intervals
        interval_width = min(max(interval_width, min_width), max_width)
        
        # Calculate bounds
        lower_bound = predicted_value - interval_width / 2
        upper_bound = predicted_value + interval_width / 2
        
        return lower_bound, upper_bound, interval_width
    
    def apply_prediction_intervals(self, predictions: pd.DataFrame) -> pd.DataFrame:
        """
        Apply prediction intervals to a DataFrame of predictions.
        
        Args:
            predictions: DataFrame with game predictions
            
        Returns:
            DataFrame with confidence intervals added
        """
        result = predictions.copy()
        
        # Add interval columns if they don't exist
        for col in ['home_lower_bound', 'home_upper_bound', 'away_lower_bound', 'away_upper_bound',
                  'home_interval_width', 'away_interval_width']:
            if col not in result.columns:
                result[col] = np.nan
        
        # Calculate intervals for each prediction
        for idx, row in result.iterrows():
            # Extract needed data with fallbacks for missing columns
            quarter = int(row.get('current_quarter', 0))
            
            # Get home and away predictions with fallbacks
            home_pred = float(row.get('predicted_home_score', 
                                   row.get('predicted_home_final', 0)))
            away_pred = float(row.get('predicted_away_score', 
                                   row.get('predicted_away_final', 0)))
            
            # Get game context information
            score_diff = abs(row.get('score_differential', 
                                  abs(home_pred - away_pred)))
            
            momentum = float(row.get('momentum_shift', 
                                  row.get('cumulative_momentum', 0)))
            
            time_remaining = row.get('time_remaining')
            win_prob = row.get('win_probability')
            
            # Calculate intervals for home team
            home_lower, home_upper, home_width = self.calculate_prediction_interval(
                home_pred, quarter, score_diff, momentum, time_remaining, win_prob)
            
            # Calculate intervals for away team
            away_lower, away_upper, away_width = self.calculate_prediction_interval(
                away_pred, quarter, score_diff, momentum, time_remaining, win_prob)
            
            # Update DataFrame
            result.loc[idx, 'home_lower_bound'] = home_lower
            result.loc[idx, 'home_upper_bound'] = home_upper
            result.loc[idx, 'home_interval_width'] = home_width
            
            result.loc[idx, 'away_lower_bound'] = away_lower
            result.loc[idx, 'away_upper_bound'] = away_upper
            result.loc[idx, 'away_interval_width'] = away_width
            
            # Add context information for reference
            result.loc[idx, 'context_adjustment'] = self.context_adjuster.calculate_adjustment(
                quarter, score_diff, momentum, time_remaining, win_prob)
            
            high_leverage = self.context_adjuster.context_classifier.is_high_leverage_situation(
                quarter, score_diff, time_remaining, win_prob)
            result.loc[idx, 'high_leverage'] = high_leverage
        
        return result
    
    def calculate_coverage(self, predictions: pd.DataFrame, actuals: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate coverage rates by comparing predictions with actual values.
        
        Args:
            predictions: DataFrame with predictions and intervals
            actuals: DataFrame with actual game results
        
        Returns:
            DataFrame with coverage analysis
        """
        # Merge predictions with actuals
        merged = pd.merge(
            predictions,
            actuals[['game_id', 'home_score', 'away_score']],
            on='game_id',
            how='inner',
            suffixes=('_pred', '')
        )
        
        if merged.empty:
            print("Warning: No matching games found between predictions and actuals")
            return pd.DataFrame()
        
        # Ensure quarter column exists (copy from current_quarter if needed)
        if 'quarter' not in merged.columns and 'current_quarter' in merged.columns:
            merged['quarter'] = merged['current_quarter']
        
        # Check if actual values fall within predicted intervals
        merged['home_in_interval'] = ((merged['home_score'] >= merged['home_lower_bound']) & 
                                    (merged['home_score'] <= merged['home_upper_bound']))
        merged['away_in_interval'] = ((merged['away_score'] >= merged['away_lower_bound']) & 
                                    (merged['away_score'] <= merged['away_upper_bound']))
        merged['in_interval'] = merged['home_in_interval'] & merged['away_in_interval']
        
        # Add error metrics
        if 'predicted_home_score' in merged.columns and 'predicted_away_score' in merged.columns:
            merged['home_error'] = (merged['predicted_home_score'] - merged['home_score']).abs()
            merged['away_error'] = (merged['predicted_away_score'] - merged['away_score']).abs()
            merged['total_error'] = merged['home_error'] + merged['away_error']
        
        return merged
    
    def update_width_factors(self, 
                           coverage_by_quarter: Dict[int, float], 
                           target_coverage: float = 0.9,
                           dampen_factor: float = 0.5) -> Dict[int, float]:
        """
        Update width factors based on observed coverage vs target.
        
        Args:
            coverage_by_quarter: Dict mapping quarters to observed coverage rates
            target_coverage: Target coverage rate (default: 0.9)
            dampen_factor: Factor to dampen adjustments (0-1)
            
        Returns:
            Dict with updated width factors
        """
        updated_factors = self.width_factors.copy()
        
        for quarter, coverage in coverage_by_quarter.items():
            if coverage < 0.01 or target_coverage < 0.01:
                # Skip very small values to avoid division issues
                continue
                
            # Calculate adjustment ratio
            adjustment_ratio = target_coverage / coverage
            
            # Apply adjustment with dampening to prevent overcorrection
            updated_factors[quarter] = updated_factors.get(quarter, 1.0) * (
                1.0 + (adjustment_ratio - 1.0) * dampen_factor
            )
            
            # Cap factors to reasonable bounds
            updated_factors[quarter] = max(0.5, min(5.0, updated_factors[quarter]))
        
        return updated_factors
    
    def run_calibration_iteration(self, 
                                predictions: pd.DataFrame, 
                                actuals: pd.DataFrame, 
                                target_coverage: float = 0.9) -> Dict[str, Any]:
        """
        Run a single calibration iteration to update width factors.
        
        Args:
            predictions: DataFrame with predictions and their metadata
            actuals: DataFrame with actual final scores
            target_coverage: Target coverage rate (default: 0.9 for 90% intervals)
            
        Returns:
            Dict with calibration results
        """
        print(f"Running calibration iteration (target coverage: {target_coverage:.1%})")
        
        # Store target coverage
        self.target_coverage = target_coverage
        
        # Apply current width factors to predictions
        augmented_predictions = self.apply_prediction_intervals(predictions)
        
        # Merge with actuals to calculate coverage
        validation_results = self.calculate_coverage(augmented_predictions, actuals)
        
        # Calculate per-quarter coverage rates
        coverage_by_quarter = {}
        for quarter in range(5):  # 0-4
            quarter_data = validation_results[validation_results['quarter'] == quarter]
            if not quarter_data.empty:
                coverage_by_quarter[quarter] = quarter_data['in_interval'].mean()
                print(f"Quarter {quarter}: Coverage = {coverage_by_quarter[quarter]:.1%} (target: {target_coverage:.1%})")
        
        # Update width factors
        old_factors = self.width_factors.copy()
        self.width_factors = self.update_width_factors(coverage_by_quarter, target_coverage)
        
        # Calculate factor changes
        factor_changes = {q: self.width_factors[q] - old_factors.get(q, 1.0) 
                        for q in self.width_factors}
        
        # Record calibration result
        result = {
            'timestamp': datetime.now().isoformat(),
            'old_factors': old_factors,
            'new_factors': self.width_factors.copy(),
            'coverage_by_quarter': coverage_by_quarter,
            'factor_changes': factor_changes,
            'target_coverage': target_coverage,
            'context_adjustments': self.context_adjuster.adjustment_factors.copy()
        }
        self.calibration_history.append(result)
        
        # Save updated calibration
        self._save_calibration_cache()
        
        return result
    
    def run_full_calibration(self, 
                           prediction_history: List[pd.DataFrame], 
                           actual_results: pd.DataFrame, 
                           iterations: int = 3,
                           target_coverage: float = 0.9) -> Dict[str, Any]:
        """
        Run a full calibration process with multiple iterations.
        
        Args:
            prediction_history: List of prediction DataFrames
            actual_results: DataFrame with actual results
            iterations: Number of calibration iterations
            target_coverage: Target coverage rate
            
        Returns:
            Dict with calibration summary
        """
        # Initialize results
        calibration_results = []
        
        # Store starting factors
        initial_factors = self.width_factors.copy()
        
        # Run iterations
        for i in range(iterations):
            print(f"\nCalibration iteration {i+1}/{iterations}")
            
            # Combine all predictions for this iteration
            try:
                all_predictions = pd.concat(prediction_history, ignore_index=True)
            except ValueError:
                if len(prediction_history) == 0:
                    print("Error: No prediction data provided")
                    return {'error': 'No prediction data provided'}
                else:
                    all_predictions = prediction_history[0]
            
            # Run calibration iteration
            result = self.run_calibration_iteration(all_predictions, actual_results, target_coverage)
            calibration_results.append(result)
            
            # Early stopping if we're close to target
            coverage_values = list(result['coverage_by_quarter'].values())
            if coverage_values:
                avg_coverage = sum(coverage_values) / len(coverage_values)
                coverage_error = abs(avg_coverage - target_coverage)
                
                if coverage_error < 0.02:  # Within 2% of target
                    print(f"Early stopping: Average coverage {avg_coverage:.1%} is within 2% of target {target_coverage:.1%}")
                    break
        
        # Calculate overall changes
        overall_changes = {}
        for quarter in self.width_factors.keys():
            if quarter in initial_factors:
                change = self.width_factors[quarter] - initial_factors[quarter]
                percent_change = (change / initial_factors[quarter]) * 100
                overall_changes[quarter] = {
                    'absolute': change,
                    'percent': percent_change
                }
        
        # Generate summary
        summary = {
            'iterations_run': len(calibration_results),
            'final_width_factors': self.width_factors,
            'initial_width_factors': initial_factors,
            'overall_changes': overall_changes,
            'final_coverage': calibration_results[-1]['coverage_by_quarter'] if calibration_results else None,
            'calibration_history': calibration_results
        }
        
        # Visualize calibration process
        self.visualize_calibration_results(summary)
        
        return summary
    
    def visualize_calibration_results(self, 
                                    calibration_summary: Dict[str, Any], 
                                    show_plot: bool = True) -> Optional[plt.Figure]:
        """
        Visualize calibration results.
        
        Args:
            calibration_summary: Dictionary with calibration results
            show_plot: Whether to display the plot immediately
        
        Returns:
            matplotlib Figure if created, None otherwise
        """
        # Skip if no calibration history
        if not calibration_summary.get('calibration_history'):
            print("No calibration history to visualize")
            return None
        
        # Create figure with two subplots
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
        
        # Extract data for plotting
        history = calibration_summary['calibration_history']
        iterations = list(range(1, len(history) + 1))
        
        # Use consistent quarters across all iterations
        all_quarters = set()
        for entry in history:
            all_quarters.update(entry['new_factors'].keys())
        quarters = sorted(all_quarters)
        
        # Colors for different quarters
        colors = ['blue', 'green', 'orange', 'red', 'purple']
        
        # Plot 1: Coverage by quarter over iterations
        ax1.set_title('Coverage by Quarter Over Calibration Iterations')
        
        for i, quarter in enumerate(quarters):
            # Extract coverage values for this quarter
            coverage_values = []
            for entry in history:
                coverage = entry['coverage_by_quarter'].get(quarter, np.nan)
                coverage_values.append(coverage)
            
            # Skip if no data for this quarter
            if all(np.isnan(x) for x in coverage_values):
                continue
                
            # Plot coverage trend
            ax1.plot(iterations, coverage_values, f'-o', color=colors[i % len(colors)], 
                    label=f'Quarter {quarter}')
        
        # Add target line
        target = history[0]['target_coverage']
        ax1.axhline(y=target, color='black', linestyle='--', label=f'Target ({target:.0%})')
        
        ax1.set_xlabel('Calibration Iteration')
        ax1.set_ylabel('Coverage Rate')
        ax1.set_ylim(0, 1.1)
        ax1.grid(True, alpha=0.3)
        ax1.legend()
        
        # Plot 2: Width factors by quarter over iterations
        ax2.set_title('Width Factors by Quarter Over Calibration Iterations')
        
        for i, quarter in enumerate(quarters):
            # Extract factor values for this quarter
            factor_values = []
            for entry in history:
                factor = entry['new_factors'].get(quarter, np.nan)
                factor_values.append(factor)
            
            # Skip if no data for this quarter
            if all(np.isnan(x) for x in factor_values):
                continue
                
            # Plot factor trend
            ax2.plot(iterations, factor_values, f'-s', color=colors[i % len(colors)], 
                    label=f'Quarter {quarter}')
        
        ax2.set_xlabel('Calibration Iteration')
        ax2.set_ylabel('Width Factor')
        ax2.grid(True, alpha=0.3)
        ax2.legend()
        
        plt.tight_layout()
        
        if show_plot:
            plt.show()
        
        return fig
    
    def analyze_high_leverage_performance(self, predictions: pd.DataFrame, 
                                       actuals: pd.DataFrame) -> Dict[str, Any]:
        """
        Analyze prediction performance specifically in high-leverage situations.
        
        Args:
            predictions: DataFrame with predictions
            actuals: DataFrame with actual results
            
        Returns:
            Dict with analysis results
        """
        # Apply intervals and identify high-leverage situations
        predictions_with_intervals = self.apply_prediction_intervals(predictions)
        
        # Calculate coverage
        coverage_data = self.calculate_coverage(predictions_with_intervals, actuals)
        
        if coverage_data.empty:
            return {'error': 'No matching data found'}
        
        # Split by high leverage vs normal
        high_leverage = coverage_data[coverage_data['high_leverage'] == True]
        normal = coverage_data[coverage_data['high_leverage'] == False]
        
        # Prepare analysis results
        results = {
            'summary': {
                'total_games': len(coverage_data),
                'high_leverage_games': len(high_leverage),
                'normal_games': len(normal)
            }
        }
        
        # Calculate coverage rates
        if not high_leverage.empty:
            results['high_leverage'] = {
                'coverage': high_leverage['in_interval'].mean(),
                'home_coverage': high_leverage['home_in_interval'].mean(),
                'away_coverage': high_leverage['away_in_interval'].mean()
            }
            
            # Add error metrics if available
            if 'home_error' in high_leverage.columns:
                results['high_leverage'].update({
                    'mean_home_error': high_leverage['home_error'].mean(),
                    'mean_away_error': high_leverage['away_error'].mean(),
                    'mean_total_error': high_leverage['total_error'].mean()
                })
        
        if not normal.empty:
            results['normal'] = {
                'coverage': normal['in_interval'].mean(),
                'home_coverage': normal['home_in_interval'].mean(),
                'away_coverage': normal['away_in_interval'].mean()
            }
            
            # Add error metrics if available
            if 'home_error' in normal.columns:
                results['normal'].update({
                    'mean_home_error': normal['home_error'].mean(),
                    'mean_away_error': normal['away_error'].mean(),
                    'mean_total_error': normal['total_error'].mean()
                })
        
        # Calculate comparison between high-leverage and normal situations
        if 'high_leverage' in results and 'normal' in results:
            results['comparison'] = {
                'coverage_diff': results['high_leverage']['coverage'] - results['normal']['coverage'],
                'home_coverage_diff': results['high_leverage']['home_coverage'] - results['normal']['home_coverage'],
                'away_coverage_diff': results['high_leverage']['away_coverage'] - results['normal']['away_coverage']
            }
            
            # Add error comparison if available
            if 'mean_total_error' in results['high_leverage'] and 'mean_total_error' in results['normal']:
                results['comparison']['error_ratio'] = (
                    results['high_leverage']['mean_total_error'] / 
                    results['normal']['mean_total_error'] if results['normal']['mean_total_error'] > 0 else np.nan
                )
        
        return results


def demo_confidence_calibration(display_details: bool = True) -> Dict[str, Any]:
    """
    Demonstrate the calibration system with sample data.
    
    Args:
        display_details: Whether to print detailed information
        
    Returns:
        Dict with demonstration results
    """
    # Create a calibration system
    calibration_system = ContextAwareConfidenceCalibration()
    
    # Create sample predictions with different game contexts
    predictions = pd.DataFrame([
        # Pre-game predictions
        {'game_id': 1001, 'current_quarter': 0, 'predicted_home_score': 105, 'predicted_away_score': 98, 
         'momentum_shift': 0, 'win_probability': 0.65},
        # Close game in Q2
        {'game_id': 1002, 'current_quarter': 2, 'predicted_home_score': 110, 'predicted_away_score': 107, 
         'momentum_shift': 0.3, 'win_probability': 0.55, 'time_remaining': 6},
        # Blowout in Q3
        {'game_id': 1003, 'current_quarter': 3, 'predicted_home_score': 95, 'predicted_away_score': 75, 
         'momentum_shift': -0.1, 'win_probability': 0.92, 'time_remaining': 8},
        # High leverage final minutes
        {'game_id': 1004, 'current_quarter': 4, 'predicted_home_score': 108, 'predicted_away_score': 106, 
         'momentum_shift': 0.6, 'win_probability': 0.52, 'time_remaining': 2},
    ])
    
    # Sample actual results
    actuals = pd.DataFrame([
        {'game_id': 1001, 'home_score': 108, 'away_score': 101},
        {'game_id': 1002, 'home_score': 112, 'away_score': 109},
        {'game_id': 1003, 'home_score': 98, 'away_score': 78},
        {'game_id': 1004, 'home_score': 110, 'away_score': 107},
    ])
    
    # Apply intervals and check coverage
    predictions_with_intervals = calibration_system.apply_prediction_intervals(predictions)
    coverage_results = calibration_system.calculate_coverage(predictions_with_intervals, actuals)
    
    if display_details:
        print("\nSample Predictions with Intervals:")
        for idx, row in predictions_with_intervals.iterrows():
            game_id = row['game_id']
            quarter = row['current_quarter']
            score_diff = abs(row['predicted_home_score'] - row['predicted_away_score'])
            context = "close game" if score_diff <= 5.0 else "blowout"
            
            if quarter == 4 and row.get('time_remaining', 12) <= 5.0:
                context = "final minutes"
            
            high_leverage = row.get('high_leverage', False)
            
            print(f"\nGame {game_id} (Q{quarter}, {context}, {'high leverage' if high_leverage else 'normal'}):")
            print(f"  Home: {row['predicted_home_score']:.1f} [{row['home_lower_bound']:.1f} - {row['home_upper_bound']:.1f}]")
            print(f"  Away: {row['predicted_away_score']:.1f} [{row['away_lower_bound']:.1f} - {row['away_upper_bound']:.1f}]")
            
            # Check if actuals fall within intervals
            actual = actuals[actuals['game_id'] == game_id]
            if not actual.empty:
                actual = actual.iloc[0]
                home_in = actual['home_score'] >= row['home_lower_bound'] and actual['home_score'] <= row['home_upper_bound']
                away_in = actual['away_score'] >= row['away_lower_bound'] and actual['away_score'] <= row['away_upper_bound']
                
                print(f"  Actual Home: {actual['home_score']} ({'within' if home_in else 'outside'} interval)")
                print(f"  Actual Away: {actual['away_score']} ({'within' if away_in else 'outside'} interval)")
        
        # Print coverage summary
        print("\nCoverage Summary:")
        for quarter in range(5):
            quarter_data = coverage_results[coverage_results['quarter'] == quarter]
            if not quarter_data.empty:
                home_coverage = quarter_data['home_in_interval'].mean()
                away_coverage = quarter_data['away_in_interval'].mean()
                full_coverage = quarter_data['in_interval'].mean()
                
                print(f"  Quarter {quarter}: Home {home_coverage:.0%}, Away {away_coverage:.0%}, Overall {full_coverage:.0%}")
    
    # Run a calibration iteration
    if display_details:
        print("\nRunning calibration iteration...")
    result = calibration_system.run_calibration_iteration(predictions, actuals)
    
    # Analyze high-leverage performance
    high_leverage_analysis = calibration_system.analyze_high_leverage_performance(
        predictions, actuals)
    
    return {
        'predictions_with_intervals': predictions_with_intervals,
        'coverage_results': coverage_results,
        'calibration_result': result,
        'high_leverage_analysis': high_leverage_analysis
    }


# Integration with previous modules
def integrate_with_uncertainty_estimator(uncertainty_estimator: Any) -> ContextAwareConfidenceCalibration:
    """
    Integrate the context-aware calibration system with the UncertaintyEstimator from Cell 27.
    
    Args:
        uncertainty_estimator: Instance of UncertaintyEstimator from Cell 27
        
    Returns:
        ContextAwareConfidenceCalibration instance initialized with the estimator's width factors
    """
    # Extract width factors from the uncertainty estimator
    width_factors = getattr(uncertainty_estimator, 'width_factors', None)
    
    # Create a context-aware calibration system with those factors
    calibration_system = ContextAwareConfidenceCalibration(
        base_width_factors=width_factors
    )
    
    return calibration_system


# Run demo if this cell is executed directly
if __name__ == "__main__":
    demo_results = demo_confidence_calibration()

In [ ]:
# Cell 27E: Comprehensive Performance Profiling and Optimization with Enhanced Feature Calculator

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, HTML
from typing import Dict, List, Tuple, Optional, Union, Any
import json
from datetime import datetime, timedelta
import pytz
import os

# ------------------ NEW HELPER: Fetch Live Games from Supabase ------------------

from caching.supabase_client import supabase

def fetch_live_games_from_supabase_27E() -> pd.DataFrame:
    """
    Pull all rows from 'nba_live_game_stats' for today's date in Pacific Time,
    filter for rows whose 'status' (uppercased) is in {Q1, Q2, Q3, Q4, OT, BT, HT},
    and set 'game_status' to 'LIVE' for those rows.
    This ensures we only see truly live games, matching Cell 29's fix.
    """
    try:
        # Fetch all rows from Supabase
        response = supabase.table("nba_live_game_stats").select("*").execute()
        if not response.data:
            print("No game data found in 'nba_live_game_stats'.")
            return pd.DataFrame()

        df = pd.DataFrame(response.data)

        # Convert game_date to datetime, localize to Pacific Time, filter for today
        if "game_date" in df.columns:
            df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce", utc=True)
            pacific_tz = pytz.timezone("America/Los_Angeles")
            df["game_date_pt"] = df["game_date"].dt.tz_convert(pacific_tz)
            df["date_only_pt"] = df["game_date_pt"].dt.date
            now_pt = datetime.now(pacific_tz)
            today_pt = now_pt.date()
            df = df[df["date_only_pt"] == today_pt].copy()
            print(f"Found {len(df)} games scheduled for today in PT (Cell 27E).")
        else:
            print("Column 'game_date' not found in the Supabase data!")
            return pd.DataFrame()

        # Normalize 'status' column and filter for live statuses
        if "status" in df.columns:
            df["status"] = df["status"].astype(str).str.upper()
        else:
            df["status"] = ""

        live_statuses = {"Q1", "Q2", "Q3", "Q4", "OT", "BT", "HT"}
        active_df = df[df["status"].isin(live_statuses)].copy()

        # Mark these as "LIVE" for consistent usage
        active_df["game_status"] = "LIVE"

        print(f"Fetched {len(active_df)} active live games (Cell 27E).")
        return active_df

    except Exception as e:
        print(f"Error in fetch_live_games_from_supabase_27E(): {e}")
        return pd.DataFrame()

# ------------------ BELOW: The classes from your original Cell 27E remain unchanged ------------------

class ColorManager:
    """
    Manages color schemes for NBA visualization components.
    ...
    (unchanged)
    """
    def __init__(self, base_scheme: str = "default"):
        # ... unchanged ...
        pass

    # ... rest of ColorManager code ...

class ConfidenceIndicator:
    """
    Creates compact, reusable visualization components for showing prediction confidence.
    ...
    (unchanged)
    """
    def __init__(self, color_manager: Optional[ColorManager] = None):
        # ... unchanged ...
        pass

    # ... rest of ConfidenceIndicator code ...

class GameVisualizer:
    """
    Creates visualizations for NBA game predictions with confidence intervals.
    ...
    (unchanged)
    """
    def __init__(self, color_manager: Optional[ColorManager] = None):
        # ... unchanged ...
        pass

    # ... rest of GameVisualizer code ...

class DashboardGenerator:
    """
    Generates complete dashboards for displaying NBA prediction confidence.
    ...
    (unchanged)
    """
    def __init__(self, 
                 color_manager: Optional[ColorManager] = None,
                 game_visualizer: Optional[GameVisualizer] = None):
        # ... unchanged ...
        pass

    # ... rest of DashboardGenerator code ...

class ConfidenceVisualizationSystem:
    """
    Complete system for generating NBA prediction confidence visualizations.
    ...
    (unchanged)
    """
    def __init__(self, color_scheme: str = 'default'):
        # ... unchanged ...
        pass

    # ... rest of ConfidenceVisualizationSystem code ...

def create_interactive_confidence_display(prediction_data: Dict[str, Any], 
                                        design_style: str = 'pwa') -> str:
    """
    Create an interactive SVG visualization showing prediction confidence.
    ...
    (unchanged)
    """
    # ... unchanged ...
    pass

# =============== REPLACED DEMO FUNCTION: fetch from supabase instead of sample data ===============

def demo_enhanced_visualization_27E():
    """
    Demonstrate the enhanced visualization using real-time data from Supabase,
    mirroring the fix from Cell 29 so we don't see old Lakers/Celtics, etc.
    """
    # 1) Fetch live games from Supabase
    live_games_df = fetch_live_games_from_supabase_27E()
    if live_games_df.empty:
        print("No active live games found for Cell 27E.")
        return

    # 2) Convert your DataFrame columns to match any needed naming. For example,
    #    if you rely on 'predicted_home_score' or 'predicted_away_score', ensure
    #    they exist or that your code can handle them. If your model is run earlier,
    #    you might already have columns. Otherwise, do a minimal placeholder:
    if 'predicted_home_score' not in live_games_df.columns:
        live_games_df['predicted_home_score'] = live_games_df['home_score'] + 10.0
    if 'predicted_away_score' not in live_games_df.columns:
        live_games_df['predicted_away_score'] = live_games_df['away_score'] + 8.0
    # For demonstration, set a basic 'win_probability' if missing
    if 'win_probability' not in live_games_df.columns:
        live_games_df['win_probability'] = 0.5

    # 3) Create a ConfidenceVisualizationSystem
    vis_system = ConfidenceVisualizationSystem(color_scheme="default")

    # 4) Generate a full dashboard for the live games
    dashboard_html = vis_system.create_dashboard(live_games_df, 
                                                 dashboard_id="confidence-dashboard-27E",
                                                 title="NBA Live Games (Cell 27E)")

    # 5) Display in a notebook if available
    try:
        from IPython.display import display, HTML
        display(HTML(dashboard_html))
    except:
        print(dashboard_html)

# If you want this code to run automatically, uncomment:
# if __name__ == "__main__":
#     demo_enhanced_visualization_27E()


In [ ]:
# Cell 28: Comprehensive Performance Profiling and Optimization with Enhanced Feature Calculator

import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import traceback
import os
import json
import hashlib
import pickle
from functools import wraps, lru_cache
from datetime import datetime, timedelta
import psutil
import gc
from typing import Dict, List, Tuple, Callable, Optional, Union, Any

# Define placeholders for imported functions to avoid missing module errors
def fetch_recent_games_for_testing(limit=5):
    """Placeholder for fetch_recent_games_for_testing function."""
    print("Missing fetch_recent_games_for_testing module, using fallback")
    return None

def prepare_features(df, model=None):
    """Placeholder for prepare_features function."""
    print("Missing prepare_features module, using fallback")
    return df.copy()

def run_predictions(model, features_df):
    """Placeholder for run_predictions function."""
    print("Missing run_predictions module, using fallback")
    return features_df.copy()

def add_uncertainty_to_predictions(predictions_df):
    """Placeholder for add_uncertainty_to_predictions function."""
    print("Missing add_uncertainty_to_predictions module, using fallback")
    return predictions_df.copy()

def load_model():
    """Placeholder for load_model function."""
    print("Missing load_model module, using fallback")
    return None

class PerformanceProfiler:
    """
    A comprehensive profiler for NBA prediction pipeline performance measurement.
    
    This class provides methods to:
    1. Measure real-world performance using actual NBA data
    2. Profile complete end-to-end pipelines, not just components
    3. Track both time and memory usage
    4. Generate visualizations and optimization recommendations
    """
    
    def __init__(self, profile_dir="./performance_profiles"):
        """
        Initialize the performance profiler.
        
        Args:
            profile_dir: Directory to store performance profiles
        """
        self.profile_dir = profile_dir
        os.makedirs(profile_dir, exist_ok=True)
        
        # Tracking data
        self.timing_data = {}
        self.memory_data = {}
        self.function_calls = {}
        self.bottlenecks = []
        
        # Store previous runs for comparison
        self.previous_profiles = self._load_previous_profiles()
    
    def _load_previous_profiles(self):
        """Load previous profiles from storage."""
        profiles = []
        try:
            profile_files = [f for f in os.listdir(self.profile_dir) 
                            if f.endswith('.json')]
            
            for filename in profile_files:
                try:
                    with open(os.path.join(self.profile_dir, filename), 'r') as f:
                        profile = json.load(f)
                        profiles.append(profile)
                except Exception as e:
                    print(f"Error loading profile {filename}: {e}")
                    
            print(f"Loaded {len(profiles)} previous performance profiles")
            return profiles
        except Exception as e:
            print(f"Error loading profiles: {e}")
            return []
    
    def profile_function(self, func=None, label=None, track_memory=True):
        """
        Decorator to profile function execution time and optional memory usage.
        
        Args:
            func: Function to profile
            label: Custom label for the function (defaults to function name)
            track_memory: Whether to track memory usage
            
        Returns:
            Decorated function
        """
        def decorator(f):
            @wraps(f)
            def wrapper(*args, **kwargs):
                # Start tracking
                func_name = label or f.__name__
                start_time = time.time()
                
                # Track memory before
                if track_memory:
                    gc.collect()  # Force garbage collection
                    before_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
                
                # Execute function
                try:
                    result = f(*args, **kwargs)
                    success = True
                except Exception as e:
                    success = False
                    print(f"Error in {func_name}: {e}")
                    traceback.print_exc()
                    raise
                finally:
                    # Record execution time
                    end_time = time.time()
                    execution_time = end_time - start_time
                    
                    # Record in timing data
                    if func_name not in self.timing_data:
                        self.timing_data[func_name] = []
                    self.timing_data[func_name].append(execution_time)
                    
                    # Track function calls
                    if func_name not in self.function_calls:
                        self.function_calls[func_name] = 0
                    self.function_calls[func_name] += 1
                    
                    # Track memory after
                    if track_memory:
                        gc.collect()  # Force garbage collection
                        after_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
                        memory_diff = after_memory - before_memory
                        
                        if func_name not in self.memory_data:
                            self.memory_data[func_name] = []
                        self.memory_data[func_name].append(memory_diff)
                        
                        # Print memory usage for long-running functions
                        if execution_time > 1.0:
                            print(f"{func_name} memory change: {memory_diff:.2f} MB")
                
                # Print timing for long-running functions
                if execution_time > 0.5:
                    print(f"{func_name} execution time: {execution_time:.3f} seconds")
                
                if success:
                    return result
            
            return wrapper
        
        # Handle direct decoration without arguments
        if func is not None:
            return decorator(func)
        return decorator
    
    def profile_pipeline(self, pipeline_func, label, *pipeline_args, **pipeline_kwargs):
        """
        Profile an entire pipeline end-to-end.
        
        Args:
            pipeline_func: Function that runs the entire pipeline
            label: Label for this pipeline run
            *pipeline_args, **pipeline_kwargs: Arguments to pass to pipeline function
            
        Returns:
            Pipeline result and performance data
        """
        print(f"\n{'='*50}\nProfiling pipeline: {label}\n{'='*50}")
        
        # Start tracking
        pipeline_start = time.time()
        gc.collect()  # Force garbage collection
        start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
        
        # Track component-level metrics with a context manager
        original_timing_data = self.timing_data.copy()
        original_memory_data = self.memory_data.copy()
        original_function_calls = self.function_calls.copy()
        
        # Run the pipeline
        try:
            result = pipeline_func(*pipeline_args, **pipeline_kwargs)
            success = True
        except Exception as e:
            success = False
            print(f"Pipeline error: {e}")
            traceback.print_exc()
            result = None
        finally:
            # Record execution time
            pipeline_end = time.time()
            execution_time = pipeline_end - pipeline_start
            
            # Record memory usage
            gc.collect()  # Force garbage collection
            end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
            memory_diff = end_memory - start_memory
            
            # Create profile data
            profile = {
                'label': label,
                'timestamp': datetime.now().isoformat(),
                'success': success,
                'execution_time': execution_time,
                'memory_diff_mb': memory_diff,
                'component_metrics': self._get_component_metrics(
                    original_timing_data, original_memory_data, original_function_calls),
                'bottlenecks': self._identify_bottlenecks(),
                'optimization_recommendations': []
            }
            
            # Add optimization recommendations
            profile['optimization_recommendations'] = self._generate_optimization_recommendations(profile)
            
            # Store profile
            profile_filename = f"profile_{label.replace(' ', '_')}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
            with open(os.path.join(self.profile_dir, profile_filename), 'w') as f:
                json.dump(profile, f, indent=2)
            
            print(f"\nPipeline {label} completed in {execution_time:.3f} seconds")
            print(f"Memory change: {memory_diff:.2f} MB")
            
            # Return both the pipeline result and profile data
            return result, profile
    
    def _get_component_metrics(self, original_timing, original_memory, original_calls):
        """
        Calculate component-specific metrics based on before/after data.
        
        Args:
            original_timing: Timing data before pipeline run
            original_memory: Memory data before pipeline run
            original_calls: Function call counts before pipeline run
            
        Returns:
            Dict with component metrics
        """
        component_metrics = {}
        
        # Process timing data
        for func_name, times in self.timing_data.items():
            original_times = original_timing.get(func_name, [])
            new_times = times[len(original_times):]
            
            if new_times:
                component_metrics[func_name] = {
                    'calls': len(new_times),
                    'total_time': sum(new_times),
                    'avg_time': sum(new_times) / len(new_times),
                    'min_time': min(new_times),
                    'max_time': max(new_times)
                }
                
                # Add memory metrics if available
                if func_name in self.memory_data:
                    original_mem = original_memory.get(func_name, [])
                    new_mem = self.memory_data[func_name][len(original_mem):]
                    
                    if new_mem:
                        component_metrics[func_name]['total_memory_mb'] = sum(new_mem)
                        component_metrics[func_name]['avg_memory_mb'] = sum(new_mem) / len(new_mem)
        
        return component_metrics
    
    def _identify_bottlenecks(self):
        """Identify performance bottlenecks in the pipeline."""
        bottlenecks = []
        
        # Time-based bottlenecks
        time_threshold = 0.5  # Functions taking > 0.5s on average
        
        for func_name, metrics in self._get_current_component_metrics().items():
            # Add to bottlenecks if average time exceeds threshold
            if metrics.get('avg_time', 0) > time_threshold:
                bottlenecks.append({
                    'component': func_name,
                    'type': 'execution_time',
                    'value': metrics['avg_time'],
                    'severity': 'high' if metrics['avg_time'] > 2.0 else 'medium',
                    'description': f"{func_name} takes {metrics['avg_time']:.2f}s on average"
                })
            
            # Memory-based bottlenecks (if function allocates > 50MB)
            if metrics.get('avg_memory_mb', 0) > 50:
                bottlenecks.append({
                    'component': func_name,
                    'type': 'memory_usage',
                    'value': metrics['avg_memory_mb'],
                    'severity': 'high' if metrics['avg_memory_mb'] > 200 else 'medium',
                    'description': f"{func_name} allocates {metrics['avg_memory_mb']:.2f}MB on average"
                })
                
            # Excessive function calls (if called more than 100 times)
            if metrics.get('calls', 0) > 100:
                bottlenecks.append({
                    'component': func_name,
                    'type': 'excessive_calls',
                    'value': metrics['calls'],
                    'severity': 'medium',
                    'description': f"{func_name} called {metrics['calls']} times"
                })
        
        # Sort by severity and value
        bottlenecks.sort(key=lambda x: (0 if x['severity'] == 'high' else 1, -x['value']))
        
        return bottlenecks
    
    def _get_current_component_metrics(self):
        """Get the most recent metrics for all components."""
        metrics = {}
        
        for func_name, times in self.timing_data.items():
            if times:
                metrics[func_name] = {
                    'calls': self.function_calls.get(func_name, 0),
                    'total_time': sum(times),
                    'avg_time': sum(times) / len(times),
                    'min_time': min(times),
                    'max_time': max(times)
                }
                
                # Add memory metrics if available
                if func_name in self.memory_data and self.memory_data[func_name]:
                    mem_values = self.memory_data[func_name]
                    metrics[func_name]['total_memory_mb'] = sum(mem_values)
                    metrics[func_name]['avg_memory_mb'] = sum(mem_values) / len(mem_values)
        
        return metrics
    
    def _generate_optimization_recommendations(self, profile):
        """Generate optimization recommendations based on performance profile."""
        recommendations = []
        
        # Process bottlenecks
        for bottleneck in profile.get('bottlenecks', []):
            component = bottleneck['component']
            bottleneck_type = bottleneck['type']
            
            if bottleneck_type == 'execution_time':
                if 'fetch' in component.lower() or 'get' in component.lower():
                    recommendations.append({
                        'component': component,
                        'recommendation': 'Implement caching for database or API requests',
                        'potential_impact': 'high',
                        'implementation_difficulty': 'medium'
                    })
                elif 'feature' in component.lower():
                    recommendations.append({
                        'component': component,
                        'recommendation': 'Optimize feature calculation with vectorized operations',
                        'potential_impact': 'high',
                        'implementation_difficulty': 'medium'
                    })
                elif 'model' in component.lower() or 'predict' in component.lower():
                    recommendations.append({
                        'component': component,
                        'recommendation': 'Consider model simplification or batch predictions',
                        'potential_impact': 'medium',
                        'implementation_difficulty': 'high'
                    })
            
            elif bottleneck_type == 'memory_usage':
                recommendations.append({
                    'component': component,
                    'recommendation': 'Reduce memory usage by processing data in chunks',
                    'potential_impact': 'medium',
                    'implementation_difficulty': 'medium'
                })
            
            elif bottleneck_type == 'excessive_calls':
                recommendations.append({
                    'component': component,
                    'recommendation': 'Reduce number of function calls through batching',
                    'potential_impact': 'high',
                    'implementation_difficulty': 'low'
                })
        
        # Add general recommendations if needed
        if profile.get('execution_time', 0) > 5.0:
            recommendations.append({
                'component': 'entire_pipeline',
                'recommendation': 'Implement parallel processing for independent tasks',
                'potential_impact': 'high',
                'implementation_difficulty': 'high'
            })
        
        return recommendations
    
    def visualize_profile(self, profile=None):
        """
        Create visualization of performance profile.
        
        Args:
            profile: Profile data dict (if None, visualizes most recent)
            
        Returns:
            Matplotlib figure
        """
        # Use the provided profile or the last run
        if profile is None:
            if not self.previous_profiles:
                print("No profiles available to visualize.")
                return None
            profile = self.previous_profiles[-1]
        
        # Create figure with subplots
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
        
        # Component timing chart (top)
        component_metrics = profile.get('component_metrics', {})
        components = []
        times = []
        colors = []
        
        for component, metrics in component_metrics.items():
            components.append(component)
            times.append(metrics.get('total_time', 0))
            
            # Determine color based on bottleneck severity
            is_bottleneck = False
            severity = 'low'
            
            for bottleneck in profile.get('bottlenecks', []):
                if bottleneck['component'] == component:
                    is_bottleneck = True
                    severity = bottleneck['severity']
                    break
            
            if is_bottleneck:
                if severity == 'high':
                    colors.append('#ff5252')  # Red for high severity
                else:
                    colors.append('#ffb142')  # Orange for medium severity
            else:
                colors.append('#2ecc71')  # Green for non-bottlenecks
        
        # Sort by execution time (descending)
        sorted_indices = np.argsort(times)[::-1]
        sorted_components = [components[i] for i in sorted_indices]
        sorted_times = [times[i] for i in sorted_indices]
        sorted_colors = [colors[i] for i in sorted_indices]
        
        # Plot component times
        ax1.barh(sorted_components, sorted_times, color=sorted_colors)
        ax1.set_title('Component Execution Times', fontsize=14)
        ax1.set_xlabel('Seconds', fontsize=12)
        ax1.set_ylabel('Component', fontsize=12)
        ax1.grid(axis='x', linestyle='--', alpha=0.7)
        
        # Add time labels
        for i, v in enumerate(sorted_times):
            if v >= 0.1:  # Only label significant times
                ax1.text(v + 0.05, i, f"{v:.2f}s", va='center')
        
        # Memory usage chart (bottom)
        memory_components = []
        memory_values = []
        memory_colors = []
        
        for component, metrics in component_metrics.items():
            if 'avg_memory_mb' in metrics:
                memory_components.append(component)
                memory_values.append(metrics['avg_memory_mb'])
                
                # Determine color based on memory usage
                if metrics['avg_memory_mb'] > 200:
                    memory_colors.append('#ff5252')  # Red for high memory
                elif metrics['avg_memory_mb'] > 50:
                    memory_colors.append('#ffb142')  # Orange for medium memory
                else:
                    memory_colors.append('#2ecc71')  # Green for low memory
        
        # Sort by memory usage (descending)
        if memory_components:
            mem_sorted_indices = np.argsort(memory_values)[::-1]
            mem_sorted_components = [memory_components[i] for i in mem_sorted_indices]
            mem_sorted_values = [memory_values[i] for i in mem_sorted_indices]
            mem_sorted_colors = [memory_colors[i] for i in mem_sorted_indices]
            
            # Plot memory usage
            ax2.barh(mem_sorted_components, mem_sorted_values, color=mem_sorted_colors)
            ax2.set_title('Component Memory Usage', fontsize=14)
            ax2.set_xlabel('Memory (MB)', fontsize=12)
            ax2.set_ylabel('Component', fontsize=12)
            ax2.grid(axis='x', linestyle='--', alpha=0.7)
            
            # Add memory labels
            for i, v in enumerate(mem_sorted_values):
                if v >= 1.0:  # Only label significant memory usage
                    ax2.text(v + 1, i, f"{v:.1f} MB", va='center')
        else:
            ax2.text(0.5, 0.5, 'No memory usage data available', 
                    ha='center', va='center', transform=ax2.transAxes, fontsize=14)
        
        # Add total metrics as a figure title
        plt.suptitle(
            f"Pipeline Performance Profile: {profile.get('label', 'Unnamed')}\n"
            f"Total Time: {profile.get('execution_time', 0):.2f}s, "
            f"Memory Change: {profile.get('memory_diff_mb', 0):.1f}MB",
            fontsize=16
        )
        
        plt.tight_layout()
        plt.subplots_adjust(top=0.9)
        
        return fig
    
    def compare_profiles(self, label1, label2=None):
        """
        Compare two performance profiles.
        
        Args:
            label1: Label of first profile
            label2: Label of second profile (if None, uses most recent)
            
        Returns:
            Comparison data and visualization
        """
        # Find profiles by label
        profile1 = None
        profile2 = None
        
        for profile in self.previous_profiles:
            if profile.get('label') == label1:
                profile1 = profile
            elif label2 and profile.get('label') == label2:
                profile2 = profile
        
        # If label2 is None, use the most recent profile that isn't profile1
        if label2 is None and profile1 is not None:
            for profile in reversed(self.previous_profiles):
                if profile.get('label') != label1:
                    profile2 = profile
                    label2 = profile.get('label', 'Unnamed')
                    break
        
        if not profile1 or not profile2:
            print(f"Couldn't find profiles to compare.")
            return None
        
        print(f"Comparing profiles: '{label1}' vs '{label2}'")
        
        # Calculate differences
        comparison = {
            'profile1': label1,
            'profile2': label2,
            'total_time_diff': profile2['execution_time'] - profile1['execution_time'],
            'total_time_pct': ((profile2['execution_time'] / profile1['execution_time']) - 1) * 100 
                             if profile1['execution_time'] > 0 else float('inf'),
            'memory_diff': profile2['memory_diff_mb'] - profile1['memory_diff_mb'],
            'component_diffs': {}
        }
        
        # Compare common components
        components1 = profile1.get('component_metrics', {})
        components2 = profile2.get('component_metrics', {})
        
        all_components = set(components1.keys()) | set(components2.keys())
        
        for component in all_components:
            metrics1 = components1.get(component, {})
            metrics2 = components2.get(component, {})
            
            # Skip if component doesn't exist in one profile
            if not metrics1 or not metrics2:
                continue
            
            # Calculate time differences
            time1 = metrics1.get('total_time', 0)
            time2 = metrics2.get('total_time', 0)
            time_diff = time2 - time1
            time_pct = ((time2 / time1) - 1) * 100 if time1 > 0 else float('inf')
            
            # Calculate memory differences if available
            mem_diff = None
            mem_pct = None
            
            if 'total_memory_mb' in metrics1 and 'total_memory_mb' in metrics2:
                mem1 = metrics1['total_memory_mb']
                mem2 = metrics2['total_memory_mb']
                mem_diff = mem2 - mem1
                mem_pct = ((mem2 / mem1) - 1) * 100 if mem1 > 0 else float('inf')
            
            # Store component differences
            comparison['component_diffs'][component] = {
                'time_diff': time_diff,
                'time_pct': time_pct,
                'memory_diff': mem_diff,
                'memory_pct': mem_pct
            }
        
        # Create visualization
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # Extract component data for plotting
        components = []
        time_pcts = []
        colors = []
        
        for component, diffs in comparison['component_diffs'].items():
            components.append(component)
            pct = diffs['time_pct']
            time_pcts.append(pct)
            
            # Color based on performance change
            if pct <= -10:  # Improved by 10%+
                colors.append('#2ecc71')  # Green
            elif pct >= 10:  # Worse by 10%+
                colors.append('#ff5252')  # Red
            else:
                colors.append('#3498db')  # Blue for minor changes
        
        # Sort by percentage change
        sorted_indices = np.argsort(time_pcts)
        sorted_components = [components[i] for i in sorted_indices]
        sorted_pcts = [time_pcts[i] for i in sorted_indices]
        sorted_colors = [colors[i] for i in sorted_indices]
        
        # Plot percentage changes
        bars = ax.barh(sorted_components, sorted_pcts, color=sorted_colors)
        
        # Add zero line
        ax.axvline(0, color='black', linestyle='-', alpha=0.7)
        
        # Add percentage labels
        for i, (bar, pct) in enumerate(zip(bars, sorted_pcts)):
            if abs(pct) >= 1.0:  # Only label significant changes
                text_color = 'white' if abs(pct) > 20 else 'black'
                x_pos = bar.get_width() if pct >= 0 else bar.get_width() - 5
                ax.text(x_pos, bar.get_y() + bar.get_height()/2, 
                       f"{pct:.1f}%", va='center', ha='right' if pct < 0 else 'left',
                       color=text_color, fontweight='bold')
        
        # Set chart title and labels
        ax.set_title(f"Performance Comparison: '{label2}' vs '{label1}'", fontsize=14)
        ax.set_xlabel('Percentage Change (%)', fontsize=12)
        ax.set_ylabel('Component', fontsize=12)
        
        # Add overall metrics as text
        time_change = comparison['total_time_pct']
        time_symbol = "↓" if time_change <= 0 else "↑"
        
        memory_pct = (comparison['memory_diff'] / profile1['memory_diff_mb']) * 100 if profile1['memory_diff_mb'] != 0 else float('inf')
        memory_symbol = "↓" if comparison['memory_diff'] <= 0 else "↑"
        
        overall_text = (
            f"Overall Time: {abs(time_change):.1f}% {time_symbol} " 
            f"({profile1['execution_time']:.2f}s → {profile2['execution_time']:.2f}s)\n"
            f"Overall Memory: {abs(memory_pct):.1f}% {memory_symbol} "
            f"({profile1['memory_diff_mb']:.1f}MB → {profile2['memory_diff_mb']:.1f}MB)"
        )
        
        plt.figtext(0.5, 0.01, overall_text, ha='center', fontsize=12, 
                   bbox=dict(facecolor='#f9f9f9', alpha=0.5, boxstyle='round,pad=0.5'))
        
        plt.grid(axis='x', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.subplots_adjust(bottom=0.1)
        
        return comparison, fig


class OptimizedFeatureCalculator:
    """
    Optimized feature calculation with caching and vectorized operations.
    
    This class implements optimized methods for calculating prediction features
    with advanced caching and performance tracking.
    """
    
    def __init__(self, cache_dir: str = "./feature_cache", 
                max_cache_entries: int = 100,
                cache_expiry_seconds: int = 3600):
        """
        Initialize the optimized feature calculator.
        
        Args:
            cache_dir: Directory to store cache files
            max_cache_entries: Maximum number of entries in memory cache
            cache_expiry_seconds: Time in seconds after which cache entries expire
        """
        # Create cache directory if it doesn't exist
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)
        
        # Cache settings
        self.max_cache_entries = max_cache_entries
        self.cache_expiry_seconds = cache_expiry_seconds
        
        # Initialize in-memory cache
        self.memory_cache = {}
        self.cache_timestamps = {}
        self.cache_hits = 0
        self.cache_misses = 0
        
        # Feature computation timing
        self.timing_stats = {
            'momentum_calculation': [],
            'rest_calculation': [],
            'matchup_calculation': [],
            'full_feature_generation': []
        }
    
    def _generate_cache_key(self, df: pd.DataFrame) -> str:
        """
        Generate a unique cache key for the DataFrame.
        
        Args:
            df: Input DataFrame
            
        Returns:
            str: Unique hash key for the DataFrame
        """
        # Extract core columns for key generation
        key_cols = ['game_id', 'current_quarter', 'home_score', 'away_score']
        key_cols = [col for col in key_cols if col in df.columns]
        
        if not key_cols:
            # Fallback to using all relevant columns
            all_cols = df.columns.tolist()
            possible_cols = ['game_id', 'home_team', 'away_team', 'current_quarter', 
                            'home_score', 'away_score', 'game_date']
            key_cols = [col for col in possible_cols if col in all_cols]
        
        # Create a string representation of key columns
        if key_cols:
            key_data = df[key_cols].to_json()
        else:
            # Last resort, use hash of the entire DataFrame
            key_data = str(hash(tuple(map(tuple, df.values))))
        
        # Generate hash
        return hashlib.md5(key_data.encode()).hexdigest()
    
    def _get_cache_path(self, cache_key: str) -> str:
        """Get cache file path from key."""
        return os.path.join(self.cache_dir, f"feature_cache_{cache_key}.pkl")
    
    def _load_from_cache(self, cache_key: str) -> Optional[pd.DataFrame]:
        """
        Load features from cache.
        
        Args:
            cache_key: Cache key
            
        Returns:
            Optional[pd.DataFrame]: Cached features or None if not found
        """
        # First check memory cache
        if cache_key in self.memory_cache:
            # Check if expired
            timestamp = self.cache_timestamps.get(cache_key, 0)
            if time.time() - timestamp <= self.cache_expiry_seconds:
                self.cache_hits += 1
                return self.memory_cache[cache_key]
            else:
                # Expired, remove from memory cache
                del self.memory_cache[cache_key]
                del self.cache_timestamps[cache_key]
        
        # Then check disk cache
        cache_path = self._get_cache_path(cache_key)
        if os.path.exists(cache_path):
            try:
                with open(cache_path, 'rb') as f:
                    cache_data = pickle.load(f)
                
                # Check timestamp
                timestamp = cache_data.get('timestamp', 0)
                if time.time() - timestamp <= self.cache_expiry_seconds:
                    # Add to memory cache
                    self.memory_cache[cache_key] = cache_data['features']
                    self.cache_timestamps[cache_key] = timestamp
                    
                    # Manage cache size
                    self._manage_cache_size()
                    
                    self.cache_hits += 1
                    return cache_data['features']
                else:
                    # Expired, remove file
                    os.remove(cache_path)
            except Exception as e:
                print(f"Error loading from cache: {str(e)}")
        
        self.cache_misses += 1
        return None
    
    def _save_to_cache(self, cache_key: str, features: pd.DataFrame):
        """
        Save features to cache.
        
        Args:
            cache_key: Cache key
            features: Feature DataFrame to cache
        """
        timestamp = time.time()
        
        # Save to memory cache
        self.memory_cache[cache_key] = features
        self.cache_timestamps[cache_key] = timestamp
        
        # Manage cache size
        self._manage_cache_size()
        
        # Save to disk cache
        cache_path = self._get_cache_path(cache_key)
        try:
            with open(cache_path, 'wb') as f:
                pickle.dump({
                    'features': features,
                    'timestamp': timestamp
                }, f)
        except Exception as e:
            print(f"Error saving to cache: {str(e)}")
    
    def _manage_cache_size(self):
        """Manage memory cache size by removing oldest entries."""
        if len(self.memory_cache) > self.max_cache_entries:
            # Sort by timestamp and remove oldest entries
            sorted_keys = sorted(self.cache_timestamps.items(), key=lambda x: x[1])
            
            # Remove oldest entries to get back to max size
            num_to_remove = len(self.memory_cache) - self.max_cache_entries
            for i in range(num_to_remove):
                key_to_remove = sorted_keys[i][0]
                del self.memory_cache[key_to_remove]
                del self.cache_timestamps[key_to_remove]
    
    def calculate_features(self, df: pd.DataFrame, force_recalc: bool = False) -> pd.DataFrame:
        """
        Calculate features with caching for performance.
        
        Args:
            df: Input DataFrame
            force_recalc: Force recalculation even if cached
            
        Returns:
            pd.DataFrame: DataFrame with calculated features
        """
        # Skip caching for empty DataFrames
        if df.empty:
            return df.copy()
        
        # Generate cache key
        cache_key = self._generate_cache_key(df)
        
        # Try to load from cache if not forcing recalculation
        if not force_recalc:
            cached_features = self._load_from_cache(cache_key)
            if cached_features is not None:
                # Check for significant differences that would invalidate cache
                if self._is_cache_valid(df, cached_features):
                    return cached_features
        
        # No valid cache hit, calculate features
        start_time = time.time()
        
        # Calculate all features
        features_df = self._calculate_all_features(df)
        
        # Track timing
        elapsed = time.time() - start_time
        self.timing_stats['full_feature_generation'].append(elapsed)
        
        # Save to cache
        self._save_to_cache(cache_key, features_df)
        
        return features_df
    
    def _is_cache_valid(self, original_df: pd.DataFrame, cached_df: pd.DataFrame) -> bool:
        """
        Check if cached features are still valid by comparing key fields.
        
        Args:
            original_df: Original input DataFrame
            cached_df: Cached feature DataFrame
            
        Returns:
            bool: True if cache is still valid
        """
        # Check if game IDs match
        if 'game_id' in original_df.columns and 'game_id' in cached_df.columns:
            orig_ids = set(original_df['game_id'])
            cached_ids = set(cached_df['game_id'])
            if orig_ids != cached_ids:
                return False
        
        # Check if quarters match
        if 'current_quarter' in original_df.columns and 'current_quarter' in cached_df.columns:
            for idx, row in original_df.iterrows():
                game_id = row.get('game_id')
                if game_id is not None:
                    orig_quarter = row.get('current_quarter')
                    cached_row = cached_df[cached_df['game_id'] == game_id]
                    
                    if not cached_row.empty:
                        cached_quarter = cached_row.iloc[0].get('current_quarter')
                        
                        # If quarter advanced, cache is invalid
                        if orig_quarter > cached_quarter:
                            return False
        
        return True
    
    def _calculate_all_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate all features using optimized methods.
        
        Args:
            df: Input DataFrame
            
        Returns:
            pd.DataFrame: DataFrame with all calculated features
        """
        try:
            # Make a copy to avoid modifying the original
            features_df = df.copy()
            
            # Calculate features in order
            features_df = self._calculate_basic_features(features_df)
            features_df = self._calculate_momentum_features(features_df)
            features_df = self._calculate_rest_features(features_df)
            features_df = self._calculate_matchup_features(features_df)
            
            return features_df
            
        except Exception as e:
            print(f"Error calculating features: {str(e)}")
            traceback.print_exc()
            # Return original if calculation fails
            return df.copy()
    
    def _calculate_basic_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calculate basic features like score ratio."""
        # Calculate total score
        df['total_score'] = 0
        
        for idx, row in df.iterrows():
            home_score = float(row.get('home_score', 0) or 0)
            away_score = float(row.get('away_score', 0) or 0)
            df.at[idx, 'total_score'] = home_score + away_score
        
        # Calculate score ratio
        df['score_ratio'] = 0.5  # Default to even
        for idx, row in df.iterrows():
            home_score = float(row.get('home_score', 0) or 0)
            total = row.get('total_score', 0)
            
            if total > 0:
                df.at[idx, 'score_ratio'] = home_score / total
        
        return df
    
    def _calculate_momentum_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate momentum features using vectorized operations.
        
        Args:
            df: Input DataFrame
            
        Returns:
            pd.DataFrame: DataFrame with momentum features
        """
        start_time = time.time()
        
        try:
            # Prepare arrays for vectorized operations
            n_rows = len(df)
            current_quarter = np.array(df['current_quarter'].fillna(0).astype(int))
            
            # Extract quarter scores as numpy arrays
            home_q1 = np.array([float(row.get('home_q1', 0) or 0) for _, row in df.iterrows()])
            home_q2 = np.array([float(row.get('home_q2', 0) or 0) for _, row in df.iterrows()])
            home_q3 = np.array([float(row.get('home_q3', 0) or 0) for _, row in df.iterrows()])
            home_q4 = np.array([float(row.get('home_q4', 0) or 0) for _, row in df.iterrows()])
            
            away_q1 = np.array([float(row.get('away_q1', 0) or 0) for _, row in df.iterrows()])
            away_q2 = np.array([float(row.get('away_q2', 0) or 0) for _, row in df.iterrows()])
            away_q3 = np.array([float(row.get('away_q3', 0) or 0) for _, row in df.iterrows()])
            away_q4 = np.array([float(row.get('away_q4', 0) or 0) for _, row in df.iterrows()])
            
            # Calculate quarter-to-quarter momentum shifts
            q1_to_q2_momentum = (home_q2 - home_q1) - (away_q2 - away_q1)
            q2_to_q3_momentum = (home_q3 - home_q2) - (away_q3 - away_q2)
            q3_to_q4_momentum = (home_q4 - home_q3) - (away_q4 - away_q3)
            
            # Initialize momentum arrays
            q1_to_q2_momentum_final = np.zeros(n_rows)
            q2_to_q3_momentum_final = np.zeros(n_rows)
            q3_to_q4_momentum_final = np.zeros(n_rows)
            cumulative_momentum = np.zeros(n_rows)
            
            # Apply masks based on current quarter
            q1_mask = current_quarter >= 2
            q1_to_q2_momentum_final[q1_mask] = q1_to_q2_momentum[q1_mask]
            
            q2_mask = current_quarter >= 3
            q2_to_q3_momentum_final[q2_mask] = q2_to_q3_momentum[q2_mask]
            
            q3_mask = current_quarter >= 4
            q3_to_q4_momentum_final[q3_mask] = q3_to_q4_momentum[q3_mask]
            
            # Calculate cumulative momentum based on quarter
            q2_only_mask = current_quarter == 2
            q3_only_mask = current_quarter == 3
            q4_only_mask = current_quarter >= 4
            
            # Weights for each quarter-to-quarter momentum
            weights = np.array([0.2, 0.3, 0.5])
            
            # Apply to each quarter scenario
            cumulative_momentum[q2_only_mask] = q1_to_q2_momentum_final[q2_only_mask]
            
            cumulative_momentum[q3_only_mask] = (
                q1_to_q2_momentum_final[q3_only_mask] * weights[0] +
                q2_to_q3_momentum_final[q3_only_mask] * weights[1]
            ) / (weights[0] + weights[1])
            
            cumulative_momentum[q4_only_mask] = (
                q1_to_q2_momentum_final[q4_only_mask] * weights[0] +
                q2_to_q3_momentum_final[q4_only_mask] * weights[1] +
                q3_to_q4_momentum_final[q4_only_mask] * weights[2]
            ) / np.sum(weights)
            
            # Normalize to [-1, 1]
            cumulative_momentum = np.clip(cumulative_momentum / 15.0, -1.0, 1.0)
            
            # Add results to DataFrame
            df['q1_to_q2_momentum'] = q1_to_q2_momentum_final
            df['q2_to_q3_momentum'] = q2_to_q3_momentum_final
            df['q3_to_q4_momentum'] = q3_to_q4_momentum_final
            df['cumulative_momentum'] = cumulative_momentum
            
        except Exception as e:
            print(f"Error in momentum calculation: {str(e)}")
            traceback.print_exc()
            
            # Initialize momentum features with zeros
            df['q1_to_q2_momentum'] = 0
            df['q2_to_q3_momentum'] = 0
            df['q3_to_q4_momentum'] = 0
            df['cumulative_momentum'] = 0
        
        # Track timing
        elapsed = time.time() - start_time
        self.timing_stats['momentum_calculation'].append(elapsed)
        
        return df
    
    def _calculate_rest_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate rest-related features.
        
        Args:
            df: Input DataFrame
            
        Returns:
            pd.DataFrame: DataFrame with rest features
        """
        start_time = time.time()
        
        try:
            # Set default values
            df['rest_days_home'] = 2
            df['rest_days_away'] = 2
            df['is_back_to_back_home'] = 0
            df['is_back_to_back_away'] = 0
            df['rest_advantage'] = 0
            
            # We'd need game schedule data to properly calculate rest
            # This would typically involve querying a database
            
            # For demonstration, we'll use default values
            # In a real implementation, you would look up previous games
            
            # Calculate rest advantage
            df['rest_advantage'] = df['rest_days_home'] - df['rest_days_away']
            
        except Exception as e:
            print(f"Error in rest calculation: {str(e)}")
            
            # Set defaults if calculation fails
            df['rest_days_home'] = 2
            df['rest_days_away'] = 2
            df['is_back_to_back_home'] = 0
            df['is_back_to_back_away'] = 0
            df['rest_advantage'] = 0
        
        # Track timing
        elapsed = time.time() - start_time
        self.timing_stats['rest_calculation'].append(elapsed)
        
        return df
    
    def _calculate_matchup_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate matchup-related features.
        
        Args:
            df: Input DataFrame
            
        Returns:
            pd.DataFrame: DataFrame with matchup features
        """
        start_time = time.time()
        
        try:
            # Set default values
            df['prev_matchup_diff'] = 0
            
            # We'd need historical data to properly calculate matchup differentials
            # This would typically involve querying a database
            
            # For demonstration, we'll use default values
            # In a real implementation, you would look up previous matchups
            
        except Exception as e:
            print(f"Error in matchup calculation: {str(e)}")
            
            # Set defaults if calculation fails
            df['prev_matchup_diff'] = 0
        
        # Track timing
        elapsed = time.time() - start_time
        self.timing_stats['matchup_calculation'].append(elapsed)
        
        return df
    
    def get_performance_stats(self) -> Dict[str, Any]:
        """
        Get performance statistics for feature calculation.
        
        Returns:
            Dict with performance statistics
        """
        stats = {
            'cache_hits': self.cache_hits,
            'cache_misses': self.cache_misses,
            'cache_hit_rate': self.cache_hits / (self.cache_hits + self.cache_misses) if (self.cache_hits + self.cache_misses) > 0 else 0
        }
        
        # Add timing stats
        for feature, times in self.timing_stats.items():
            if times:
                stats[f'{feature}_avg_ms'] = np.mean(times) * 1000
                stats[f'{feature}_min_ms'] = np.min(times) * 1000
                stats[f'{feature}_max_ms'] = np.max(times) * 1000
        
        return stats
    
    def clear_cache(self, older_than_seconds: Optional[int] = None):
        """
        Clear the feature cache.
        
        Args:
            older_than_seconds: Only clear entries older than this many seconds
        """
        # Clear memory cache
        if older_than_seconds is None:
            # Clear all
            self.memory_cache = {}
            self.cache_timestamps = {}
        else:
            # Clear only old entries
            current_time = time.time()
            keys_to_remove = []
            
            for key, timestamp in self.cache_timestamps.items():
                if current_time - timestamp > older_than_seconds:
                    keys_to_remove.append(key)
            
            for key in keys_to_remove:
                del self.memory_cache[key]
                del self.cache_timestamps[key]
        
        # Clear disk cache
        for filename in os.listdir(self.cache_dir):
            if filename.startswith("feature_cache_") and filename.endswith(".pkl"):
                file_path = os.path.join(self.cache_dir, filename)
                
                if older_than_seconds is None:
                    # Remove all cache files
                    os.remove(file_path)
                else:
                    # Check timestamp
                    try:
                        with open(file_path, 'rb') as f:
                            cache_data = pickle.load(f)
                        
                        timestamp = cache_data.get('timestamp', 0)
                        if time.time() - timestamp > older_than_seconds:
                            os.remove(file_path)
                    except:
                        # Remove if can't read timestamp
                        os.remove(file_path)


class PredictionPipeline:
    """
    End-to-end prediction pipeline for benchmarking.
    
    This class implements a complete prediction pipeline that can be used
    to benchmark performance with various optimization strategies.
    """
    
    def __init__(self, use_optimization=False, model=None):
        """
        Initialize the prediction pipeline.
        
        Args:
            use_optimization: Whether to use optimized implementations
            model: Pre-loaded prediction model
        """
        self.use_optimization = use_optimization
        self.model = model
        
        # Create optimizer if needed
        if use_optimization:
            self.feature_calculator = OptimizedFeatureCalculator()
        
        # Initialize profiler for internal measurement
        self.profiler = PerformanceProfiler()
    
    @property
    def optimization_status(self):
        """Get the current optimization status string."""
        return "Optimized" if self.use_optimization else "Standard"
    
    def set_optimization(self, use_optimization):
        """Toggle optimization on/off."""
        self.use_optimization = use_optimization
        if use_optimization and not hasattr(self, 'feature_calculator'):
            self.feature_calculator = OptimizedFeatureCalculator()
    
    def load_test_data(self, sample_size=5):
        """
        Load test data for benchmarking.
        
        Args:
            sample_size: Number of games to load
            
        Returns:
            DataFrame with test data
        """
        # Use decorator for profiling
        @self.profiler.profile_function(label="load_test_data")
        def _load_data():
            # Use our pre-defined fallback function
            try:
                return fetch_recent_games_for_testing(limit=sample_size)
            except:
                # Fallback to generating synthetic data
                print("Using synthetic test data")
                return self._generate_synthetic_data(sample_size)
        
        return _load_data()
    
    def _generate_synthetic_data(self, sample_size):
        """Generate synthetic game data for testing."""
        import random
        
        synthetic_games = []
        
        for i in range(sample_size):
            # Generate a random quarter (1-4)
            quarter = random.randint(1, 4)
            
            # Generate quarter scores
            home_quarters = [random.randint(20, 35) for _ in range(quarter)]
            away_quarters = [random.randint(20, 35) for _ in range(quarter)]
            
            # Fill remaining quarters with zeros
            home_quarters.extend([0] * (4 - quarter))
            away_quarters.extend([0] * (4 - quarter))
            
            # Calculate current score
            home_score = sum(home_quarters)
            away_score = sum(away_quarters)
            
            # Create game dictionary
            game = {
                'game_id': 1000 + i,
                'home_team': f"Team{2*i}",
                'away_team': f"Team{2*i+1}",
                'current_quarter': quarter,
                'home_score': home_score,
                'away_score': away_score,
                'home_q1': home_quarters[0],
                'home_q2': home_quarters[1],
                'home_q3': home_quarters[2],
                'home_q4': home_quarters[3],
                'away_q1': away_quarters[0],
                'away_q2': away_quarters[1],
                'away_q3': away_quarters[2],
                'away_q4': away_quarters[3],
                'simulated': True
            }
            
            synthetic_games.append(game)
        
        return pd.DataFrame(synthetic_games)
    
    def prepare_features(self, games_df):
        """
        Prepare features for prediction.
        
        Args:
            games_df: DataFrame with game data
            
        Returns:
            DataFrame with features for prediction
        """
        # Use decorator for profiling
        @self.profiler.profile_function(label="prepare_features")
        def _prepare():
            if self.use_optimization:
                return self.feature_calculator.calculate_features(games_df)
            else:
                # Use our pre-defined fallback function
                try:
                    return prepare_features(games_df, self.model)
                except:
                    # Fallback to basic implementation
                    print("Using basic feature preparation")
                    return self._basic_feature_preparation(games_df)
        
        return _prepare()
    
    def _basic_feature_preparation(self, games_df):
        """Basic feature preparation implementation."""
        features_df = games_df.copy()
        
        # Ensure numeric types
        for col in games_df.columns:
            if col.startswith('home_q') or col.startswith('away_q') or col in ['home_score', 'away_score']:
                features_df[col] = pd.to_numeric(features_df[col], errors='coerce').fillna(0)
        
        # Calculate score ratio
        total_score = features_df['home_score'] + features_df['away_score']
        features_df['score_ratio'] = np.where(
            total_score > 0,
            features_df['home_score'] / total_score,
            0.5
        )
        
        # Add placeholder values for other features
        features_df['prev_matchup_diff'] = 0
        features_df['rolling_home_score'] = 105.0
        features_df['rolling_away_score'] = 105.0
        
        return features_df
    
    def make_predictions(self, features_df):
        """
        Make predictions using the model.
        
        Args:
            features_df: DataFrame with features
            
        Returns:
            DataFrame with predictions
        """
        # Use decorator for profiling
        @self.profiler.profile_function(label="make_predictions")
        def _predict():
            if self.model is None:
                return self._simulate_predictions(features_df)
            
            # Use our pre-defined fallback function
            try:
                return run_predictions(self.model, features_df)
            except Exception as e:
                print(f"Error using run_predictions: {e}")
                return self._basic_predictions(features_df)
        
        return _predict()
    
    def _simulate_predictions(self, features_df):
        """Simulate predictions when no model is available."""
        results = []
        
        for _, game in features_df.iterrows():
            # Extract game info
            game_id = game['game_id']
            home_team = game['home_team']
            away_team = game['away_team']
            current_quarter = int(game['current_quarter'])
            home_score = float(game['home_score'])
            away_score = float(game['away_score'])
            
            # Simple projection based on current score and quarter
            if current_quarter > 0:
                # Project final score based on current pace
                quarter_ratio = 4 / current_quarter
                predicted_home = home_score * quarter_ratio
                predicted_away = away_score * quarter_ratio
            else:
                # Pre-game prediction
                predicted_home = 105.0
                predicted_away = 102.0
            
            # Create result dictionary
            result = {
                'game_id': game_id,
                'home_team': home_team,
                'away_team': away_team,
                'current_quarter': current_quarter,
                'current_home_score': home_score,
                'current_away_score': away_score,
                'predicted_home_final': predicted_home,
                'predicted_away_final': predicted_away,
                'win_probability': 0.5 + (predicted_home - predicted_away) / 40.0  # Simple win probability
            }
            
            results.append(result)
        
        return pd.DataFrame(results)
    
    def _basic_predictions(self, features_df):
        """Basic prediction implementation when standard function isn't available."""
        if self.model is None:
            return self._simulate_predictions(features_df)
        
        # Get model features based on model type
        model_features = []
        
        if hasattr(self.model, 'feature_names_in_'):
            model_features = self.model.feature_names_in_
        elif hasattr(self.model, 'feature_importances_'):
            # Use the first 8 feature names as an approximation
            base_features = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4', 
                'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
            ]
            model_features = base_features[:len(self.model.feature_importances_)]
        else:
            # Default basic features
            model_features = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4', 
                'score_ratio'
            ]
        
        # Ensure all features exist
        for feature in model_features:
            if feature not in features_df.columns:
                features_df[feature] = 0
        
        # Make predictions
        X = features_df[model_features]
        predictions = self.model.predict(X)
        
        # Create results
        results_df = features_df.copy()
        results_df['predicted_home_final'] = predictions
        
        # Estimate away score based on home score and score ratio
        results_df['predicted_away_final'] = predictions * (1 - results_df['score_ratio']) / results_df['score_ratio']
        
        # Calculate win probability
        score_diff = results_df['predicted_home_final'] - results_df['predicted_away_final']
        results_df['win_probability'] = 1.0 / (1.0 + np.exp(-0.1 * score_diff))
        
        return results_df
    
    def post_process_predictions(self, predictions_df):
        """
        Apply post-processing to predictions.
        
        Args:
            predictions_df: DataFrame with raw predictions
            
        Returns:
            DataFrame with processed predictions
        """
        # Use decorator for profiling
        @self.profiler.profile_function(label="post_process_predictions")
        def _post_process():
            processed_df = predictions_df.copy()
            
            # Apply uncertainty estimation if optimized
            if self.use_optimization:
                return self._apply_optimized_post_processing(processed_df)
            else:
                return self._basic_post_processing(processed_df)
        
        return _post_process()
    
    def _apply_optimized_post_processing(self, predictions_df):
        """Apply optimized post-processing."""
        # Use our pre-defined fallback function
        try:
            return add_uncertainty_to_predictions(predictions_df)
        except:
            # Fallback to basic
            return self._basic_post_processing(predictions_df)
    
    def _basic_post_processing(self, predictions_df):
        """Apply basic post-processing to predictions."""
        # Ensure home prediction is at least current score
        predictions_df['predicted_home_final'] = np.maximum(
            predictions_df['predicted_home_final'],
            predictions_df['current_home_score']
        )
        
        # Ensure away prediction is at least current score
        if 'current_away_score' in predictions_df.columns:
            predictions_df['predicted_away_final'] = np.maximum(
                predictions_df['predicted_away_final'],
                predictions_df['current_away_score']
            )
        
        # Add simple confidence intervals
        quarter_width = {0: 30, 1: 25, 2: 20, 3: 15, 4: 10}
        
        predictions_df['home_lower_bound'] = predictions_df.apply(
            lambda row: row['predicted_home_final'] - quarter_width.get(row['current_quarter'], 20) / 2,
            axis=1
        )
        
        predictions_df['home_upper_bound'] = predictions_df.apply(
            lambda row: row['predicted_home_final'] + quarter_width.get(row['current_quarter'], 20) / 2,
            axis=1
        )
        
        predictions_df['away_lower_bound'] = predictions_df.apply(
            lambda row: row['predicted_away_final'] - quarter_width.get(row['current_quarter'], 20) / 2,
            axis=1
        )
        
        predictions_df['away_upper_bound'] = predictions_df.apply(
            lambda row: row['predicted_away_final'] + quarter_width.get(row['current_quarter'], 20) / 2,
            axis=1
        )
        
        return predictions_df
    
    def run_complete_pipeline(self, input_data=None, sample_size=5):
        """
        Run the complete prediction pipeline end-to-end.
        
        Args:
            input_data: Optional input data (loads test data if None)
            sample_size: Number of games to load if input_data is None
            
        Returns:
            DataFrame with complete prediction results
        """
        # Load data if not provided
        if input_data is None:
            games_df = self.load_test_data(sample_size)
        else:
            games_df = input_data
        
        if games_df is None or (isinstance(games_df, pd.DataFrame) and games_df.empty):
            print("No games available for processing")
            return pd.DataFrame()
        
        # Generate and process features
        features_df = self.prepare_features(games_df)
        
        # Make predictions
        predictions_df = self.make_predictions(features_df)
        
        # Post-process predictions
        final_predictions = self.post_process_predictions(predictions_df)
        
        return final_predictions


def benchmark_performance(compare_optimization=True, sample_sizes=[1, 5, 10], iterations=3):
    """
    Run comprehensive performance benchmarks.
    
    Args:
        compare_optimization: Whether to compare optimized vs. standard
        sample_sizes: List of sample sizes to test
        iterations: Number of iterations for each test
        
    Returns:
        Dictionary with benchmark results
    """
    print("\n" + "="*80)
    print(f"Running performance benchmarks ({iterations} iterations each)")
    print("="*80)
    
    # Initialize profiler
    profiler = PerformanceProfiler()
    
    # Try to load model - use our pre-defined fallback function
    model = None
    try:
        model = load_model()
        print("Successfully loaded prediction model")
    except Exception as e:
        print(f"Couldn't load model, will use simulated predictions: {e}")
    
    # Initialize pipelines
    standard_pipeline = PredictionPipeline(use_optimization=False, model=model)
    optimized_pipeline = PredictionPipeline(use_optimization=True, model=model)
    
    # Initialize results
    results = {
        'standard': {},
        'optimized': {},
        'comparison': {},
        'sample_sizes': sample_sizes,
        'iterations': iterations
    }
    
    # Run benchmarks for different sample sizes
    for sample_size in sample_sizes:
        print(f"\nTesting with {sample_size} games:")
        
        # Standard pipeline
        standard_times = []
        standard_memory = []
        
        for i in range(iterations):
            print(f"  Standard pipeline iteration {i+1}/{iterations}...")
            
            # Profile the complete pipeline
            _, profile = profiler.profile_pipeline(
                standard_pipeline.run_complete_pipeline,
                f"Standard_Size{sample_size}_Run{i+1}",
                None,
                sample_size
            )
            
            standard_times.append(profile['execution_time'])
            standard_memory.append(profile['memory_diff_mb'])
        
        # Store standard results
        results['standard'][sample_size] = {
            'avg_time': sum(standard_times) / len(standard_times),
            'min_time': min(standard_times),
            'max_time': max(standard_times),
            'avg_memory': sum(standard_memory) / len(standard_memory),
            'bottlenecks': profile.get('bottlenecks', [])
        }
        
        # Skip optimized if not comparing
        if not compare_optimization:
            continue
        
        # Optimized pipeline
        optimized_times = []
        optimized_memory = []
        
        for i in range(iterations):
            print(f"  Optimized pipeline iteration {i+1}/{iterations}...")
            
            # Profile the complete pipeline
            _, profile = profiler.profile_pipeline(
                optimized_pipeline.run_complete_pipeline,
                f"Optimized_Size{sample_size}_Run{i+1}",
                None,
                sample_size
            )
            
            optimized_times.append(profile['execution_time'])
            optimized_memory.append(profile['memory_diff_mb'])
        
        # Store optimized results
        results['optimized'][sample_size] = {
            'avg_time': sum(optimized_times) / len(optimized_times),
            'min_time': min(optimized_times),
            'max_time': max(optimized_times),
            'avg_memory': sum(optimized_memory) / len(optimized_memory),
            'bottlenecks': profile.get('bottlenecks', [])
        }
        
        # Calculate comparison metrics
        std_avg_time = results['standard'][sample_size]['avg_time']
        opt_avg_time = results['optimized'][sample_size]['avg_time']
        
        time_diff = std_avg_time - opt_avg_time
        time_pct = (time_diff / std_avg_time) * 100 if std_avg_time > 0 else 0
        
        results['comparison'][sample_size] = {
            'time_diff': time_diff,
            'time_pct_reduction': time_pct,
            'memory_diff': results['standard'][sample_size]['avg_memory'] - results['optimized'][sample_size]['avg_memory'],
            'speedup_factor': std_avg_time / opt_avg_time if opt_avg_time > 0 else 0
        }
        
        print(f"  Results for {sample_size} games:")
        print(f"    Standard: {std_avg_time:.3f}s")
        print(f"    Optimized: {opt_avg_time:.3f}s")
        print(f"    Improvement: {time_pct:.1f}% faster")
    
    # Visualize results
    if compare_optimization:
        fig = visualize_benchmark_results(results)
        plt.show()
    
    return results

def visualize_benchmark_results(results):
    """Create visualization of benchmark results."""
    # Create figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Extract data for plotting
    sample_sizes = results['sample_sizes']
    
    std_times = [results['standard'][size]['avg_time'] for size in sample_sizes]
    std_err = [(results['standard'][size]['max_time'] - results['standard'][size]['min_time'])/2 
               for size in sample_sizes]
    
    if 'optimized' in results and results['optimized']:
        opt_times = [results['optimized'][size]['avg_time'] for size in sample_sizes]
        opt_err = [(results['optimized'][size]['max_time'] - results['optimized'][size]['min_time'])/2 
                  for size in sample_sizes]
        
        improvements = [results['comparison'][size]['time_pct_reduction'] for size in sample_sizes]
    
    # Plot execution times
    x = np.arange(len(sample_sizes))
    width = 0.35
    
    ax1.bar(x - width/2, std_times, width, label='Standard', color='#ff9999', 
           yerr=std_err, capsize=5)
    
    if 'optimized' in results and results['optimized']:
        ax1.bar(x + width/2, opt_times, width, label='Optimized', color='#66b3ff',
               yerr=opt_err, capsize=5)
    
    ax1.set_xlabel('Sample Size (number of games)')
    ax1.set_ylabel('Execution Time (seconds)')
    ax1.set_title('Pipeline Execution Time Comparison')
    ax1.set_xticks(x)
    ax1.set_xticklabels(sample_sizes)
    ax1.legend()
    ax1.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Add time labels
    for i, v in enumerate(std_times):
        ax1.text(i - width/2, v + 0.1, f"{v:.2f}s", ha='center')
    
    if 'optimized' in results and results['optimized']:
        for i, v in enumerate(opt_times):
            ax1.text(i + width/2, v + 0.1, f"{v:.2f}s", ha='center')
    
    # Plot improvement percentages if available
    if 'optimized' in results and results['optimized']:
        ax2.bar(sample_sizes, improvements, color='#99ff99')
        ax2.set_xlabel('Sample Size (number of games)')
        ax2.set_ylabel('Performance Improvement (%)')
        ax2.set_title('Optimization Improvement')
        ax2.grid(axis='y', linestyle='--', alpha=0.7)
        
        # Add percentage labels
        for i, v in enumerate(improvements):
            ax2.text(sample_sizes[i], v + 1, f"{v:.1f}%", ha='center')
    else:
        ax2.text(0.5, 0.5, 'No optimization comparison available', 
                ha='center', va='center', transform=ax2.transAxes, fontsize=12)
    
    plt.tight_layout()
    return fig

# Run a benchmark demonstration
if __name__ == "__main__":
    # Create test data
    def create_test_data(num_games=5):
        """Create sample test data."""
        import random
        
        test_games = []
        for i in range(num_games):
            quarter = random.randint(1, 4)
            game = {
                'game_id': 1000 + i,
                'home_team': f"Team{i*2}",
                'away_team': f"Team{i*2+1}",
                'current_quarter': quarter,
                'home_q1': random.randint(20, 35) if quarter >= 1 else 0,
                'home_q2': random.randint(20, 35) if quarter >= 2 else 0,
                'home_q3': random.randint(20, 35) if quarter >= 3 else 0,
                'home_q4': random.randint(20, 35) if quarter >= 4 else 0,
                'away_q1': random.randint(20, 35) if quarter >= 1 else 0,
                'away_q2': random.randint(20, 35) if quarter >= 2 else 0,
                'away_q3': random.randint(20, 35) if quarter >= 3 else 0,
                'away_q4': random.randint(20, 35) if quarter >= 4 else 0
            }
            
            # Calculate current scores
            game['home_score'] = sum([game.get(f'home_q{q}', 0) for q in range(1, quarter+1)])
            game['away_score'] = sum([game.get(f'away_q{q}', 0) for q in range(1, quarter+1)])
            
            test_games.append(game)
        
        return pd.DataFrame(test_games)
    
    # Define a test pipeline
    def test_pipeline():
        """Run a simple test pipeline."""
        # Create test data
        test_data = create_test_data(num_games=5)
        
        # Create pipeline
        pipeline = PredictionPipeline(use_optimization=True)
        
        # Run quick benchmark
        profiler = PerformanceProfiler()
        _, profile = profiler.profile_pipeline(
            pipeline.run_complete_pipeline,
            "Sample_Benchmark",
            test_data
        )
        
        # Visualize profile
        fig = profiler.visualize_profile(profile)
        plt.show()
        
        # Run a more comprehensive benchmark
        print("\nRunning comprehensive benchmark...")
        benchmark_results = benchmark_performance(
            compare_optimization=True,
            sample_sizes=[1, 5, 10],
            iterations=2
        )
        
        print("\nBenchmark Summary:")
        for size in benchmark_results['sample_sizes']:
            if 'comparison' in benchmark_results and size in benchmark_results['comparison']:
                comparison = benchmark_results['comparison'][size]
                print(f"Sample size {size}: {comparison['time_pct_reduction']:.1f}% improvement, " 
                      f"{comparison['speedup_factor']:.2f}x speedup")
    
    test_pipeline()

In [ ]:
# Cell 29: Modular NBA Game Prediction Monitoring System

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output
import datetime
import time
import json
import os
from typing import Dict, List, Tuple, Optional, Union, Any, Callable
import logging
from functools import wraps
import pytz
from datetime import datetime, timedelta

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger("nba_monitoring")

# Import Supabase client (assumes correct configuration)
from caching.supabase_client import supabase

# --------------------------------------------------------------------
# HELPER: Fetch live game data from Supabase (mirroring Cells 1/3)
# --------------------------------------------------------------------

def fetch_live_games_from_supabase() -> pd.DataFrame:
    """
    Pull all rows from the 'nba_live_game_stats' table,
    convert game_date to Pacific Time, filter for today's games,
    and then select only rows where the stored status (uppercased)
    is in the live set {"Q1", "Q2", "Q3", "Q4", "OT", "BT", "HT"}.
    Also adds a 'game_status' column set to 'live'.
    """
    try:
        response = supabase.table("nba_live_game_stats").select("*").execute()
        if not response.data:
            print("No live game data available from Supabase.")
            return pd.DataFrame()
        
        df = pd.DataFrame(response.data)
        if "game_date" in df.columns:
            df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce", utc=True)
            pacific_tz = pytz.timezone("America/Los_Angeles")
            df["game_date_pt"] = df["game_date"].dt.tz_convert(pacific_tz)
            df["date_only_pt"] = df["game_date_pt"].dt.date
            now_pt = datetime.now(pacific_tz)
            today_pt = now_pt.date()
            df = df[df["date_only_pt"] == today_pt].copy()
            print(f"Found {len(df)} games scheduled for today in PT.")
        else:
            print("Column 'game_date' not found in Supabase data!")
        
        if "status" in df.columns:
            df["status"] = df["status"].astype(str).str.upper()
        else:
            df["status"] = ""
        
        live_statuses = {"Q1", "Q2", "Q3", "Q4", "OT", "BT", "HT"}
        active_df = df[df["status"].isin(live_statuses)].copy()
        # Add a new column 'game_status' set to 'live' for these filtered rows
        active_df["game_status"] = "live"
        print(f"Fetched {len(active_df)} active live games from Supabase.")
        return active_df
    except Exception as e:
        print(f"Error fetching live games from Supabase: {e}")
        return pd.DataFrame()

# ====================================================================
# DATA MANAGEMENT LAYER
# ====================================================================

class DataStore:
    """
    Manages storage and retrieval of prediction data and history.
    """
    def __init__(self, storage_dir: str = "./prediction_history"):
        self.storage_dir = storage_dir
        os.makedirs(storage_dir, exist_ok=True)
        self.recent_predictions = {}
        self.prediction_history = {}
        self._load_history()
        
    def _load_history(self):
        try:
            history_files = [f for f in os.listdir(self.storage_dir) if f.endswith('.json')]
            for file in history_files:
                try:
                    game_id = file.split('_')[1].split('.')[0]
                    file_path = os.path.join(self.storage_dir, file)
                    with open(file_path, 'r') as f:
                        game_history = json.load(f)
                    self.prediction_history[game_id] = game_history
                except Exception as e:
                    logger.warning(f"Error loading history file {file}: {str(e)}")
            logger.info(f"Loaded prediction history for {len(self.prediction_history)} games")
        except Exception as e:
            logger.error(f"Failed to load prediction history: {str(e)}")
    
    def save_prediction(self, prediction: Dict[str, Any]) -> bool:
        try:
            game_id = str(prediction.get('game_id'))
            if not game_id:
                logger.warning("Prediction missing game_id, cannot save")
                return False
            if 'timestamp' not in prediction:
                prediction['timestamp'] = datetime.now().isoformat()
            self.recent_predictions[game_id] = prediction
            if game_id not in self.prediction_history:
                self.prediction_history[game_id] = []
            self.prediction_history[game_id].append(prediction)
            self._save_game_history(game_id)
            return True
        except Exception as e:
            logger.error(f"Failed to save prediction: {str(e)}")
            return False
    
    def _save_game_history(self, game_id: str) -> bool:
        try:
            if game_id not in self.prediction_history:
                return False
            file_path = os.path.join(self.storage_dir, f"game_{game_id}.json")
            history = self.prediction_history[game_id]
            for pred in history:
                for key, value in pred.items():
                    if hasattr(value, "isoformat"):
                        pred[key] = value.isoformat()
            with open(file_path, 'w') as f:
                json.dump(history, f)
            return True
        except Exception as e:
            logger.error(f"Failed to save game history for {game_id}: {str(e)}")
            return False
    
    def get_game_history(self, game_id: str) -> List[Dict[str, Any]]:
        return self.prediction_history.get(game_id, [])
    
    def get_recent_prediction(self, game_id: str) -> Optional[Dict[str, Any]]:
        return self.recent_predictions.get(game_id)
    
    def get_all_recent_predictions(self) -> Dict[str, Dict[str, Any]]:
        return self.recent_predictions
    
    def clear_old_predictions(self, max_age_days: int = 7) -> int:
        cleared_count = 0
        cutoff_date = datetime.now() - timedelta(days=max_age_days)
        cutoff_str = cutoff_date.isoformat()
        for game_id in list(self.prediction_history.keys()):
            game_predictions = self.prediction_history[game_id]
            all_old = True
            for pred in game_predictions:
                if pred.get('timestamp', '') > cutoff_str:
                    all_old = False
                    break
            if all_old:
                del self.prediction_history[game_id]
                file_path = os.path.join(self.storage_dir, f"game_{game_id}.json")
                if os.path.exists(file_path):
                    os.remove(file_path)
                cleared_count += 1
        return cleared_count

# ====================================================================
# ALERT MANAGEMENT LAYER
# ====================================================================

class AlertManager:
    """
    Manages alerts and notifications for the monitoring system.
    """
    def __init__(self, data_store: DataStore, max_alerts: int = 100):
        self.data_store = data_store
        self.max_alerts = max_alerts
        self.alerts = []
        self.alert_thresholds = {
            'score_change': 15.0,
            'win_prob_change': 0.25,
            'unusual_momentum': 0.7,
            'prediction_error': 20.0,
            'confidence_mismatch': 0.3
        }
    
    def check_for_alerts(self, game_id: str) -> List[Dict[str, Any]]:
        """
        Check for alert conditions for a specific game.
        """
        game_history = self.data_store.get_game_history(game_id)
        if len(game_history) < 2:
            return []
        current = game_history[-1]
        previous = game_history[-2]
        new_alerts = []
        try:
            home_score_change = abs(
                current.get('predicted_home_score', current.get('predicted_home_final', 0)) - 
                previous.get('predicted_home_score', previous.get('predicted_home_final', 0))
            )
            away_score_change = abs(
                current.get('predicted_away_score', current.get('predicted_away_final', 0)) - 
                previous.get('predicted_away_score', previous.get('predicted_away_final', 0))
            )
            if home_score_change > self.alert_thresholds['score_change'] or away_score_change > self.alert_thresholds['score_change']:
                alert = {
                    'timestamp': datetime.now().isoformat(),
                    'game_id': game_id,
                    'alert_type': 'significant_score_change',
                    'message': f"Significant change in predicted score: Home {home_score_change:.1f} pts, Away {away_score_change:.1f} pts",
                    'severity': 'medium',
                    'data': {
                        'home_change': home_score_change,
                        'away_change': away_score_change,
                        'current': {
                            'home': current.get('predicted_home_score', current.get('predicted_home_final', 0)),
                            'away': current.get('predicted_away_score', current.get('predicted_away_final', 0))
                        },
                        'previous': {
                            'home': previous.get('predicted_home_score', previous.get('predicted_home_final', 0)),
                            'away': previous.get('predicted_away_score', previous.get('predicted_away_final', 0))
                        }
                    }
                }
                new_alerts.append(alert)
            
            win_prob_change = abs(
                current.get('win_probability', 0.5) - previous.get('win_probability', 0.5)
            )
            if win_prob_change > self.alert_thresholds['win_prob_change']:
                alert = {
                    'timestamp': datetime.now().isoformat(),
                    'game_id': game_id,
                    'alert_type': 'significant_win_prob_change',
                    'message': f"Win probability changed by {win_prob_change*100:.1f}% points",
                    'severity': 'high' if win_prob_change > 0.4 else 'medium',
                    'data': {
                        'win_prob_change': win_prob_change,
                        'current': current.get('win_probability', 0.5),
                        'previous': previous.get('win_probability', 0.5),
                        'direction': 'increased' if current.get('win_probability', 0.5) > previous.get('win_probability', 0.5) else 'decreased'
                    }
                }
                new_alerts.append(alert)
            
            momentum = abs(current.get('momentum_shift', current.get('cumulative_momentum', 0)))
            if momentum > self.alert_thresholds['unusual_momentum']:
                direction = "positive" if current.get('momentum_shift', current.get('cumulative_momentum', 0)) > 0 else "negative"
                alert = {
                    'timestamp': datetime.now().isoformat(),
                    'game_id': game_id,
                    'alert_type': 'unusual_momentum',
                    'message': f"Unusually high {direction} momentum: {momentum:.2f}",
                    'severity': 'medium',
                    'data': {
                        'momentum': current.get('momentum_shift', current.get('cumulative_momentum', 0)),
                        'threshold': self.alert_thresholds['unusual_momentum'],
                        'direction': direction
                    }
                }
                new_alerts.append(alert)
            
            if 'actual_home_score' in current and 'actual_away_score' in current:
                home_error = abs(
                    current.get('predicted_home_score', current.get('predicted_home_final', 0)) - current.get('actual_home_score', 0)
                )
                away_error = abs(
                    current.get('predicted_away_score', current.get('predicted_away_final', 0)) - current.get('actual_away_score', 0)
                )
                avg_error = (home_error + away_error) / 2
                if avg_error > self.alert_thresholds['prediction_error']:
                    alert = {
                        'timestamp': datetime.now().isoformat(),
                        'game_id': game_id,
                        'alert_type': 'high_prediction_error',
                        'message': f"High prediction error: {avg_error:.1f} points average",
                        'severity': 'high' if avg_error > 25 else 'medium',
                        'data': {
                            'home_error': home_error,
                            'away_error': away_error,
                            'avg_error': avg_error
                        }
                    }
                    new_alerts.append(alert)
            
            if 'prediction_confidence' in current and 'actual_home_score' in current and 'actual_away_score' in current:
                confidence = current.get('prediction_confidence', 0.8)
                actual_home = current.get('actual_home_score', 0)
                actual_away = current.get('actual_away_score', 0)
                predicted_home = current.get('predicted_home_score', current.get('predicted_home_final', 0))
                predicted_away = current.get('predicted_away_score', current.get('predicted_away_final', 0))
                actual_winner = 'home' if actual_home > actual_away else 'away'
                predicted_winner = 'home' if predicted_home > predicted_away else 'away'
                winner_correct = int(actual_winner == predicted_winner)
                confidence_mismatch = (confidence > 0.8 and not winner_correct) or (confidence < 0.5 and winner_correct)
                confidence_gap = confidence if not winner_correct else (1 - confidence) if winner_correct else 0
                if confidence_mismatch and confidence_gap > self.alert_thresholds['confidence_mismatch']:
                    alert = {
                        'timestamp': datetime.now().isoformat(),
                        'game_id': game_id,
                        'alert_type': 'confidence_mismatch',
                        'message': f"{'High confidence wrong prediction' if not winner_correct else 'Low confidence correct prediction'}",
                        'severity': 'medium',
                        'data': {
                            'confidence': confidence,
                            'winner_correct': winner_correct,
                            'confidence_gap': confidence_gap
                        }
                    }
                    new_alerts.append(alert)
            
            self.add_alerts(new_alerts)
        except Exception as e:
            logger.error(f"Error in check_for_alerts for game {game_id}: {e}")
        return new_alerts
    
    def add_alerts(self, alerts: List[Dict[str, Any]]):
        self.alerts.extend(alerts)
        if len(self.alerts) > self.max_alerts:
            self.alerts = self.alerts[-self.max_alerts:]
    
    def get_recent_alerts(self, limit: int = 10) -> List[Dict[str, Any]]:
        sorted_alerts = sorted(self.alerts, key=lambda x: x.get('timestamp', ''), reverse=True)
        return sorted_alerts[:limit]
    
    def get_alerts_by_game(self, game_id: str) -> List[Dict[str, Any]]:
        return [alert for alert in self.alerts if alert.get('game_id') == game_id]
    
    def set_alert_threshold(self, alert_type: str, value: float):
        if alert_type in self.alert_thresholds:
            self.alert_thresholds[alert_type] = value

# ====================================================================
# ANALYTICS LAYER
# ====================================================================

class PredictionAnalyzer:
    """
    Analyzes prediction data to generate insights and metrics.
    """
    def __init__(self, data_store: DataStore):
        self.data_store = data_store
        self.accuracy_metrics = {
            'by_quarter': {q: {'count': 0, 'mae': [], 'winner_accuracy': []} for q in range(5)},
            'by_context': {
                'close_game': {'count': 0, 'mae': [], 'winner_accuracy': []},
                'blowout': {'count': 0, 'mae': [], 'winner_accuracy': []},
                'high_momentum': {'count': 0, 'mae': [], 'winner_accuracy': []},
                'final_minutes': {'count': 0, 'mae': [], 'winner_accuracy': []}
            },
            'overall': {'count': 0, 'mae': [], 'winner_accuracy': []}
        }
    # (Other PredictionAnalyzer methods remain unchanged)
    def update_accuracy_metrics(self, game_id: str, actual_results: Dict[str, Any]) -> Dict[str, Any]:
        history = self.data_store.get_game_history(game_id)
        if not history:
            return {'status': 'error', 'message': 'No prediction history found'}
        update_summary = {
            'game_id': game_id,
            'predictions_processed': 0,
            'actual_home_score': actual_results.get('home_score', 0),
            'actual_away_score': actual_results.get('away_score', 0)
        }
        for prediction in history:
            quarter = prediction.get('current_quarter', 0)
            contexts = []
            home_score = prediction.get('home_score', prediction.get('current_home_score', 0))
            away_score = prediction.get('away_score', prediction.get('current_away_score', 0))
            score_diff = abs(home_score - away_score)
            if score_diff <= 5:
                contexts.append('close_game')
            if score_diff >= 20:
                contexts.append('blowout')
            momentum = abs(prediction.get('momentum_shift', prediction.get('cumulative_momentum', 0)))
            if momentum >= 0.6:
                contexts.append('high_momentum')
            if quarter == 4 and prediction.get('time_remaining', 12) <= 5:
                contexts.append('final_minutes')
            predicted_home = prediction.get('predicted_home_score', prediction.get('predicted_home_final', 0))
            predicted_away = prediction.get('predicted_away_score', prediction.get('predicted_away_final', 0))
            actual_home = actual_results.get('home_score', 0)
            actual_away = actual_results.get('away_score', 0)
            home_error = abs(predicted_home - actual_home)
            away_error = abs(predicted_away - actual_away)
            mae = (home_error + away_error) / 2
            actual_winner = 'home' if actual_home > actual_away else 'away'
            predicted_winner = 'home' if predicted_home > predicted_away else 'away'
            winner_correct = int(actual_winner == predicted_winner)
            if 0 <= quarter <= 4:
                self.accuracy_metrics['by_quarter'][quarter]['mae'].append(mae)
                self.accuracy_metrics['by_quarter'][quarter]['winner_accuracy'].append(winner_correct)
                self.accuracy_metrics['by_quarter'][quarter]['count'] += 1
            for context in contexts:
                if context in self.accuracy_metrics['by_context']:
                    self.accuracy_metrics['by_context'][context]['mae'].append(mae)
                    self.accuracy_metrics['by_context'][context]['winner_accuracy'].append(winner_correct)
                    self.accuracy_metrics['by_context'][context]['count'] += 1
            self.accuracy_metrics['overall']['mae'].append(mae)
            self.accuracy_metrics['overall']['winner_accuracy'].append(winner_correct)
            self.accuracy_metrics['overall']['count'] += 1
            update_summary['predictions_processed'] += 1
        if self.accuracy_metrics['overall']['mae']:
            update_summary['average_error'] = sum(self.accuracy_metrics['overall']['mae']) / len(self.accuracy_metrics['overall']['mae'])
        if self.accuracy_metrics['overall']['winner_accuracy']:
            update_summary['winner_accuracy'] = sum(self.accuracy_metrics['overall']['winner_accuracy']) / len(self.accuracy_metrics['overall']['winner_accuracy'])
        return update_summary
    
    def get_accuracy_summary(self) -> Dict[str, Any]:
        summary = {
            'by_quarter': {},
            'by_context': {},
            'overall': {},
            'count': self.accuracy_metrics['overall']['count']
        }
        for quarter, metrics in self.accuracy_metrics['by_quarter'].items():
            if metrics['mae']:
                summary['by_quarter'][quarter] = {
                    'mae': sum(metrics['mae']) / len(metrics['mae']),
                    'winner_accuracy': sum(metrics['winner_accuracy']) / len(metrics['winner_accuracy']),
                    'sample_size': len(metrics['mae']),
                    'count': metrics['count']
                }
        for context, metrics in self.accuracy_metrics['by_context'].items():
            if metrics['mae']:
                summary['by_context'][context] = {
                    'mae': sum(metrics['mae']) / len(metrics['mae']),
                    'winner_accuracy': sum(metrics['winner_accuracy']) / len(metrics['winner_accuracy']),
                    'sample_size': len(metrics['mae']),
                    'count': metrics['count']
                }
        overall = self.accuracy_metrics['overall']
        if overall['mae']:
            summary['overall'] = {
                'mae': sum(overall['mae']) / len(overall['mae']),
                'winner_accuracy': sum(overall['winner_accuracy']) / len(overall['winner_accuracy']),
                'sample_size': len(overall['mae']),
                'count': overall['count']
            }
        return summary
    
    def find_similar_games(self, prediction: Dict[str, Any], max_games: int = 5) -> List[Dict[str, Any]]:
        similar_games = []
        game_id = prediction.get('game_id', '')
        home_team = prediction.get('home_team', '')
        away_team = prediction.get('away_team', '')
        current_quarter = prediction.get('current_quarter', 0)
        score_diff = abs(
            prediction.get('home_score', prediction.get('current_home_score', 0)) - 
            prediction.get('away_score', prediction.get('current_away_score', 0))
        )
        momentum = abs(prediction.get('momentum_shift', prediction.get('cumulative_momentum', 0)))
        for hist_id, history in self.data_store.prediction_history.items():
            if hist_id == game_id:
                continue
            final_pred = history[-1] if history else None
            if not final_pred or 'actual_home_score' not in final_pred:
                continue
            similar_quarter_pred = None
            for pred in history:
                if pred.get('current_quarter', 0) == current_quarter:
                    similar_quarter_pred = pred
                    break
            if not similar_quarter_pred:
                continue
            similarity_factors = []
            teams_match = ((similar_quarter_pred.get('home_team', '') == home_team and 
                           similar_quarter_pred.get('away_team', '') == away_team) or
                          (similar_quarter_pred.get('home_team', '') == away_team and 
                           similar_quarter_pred.get('away_team', '') == home_team))
            if teams_match:
                similarity_factors.append(3.0)
            hist_score_diff = abs(
                similar_quarter_pred.get('home_score', similar_quarter_pred.get('current_home_score', 0)) - 
                similar_quarter_pred.get('away_score', similar_quarter_pred.get('current_away_score', 0))
            )
            score_diff_similarity = 1.0 - min(abs(score_diff - hist_score_diff) / 20.0, 1.0)
            similarity_factors.append(score_diff_similarity * 2.0)
            hist_momentum = abs(similar_quarter_pred.get('momentum_shift', similar_quarter_pred.get('cumulative_momentum', 0)))
            momentum_similarity = 1.0 - min(abs(momentum - hist_momentum) / 1.0, 1.0)
            similarity_factors.append(momentum_similarity)
            similarity = sum(similarity_factors) / len(similarity_factors) if similarity_factors else 0
            if similarity >= 0.6:
                actual_home = final_pred.get('actual_home_score', 0)
                actual_away = final_pred.get('actual_away_score', 0)
                predicted_home = similar_quarter_pred.get('predicted_home_score', similar_quarter_pred.get('predicted_home_final', 0))
                predicted_away = similar_quarter_pred.get('predicted_away_score', similar_quarter_pred.get('predicted_away_final', 0))
                home_error = predicted_home - actual_home
                away_error = predicted_away - actual_away
                similar_games.append({
                    'game_id': hist_id,
                    'similarity': similarity,
                    'home_team': similar_quarter_pred.get('home_team', ''),
                    'away_team': similar_quarter_pred.get('away_team', ''),
                    'quarter': similar_quarter_pred.get('current_quarter', 0),
                    'prediction': {
                        'home': predicted_home,
                        'away': predicted_away
                    },
                    'actual': {
                        'home': actual_home,
                        'away': actual_away
                    },
                    'error': {
                        'home': home_error,
                        'away': away_error,
                        'avg': (abs(home_error) + abs(away_error)) / 2
                    }
                })
        similar_games.sort(key=lambda x: x['similarity'], reverse=True)
        return similar_games[:max_games]
    
    def get_prediction_evolution(self, game_id: str) -> Dict[str, Any]:
        history = self.data_store.get_game_history(game_id)
        if not history:
            return {'game_id': game_id, 'status': 'error', 'message': 'No prediction history found'}
        timestamps = []
        quarters = []
        home_scores = []
        away_scores = []
        win_probs = []
        confidence_values = []
        for pred in history:
            ts = pred.get('timestamp', '')
            if isinstance(ts, str) and 'T' in ts:
                ts = ts.split('T')[1].split('.')[0]
            timestamps.append(ts)
            quarters.append(pred.get('current_quarter', 0))
            home_scores.append(pred.get('predicted_home_score', pred.get('predicted_home_final', 0)))
            away_scores.append(pred.get('predicted_away_score', pred.get('predicted_away_final', 0)))
            win_probs.append(pred.get('win_probability', 0.5))
            confidence = pred.get('prediction_confidence', None)
            confidence_values.append(confidence)
        actuals = None
        if history and 'actual_home_score' in history[-1]:
            actuals = {
                'home_score': history[-1].get('actual_home_score', 0),
                'away_score': history[-1].get('actual_away_score', 0)
            }
        return {
            'game_id': game_id,
            'home_team': history[0].get('home_team', '') if history else '',
            'away_team': history[0].get('away_team', '') if history else '',
            'timestamps': timestamps,
            'quarters': quarters,
            'home_scores': home_scores,
            'away_scores': away_scores,
            'win_probabilities': win_probs,
            'confidence_values': confidence_values,
            'count': len(history),
            'actual_results': actuals
        }

# ====================================================================
# VISUALIZATION LAYER
# ====================================================================

class DashboardTemplateEngine:
    """
    Handles HTML template management and rendering.
    """
    def __init__(self):
        self.global_css = """
            /* Global CSS styles */
            .dashboard { font-family: system-ui, -apple-system, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; max-width: 1200px; margin: 0 auto; }
            .dashboard-header { text-align: center; padding: 1rem; margin-bottom: 1rem; background: #0d6efd; color: white; border-radius: 0.5rem; }
            .dashboard-section { margin-bottom: 2rem; }
            .section-header { border-bottom: 2px solid #dee2e6; padding-bottom: 0.5rem; margin-bottom: 1rem; display: flex; justify-content: space-between; align-items: center; }
            .section-header h2 { margin: 0; font-size: 1.5rem; }
            .game-cards { display: grid; grid-template-columns: repeat(auto-fill, minmax(350px, 1fr)); gap: 1rem; }
            .game-card { border: 1px solid #dee2e6; border-radius: 0.5rem; padding: 1rem; margin-bottom: 1rem; background: white; box-shadow: 0 0.125rem 0.25rem rgba(0,0,0,0.075); }
            .alert-list { border: 1px solid #dee2e6; border-radius: 0.5rem; padding: 1rem; }
            .alert-item { padding: 0.5rem; margin-bottom: 0.5rem; border-radius: 0.25rem; }
            .alert-high { background-color: #f8d7da; border-left: 4px solid #dc3545; }
            .alert-medium { background-color: #fff3cd; border-left: 4px solid #ffc107; }
            .alert-low { background-color: #d1e7dd; border-left: 4px solid #198754; }
        """
        self.dashboard_js = """
            function switchTab(tabId) {
                document.querySelectorAll('.tab-content').forEach(tab => { tab.classList.remove('active'); });
                document.querySelectorAll('.tab').forEach(tab => { tab.classList.remove('active'); });
                document.getElementById(tabId).classList.add('active');
                document.querySelectorAll('.tab').forEach(tab => {
                    if (tab.getAttribute('onclick').includes(tabId)) { tab.classList.add('active'); }
                });
            }
            document.addEventListener('DOMContentLoaded', function() { /* initialize charts if needed */ });
        """
    
    def render_header(self, title="NBA Prediction Monitoring Dashboard"):
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        return f"""
        <div class="dashboard-header">
            <h1>{title}</h1>
            <p>Updated: {timestamp}</p>
        </div>
        """
    
    def render_active_games(self, games_df):
        if games_df.empty:
            return """
            <div class="dashboard-section">
                <div class="section-header">
                    <h2>Active Games</h2>
                </div>
                <p>No active games at this time.</p>
            </div>
            """
        html = """
        <div class="dashboard-section">
            <div class="section-header">
                <h2>Active Games</h2>
                <span class="update-status">Live updates</span>
            </div>
            <div class="game-cards">
        """
        for _, game in games_df.iterrows():
            game_id = game.get('game_id', '')
            home_team = game.get('home_team', '')
            away_team = game.get('away_team', '')
            current_quarter = game.get('current_quarter', 0)
            game_status = game.get('game_status', 'SCHEDULED')
            status_class = ""
            if game_status.upper() == "LIVE":
                status_class = "bg-danger text-white"
                status_text = f"LIVE Q{current_quarter}"
            elif game_status.upper() == "FINAL":
                status_class = "bg-success text-white"
                status_text = "FINAL"
            else:
                status_class = "bg-secondary text-white"
                status_text = "SCHEDULED"
            home_score = game.get('home_score', 0)
            away_score = game.get('away_score', 0)
            predicted_home = game.get('predicted_home_score', 0)
            predicted_away = game.get('predicted_away_score', 0)
            win_prob = game.get('win_probability', 0.5)
            win_prob_text = f"{win_prob*100:.1f}%"
            confidence = game.get('prediction_confidence', None)
            confidence_text = f"{confidence*100:.1f}%" if confidence is not None else "N/A"
            html += f"""
            <div class="game-card" data-game-id="{game_id}">
                <div class="d-flex justify-content-between align-items-center mb-2">
                    <h5 class="mb-0">{home_team} vs {away_team}</h5>
                    <span class="badge rounded-pill {status_class} px-3 py-2">{status_text}</span>
                </div>
                <div class="d-flex justify-content-between align-items-center mb-3">
                    <div class="text-center">
                        <div style="font-weight: bold; font-size: 1.5rem;">{home_score}</div>
                        <div style="color: #666;">{home_team}</div>
                    </div>
                    <div class="text-center">
                        <div style="font-weight: bold;">vs</div>
                    </div>
                    <div class="text-center">
                        <div style="font-weight: bold; font-size: 1.5rem;">{away_score}</div>
                        <div style="color: #666;">{away_team}</div>
                    </div>
                </div>
                <div class="mt-3">
                    <div class="d-flex justify-content-between text-muted">
                        <span>Predicted Final</span>
                        <span>Win Probability</span>
                    </div>
                    <div class="d-flex justify-content-between">
                        <strong>{predicted_home:.1f} - {predicted_away:.1f}</strong>
                        <strong>{win_prob_text}</strong>
                    </div>
                    <div class="progress mt-2" style="height: 8px;">
                        <div class="progress-bar" role="progressbar" style="width: {win_prob*100}%"></div>
                    </div>
                </div>
                <div class="mt-3 text-muted d-flex justify-content-between">
                    <small>Confidence: {confidence_text}</small>
                    <small>Game ID: {game_id}</small>
                </div>
            </div>
            """
        html += """
            </div>
        </div>
        """
        return html
    
    def render_alerts(self, alerts):
        if not alerts:
            return """
            <div class="dashboard-section">
                <div class="section-header">
                    <h2>Alerts & Notifications</h2>
                </div>
                <p>No alerts at this time.</p>
            </div>
            """
        html = """
        <div class="dashboard-section">
            <div class="section-header">
                <h2>Alerts & Notifications</h2>
            </div>
            <div class="alert-list">
        """
        for alert in alerts:
            severity = alert.get('severity', 'medium')
            alert_class = f"alert-{severity}"
            timestamp = alert.get('timestamp', '')
            if isinstance(timestamp, str) and 'T' in timestamp:
                timestamp = timestamp.split('T')[1].split('.')[0]
            html += f"""
            <div class="alert-item {alert_class}">
                <div>
                    <strong>{alert.get('alert_type', 'Alert').replace('_', ' ').title()}</strong> - {alert.get('message', '')}
                </div>
                <div style="font-size:0.8rem; color:#666">
                    Game: {alert.get('game_id', '')}, Time: {timestamp}
                </div>
            </div>
            """
        html += """
            </div>
        </div>
        """
        return html
    
    def render_historical_comparisons(self, active_games_df, prediction_analyzer):
        if active_games_df.empty:
            return """
            <div class="dashboard-section">
                <div class="section-header">
                    <h2>Historical Context & Performance</h2>
                </div>
                <p>No active games to compare.</p>
            </div>
            """
        html = """
        <div class="dashboard-section">
            <div class="section-header">
                <h2>Historical Context & Performance</h2>
            </div>
        """
        html += """<div class="tabs">"""
        for i, (_, game) in enumerate(active_games_df.iterrows()):
            game_id = game.get('game_id', i)
            home_team = game.get('home_team', '')
            away_team = game.get('away_team', '')
            active_class = "active" if i == 0 else ""
            html += f"""
            <div class="tab {active_class}" onclick="switchTab('game-tab-{game_id}')">
                {home_team} vs {away_team}
            </div>
            """
        html += """</div>"""
        for i, (_, game) in enumerate(active_games_df.iterrows()):
            game_id = str(game.get('game_id', i))
            active_class = "active" if i == 0 else ""
            html += f"""
            <div id="game-tab-{game_id}" class="tab-content {active_class}">
            """
            accuracy_summary = prediction_analyzer.get_accuracy_summary()
            html += """
            <div class="prediction-metrics">
            """
            quarter = int(game.get('current_quarter', 0))
            quarter_metrics = accuracy_summary.get('by_quarter', {}).get(quarter, {})
            quarter_mae = quarter_metrics.get('mae', 0)
            quarter_accuracy = quarter_metrics.get('winner_accuracy', 0)
            quarter_count = quarter_metrics.get('count', 0)
            overall_metrics = accuracy_summary.get('overall', {})
            overall_mae = overall_metrics.get('mae', 0)
            overall_accuracy = overall_metrics.get('winner_accuracy', 0)
            html += f"""
            <div class="metric-card">
                <div class="metric-label">Quarter {quarter} Error</div>
                <div class="metric-value">{quarter_mae:.1f}</div>
                <div class="metric-label">points (avg)</div>
                <div class="metric-label">Sample: {quarter_count}</div>
            </div>
            
            <div class="metric-card">
                <div class="metric-label">Quarter {quarter} Accuracy</div>
                <div class="metric-value">{quarter_accuracy*100:.1f}%</div>
                <div class="metric-label">winner prediction</div>
            </div>
            
            <div class="metric-card">
                <div class="metric-label">Overall Error</div>
                <div class="metric-value">{overall_mae:.1f}</div>
                <div class="metric-label">points (avg)</div>
            </div>
            
            <div class="metric-card">
                <div class="metric-label">Overall Accuracy</div>
                <div class="metric-value">{overall_accuracy*100:.1f}%</div>
                <div class="metric-label">winner prediction</div>
            </div>
            """
            html += """</div>"""
            similar_games = prediction_analyzer.find_similar_games(game.to_dict())
            if similar_games:
                html += """
                <div class="similar-games">
                    <h3>Similar Historical Games</h3>
                """
                for similar in similar_games:
                    similarity = similar.get('similarity', 0) * 100
                    html += f"""
                    <div class="similar-game-item">
                        <div>
                            <strong>{similar.get('home_team', '')} vs {similar.get('away_team', '')}</strong>
                            <span style="float:right; color:#666">Similarity: {similarity:.0f}%</span>
                        </div>
                        <div style="margin-top:0.5rem">
                            <div style="display:flex; justify-content:space-between">
                                <div>
                                    <span style="color:#666">Predicted:</span> 
                                    {similar.get('prediction', {}).get('home', 0):.1f} - {similar.get('prediction', {}).get('away', 0):.1f}
                                </div>
                                <div>
                                    <span style="color:#666">Actual:</span> 
                                    {similar.get('actual', {}).get('home', 0)} - {similar.get('actual', {}).get('away', 0)}
                                </div>
                            </div>
                            <div style="font-size:0.9rem; color:#666; text-align:right">
                                Error: {similar.get('error', {}).get('avg', 0):.1f} points average
                            </div>
                        </div>
                    </div>
                    """
                html += """</div>"""
            evolution_data = prediction_analyzer.get_prediction_evolution(game_id)
            if evolution_data and evolution_data.get('count', 0) > 1:
                chart_data = {
                    'labels': [f"Update {i+1}" for i in range(len(evolution_data.get('timestamps', [])))],
                    'datasets': [
                        {
                            'label': f"{evolution_data.get('home_team', 'Home')} Score",
                            'data': evolution_data.get('home_scores', []),
                            'borderColor': '#0d6efd',
                            'backgroundColor': 'rgba(13, 110, 253, 0.1)',
                        },
                        {
                            'label': f"{evolution_data.get('away_team', 'Away')} Score",
                            'data': evolution_data.get('away_scores', []),
                            'borderColor': '#dc3545',
                            'backgroundColor': 'rgba(220, 53, 69, 0.1)',
                        },
                        {
                            'label': 'Win Probability (%)',
                            'data': [wp * 100 for wp in evolution_data.get('win_probabilities', [])],
                            'borderColor': '#198754',
                            'backgroundColor': 'rgba(25, 135, 84, 0.1)',
                            'yAxisID': 'y1'
                        }
                    ],
                    'options': {
                        'scales': {
                            'y': {
                                'title': {'display': True, 'text': 'Score'}
                            },
                            'y1': {
                                'position': 'right',
                                'title': {'display': True, 'text': 'Win Probability (%)'},
                                'min': 0,
                                'max': 100,
                                'grid': {'drawOnChartArea': False}
                            }
                        }
                    }
                }
                if evolution_data.get('actual_results'):
                    actuals = evolution_data.get('actual_results')
                    chart_data['datasets'].append({
                        'label': 'Actual Home',
                        'data': [None] * (len(evolution_data.get('quarters', [])) - 1) + [actuals.get('home_score')],
                        'borderColor': '#0d6efd',
                        'backgroundColor': 'rgba(13, 110, 253, 0.5)',
                        'borderWidth': 2,
                        'pointRadius': 6,
                        'pointStyle': 'star'
                    })
                    chart_data['datasets'].append({
                        'label': 'Actual Away',
                        'data': [None] * (len(evolution_data.get('quarters', [])) - 1) + [actuals.get('away_score')],
                        'borderColor': '#dc3545',
                        'backgroundColor': 'rgba(220, 53, 69, 0.5)',
                        'borderWidth': 2,
                        'pointRadius': 6,
                        'pointStyle': 'star'
                    })
                html += f"""
                <div class="prediction-evolution mt-4">
                    <h3>Prediction Evolution</h3>
                    <div class="chart-container" style="position: relative; height: 300px;"></div>
                    <div class="evolution-data" style="display: none;">
                        {json.dumps(chart_data)}
                    </div>
                </div>
                """
            html += """</div>"""  # Close tab content
        html += """
        </div>
        """
        return html
    
    def render_dashboard(self, active_games_df, alerts, prediction_analyzer, container_id="monitoring-dashboard"):
        # IMPORTANT: Use only the active games that are marked as live.
        live_games_df = active_games_df[active_games_df['game_status'].str.lower() == 'live']
        html = f"""
        <div id="{container_id}" class="dashboard">
            <style>
                {self.global_css}
            </style>
            {self.render_header()}
            {self.render_active_games(live_games_df)}
            {self.render_alerts(alerts)}
            {self.render_historical_comparisons(live_games_df, prediction_analyzer)}
            <script>
                {self.dashboard_js}
            </script>
        </div>
        """
        return html

# ====================================================================
# CONTROLLER / COORDINATOR
# ====================================================================

class MonitoringDashboard:
    """
    Main dashboard controller that coordinates components.
    """
    def __init__(self, storage_dir: str = "./prediction_history"):
        self.data_store = DataStore(storage_dir)
        self.alert_manager = AlertManager(self.data_store)
        self.prediction_analyzer = PredictionAnalyzer(self.data_store)
        self.template_engine = DashboardTemplateEngine()
        self.active_games_df = pd.DataFrame()
        self.last_update = datetime.now()
    
    def add_prediction(self, prediction_data, check_for_alerts=True):
        if not isinstance(prediction_data, dict) or 'game_id' not in prediction_data:
            logger.warning("Invalid prediction data: missing game_id")
            return False
        success = self.data_store.save_prediction(prediction_data)
        if not success:
            return False
        if check_for_alerts:
            self.alert_manager.check_for_alerts(prediction_data['game_id'])
        return True
    
    def update_with_active_games(self):
        # Use our helper function to fetch live games from Supabase
        active_games_df = fetch_live_games_from_supabase()
        if active_games_df.empty:
            logger.info("No active games found.")
            self.active_games_df = active_games_df
            return False
        self.active_games_df = active_games_df
        for _, game in active_games_df.iterrows():
            self.add_prediction(game.to_dict(), check_for_alerts=True)
        self.last_update = datetime.now()
        return True
    
    def update_with_actual_results(self, game_id, actual_results):
        latest_prediction = self.data_store.get_recent_prediction(str(game_id))
        if not latest_prediction:
            return {'status': 'error', 'message': 'Game not found'}
        updated_prediction = latest_prediction.copy()
        updated_prediction['actual_home_score'] = actual_results.get('home_score', 0)
        updated_prediction['actual_away_score'] = actual_results.get('away_score', 0)
        updated_prediction['timestamp'] = datetime.now().isoformat()
        self.data_store.save_prediction(updated_prediction)
        update_summary = self.prediction_analyzer.update_accuracy_metrics(str(game_id), actual_results)
        return update_summary
    
    def render_dashboard(self, include_alerts=True, include_history=True):
        alerts = []
        if include_alerts:
            alerts = self.alert_manager.get_recent_alerts()
        html = self.template_engine.render_dashboard(self.active_games_df, alerts, self.prediction_analyzer)
        return html
    
    def display_in_notebook(self, include_alerts=True, include_history=True):
        from IPython.display import display, HTML, clear_output
        dashboard_html = self.render_dashboard(include_alerts=include_alerts, include_history=include_history)
        clear_output(wait=True)
        display(HTML(dashboard_html))
    
    def start_live_monitoring(self, fetch_games_func, update_interval=60, max_updates=None, is_notebook=True):
        update_count = 0
        try:
            while max_updates is None or update_count < max_updates:
                logger.info(f"Running monitoring update #{update_count+1}")
                try:
                    active_games = fetch_games_func()
                    if active_games is not None and not active_games.empty:
                        self.update_with_active_games()
                        if is_notebook:
                            self.display_in_notebook()
                        else:
                            print(f"\n=== Monitoring Update #{update_count+1} ===")
                            print(f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
                            print(f"Active Games: {len(active_games)}")
                            print(f"Total Alerts: {len(self.alert_manager.alerts)}")
                            recent_alerts = self.alert_manager.get_recent_alerts(3)
                            if recent_alerts:
                                print("\nRecent Alerts:")
                                for alert in recent_alerts:
                                    print(f"- {alert.get('alert_type', 'Alert')}: {alert.get('message', '')}")
                    else:
                        logger.info("No active games found for monitoring")
                        if is_notebook:
                            self.display_in_notebook()
                        else:
                            print("No active games found for monitoring")
                except Exception as e:
                    logger.error(f"Error fetching games: {e}")
                update_count += 1
                if max_updates is None or update_count < max_updates:
                    time.sleep(update_interval)
        except KeyboardInterrupt:
            logger.info("Monitoring stopped by user")
        except Exception as e:
            logger.error(f"Error in monitoring: {e}")
            import traceback
            traceback.print_exc()
        finally:
            logger.info("Monitoring ended")
    
    def get_active_games(self):
        return self.active_games_df
    
    def get_alerts(self, limit=10):
        return self.alert_manager.get_recent_alerts(limit)
    
    def get_game_history(self, game_id):
        return self.data_store.get_game_history(str(game_id))
    
    def get_accuracy_summary(self):
        return self.prediction_analyzer.get_accuracy_summary()

# ====================================================================
# USAGE EXAMPLE
# ====================================================================

def demonstrate_dashboard():
    # For demonstration, fetch live data from Supabase.
    dashboard = MonitoringDashboard()
    dashboard.update_with_active_games()
    try:
        dashboard.display_in_notebook()
    except:
        print(dashboard.render_dashboard())
    return dashboard

if __name__ == "__main__":
    dashboard = demonstrate_dashboard()
